In [1]:
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv
from torch_geometric.loader import NeighborLoader
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score
from tqdm import tqdm
import gc
import numpy as np
import psutil
import os

def log_memory(tag=""):
    proc = psutil.Process(os.getpid())
    rss_gb = proc.memory_info().rss / (1024 ** 3)
    gpu_str = ""
    if torch.cuda.is_available():
        gpu_alloc = torch.cuda.memory_allocated() / (1024 ** 3)
        gpu_str = f" | GPU: {gpu_alloc:.2f}GB"
    print(f"  💾 [{tag}] RAM: {rss_gb:.2f}GB{gpu_str}")

print("=" * 60)
print("🛡️ [TGAT v2] TransformerConv + 그래프 보존 + 튜닝 파이프라인")
print("=" * 60)

# ===================== 설정 =====================
# 엣지 다운샘플링 제거 (그래프 구조 100% 보존)
# GNN 노드 다운샘플링만 최소한으로 적용
NEG_NODE_RATIO_GNN = 50       # 양성 대비 음성 노드 50배 (이전 10배 → 50배)
NEG_ROW_RATIO_XGB = 20        # XGBoost는 유지
RANDOM_SEED = 42

# GNN 학습 하이퍼파라미터
HIDDEN_DIM = 64
EDGE_DIM = 16                 # TransformerConv edge feature 차원
N_HEADS = 4                   # 어텐션 헤드 수 (2 → 4)
N_EPOCHS = 30                 # 에포크 수 (20 → 30)
LR = 0.003                    # 학습률
BATCH_SIZE = 2048
NUM_NEIGHBORS = [15, 10]      # 이웃 샘플링 수 복원
# ================================================

raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"

# =============================================================
# 1. 데이터 로드 및 6:2:2 시간 분할
# =============================================================
print("📂 Raw 데이터 및 V2 데이터 로딩 중...")
df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))

if df_raw["Timestamp"].dtype == pl.Utf8:
    df_raw = df_raw.with_columns(
        pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False)
    )

df_v2 = df_v2.sort("time_group")
total_count = len(df_v2)
train_idx = int(total_count * 0.6)
val_idx = int(total_count * 0.8)

train_cutoff = df_v2["time_group"][train_idx]
val_cutoff = df_v2["time_group"][val_idx]

print(f"⏱️ Train (60%) 컷오프 시간: {train_cutoff}")
print(f"⏱️ Val   (80%) 컷오프 시간: {val_cutoff}")

all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")

num_nodes = len(all_accounts)
print(f"📊 전체 노드 수: {num_nodes}")
log_memory("데이터 로드 완료")

# =============================================================
# 2. 미래 정보 차단 — Train 기간 데이터만으로 그래프 구축
#    ★ 엣지 다운샘플링 제거: 구조 100% 보존
# =============================================================
print(f"\n✂️ {train_cutoff} 이전 데이터로 그래프 구축 (엣지 100% 보존)...")

df_raw_train = df_raw.filter(pl.col("Timestamp") < train_cutoff)
del df_raw; gc.collect()  # Raw 데이터 즉시 해제

# 엣지 구성
df_edges_train = df_raw_train.select(["from_acc", "to_acc", "Timestamp"]).with_columns([
    pl.col("from_acc").cast(pl.Utf8),
    pl.col("to_acc").cast(pl.Utf8)
])
df_edges_train = df_edges_train.join(
    all_accounts.rename({"account_id": "from_acc", "node_id": "src_id"}), on="from_acc", how="left"
)
df_edges_train = df_edges_train.join(
    all_accounts.rename({"account_id": "to_acc", "node_id": "dst_id"}), on="to_acc", how="left"
)

# 엣지 타임스탬프 → 상대 초
timestamps_raw = df_edges_train["Timestamp"]
min_ts = timestamps_raw.min()
max_ts = timestamps_raw.max()
total_seconds = (max_ts - min_ts).total_seconds()

# 0~1 정규화 (TransformerConv edge feature로 활용)
edge_time_normalized = (
    (timestamps_raw - min_ts).dt.total_microseconds().cast(pl.Float64) / 1e6 / max(total_seconds, 1.0)
).to_numpy().astype(np.float32)

edge_index_train = torch.tensor(
    df_edges_train.select(["src_id", "dst_id"]).to_numpy().T, dtype=torch.long
)

# edge_attr: 정규화된 시간 + 추가 시간 피처 (주기성)
# [normalized_time, sin(hour), cos(hour), sin(day_of_week), cos(day_of_week)]
hours = df_edges_train["Timestamp"].dt.hour().to_numpy().astype(np.float32)
dow = df_edges_train["Timestamp"].dt.weekday().to_numpy().astype(np.float32)

edge_features = np.stack([
    edge_time_normalized,
    np.sin(2 * np.pi * hours / 24),
    np.cos(2 * np.pi * hours / 24),
    np.sin(2 * np.pi * dow / 7),
    np.cos(2 * np.pi * dow / 7),
], axis=1)  # (E, 5)

edge_attr_train = torch.tensor(edge_features, dtype=torch.float32)
EDGE_RAW_DIM = edge_attr_train.shape[1]  # 5

n_edges = edge_index_train.shape[1]
print(f"📊 Train 엣지 수: {n_edges:,} (100% 보존)")
print(f"📊 Edge feature 차원: {EDGE_RAW_DIM} (time_norm, sin/cos_hour, sin/cos_dow)")

del df_raw_train, df_edges_train, timestamps_raw, hours, dow, edge_features, edge_time_normalized
gc.collect()
log_memory("엣지 구성 완료")

# 노드 피처 (Train 기간만)
df_v2_train = df_v2.filter(pl.col("time_group") < train_cutoff)

exclude_cols = ["account_id", "time_group", "is_laundering", "mode_format", "currency_mode"]
gnn_feature_cols = [c for c in df_v2.columns if c not in exclude_cols]

df_v2_node = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols
])
df_node_features = all_accounts.join(df_v2_node, on="account_id", how="left").fill_null(0.0)
X_node = torch.tensor(df_node_features.select(gnn_feature_cols).to_numpy(), dtype=torch.float32)
IN_DIM = X_node.shape[1]

# 레이블
target_labels = (
    df_v2_train.filter(pl.col("is_laundering") == 1)
    .select("account_id").unique()
    .with_columns(pl.lit(1).alias("label"))
)
df_labels = all_accounts.join(target_labels, on="account_id", how="left").fill_null(0)
Y_node = torch.tensor(df_labels["label"].to_numpy(), dtype=torch.long)

n_pos_total = int(Y_node.sum().item())
print(f"📊 양성 노드: {n_pos_total:,} / {num_nodes:,} ({n_pos_total/num_nodes*100:.2f}%)")

# Train 마스크 (다운샘플링 완화: 양성 전체 + 음성 50배)
active_accounts = df_v2_train["account_id"].unique()
active_df = pl.DataFrame({"account_id": active_accounts, "is_active": True})
mask_df = all_accounts.join(active_df, on="account_id", how="left").fill_null(False)

pos_node_ids = set(df_labels.filter(pl.col("label") == 1)["node_id"].to_list())
active_node_ids = mask_df.filter(pl.col("is_active"))["node_id"].to_list()

active_pos = [nid for nid in active_node_ids if nid in pos_node_ids]
active_neg = [nid for nid in active_node_ids if nid not in pos_node_ids]

n_neg_keep_gnn = min(len(active_pos) * NEG_NODE_RATIO_GNN, len(active_neg))
np.random.seed(RANDOM_SEED)
sampled_neg = np.random.choice(active_neg, size=n_neg_keep_gnn, replace=False).tolist()
sampled_nodes = set(active_pos + sampled_neg)

train_mask_np = np.zeros(num_nodes, dtype=bool)
for nid in sampled_nodes:
    train_mask_np[nid] = True
train_mask = torch.tensor(train_mask_np, dtype=torch.bool)

print(f"📊 GNN 학습 노드: 양성 {len(active_pos):,} + 음성 {n_neg_keep_gnn:,} "
      f"= {len(sampled_nodes):,} (전체 활성 {len(active_node_ids):,} 중, "
      f"비율 {len(sampled_nodes)/len(active_node_ids)*100:.1f}%)")

del df_v2_node, df_node_features, mask_df, active_df
gc.collect()
log_memory("노드 피처+마스크 완료")

# =============================================================
# 3. 모델 정의: TransformerConv 기반 TGAT
# =============================================================
print("\n🧠 TransformerConv 기반 TGAT 모델 정의...")


class EdgeProjection(nn.Module):
    """엣지 raw feature를 EDGE_DIM 차원으로 투영."""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim)
        )

    def forward(self, edge_attr):
        return self.proj(edge_attr)


class TGAT_v2(nn.Module):
    """
    2-layer Temporal Graph Attention using PyG TransformerConv.
    - TransformerConv은 안정적인 multi-head attention + edge feature 지원
    - edge_attr로 시간 정보를 직접 주입
    """
    def __init__(self, in_dim, hidden_dim, edge_raw_dim, edge_dim, n_heads, dropout=0.3):
        super().__init__()

        # Edge feature 투영
        self.edge_proj = EdgeProjection(edge_raw_dim, edge_dim)

        # Layer 1: in_dim → hidden_dim
        self.conv1 = TransformerConv(
            in_channels=in_dim,
            out_channels=hidden_dim // n_heads,
            heads=n_heads,
            edge_dim=edge_dim,
            dropout=dropout,
            concat=True   # concat heads → hidden_dim
        )
        self.norm1 = nn.LayerNorm(hidden_dim)

        # Layer 2: hidden_dim → hidden_dim
        self.conv2 = TransformerConv(
            in_channels=hidden_dim,
            out_channels=hidden_dim // n_heads,
            heads=n_heads,
            edge_dim=edge_dim,
            dropout=dropout,
            concat=True
        )
        self.norm2 = nn.LayerNorm(hidden_dim)

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )
        self.dropout = dropout

    def forward(self, x, edge_index, edge_attr):
        edge_emb = self.edge_proj(edge_attr)

        h = self.conv1(x, edge_index, edge_emb)
        h = self.norm1(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        h = self.conv2(h, edge_index, edge_emb)
        h = self.norm2(h)
        h = F.relu(h)

        return self.classifier(h).squeeze(-1)

    @torch.no_grad()
    def get_embedding(self, x, edge_index, edge_attr):
        edge_emb = self.edge_proj(edge_attr)

        h = self.conv1(x, edge_index, edge_emb)
        h = self.norm1(h)
        h = F.relu(h)

        h = self.conv2(h, edge_index, edge_emb)
        h = self.norm2(h)
        h = F.relu(h)
        return h


# =============================================================
# 4. 그래프 데이터 + 학습
# =============================================================
print("\n🔥 TGAT v2 학습 시작...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"📌 디바이스: {device}")

graph_data = Data(
    x=X_node,
    edge_index=edge_index_train,
    edge_attr=edge_attr_train,
    y=Y_node
)
graph_data.train_mask = train_mask
graph_data.num_nodes = num_nodes

del X_node, edge_index_train, edge_attr_train, Y_node, train_mask
gc.collect()

train_loader = NeighborLoader(
    graph_data,
    num_neighbors=NUM_NEIGHBORS,
    batch_size=BATCH_SIZE,
    input_nodes=graph_data.train_mask,
    shuffle=True,
    num_workers=4
)
full_loader = NeighborLoader(
    graph_data,
    num_neighbors=NUM_NEIGHBORS,
    batch_size=BATCH_SIZE,
    input_nodes=None,
    shuffle=False,
    num_workers=4
)

model = TGAT_v2(
    in_dim=IN_DIM,
    hidden_dim=HIDDEN_DIM,
    edge_raw_dim=EDGE_RAW_DIM,
    edge_dim=EDGE_DIM,
    n_heads=N_HEADS,
    dropout=0.3
).to(device)

# pos_weight 계산: 실제 학습 마스크 내 양성 비율 기반
n_train_pos = len(active_pos)
n_train_total = len(sampled_nodes)
pw = (n_train_total - n_train_pos) / max(n_train_pos, 1)
print(f"📊 BCEWithLogitsLoss pos_weight: {pw:.1f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw], device=device))
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS, eta_min=1e-5)

log_memory("학습 시작 전")

model.train()
best_loss = float('inf')
patience_counter = 0
PATIENCE = 7

for epoch in range(1, N_EPOCHS + 1):
    total_loss = 0.0
    num_batches = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{N_EPOCHS}", leave=False):
        batch = batch.to(device)
        optimizer.zero_grad()

        # edge_attr 처리
        if batch.edge_attr is not None and batch.edge_attr.numel() > 0:
            batch_edge_attr = batch.edge_attr
        else:
            batch_edge_attr = torch.zeros(batch.edge_index.size(1), EDGE_RAW_DIM, device=device)

        out = model(batch.x, batch.edge_index, batch_edge_attr)
        loss = criterion(out[:batch.batch_size], batch.y[:batch.batch_size].float())
        loss.backward()

        # Gradient clipping으로 학습 안정화
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    scheduler.step()
    avg_loss = total_loss / max(num_batches, 1)
    current_lr = optimizer.param_groups[0]['lr']

    if epoch % 5 == 0 or epoch == 1:
        print(f"  → Epoch {epoch:2d} | Loss: {avg_loss:.4f} | LR: {current_lr:.6f}")
        log_memory(f"Epoch {epoch}")

    # Early stopping
    if avg_loss < best_loss - 0.001:
        best_loss = avg_loss
        patience_counter = 0
        # 최적 모델 저장
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print(f"  ⏹️ Early stopping at epoch {epoch} (patience {PATIENCE})")
        break

# 최적 모델 복원
if 'best_state' in dir() or 'best_state' in locals():
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    print(f"  ✅ Best model 복원 (loss: {best_loss:.4f})")
    del best_state

# =============================================================
# 5. 임베딩 추출
# =============================================================
print("\n📦 TGAT v2 임베딩 추출 중...")
model.eval()
all_emb = []
with torch.no_grad():
    for batch in tqdm(full_loader, desc="Embedding Extraction"):
        batch = batch.to(device)
        if batch.edge_attr is not None and batch.edge_attr.numel() > 0:
            batch_edge_attr = batch.edge_attr
        else:
            batch_edge_attr = torch.zeros(batch.edge_index.size(1), EDGE_RAW_DIM, device=device)

        emb = model.get_embedding(batch.x, batch.edge_index, batch_edge_attr)
        all_emb.append(emb[:batch.batch_size].cpu())

final_emb = torch.cat(all_emb, dim=0).numpy()
emb_cols = [f"tgat_emb_{i}" for i in range(HIDDEN_DIM)]

df_tgat = pl.concat([
    all_accounts.select("account_id"),
    pl.DataFrame(final_emb, schema=emb_cols)
], how="horizontal")

print(f"✅ 임베딩 shape: {final_emb.shape}")

del graph_data, all_emb, final_emb, all_accounts
torch.cuda.empty_cache()
gc.collect()
log_memory("임베딩 추출 완료")

# =============================================================
# 6. XGBoost 결합 및 6:2:2 분할
# =============================================================
print("\n🚀 XGBoost 결합 및 6:2:2 시간 기준 분할...")

xgb_exclude_cols = ["account_id", "time_group", "is_laundering", "mode_format", "currency_mode", "date"]
xgb_feature_cols = [c for c in df_v2.columns if c not in xgb_exclude_cols] + emb_cols

# [1] Train — 다운샘플링 (양성 전체 + 음성 20배)
df_train_full = (
    df_v2.filter(pl.col("time_group") < train_cutoff)
    .join(df_tgat, on="account_id", how="left")
    .fill_null(0.0)
)
df_train_pos = df_train_full.filter(pl.col("is_laundering") == 1)
df_train_neg = df_train_full.filter(pl.col("is_laundering") == 0)
n_neg_keep_xgb = min(len(df_train_pos) * NEG_ROW_RATIO_XGB, len(df_train_neg))
df_train_neg_sampled = df_train_neg.sample(n=n_neg_keep_xgb, seed=RANDOM_SEED)
df_train = pl.concat([df_train_pos, df_train_neg_sampled]).sample(fraction=1.0, seed=RANDOM_SEED)

X_train = df_train.select(xgb_feature_cols).to_pandas()
y_train = df_train["is_laundering"].to_numpy()
print(f"📊 Train: {len(df_train_full):,} → {len(df_train):,} "
      f"(양성 {len(df_train_pos):,} + 음성 {n_neg_keep_xgb:,}), 양성비율: {y_train.mean()*100:.2f}%")
del df_train, df_train_full, df_train_pos, df_train_neg, df_train_neg_sampled; gc.collect()

# [2] Val (다운샘플링 없음 — 실제 분포 유지)
df_val = (
    df_v2.filter((pl.col("time_group") >= train_cutoff) & (pl.col("time_group") < val_cutoff))
    .join(df_tgat, on="account_id", how="left")
    .fill_null(0.0)
)
X_val = df_val.select(xgb_feature_cols).to_pandas()
y_val = df_val["is_laundering"].to_numpy()
print(f"📊 Val  : {len(X_val):,} rows, 양성비율: {y_val.mean()*100:.2f}%")
del df_val; gc.collect()

# [3] Test (다운샘플링 없음)
df_test = (
    df_v2.filter(pl.col("time_group") >= val_cutoff)
    .join(df_tgat, on="account_id", how="left")
    .fill_null(0.0)
)
df_test_meta = df_test.select(["account_id", "time_group", "is_laundering"])
X_test = df_test.select(xgb_feature_cols).to_pandas()
y_test = df_test["is_laundering"].to_numpy()
print(f"📊 Test : {len(X_test):,} rows, 양성비율: {y_test.mean()*100:.2f}%")
del df_test, df_v2, df_tgat; gc.collect()
log_memory("XGBoost 데이터 준비 완료")

# =============================================================
# 7. XGBoost 학습
# =============================================================
print("\n🔥 XGBoost 학습 시작...")

actual_pos_ratio = y_train.sum() / len(y_train) if len(y_train) > 0 else 0.01
adjusted_spw = max((1 - actual_pos_ratio) / actual_pos_ratio, 1.0)
print(f"📊 scale_pos_weight: {adjusted_spw:.1f}")

xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "max_depth": 8,
    "learning_rate": 0.03,
    "scale_pos_weight": adjusted_spw,
    "subsample": 0.8,
    "colsample_bytree": 0.7,
    "min_child_weight": 5,
    "gamma": 0.1,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "device": "cuda",
    "random_state": 42,
}
model_xgb = xgb.XGBClassifier(**xgb_params, n_estimators=1000, early_stopping_rounds=80)
model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)

y_prob = model_xgb.predict_proba(X_test)[:, 1]

print("\n🔍 XGBoost Feature Importance Top 15")
importance = model_xgb.feature_importances_
top_indices = np.argsort(importance)[::-1][:15]
for i, idx in enumerate(top_indices):
    print(f"  {i+1:2d}. {xgb_feature_cols[idx]:<30s} : {importance[idx]:.4f}")

# TGAT 임베딩이 전체 importance에서 차지하는 비중
tgat_importance = sum(importance[i] for i, c in enumerate(xgb_feature_cols) if c.startswith("tgat_emb_"))
total_importance = importance.sum()
print(f"\n📊 TGAT 임베딩 총 importance: {tgat_importance:.4f} / {total_importance:.4f} "
      f"({tgat_importance/total_importance*100:.1f}%)")

del X_train, X_val, X_test; gc.collect()

# =============================================================
# 8. 최종 평가 리포트
# =============================================================
print("\n" + "=" * 60)
print("🏆 [TGAT v2 + XGBoost] 최종 성능 리포트")
print("=" * 60)

# 다중 임계값 평가
print(f"\n📌 [다중 임계값 분석 (Test Set: {val_cutoff} 이후)]")
auprc = average_precision_score(y_test, y_prob)
print(f"  AUPRC: {auprc:.4f}")

best_f1 = 0
best_thresh = 0
for thresh in np.arange(0.05, 0.95, 0.05):
    y_pred_t = (y_prob >= thresh).astype(int)
    p = precision_score(y_test, y_pred_t, zero_division=0)
    r = recall_score(y_test, y_pred_t, zero_division=0)
    f = f1_score(y_test, y_pred_t, zero_division=0)
    if f > best_f1:
        best_f1 = f
        best_thresh = thresh
    if thresh in [0.1, 0.2, 0.3, 0.5, 0.7]:
        print(f"  Thresh={thresh:.2f} | P={p:.4f} R={r:.4f} F1={f:.4f}")

y_pred = (y_prob >= best_thresh).astype(int)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print(f"\n📌 [최적 임계값 결과]")
print(f"  - Best Threshold: {best_thresh:.2f}")
print(f"  - AUPRC    : {auprc:.4f}")
print(f"  - F1-Score : {best_f1:.4f}")
print(f"  - Precision: {precision:.4f}")
print(f"  - Recall   : {recall:.4f}")

# 기존 방식 (dynamic threshold) 결과도 참고용으로 출력
max_prob = np.max(y_prob)
dyn_thresh = max_prob * 0.8 if max_prob > 0 else 0.5
y_pred_dyn = (y_prob >= dyn_thresh).astype(int)
print(f"\n  (참고) Dynamic Threshold={dyn_thresh:.4f} → "
      f"F1={f1_score(y_test, y_pred_dyn, zero_division=0):.4f}, "
      f"P={precision_score(y_test, y_pred_dyn, zero_division=0):.4f}, "
      f"R={recall_score(y_test, y_pred_dyn, zero_division=0):.4f}")

# Top-K 탐지
df_res = df_test_meta.with_columns([
    pl.col("time_group").dt.date().alias("date"),
    pl.Series("pred_prob", y_prob)
])
df_distinct = df_res.sort("pred_prob", descending=True).unique(subset=["account_id"], maintain_order=True)

print(f"\n📌 [전체 Test 데이터 Top-K 탐지 (Distinct Account 기준)]")
for k in [100, 500, 1000, 5000]:
    current_k = min(k, len(df_distinct))
    if current_k > 0:
        hits = df_distinct.head(current_k)["is_laundering"].sum()
        print(f"  📍 Top-{k:4d} Hits: {hits:4d}명 | Precision: {(hits/current_k)*100:6.2f}%")

print(f"\n📌 [일별 Top-100 탐지 현황]")
unique_dates = df_distinct["date"].unique(maintain_order=True).sort(descending=True)
daily_stats = []
for d in unique_dates:
    hits = df_distinct.filter(pl.col("date") == d).head(100)["is_laundering"].sum()
    daily_stats.append({"Date": str(d), "Top-100 Hits": hits})
print(pl.DataFrame(daily_stats))

log_memory("최종 완료")
print("\n✅ TGAT v2 파이프라인 완료!")

🛡️ [TGAT v2] TransformerConv + 그래프 보존 + 튜닝 파이프라인
📂 Raw 데이터 및 V2 데이터 로딩 중...
⏱️ Train (60%) 컷오프 시간: 2022-09-09 20:00:00
⏱️ Val   (80%) 컷오프 시간: 2022-09-14 05:00:00
📊 전체 노드 수: 2076999
  💾 [데이터 로드 완료] RAM: 34.00GB | GPU: 0.00GB

✂️ 2022-09-09 20:00:00 이전 데이터로 그래프 구축 (엣지 100% 보존)...
📊 Train 엣지 수: 19,060,343 (100% 보존)
📊 Edge feature 차원: 5 (time_norm, sin/cos_hour, sin/cos_dow)
  💾 [엣지 구성 완료] RAM: 36.56GB | GPU: 0.00GB
📊 양성 노드: 20,400 / 2,076,999 (0.98%)
📊 GNN 학습 노드: 양성 20,400 + 음성 1,020,000 = 1,040,400 (전체 활성 2,065,094 중, 비율 50.4%)
  💾 [노드 피처+마스크 완료] RAM: 37.19GB | GPU: 0.00GB

🧠 TransformerConv 기반 TGAT 모델 정의...

🔥 TGAT v2 학습 시작...
📌 디바이스: cuda


/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "


📊 BCEWithLogitsLoss pos_weight: 50.0
  💾 [학습 시작 전] RAM: 37.91GB | GPU: 0.00GB


  → Epoch  1 | Loss: 1.2263 | LR: 0.002992
  💾 [Epoch 1] RAM: 38.39GB | GPU: 0.02GB


  → Epoch  5 | Loss: 1.4035 | LR: 0.002800
  💾 [Epoch 5] RAM: 38.40GB | GPU: 0.02GB


  → Epoch 10 | Loss: 9.7110 | LR: 0.002252
  💾 [Epoch 10] RAM: 38.41GB | GPU: 0.02GB
  ⏹️ Early stopping at epoch 10 (patience 7)
  ✅ Best model 복원 (loss: 1.1493)

📦 TGAT v2 임베딩 추출 중...


Embedding Extraction: 100%|█████████████████████████████████████████████████| 1015/1015 [00:13<00:00, 75.60it/s]


✅ 임베딩 shape: (2076999, 64)
  💾 [임베딩 추출 완료] RAM: 39.38GB | GPU: 0.02GB

🚀 XGBoost 결합 및 6:2:2 시간 기준 분할...
📊 Train: 17,172,747 → 619,437 (양성 29,497 + 음성 589,940), 양성비율: 4.76%
📊 Val  : 5,790,644 rows, 양성비율: 0.30%
📊 Test : 5,792,913 rows, 양성비율: 0.36%
  💾 [XGBoost 데이터 준비 완료] RAM: 54.86GB | GPU: 0.02GB

🔥 XGBoost 학습 시작...
📊 scale_pos_weight: 20.0
[0]	validation_0-aucpr:0.10156
[50]	validation_0-aucpr:0.24281
[100]	validation_0-aucpr:0.31213
[150]	validation_0-aucpr:0.35549
[200]	validation_0-aucpr:0.38187
[250]	validation_0-aucpr:0.39435
[300]	validation_0-aucpr:0.39988
[350]	validation_0-aucpr:0.40357
[400]	validation_0-aucpr:0.40698
[450]	validation_0-aucpr:0.40995
[500]	validation_0-aucpr:0.41166
[550]	validation_0-aucpr:0.41483
[600]	validation_0-aucpr:0.41695
[650]	validation_0-aucpr:0.41814
[700]	validation_0-aucpr:0.41953
[750]	validation_0-aucpr:0.42252
[800]	validation_0-aucpr:0.42344
[850]	validation_0-aucpr:0.42444
[900]	validation_0-aucpr:0.42602
[950]	validation_0-aucpr:0.42805
[

/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [09:29:17] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



🔍 XGBoost Feature Importance Top 15
   1. cnt_risk_format                : 0.1559
   2. ratio_cross_border             : 0.1225
   3. cnt_inter_bank                 : 0.0891
   4. amount_kurtosis                : 0.0845
   5. degree_1h                      : 0.0706
   6. cnt_wire                       : 0.0466
   7. cnt_1h                         : 0.0367
   8. entity_acct_cnt                : 0.0297
   9. tgat_emb_48                    : 0.0227
  10. cnt_currencies                 : 0.0222
  11. cnt_intra_bank                 : 0.0205
  12. avg_log_amount                 : 0.0157
  13. sum_1h                         : 0.0155
  14. burst_ratio                    : 0.0128
  15. cnt_night                      : 0.0119

📊 TGAT 임베딩 총 importance: 0.1763 / 1.0000 (17.6%)

🏆 [TGAT v2 + XGBoost] 최종 성능 리포트

📌 [다중 임계값 분석 (Test Set: 2022-09-14 05:00:00 이후)]
  AUPRC: 0.5086
  Thresh=0.10 | P=0.0276 R=0.9789 F1=0.0536
  Thresh=0.20 | P=0.0374 R=0.9583 F1=0.0721
  Thresh=0.30 | P=0.0479 R=0.9368 F1

In [1]:
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv
from torch_geometric.loader import NeighborLoader
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score
from tqdm import tqdm
import gc
import numpy as np
import psutil
import os

def log_memory(tag=""):
    proc = psutil.Process(os.getpid())
    rss_gb = proc.memory_info().rss / (1024 ** 3)
    gpu_str = ""
    if torch.cuda.is_available():
        gpu_alloc = torch.cuda.memory_allocated() / (1024 ** 3)
        gpu_str = f" | GPU: {gpu_alloc:.2f}GB"
    print(f"  💾 [{tag}] RAM: {rss_gb:.2f}GB{gpu_str}")

print("=" * 60)
print("🛡️ [TGAT v2.1] TransformerConv + v3 안정화 + LR Warmup + Cosine")
print("   ▸ v3 적용: input_proj + pre-norm + residual + GELU + dropout 0.2")
print("   ▸ LR: warmup 5ep (1e-6→1e-3) + cosine decay, patience 10")
print("=" * 60)

# ===================== 설정 =====================
NEG_NODE_RATIO_GNN = 50
NEG_ROW_RATIO_XGB = 20
RANDOM_SEED = 42

# GNN 학습 하이퍼파라미터
HIDDEN_DIM = 64
EDGE_DIM = 16
N_HEADS = 4
N_EPOCHS = 40                 # ★ 30 → 40 (v4와 동일)
BATCH_SIZE = 2048
NUM_NEIGHBORS = [15, 10]

# ★ 변경: LR 스케줄링 파라미터
BASE_LR = 0.001               # ★ v2의 0.003 → 0.001 (v4와 동일)
WARMUP_EPOCHS = 5              # ★ 추가: 5 에폭 warmup
PATIENCE = 10                  # ★ 7 → 10 (cosine 스케줄에 맞게 여유)
# ================================================

raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"

# =============================================================
# 1. 데이터 로드 및 6:2:2 시간 분할
# =============================================================
print("📂 Raw 데이터 및 V2 데이터 로딩 중...")
df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))

if df_raw["Timestamp"].dtype == pl.Utf8:
    df_raw = df_raw.with_columns(
        pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False)
    )

df_v2 = df_v2.sort("time_group")
total_count = len(df_v2)
train_idx = int(total_count * 0.6)
val_idx = int(total_count * 0.8)

train_cutoff = df_v2["time_group"][train_idx]
val_cutoff = df_v2["time_group"][val_idx]

print(f"⏱️ Train (60%) 컷오프 시간: {train_cutoff}")
print(f"⏱️ Val   (80%) 컷오프 시간: {val_cutoff}")

all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")

num_nodes = len(all_accounts)
print(f"📊 전체 노드 수: {num_nodes}")
log_memory("데이터 로드 완료")

# =============================================================
# 2. 미래 정보 차단 — Train 기간 데이터만으로 그래프 구축
# =============================================================
print(f"\n✂️ {train_cutoff} 이전 데이터로 그래프 구축 (엣지 100% 보존)...")

df_raw_train = df_raw.filter(pl.col("Timestamp") < train_cutoff)
del df_raw; gc.collect()

df_edges_train = df_raw_train.select(["from_acc", "to_acc", "Timestamp"]).with_columns([
    pl.col("from_acc").cast(pl.Utf8),
    pl.col("to_acc").cast(pl.Utf8)
])
df_edges_train = df_edges_train.join(
    all_accounts.rename({"account_id": "from_acc", "node_id": "src_id"}), on="from_acc", how="left"
)
df_edges_train = df_edges_train.join(
    all_accounts.rename({"account_id": "to_acc", "node_id": "dst_id"}), on="to_acc", how="left"
)

timestamps_raw = df_edges_train["Timestamp"]
min_ts = timestamps_raw.min()
max_ts = timestamps_raw.max()
total_seconds = (max_ts - min_ts).total_seconds()

edge_time_normalized = (
    (timestamps_raw - min_ts).dt.total_microseconds().cast(pl.Float64) / 1e6 / max(total_seconds, 1.0)
).to_numpy().astype(np.float32)

edge_index_train = torch.tensor(
    df_edges_train.select(["src_id", "dst_id"]).to_numpy().T, dtype=torch.long
)

hours = df_edges_train["Timestamp"].dt.hour().to_numpy().astype(np.float32)
dow = df_edges_train["Timestamp"].dt.weekday().to_numpy().astype(np.float32)

edge_features = np.stack([
    edge_time_normalized,
    np.sin(2 * np.pi * hours / 24),
    np.cos(2 * np.pi * hours / 24),
    np.sin(2 * np.pi * dow / 7),
    np.cos(2 * np.pi * dow / 7),
], axis=1)

edge_attr_train = torch.tensor(edge_features, dtype=torch.float32)
EDGE_RAW_DIM = edge_attr_train.shape[1]

n_edges = edge_index_train.shape[1]
print(f"📊 Train 엣지 수: {n_edges:,} (100% 보존)")
print(f"📊 Edge feature 차원: {EDGE_RAW_DIM}")

del df_raw_train, df_edges_train, timestamps_raw, hours, dow, edge_features, edge_time_normalized
gc.collect()
log_memory("엣지 구성 완료")

# 노드 피처 (Train 기간만)
df_v2_train = df_v2.filter(pl.col("time_group") < train_cutoff)

exclude_cols = ["account_id", "time_group", "is_laundering", "mode_format", "currency_mode"]
gnn_feature_cols = [c for c in df_v2.columns if c not in exclude_cols]

df_v2_node = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols
])
df_node_features = all_accounts.join(df_v2_node, on="account_id", how="left").fill_null(0.0)
X_node = torch.tensor(df_node_features.select(gnn_feature_cols).to_numpy(), dtype=torch.float32)
IN_DIM = X_node.shape[1]

# 레이블
target_labels = (
    df_v2_train.filter(pl.col("is_laundering") == 1)
    .select("account_id").unique()
    .with_columns(pl.lit(1).alias("label"))
)
df_labels = all_accounts.join(target_labels, on="account_id", how="left").fill_null(0)
Y_node = torch.tensor(df_labels["label"].to_numpy(), dtype=torch.long)

n_pos_total = int(Y_node.sum().item())
print(f"📊 양성 노드: {n_pos_total:,} / {num_nodes:,} ({n_pos_total/num_nodes*100:.2f}%)")

# Train 마스크
active_accounts = df_v2_train["account_id"].unique()
active_df = pl.DataFrame({"account_id": active_accounts, "is_active": True})
mask_df = all_accounts.join(active_df, on="account_id", how="left").fill_null(False)

pos_node_ids = set(df_labels.filter(pl.col("label") == 1)["node_id"].to_list())
active_node_ids = mask_df.filter(pl.col("is_active"))["node_id"].to_list()

active_pos = [nid for nid in active_node_ids if nid in pos_node_ids]
active_neg = [nid for nid in active_node_ids if nid not in pos_node_ids]

n_neg_keep_gnn = min(len(active_pos) * NEG_NODE_RATIO_GNN, len(active_neg))
np.random.seed(RANDOM_SEED)
sampled_neg = np.random.choice(active_neg, size=n_neg_keep_gnn, replace=False).tolist()
sampled_nodes = set(active_pos + sampled_neg)

train_mask_np = np.zeros(num_nodes, dtype=bool)
for nid in sampled_nodes:
    train_mask_np[nid] = True
train_mask = torch.tensor(train_mask_np, dtype=torch.bool)

print(f"📊 GNN 학습 노드: 양성 {len(active_pos):,} + 음성 {n_neg_keep_gnn:,} = {len(sampled_nodes):,}")

del df_v2_node, df_node_features, mask_df, active_df
gc.collect()
log_memory("노드 피처+마스크 완료")

# =============================================================
# 3. 모델 정의: TransformerConv 기반 TGAT
# =============================================================
print("\n🧠 TransformerConv 기반 TGAT 모델 정의...")


class EdgeProjection(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.GELU(),                    # ★ ReLU → GELU
            nn.Linear(out_dim, out_dim)
        )
    def forward(self, edge_attr):
        return self.proj(edge_attr)


class TGAT_v2(nn.Module):
    """v3 안정화 적용: input_proj + pre-norm + residual + GELU + dropout 0.2"""
    def __init__(self, in_dim, hidden_dim, edge_raw_dim, edge_dim, n_heads, dropout=0.2):
        super().__init__()
        self.edge_proj = EdgeProjection(edge_raw_dim, edge_dim)
        self.input_proj = nn.Linear(in_dim, hidden_dim)   # ★ 추가: 입력 투영
        self.norm1 = nn.LayerNorm(hidden_dim)              # ★ pre-norm 위치 변경
        self.conv1 = TransformerConv(
            in_channels=hidden_dim, out_channels=hidden_dim // n_heads,
            heads=n_heads, edge_dim=edge_dim, dropout=dropout, concat=True)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.conv2 = TransformerConv(
            in_channels=hidden_dim, out_channels=hidden_dim // n_heads,
            heads=n_heads, edge_dim=edge_dim, dropout=dropout, concat=True)
        self.norm_out = nn.LayerNorm(hidden_dim)           # ★ 추가: 출력 norm
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),                                     # ★ ReLU → GELU
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )
        self.dropout = dropout

    def forward(self, x, edge_index, edge_attr):
        edge_emb = self.edge_proj(edge_attr)
        h = self.input_proj(x)                             # ★ 입력 투영
        # ★ Pre-norm + Residual connection
        h = h + F.dropout(self.conv1(self.norm1(h), edge_index, edge_emb),
                          p=self.dropout, training=self.training)
        h = h + F.dropout(self.conv2(self.norm2(h), edge_index, edge_emb),
                          p=self.dropout, training=self.training)
        h = self.norm_out(h)
        return self.classifier(h).squeeze(-1)

    @torch.no_grad()
    def get_embedding(self, x, edge_index, edge_attr):
        edge_emb = self.edge_proj(edge_attr)
        h = self.input_proj(x)
        h = h + self.conv1(self.norm1(h), edge_index, edge_emb)
        h = h + self.conv2(self.norm2(h), edge_index, edge_emb)
        h = self.norm_out(h)
        return h


# =============================================================
# 4. 그래프 데이터 + 학습 (★ LR Warmup + Cosine Annealing)
# =============================================================
print("\n🔥 TGAT v2.1 학습 시작...")
print(f"   ▸ LR: warmup {WARMUP_EPOCHS}ep (1e-6→{BASE_LR}) + cosine decay")
print(f"   ▸ Epochs: {N_EPOCHS}, Patience: {PATIENCE}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"📌 디바이스: {device}")

graph_data = Data(
    x=X_node, edge_index=edge_index_train, edge_attr=edge_attr_train, y=Y_node
)
graph_data.train_mask = train_mask
graph_data.num_nodes = num_nodes

del X_node, edge_index_train, edge_attr_train, Y_node, train_mask
gc.collect()

train_loader = NeighborLoader(
    graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
    input_nodes=graph_data.train_mask, shuffle=True, num_workers=4)
full_loader = NeighborLoader(
    graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
    input_nodes=None, shuffle=False, num_workers=4)

model = TGAT_v2(
    in_dim=IN_DIM, hidden_dim=HIDDEN_DIM, edge_raw_dim=EDGE_RAW_DIM,
    edge_dim=EDGE_DIM, n_heads=N_HEADS, dropout=0.2
).to(device)

n_train_pos = len(active_pos)
n_train_total = len(sampled_nodes)
pw = (n_train_total - n_train_pos) / max(n_train_pos, 1)
print(f"📊 BCEWithLogitsLoss pos_weight: {pw:.1f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw], device=device))

# ★ 변경: optimizer 초기 LR을 1e-6으로 (warmup 시작점)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6, weight_decay=1e-4)

# ★ 변경: 수동 LR 스케줄링 (warmup + cosine annealing)
def set_lr(optimizer, epoch):
    """Warmup (linear) + Cosine Annealing 스케줄러"""
    if epoch < WARMUP_EPOCHS:
        # Linear warmup: 1e-6 → BASE_LR
        lr = 1e-6 + (BASE_LR - 1e-6) * (epoch / WARMUP_EPOCHS)
    else:
        # Cosine annealing: BASE_LR → 1e-5
        progress = (epoch - WARMUP_EPOCHS) / max(N_EPOCHS - WARMUP_EPOCHS, 1)
        lr = 1e-5 + (BASE_LR - 1e-5) * 0.5 * (1 + np.cos(np.pi * progress))
    for pg in optimizer.param_groups:
        pg['lr'] = lr
    return lr

log_memory("학습 시작 전")

model.train()
best_loss = float('inf')
patience_counter = 0
best_state = None

for epoch in range(N_EPOCHS):
    # ★ LR 설정 (매 에폭 시작 시)
    current_lr = set_lr(optimizer, epoch)

    total_loss = 0.0
    num_batches = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{N_EPOCHS}", leave=False):
        batch = batch.to(device)
        optimizer.zero_grad()

        if batch.edge_attr is not None and batch.edge_attr.numel() > 0:
            batch_edge_attr = batch.edge_attr
        else:
            batch_edge_attr = torch.zeros(batch.edge_index.size(1), EDGE_RAW_DIM, device=device)

        out = model(batch.x, batch.edge_index, batch_edge_attr)
        loss = criterion(out[:batch.batch_size], batch.y[:batch.batch_size].float())

        if torch.isnan(loss) or torch.isinf(loss):
            optimizer.zero_grad()
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)  # ★ 1.0 → 0.5
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    avg_loss = total_loss / max(num_batches, 1)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  → Epoch {epoch+1:2d} | Loss: {avg_loss:.4f} | LR: {current_lr:.6f}")

    # ★ Early stopping 기준: 0.001 → 0.0005 (더 세밀하게)
    if avg_loss < best_loss - 0.0005:
        best_loss = avg_loss
        patience_counter = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print(f"  ⏹️ Early stopping at epoch {epoch+1} (patience {PATIENCE})")
        break

if best_state is not None:
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    print(f"  ✅ Best model 복원 (loss: {best_loss:.4f})")
    del best_state

# =============================================================
# 5. 임베딩 추출
# =============================================================
print("\n📦 TGAT v2.1 임베딩 추출 중...")
model.eval()
all_emb = []
with torch.no_grad():
    for batch in tqdm(full_loader, desc="Embedding Extraction"):
        batch = batch.to(device)
        if batch.edge_attr is not None and batch.edge_attr.numel() > 0:
            batch_edge_attr = batch.edge_attr
        else:
            batch_edge_attr = torch.zeros(batch.edge_index.size(1), EDGE_RAW_DIM, device=device)

        emb = model.get_embedding(batch.x, batch.edge_index, batch_edge_attr)
        all_emb.append(emb[:batch.batch_size].cpu())

final_emb = torch.cat(all_emb, dim=0).numpy()
emb_cols = [f"tgat_emb_{i}" for i in range(HIDDEN_DIM)]

df_tgat = pl.concat([
    all_accounts.select("account_id"),
    pl.DataFrame(final_emb, schema=emb_cols)
], how="horizontal")

print(f"✅ 임베딩 shape: {final_emb.shape}")

del graph_data, all_emb, final_emb, all_accounts
torch.cuda.empty_cache()
gc.collect()
log_memory("임베딩 추출 완료")

# =============================================================
# 6. XGBoost 결합 및 6:2:2 분할
# =============================================================
print("\n🚀 XGBoost 결합 및 6:2:2 시간 기준 분할...")

xgb_exclude_cols = ["account_id", "time_group", "is_laundering", "mode_format", "currency_mode", "date"]
xgb_feature_cols = [c for c in df_v2.columns if c not in xgb_exclude_cols] + emb_cols

# [1] Train
df_train_full = (
    df_v2.filter(pl.col("time_group") < train_cutoff)
    .join(df_tgat, on="account_id", how="left")
    .fill_null(0.0)
)
df_train_pos = df_train_full.filter(pl.col("is_laundering") == 1)
df_train_neg = df_train_full.filter(pl.col("is_laundering") == 0)
n_neg_keep_xgb = min(len(df_train_pos) * NEG_ROW_RATIO_XGB, len(df_train_neg))
df_train_neg_sampled = df_train_neg.sample(n=n_neg_keep_xgb, seed=RANDOM_SEED)
df_train = pl.concat([df_train_pos, df_train_neg_sampled]).sample(fraction=1.0, seed=RANDOM_SEED)

X_train = df_train.select(xgb_feature_cols).to_pandas()
y_train = df_train["is_laundering"].to_numpy()
print(f"📊 Train: {len(df_train_full):,} → {len(df_train):,} (양성비율: {y_train.mean()*100:.2f}%)")
del df_train, df_train_full, df_train_pos, df_train_neg, df_train_neg_sampled; gc.collect()

# [2] Val
df_val = (
    df_v2.filter((pl.col("time_group") >= train_cutoff) & (pl.col("time_group") < val_cutoff))
    .join(df_tgat, on="account_id", how="left")
    .fill_null(0.0)
)
X_val = df_val.select(xgb_feature_cols).to_pandas()
y_val = df_val["is_laundering"].to_numpy()
print(f"📊 Val  : {len(X_val):,} rows, 양성비율: {y_val.mean()*100:.2f}%")
del df_val; gc.collect()

# [3] Test
df_test = (
    df_v2.filter(pl.col("time_group") >= val_cutoff)
    .join(df_tgat, on="account_id", how="left")
    .fill_null(0.0)
)
df_test_meta = df_test.select(["account_id", "time_group", "is_laundering"])
X_test = df_test.select(xgb_feature_cols).to_pandas()
y_test = df_test["is_laundering"].to_numpy()
print(f"📊 Test : {len(X_test):,} rows, 양성비율: {y_test.mean()*100:.2f}%")
del df_test, df_v2, df_tgat; gc.collect()
log_memory("XGBoost 데이터 준비 완료")

# =============================================================
# 7. XGBoost 학습
# =============================================================
print("\n🔥 XGBoost 학습 시작...")

actual_pos_ratio = y_train.sum() / len(y_train) if len(y_train) > 0 else 0.01
adjusted_spw = max((1 - actual_pos_ratio) / actual_pos_ratio, 1.0)
print(f"📊 scale_pos_weight: {adjusted_spw:.1f}")

xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "max_depth": 8,
    "learning_rate": 0.03,
    "scale_pos_weight": adjusted_spw,
    "subsample": 0.8,
    "colsample_bytree": 0.7,
    "min_child_weight": 5,
    "gamma": 0.1,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "device": "cuda",
    "random_state": 42,
}
model_xgb = xgb.XGBClassifier(**xgb_params, n_estimators=1500, early_stopping_rounds=100)
model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)

y_prob = model_xgb.predict_proba(X_test)[:, 1]

print("\n🔍 XGBoost Feature Importance Top 15")
importance = model_xgb.feature_importances_
top_indices = np.argsort(importance)[::-1][:15]
for i, idx in enumerate(top_indices):
    print(f"  {i+1:2d}. {xgb_feature_cols[idx]:<30s} : {importance[idx]:.4f}")

tgat_importance = sum(importance[i] for i, c in enumerate(xgb_feature_cols) if c.startswith("tgat_emb_"))
total_importance = importance.sum()
print(f"\n📊 TGAT 임베딩 총 importance: {tgat_importance:.4f} / {total_importance:.4f} "
      f"({tgat_importance/total_importance*100:.1f}%)")

del X_train, X_val, X_test; gc.collect()

# =============================================================
# 8. 최종 평가 리포트
# =============================================================
print("\n" + "=" * 60)
print("🏆 [TGAT v2.1 + XGBoost] 최종 성능 리포트")
print("=" * 60)

auprc = average_precision_score(y_test, y_prob)
print(f"\n  AUPRC: {auprc:.4f}")

best_f1 = 0
best_thresh = 0
for thresh in np.arange(0.05, 0.95, 0.01):
    y_pred_t = (y_prob >= thresh).astype(int)
    f = f1_score(y_test, y_pred_t, zero_division=0)
    if f > best_f1:
        best_f1 = f
        best_thresh = thresh

y_pred = (y_prob >= best_thresh).astype(int)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

from sklearn.metrics import confusion_matrix
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print(f"\n📌 [최적 임계값 결과]")
print(f"  - Best Threshold: {best_thresh:.2f}")
print(f"  - AUPRC    : {auprc:.4f}")
print(f"  - F1-Score : {best_f1:.4f}")
print(f"  - Precision: {precision:.4f}")
print(f"  - Recall   : {recall:.4f}")
print(f"\n📌 [TP/FP/FN/TN]")
print(f"  - TP (정탐): {tp:,}")
print(f"  - FP (오탐): {fp:,}")
print(f"  - FN (미탐): {fn:,}")
print(f"  - TN (정상): {tn:,}")

print(f"\n📌 [주요 임계값별 성능]")
for t in [0.1, 0.2, 0.3, 0.5, 0.7, 0.8, 0.9]:
    yp = (y_prob >= t).astype(int)
    print(f"  T={t:.1f} | P={precision_score(y_test,yp,zero_division=0):.4f} "
          f"R={recall_score(y_test,yp,zero_division=0):.4f} "
          f"F1={f1_score(y_test,yp,zero_division=0):.4f}")

# Top-K
df_res = df_test_meta.with_columns([
    pl.col("time_group").dt.date().alias("date"),
    pl.Series("pred_prob", y_prob)
])
df_distinct = df_res.sort("pred_prob", descending=True).unique(subset=["account_id"], maintain_order=True)

print(f"\n📌 [Top-K 탐지 (Distinct Account)]")
for k in [100, 500, 1000, 5000, 10000]:
    current_k = min(k, len(df_distinct))
    if current_k > 0:
        hits = df_distinct.head(current_k)["is_laundering"].sum()
        print(f"  📍 Top-{k:5d} Hits: {hits:5d}명 | Precision: {(hits/current_k)*100:6.2f}%")

print(f"\n📌 [일별 Top-100 탐지]")
unique_dates = df_distinct["date"].unique(maintain_order=True).sort(descending=True)
daily_stats = []
for d in unique_dates:
    hits = df_distinct.filter(pl.col("date") == d).head(100)["is_laundering"].sum()
    daily_stats.append({"Date": str(d), "Top-100 Hits": hits})
print(pl.DataFrame(daily_stats))

log_memory("최종 완료")
print("\n✅ TGAT v2.1 파이프라인 완료!")

🛡️ [TGAT v2.1] TransformerConv + v3 안정화 + LR Warmup + Cosine
   ▸ v3 적용: input_proj + pre-norm + residual + GELU + dropout 0.2
   ▸ LR: warmup 5ep (1e-6→1e-3) + cosine decay, patience 10
📂 Raw 데이터 및 V2 데이터 로딩 중...
⏱️ Train (60%) 컷오프 시간: 2022-09-09 20:00:00
⏱️ Val   (80%) 컷오프 시간: 2022-09-14 05:00:00
📊 전체 노드 수: 2076999
  💾 [데이터 로드 완료] RAM: 33.17GB | GPU: 0.00GB

✂️ 2022-09-09 20:00:00 이전 데이터로 그래프 구축 (엣지 100% 보존)...
📊 Train 엣지 수: 19,060,343 (100% 보존)
📊 Edge feature 차원: 5
  💾 [엣지 구성 완료] RAM: 35.75GB | GPU: 0.00GB
📊 양성 노드: 20,400 / 2,076,999 (0.98%)
📊 GNN 학습 노드: 양성 20,400 + 음성 1,020,000 = 1,040,400
  💾 [노드 피처+마스크 완료] RAM: 36.34GB | GPU: 0.00GB

🧠 TransformerConv 기반 TGAT 모델 정의...

🔥 TGAT v2.1 학습 시작...
   ▸ LR: warmup 5ep (1e-6→0.001) + cosine decay
   ▸ Epochs: 40, Patience: 10
📌 디바이스: cuda


/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "


📊 BCEWithLogitsLoss pos_weight: 50.0
  💾 [학습 시작 전] RAM: 37.06GB | GPU: 0.00GB


  → Epoch  1 | Loss: 1.3625 | LR: 0.000001


  → Epoch  5 | Loss: 1.1609 | LR: 0.000800


...

  → Epoch 40 | Loss: 1.0313 | LR: 0.000012
  ✅ Best model 복원 (loss: 1.0308)

📦 TGAT v2.1 임베딩 추출 중...


Embedding Extraction: 100%|█████████████████████████████████████████████████| 1015/1015 [00:16<00:00, 62.27it/s]


✅ 임베딩 shape: (2076999, 64)
  💾 [임베딩 추출 완료] RAM: 38.58GB | GPU: 0.02GB

🚀 XGBoost 결합 및 6:2:2 시간 기준 분할...
📊 Train: 17,172,747 → 619,437 (양성비율: 4.76%)
📊 Val  : 5,790,644 rows, 양성비율: 0.30%
📊 Test : 5,792,913 rows, 양성비율: 0.36%
  💾 [XGBoost 데이터 준비 완료] RAM: 55.16GB | GPU: 0.02GB

🔥 XGBoost 학습 시작...
📊 scale_pos_weight: 20.0
[0]	validation_0-aucpr:0.10065
[100]	validation_0-aucpr:0.34802
[200]	validation_0-aucpr:0.41368
[300]	validation_0-aucpr:0.42135
[400]	validation_0-aucpr:0.42612
[500]	validation_0-aucpr:0.43020
[600]	validation_0-aucpr:0.43504
[700]	validation_0-aucpr:0.43770
[800]	validation_0-aucpr:0.44067
[900]	validation_0-aucpr:0.44546
[1000]	validation_0-aucpr:0.44881
[1100]	validation_0-aucpr:0.44902
[1200]	validation_0-aucpr:0.45135
[1300]	validation_0-aucpr:0.45419
[1400]	validation_0-aucpr:0.45525
[1499]	validation_0-aucpr:0.45717


/home/tracerofjageum/my-lab/.venv/lib/python3.12/site-packages/xgboost/core.py:751: UserWarning: [02:08:04] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



🔍 XGBoost Feature Importance Top 15
   1. ratio_cross_border             : 0.1517
   2. cnt_risk_format                : 0.1158
   3. cnt_inter_bank                 : 0.0956
   4. degree_1h                      : 0.0823
   5. amount_kurtosis                : 0.0717
   6. cnt_wire                       : 0.0473
   7. cnt_currencies                 : 0.0330
   8. cnt_1h                         : 0.0326
   9. cnt_intra_bank                 : 0.0303
  10. entity_acct_cnt                : 0.0269
  11. burst_ratio                    : 0.0189
  12. sum_1h                         : 0.0153
  13. avg_log_amount                 : 0.0142
  14. cnt_night                      : 0.0113
  15. max_1h                         : 0.0108

📊 TGAT 임베딩 총 importance: 0.1596 / 1.0000 (16.0%)

🏆 [TGAT v2.1 + XGBoost] 최종 성능 리포트

  AUPRC: 0.5408

📌 [최적 임계값 결과]
  - Best Threshold: 0.94
  - AUPRC    : 0.5408
  - F1-Score : 0.4249
  - Precision: 0.3152
  - Recall   : 0.6519

📌 [TP/FP/FN/TN]
  - TP (정탐): 13,468
  - FP

In [1]:
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv
from torch_geometric.loader import NeighborLoader
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score
from tqdm import tqdm
import gc
import numpy as np
import psutil
import os
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

def log_memory(tag=""):
    proc = psutil.Process(os.getpid())
    rss_gb = proc.memory_info().rss / (1024 ** 3)
    gpu_str = ""
    if torch.cuda.is_available():
        gpu_alloc = torch.cuda.memory_allocated() / (1024 ** 3)
        gpu_str = f" | GPU: {gpu_alloc:.2f}GB"
    print(f"  💾 [{tag}] RAM: {rss_gb:.2f}GB{gpu_str}")

print("=" * 60)
print("🛡️ [TGAT v3] GNN 안정화 + 그래프 피처 (Optuna 없음)")
print("=" * 60)

# ===================== 설정 =====================
NEG_NODE_RATIO_GNN = 50
NEG_ROW_RATIO_XGB = 20
RANDOM_SEED = 42
HIDDEN_DIM = 64
EDGE_DIM = 16
N_HEADS = 4
N_EPOCHS_GNN = 40
BATCH_SIZE = 2048
NUM_NEIGHBORS = [15, 10]
# ================================================

raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"

# =============================================================
# 1. 데이터 로드 및 6:2:2 시간 분할
# =============================================================
print("\n📂 데이터 로딩 중...")
df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))

if df_raw["Timestamp"].dtype == pl.Utf8:
    df_raw = df_raw.with_columns(
        pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False)
    )

df_v2 = df_v2.sort("time_group")
total_count = len(df_v2)
train_idx = int(total_count * 0.6)
val_idx = int(total_count * 0.8)

train_cutoff = df_v2["time_group"][train_idx]
val_cutoff = df_v2["time_group"][val_idx]

print(f"⏱️ Train cutoff: {train_cutoff}")
print(f"⏱️ Val   cutoff: {val_cutoff}")

all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")

num_nodes = len(all_accounts)
print(f"📊 전체 노드 수: {num_nodes:,}")
log_memory("데이터 로드 완료")

# =============================================================
# 2. 그래프 구조 피처 계산 (Train 기간만)
# =============================================================
print("\n📐 그래프 구조 피처 계산 중 (Train 기간)...")

df_raw_train = df_raw.filter(pl.col("Timestamp") < train_cutoff)
del df_raw; gc.collect()

# --- Degree 피처 ---
df_from = df_raw_train.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id"))
df_to = df_raw_train.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))

out_degree = df_from.group_by("account_id").len().rename({"len": "graph_out_degree"})
in_degree = df_to.group_by("account_id").len().rename({"len": "graph_in_degree"})

# --- Unique 거래 상대방 수 ---
unique_out = (
    df_raw_train.select([
        pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
        pl.col("to_acc").cast(pl.Utf8).alias("partner")
    ])
    .group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_out_partners"))
)
unique_in = (
    df_raw_train.select([
        pl.col("to_acc").cast(pl.Utf8).alias("account_id"),
        pl.col("from_acc").cast(pl.Utf8).alias("partner")
    ])
    .group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_in_partners"))
)

# --- 양방향 거래 비율 ---
edges_set_from = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("a"),
    pl.col("to_acc").cast(pl.Utf8).alias("b")
]).unique()
edges_set_to = edges_set_from.select([
    pl.col("b").alias("a"),
    pl.col("a").alias("b")
])
bidirectional = edges_set_from.join(edges_set_to, on=["a", "b"], how="inner")

bidir_count = (
    bidirectional.select(pl.col("a").alias("account_id"))
    .group_by("account_id").len().rename({"len": "graph_bidir_count"})
)

del edges_set_from, edges_set_to, bidirectional; gc.collect()

# --- 거래 시간 패턴 피처 ---
time_features = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("Timestamp")
]).group_by("account_id").agg([
    pl.col("Timestamp").sort().diff().dt.total_seconds().mean().alias("graph_avg_tx_interval"),
    pl.col("Timestamp").sort().diff().dt.total_seconds().std().alias("graph_std_tx_interval"),
    pl.col("Timestamp").count().alias("graph_total_tx_count"),
])

# --- PageRank (power iteration) ---
print("  📊 PageRank 계산 중 (10 iterations)...")

account_to_id = dict(zip(
    all_accounts["account_id"].to_list(),
    all_accounts["node_id"].to_list()
))

src_ids = df_raw_train["from_acc"].cast(pl.Utf8).map_elements(
    lambda x: account_to_id.get(x, -1), return_dtype=pl.Int64
).to_numpy()
dst_ids = df_raw_train["to_acc"].cast(pl.Utf8).map_elements(
    lambda x: account_to_id.get(x, -1), return_dtype=pl.Int64
).to_numpy()

valid_mask = (src_ids >= 0) & (dst_ids >= 0)
src_valid = src_ids[valid_mask]
dst_valid = dst_ids[valid_mask]

out_deg = np.zeros(num_nodes, dtype=np.float64)
np.add.at(out_deg, src_valid, 1.0)
out_deg = np.maximum(out_deg, 1.0)

damping = 0.85
pr = np.ones(num_nodes, dtype=np.float64) / num_nodes
for _ in range(10):
    new_pr = np.ones(num_nodes, dtype=np.float64) * (1 - damping) / num_nodes
    contributions = pr[src_valid] / out_deg[src_valid]
    np.add.at(new_pr, dst_valid, damping * contributions)
    pr = new_pr

pr_df = pl.DataFrame({
    "node_id": np.arange(num_nodes, dtype=np.uint32),
    "graph_pagerank": pr.astype(np.float32)
})
pr_df = all_accounts.join(pr_df, on="node_id", how="left").select(["account_id", "graph_pagerank"])

del src_ids, dst_ids, src_valid, dst_valid, out_deg, pr, new_pr
gc.collect()

# --- 모든 그래프 피처 결합 ---
df_graph_feats = (
    all_accounts.select("account_id")
    .join(out_degree, on="account_id", how="left")
    .join(in_degree, on="account_id", how="left")
    .join(unique_out, on="account_id", how="left")
    .join(unique_in, on="account_id", how="left")
    .join(bidir_count, on="account_id", how="left")
    .join(time_features, on="account_id", how="left")
    .join(pr_df, on="account_id", how="left")
    .fill_null(0.0)
)

df_graph_feats = df_graph_feats.with_columns([
    (pl.col("graph_out_degree") + pl.col("graph_in_degree")).alias("graph_total_degree"),
    (pl.col("graph_out_degree") / (pl.col("graph_in_degree") + 1)).alias("graph_out_in_ratio"),
    (pl.col("graph_bidir_count") / (pl.col("graph_out_degree") + 1)).alias("graph_reciprocity"),
    (pl.col("graph_unique_out_partners") / (pl.col("graph_out_degree") + 1)).alias("graph_partner_diversity"),
])

graph_feature_cols = [c for c in df_graph_feats.columns if c.startswith("graph_")]
print(f"✅ 그래프 피처 {len(graph_feature_cols)}개 생성: {graph_feature_cols}")

del out_degree, in_degree, unique_out, unique_in, bidir_count, time_features, pr_df, df_from, df_to
gc.collect()
log_memory("그래프 피처 완료")

# =============================================================
# 3. GNN용 엣지 + 노드 피처 구성
# =============================================================
print("\n🔗 GNN 엣지/노드 구성 중...")

df_edges_train = df_raw_train.select(["from_acc", "to_acc", "Timestamp"]).with_columns([
    pl.col("from_acc").cast(pl.Utf8),
    pl.col("to_acc").cast(pl.Utf8)
])
df_edges_train = df_edges_train.join(
    all_accounts.rename({"account_id": "from_acc", "node_id": "src_id"}), on="from_acc", how="left"
)
df_edges_train = df_edges_train.join(
    all_accounts.rename({"account_id": "to_acc", "node_id": "dst_id"}), on="to_acc", how="left"
)

timestamps_raw = df_edges_train["Timestamp"]
min_ts = timestamps_raw.min()
max_ts = timestamps_raw.max()
total_seconds = (max_ts - min_ts).total_seconds()

edge_time_normalized = (
    (timestamps_raw - min_ts).dt.total_microseconds().cast(pl.Float64) / 1e6 / max(total_seconds, 1.0)
).to_numpy().astype(np.float32)

hours = df_edges_train["Timestamp"].dt.hour().to_numpy().astype(np.float32)
dow = df_edges_train["Timestamp"].dt.weekday().to_numpy().astype(np.float32)

edge_features = np.stack([
    edge_time_normalized,
    np.sin(2 * np.pi * hours / 24),
    np.cos(2 * np.pi * hours / 24),
    np.sin(2 * np.pi * dow / 7),
    np.cos(2 * np.pi * dow / 7),
], axis=1)

edge_index_train = torch.tensor(
    df_edges_train.select(["src_id", "dst_id"]).to_numpy().T, dtype=torch.long
)
edge_attr_train = torch.tensor(edge_features, dtype=torch.float32)
EDGE_RAW_DIM = edge_attr_train.shape[1]

print(f"📊 Train 엣지: {edge_index_train.shape[1]:,}")

del df_raw_train, df_edges_train, timestamps_raw, hours, dow, edge_features, edge_time_normalized
gc.collect()

# 노드 피처
df_v2_train = df_v2.filter(pl.col("time_group") < train_cutoff)
exclude_cols = ["account_id", "time_group", "is_laundering", "mode_format", "currency_mode"]
gnn_feature_cols = [c for c in df_v2.columns if c not in exclude_cols]

df_v2_node = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols
])
df_node_features = all_accounts.join(df_v2_node, on="account_id", how="left").fill_null(0.0)
X_node = torch.tensor(df_node_features.select(gnn_feature_cols).to_numpy(), dtype=torch.float32)
IN_DIM = X_node.shape[1]

# 레이블
target_labels = (
    df_v2_train.filter(pl.col("is_laundering") == 1)
    .select("account_id").unique()
    .with_columns(pl.lit(1).alias("label"))
)
df_labels = all_accounts.join(target_labels, on="account_id", how="left").fill_null(0)
Y_node = torch.tensor(df_labels["label"].to_numpy(), dtype=torch.long)

# Train 마스크
active_accounts = df_v2_train["account_id"].unique()
active_df = pl.DataFrame({"account_id": active_accounts, "is_active": True})
mask_df = all_accounts.join(active_df, on="account_id", how="left").fill_null(False)

pos_node_ids = set(df_labels.filter(pl.col("label") == 1)["node_id"].to_list())
active_node_ids = mask_df.filter(pl.col("is_active"))["node_id"].to_list()
active_pos = [nid for nid in active_node_ids if nid in pos_node_ids]
active_neg = [nid for nid in active_node_ids if nid not in pos_node_ids]

n_neg_keep_gnn = min(len(active_pos) * NEG_NODE_RATIO_GNN, len(active_neg))
np.random.seed(RANDOM_SEED)
sampled_neg = np.random.choice(active_neg, size=n_neg_keep_gnn, replace=False).tolist()
sampled_nodes = set(active_pos + sampled_neg)

train_mask_np = np.zeros(num_nodes, dtype=bool)
for nid in sampled_nodes:
    train_mask_np[nid] = True
train_mask = torch.tensor(train_mask_np, dtype=torch.bool)

print(f"📊 GNN 학습 노드: 양성 {len(active_pos):,} + 음성 {n_neg_keep_gnn:,} = {len(sampled_nodes):,}")

del df_v2_node, df_node_features, mask_df, active_df
gc.collect()
log_memory("GNN 데이터 준비 완료")

# =============================================================
# 4. TGAT v3 모델
# =============================================================
print("\n🧠 TGAT v3 모델 정의 (안정화 버전)...")


class EdgeProjection(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.GELU(),
            nn.Linear(out_dim, out_dim)
        )

    def forward(self, edge_attr):
        return self.proj(edge_attr)


class TGAT_v3(nn.Module):
    def __init__(self, in_dim, hidden_dim, edge_raw_dim, edge_dim, n_heads, dropout=0.2):
        super().__init__()
        self.edge_proj = EdgeProjection(edge_raw_dim, edge_dim)
        self.input_proj = nn.Linear(in_dim, hidden_dim)

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.conv1 = TransformerConv(
            in_channels=hidden_dim, out_channels=hidden_dim // n_heads,
            heads=n_heads, edge_dim=edge_dim, dropout=dropout, concat=True
        )
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.conv2 = TransformerConv(
            in_channels=hidden_dim, out_channels=hidden_dim // n_heads,
            heads=n_heads, edge_dim=edge_dim, dropout=dropout, concat=True
        )
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim // 2, 1)
        )
        self.dropout = dropout

    def forward(self, x, edge_index, edge_attr):
        edge_emb = self.edge_proj(edge_attr)
        h = self.input_proj(x)
        h = h + F.dropout(self.conv1(self.norm1(h), edge_index, edge_emb), p=self.dropout, training=self.training)
        h = h + F.dropout(self.conv2(self.norm2(h), edge_index, edge_emb), p=self.dropout, training=self.training)
        h = self.norm_out(h)
        return self.classifier(h).squeeze(-1)

    @torch.no_grad()
    def get_embedding(self, x, edge_index, edge_attr):
        edge_emb = self.edge_proj(edge_attr)
        h = self.input_proj(x)
        h = h + self.conv1(self.norm1(h), edge_index, edge_emb)
        h = h + self.conv2(self.norm2(h), edge_index, edge_emb)
        h = self.norm_out(h)
        return h


# =============================================================
# 5. GNN 학습 (Warm-up + Cosine + 안정화)
# =============================================================
print("\n🔥 TGAT v3 학습 시작...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"📌 디바이스: {device}")

graph_data = Data(x=X_node, edge_index=edge_index_train, edge_attr=edge_attr_train, y=Y_node)
graph_data.train_mask = train_mask
graph_data.num_nodes = num_nodes

del X_node, edge_index_train, edge_attr_train, Y_node, train_mask
gc.collect()

train_loader = NeighborLoader(
    graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
    input_nodes=graph_data.train_mask, shuffle=True, num_workers=4
)
full_loader = NeighborLoader(
    graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
    input_nodes=None, shuffle=False, num_workers=4
)

model = TGAT_v3(
    in_dim=IN_DIM, hidden_dim=HIDDEN_DIM, edge_raw_dim=EDGE_RAW_DIM,
    edge_dim=EDGE_DIM, n_heads=N_HEADS, dropout=0.2
).to(device)

pw = (len(sampled_nodes) - len(active_pos)) / max(len(active_pos), 1)
print(f"📊 pos_weight: {pw:.1f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw], device=device))

BASE_LR = 0.001
WARMUP_EPOCHS = 5

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6, weight_decay=1e-4)

def set_lr(optimizer, epoch):
    if epoch < WARMUP_EPOCHS:
        lr = 1e-6 + (BASE_LR - 1e-6) * (epoch / WARMUP_EPOCHS)
    else:
        progress = (epoch - WARMUP_EPOCHS) / max(N_EPOCHS_GNN - WARMUP_EPOCHS, 1)
        lr = 1e-5 + (BASE_LR - 1e-5) * 0.5 * (1 + np.cos(np.pi * progress))
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    return lr

log_memory("학습 시작 전")

model.train()
best_loss = float('inf')
patience_counter = 0
PATIENCE = 10
best_state = None

for epoch in range(N_EPOCHS_GNN):
    current_lr = set_lr(optimizer, epoch)
    total_loss = 0.0
    num_batches = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{N_EPOCHS_GNN}", leave=False):
        batch = batch.to(device)
        optimizer.zero_grad()

        if batch.edge_attr is not None and batch.edge_attr.numel() > 0:
            batch_edge_attr = batch.edge_attr
        else:
            batch_edge_attr = torch.zeros(batch.edge_index.size(1), EDGE_RAW_DIM, device=device)

        out = model(batch.x, batch.edge_index, batch_edge_attr)
        loss = criterion(out[:batch.batch_size], batch.y[:batch.batch_size].float())

        if torch.isnan(loss) or torch.isinf(loss):
            print(f"  ⚠️ Epoch {epoch+1}: NaN/Inf loss 감지, 배치 스킵")
            optimizer.zero_grad()
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    avg_loss = total_loss / max(num_batches, 1)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  → Epoch {epoch+1:2d} | Loss: {avg_loss:.4f} | LR: {current_lr:.6f}")

    if avg_loss < best_loss - 0.0005:
        best_loss = avg_loss
        patience_counter = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print(f"  ⏹️ Early stopping at epoch {epoch+1} (patience {PATIENCE})")
        break

if best_state is not None:
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    print(f"  ✅ Best model 복원 (loss: {best_loss:.4f})")
    del best_state
else:
    print("  ⚠️ Best state 없음 — 현재 모델 사용")

# =============================================================
# 6. 임베딩 추출
# =============================================================
print("\n📦 TGAT v3 임베딩 추출 중...")
model.eval()
all_emb = []
with torch.no_grad():
    for batch in tqdm(full_loader, desc="Embedding Extraction"):
        batch = batch.to(device)
        if batch.edge_attr is not None and batch.edge_attr.numel() > 0:
            batch_edge_attr = batch.edge_attr
        else:
            batch_edge_attr = torch.zeros(batch.edge_index.size(1), EDGE_RAW_DIM, device=device)
        emb = model.get_embedding(batch.x, batch.edge_index, batch_edge_attr)
        all_emb.append(emb[:batch.batch_size].cpu())

final_emb = torch.cat(all_emb, dim=0).numpy()
emb_cols = [f"tgat_emb_{i}" for i in range(HIDDEN_DIM)]

df_tgat = pl.concat([
    all_accounts.select("account_id"),
    pl.DataFrame(final_emb, schema=emb_cols)
], how="horizontal")

print(f"✅ 임베딩 shape: {final_emb.shape}")

del graph_data, all_emb, final_emb, all_accounts
torch.cuda.empty_cache()
gc.collect()
log_memory("임베딩 추출 완료")

# =============================================================
# 7. XGBoost 데이터 구성 (V2 + 그래프 피처 + TGAT 임베딩)
# =============================================================
print("\n🚀 XGBoost 데이터 구성 중...")

xgb_exclude_cols = ["account_id", "time_group", "is_laundering", "mode_format", "currency_mode", "date"]
xgb_feature_cols = [c for c in df_v2.columns if c not in xgb_exclude_cols] + graph_feature_cols + emb_cols

print(f"📊 총 피처 수: {len(xgb_feature_cols)} "
      f"(V2: {len([c for c in df_v2.columns if c not in xgb_exclude_cols])}, "
      f"그래프: {len(graph_feature_cols)}, TGAT: {len(emb_cols)})")

# [1] Train — 다운샘플링
df_train_full = (
    df_v2.filter(pl.col("time_group") < train_cutoff)
    .join(df_graph_feats, on="account_id", how="left")
    .join(df_tgat, on="account_id", how="left")
    .fill_null(0.0)
)
df_train_pos = df_train_full.filter(pl.col("is_laundering") == 1)
df_train_neg = df_train_full.filter(pl.col("is_laundering") == 0)
n_neg_keep_xgb = min(len(df_train_pos) * NEG_ROW_RATIO_XGB, len(df_train_neg))
df_train_neg_sampled = df_train_neg.sample(n=n_neg_keep_xgb, seed=RANDOM_SEED)
df_train = pl.concat([df_train_pos, df_train_neg_sampled]).sample(fraction=1.0, seed=RANDOM_SEED)

X_train = df_train.select(xgb_feature_cols).to_pandas()
y_train = df_train["is_laundering"].to_numpy()
print(f"📊 Train: {len(df_train_full):,} → {len(df_train):,} (양성비율: {y_train.mean()*100:.2f}%)")
del df_train, df_train_full, df_train_pos, df_train_neg, df_train_neg_sampled; gc.collect()

# [2] Val
df_val = (
    df_v2.filter((pl.col("time_group") >= train_cutoff) & (pl.col("time_group") < val_cutoff))
    .join(df_graph_feats, on="account_id", how="left")
    .join(df_tgat, on="account_id", how="left")
    .fill_null(0.0)
)
X_val = df_val.select(xgb_feature_cols).to_pandas()
y_val = df_val["is_laundering"].to_numpy()
print(f"📊 Val  : {len(X_val):,} rows, 양성비율: {y_val.mean()*100:.2f}%")
del df_val; gc.collect()

# [3] Test
df_test = (
    df_v2.filter(pl.col("time_group") >= val_cutoff)
    .join(df_graph_feats, on="account_id", how="left")
    .join(df_tgat, on="account_id", how="left")
    .fill_null(0.0)
)
df_test_meta = df_test.select(["account_id", "time_group", "is_laundering"])
X_test = df_test.select(xgb_feature_cols).to_pandas()
y_test = df_test["is_laundering"].to_numpy()
print(f"📊 Test : {len(X_test):,} rows, 양성비율: {y_test.mean()*100:.2f}%")
del df_test, df_v2, df_tgat, df_graph_feats; gc.collect()
log_memory("XGBoost 데이터 준비 완료")

# =============================================================
# 8. XGBoost 학습 (고정 하이퍼파라미터)
# =============================================================
print("\n🔥 XGBoost 학습 시작 (고정 하이퍼파라미터)...")

actual_pos_ratio = y_train.sum() / len(y_train) if len(y_train) > 0 else 0.01
adjusted_spw = max((1 - actual_pos_ratio) / actual_pos_ratio, 1.0)
print(f"📊 scale_pos_weight: {adjusted_spw:.1f}")

xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "device": "cuda",
    "random_state": 42,
    "max_depth": 8,
    "learning_rate": 0.03,
    "scale_pos_weight": adjusted_spw,
    "subsample": 0.8,
    "colsample_bytree": 0.7,
    "min_child_weight": 5,
    "gamma": 0.1,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
}

model_xgb = xgb.XGBClassifier(**xgb_params, n_estimators=1500, early_stopping_rounds=100)
model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)

y_prob = model_xgb.predict_proba(X_test)[:, 1]

# Feature Importance
print("\n🔍 Feature Importance Top 20")
importance = model_xgb.feature_importances_
top_indices = np.argsort(importance)[::-1][:20]
for i, idx in enumerate(top_indices):
    tag = ""
    if xgb_feature_cols[idx].startswith("tgat_emb_"):
        tag = " [TGAT]"
    elif xgb_feature_cols[idx].startswith("graph_"):
        tag = " [GRAPH]"
    print(f"  {i+1:2d}. {xgb_feature_cols[idx]:<30s}: {importance[idx]:.4f}{tag}")

tgat_imp = sum(importance[i] for i, c in enumerate(xgb_feature_cols) if c.startswith("tgat_emb_"))
graph_imp = sum(importance[i] for i, c in enumerate(xgb_feature_cols) if c.startswith("graph_"))
v2_imp = importance.sum() - tgat_imp - graph_imp
print(f"\n📊 피처 그룹별 importance:")
print(f"  V2 피처  : {v2_imp:.4f} ({v2_imp/importance.sum()*100:.1f}%)")
print(f"  그래프   : {graph_imp:.4f} ({graph_imp/importance.sum()*100:.1f}%)")
print(f"  TGAT 임베딩: {tgat_imp:.4f} ({tgat_imp/importance.sum()*100:.1f}%)")

del X_train, X_val, X_test; gc.collect()

# =============================================================
# 9. 최종 평가 리포트
# =============================================================
print("\n" + "=" * 60)
print("🏆 [TGAT v3 + 그래프 피처] 최종 성능 리포트")
print("=" * 60)

auprc = average_precision_score(y_test, y_prob)

print(f"\n📌 [다중 임계값 분석 (Test: {val_cutoff} 이후)]")
print(f"  AUPRC: {auprc:.4f}")

best_f1 = 0
best_thresh = 0
for thresh in np.arange(0.05, 0.95, 0.01):
    y_pred_t = (y_prob >= thresh).astype(int)
    f = f1_score(y_test, y_pred_t, zero_division=0)
    if f > best_f1:
        best_f1 = f
        best_thresh = thresh

y_pred = (y_prob >= best_thresh).astype(int)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print(f"\n📌 [최적 임계값 결과]")
print(f"  - Best Threshold: {best_thresh:.2f}")
print(f"  - AUPRC    : {auprc:.4f}")
print(f"  - F1-Score : {best_f1:.4f}")
print(f"  - Precision: {precision:.4f}")
print(f"  - Recall   : {recall:.4f}")

print(f"\n📌 [주요 임계값별 성능]")
for thresh in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    y_pred_t = (y_prob >= thresh).astype(int)
    p = precision_score(y_test, y_pred_t, zero_division=0)
    r = recall_score(y_test, y_pred_t, zero_division=0)
    f = f1_score(y_test, y_pred_t, zero_division=0)
    print(f"  T={thresh:.1f} | P={p:.4f} R={r:.4f} F1={f:.4f}")

# Top-K
df_res = df_test_meta.with_columns([
    pl.col("time_group").dt.date().alias("date"),
    pl.Series("pred_prob", y_prob)
])
df_distinct = df_res.sort("pred_prob", descending=True).unique(subset=["account_id"], maintain_order=True)

print(f"\n📌 [Top-K 탐지 (Distinct Account 기준)]")
for k in [100, 500, 1000, 5000, 10000]:
    current_k = min(k, len(df_distinct))
    if current_k > 0:
        hits = df_distinct.head(current_k)["is_laundering"].sum()
        print(f"  📍 Top-{k:5d} Hits: {hits:5d}명 | Precision: {(hits/current_k)*100:6.2f}%")

print(f"\n📌 [일별 Top-100 탐지 현황]")
unique_dates = df_distinct["date"].unique(maintain_order=True).sort(descending=True)
daily_stats = []
for d in unique_dates:
    hits = df_distinct.filter(pl.col("date") == d).head(100)["is_laundering"].sum()
    daily_stats.append({"Date": str(d), "Top-100 Hits": hits})
print(pl.DataFrame(daily_stats))

log_memory("최종 완료")
print("\n✅ TGAT v3 파이프라인 완료 (Optuna 없음)!")

🛡️ [TGAT v3] GNN 안정화 + 그래프 피처 (Optuna 없음)

📂 데이터 로딩 중...
⏱️ Train cutoff: 2022-09-09 20:00:00
⏱️ Val   cutoff: 2022-09-14 05:00:00
📊 전체 노드 수: 2,076,999
  💾 [데이터 로드 완료] RAM: 33.01GB | GPU: 0.00GB

📐 그래프 구조 피처 계산 중 (Train 기간)...
  📊 PageRank 계산 중 (10 iterations)...
✅ 그래프 피처 13개 생성: ['graph_out_degree', 'graph_in_degree', 'graph_unique_out_partners', 'graph_unique_in_partners', 'graph_bidir_count', 'graph_avg_tx_interval', 'graph_std_tx_interval', 'graph_total_tx_count', 'graph_pagerank', 'graph_total_degree', 'graph_out_in_ratio', 'graph_reciprocity', 'graph_partner_diversity']
  💾 [그래프 피처 완료] RAM: 36.11GB | GPU: 0.00GB

🔗 GNN 엣지/노드 구성 중...
📊 Train 엣지: 19,060,343
📊 GNN 학습 노드: 양성 20,400 + 음성 1,020,000 = 1,040,400
  💾 [GNN 데이터 준비 완료] RAM: 37.31GB | GPU: 0.00GB

🧠 TGAT v3 모델 정의 (안정화 버전)...

🔥 TGAT v3 학습 시작...
📌 디바이스: cuda
📊 pos_weight: 50.0
  💾 [학습 시작 전] RAM: 38.02GB | GPU: 0.00GB


  → Epoch  1 | Loss: 1.3762 | LR: 0.000001


  → Epoch  5 | Loss: 1.1558 | LR: 0.000800


  → Epoch 10 | Loss: 1.1004 | LR: 0.000968


  → Epoch 15 | Loss: 1.0748 | LR: 0.000847


...

  → Epoch 30 | Loss: 1.0390 | LR: 0.000232


  → Epoch 35 | Loss: 1.0311 | LR: 0.000080


  → Epoch 40 | Loss: 1.0281 | LR: 0.000012
  ✅ Best model 복원 (loss: 1.0275)

📦 TGAT v3 임베딩 추출 중...


Embedding Extraction: 100%|█████████████████████████████████████████████████| 1015/1015 [00:14<00:00, 70.59it/s]


✅ 임베딩 shape: (2076999, 64)
  💾 [임베딩 추출 완료] RAM: 39.53GB | GPU: 0.02GB

🚀 XGBoost 데이터 구성 중...
📊 총 피처 수: 115 (V2: 38, 그래프: 13, TGAT: 64)
📊 Train: 17,172,747 → 619,437 (양성비율: 4.76%)
📊 Val  : 5,790,644 rows, 양성비율: 0.30%
📊 Test : 5,792,913 rows, 양성비율: 0.36%
  💾 [XGBoost 데이터 준비 완료] RAM: 60.58GB | GPU: 0.02GB

🔥 XGBoost 학습 시작 (고정 하이퍼파라미터)...
📊 scale_pos_weight: 20.0
[0]	validation_0-aucpr:0.06676
[100]	validation_0-aucpr:0.32648
[200]	validation_0-aucpr:0.40855
[300]	validation_0-aucpr:0.42853
[400]	validation_0-aucpr:0.44005
[500]	validation_0-aucpr:0.44571
[600]	validation_0-aucpr:0.45101
[700]	validation_0-aucpr:0.45325
[800]	validation_0-aucpr:0.45700
[900]	validation_0-aucpr:0.45791
[1000]	validation_0-aucpr:0.46201
[1100]	validation_0-aucpr:0.46293
[1200]	validation_0-aucpr:0.46500
[1300]	validation_0-aucpr:0.46710
[1400]	validation_0-aucpr:0.46921
[1499]	validation_0-aucpr:0.47061

🔍 Feature Importance Top 20
   1. ratio_cross_border            : 0.1425
   2. cnt_risk_format           

In [2]:
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv
from torch_geometric.loader import NeighborLoader
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score
from tqdm import tqdm
import gc
import numpy as np
import psutil
import os
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

def log_memory(tag=""):
    proc = psutil.Process(os.getpid())
    rss_gb = proc.memory_info().rss / (1024 ** 3)
    gpu_str = ""
    if torch.cuda.is_available():
        gpu_alloc = torch.cuda.memory_allocated() / (1024 ** 3)
        gpu_str = f" | GPU: {gpu_alloc:.2f}GB"
    print(f"  💾 [{tag}] RAM: {rss_gb:.2f}GB{gpu_str}")

print("=" * 60)
print("🛡️ [TGAT v4] 구조×행동 피처 고도화 파이프라인")
print("=" * 60)

# ===================== 설정 =====================
NEG_NODE_RATIO_GNN = 50
NEG_ROW_RATIO_XGB = 20
RANDOM_SEED = 42
HIDDEN_DIM = 64
EDGE_DIM = 16
N_HEADS = 4
N_EPOCHS_GNN = 40
BATCH_SIZE = 2048
NUM_NEIGHBORS = [15, 10]
N_INTERACTION_DIMS = 8   # TGAT×PageRank interaction 차원 수
# ================================================

raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"

# =============================================================
# 1. 데이터 로드 및 6:2:2 시간 분할
# =============================================================
print("\n📂 데이터 로딩 중...")
df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))

if df_raw["Timestamp"].dtype == pl.Utf8:
    df_raw = df_raw.with_columns(
        pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False)
    )

df_v2 = df_v2.sort("time_group")
total_count = len(df_v2)
train_idx = int(total_count * 0.6)
val_idx = int(total_count * 0.8)

train_cutoff = df_v2["time_group"][train_idx]
val_cutoff = df_v2["time_group"][val_idx]

print(f"⏱️ Train cutoff: {train_cutoff}")
print(f"⏱️ Val   cutoff: {val_cutoff}")

all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")

num_nodes = len(all_accounts)
print(f"📊 전체 노드 수: {num_nodes:,}")
log_memory("데이터 로드 완료")

# =============================================================
# 2. 기본 그래프 구조 피처 (v3 동일)
# =============================================================
print("\n📐 [Step 2] 기본 그래프 구조 피처 계산...")

df_raw_train = df_raw.filter(pl.col("Timestamp") < train_cutoff)
del df_raw; gc.collect()

# Degree
df_from = df_raw_train.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id"))
df_to = df_raw_train.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
out_degree = df_from.group_by("account_id").len().rename({"len": "graph_out_degree"})
in_degree = df_to.group_by("account_id").len().rename({"len": "graph_in_degree"})

# Unique partners
unique_out = (
    df_raw_train.select([
        pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
        pl.col("to_acc").cast(pl.Utf8).alias("partner")
    ]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_out_partners"))
)
unique_in = (
    df_raw_train.select([
        pl.col("to_acc").cast(pl.Utf8).alias("account_id"),
        pl.col("from_acc").cast(pl.Utf8).alias("partner")
    ]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_in_partners"))
)

# Bidirectional
edges_set_from = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("a"),
    pl.col("to_acc").cast(pl.Utf8).alias("b")
]).unique()
edges_set_to = edges_set_from.select([pl.col("b").alias("a"), pl.col("a").alias("b")])
bidirectional = edges_set_from.join(edges_set_to, on=["a", "b"], how="inner")
bidir_count = (
    bidirectional.select(pl.col("a").alias("account_id"))
    .group_by("account_id").len().rename({"len": "graph_bidir_count"})
)
del edges_set_from, edges_set_to, bidirectional; gc.collect()

# Time features
time_features = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("Timestamp")
]).group_by("account_id").agg([
    pl.col("Timestamp").sort().diff().dt.total_seconds().mean().alias("graph_avg_tx_interval"),
    pl.col("Timestamp").sort().diff().dt.total_seconds().std().alias("graph_std_tx_interval"),
    pl.col("Timestamp").count().alias("graph_total_tx_count"),
])

# PageRank
print("  📊 PageRank 계산 중 (10 iterations)...")
account_to_id = dict(zip(all_accounts["account_id"].to_list(), all_accounts["node_id"].to_list()))

src_ids = df_raw_train["from_acc"].cast(pl.Utf8).map_elements(
    lambda x: account_to_id.get(x, -1), return_dtype=pl.Int64
).to_numpy()
dst_ids = df_raw_train["to_acc"].cast(pl.Utf8).map_elements(
    lambda x: account_to_id.get(x, -1), return_dtype=pl.Int64
).to_numpy()

valid_mask = (src_ids >= 0) & (dst_ids >= 0)
src_valid = src_ids[valid_mask]
dst_valid = dst_ids[valid_mask]

out_deg_arr = np.zeros(num_nodes, dtype=np.float64)
np.add.at(out_deg_arr, src_valid, 1.0)
out_deg_arr = np.maximum(out_deg_arr, 1.0)

damping = 0.85
pr = np.ones(num_nodes, dtype=np.float64) / num_nodes
for _ in range(10):
    new_pr = np.ones(num_nodes, dtype=np.float64) * (1 - damping) / num_nodes
    contributions = pr[src_valid] / out_deg_arr[src_valid]
    np.add.at(new_pr, dst_valid, damping * contributions)
    pr = new_pr

pr_df = pl.DataFrame({
    "node_id": np.arange(num_nodes, dtype=np.uint32),
    "graph_pagerank": pr.astype(np.float32)
})
pr_df = all_accounts.join(pr_df, on="node_id", how="left").select(["account_id", "graph_pagerank"])

del src_ids, dst_ids, src_valid, dst_valid, out_deg_arr, pr, new_pr; gc.collect()

# 기본 그래프 피처 결합
df_graph_feats = (
    all_accounts.select("account_id")
    .join(out_degree, on="account_id", how="left")
    .join(in_degree, on="account_id", how="left")
    .join(unique_out, on="account_id", how="left")
    .join(unique_in, on="account_id", how="left")
    .join(bidir_count, on="account_id", how="left")
    .join(time_features, on="account_id", how="left")
    .join(pr_df, on="account_id", how="left")
    .fill_null(0.0)
)
df_graph_feats = df_graph_feats.with_columns([
    (pl.col("graph_out_degree") + pl.col("graph_in_degree")).alias("graph_total_degree"),
    (pl.col("graph_out_degree") / (pl.col("graph_in_degree") + 1)).alias("graph_out_in_ratio"),
    (pl.col("graph_bidir_count") / (pl.col("graph_out_degree") + 1)).alias("graph_reciprocity"),
    (pl.col("graph_unique_out_partners") / (pl.col("graph_out_degree") + 1)).alias("graph_partner_diversity"),
])

del out_degree, in_degree, unique_out, unique_in, bidir_count, time_features, pr_df, df_from, df_to
gc.collect()
print(f"  ✅ 기본 그래프 피처: {len([c for c in df_graph_feats.columns if c.startswith('graph_')])}개")

# =============================================================
# 2-A. ★ 구조-조건부 행동 피처 10개 (NEW)
#    "구조적 위치가 X인 노드의 행동 패턴 Y" 형태
# =============================================================
print("\n🔬 [Step 2-A] 구조-조건부 행동 피처 10개 계산...")

# V2 피처를 account_id 기준으로 Train 기간 평균
df_v2_train = df_v2.filter(pl.col("time_group") < train_cutoff)
exclude_cols = ["account_id", "time_group", "is_laundering", "mode_format", "currency_mode"]
gnn_feature_cols = [c for c in df_v2.columns if c not in exclude_cols]

df_v2_agg = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols
])

# V2 + 그래프 구조 결합 (조건부 피처 계산용)
df_cond = df_v2_agg.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)

# 주요 V2 컬럼 존재 여부 확인 후 안전하게 접근
v2_cols_available = set(df_cond.columns)

def safe_col(name, default=0.0):
    return pl.col(name) if name in v2_cols_available else pl.lit(default)

# --- 10개 구조-조건부 행동 피처 ---
cond_exprs = []

# 1) 고degree 노드의 야간거래 집중도: degree가 높은데 야간에 거래가 몰리면 의심
if "cnt_night" in v2_cols_available and "cnt_1h" in v2_cols_available:
    cond_exprs.append(
        (pl.col("graph_total_degree") * safe_col("cnt_night") / (safe_col("cnt_1h") + 1))
        .alias("cond_highdeg_night_intensity")
    )

# 2) 저degree 노드의 고액거래 비율: 거래 적은데 금액이 크면 mule 의심
if "avg_log_amount" in v2_cols_available:
    cond_exprs.append(
        (safe_col("avg_log_amount") / (pl.col("graph_total_degree").log1p() + 1))
        .alias("cond_lowdeg_high_amount")
    )

# 3) 단방향(비reciprocal) 노드의 cross-border 비율: 한방향으로만 해외 송금
if "ratio_cross_border" in v2_cols_available:
    cond_exprs.append(
        (safe_col("ratio_cross_border") * (1 - pl.col("graph_reciprocity")))
        .alias("cond_oneway_crossborder")
    )

# 4) fan-out 집중도 × 거래 빈도: 짧은 시간에 다수 상대에게 송금 (layering 패턴)
if "cnt_1h" in v2_cols_available:
    cond_exprs.append(
        (pl.col("graph_unique_out_partners") * safe_col("cnt_1h") / (pl.col("graph_out_degree") + 1))
        .alias("cond_fanout_burst")
    )

# 5) PageRank × 위험 포맷 거래 비율: 중요 노드가 위험 포맷을 많이 쓰면 허브 역할
if "cnt_risk_format" in v2_cols_available:
    cond_exprs.append(
        (pl.col("graph_pagerank") * safe_col("cnt_risk_format"))
        .alias("cond_hub_risk_format")
    )

# 6) in/out 비대칭 × wire 거래 비율: 들어오기만 하는 계좌가 wire 거래 많으면 mule
if "cnt_wire" in v2_cols_available:
    cond_exprs.append(
        (pl.col("graph_out_in_ratio") * safe_col("cnt_wire"))
        .alias("cond_asymmetric_wire")
    )

# 7) 거래 간격 불규칙성 × interbank 비율: 불규칙하게 은행 간 거래 (분산 세탁)
if "cnt_inter_bank" in v2_cols_available:
    cond_exprs.append(
        (pl.col("graph_std_tx_interval").fill_null(0) / (pl.col("graph_avg_tx_interval").fill_null(1) + 1)
         * safe_col("cnt_inter_bank"))
        .alias("cond_irregular_interbank")
    )

# 8) partner diversity × 통화 다양성: 다양한 상대 + 다양한 통화 = layering
if "cnt_currencies" in v2_cols_available:
    cond_exprs.append(
        (pl.col("graph_partner_diversity") * safe_col("cnt_currencies"))
        .alias("cond_diverse_partner_currency")
    )

# 9) 양방향 거래 노드의 금액 변동성: 왕복 거래하면서 금액 분산이 크면 세탁 은폐
if "amount_kurtosis" in v2_cols_available:
    cond_exprs.append(
        (pl.col("graph_reciprocity") * safe_col("amount_kurtosis"))
        .alias("cond_bidir_amount_volatility")
    )

# 10) 고 entity 계좌 수 × burst ratio: 한 entity가 다수 계좌 + 폭발적 거래
if "entity_acct_cnt" in v2_cols_available and "burst_ratio" in v2_cols_available:
    cond_exprs.append(
        (safe_col("entity_acct_cnt") * safe_col("burst_ratio"))
        .alias("cond_multi_acct_burst")
    )

df_cond_features = df_cond.select("account_id").with_columns(
    [df_cond[expr.meta.output_name()] if hasattr(expr, 'meta') else pl.lit(0.0) for expr in []]
)

# 실제 계산
df_cond = df_cond.with_columns(cond_exprs)
cond_feature_cols = [expr.meta.output_name() for expr in cond_exprs]

# inf/NaN 안전 처리
for cn in cond_feature_cols:
    df_cond = df_cond.with_columns(
        pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
        .then(0.0)
        .otherwise(pl.col(cn))
        .alias(cn)
    )

df_cond_features = df_cond.select(["account_id"] + cond_feature_cols)

print(f"  ✅ 구조-조건부 행동 피처 {len(cond_feature_cols)}개: {cond_feature_cols}")

del df_cond; gc.collect()

# =============================================================
# 2-B. ★ Smurf / Mule / Layering 점수 5개 (NEW)
# =============================================================
print("\n🎯 [Step 2-B] AML 패턴 점수 5개 설계...")

# V2+그래프를 다시 결합 (점수 계산용)
df_score_base = df_v2_agg.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)

score_exprs = []

# --- Smurf Score (구조화 거래) ---
# 특징: 소액 분산, 많은 거래 횟수, 다수 계좌로 분산, 임계값 이하 금액
# = (거래 빈도 × 상대 다양성) / (평균 금액 + 1) × burst_ratio
smurf_parts = []
if "cnt_1h" in v2_cols_available:
    smurf_parts.append("safe_cnt_1h")
if "burst_ratio" in v2_cols_available:
    smurf_parts.append("safe_burst")

score_exprs.append(
    (
        (safe_col("cnt_1h").fill_null(0).clip(0, 1e6) + 1).log1p()
        * pl.col("graph_unique_out_partners").clip(0, 1e6).log1p()
        * safe_col("burst_ratio").fill_null(0).clip(0, 1e6)
        / (safe_col("avg_log_amount").fill_null(1).clip(0, 1e6) + 1)
    ).alias("score_smurf")
)

# --- Mule Score (자금 전달) ---
# 특징: 돈이 들어오고 곧 나감 (in≈out), 적은 고유 파트너, 비reciprocal, wire 비중 높음
score_exprs.append(
    (
        (1.0 / ((pl.col("graph_out_in_ratio").clip(0.01, 100) - 1.0).abs() + 0.1))  # in≈out일수록 높음, 안전한 분모
        * safe_col("cnt_wire").fill_null(0).clip(0, 1e6).log1p()
        * (1 - pl.col("graph_reciprocity").clip(0, 1))
        / (pl.col("graph_unique_out_partners").clip(0, 1e6).log1p() + 1)
    ).alias("score_mule")
)

# --- Layering Score (겹겹이 거래) ---
# 특징: 높은 fan-out, 다양한 통화, cross-border, 빠른 연속 거래
score_exprs.append(
    (
        pl.col("graph_unique_out_partners").clip(0, 1e6).log1p()
        * safe_col("cnt_currencies").fill_null(0).clip(0, 1e6).log1p()
        * safe_col("ratio_cross_border").fill_null(0).clip(0, 1)
        / (pl.col("graph_avg_tx_interval").fill_null(3600).clip(0, 1e9) / 3600 + 1)
    ).alias("score_layering")
)

# --- Integration Score (통합/현금화) ---
# 특징: 높은 in-degree, PageRank 높음, 적은 out 거래, risk format 사용
score_exprs.append(
    (
        pl.col("graph_in_degree").clip(0, 1e6).log1p()
        * (pl.col("graph_pagerank").clip(0, 1) * 1e6)
        * safe_col("cnt_risk_format").fill_null(0).clip(0, 1e6).log1p()
        / (pl.col("graph_out_degree").clip(0, 1e6).log1p() + 1)
    ).alias("score_integration")
)

# --- Anomaly Composite Score (종합 이상 점수) ---
# 위 4개 점수의 가중합 (모든 패턴에 걸쳐 전반적 의심도)
# → 개별 점수 계산 후 후처리로 생성

df_score_base = df_score_base.with_columns(score_exprs)

# inf/NaN 안전 처리
score_names = ["score_smurf", "score_mule", "score_layering", "score_integration"]
for sn in score_names:
    df_score_base = df_score_base.with_columns(
        pl.when(pl.col(sn).is_infinite() | pl.col(sn).is_nan())
        .then(0.0)
        .otherwise(pl.col(sn))
        .alias(sn)
    )

# 각 점수를 0~1 min-max 정규화 후 composite 생성
for sn in score_names:
    col_min = df_score_base[sn].min()
    col_max = df_score_base[sn].max()
    col_range = col_max - col_min if (col_max - col_min) > 0 else 1.0
    df_score_base = df_score_base.with_columns(
        ((pl.col(sn) - col_min) / col_range).alias(f"{sn}_norm")
    )

df_score_base = df_score_base.with_columns(
    (
        pl.col("score_smurf_norm") * 0.3
        + pl.col("score_mule_norm") * 0.25
        + pl.col("score_layering_norm") * 0.3
        + pl.col("score_integration_norm") * 0.15
    ).alias("score_composite")
)

# 정규화 임시 컬럼 제거
aml_score_cols = score_names + ["score_composite"]
df_aml_scores = df_score_base.select(["account_id"] + aml_score_cols)

print(f"  ✅ AML 패턴 점수 {len(aml_score_cols)}개: {aml_score_cols}")

del df_score_base, df_v2_agg; gc.collect()
log_memory("구조-행동 피처 + AML 점수 완료")

# =============================================================
# 3. GNN 엣지 + 노드 피처 구성
# =============================================================
print("\n🔗 [Step 3] GNN 엣지/노드 구성 중...")

df_edges_train = df_raw_train.select(["from_acc", "to_acc", "Timestamp"]).with_columns([
    pl.col("from_acc").cast(pl.Utf8),
    pl.col("to_acc").cast(pl.Utf8)
])
df_edges_train = df_edges_train.join(
    all_accounts.rename({"account_id": "from_acc", "node_id": "src_id"}), on="from_acc", how="left"
)
df_edges_train = df_edges_train.join(
    all_accounts.rename({"account_id": "to_acc", "node_id": "dst_id"}), on="to_acc", how="left"
)

timestamps_raw = df_edges_train["Timestamp"]
min_ts = timestamps_raw.min()
max_ts = timestamps_raw.max()
total_seconds = (max_ts - min_ts).total_seconds()

edge_time_normalized = (
    (timestamps_raw - min_ts).dt.total_microseconds().cast(pl.Float64) / 1e6 / max(total_seconds, 1.0)
).to_numpy().astype(np.float32)

hours = df_edges_train["Timestamp"].dt.hour().to_numpy().astype(np.float32)
dow = df_edges_train["Timestamp"].dt.weekday().to_numpy().astype(np.float32)

edge_features = np.stack([
    edge_time_normalized,
    np.sin(2 * np.pi * hours / 24),
    np.cos(2 * np.pi * hours / 24),
    np.sin(2 * np.pi * dow / 7),
    np.cos(2 * np.pi * dow / 7),
], axis=1)

edge_index_train = torch.tensor(
    df_edges_train.select(["src_id", "dst_id"]).to_numpy().T, dtype=torch.long
)
edge_attr_train = torch.tensor(edge_features, dtype=torch.float32)
EDGE_RAW_DIM = edge_attr_train.shape[1]
print(f"📊 Train 엣지: {edge_index_train.shape[1]:,}")

del df_raw_train, df_edges_train, timestamps_raw, hours, dow, edge_features, edge_time_normalized
gc.collect()

# 노드 피처 (GNN용은 V2만 사용 — 차원 안정성)
df_v2_node = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols
])
df_node_features = all_accounts.join(df_v2_node, on="account_id", how="left").fill_null(0.0)
X_node = torch.tensor(df_node_features.select(gnn_feature_cols).to_numpy(), dtype=torch.float32)
IN_DIM = X_node.shape[1]

# 레이블
target_labels = (
    df_v2_train.filter(pl.col("is_laundering") == 1)
    .select("account_id").unique()
    .with_columns(pl.lit(1).alias("label"))
)
df_labels = all_accounts.join(target_labels, on="account_id", how="left").fill_null(0)
Y_node = torch.tensor(df_labels["label"].to_numpy(), dtype=torch.long)

# Train 마스크
active_accounts = df_v2_train["account_id"].unique()
active_df = pl.DataFrame({"account_id": active_accounts, "is_active": True})
mask_df = all_accounts.join(active_df, on="account_id", how="left").fill_null(False)

pos_node_ids = set(df_labels.filter(pl.col("label") == 1)["node_id"].to_list())
active_node_ids = mask_df.filter(pl.col("is_active"))["node_id"].to_list()
active_pos = [nid for nid in active_node_ids if nid in pos_node_ids]
active_neg = [nid for nid in active_node_ids if nid not in pos_node_ids]

n_neg_keep_gnn = min(len(active_pos) * NEG_NODE_RATIO_GNN, len(active_neg))
np.random.seed(RANDOM_SEED)
sampled_neg = np.random.choice(active_neg, size=n_neg_keep_gnn, replace=False).tolist()
sampled_nodes = set(active_pos + sampled_neg)

train_mask_np = np.zeros(num_nodes, dtype=bool)
for nid in sampled_nodes:
    train_mask_np[nid] = True
train_mask = torch.tensor(train_mask_np, dtype=torch.bool)

print(f"📊 GNN 학습 노드: 양성 {len(active_pos):,} + 음성 {n_neg_keep_gnn:,} = {len(sampled_nodes):,}")

del df_v2_node, df_node_features, mask_df, active_df
gc.collect()
log_memory("GNN 데이터 준비 완료")

# =============================================================
# 4. TGAT v3 모델 (v3과 동일)
# =============================================================
print("\n🧠 [Step 4] TGAT v3 모델 정의...")


class EdgeProjection(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, out_dim), nn.GELU(), nn.Linear(out_dim, out_dim)
        )

    def forward(self, edge_attr):
        return self.proj(edge_attr)


class TGAT_v3(nn.Module):
    def __init__(self, in_dim, hidden_dim, edge_raw_dim, edge_dim, n_heads, dropout=0.2):
        super().__init__()
        self.edge_proj = EdgeProjection(edge_raw_dim, edge_dim)
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.conv1 = TransformerConv(
            hidden_dim, hidden_dim // n_heads, heads=n_heads,
            edge_dim=edge_dim, dropout=dropout, concat=True
        )
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.conv2 = TransformerConv(
            hidden_dim, hidden_dim // n_heads, heads=n_heads,
            edge_dim=edge_dim, dropout=dropout, concat=True
        )
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim // 2, 1)
        )
        self.dropout = dropout

    def forward(self, x, edge_index, edge_attr):
        edge_emb = self.edge_proj(edge_attr)
        h = self.input_proj(x)
        h = h + F.dropout(self.conv1(self.norm1(h), edge_index, edge_emb), p=self.dropout, training=self.training)
        h = h + F.dropout(self.conv2(self.norm2(h), edge_index, edge_emb), p=self.dropout, training=self.training)
        h = self.norm_out(h)
        return self.classifier(h).squeeze(-1)

    @torch.no_grad()
    def get_embedding(self, x, edge_index, edge_attr):
        edge_emb = self.edge_proj(edge_attr)
        h = self.input_proj(x)
        h = h + self.conv1(self.norm1(h), edge_index, edge_emb)
        h = h + self.conv2(self.norm2(h), edge_index, edge_emb)
        return self.norm_out(h)


# =============================================================
# 5. GNN 학습
# =============================================================
print("\n🔥 [Step 5] TGAT 학습 시작...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"📌 디바이스: {device}")

graph_data = Data(x=X_node, edge_index=edge_index_train, edge_attr=edge_attr_train, y=Y_node)
graph_data.train_mask = train_mask
graph_data.num_nodes = num_nodes

del X_node, edge_index_train, edge_attr_train, Y_node, train_mask
gc.collect()

train_loader = NeighborLoader(
    graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
    input_nodes=graph_data.train_mask, shuffle=True, num_workers=4
)
full_loader = NeighborLoader(
    graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
    input_nodes=None, shuffle=False, num_workers=4
)

model = TGAT_v3(
    in_dim=IN_DIM, hidden_dim=HIDDEN_DIM, edge_raw_dim=EDGE_RAW_DIM,
    edge_dim=EDGE_DIM, n_heads=N_HEADS, dropout=0.2
).to(device)

pw = (len(sampled_nodes) - len(active_pos)) / max(len(active_pos), 1)
print(f"📊 pos_weight: {pw:.1f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw], device=device))
BASE_LR = 0.001
WARMUP_EPOCHS = 5
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6, weight_decay=1e-4)

def set_lr(optimizer, epoch):
    if epoch < WARMUP_EPOCHS:
        lr = 1e-6 + (BASE_LR - 1e-6) * (epoch / WARMUP_EPOCHS)
    else:
        progress = (epoch - WARMUP_EPOCHS) / max(N_EPOCHS_GNN - WARMUP_EPOCHS, 1)
        lr = 1e-5 + (BASE_LR - 1e-5) * 0.5 * (1 + np.cos(np.pi * progress))
    for pg in optimizer.param_groups:
        pg['lr'] = lr
    return lr

model.train()
best_loss = float('inf')
patience_counter = 0
PATIENCE = 10
best_state = None

for epoch in range(N_EPOCHS_GNN):
    current_lr = set_lr(optimizer, epoch)
    total_loss = 0.0
    num_batches = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{N_EPOCHS_GNN}", leave=False):
        batch = batch.to(device)
        optimizer.zero_grad()
        if batch.edge_attr is not None and batch.edge_attr.numel() > 0:
            batch_edge_attr = batch.edge_attr
        else:
            batch_edge_attr = torch.zeros(batch.edge_index.size(1), EDGE_RAW_DIM, device=device)

        out = model(batch.x, batch.edge_index, batch_edge_attr)
        loss = criterion(out[:batch.batch_size], batch.y[:batch.batch_size].float())

        if torch.isnan(loss) or torch.isinf(loss):
            optimizer.zero_grad()
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        optimizer.step()
        total_loss += loss.item()
        num_batches += 1

    avg_loss = total_loss / max(num_batches, 1)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  → Epoch {epoch+1:2d} | Loss: {avg_loss:.4f} | LR: {current_lr:.6f}")

    if avg_loss < best_loss - 0.0005:
        best_loss = avg_loss
        patience_counter = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1
    if patience_counter >= PATIENCE:
        print(f"  ⏹️ Early stopping at epoch {epoch+1}")
        break

if best_state is not None:
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    print(f"  ✅ Best model 복원 (loss: {best_loss:.4f})")
    del best_state

# =============================================================
# 6. 임베딩 추출
# =============================================================
print("\n📦 [Step 6] TGAT 임베딩 추출...")
model.eval()
all_emb = []
with torch.no_grad():
    for batch in tqdm(full_loader, desc="Embedding Extraction"):
        batch = batch.to(device)
        if batch.edge_attr is not None and batch.edge_attr.numel() > 0:
            batch_edge_attr = batch.edge_attr
        else:
            batch_edge_attr = torch.zeros(batch.edge_index.size(1), EDGE_RAW_DIM, device=device)
        emb = model.get_embedding(batch.x, batch.edge_index, batch_edge_attr)
        all_emb.append(emb[:batch.batch_size].cpu())

final_emb = torch.cat(all_emb, dim=0).numpy()
emb_cols = [f"tgat_emb_{i}" for i in range(HIDDEN_DIM)]

df_tgat = pl.concat([
    all_accounts.select("account_id"),
    pl.DataFrame(final_emb, schema=emb_cols)
], how="horizontal")

print(f"✅ 임베딩 shape: {final_emb.shape}")

del graph_data, all_emb, final_emb
torch.cuda.empty_cache()
gc.collect()

# =============================================================
# 6-A. ★ TGAT 임베딩 × PageRank interaction 피처 (NEW)
# =============================================================
print("\n🔗 [Step 6-A] TGAT × PageRank interaction 피처 생성...")

# PageRank를 TGAT 데이터프레임에 조인
df_tgat_with_pr = df_tgat.join(
    df_graph_feats.select(["account_id", "graph_pagerank"]),
    on="account_id", how="left"
).fill_null(0.0)

# TGAT 임베딩 중 variance가 높은 상위 N_INTERACTION_DIMS 차원 선택
emb_np = df_tgat_with_pr.select(emb_cols).to_numpy()
emb_var = np.var(emb_np, axis=0)
top_emb_indices = np.argsort(emb_var)[::-1][:N_INTERACTION_DIMS]
top_emb_cols = [emb_cols[i] for i in top_emb_indices]
print(f"  📊 Variance 상위 {N_INTERACTION_DIMS} 임베딩 차원: {top_emb_cols}")

# interaction: emb_i × pagerank
interaction_exprs = []
interaction_cols = []
for col_name in top_emb_cols:
    new_name = f"interact_pr_x_{col_name}"
    interaction_exprs.append(
        (pl.col(col_name) * pl.col("graph_pagerank") * 1e6).alias(new_name)  # PR 스케일링
    )
    interaction_cols.append(new_name)

# 추가: TGAT L2 norm × PageRank (임베딩 크기 자체가 정보)
interaction_exprs.append(
    (
        sum(pl.col(c) ** 2 for c in emb_cols).sqrt() * pl.col("graph_pagerank") * 1e6
    ).alias("interact_pr_x_emb_norm")
)
interaction_cols.append("interact_pr_x_emb_norm")

# 추가: TGAT 상위 차원 합 × PageRank
interaction_exprs.append(
    (
        sum(pl.col(c) for c in top_emb_cols) * pl.col("graph_pagerank") * 1e6
    ).alias("interact_pr_x_emb_topsum")
)
interaction_cols.append("interact_pr_x_emb_topsum")

df_interactions = df_tgat_with_pr.with_columns(interaction_exprs).select(
    ["account_id"] + interaction_cols
)

# inf/NaN 안전 처리
for ic in interaction_cols:
    df_interactions = df_interactions.with_columns(
        pl.when(pl.col(ic).is_infinite() | pl.col(ic).is_nan())
        .then(0.0)
        .otherwise(pl.col(ic))
        .alias(ic)
    )

print(f"  ✅ Interaction 피처 {len(interaction_cols)}개 생성")

del df_tgat_with_pr, emb_np
gc.collect()
log_memory("전체 피처 엔지니어링 완료")

# =============================================================
# 7. XGBoost 데이터 구성 (V2 + 그래프 + 조건부행동 + AML점수 + TGAT + Interaction)
# =============================================================
print("\n🚀 [Step 7] XGBoost 데이터 구성...")

xgb_exclude_cols = ["account_id", "time_group", "is_laundering", "mode_format", "currency_mode", "date"]
xgb_v2_cols = [c for c in df_v2.columns if c not in xgb_exclude_cols]
graph_feature_cols = [c for c in df_graph_feats.columns if c.startswith("graph_")]

xgb_feature_cols = (
    xgb_v2_cols
    + graph_feature_cols
    + cond_feature_cols
    + aml_score_cols
    + emb_cols
    + interaction_cols
)

print(f"📊 총 피처 수: {len(xgb_feature_cols)}")
print(f"  V2: {len(xgb_v2_cols)} | 그래프: {len(graph_feature_cols)} | "
      f"조건부행동: {len(cond_feature_cols)} | AML점수: {len(aml_score_cols)} | "
      f"TGAT: {len(emb_cols)} | Interaction: {len(interaction_cols)}")


def build_xgb_df(df_base, label=""):
    """V2 데이터에 모든 피처 테이블을 조인 + inf/NaN 안전 처리."""
    df = (
        df_base
        .join(df_graph_feats, on="account_id", how="left")
        .join(df_cond_features, on="account_id", how="left")
        .join(df_aml_scores, on="account_id", how="left")
        .join(df_tgat, on="account_id", how="left")
        .join(df_interactions, on="account_id", how="left")
        .fill_null(0.0)
        .fill_nan(0.0)
    )
    # inf → 0 치환 (Polars의 fill_nan은 NaN만 처리, inf는 별도)
    for col_name in xgb_feature_cols:
        if col_name in df.columns:
            df = df.with_columns(
                pl.when(pl.col(col_name).is_infinite())
                .then(0.0)
                .otherwise(pl.col(col_name))
                .alias(col_name)
            )
    return df


# [1] Train — 다운샘플링
df_train_full = build_xgb_df(df_v2.filter(pl.col("time_group") < train_cutoff))
df_train_pos = df_train_full.filter(pl.col("is_laundering") == 1)
df_train_neg = df_train_full.filter(pl.col("is_laundering") == 0)
n_neg_keep_xgb = min(len(df_train_pos) * NEG_ROW_RATIO_XGB, len(df_train_neg))
df_train_neg_sampled = df_train_neg.sample(n=n_neg_keep_xgb, seed=RANDOM_SEED)
df_train = pl.concat([df_train_pos, df_train_neg_sampled]).sample(fraction=1.0, seed=RANDOM_SEED)

X_train = df_train.select(xgb_feature_cols).to_pandas()
y_train = df_train["is_laundering"].to_numpy()
print(f"📊 Train: {len(df_train_full):,} → {len(df_train):,} (양성비율: {y_train.mean()*100:.2f}%)")
del df_train, df_train_full, df_train_pos, df_train_neg, df_train_neg_sampled; gc.collect()

# [2] Val
df_val = build_xgb_df(
    df_v2.filter((pl.col("time_group") >= train_cutoff) & (pl.col("time_group") < val_cutoff))
)
X_val = df_val.select(xgb_feature_cols).to_pandas()
y_val = df_val["is_laundering"].to_numpy()
print(f"📊 Val  : {len(X_val):,} (양성비율: {y_val.mean()*100:.2f}%)")
del df_val; gc.collect()

# [3] Test
df_test = build_xgb_df(df_v2.filter(pl.col("time_group") >= val_cutoff))
df_test_meta = df_test.select(["account_id", "time_group", "is_laundering"])
X_test = df_test.select(xgb_feature_cols).to_pandas()
y_test = df_test["is_laundering"].to_numpy()
print(f"📊 Test : {len(X_test):,} (양성비율: {y_test.mean()*100:.2f}%)")
del df_test, df_v2, df_tgat, df_graph_feats, df_cond_features, df_aml_scores, df_interactions
gc.collect()
log_memory("XGBoost 데이터 준비 완료")

# =============================================================
# 8. XGBoost 학습
# =============================================================
print("\n🔥 [Step 8] XGBoost 학습...")

actual_pos_ratio = y_train.sum() / len(y_train) if len(y_train) > 0 else 0.01
adjusted_spw = max((1 - actual_pos_ratio) / actual_pos_ratio, 1.0)
print(f"📊 scale_pos_weight: {adjusted_spw:.1f}")

xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "device": "cuda",
    "random_state": 42,
    "max_depth": 8,
    "learning_rate": 0.03,
    "scale_pos_weight": adjusted_spw,
    "subsample": 0.8,
    "colsample_bytree": 0.7,
    "min_child_weight": 5,
    "gamma": 0.1,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
}

model_xgb = xgb.XGBClassifier(**xgb_params, n_estimators=1500, early_stopping_rounds=100)
model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)

y_prob = model_xgb.predict_proba(X_test)[:, 1]

# Feature Importance
print("\n🔍 Feature Importance Top 25")
importance = model_xgb.feature_importances_
top_indices = np.argsort(importance)[::-1][:25]
for i, idx in enumerate(top_indices):
    tag = ""
    col = xgb_feature_cols[idx]
    if col.startswith("tgat_emb_"): tag = " [TGAT]"
    elif col.startswith("graph_"): tag = " [GRAPH]"
    elif col.startswith("cond_"): tag = " [COND]"
    elif col.startswith("score_"): tag = " [SCORE]"
    elif col.startswith("interact_"): tag = " [INTER]"
    print(f"  {i+1:2d}. {col:<40s}: {importance[idx]:.4f}{tag}")

# 그룹별 기여도
v2_imp = sum(importance[i] for i, c in enumerate(xgb_feature_cols) if c in xgb_v2_cols)
graph_imp = sum(importance[i] for i, c in enumerate(xgb_feature_cols) if c.startswith("graph_"))
cond_imp = sum(importance[i] for i, c in enumerate(xgb_feature_cols) if c.startswith("cond_"))
score_imp = sum(importance[i] for i, c in enumerate(xgb_feature_cols) if c.startswith("score_"))
tgat_imp = sum(importance[i] for i, c in enumerate(xgb_feature_cols) if c.startswith("tgat_emb_"))
inter_imp = sum(importance[i] for i, c in enumerate(xgb_feature_cols) if c.startswith("interact_"))
total_imp = importance.sum()

print(f"\n📊 피처 그룹별 기여도:")
print(f"  V2 피처       : {v2_imp/total_imp*100:5.1f}%")
print(f"  그래프 구조    : {graph_imp/total_imp*100:5.1f}%")
print(f"  구조×행동 조건부: {cond_imp/total_imp*100:5.1f}%")
print(f"  AML 패턴 점수  : {score_imp/total_imp*100:5.1f}%")
print(f"  TGAT 임베딩    : {tgat_imp/total_imp*100:5.1f}%")
print(f"  TGAT×PR 교차   : {inter_imp/total_imp*100:5.1f}%")

del X_train, X_val, X_test; gc.collect()

# =============================================================
# 9. 최종 평가 리포트
# =============================================================
print("\n" + "=" * 60)
print("🏆 [TGAT v4] 최종 성능 리포트")
print("=" * 60)

auprc = average_precision_score(y_test, y_prob)
print(f"\n  AUPRC: {auprc:.4f}")

best_f1 = 0
best_thresh = 0
for thresh in np.arange(0.05, 0.95, 0.01):
    f = f1_score(y_test, (y_prob >= thresh).astype(int), zero_division=0)
    if f > best_f1:
        best_f1 = f
        best_thresh = thresh

y_pred = (y_prob >= best_thresh).astype(int)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print(f"\n📌 [최적 임계값 결과]")
print(f"  - Threshold: {best_thresh:.2f}")
print(f"  - AUPRC    : {auprc:.4f}")
print(f"  - F1-Score : {best_f1:.4f}")
print(f"  - Precision: {precision:.4f}")
print(f"  - Recall   : {recall:.4f}")

print(f"\n📌 [주요 임계값별 성능]")
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    yp = (y_prob >= t).astype(int)
    print(f"  T={t:.1f} | P={precision_score(y_test,yp,zero_division=0):.4f} "
          f"R={recall_score(y_test,yp,zero_division=0):.4f} "
          f"F1={f1_score(y_test,yp,zero_division=0):.4f}")

# Top-K
df_res = df_test_meta.with_columns([
    pl.col("time_group").dt.date().alias("date"),
    pl.Series("pred_prob", y_prob)
])
df_distinct = df_res.sort("pred_prob", descending=True).unique(subset=["account_id"], maintain_order=True)

print(f"\n📌 [Top-K 탐지 (Distinct Account)]")
for k in [100, 500, 1000, 5000, 10000]:
    ck = min(k, len(df_distinct))
    if ck > 0:
        hits = df_distinct.head(ck)["is_laundering"].sum()
        print(f"  📍 Top-{k:5d} Hits: {hits:5d}명 | Precision: {(hits/ck)*100:6.2f}%")

print(f"\n📌 [일별 Top-100 탐지]")
unique_dates = df_distinct["date"].unique(maintain_order=True).sort(descending=True)
daily_stats = []
for d in unique_dates:
    hits = df_distinct.filter(pl.col("date") == d).head(100)["is_laundering"].sum()
    daily_stats.append({"Date": str(d), "Top-100 Hits": hits})
print(pl.DataFrame(daily_stats))

# v3 대비 비교
print(f"\n📌 [v3 → v4 성능 비교]")
print(f"  AUPRC    : 0.5311 → {auprc:.4f} ({'↑' if auprc > 0.5311 else '↓'} {abs(auprc-0.5311):.4f})")
print(f"  Best F1  : 0.4777 → {best_f1:.4f} ({'↑' if best_f1 > 0.4777 else '↓'} {abs(best_f1-0.4777):.4f})")

log_memory("최종 완료")
print("\n✅ TGAT v4 파이프라인 완료!")

🛡️ [TGAT v4] 구조×행동 피처 고도화 파이프라인

📂 데이터 로딩 중...
⏱️ Train cutoff: 2022-09-09 20:00:00
⏱️ Val   cutoff: 2022-09-14 05:00:00
📊 전체 노드 수: 2,076,999
  💾 [데이터 로드 완료] RAM: 34.14GB | GPU: 0.00GB

📐 [Step 2] 기본 그래프 구조 피처 계산...
  📊 PageRank 계산 중 (10 iterations)...
  ✅ 기본 그래프 피처: 13개

🔬 [Step 2-A] 구조-조건부 행동 피처 10개 계산...
  ✅ 구조-조건부 행동 피처 10개: ['cond_highdeg_night_intensity', 'cond_lowdeg_high_amount', 'cond_oneway_crossborder', 'cond_fanout_burst', 'cond_hub_risk_format', 'cond_asymmetric_wire', 'cond_irregular_interbank', 'cond_diverse_partner_currency', 'cond_bidir_amount_volatility', 'cond_multi_acct_burst']

🎯 [Step 2-B] AML 패턴 점수 5개 설계...
  ✅ AML 패턴 점수 5개: ['score_smurf', 'score_mule', 'score_layering', 'score_integration', 'score_composite']
  💾 [구조-행동 피처 + AML 점수 완료] RAM: 37.42GB | GPU: 0.00GB

🔗 [Step 3] GNN 엣지/노드 구성 중...
📊 Train 엣지: 19,060,343
📊 GNN 학습 노드: 양성 20,400 + 음성 1,020,000 = 1,040,400
  💾 [GNN 데이터 준비 완료] RAM: 38.58GB | GPU: 0.00GB

🧠 [Step 4] TGAT v3 모델 정의...

🔥 [Step 5] TGAT 학습 시작.

  → Epoch  1 | Loss: 1.3552 | LR: 0.000001


  → Epoch  5 | Loss: 1.1508 | LR: 0.000800


  → Epoch 10 | Loss: 1.0847 | LR: 0.000968


  → Epoch 15 | Loss: 1.0714 | LR: 0.000847


...

  → Epoch 30 | Loss: 1.0361 | LR: 0.000232


  → Epoch 35 | Loss: 1.0367 | LR: 0.000080


  → Epoch 40 | Loss: 1.0276 | LR: 0.000012
  ✅ Best model 복원 (loss: 1.0264)

📦 [Step 6] TGAT 임베딩 추출...


Embedding Extraction: 100%|█████████████████████████████████████████████████| 1015/1015 [00:15<00:00, 66.30it/s]


✅ 임베딩 shape: (2076999, 64)

🔗 [Step 6-A] TGAT × PageRank interaction 피처 생성...
  📊 Variance 상위 8 임베딩 차원: ['tgat_emb_21', 'tgat_emb_45', 'tgat_emb_51', 'tgat_emb_29', 'tgat_emb_39', 'tgat_emb_56', 'tgat_emb_0', 'tgat_emb_47']
  ✅ Interaction 피처 10개 생성
  💾 [전체 피처 엔지니어링 완료] RAM: 40.83GB | GPU: 0.02GB

🚀 [Step 7] XGBoost 데이터 구성...
📊 총 피처 수: 140
  V2: 38 | 그래프: 13 | 조건부행동: 10 | AML점수: 5 | TGAT: 64 | Interaction: 10
📊 Train: 17,172,747 → 619,437 (양성비율: 4.76%)
📊 Val  : 5,790,644 (양성비율: 0.30%)
📊 Test : 5,792,913 (양성비율: 0.36%)
  💾 [XGBoost 데이터 준비 완료] RAM: 57.47GB | GPU: 0.02GB

🔥 [Step 8] XGBoost 학습...
📊 scale_pos_weight: 20.0
[0]	validation_0-aucpr:0.11627
[100]	validation_0-aucpr:0.33651
[200]	validation_0-aucpr:0.41842
[300]	validation_0-aucpr:0.44234
[400]	validation_0-aucpr:0.45002
[500]	validation_0-aucpr:0.45367
[600]	validation_0-aucpr:0.45812
[700]	validation_0-aucpr:0.46151
[800]	validation_0-aucpr:0.46482
[900]	validation_0-aucpr:0.46803
[1000]	validation_0-aucpr:0.47114
[1100]	valida

In [4]:
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv
from torch_geometric.loader import NeighborLoader
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm
import gc
import numpy as np
import psutil
import os
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

def log_memory(tag=""):
    proc = psutil.Process(os.getpid())
    rss_gb = proc.memory_info().rss / (1024 ** 3)
    gpu_str = ""
    if torch.cuda.is_available():
        gpu_alloc = torch.cuda.memory_allocated() / (1024 ** 3)
        gpu_str = f" | GPU: {gpu_alloc:.2f}GB"
    print(f"  💾 [{tag}] RAM: {rss_gb:.2f}GB{gpu_str}")

print("=" * 60)
print("🛡️ [TGAT v4] 구조×행동 피처 고도화 파이프라인")
print("=" * 60)

# ===================== 설정 =====================
NEG_NODE_RATIO_GNN = 50
NEG_ROW_RATIO_XGB = 20
RANDOM_SEED = 42
HIDDEN_DIM = 64
EDGE_DIM = 16
N_HEADS = 4
N_EPOCHS_GNN = 40
BATCH_SIZE = 2048
NUM_NEIGHBORS = [15, 10]
N_INTERACTION_DIMS = 8   # TGAT×PageRank interaction 차원 수
# ================================================

raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"

# =============================================================
# 1. 데이터 로드 및 6:2:2 시간 분할
# =============================================================
print("\n📂 데이터 로딩 중...")
df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))

if df_raw["Timestamp"].dtype == pl.Utf8:
    df_raw = df_raw.with_columns(
        pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False)
    )

df_v2 = df_v2.sort("time_group")
total_count = len(df_v2)
train_idx = int(total_count * 0.6)
val_idx = int(total_count * 0.8)

train_cutoff = df_v2["time_group"][train_idx]
val_cutoff = df_v2["time_group"][val_idx]

print(f"⏱️ Train cutoff: {train_cutoff}")
print(f"⏱️ Val   cutoff: {val_cutoff}")

all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")

num_nodes = len(all_accounts)
print(f"📊 전체 노드 수: {num_nodes:,}")
log_memory("데이터 로드 완료")

# =============================================================
# 2. 기본 그래프 구조 피처 (v3 동일)
# =============================================================
print("\n📐 [Step 2] 기본 그래프 구조 피처 계산...")

df_raw_train = df_raw.filter(pl.col("Timestamp") < train_cutoff)
del df_raw; gc.collect()

# Degree
df_from = df_raw_train.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id"))
df_to = df_raw_train.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
out_degree = df_from.group_by("account_id").len().rename({"len": "graph_out_degree"})
in_degree = df_to.group_by("account_id").len().rename({"len": "graph_in_degree"})

# Unique partners
unique_out = (
    df_raw_train.select([
        pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
        pl.col("to_acc").cast(pl.Utf8).alias("partner")
    ]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_out_partners"))
)
unique_in = (
    df_raw_train.select([
        pl.col("to_acc").cast(pl.Utf8).alias("account_id"),
        pl.col("from_acc").cast(pl.Utf8).alias("partner")
    ]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_in_partners"))
)

# Bidirectional
edges_set_from = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("a"),
    pl.col("to_acc").cast(pl.Utf8).alias("b")
]).unique()
edges_set_to = edges_set_from.select([pl.col("b").alias("a"), pl.col("a").alias("b")])
bidirectional = edges_set_from.join(edges_set_to, on=["a", "b"], how="inner")
bidir_count = (
    bidirectional.select(pl.col("a").alias("account_id"))
    .group_by("account_id").len().rename({"len": "graph_bidir_count"})
)
del edges_set_from, edges_set_to, bidirectional; gc.collect()

# Time features
time_features = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("Timestamp")
]).group_by("account_id").agg([
    pl.col("Timestamp").sort().diff().dt.total_seconds().mean().alias("graph_avg_tx_interval"),
    pl.col("Timestamp").sort().diff().dt.total_seconds().std().alias("graph_std_tx_interval"),
    pl.col("Timestamp").count().alias("graph_total_tx_count"),
])

# PageRank
print("  📊 PageRank 계산 중 (10 iterations)...")
account_to_id = dict(zip(all_accounts["account_id"].to_list(), all_accounts["node_id"].to_list()))

src_ids = df_raw_train["from_acc"].cast(pl.Utf8).map_elements(
    lambda x: account_to_id.get(x, -1), return_dtype=pl.Int64
).to_numpy()
dst_ids = df_raw_train["to_acc"].cast(pl.Utf8).map_elements(
    lambda x: account_to_id.get(x, -1), return_dtype=pl.Int64
).to_numpy()

valid_mask = (src_ids >= 0) & (dst_ids >= 0)
src_valid = src_ids[valid_mask]
dst_valid = dst_ids[valid_mask]

out_deg_arr = np.zeros(num_nodes, dtype=np.float64)
np.add.at(out_deg_arr, src_valid, 1.0)
out_deg_arr = np.maximum(out_deg_arr, 1.0)

damping = 0.85
pr = np.ones(num_nodes, dtype=np.float64) / num_nodes
for _ in range(10):
    new_pr = np.ones(num_nodes, dtype=np.float64) * (1 - damping) / num_nodes
    contributions = pr[src_valid] / out_deg_arr[src_valid]
    np.add.at(new_pr, dst_valid, damping * contributions)
    pr = new_pr

pr_df = pl.DataFrame({
    "node_id": np.arange(num_nodes, dtype=np.uint32),
    "graph_pagerank": pr.astype(np.float32)
})
pr_df = all_accounts.join(pr_df, on="node_id", how="left").select(["account_id", "graph_pagerank"])

del src_ids, dst_ids, src_valid, dst_valid, out_deg_arr, pr, new_pr; gc.collect()

# 기본 그래프 피처 결합
df_graph_feats = (
    all_accounts.select("account_id")
    .join(out_degree, on="account_id", how="left")
    .join(in_degree, on="account_id", how="left")
    .join(unique_out, on="account_id", how="left")
    .join(unique_in, on="account_id", how="left")
    .join(bidir_count, on="account_id", how="left")
    .join(time_features, on="account_id", how="left")
    .join(pr_df, on="account_id", how="left")
    .fill_null(0.0)
)
df_graph_feats = df_graph_feats.with_columns([
    (pl.col("graph_out_degree") + pl.col("graph_in_degree")).alias("graph_total_degree"),
    (pl.col("graph_out_degree") / (pl.col("graph_in_degree") + 1)).alias("graph_out_in_ratio"),
    (pl.col("graph_bidir_count") / (pl.col("graph_out_degree") + 1)).alias("graph_reciprocity"),
    (pl.col("graph_unique_out_partners") / (pl.col("graph_out_degree") + 1)).alias("graph_partner_diversity"),
])

del out_degree, in_degree, unique_out, unique_in, bidir_count, time_features, pr_df, df_from, df_to
gc.collect()
print(f"  ✅ 기본 그래프 피처: {len([c for c in df_graph_feats.columns if c.startswith('graph_')])}개")

# =============================================================
# 2-A. ★ 구조-조건부 행동 피처 10개 (NEW)
# =============================================================
print("\n🔬 [Step 2-A] 구조-조건부 행동 피처 10개 계산...")

df_v2_train = df_v2.filter(pl.col("time_group") < train_cutoff)
exclude_cols = ["account_id", "time_group", "is_laundering", "mode_format", "currency_mode"]
gnn_feature_cols = [c for c in df_v2.columns if c not in exclude_cols]

df_v2_agg = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols
])

df_cond = df_v2_agg.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)
v2_cols_available = set(df_cond.columns)

def safe_col(name, default=0.0):
    return pl.col(name) if name in v2_cols_available else pl.lit(default)

cond_exprs = []

if "cnt_night" in v2_cols_available and "cnt_1h" in v2_cols_available:
    cond_exprs.append(
        (pl.col("graph_total_degree") * safe_col("cnt_night") / (safe_col("cnt_1h") + 1))
        .alias("cond_highdeg_night_intensity")
    )

if "avg_log_amount" in v2_cols_available:
    cond_exprs.append(
        (safe_col("avg_log_amount") / (pl.col("graph_total_degree").log1p() + 1))
        .alias("cond_lowdeg_high_amount")
    )

if "ratio_cross_border" in v2_cols_available:
    cond_exprs.append(
        (safe_col("ratio_cross_border") * (1 - pl.col("graph_reciprocity")))
        .alias("cond_oneway_crossborder")
    )

if "cnt_1h" in v2_cols_available:
    cond_exprs.append(
        (pl.col("graph_unique_out_partners") * safe_col("cnt_1h") / (pl.col("graph_out_degree") + 1))
        .alias("cond_fanout_burst")
    )

if "cnt_risk_format" in v2_cols_available:
    cond_exprs.append(
        (pl.col("graph_pagerank") * safe_col("cnt_risk_format"))
        .alias("cond_hub_risk_format")
    )

if "cnt_wire" in v2_cols_available:
    cond_exprs.append(
        (pl.col("graph_out_in_ratio") * safe_col("cnt_wire"))
        .alias("cond_asymmetric_wire")
    )

if "cnt_inter_bank" in v2_cols_available:
    cond_exprs.append(
        (pl.col("graph_std_tx_interval").fill_null(0) / (pl.col("graph_avg_tx_interval").fill_null(1) + 1)
         * safe_col("cnt_inter_bank"))
        .alias("cond_irregular_interbank")
    )

if "cnt_currencies" in v2_cols_available:
    cond_exprs.append(
        (pl.col("graph_partner_diversity") * safe_col("cnt_currencies"))
        .alias("cond_diverse_partner_currency")
    )

if "amount_kurtosis" in v2_cols_available:
    cond_exprs.append(
        (pl.col("graph_reciprocity") * safe_col("amount_kurtosis"))
        .alias("cond_bidir_amount_volatility")
    )

if "entity_acct_cnt" in v2_cols_available and "burst_ratio" in v2_cols_available:
    cond_exprs.append(
        (safe_col("entity_acct_cnt") * safe_col("burst_ratio"))
        .alias("cond_multi_acct_burst")
    )

df_cond_features = df_cond.select("account_id").with_columns(
    [df_cond[expr.meta.output_name()] if hasattr(expr, 'meta') else pl.lit(0.0) for expr in []]
)

df_cond = df_cond.with_columns(cond_exprs)
cond_feature_cols = [expr.meta.output_name() for expr in cond_exprs]

for cn in cond_feature_cols:
    df_cond = df_cond.with_columns(
        pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
        .then(0.0)
        .otherwise(pl.col(cn))
        .alias(cn)
    )

df_cond_features = df_cond.select(["account_id"] + cond_feature_cols)
print(f"  ✅ 구조-조건부 행동 피처 {len(cond_feature_cols)}개: {cond_feature_cols}")
del df_cond; gc.collect()

# =============================================================
# 2-B. ★ Smurf / Mule / Layering 점수 5개 (NEW)
# =============================================================
print("\n🎯 [Step 2-B] AML 패턴 점수 5개 설계...")

df_score_base = df_v2_agg.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)

score_exprs = []

score_exprs.append(
    (
        (safe_col("cnt_1h").fill_null(0).clip(0, 1e6) + 1).log1p()
        * pl.col("graph_unique_out_partners").clip(0, 1e6).log1p()
        * safe_col("burst_ratio").fill_null(0).clip(0, 1e6)
        / (safe_col("avg_log_amount").fill_null(1).clip(0, 1e6) + 1)
    ).alias("score_smurf")
)

score_exprs.append(
    (
        (1.0 / ((pl.col("graph_out_in_ratio").clip(0.01, 100) - 1.0).abs() + 0.1))
        * safe_col("cnt_wire").fill_null(0).clip(0, 1e6).log1p()
        * (1 - pl.col("graph_reciprocity").clip(0, 1))
        / (pl.col("graph_unique_out_partners").clip(0, 1e6).log1p() + 1)
    ).alias("score_mule")
)

score_exprs.append(
    (
        pl.col("graph_unique_out_partners").clip(0, 1e6).log1p()
        * safe_col("cnt_currencies").fill_null(0).clip(0, 1e6).log1p()
        * safe_col("ratio_cross_border").fill_null(0).clip(0, 1)
        / (pl.col("graph_avg_tx_interval").fill_null(3600).clip(0, 1e9) / 3600 + 1)
    ).alias("score_layering")
)

score_exprs.append(
    (
        pl.col("graph_in_degree").clip(0, 1e6).log1p()
        * (pl.col("graph_pagerank").clip(0, 1) * 1e6)
        * safe_col("cnt_risk_format").fill_null(0).clip(0, 1e6).log1p()
        / (pl.col("graph_out_degree").clip(0, 1e6).log1p() + 1)
    ).alias("score_integration")
)

df_score_base = df_score_base.with_columns(score_exprs)

score_names = ["score_smurf", "score_mule", "score_layering", "score_integration"]
for sn in score_names:
    df_score_base = df_score_base.with_columns(
        pl.when(pl.col(sn).is_infinite() | pl.col(sn).is_nan())
        .then(0.0)
        .otherwise(pl.col(sn))
        .alias(sn)
    )

for sn in score_names:
    col_min = df_score_base[sn].min()
    col_max = df_score_base[sn].max()
    col_range = col_max - col_min if (col_max - col_min) > 0 else 1.0
    df_score_base = df_score_base.with_columns(
        ((pl.col(sn) - col_min) / col_range).alias(f"{sn}_norm")
    )

df_score_base = df_score_base.with_columns(
    (
        pl.col("score_smurf_norm") * 0.3
        + pl.col("score_mule_norm") * 0.25
        + pl.col("score_layering_norm") * 0.3
        + pl.col("score_integration_norm") * 0.15
    ).alias("score_composite")
)

aml_score_cols = score_names + ["score_composite"]
df_aml_scores = df_score_base.select(["account_id"] + aml_score_cols)

print(f"  ✅ AML 패턴 점수 {len(aml_score_cols)}개: {aml_score_cols}")
del df_score_base, df_v2_agg; gc.collect()
log_memory("구조-행동 피처 + AML 점수 완료")

# =============================================================
# 3. GNN 엣지 + 노드 피처 구성
# =============================================================
print("\n🔗 [Step 3] GNN 엣지/노드 구성 중...")

df_edges_train = df_raw_train.select(["from_acc", "to_acc", "Timestamp"]).with_columns([
    pl.col("from_acc").cast(pl.Utf8),
    pl.col("to_acc").cast(pl.Utf8)
])
df_edges_train = df_edges_train.join(
    all_accounts.rename({"account_id": "from_acc", "node_id": "src_id"}), on="from_acc", how="left"
)
df_edges_train = df_edges_train.join(
    all_accounts.rename({"account_id": "to_acc", "node_id": "dst_id"}), on="to_acc", how="left"
)

timestamps_raw = df_edges_train["Timestamp"]
min_ts = timestamps_raw.min()
max_ts = timestamps_raw.max()
total_seconds = (max_ts - min_ts).total_seconds()

edge_time_normalized = (
    (timestamps_raw - min_ts).dt.total_microseconds().cast(pl.Float64) / 1e6 / max(total_seconds, 1.0)
).to_numpy().astype(np.float32)

hours = df_edges_train["Timestamp"].dt.hour().to_numpy().astype(np.float32)
dow = df_edges_train["Timestamp"].dt.weekday().to_numpy().astype(np.float32)

edge_features = np.stack([
    edge_time_normalized,
    np.sin(2 * np.pi * hours / 24),
    np.cos(2 * np.pi * hours / 24),
    np.sin(2 * np.pi * dow / 7),
    np.cos(2 * np.pi * dow / 7),
], axis=1)

edge_index_train = torch.tensor(
    df_edges_train.select(["src_id", "dst_id"]).to_numpy().T, dtype=torch.long
)
edge_attr_train = torch.tensor(edge_features, dtype=torch.float32)
EDGE_RAW_DIM = edge_attr_train.shape[1]
print(f"📊 Train 엣지: {edge_index_train.shape[1]:,}")

del df_raw_train, df_edges_train, timestamps_raw, hours, dow, edge_features, edge_time_normalized
gc.collect()

df_v2_node = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols
])
df_node_features = all_accounts.join(df_v2_node, on="account_id", how="left").fill_null(0.0)
X_node = torch.tensor(df_node_features.select(gnn_feature_cols).to_numpy(), dtype=torch.float32)
IN_DIM = X_node.shape[1]

target_labels = (
    df_v2_train.filter(pl.col("is_laundering") == 1)
    .select("account_id").unique()
    .with_columns(pl.lit(1).alias("label"))
)
df_labels = all_accounts.join(target_labels, on="account_id", how="left").fill_null(0)
Y_node = torch.tensor(df_labels["label"].to_numpy(), dtype=torch.long)

active_accounts = df_v2_train["account_id"].unique()
active_df = pl.DataFrame({"account_id": active_accounts, "is_active": True})
mask_df = all_accounts.join(active_df, on="account_id", how="left").fill_null(False)

pos_node_ids = set(df_labels.filter(pl.col("label") == 1)["node_id"].to_list())
active_node_ids = mask_df.filter(pl.col("is_active"))["node_id"].to_list()
active_pos = [nid for nid in active_node_ids if nid in pos_node_ids]
active_neg = [nid for nid in active_node_ids if nid not in pos_node_ids]

n_neg_keep_gnn = min(len(active_pos) * NEG_NODE_RATIO_GNN, len(active_neg))
np.random.seed(RANDOM_SEED)
sampled_neg = np.random.choice(active_neg, size=n_neg_keep_gnn, replace=False).tolist()
sampled_nodes = set(active_pos + sampled_neg)

train_mask_np = np.zeros(num_nodes, dtype=bool)
for nid in sampled_nodes:
    train_mask_np[nid] = True
train_mask = torch.tensor(train_mask_np, dtype=torch.bool)

print(f"📊 GNN 학습 노드: 양성 {len(active_pos):,} + 음성 {n_neg_keep_gnn:,} = {len(sampled_nodes):,}")

del df_v2_node, df_node_features, mask_df, active_df
gc.collect()
log_memory("GNN 데이터 준비 완료")

# =============================================================
# 4. TGAT v3 모델
# =============================================================
print("\n🧠 [Step 4] TGAT v3 모델 정의...")


class EdgeProjection(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, out_dim), nn.GELU(), nn.Linear(out_dim, out_dim)
        )

    def forward(self, edge_attr):
        return self.proj(edge_attr)


class TGAT_v3(nn.Module):
    def __init__(self, in_dim, hidden_dim, edge_raw_dim, edge_dim, n_heads, dropout=0.2):
        super().__init__()
        self.edge_proj = EdgeProjection(edge_raw_dim, edge_dim)
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.conv1 = TransformerConv(
            hidden_dim, hidden_dim // n_heads, heads=n_heads,
            edge_dim=edge_dim, dropout=dropout, concat=True
        )
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.conv2 = TransformerConv(
            hidden_dim, hidden_dim // n_heads, heads=n_heads,
            edge_dim=edge_dim, dropout=dropout, concat=True
        )
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim // 2, 1)
        )
        self.dropout = dropout

    def forward(self, x, edge_index, edge_attr):
        edge_emb = self.edge_proj(edge_attr)
        h = self.input_proj(x)
        h = h + F.dropout(self.conv1(self.norm1(h), edge_index, edge_emb), p=self.dropout, training=self.training)
        h = h + F.dropout(self.conv2(self.norm2(h), edge_index, edge_emb), p=self.dropout, training=self.training)
        h = self.norm_out(h)
        return self.classifier(h).squeeze(-1)

    @torch.no_grad()
    def get_embedding(self, x, edge_index, edge_attr):
        edge_emb = self.edge_proj(edge_attr)
        h = self.input_proj(x)
        h = h + self.conv1(self.norm1(h), edge_index, edge_emb)
        h = h + self.conv2(self.norm2(h), edge_index, edge_emb)
        return self.norm_out(h)


# =============================================================
# 5. GNN 학습
# =============================================================
print("\n🔥 [Step 5] TGAT 학습 시작...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"📌 디바이스: {device}")

graph_data = Data(x=X_node, edge_index=edge_index_train, edge_attr=edge_attr_train, y=Y_node)
graph_data.train_mask = train_mask
graph_data.num_nodes = num_nodes

del X_node, edge_index_train, edge_attr_train, Y_node, train_mask
gc.collect()

train_loader = NeighborLoader(
    graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
    input_nodes=graph_data.train_mask, shuffle=True, num_workers=4
)
full_loader = NeighborLoader(
    graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
    input_nodes=None, shuffle=False, num_workers=4
)

model = TGAT_v3(
    in_dim=IN_DIM, hidden_dim=HIDDEN_DIM, edge_raw_dim=EDGE_RAW_DIM,
    edge_dim=EDGE_DIM, n_heads=N_HEADS, dropout=0.2
).to(device)

pw = (len(sampled_nodes) - len(active_pos)) / max(len(active_pos), 1)
print(f"📊 pos_weight: {pw:.1f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw], device=device))
BASE_LR = 0.001
WARMUP_EPOCHS = 5
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6, weight_decay=1e-4)

def set_lr(optimizer, epoch):
    if epoch < WARMUP_EPOCHS:
        lr = 1e-6 + (BASE_LR - 1e-6) * (epoch / WARMUP_EPOCHS)
    else:
        progress = (epoch - WARMUP_EPOCHS) / max(N_EPOCHS_GNN - WARMUP_EPOCHS, 1)
        lr = 1e-5 + (BASE_LR - 1e-5) * 0.5 * (1 + np.cos(np.pi * progress))
    for pg in optimizer.param_groups:
        pg['lr'] = lr
    return lr

model.train()
best_loss = float('inf')
patience_counter = 0
PATIENCE = 10
best_state = None

for epoch in range(N_EPOCHS_GNN):
    current_lr = set_lr(optimizer, epoch)
    total_loss = 0.0
    num_batches = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{N_EPOCHS_GNN}", leave=False):
        batch = batch.to(device)
        optimizer.zero_grad()
        if batch.edge_attr is not None and batch.edge_attr.numel() > 0:
            batch_edge_attr = batch.edge_attr
        else:
            batch_edge_attr = torch.zeros(batch.edge_index.size(1), EDGE_RAW_DIM, device=device)

        out = model(batch.x, batch.edge_index, batch_edge_attr)
        loss = criterion(out[:batch.batch_size], batch.y[:batch.batch_size].float())

        if torch.isnan(loss) or torch.isinf(loss):
            optimizer.zero_grad()
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        optimizer.step()
        total_loss += loss.item()
        num_batches += 1

    avg_loss = total_loss / max(num_batches, 1)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  → Epoch {epoch+1:2d} | Loss: {avg_loss:.4f} | LR: {current_lr:.6f}")

    if avg_loss < best_loss - 0.0005:
        best_loss = avg_loss
        patience_counter = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1
    if patience_counter >= PATIENCE:
        print(f"  ⏹️ Early stopping at epoch {epoch+1}")
        break

if best_state is not None:
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    print(f"  ✅ Best model 복원 (loss: {best_loss:.4f})")
    del best_state

# =============================================================
# 6. 임베딩 추출
# =============================================================
print("\n📦 [Step 6] TGAT 임베딩 추출...")
model.eval()
all_emb = []
with torch.no_grad():
    for batch in tqdm(full_loader, desc="Embedding Extraction"):
        batch = batch.to(device)
        if batch.edge_attr is not None and batch.edge_attr.numel() > 0:
            batch_edge_attr = batch.edge_attr
        else:
            batch_edge_attr = torch.zeros(batch.edge_index.size(1), EDGE_RAW_DIM, device=device)
        emb = model.get_embedding(batch.x, batch.edge_index, batch_edge_attr)
        all_emb.append(emb[:batch.batch_size].cpu())

final_emb = torch.cat(all_emb, dim=0).numpy()
emb_cols = [f"tgat_emb_{i}" for i in range(HIDDEN_DIM)]

df_tgat = pl.concat([
    all_accounts.select("account_id"),
    pl.DataFrame(final_emb, schema=emb_cols)
], how="horizontal")

print(f"✅ 임베딩 shape: {final_emb.shape}")

del graph_data, all_emb, final_emb
torch.cuda.empty_cache()
gc.collect()

# =============================================================
# 6-A. ★ TGAT 임베딩 × PageRank interaction 피처 (NEW)
# =============================================================
print("\n🔗 [Step 6-A] TGAT × PageRank interaction 피처 생성...")

df_tgat_with_pr = df_tgat.join(
    df_graph_feats.select(["account_id", "graph_pagerank"]),
    on="account_id", how="left"
).fill_null(0.0)

emb_np = df_tgat_with_pr.select(emb_cols).to_numpy()
emb_var = np.var(emb_np, axis=0)
top_emb_indices = np.argsort(emb_var)[::-1][:N_INTERACTION_DIMS]
top_emb_cols = [emb_cols[i] for i in top_emb_indices]
print(f"  📊 Variance 상위 {N_INTERACTION_DIMS} 임베딩 차원: {top_emb_cols}")

interaction_exprs = []
interaction_cols = []
for col_name in top_emb_cols:
    new_name = f"interact_pr_x_{col_name}"
    interaction_exprs.append(
        (pl.col(col_name) * pl.col("graph_pagerank") * 1e6).alias(new_name)
    )
    interaction_cols.append(new_name)

interaction_exprs.append(
    (
        sum(pl.col(c) ** 2 for c in emb_cols).sqrt() * pl.col("graph_pagerank") * 1e6
    ).alias("interact_pr_x_emb_norm")
)
interaction_cols.append("interact_pr_x_emb_norm")

interaction_exprs.append(
    (
        sum(pl.col(c) for c in top_emb_cols) * pl.col("graph_pagerank") * 1e6
    ).alias("interact_pr_x_emb_topsum")
)
interaction_cols.append("interact_pr_x_emb_topsum")

df_interactions = df_tgat_with_pr.with_columns(interaction_exprs).select(
    ["account_id"] + interaction_cols
)

for ic in interaction_cols:
    df_interactions = df_interactions.with_columns(
        pl.when(pl.col(ic).is_infinite() | pl.col(ic).is_nan())
        .then(0.0)
        .otherwise(pl.col(ic))
        .alias(ic)
    )

print(f"  ✅ Interaction 피처 {len(interaction_cols)}개 생성")

del df_tgat_with_pr, emb_np
gc.collect()
log_memory("전체 피처 엔지니어링 완료")

# =============================================================
# 7. XGBoost 데이터 구성
# =============================================================
print("\n🚀 [Step 7] XGBoost 데이터 구성...")

xgb_exclude_cols = ["account_id", "time_group", "is_laundering", "mode_format", "currency_mode", "date"]
xgb_v2_cols = [c for c in df_v2.columns if c not in xgb_exclude_cols]
graph_feature_cols = [c for c in df_graph_feats.columns if c.startswith("graph_")]

xgb_feature_cols = (
    xgb_v2_cols
    + graph_feature_cols
    + cond_feature_cols
    + aml_score_cols
    + emb_cols
    + interaction_cols
)

print(f"📊 총 피처 수: {len(xgb_feature_cols)}")
print(f"  V2: {len(xgb_v2_cols)} | 그래프: {len(graph_feature_cols)} | "
      f"조건부행동: {len(cond_feature_cols)} | AML점수: {len(aml_score_cols)} | "
      f"TGAT: {len(emb_cols)} | Interaction: {len(interaction_cols)}")


def build_xgb_df(df_base, label=""):
    df = (
        df_base
        .join(df_graph_feats, on="account_id", how="left")
        .join(df_cond_features, on="account_id", how="left")
        .join(df_aml_scores, on="account_id", how="left")
        .join(df_tgat, on="account_id", how="left")
        .join(df_interactions, on="account_id", how="left")
        .fill_null(0.0)
        .fill_nan(0.0)
    )
    for col_name in xgb_feature_cols:
        if col_name in df.columns:
            df = df.with_columns(
                pl.when(pl.col(col_name).is_infinite())
                .then(0.0)
                .otherwise(pl.col(col_name))
                .alias(col_name)
            )
    return df


# [1] Train
df_train_full = build_xgb_df(df_v2.filter(pl.col("time_group") < train_cutoff))
df_train_pos = df_train_full.filter(pl.col("is_laundering") == 1)
df_train_neg = df_train_full.filter(pl.col("is_laundering") == 0)
n_neg_keep_xgb = min(len(df_train_pos) * NEG_ROW_RATIO_XGB, len(df_train_neg))
df_train_neg_sampled = df_train_neg.sample(n=n_neg_keep_xgb, seed=RANDOM_SEED)
df_train = pl.concat([df_train_pos, df_train_neg_sampled]).sample(fraction=1.0, seed=RANDOM_SEED)

X_train = df_train.select(xgb_feature_cols).to_pandas()
y_train = df_train["is_laundering"].to_numpy()
print(f"📊 Train: {len(df_train_full):,} → {len(df_train):,} (양성비율: {y_train.mean()*100:.2f}%)")
del df_train, df_train_full, df_train_pos, df_train_neg, df_train_neg_sampled; gc.collect()

# [2] Val
df_val = build_xgb_df(
    df_v2.filter((pl.col("time_group") >= train_cutoff) & (pl.col("time_group") < val_cutoff))
)
X_val = df_val.select(xgb_feature_cols).to_pandas()
y_val = df_val["is_laundering"].to_numpy()
print(f"📊 Val  : {len(X_val):,} (양성비율: {y_val.mean()*100:.2f}%)")
del df_val; gc.collect()

# [3] Test — ★ 미탐/오탐 분석을 위해 피처 DataFrame을 유지
df_test_full = build_xgb_df(df_v2.filter(pl.col("time_group") >= val_cutoff))
df_test_meta = df_test_full.select(["account_id", "time_group", "is_laundering"])
X_test = df_test_full.select(xgb_feature_cols).to_pandas()
y_test = df_test_full["is_laundering"].to_numpy()
print(f"📊 Test : {len(X_test):,} (양성비율: {y_test.mean()*100:.2f}%)")

# ★ 분석에 쓸 Polars DF 보관 (원본 대비 변경: del 하지 않음)
df_test_with_features = df_test_full  # 피처 + 메타 모두 포함
del df_test_full
del df_v2, df_tgat, df_graph_feats, df_cond_features, df_aml_scores, df_interactions
gc.collect()
log_memory("XGBoost 데이터 준비 완료")

# =============================================================
# 8. XGBoost 학습
# =============================================================
print("\n🔥 [Step 8] XGBoost 학습...")

actual_pos_ratio = y_train.sum() / len(y_train) if len(y_train) > 0 else 0.01
adjusted_spw = max((1 - actual_pos_ratio) / actual_pos_ratio, 1.0)
print(f"📊 scale_pos_weight: {adjusted_spw:.1f}")

xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "device": "cuda",
    "random_state": 42,
    "max_depth": 8,
    "learning_rate": 0.03,
    "scale_pos_weight": adjusted_spw,
    "subsample": 0.8,
    "colsample_bytree": 0.7,
    "min_child_weight": 5,
    "gamma": 0.1,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
}

model_xgb = xgb.XGBClassifier(**xgb_params, n_estimators=1500, early_stopping_rounds=100)
model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)

y_prob = model_xgb.predict_proba(X_test)[:, 1]

# Feature Importance
print("\n🔍 Feature Importance Top 25")
importance = model_xgb.feature_importances_
top_indices = np.argsort(importance)[::-1][:25]
for i, idx in enumerate(top_indices):
    tag = ""
    col = xgb_feature_cols[idx]
    if col.startswith("tgat_emb_"): tag = " [TGAT]"
    elif col.startswith("graph_"): tag = " [GRAPH]"
    elif col.startswith("cond_"): tag = " [COND]"
    elif col.startswith("score_"): tag = " [SCORE]"
    elif col.startswith("interact_"): tag = " [INTER]"
    print(f"  {i+1:2d}. {col:<40s}: {importance[idx]:.4f}{tag}")

v2_imp = sum(importance[i] for i, c in enumerate(xgb_feature_cols) if c in xgb_v2_cols)
graph_imp = sum(importance[i] for i, c in enumerate(xgb_feature_cols) if c.startswith("graph_"))
cond_imp = sum(importance[i] for i, c in enumerate(xgb_feature_cols) if c.startswith("cond_"))
score_imp = sum(importance[i] for i, c in enumerate(xgb_feature_cols) if c.startswith("score_"))
tgat_imp = sum(importance[i] for i, c in enumerate(xgb_feature_cols) if c.startswith("tgat_emb_"))
inter_imp = sum(importance[i] for i, c in enumerate(xgb_feature_cols) if c.startswith("interact_"))
total_imp = importance.sum()

print(f"\n📊 피처 그룹별 기여도:")
print(f"  V2 피처       : {v2_imp/total_imp*100:5.1f}%")
print(f"  그래프 구조    : {graph_imp/total_imp*100:5.1f}%")
print(f"  구조×행동 조건부: {cond_imp/total_imp*100:5.1f}%")
print(f"  AML 패턴 점수  : {score_imp/total_imp*100:5.1f}%")
print(f"  TGAT 임베딩    : {tgat_imp/total_imp*100:5.1f}%")
print(f"  TGAT×PR 교차   : {inter_imp/total_imp*100:5.1f}%")

del X_train, X_val; gc.collect()
# ★ X_test는 분석에 사용하므로 del 하지 않음

# =============================================================
# 9. 최종 평가 리포트
# =============================================================
print("\n" + "=" * 60)
print("🏆 [TGAT v4] 최종 성능 리포트")
print("=" * 60)

auprc = average_precision_score(y_test, y_prob)
print(f"\n  AUPRC: {auprc:.4f}")

best_f1 = 0
best_thresh = 0
for thresh in np.arange(0.05, 0.95, 0.01):
    f = f1_score(y_test, (y_prob >= thresh).astype(int), zero_division=0)
    if f > best_f1:
        best_f1 = f
        best_thresh = thresh

y_pred = (y_prob >= best_thresh).astype(int)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print(f"\n📌 [최적 임계값 결과]")
print(f"  - Threshold: {best_thresh:.2f}")
print(f"  - AUPRC    : {auprc:.4f}")
print(f"  - F1-Score : {best_f1:.4f}")
print(f"  - Precision: {precision:.4f}")
print(f"  - Recall   : {recall:.4f}")

print(f"\n📌 [주요 임계값별 성능]")
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    yp = (y_prob >= t).astype(int)
    print(f"  T={t:.1f} | P={precision_score(y_test,yp,zero_division=0):.4f} "
          f"R={recall_score(y_test,yp,zero_division=0):.4f} "
          f"F1={f1_score(y_test,yp,zero_division=0):.4f}")

# Top-K
df_res = df_test_meta.with_columns([
    pl.col("time_group").dt.date().alias("date"),
    pl.Series("pred_prob", y_prob)
])
df_distinct = df_res.sort("pred_prob", descending=True).unique(subset=["account_id"], maintain_order=True)

print(f"\n📌 [Top-K 탐지 (Distinct Account)]")
for k in [100, 500, 1000, 5000, 10000]:
    ck = min(k, len(df_distinct))
    if ck > 0:
        hits = df_distinct.head(ck)["is_laundering"].sum()
        print(f"  📍 Top-{k:5d} Hits: {hits:5d}명 | Precision: {(hits/ck)*100:6.2f}%")

print(f"\n📌 [일별 Top-100 탐지]")
unique_dates = df_distinct["date"].unique(maintain_order=True).sort(descending=True)
daily_stats = []
for d in unique_dates:
    hits = df_distinct.filter(pl.col("date") == d).head(100)["is_laundering"].sum()
    daily_stats.append({"Date": str(d), "Top-100 Hits": hits})
print(pl.DataFrame(daily_stats))

print(f"\n📌 [v3 → v4 성능 비교]")
print(f"  AUPRC    : 0.5311 → {auprc:.4f} ({'↑' if auprc > 0.5311 else '↓'} {abs(auprc-0.5311):.4f})")
print(f"  Best F1  : 0.4777 → {best_f1:.4f} ({'↑' if best_f1 > 0.4777 else '↓'} {abs(best_f1-0.4777):.4f})")

log_memory("최종 완료")
print("\n✅ TGAT v4 파이프라인 완료!")


# #############################################################
# #############################################################
# ##                                                         ##
# ##   Step 10. 미탐(FN) / 오탐(FP) 심층 분석                ##
# ##                                                         ##
# #############################################################
# #############################################################

print("\n" + "=" * 60)
print("🔍 [Step 10] 미탐(FN) / 오탐(FP) 심층 분석 시작")
print("=" * 60)

# ----------------------------------------------------------
# 10-0. 분석용 통합 DataFrame 구성
# ----------------------------------------------------------
# df_test_with_features 에 pred_prob 추가
df_analysis = df_test_with_features.with_columns(
    pl.Series("pred_prob", y_prob)
)

# 피처 그룹 정의 (이후 분석에서 반복 사용)
feature_groups = {
    "V2 행동":     [c for c in xgb_v2_cols if c in df_analysis.columns],
    "그래프 구조":  [c for c in xgb_feature_cols if c.startswith("graph_") and c in df_analysis.columns],
    "조건부행동":   [c for c in xgb_feature_cols if c.startswith("cond_") and c in df_analysis.columns],
    "AML 점수":    [c for c in xgb_feature_cols if c.startswith("score_") and c in df_analysis.columns],
    "TGAT 임베딩":  [c for c in xgb_feature_cols if c.startswith("tgat_emb_") and c in df_analysis.columns],
    "Interaction":  [c for c in xgb_feature_cols if c.startswith("interact_") and c in df_analysis.columns],
}

THRESH = best_thresh  # 최적 F1 임계값 사용

# 4분류 컬럼 추가
df_analysis = df_analysis.with_columns(
    pl.when((pl.col("is_laundering") == 1) & (pl.col("pred_prob") >= THRESH))
      .then(pl.lit("TP"))
    .when((pl.col("is_laundering") == 1) & (pl.col("pred_prob") < THRESH))
      .then(pl.lit("FN"))
    .when((pl.col("is_laundering") == 0) & (pl.col("pred_prob") >= THRESH))
      .then(pl.lit("FP"))
    .otherwise(pl.lit("TN"))
    .alias("pred_class")
)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print(f"\n📊 Confusion Matrix (Threshold = {THRESH:.2f})")
print(f"  TP (정탐) : {tp:,}")
print(f"  FP (오탐) : {fp:,}")
print(f"  FN (미탐) : {fn:,}")
print(f"  TN (정상) : {tn:,}")

# ----------------------------------------------------------
# 10-1. 미탐(FN) 분석
# ----------------------------------------------------------
print("\n" + "-" * 60)
print("🔴 [10-1] 미탐(FN) 심층 분석")
print("-" * 60)

df_fn = df_analysis.filter(pl.col("pred_class") == "FN")
df_tp = df_analysis.filter(pl.col("pred_class") == "TP")

print(f"\n  총 미탐 건수: {len(df_fn):,}")
if len(df_fn) > 0:
    fn_probs = df_fn["pred_prob"].to_numpy()

    # (A) 미탐 예측 확률 분포
    print(f"\n  📈 미탐 예측 확률 분포:")
    print(f"     Mean: {np.mean(fn_probs):.4f} | Median: {np.median(fn_probs):.4f} | "
          f"Std: {np.std(fn_probs):.4f} | Max: {np.max(fn_probs):.4f}")

    bins = [0, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0]
    print(f"\n  📊 미탐 확률 구간별 분포:")
    for i in range(len(bins) - 1):
        cnt = int(((fn_probs >= bins[i]) & (fn_probs < bins[i+1])).sum())
        pct = cnt / len(fn_probs) * 100
        bar = "█" * int(pct / 2)
        print(f"     [{bins[i]:.2f}, {bins[i+1]:.2f}) : {cnt:6d} ({pct:5.1f}%) {bar}")

    # (B) 미탐 유형 분류
    n_hard   = int((fn_probs < 0.05).sum())
    n_medium = int(((fn_probs >= 0.05) & (fn_probs < THRESH - 0.1)).sum())
    n_near   = int((fn_probs >= THRESH - 0.1).sum())
    print(f"\n  🔬 미탐 유형 분류:")
    print(f"     Hard FN    (prob < 0.05)          : {n_hard:,}건 — 모델이 전혀 감지 못함")
    print(f"     Medium FN  (0.05 ~ {THRESH-0.1:.2f})        : {n_medium:,}건 — 부분 감지, threshold 미달")
    print(f"     Near-miss  (prob >= {THRESH-0.1:.2f})        : {n_near:,}건 — 거의 탐지 성공")

    # (C) 피처 그룹별 FN vs TP 비교
    if len(df_tp) > 0:
        print(f"\n  📊 피처 그룹별 FN vs TP 평균 비교 (차이 큰 상위 5개):")
        for grp_name, cols in feature_groups.items():
            if not cols:
                continue
            fn_means = df_fn.select(cols).mean().to_numpy().flatten()
            tp_means = df_tp.select(cols).mean().to_numpy().flatten()
            diffs = fn_means - tp_means
            abs_diffs = np.abs(diffs)

            print(f"\n     [{grp_name}] (피처 {len(cols)}개, 평균|차이|: {np.mean(abs_diffs):.4f})")
            top_n = min(5, len(cols))
            top_idx = np.argsort(abs_diffs)[::-1][:top_n]
            for idx in top_idx:
                c = cols[idx]
                d = "↑" if diffs[idx] > 0 else "↓"
                print(f"       {c:<40s}: FN={fn_means[idx]:10.4f}  TP={tp_means[idx]:10.4f}  ({d}{abs_diffs[idx]:.4f})")

    # (D) AML 점수별 FN vs TP
    aml_cols_in_df = [c for c in ["score_smurf", "score_mule", "score_layering",
                                    "score_integration", "score_composite"] if c in df_analysis.columns]
    if aml_cols_in_df and len(df_tp) > 0:
        print(f"\n  🎯 AML 패턴 점수 — FN vs TP:")
        for sc in aml_cols_in_df:
            fn_v = df_fn[sc].mean()
            tp_v = df_tp[sc].mean()
            print(f"     {sc:<25s}: FN={fn_v:.4f}  TP={tp_v:.4f}  (차이: {fn_v - tp_v:+.4f})")

    # (E) 시간대별 미탐
    if "time_group" in df_fn.columns:
        print(f"\n  📅 미탐 상위 날짜 (건수 기준):")
        fn_by_date = (
            df_fn.with_columns(pl.col("time_group").dt.date().alias("date"))
            .group_by("date").len().sort("len", descending=True)
        )
        for row in fn_by_date.head(10).iter_rows(named=True):
            print(f"     {row['date']} : {row['len']:5d}건")


# ----------------------------------------------------------
# 10-2. 오탐(FP) 분석
# ----------------------------------------------------------
print("\n" + "-" * 60)
print("🟡 [10-2] 오탐(FP) 심층 분석")
print("-" * 60)

df_fp = df_analysis.filter(pl.col("pred_class") == "FP")
df_tn = df_analysis.filter(pl.col("pred_class") == "TN")

print(f"\n  총 오탐 건수: {len(df_fp):,}")
if len(df_fp) > 0:
    fp_probs = df_fp["pred_prob"].to_numpy()

    # (A) 오탐 예측 확률 분포
    print(f"\n  📈 오탐 예측 확률 분포:")
    print(f"     Mean: {np.mean(fp_probs):.4f} | Median: {np.median(fp_probs):.4f} | "
          f"Std: {np.std(fp_probs):.4f} | Max: {np.max(fp_probs):.4f}")

    bins_fp = [THRESH, THRESH + 0.1, THRESH + 0.2, 0.5, 0.7, 0.9, 1.01]
    # 중복 제거 및 정렬
    bins_fp = sorted(set([b for b in bins_fp if b >= THRESH]))
    if bins_fp[-1] <= 1.0:
        bins_fp.append(1.01)
    print(f"\n  📊 오탐 확률 구간별 분포:")
    for i in range(len(bins_fp) - 1):
        cnt = int(((fp_probs >= bins_fp[i]) & (fp_probs < bins_fp[i+1])).sum())
        pct = cnt / len(fp_probs) * 100 if len(fp_probs) > 0 else 0
        bar = "█" * int(pct / 2)
        print(f"     [{bins_fp[i]:.2f}, {bins_fp[i+1]:.2f}) : {cnt:6d} ({pct:5.1f}%) {bar}")

    # (B) 오탐 유형 분류
    n_border = int((fp_probs < THRESH + 0.1).sum())
    n_mid    = int(((fp_probs >= THRESH + 0.1) & (fp_probs < 0.8)).sum())
    n_high   = int((fp_probs >= 0.8).sum())
    print(f"\n  🔬 오탐 유형 분류:")
    print(f"     경계선 FP  (prob < {THRESH+0.1:.2f})      : {n_border:,}건 — threshold 약간 초과")
    print(f"     중간 확신  ({THRESH+0.1:.2f} ~ 0.80)       : {n_mid:,}건")
    print(f"     고확신 FP  (prob >= 0.80)          : {n_high:,}건 — 모델이 강하게 양성 판단")

    # (C) 피처 그룹별 FP vs TN 비교
    if len(df_tn) > 0:
        print(f"\n  📊 피처 그룹별 FP vs TN 평균 비교 (차이 큰 상위 5개):")
        for grp_name, cols in feature_groups.items():
            if not cols:
                continue
            fp_means = df_fp.select(cols).mean().to_numpy().flatten()
            tn_means = df_tn.select(cols).mean().to_numpy().flatten()
            diffs = fp_means - tn_means
            abs_diffs = np.abs(diffs)

            print(f"\n     [{grp_name}] (피처 {len(cols)}개, 평균|차이|: {np.mean(abs_diffs):.4f})")
            top_n = min(5, len(cols))
            top_idx = np.argsort(abs_diffs)[::-1][:top_n]
            for idx in top_idx:
                c = cols[idx]
                d = "↑" if diffs[idx] > 0 else "↓"
                print(f"       {c:<40s}: FP={fp_means[idx]:10.4f}  TN={tn_means[idx]:10.4f}  ({d}{abs_diffs[idx]:.4f})")

    # (D) 오탐 vs 정탐(TP) 유사도 — 왜 양성으로 오판했는지
    if len(df_tp) > 0 and aml_cols_in_df:
        print(f"\n  🔍 오탐이 정탐(TP)과 얼마나 유사한가? (AML 점수 비교):")
        for sc in aml_cols_in_df:
            fp_v = df_fp[sc].mean()
            tp_v = df_tp[sc].mean()
            sim = 1 - abs(fp_v - tp_v) / (abs(tp_v) + 1e-8)
            print(f"     {sc:<25s}: FP={fp_v:.4f}  TP={tp_v:.4f}  유사도={sim:.2%}")

    # (E) 고확신 오탐 상세
    if n_high > 0:
        print(f"\n  🚨 고확신 오탐 (prob >= 0.80) 상위 20건:")
        df_high_fp = df_fp.filter(pl.col("pred_prob") >= 0.8).sort("pred_prob", descending=True)
        show_cols = ["account_id", "pred_prob"] + [c for c in aml_cols_in_df if c in df_high_fp.columns]
        graph_show = [c for c in ["graph_total_degree", "graph_pagerank", "graph_out_in_ratio",
                                   "graph_reciprocity"] if c in df_high_fp.columns]
        show_cols += graph_show
        print(df_high_fp.head(20).select(show_cols))

    # (F) 시간대별 오탐
    if "time_group" in df_fp.columns:
        print(f"\n  📅 오탐 상위 날짜 (건수 기준):")
        fp_by_date = (
            df_fp.with_columns(pl.col("time_group").dt.date().alias("date"))
            .group_by("date").len().sort("len", descending=True)
        )
        for row in fp_by_date.head(10).iter_rows(named=True):
            print(f"     {row['date']} : {row['len']:5d}건")


# ----------------------------------------------------------
# 10-3. 임계값 민감도 분석
# ----------------------------------------------------------
print("\n" + "-" * 60)
print("📐 [10-3] 임계값 민감도 분석 (FN/FP 트레이드오프)")
print("-" * 60)

print(f"\n  {'Thresh':>7s} | {'TP':>6s} {'FP':>6s} {'FN':>6s} {'TN':>7s} | "
      f"{'Prec':>6s} {'Recall':>6s} {'F1':>6s} | {'FP/TP':>6s} {'FN率':>6s}")
print("  " + "-" * 85)

for t in np.arange(0.05, 0.96, 0.05):
    yp = (y_prob >= t).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, yp).ravel()
    p_t = precision_score(y_test, yp, zero_division=0)
    r_t = recall_score(y_test, yp, zero_division=0)
    f1_t = f1_score(y_test, yp, zero_division=0)
    fp_tp = fp_t / tp_t if tp_t > 0 else float('inf')
    fn_rate = fn_t / (fn_t + tp_t) if (fn_t + tp_t) > 0 else 0
    marker = " ◀ best" if abs(t - THRESH) < 0.01 else ""
    print(f"  {t:7.2f} | {tp_t:6d} {fp_t:6d} {fn_t:6d} {tn_t:7d} | "
          f"{p_t:6.4f} {r_t:6.4f} {f1_t:6.4f} | {fp_tp:6.2f} {fn_rate:6.2%}{marker}")


# ----------------------------------------------------------
# 10-4. 계좌 단위(Distinct Account) FN/FP 분석
# ----------------------------------------------------------
print("\n" + "-" * 60)
print("👤 [10-4] 계좌 단위 FN/FP 분석")
print("-" * 60)

df_acct = df_analysis.group_by("account_id").agg([
    pl.col("pred_prob").max().alias("max_prob"),
    pl.col("pred_prob").mean().alias("avg_prob"),
    pl.col("pred_prob").count().alias("n_rows"),
    pl.col("is_laundering").max().alias("is_laundering"),
])

y_acct_true = df_acct["is_laundering"].to_numpy()
y_acct_prob = df_acct["max_prob"].to_numpy()
y_acct_pred = (y_acct_prob >= THRESH).astype(int)

tn_a, fp_a, fn_a, tp_a = confusion_matrix(y_acct_true, y_acct_pred).ravel()
print(f"\n  총 고유 계좌: {len(df_acct):,}")
print(f"  TP: {tp_a:,} | FP: {fp_a:,} | FN: {fn_a:,} | TN: {tn_a:,}")
print(f"  Precision: {precision_score(y_acct_true, y_acct_pred, zero_division=0):.4f}")
print(f"  Recall   : {recall_score(y_acct_true, y_acct_pred, zero_division=0):.4f}")
print(f"  F1       : {f1_score(y_acct_true, y_acct_pred, zero_division=0):.4f}")

# 미탐 계좌
df_fn_acct = df_acct.filter(
    (pl.col("is_laundering") == 1) & (pl.col("max_prob") < THRESH)
)
if len(df_fn_acct) > 0:
    fn_a_probs = df_fn_acct["max_prob"].to_numpy()
    print(f"\n  🔴 미탐 계좌 {len(df_fn_acct):,}개:")
    print(f"     max_prob — mean: {np.mean(fn_a_probs):.4f}, "
          f"median: {np.median(fn_a_probs):.4f}, max: {np.max(fn_a_probs):.4f}")
    print(f"     거래 행 수 — mean: {df_fn_acct['n_rows'].mean():.1f}, "
          f"median: {df_fn_acct['n_rows'].median():.0f}, max: {df_fn_acct['n_rows'].max()}")

    # 미탐 계좌 중 거래가 아주 적은(≤3건) 계좌 비율
    few_tx = df_fn_acct.filter(pl.col("n_rows") <= 3)
    print(f"     거래 ≤3건 계좌: {len(few_tx):,}개 ({len(few_tx)/len(df_fn_acct)*100:.1f}%)")

# 오탐 계좌
df_fp_acct = df_acct.filter(
    (pl.col("is_laundering") == 0) & (pl.col("max_prob") >= THRESH)
)
if len(df_fp_acct) > 0:
    fp_a_probs = df_fp_acct["max_prob"].to_numpy()
    print(f"\n  🟡 오탐 계좌 {len(df_fp_acct):,}개:")
    print(f"     max_prob — mean: {np.mean(fp_a_probs):.4f}, "
          f"median: {np.median(fp_a_probs):.4f}, max: {np.max(fp_a_probs):.4f}")
    print(f"     거래 행 수 — mean: {df_fp_acct['n_rows'].mean():.1f}, "
          f"median: {df_fp_acct['n_rows'].median():.0f}, max: {df_fp_acct['n_rows'].max()}")


# ----------------------------------------------------------
# 10-5. XGBoost Feature Importance 기반 — FN/FP 유발 주요 피처
# ----------------------------------------------------------
print("\n" + "-" * 60)
print("🧠 [10-5] Feature Importance 가중 FN/FP 원인 피처 분석")
print("-" * 60)

# Importance가 높은 피처에서 FN/FP가 얼마나 벗어나는지 분석
if len(df_fn) > 0 and len(df_tp) > 0:
    print(f"\n  🔴 미탐 원인 — Importance Top 20 피처 중 FN≠TP 차이:")
    top20_idx = np.argsort(importance)[::-1][:20]
    fn_cause_list = []
    for idx in top20_idx:
        col = xgb_feature_cols[idx]
        if col not in df_analysis.columns:
            continue
        fn_val = df_fn[col].mean()
        tp_val = df_tp[col].mean()
        diff = fn_val - tp_val
        imp_weighted_diff = abs(diff) * importance[idx]
        fn_cause_list.append((col, fn_val, tp_val, diff, importance[idx], imp_weighted_diff))

    fn_cause_list.sort(key=lambda x: x[5], reverse=True)
    for col, fn_v, tp_v, diff, imp, wimp in fn_cause_list:
        d = "↑" if diff > 0 else "↓"
        print(f"     {col:<40s}: FN={fn_v:10.4f} TP={tp_v:10.4f} ({d}{abs(diff):.4f}) "
              f"imp={imp:.4f} → 가중차이={wimp:.6f}")

if len(df_fp) > 0 and len(df_tn) > 0:
    print(f"\n  🟡 오탐 원인 — Importance Top 20 피처 중 FP≠TN 차이:")
    fp_cause_list = []
    for idx in top20_idx:
        col = xgb_feature_cols[idx]
        if col not in df_analysis.columns:
            continue
        fp_val = df_fp[col].mean()
        tn_val = df_tn[col].mean()
        diff = fp_val - tn_val
        imp_weighted_diff = abs(diff) * importance[idx]
        fp_cause_list.append((col, fp_val, tn_val, diff, importance[idx], imp_weighted_diff))

    fp_cause_list.sort(key=lambda x: x[5], reverse=True)
    for col, fp_v, tn_v, diff, imp, wimp in fp_cause_list:
        d = "↑" if diff > 0 else "↓"
        print(f"     {col:<40s}: FP={fp_v:10.4f} TN={tn_v:10.4f} ({d}{abs(diff):.4f}) "
              f"imp={imp:.4f} → 가중차이={wimp:.6f}")


# ----------------------------------------------------------
# 10-6. 미탐 Hard Case 샘플 출력
# ----------------------------------------------------------
print("\n" + "-" * 60)
print("📋 [10-6] Hard FN 샘플 (prob < 0.05인 양성 사례)")
print("-" * 60)

df_hard_fn = df_fn.filter(pl.col("pred_prob") < 0.05).sort("pred_prob")
if len(df_hard_fn) > 0:
    show_cols_fn = ["account_id", "pred_prob"] + \
                   [c for c in aml_cols_in_df if c in df_hard_fn.columns] + \
                   [c for c in ["graph_total_degree", "graph_pagerank", "graph_out_in_ratio",
                                "graph_reciprocity"] if c in df_hard_fn.columns]
    print(f"\n  Hard FN 총 {len(df_hard_fn):,}건 중 상위 20건:")
    print(df_hard_fn.head(20).select(show_cols_fn))
else:
    print("  Hard FN (prob < 0.05) 사례 없음")


# ----------------------------------------------------------
# 10-7. 종합 요약
# ----------------------------------------------------------
print("\n" + "=" * 60)
print("📝 [10-7] 미탐/오탐 분석 종합 요약")
print("=" * 60)

total_pos = int((y_test == 1).sum())
total_neg = int((y_test == 0).sum())

print(f"""
  ┌──────────────────────────────────────────────────────┐
  │  전체 테스트: {len(y_test):,} (양성: {total_pos:,} / 음성: {total_neg:,})  │
  │  Threshold  : {THRESH:.2f}                                    │
  │  AUPRC      : {auprc:.4f}                                  │
  │  F1-Score   : {best_f1:.4f}                                  │
  ├──────────────────────────────────────────────────────┤
  │  TP (정탐)  : {tp:>7,}                                    │
  │  FP (오탐)  : {fp:>7,}  ← 실무 조사 부담                 │
  │  FN (미탐)  : {fn:>7,}  ← 놓친 자금세탁                  │
  │  TN (정상)  : {tn:>7,}                                    │
  ├──────────────────────────────────────────────────────┤
  │  미탐 중 Hard FN (prob<0.05): {n_hard if len(df_fn)>0 else 0:>6,}건                │
  │  미탐 중 Near-miss (prob>={THRESH-0.1:.2f}): {n_near if len(df_fn)>0 else 0:>6,}건                │
  │  오탐 중 고확신 FP (prob>=0.80): {n_high if len(df_fp)>0 else 0:>5,}건                │
  │  계좌단위 미탐: {len(df_fn_acct) if len(df_fn_acct)>0 else 0:>6,}개 / 오탐: {len(df_fp_acct) if len(df_fp_acct)>0 else 0:>6,}개       │
  └──────────────────────────────────────────────────────┘
""")

# 개선 방향 제안
print("  💡 개선 방향 제안:")
if len(df_fn) > 0 and n_hard > len(df_fn) * 0.5:
    print(f"     → Hard FN이 전체 미탐의 {n_hard/len(df_fn)*100:.0f}%: "
          f"이 사례들은 현재 피처 체계로 포착 불가능한 패턴일 가능성 높음")
    print(f"       ⇒ 새로운 피처(예: 2-hop 이웃 통계, 시계열 패턴) 탐색 필요")

if len(df_fn) > 0 and n_near > len(df_fn) * 0.2:
    print(f"     → Near-miss FN이 {n_near/len(df_fn)*100:.0f}%: "
          f"임계값 하향으로 일부 회수 가능 (오탐 증가 트레이드오프)")

if len(df_fp) > 0 and n_high > 0:
    print(f"     → 고확신 오탐 {n_high:,}건: 정상인데 세탁 패턴과 매우 유사한 계좌")
    print(f"       ⇒ 이 계좌들의 공통 특성을 별도 분석하여 모델 보완 필요")

# 메모리 정리
del df_analysis, df_fn, df_tp, df_fp, df_tn, df_test_with_features, X_test
gc.collect()

log_memory("미탐/오탐 분석 완료")
print("\n✅ TGAT v4 + 미탐/오탐 분석 전체 파이프라인 완료!")

🛡️ [TGAT v4] 구조×행동 피처 고도화 파이프라인

📂 데이터 로딩 중...
⏱️ Train cutoff: 2022-09-09 20:00:00
⏱️ Val   cutoff: 2022-09-14 05:00:00
📊 전체 노드 수: 2,076,999
  💾 [데이터 로드 완료] RAM: 57.63GB | GPU: 0.02GB

📐 [Step 2] 기본 그래프 구조 피처 계산...
  📊 PageRank 계산 중 (10 iterations)...
  ✅ 기본 그래프 피처: 13개

🔬 [Step 2-A] 구조-조건부 행동 피처 10개 계산...
  ✅ 구조-조건부 행동 피처 10개: ['cond_highdeg_night_intensity', 'cond_lowdeg_high_amount', 'cond_oneway_crossborder', 'cond_fanout_burst', 'cond_hub_risk_format', 'cond_asymmetric_wire', 'cond_irregular_interbank', 'cond_diverse_partner_currency', 'cond_bidir_amount_volatility', 'cond_multi_acct_burst']

🎯 [Step 2-B] AML 패턴 점수 5개 설계...
  ✅ AML 패턴 점수 5개: ['score_smurf', 'score_mule', 'score_layering', 'score_integration', 'score_composite']
  💾 [구조-행동 피처 + AML 점수 완료] RAM: 57.18GB | GPU: 0.02GB

🔗 [Step 3] GNN 엣지/노드 구성 중...
📊 Train 엣지: 19,060,343
📊 GNN 학습 노드: 양성 20,400 + 음성 1,020,000 = 1,040,400
  💾 [GNN 데이터 준비 완료] RAM: 57.66GB | GPU: 0.02GB

🧠 [Step 4] TGAT v3 모델 정의...

🔥 [Step 5] TGAT 학습 시작.

  → Epoch  1 | Loss: 1.3407 | LR: 0.000001


  → Epoch  5 | Loss: 1.1512 | LR: 0.000800


  → Epoch 10 | Loss: 1.0844 | LR: 0.000968


  → Epoch 15 | Loss: 1.0764 | LR: 0.000847


...

  → Epoch 30 | Loss: 1.0385 | LR: 0.000232


  → Epoch 35 | Loss: 1.0337 | LR: 0.000080


  → Epoch 40 | Loss: 1.0312 | LR: 0.000012
  ✅ Best model 복원 (loss: 1.0294)

📦 [Step 6] TGAT 임베딩 추출...


Embedding Extraction: 100%|█████████████████████████████████████████████████| 1015/1015 [00:17<00:00, 56.43it/s]


✅ 임베딩 shape: (2076999, 64)

🔗 [Step 6-A] TGAT × PageRank interaction 피처 생성...
  📊 Variance 상위 8 임베딩 차원: ['tgat_emb_59', 'tgat_emb_24', 'tgat_emb_37', 'tgat_emb_21', 'tgat_emb_18', 'tgat_emb_44', 'tgat_emb_33', 'tgat_emb_4']
  ✅ Interaction 피처 10개 생성
  💾 [전체 피처 엔지니어링 완료] RAM: 55.50GB | GPU: 0.02GB

🚀 [Step 7] XGBoost 데이터 구성...
📊 총 피처 수: 140
  V2: 38 | 그래프: 13 | 조건부행동: 10 | AML점수: 5 | TGAT: 64 | Interaction: 10
📊 Train: 17,172,747 → 619,437 (양성비율: 4.76%)
📊 Val  : 5,790,644 (양성비율: 0.30%)
📊 Test : 5,792,913 (양성비율: 0.36%)
  💾 [XGBoost 데이터 준비 완료] RAM: 57.26GB | GPU: 0.02GB

🔥 [Step 8] XGBoost 학습...
📊 scale_pos_weight: 20.0
[0]	validation_0-aucpr:0.07865
[100]	validation_0-aucpr:0.32392
[200]	validation_0-aucpr:0.41327
[300]	validation_0-aucpr:0.43845
[400]	validation_0-aucpr:0.44617
[500]	validation_0-aucpr:0.45196
[600]	validation_0-aucpr:0.45653
[700]	validation_0-aucpr:0.45906
[800]	validation_0-aucpr:0.46021
[900]	validation_0-aucpr:0.46389
[1000]	validation_0-aucpr:0.46654
[1100]	valida

In [9]:
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv
from torch_geometric.loader import NeighborLoader
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm
import gc
import numpy as np
import psutil
import os
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

def log_memory(tag=""):
    proc = psutil.Process(os.getpid())
    rss_gb = proc.memory_info().rss / (1024 ** 3)
    gpu_str = ""
    if torch.cuda.is_available():
        gpu_alloc = torch.cuda.memory_allocated() / (1024 ** 3)
        gpu_str = f" | GPU: {gpu_alloc:.2f}GB"
    print(f"  💾 [{tag}] RAM: {rss_gb:.2f}GB{gpu_str}")

print("=" * 60)
print("🛡️ [TGAT v5.4] v4 피처 + 2-Stage Pruning 최적화")
print("   ▸ v4 원본 140개 피처 그대로 유지")
print("   ▸ 2-Stage: Importance Pruning + HP 탐색")
print("   ▸ 새 피처 추가 없음 — 순수 최적화 실험")
print("=" * 60)

NEG_NODE_RATIO_GNN = 50
NEG_ROW_RATIO_XGB = 20
RANDOM_SEED = 42
HIDDEN_DIM = 64
EDGE_DIM = 16
N_HEADS = 4
N_EPOCHS_GNN = 40
BATCH_SIZE = 2048
NUM_NEIGHBORS = [15, 10]
N_INTERACTION_DIMS = 8

raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"

# === 1. 데이터 로드 ===
print("\n📂 데이터 로딩 중...")
df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))
if df_raw["Timestamp"].dtype == pl.Utf8:
    df_raw = df_raw.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_v2 = df_v2.sort("time_group")
total_count = len(df_v2)
train_idx = int(total_count * 0.6); val_idx = int(total_count * 0.8)
train_cutoff = df_v2["time_group"][train_idx]; val_cutoff = df_v2["time_group"][val_idx]
print(f"⏱️ Train cutoff: {train_cutoff}\n⏱️ Val   cutoff: {val_cutoff}")
all_accounts = pl.concat([df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))]).unique().with_row_index("node_id")
num_nodes = len(all_accounts); print(f"📊 전체 노드 수: {num_nodes:,}"); log_memory("데이터 로드 완료")

# === 2. 그래프 피처 ===
print("\n📐 [Step 2] 기본 그래프 구조 피처 계산...")
df_raw_train = df_raw.filter(pl.col("Timestamp") < train_cutoff); del df_raw; gc.collect()
df_from = df_raw_train.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id"))
df_to = df_raw_train.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
out_degree = df_from.group_by("account_id").len().rename({"len":"graph_out_degree"})
in_degree = df_to.group_by("account_id").len().rename({"len":"graph_in_degree"})
unique_out = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("to_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_out_partners"))
unique_in = df_raw_train.select([pl.col("to_acc").cast(pl.Utf8).alias("account_id"),pl.col("from_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_in_partners"))
edges_set_from = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("a"),pl.col("to_acc").cast(pl.Utf8).alias("b")]).unique()
edges_set_to = edges_set_from.select([pl.col("b").alias("a"),pl.col("a").alias("b")])
bidirectional = edges_set_from.join(edges_set_to, on=["a","b"], how="inner")
bidir_count = bidirectional.select(pl.col("a").alias("account_id")).group_by("account_id").len().rename({"len":"graph_bidir_count"})
del edges_set_from, edges_set_to, bidirectional; gc.collect()
time_features = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("Timestamp")]).group_by("account_id").agg([pl.col("Timestamp").sort().diff().dt.total_seconds().mean().alias("graph_avg_tx_interval"),pl.col("Timestamp").sort().diff().dt.total_seconds().std().alias("graph_std_tx_interval"),pl.col("Timestamp").count().alias("graph_total_tx_count")])
print("  📊 PageRank 계산 중...")
account_to_id = dict(zip(all_accounts["account_id"].to_list(), all_accounts["node_id"].to_list()))
src_ids = df_raw_train["from_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
dst_ids = df_raw_train["to_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
valid_mask = (src_ids>=0)&(dst_ids>=0); src_valid=src_ids[valid_mask]; dst_valid=dst_ids[valid_mask]
out_deg_arr = np.zeros(num_nodes, dtype=np.float64); np.add.at(out_deg_arr, src_valid, 1.0); out_deg_arr = np.maximum(out_deg_arr, 1.0)
damping=0.85; pr=np.ones(num_nodes,dtype=np.float64)/num_nodes
for _ in range(10):
    new_pr = np.ones(num_nodes,dtype=np.float64)*(1-damping)/num_nodes; contributions=pr[src_valid]/out_deg_arr[src_valid]; np.add.at(new_pr, dst_valid, damping*contributions); pr=new_pr
pr_df = pl.DataFrame({"node_id":np.arange(num_nodes,dtype=np.uint32),"graph_pagerank":pr.astype(np.float32)})
pr_df = all_accounts.join(pr_df, on="node_id", how="left").select(["account_id","graph_pagerank"])
del src_ids,dst_ids,src_valid,dst_valid,out_deg_arr,pr,new_pr; gc.collect()
df_graph_feats = all_accounts.select("account_id").join(out_degree,on="account_id",how="left").join(in_degree,on="account_id",how="left").join(unique_out,on="account_id",how="left").join(unique_in,on="account_id",how="left").join(bidir_count,on="account_id",how="left").join(time_features,on="account_id",how="left").join(pr_df,on="account_id",how="left").fill_null(0.0)
df_graph_feats = df_graph_feats.with_columns([(pl.col("graph_out_degree")+pl.col("graph_in_degree")).alias("graph_total_degree"),(pl.col("graph_out_degree")/(pl.col("graph_in_degree")+1)).alias("graph_out_in_ratio"),(pl.col("graph_bidir_count")/(pl.col("graph_out_degree")+1)).alias("graph_reciprocity"),(pl.col("graph_unique_out_partners")/(pl.col("graph_out_degree")+1)).alias("graph_partner_diversity")])
del out_degree,in_degree,unique_out,unique_in,bidir_count,time_features,pr_df,df_from,df_to; gc.collect()
print(f"  ✅ 기본 그래프 피처: {len([c for c in df_graph_feats.columns if c.startswith('graph_')])}개")

# === 2-A. 조건부행동 피처 ===
print("\n🔬 [Step 2-A] 구조-조건부 행동 피처 10개...")
df_v2_train = df_v2.filter(pl.col("time_group")<train_cutoff)
exclude_cols = ["account_id","time_group","is_laundering","mode_format","currency_mode"]
gnn_feature_cols = [c for c in df_v2.columns if c not in exclude_cols]
df_v2_agg = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols])
df_cond = df_v2_agg.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)
v2_cols_available = set(df_cond.columns)
def safe_col(name, default=0.0): return pl.col(name) if name in v2_cols_available else pl.lit(default)
cond_exprs = []
if "cnt_night" in v2_cols_available and "cnt_1h" in v2_cols_available: cond_exprs.append((pl.col("graph_total_degree")*safe_col("cnt_night")/(safe_col("cnt_1h")+1)).alias("cond_highdeg_night_intensity"))
if "avg_log_amount" in v2_cols_available: cond_exprs.append((safe_col("avg_log_amount")/(pl.col("graph_total_degree").log1p()+1)).alias("cond_lowdeg_high_amount"))
if "ratio_cross_border" in v2_cols_available: cond_exprs.append((safe_col("ratio_cross_border")*(1-pl.col("graph_reciprocity"))).alias("cond_oneway_crossborder"))
if "cnt_1h" in v2_cols_available: cond_exprs.append((pl.col("graph_unique_out_partners")*safe_col("cnt_1h")/(pl.col("graph_out_degree")+1)).alias("cond_fanout_burst"))
if "cnt_risk_format" in v2_cols_available: cond_exprs.append((pl.col("graph_pagerank")*safe_col("cnt_risk_format")).alias("cond_hub_risk_format"))
if "cnt_wire" in v2_cols_available: cond_exprs.append((pl.col("graph_out_in_ratio")*safe_col("cnt_wire")).alias("cond_asymmetric_wire"))
if "cnt_inter_bank" in v2_cols_available: cond_exprs.append((pl.col("graph_std_tx_interval").fill_null(0)/(pl.col("graph_avg_tx_interval").fill_null(1)+1)*safe_col("cnt_inter_bank")).alias("cond_irregular_interbank"))
if "cnt_currencies" in v2_cols_available: cond_exprs.append((pl.col("graph_partner_diversity")*safe_col("cnt_currencies")).alias("cond_diverse_partner_currency"))
if "amount_kurtosis" in v2_cols_available: cond_exprs.append((pl.col("graph_reciprocity")*safe_col("amount_kurtosis")).alias("cond_bidir_amount_volatility"))
if "entity_acct_cnt" in v2_cols_available and "burst_ratio" in v2_cols_available: cond_exprs.append((safe_col("entity_acct_cnt")*safe_col("burst_ratio")).alias("cond_multi_acct_burst"))
df_cond = df_cond.with_columns(cond_exprs); cond_feature_cols = [e.meta.output_name() for e in cond_exprs]
for cn in cond_feature_cols: df_cond = df_cond.with_columns(pl.when(pl.col(cn).is_infinite()|pl.col(cn).is_nan()).then(0.0).otherwise(pl.col(cn)).alias(cn))
df_cond_features = df_cond.select(["account_id"]+cond_feature_cols); print(f"  ✅ {len(cond_feature_cols)}개"); del df_cond; gc.collect()

# === 2-B. AML 점수 ===
print("\n🎯 [Step 2-B] AML 패턴 점수 5개...")
df_score_base = df_v2_agg.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)
score_exprs = [((safe_col("cnt_1h").fill_null(0).clip(0,1e6)+1).log1p()*pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*safe_col("burst_ratio").fill_null(0).clip(0,1e6)/(safe_col("avg_log_amount").fill_null(1).clip(0,1e6)+1)).alias("score_smurf"),((1.0/((pl.col("graph_out_in_ratio").clip(0.01,100)-1.0).abs()+0.1))*safe_col("cnt_wire").fill_null(0).clip(0,1e6).log1p()*(1-pl.col("graph_reciprocity").clip(0,1))/(pl.col("graph_unique_out_partners").clip(0,1e6).log1p()+1)).alias("score_mule"),(pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*safe_col("cnt_currencies").fill_null(0).clip(0,1e6).log1p()*safe_col("ratio_cross_border").fill_null(0).clip(0,1)/(pl.col("graph_avg_tx_interval").fill_null(3600).clip(0,1e9)/3600+1)).alias("score_layering"),(pl.col("graph_in_degree").clip(0,1e6).log1p()*(pl.col("graph_pagerank").clip(0,1)*1e6)*safe_col("cnt_risk_format").fill_null(0).clip(0,1e6).log1p()/(pl.col("graph_out_degree").clip(0,1e6).log1p()+1)).alias("score_integration")]
df_score_base = df_score_base.with_columns(score_exprs)
score_names = ["score_smurf","score_mule","score_layering","score_integration"]
for sn in score_names: df_score_base = df_score_base.with_columns(pl.when(pl.col(sn).is_infinite()|pl.col(sn).is_nan()).then(0.0).otherwise(pl.col(sn)).alias(sn))
for sn in score_names:
    mn=df_score_base[sn].min(); mx=df_score_base[sn].max(); rng=mx-mn if (mx-mn)>0 else 1.0
    df_score_base = df_score_base.with_columns(((pl.col(sn)-mn)/rng).alias(f"{sn}_norm"))
df_score_base = df_score_base.with_columns((pl.col("score_smurf_norm")*0.3+pl.col("score_mule_norm")*0.25+pl.col("score_layering_norm")*0.3+pl.col("score_integration_norm")*0.15).alias("score_composite"))
aml_score_cols = score_names+["score_composite"]
df_aml_scores = df_score_base.select(["account_id"]+aml_score_cols); print(f"  ✅ {len(aml_score_cols)}개")
del df_score_base, df_v2_agg; gc.collect(); log_memory("피처 완료")

# === 3. GNN 엣지/노드 ===
print("\n🔗 [Step 3] GNN 엣지/노드 구성 중...")
df_edges_train = df_raw_train.select(["from_acc","to_acc","Timestamp"]).with_columns([pl.col("from_acc").cast(pl.Utf8),pl.col("to_acc").cast(pl.Utf8)])
df_edges_train = df_edges_train.join(all_accounts.rename({"account_id":"from_acc","node_id":"src_id"}),on="from_acc",how="left").join(all_accounts.rename({"account_id":"to_acc","node_id":"dst_id"}),on="to_acc",how="left")
timestamps_raw = df_edges_train["Timestamp"]; min_ts=timestamps_raw.min(); max_ts=timestamps_raw.max(); total_seconds=(max_ts-min_ts).total_seconds()
edge_time_normalized = ((timestamps_raw-min_ts).dt.total_microseconds().cast(pl.Float64)/1e6/max(total_seconds,1.0)).to_numpy().astype(np.float32)
hours=df_edges_train["Timestamp"].dt.hour().to_numpy().astype(np.float32); dow=df_edges_train["Timestamp"].dt.weekday().to_numpy().astype(np.float32)
edge_features = np.stack([edge_time_normalized,np.sin(2*np.pi*hours/24),np.cos(2*np.pi*hours/24),np.sin(2*np.pi*dow/7),np.cos(2*np.pi*dow/7)],axis=1)
edge_index_train = torch.tensor(df_edges_train.select(["src_id","dst_id"]).to_numpy().T, dtype=torch.long)
edge_attr_train = torch.tensor(edge_features, dtype=torch.float32); EDGE_RAW_DIM=edge_attr_train.shape[1]
print(f"📊 Train 엣지: {edge_index_train.shape[1]:,}")
del df_raw_train,df_edges_train,timestamps_raw,hours,dow,edge_features,edge_time_normalized; gc.collect()
df_v2_node = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols])
df_node_features = all_accounts.join(df_v2_node, on="account_id", how="left").fill_null(0.0)
X_node = torch.tensor(df_node_features.select(gnn_feature_cols).to_numpy(), dtype=torch.float32); IN_DIM=X_node.shape[1]
target_labels = df_v2_train.filter(pl.col("is_laundering")==1).select("account_id").unique().with_columns(pl.lit(1).alias("label"))
df_labels = all_accounts.join(target_labels, on="account_id", how="left").fill_null(0)
Y_node = torch.tensor(df_labels["label"].to_numpy(), dtype=torch.long)
active_accounts = df_v2_train["account_id"].unique(); active_df = pl.DataFrame({"account_id":active_accounts,"is_active":True})
mask_df = all_accounts.join(active_df, on="account_id", how="left").fill_null(False)
pos_node_ids = set(df_labels.filter(pl.col("label")==1)["node_id"].to_list())
active_node_ids = mask_df.filter(pl.col("is_active"))["node_id"].to_list()
active_pos = [n for n in active_node_ids if n in pos_node_ids]; active_neg = [n for n in active_node_ids if n not in pos_node_ids]
n_neg_keep_gnn = min(len(active_pos)*NEG_NODE_RATIO_GNN, len(active_neg)); np.random.seed(RANDOM_SEED)
sampled_neg = np.random.choice(active_neg, size=n_neg_keep_gnn, replace=False).tolist(); sampled_nodes = set(active_pos+sampled_neg)
train_mask_np = np.zeros(num_nodes, dtype=bool)
for nid in sampled_nodes: train_mask_np[nid]=True
train_mask = torch.tensor(train_mask_np, dtype=torch.bool)
print(f"📊 GNN 학습 노드: 양성 {len(active_pos):,} + 음성 {n_neg_keep_gnn:,} = {len(sampled_nodes):,}")
del df_v2_node,df_node_features,mask_df,active_df; gc.collect(); log_memory("GNN 데이터 준비 완료")

# === 4-5. TGAT 모델 + 학습 ===
print("\n🧠 [Step 4] TGAT v3 모델 정의...")
class EdgeProjection(nn.Module):
    def __init__(s,i,o): super().__init__(); s.proj=nn.Sequential(nn.Linear(i,o),nn.GELU(),nn.Linear(o,o))
    def forward(s,e): return s.proj(e)
class TGAT_v3(nn.Module):
    def __init__(s,in_dim,hidden_dim,edge_raw_dim,edge_dim,n_heads,dropout=0.2):
        super().__init__(); s.edge_proj=EdgeProjection(edge_raw_dim,edge_dim); s.input_proj=nn.Linear(in_dim,hidden_dim)
        s.norm1=nn.LayerNorm(hidden_dim); s.conv1=TransformerConv(hidden_dim,hidden_dim//n_heads,heads=n_heads,edge_dim=edge_dim,dropout=dropout,concat=True)
        s.norm2=nn.LayerNorm(hidden_dim); s.conv2=TransformerConv(hidden_dim,hidden_dim//n_heads,heads=n_heads,edge_dim=edge_dim,dropout=dropout,concat=True)
        s.norm_out=nn.LayerNorm(hidden_dim); s.classifier=nn.Sequential(nn.Linear(hidden_dim,hidden_dim//2),nn.GELU(),nn.Dropout(dropout),nn.Linear(hidden_dim//2,1)); s.dropout=dropout
    def forward(s,x,ei,ea):
        ee=s.edge_proj(ea); h=s.input_proj(x); h=h+F.dropout(s.conv1(s.norm1(h),ei,ee),p=s.dropout,training=s.training); h=h+F.dropout(s.conv2(s.norm2(h),ei,ee),p=s.dropout,training=s.training); h=s.norm_out(h); return s.classifier(h).squeeze(-1)
    @torch.no_grad()
    def get_embedding(s,x,ei,ea):
        ee=s.edge_proj(ea); h=s.input_proj(x); h=h+s.conv1(s.norm1(h),ei,ee); h=h+s.conv2(s.norm2(h),ei,ee); return s.norm_out(h)

print("\n🔥 [Step 5] TGAT 학습 시작...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print(f"📌 디바이스: {device}")
graph_data = Data(x=X_node, edge_index=edge_index_train, edge_attr=edge_attr_train, y=Y_node); graph_data.train_mask=train_mask; graph_data.num_nodes=num_nodes
del X_node,edge_index_train,edge_attr_train,Y_node,train_mask; gc.collect()
train_loader = NeighborLoader(graph_data,num_neighbors=NUM_NEIGHBORS,batch_size=BATCH_SIZE,input_nodes=graph_data.train_mask,shuffle=True,num_workers=4)
full_loader = NeighborLoader(graph_data,num_neighbors=NUM_NEIGHBORS,batch_size=BATCH_SIZE,input_nodes=None,shuffle=False,num_workers=4)
model = TGAT_v3(in_dim=IN_DIM,hidden_dim=HIDDEN_DIM,edge_raw_dim=EDGE_RAW_DIM,edge_dim=EDGE_DIM,n_heads=N_HEADS,dropout=0.2).to(device)
pw=(len(sampled_nodes)-len(active_pos))/max(len(active_pos),1); print(f"📊 pos_weight: {pw:.1f}")
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw],device=device))
BASE_LR=0.001; WARMUP_EPOCHS=5; optimizer=torch.optim.AdamW(model.parameters(),lr=1e-6,weight_decay=1e-4)
def set_lr(opt,ep):
    if ep<WARMUP_EPOCHS: lr=1e-6+(BASE_LR-1e-6)*(ep/WARMUP_EPOCHS)
    else: p=(ep-WARMUP_EPOCHS)/max(N_EPOCHS_GNN-WARMUP_EPOCHS,1); lr=1e-5+(BASE_LR-1e-5)*0.5*(1+np.cos(np.pi*p))
    for pg in opt.param_groups: pg['lr']=lr
    return lr
model.train(); best_loss=float('inf'); patience_counter=0; PATIENCE=10; best_state=None
for epoch in range(N_EPOCHS_GNN):
    clr=set_lr(optimizer,epoch); tl=0.0; nb=0
    for batch in tqdm(train_loader,desc=f"Epoch {epoch+1}/{N_EPOCHS_GNN}",leave=False):
        batch=batch.to(device); optimizer.zero_grad()
        bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
        out=model(batch.x,batch.edge_index,bea); loss=criterion(out[:batch.batch_size],batch.y[:batch.batch_size].float())
        if torch.isnan(loss) or torch.isinf(loss): optimizer.zero_grad(); continue
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=0.5); optimizer.step(); tl+=loss.item(); nb+=1
    al=tl/max(nb,1)
    if (epoch+1)%5==0 or epoch==0: print(f"  → Epoch {epoch+1:2d} | Loss: {al:.4f} | LR: {clr:.6f}")
    if al<best_loss-0.0005: best_loss=al; patience_counter=0; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
    else: patience_counter+=1
    if patience_counter>=PATIENCE: print(f"  ⏹️ Early stopping at epoch {epoch+1}"); break
if best_state: model.load_state_dict({k:v.to(device) for k,v in best_state.items()}); print(f"  ✅ Best model 복원 (loss: {best_loss:.4f})"); del best_state

# === 6. 임베딩 추출 ===
print("\n📦 [Step 6] TGAT 임베딩 추출...")
model.eval(); all_emb=[]
with torch.no_grad():
    for batch in tqdm(full_loader,desc="Embedding Extraction"):
        batch=batch.to(device); bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
        emb=model.get_embedding(batch.x,batch.edge_index,bea); all_emb.append(emb[:batch.batch_size].cpu())
final_emb=torch.cat(all_emb,dim=0).numpy(); emb_cols=[f"tgat_emb_{i}" for i in range(HIDDEN_DIM)]
df_tgat=pl.concat([all_accounts.select("account_id"),pl.DataFrame(final_emb,schema=emb_cols)],how="horizontal")
print(f"✅ 임베딩 shape: {final_emb.shape}"); del graph_data,all_emb,final_emb; torch.cuda.empty_cache(); gc.collect()

# === 6-A. Interaction 피처 ===
print("\n🔗 [Step 6-A] TGAT × PageRank interaction 피처...")
df_tgat_with_pr=df_tgat.join(df_graph_feats.select(["account_id","graph_pagerank"]),on="account_id",how="left").fill_null(0.0)
emb_np=df_tgat_with_pr.select(emb_cols).to_numpy(); emb_var=np.var(emb_np,axis=0)
top_emb_indices=np.argsort(emb_var)[::-1][:N_INTERACTION_DIMS]; top_emb_cols=[emb_cols[i] for i in top_emb_indices]
interaction_exprs=[]; interaction_cols=[]
for cn in top_emb_cols: nn2=f"interact_pr_x_{cn}"; interaction_exprs.append((pl.col(cn)*pl.col("graph_pagerank")*1e6).alias(nn2)); interaction_cols.append(nn2)
interaction_exprs.append((sum(pl.col(c)**2 for c in emb_cols).sqrt()*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_norm")); interaction_cols.append("interact_pr_x_emb_norm")
interaction_exprs.append((sum(pl.col(c) for c in top_emb_cols)*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_topsum")); interaction_cols.append("interact_pr_x_emb_topsum")
df_interactions=df_tgat_with_pr.with_columns(interaction_exprs).select(["account_id"]+interaction_cols)
for ic in interaction_cols: df_interactions=df_interactions.with_columns(pl.when(pl.col(ic).is_infinite()|pl.col(ic).is_nan()).then(0.0).otherwise(pl.col(ic)).alias(ic))
print(f"  ✅ Interaction 피처 {len(interaction_cols)}개"); del df_tgat_with_pr,emb_np; gc.collect(); log_memory("피처 완료")

# === 7. XGBoost 데이터 구성 (v4 동일 140개) ===
print("\n🚀 [Step 7] XGBoost 데이터 구성...")
xgb_exclude_cols=["account_id","time_group","is_laundering","mode_format","currency_mode","date"]
xgb_v2_cols=[c for c in df_v2.columns if c not in xgb_exclude_cols]
graph_feature_cols=[c for c in df_graph_feats.columns if c.startswith("graph_")]
xgb_feature_cols = xgb_v2_cols+graph_feature_cols+cond_feature_cols+aml_score_cols+emb_cols+interaction_cols
print(f"📊 총 피처 수: {len(xgb_feature_cols)} (v4 동일)")
def build_xgb_df(db):
    df=db.join(df_graph_feats,on="account_id",how="left").join(df_cond_features,on="account_id",how="left").join(df_aml_scores,on="account_id",how="left").join(df_tgat,on="account_id",how="left").join(df_interactions,on="account_id",how="left").fill_null(0.0).fill_nan(0.0)
    for cn in xgb_feature_cols:
        if cn in df.columns: df=df.with_columns(pl.when(pl.col(cn).is_infinite()).then(0.0).otherwise(pl.col(cn)).alias(cn))
    return df
df_train_full=build_xgb_df(df_v2.filter(pl.col("time_group")<train_cutoff)); df_train_pos=df_train_full.filter(pl.col("is_laundering")==1); df_train_neg=df_train_full.filter(pl.col("is_laundering")==0)
n_neg_keep_xgb=min(len(df_train_pos)*NEG_ROW_RATIO_XGB,len(df_train_neg)); df_train=pl.concat([df_train_pos,df_train_neg.sample(n=n_neg_keep_xgb,seed=RANDOM_SEED)]).sample(fraction=1.0,seed=RANDOM_SEED)
X_train=df_train.select(xgb_feature_cols).to_pandas(); y_train=df_train["is_laundering"].to_numpy()
print(f"📊 Train: {len(df_train_full):,} → {len(df_train):,} (양성: {y_train.mean()*100:.2f}%)")
del df_train,df_train_full,df_train_pos,df_train_neg; gc.collect()
df_val=build_xgb_df(df_v2.filter((pl.col("time_group")>=train_cutoff)&(pl.col("time_group")<val_cutoff)))
X_val=df_val.select(xgb_feature_cols).to_pandas(); y_val=df_val["is_laundering"].to_numpy(); print(f"📊 Val  : {len(X_val):,}"); del df_val; gc.collect()
df_test_full=build_xgb_df(df_v2.filter(pl.col("time_group")>=val_cutoff)); df_test_meta=df_test_full.select(["account_id","time_group","is_laundering"])
X_test=df_test_full.select(xgb_feature_cols).to_pandas(); y_test=df_test_full["is_laundering"].to_numpy(); print(f"📊 Test : {len(X_test):,}")
del df_test_full,df_v2,df_tgat,df_graph_feats,df_cond_features,df_aml_scores,df_interactions; gc.collect(); log_memory("XGBoost 준비 완료")

# =============================================================
# 8. ★ 2-Stage Pruning + HP 탐색
# =============================================================
print("\n🔥 [Step 8] 2-Stage Pruning + HP 탐색...")
actual_pos_ratio=y_train.sum()/len(y_train); adjusted_spw=max((1-actual_pos_ratio)/actual_pos_ratio,1.0)
print(f"📊 scale_pos_weight: {adjusted_spw:.1f} | 피처: {len(xgb_feature_cols)}")

# Stage 1
print("\n── [Stage 1] 전체 140개 피처 학습 ──")
s1p={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda","random_state":42,"max_depth":8,"learning_rate":0.03,"scale_pos_weight":adjusted_spw,"subsample":0.8,"colsample_bytree":0.7,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
m_s1=xgb.XGBClassifier(**s1p,n_estimators=1500,early_stopping_rounds=100); m_s1.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=100)
s1_val=m_s1.best_score; s1_test=average_precision_score(y_test,m_s1.predict_proba(X_test)[:,1])
print(f"\n  Stage 1: Val={s1_val:.5f} Test={s1_test:.4f} (iter: {m_s1.best_iteration})")
imp_s1=m_s1.feature_importances_; feat_imp=sorted(zip(xgb_feature_cols,imp_s1),key=lambda x:x[1],reverse=True)
print(f"  imp>=1%: {sum(1 for _,i in feat_imp if i>=0.01)} | imp>=0.5%: {sum(1 for _,i in feat_imp if i>=0.005)} | imp<0.1%: {sum(1 for _,i in feat_imp if i<0.001)}")
del m_s1; gc.collect()

# Stage 2
print("\n── [Stage 2] Pruning + HP 탐색 ──")
bp2={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda","random_state":42,"scale_pos_weight":adjusted_spw,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
configs=[]
for nf in [25,30,35,40,50,60]:
    sel=[n for n,_ in feat_imp[:nf]]
    configs.append({"label":f"top{nf}_base","features":sel,"hp":{"max_depth":8,"learning_rate":0.03,"colsample_bytree":0.7,"subsample":0.8,"min_child_weight":5,"n_estimators":1500}})
    configs.append({"label":f"top{nf}_col85","features":sel,"hp":{"max_depth":8,"learning_rate":0.03,"colsample_bytree":0.85,"subsample":0.8,"min_child_weight":5,"n_estimators":1500}})
    configs.append({"label":f"top{nf}_lr02","features":sel,"hp":{"max_depth":8,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"n_estimators":2500}})
configs.append({"label":"all_140","features":xgb_feature_cols,"hp":{"max_depth":8,"learning_rate":0.03,"colsample_bytree":0.7,"subsample":0.8,"min_child_weight":5,"n_estimators":1500}})

print(f"\n📊 {len(configs)}개 조합 탐색...")
print(f"  {'#':>2s} {'Label':<18s} | {'#f':>3s} {'d':>2s} {'LR':>5s} {'col':>4s} | {'ValAUCPR':>9s} {'it':>5s}")
print("  "+"-"*60)
res=[]
for i,c in enumerate(configs):
    lb=c["label"]; sf=c["features"]; hp=c["hp"].copy(); ne=hp.pop("n_estimators")
    m=xgb.XGBClassifier(**{**bp2,**hp},n_estimators=ne,early_stopping_rounds=100)
    m.fit(X_train[sf],y_train,eval_set=[(X_val[sf],y_val)],verbose=0)
    va=m.best_score; bi=m.best_iteration
    res.append({"label":lb,"features":sf,"hp":{**hp,"n_estimators":ne},"val":va,"iter":bi,"model":m})
    print(f"  {i+1:2d} {lb:<18s} | {len(sf):3d} {hp.get('max_depth',8):2d} {hp.get('learning_rate',0.03):5.3f} {hp.get('colsample_bytree',0.7):4.2f} | {va:9.5f} {bi:5d}")
    if len(res)>1:
        bsf=max(res,key=lambda x:x["val"])
        for r in res:
            if r is not bsf and r.get("model"): del r["model"]; r["model"]=None
        gc.collect()

br=max(res,key=lambda x:x["val"]); model_xgb=br["model"]; best_features=br["features"]
print(f"\n  🏆 Best: {br['label']} (Val: {br['val']:.5f}, {len(best_features)}개 피처, iter: {br['iter']})")
print(f"     Stage1 대비: {'↑' if br['val']>s1_val else '↓'} {abs(br['val']-s1_val):.5f}")
y_prob=model_xgb.predict_proba(X_test[best_features])[:,1]
for r in res:
    if r.get("model") and r is not br: del r["model"]
gc.collect()

# Importance
print(f"\n🔍 Feature Importance Top 25 ({len(best_features)}개)")
imp=model_xgb.feature_importances_; tidx=np.argsort(imp)[::-1][:25]
for i,idx in enumerate(tidx):
    c=best_features[idx]; t=""
    if c.startswith("tgat_emb_"): t=" [TGAT]"
    elif c.startswith("graph_"): t=" [GRAPH]"
    elif c.startswith("cond_"): t=" [COND]"
    elif c.startswith("score_"): t=" [SCORE]"
    elif c.startswith("interact_"): t=" [INTER]"
    print(f"  {i+1:2d}. {c:<40s}: {imp[idx]:.4f}{t}")
ti=imp.sum(); v2i=sum(imp[i] for i,c in enumerate(best_features) if c in xgb_v2_cols)
gi=sum(imp[i] for i,c in enumerate(best_features) if c.startswith("graph_"))
ci=sum(imp[i] for i,c in enumerate(best_features) if c.startswith("cond_"))
si=sum(imp[i] for i,c in enumerate(best_features) if c.startswith("score_"))
ei=sum(imp[i] for i,c in enumerate(best_features) if c.startswith("tgat_emb_"))
ii=sum(imp[i] for i,c in enumerate(best_features) if c.startswith("interact_"))
print(f"\n📊 그룹별: V2={v2i/ti*100:.1f}% 그래프={gi/ti*100:.1f}% 조건부={ci/ti*100:.1f}% AML={si/ti*100:.1f}% TGAT={ei/ti*100:.1f}% Inter={ii/ti*100:.1f}%")
del X_train,X_val; gc.collect()

# === 9. 리포트 ===
print("\n"+"="*60); print("🏆 [TGAT v5.4] 최종 성능 리포트"); print("="*60)
auprc=average_precision_score(y_test,y_prob); print(f"\n  AUPRC: {auprc:.4f}")
best_f1=0; best_thresh=0
for th in np.arange(0.05,0.95,0.01):
    f=f1_score(y_test,(y_prob>=th).astype(int),zero_division=0)
    if f>best_f1: best_f1=f; best_thresh=th
y_pred=(y_prob>=best_thresh).astype(int); prec=precision_score(y_test,y_pred,zero_division=0); rec=recall_score(y_test,y_pred,zero_division=0)
print(f"\n📌 [최적 임계값] Thresh={best_thresh:.2f} | AUPRC={auprc:.4f} | F1={best_f1:.4f} | P={prec:.4f} | R={rec:.4f}")
print(f"\n📌 [임계값별 성능]")
for t in [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]:
    yp=(y_prob>=t).astype(int); print(f"  T={t:.1f} | P={precision_score(y_test,yp,zero_division=0):.4f} R={recall_score(y_test,yp,zero_division=0):.4f} F1={f1_score(y_test,yp,zero_division=0):.4f}")
df_res=df_test_meta.with_columns([pl.col("time_group").dt.date().alias("date"),pl.Series("pred_prob",y_prob)])
df_distinct=df_res.sort("pred_prob",descending=True).unique(subset=["account_id"],maintain_order=True)
print(f"\n📌 [Top-K 탐지]")
for k in [100,500,1000,5000,10000]:
    ck=min(k,len(df_distinct)); hits=df_distinct.head(ck)["is_laundering"].sum() if ck>0 else 0
    print(f"  Top-{k:5d}: {hits:5d}명 | P={hits/ck*100:.2f}%")
print(f"\n📌 [일별 Top-100]")
for d in df_distinct["date"].unique(maintain_order=True).sort(descending=True):
    hits=df_distinct.filter(pl.col("date")==d).head(100)["is_laundering"].sum(); print(f"  {d}: {hits}")
tn,fp,fn,tp=confusion_matrix(y_test,y_pred).ravel()
print(f"\n📌 [v4→v5.4] AUPRC: 0.5370→{auprc:.4f} ({'↑' if auprc>0.537 else '↓'}{abs(auprc-0.537):.4f}) | F1: 0.4940→{best_f1:.4f}")
print(f"  TP:{tp:,} FP:{fp:,} FN:{fn:,} TN:{tn:,}")
print(f"\n📌 [2-Stage] S1(140개)={s1_val:.5f}/{s1_test:.4f} → S2({br['label']},{len(best_features)}개)={br['val']:.5f}/{auprc:.4f}")
del X_test; gc.collect(); log_memory("최종 완료"); print("\n✅ TGAT v5.4 완료!")

🛡️ [TGAT v5.4] v4 피처 + 2-Stage Pruning 최적화
   ▸ v4 원본 140개 피처 그대로 유지
   ▸ 2-Stage: Importance Pruning + HP 탐색
   ▸ 새 피처 추가 없음 — 순수 최적화 실험

📂 데이터 로딩 중...
⏱️ Train cutoff: 2022-09-09 20:00:00
⏱️ Val   cutoff: 2022-09-14 05:00:00
📊 전체 노드 수: 2,076,999
  💾 [데이터 로드 완료] RAM: 56.85GB | GPU: 0.02GB

📐 [Step 2] 기본 그래프 구조 피처 계산...
  📊 PageRank 계산 중...
  ✅ 기본 그래프 피처: 13개

🔬 [Step 2-A] 구조-조건부 행동 피처 10개...
  ✅ 10개

🎯 [Step 2-B] AML 패턴 점수 5개...
  ✅ 5개
  💾 [피처 완료] RAM: 55.97GB | GPU: 0.02GB

🔗 [Step 3] GNN 엣지/노드 구성 중...
📊 Train 엣지: 19,060,343
📊 GNN 학습 노드: 양성 20,400 + 음성 1,020,000 = 1,040,400
  💾 [GNN 데이터 준비 완료] RAM: 56.50GB | GPU: 0.02GB

🧠 [Step 4] TGAT v3 모델 정의...

🔥 [Step 5] TGAT 학습 시작...
📌 디바이스: cuda
📊 pos_weight: 50.0


  → Epoch  1 | Loss: 1.3702 | LR: 0.000001


  → Epoch  5 | Loss: 1.1564 | LR: 0.000800


  → Epoch 10 | Loss: 1.0870 | LR: 0.000968


  → Epoch 15 | Loss: 1.0610 | LR: 0.000847


...

  → Epoch 30 | Loss: 1.0343 | LR: 0.000232


  → Epoch 35 | Loss: 1.0313 | LR: 0.000080


  → Epoch 40 | Loss: 1.0280 | LR: 0.000012
  ✅ Best model 복원 (loss: 1.0266)

📦 [Step 6] TGAT 임베딩 추출...


Embedding Extraction: 100%|█████████████████████████████████████████████████| 1015/1015 [00:17<00:00, 56.73it/s]


✅ 임베딩 shape: (2076999, 64)

🔗 [Step 6-A] TGAT × PageRank interaction 피처...
  ✅ Interaction 피처 10개
  💾 [피처 완료] RAM: 54.10GB | GPU: 0.02GB

🚀 [Step 7] XGBoost 데이터 구성...
📊 총 피처 수: 140 (v4 동일)
📊 Train: 17,172,747 → 619,437 (양성: 4.76%)
📊 Val  : 5,790,644
📊 Test : 5,792,913
  💾 [XGBoost 준비 완료] RAM: 57.44GB | GPU: 0.02GB

🔥 [Step 8] 2-Stage Pruning + HP 탐색...
📊 scale_pos_weight: 20.0 | 피처: 140

── [Stage 1] 전체 140개 피처 학습 ──
[0]	validation_0-aucpr:0.11812
[100]	validation_0-aucpr:0.32693
[200]	validation_0-aucpr:0.41220
[300]	validation_0-aucpr:0.43272
[400]	validation_0-aucpr:0.44198
[500]	validation_0-aucpr:0.44430
[600]	validation_0-aucpr:0.45073
[700]	validation_0-aucpr:0.45378
[800]	validation_0-aucpr:0.45602
[900]	validation_0-aucpr:0.45996
[1000]	validation_0-aucpr:0.46217
[1100]	validation_0-aucpr:0.46668
[1200]	validation_0-aucpr:0.46891
[1300]	validation_0-aucpr:0.47141
[1400]	validation_0-aucpr:0.47279
[1499]	validation_0-aucpr:0.47524

  Stage 1: Val=0.47524 Test=0.5293 (iter: 1499

In [3]:
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv
from torch_geometric.loader import NeighborLoader
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm
import gc, numpy as np, psutil, os, warnings
warnings.filterwarnings("ignore", category=UserWarning)

def log_memory(tag=""):
    rss = psutil.Process(os.getpid()).memory_info().rss/(1024**3)
    gpu = torch.cuda.memory_allocated()/(1024**3) if torch.cuda.is_available() else 0
    print(f"  💾 [{tag}] RAM: {rss:.2f}GB | GPU: {gpu:.2f}GB")

print("=" * 60)
print("🛡️ [TGAT v7c] Input Enrichment + Edge Enhancement")
print("   ▸ 노드 입력: V2(38)+그래프(13)+조건부(10)+AML(5)+규칙성(7)+금액(8) = 81개")
print("   ▸ Edge 입력: 시간(5)+금액(3)+유형(2) = 10개")
print("   ▸ 모델 구조: v4 TGAT 동일 (TransformerConv 2층)")
print("   ▸ XGBoost: v4 피처 + Enriched TGAT 임베딩 + 2-Stage Pruning")
print("=" * 60)

# ===================== 설정 =====================
NEG_NODE_RATIO_GNN = 50; NEG_ROW_RATIO_XGB = 20; RANDOM_SEED = 42
HIDDEN_DIM = 64; EDGE_DIM = 16; N_HEADS = 4
N_EPOCHS_GNN = 40; BATCH_SIZE = 2048; NUM_NEIGHBORS = [15, 10]
N_INTERACTION_DIMS = 8; PATIENCE = 10; BASE_LR = 0.001; WARMUP_EPOCHS = 5

raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"

# =============================================================
# 1. 데이터 로드
# =============================================================
print("\n📂 데이터 로딩 중...")
df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))
if df_raw["Timestamp"].dtype == pl.Utf8:
    df_raw = df_raw.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_v2 = df_v2.sort("time_group")
total_count = len(df_v2)
train_cutoff = df_v2["time_group"][int(total_count*0.6)]
val_cutoff = df_v2["time_group"][int(total_count*0.8)]
print(f"⏱️ Train cutoff: {train_cutoff}\n⏱️ Val   cutoff: {val_cutoff}")

all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")
num_nodes = len(all_accounts); print(f"📊 전체 노드 수: {num_nodes:,}")
log_memory("데이터 로드 완료")

# =============================================================
# 2. 모든 노드 피처 계산 (GNN 입력용 81개 + XGBoost 겸용)
# =============================================================
print("\n📐 [Step 2] 전체 노드 피처 계산 (81개)...")
df_raw_train = df_raw.filter(pl.col("Timestamp") < train_cutoff)

# ─── 2-1. 그래프 구조 피처 13개 ───
print("  📊 그래프 구조 피처...")
df_from = df_raw_train.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id"))
df_to = df_raw_train.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
out_degree = df_from.group_by("account_id").len().rename({"len":"graph_out_degree"})
in_degree = df_to.group_by("account_id").len().rename({"len":"graph_in_degree"})
unique_out = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("to_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_out_partners"))
unique_in = df_raw_train.select([pl.col("to_acc").cast(pl.Utf8).alias("account_id"),pl.col("from_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_in_partners"))
esf = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("a"),pl.col("to_acc").cast(pl.Utf8).alias("b")]).unique()
est = esf.select([pl.col("b").alias("a"),pl.col("a").alias("b")])
bidir = esf.join(est, on=["a","b"], how="inner")
bidir_count = bidir.select(pl.col("a").alias("account_id")).group_by("account_id").len().rename({"len":"graph_bidir_count"})
del esf, est, bidir; gc.collect()
time_features = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("Timestamp")]).group_by("account_id").agg([
    pl.col("Timestamp").sort().diff().dt.total_seconds().mean().alias("graph_avg_tx_interval"),
    pl.col("Timestamp").sort().diff().dt.total_seconds().std().alias("graph_std_tx_interval"),
    pl.col("Timestamp").count().alias("graph_total_tx_count")])

# PageRank
print("  📊 PageRank...")
account_to_id = dict(zip(all_accounts["account_id"].to_list(), all_accounts["node_id"].to_list()))
src_ids = df_raw_train["from_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
dst_ids = df_raw_train["to_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
vm=(src_ids>=0)&(dst_ids>=0); sv=src_ids[vm]; dv=dst_ids[vm]
oda=np.zeros(num_nodes,dtype=np.float64); np.add.at(oda,sv,1.0); oda=np.maximum(oda,1.0)
pr=np.ones(num_nodes,dtype=np.float64)/num_nodes
for _ in range(10): npr=np.ones(num_nodes,dtype=np.float64)*0.15/num_nodes; np.add.at(npr,dv,0.85*pr[sv]/oda[sv]); pr=npr
pr_df = pl.DataFrame({"node_id":np.arange(num_nodes,dtype=np.uint32),"graph_pagerank":pr.astype(np.float32)})
pr_df = all_accounts.join(pr_df,on="node_id",how="left").select(["account_id","graph_pagerank"])
del src_ids,dst_ids,sv,dv,oda,pr,npr; gc.collect()

df_graph_feats = (all_accounts.select("account_id").join(out_degree,on="account_id",how="left").join(in_degree,on="account_id",how="left")
    .join(unique_out,on="account_id",how="left").join(unique_in,on="account_id",how="left")
    .join(bidir_count,on="account_id",how="left").join(time_features,on="account_id",how="left")
    .join(pr_df,on="account_id",how="left").fill_null(0.0)
    .with_columns([(pl.col("graph_out_degree")+pl.col("graph_in_degree")).alias("graph_total_degree"),
        (pl.col("graph_out_degree")/(pl.col("graph_in_degree")+1)).alias("graph_out_in_ratio"),
        (pl.col("graph_bidir_count")/(pl.col("graph_out_degree")+1)).alias("graph_reciprocity"),
        (pl.col("graph_unique_out_partners")/(pl.col("graph_out_degree")+1)).alias("graph_partner_diversity")]))
graph_feature_cols = [c for c in df_graph_feats.columns if c.startswith("graph_")]
del out_degree,in_degree,unique_out,unique_in,bidir_count,time_features,pr_df,df_from,df_to; gc.collect()
print(f"  ✅ 그래프 피처 {len(graph_feature_cols)}개")

# ─── 2-2. 조건부행동 피처 10개 ───
print("  📊 조건부행동 피처...")
df_v2_train = df_v2.filter(pl.col("time_group")<train_cutoff)
exclude_cols = ["account_id","time_group","is_laundering","mode_format","currency_mode"]
gnn_feature_cols_v2 = [c for c in df_v2.columns if c not in exclude_cols]  # V2 원본 38개
df_v2_agg = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])

df_cond = df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
v2a=set(df_cond.columns)
def sc(n): return pl.col(n) if n in v2a else pl.lit(0.0)
ce=[]
if "cnt_night" in v2a and "cnt_1h" in v2a: ce.append((pl.col("graph_total_degree")*sc("cnt_night")/(sc("cnt_1h")+1)).alias("cond_highdeg_night_intensity"))
if "avg_log_amount" in v2a: ce.append((sc("avg_log_amount")/(pl.col("graph_total_degree").log1p()+1)).alias("cond_lowdeg_high_amount"))
if "ratio_cross_border" in v2a: ce.append((sc("ratio_cross_border")*(1-pl.col("graph_reciprocity"))).alias("cond_oneway_crossborder"))
if "cnt_1h" in v2a: ce.append((pl.col("graph_unique_out_partners")*sc("cnt_1h")/(pl.col("graph_out_degree")+1)).alias("cond_fanout_burst"))
if "cnt_risk_format" in v2a: ce.append((pl.col("graph_pagerank")*sc("cnt_risk_format")).alias("cond_hub_risk_format"))
if "cnt_wire" in v2a: ce.append((pl.col("graph_out_in_ratio")*sc("cnt_wire")).alias("cond_asymmetric_wire"))
if "cnt_inter_bank" in v2a: ce.append((pl.col("graph_std_tx_interval").fill_null(0)/(pl.col("graph_avg_tx_interval").fill_null(1)+1)*sc("cnt_inter_bank")).alias("cond_irregular_interbank"))
if "cnt_currencies" in v2a: ce.append((pl.col("graph_partner_diversity")*sc("cnt_currencies")).alias("cond_diverse_partner_currency"))
if "amount_kurtosis" in v2a: ce.append((pl.col("graph_reciprocity")*sc("amount_kurtosis")).alias("cond_bidir_amount_volatility"))
if "entity_acct_cnt" in v2a and "burst_ratio" in v2a: ce.append((sc("entity_acct_cnt")*sc("burst_ratio")).alias("cond_multi_acct_burst"))
df_cond=df_cond.with_columns(ce); cond_feature_cols=[e.meta.output_name() for e in ce]
for cn in cond_feature_cols: df_cond=df_cond.with_columns(pl.when(pl.col(cn).is_infinite()|pl.col(cn).is_nan()).then(0.0).otherwise(pl.col(cn)).alias(cn))
df_cond_features=df_cond.select(["account_id"]+cond_feature_cols); del df_cond; gc.collect()
print(f"  ✅ 조건부행동 {len(cond_feature_cols)}개")

# ─── 2-3. AML 점수 5개 ───
print("  📊 AML 점수...")
df_sb=df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
se=[((sc("cnt_1h").fill_null(0).clip(0,1e6)+1).log1p()*pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("burst_ratio").fill_null(0).clip(0,1e6)/(sc("avg_log_amount").fill_null(1).clip(0,1e6)+1)).alias("score_smurf"),
    ((1.0/((pl.col("graph_out_in_ratio").clip(0.01,100)-1.0).abs()+0.1))*sc("cnt_wire").fill_null(0).clip(0,1e6).log1p()*(1-pl.col("graph_reciprocity").clip(0,1))/(pl.col("graph_unique_out_partners").clip(0,1e6).log1p()+1)).alias("score_mule"),
    (pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("cnt_currencies").fill_null(0).clip(0,1e6).log1p()*sc("ratio_cross_border").fill_null(0).clip(0,1)/(pl.col("graph_avg_tx_interval").fill_null(3600).clip(0,1e9)/3600+1)).alias("score_layering"),
    (pl.col("graph_in_degree").clip(0,1e6).log1p()*(pl.col("graph_pagerank").clip(0,1)*1e6)*sc("cnt_risk_format").fill_null(0).clip(0,1e6).log1p()/(pl.col("graph_out_degree").clip(0,1e6).log1p()+1)).alias("score_integration")]
df_sb=df_sb.with_columns(se)
sn=["score_smurf","score_mule","score_layering","score_integration"]
for s in sn: df_sb=df_sb.with_columns(pl.when(pl.col(s).is_infinite()|pl.col(s).is_nan()).then(0.0).otherwise(pl.col(s)).alias(s))
for s in sn:
    mn=df_sb[s].min();mx=df_sb[s].max();rng=mx-mn if(mx-mn)>0 else 1.0
    df_sb=df_sb.with_columns(((pl.col(s)-mn)/rng).alias(f"{s}_norm"))
df_sb=df_sb.with_columns((pl.col("score_smurf_norm")*0.3+pl.col("score_mule_norm")*0.25+pl.col("score_layering_norm")*0.3+pl.col("score_integration_norm")*0.15).alias("score_composite"))
aml_score_cols=sn+["score_composite"]
df_aml_scores=df_sb.select(["account_id"]+aml_score_cols)
del df_sb, df_v2_agg; gc.collect()
print(f"  ✅ AML 점수 {len(aml_score_cols)}개")

# ─── 2-4. 규칙성 피처 7개 ───
print("  📊 규칙성 피처...")
df_tx_time = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner"),
    pl.col("Timestamp"),
    pl.col("Timestamp").dt.hour().alias("hour"),
    pl.col("Timestamp").dt.weekday().alias("dow"),
])
reg_agg = df_tx_time.group_by("account_id").agg([
    (pl.col("hour").is_between(9,17).mean()).alias("reg_business_hour_ratio"),
    (pl.col("dow").is_in([5,6]).mean()).alias("reg_weekend_ratio"),
    (pl.col("hour").is_between(0,5).mean()).alias("reg_night_strict_ratio"),
    pl.col("partner").count().alias("_total_tx"),
])
# Hour entropy
hc=df_tx_time.group_by(["account_id","hour"]).len().rename({"len":"h_count"})
hp=hc.pivot(on="hour",index="account_id",values="h_count").fill_null(0)
hcols=[c for c in hp.columns if c!="account_id"]
hmat=hp.select(hcols).to_numpy().astype(np.float64); hsum=np.maximum(hmat.sum(axis=1,keepdims=True),1.0)
hprobs=hmat/hsum; hlog=np.where(hprobs>0,np.log(hprobs+1e-12),0.0)
df_h_ent=pl.DataFrame({"account_id":hp["account_id"],"reg_hour_entropy":-np.sum(hprobs*hlog,axis=1).astype(np.float32)})
del hc,hp,hmat,hprobs,hlog; gc.collect()
# DOW entropy
dc=df_tx_time.group_by(["account_id","dow"]).len().rename({"len":"d_count"})
dp=dc.pivot(on="dow",index="account_id",values="d_count").fill_null(0)
dcols=[c for c in dp.columns if c!="account_id"]
dmat=dp.select(dcols).to_numpy().astype(np.float64); dsum=np.maximum(dmat.sum(axis=1,keepdims=True),1.0)
dprobs=dmat/dsum; dlog=np.where(dprobs>0,np.log(dprobs+1e-12),0.0)
df_d_ent=pl.DataFrame({"account_id":dp["account_id"],"reg_dow_entropy":-np.sum(dprobs*dlog,axis=1).astype(np.float32)})
del dc,dp,dmat,dprobs,dlog; gc.collect()
# Partner consistency + top concentration
pf=df_tx_time.group_by(["account_id","partner"]).len().rename({"len":"p_count"})
rp=pf.filter(pl.col("p_count")>=2).group_by("account_id").len().rename({"len":"repeat_cnt"})
tp2=pf.group_by("account_id").len().rename({"len":"total_cnt"})
df_pc=tp2.join(rp,on="account_id",how="left").fill_null(0).with_columns((pl.col("repeat_cnt")/(pl.col("total_cnt")+1)).alias("reg_consistent_partner_ratio")).select(["account_id","reg_consistent_partner_ratio"])
top_p=pf.sort(["account_id","p_count"],descending=[False,True]).unique(subset=["account_id"],maintain_order=True).rename({"p_count":"top_cnt"}).select(["account_id","top_cnt"])
df_tc=reg_agg.select(["account_id","_total_tx"]).join(top_p,on="account_id",how="left").fill_null(0).with_columns((pl.col("top_cnt")/(pl.col("_total_tx")+1)).alias("reg_top_partner_concentration")).select(["account_id","reg_top_partner_concentration"])
del pf,rp,tp2,top_p,df_tx_time; gc.collect()
df_regularity=(reg_agg.select(["account_id","reg_business_hour_ratio","reg_weekend_ratio","reg_night_strict_ratio"])
    .join(df_h_ent,on="account_id",how="left").join(df_d_ent,on="account_id",how="left")
    .join(df_pc,on="account_id",how="left").join(df_tc,on="account_id",how="left").fill_null(0.0))
regularity_cols=[c for c in df_regularity.columns if c.startswith("reg_")]
for rc in regularity_cols: df_regularity=df_regularity.with_columns(pl.when(pl.col(rc).is_infinite()|pl.col(rc).is_nan()).then(0.0).otherwise(pl.col(rc)).alias(rc))
del reg_agg,df_h_ent,df_d_ent,df_pc,df_tc; gc.collect()
print(f"  ✅ 규칙성 {len(regularity_cols)}개")

# ─── 2-5. 금액 분포 피처 8개 ───
print("  📊 금액 분포 피처...")
amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train.columns: amount_col = cand; break
if amount_col:
    df_amounts=df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col(amount_col).cast(pl.Float64).alias("amount")]).filter(pl.col("amount")>0)
    df_amt=df_amounts.group_by("account_id").agg([
        ((pl.col("amount")%1000==0)|(pl.col("amount")%5000==0)).mean().alias("amt_round_number_ratio"),
        (pl.col("amount").std()/(pl.col("amount").mean()+1)).alias("amt_cv"),
        (pl.col("amount").max()/(pl.col("amount").mean()+1)).alias("amt_max_mean_ratio"),
        (pl.col("amount").median()/(pl.col("amount").mean()+1)).alias("amt_median_mean_ratio"),
        pl.col("amount").quantile(0.9).alias("_q90"), pl.col("amount").sum().alias("_tsum"),
        pl.col("amount").count().alias("_cnt"),
        pl.col("amount").quantile(0.75).alias("_q75"), pl.col("amount").quantile(0.25).alias("_q25"),
        pl.col("amount").median().alias("_med"),
    ]).fill_null(0.0).with_columns([
        (pl.col("_q90")*0.1*pl.col("_cnt")/(pl.col("_tsum")+1)).alias("amt_top10pct_concentration"),
        ((pl.col("_q75")-pl.col("_q25"))/(pl.col("_med")+1)).alias("amt_iqr_median_ratio")])
    gm=df_amounts["amount"].mean(); st=max(gm*0.1,1.0)
    df_sr=df_amounts.with_columns((pl.col("amount")<st).alias("is_small")).group_by("account_id").agg(pl.col("is_small").mean().alias("amt_small_tx_ratio"))
    df_amt=df_amt.join(df_sr,on="account_id",how="left").fill_null(0.0)
    df_al=df_amounts.with_columns((pl.col("amount")+1).log().cast(pl.Int32).alias("log_bucket"))
    bc=df_al.group_by(["account_id","log_bucket"]).len().rename({"len":"b_count"})
    bt=bc.group_by("account_id").agg(pl.col("b_count").sum().alias("b_total"))
    bp=bc.join(bt,on="account_id").with_columns((pl.col("b_count")/pl.col("b_total")).alias("b_prob"))
    ae=bp.group_by("account_id").agg((-1*(pl.col("b_prob")*(pl.col("b_prob")+1e-12).log()).sum()).alias("amt_entropy"))
    df_amt=df_amt.join(ae,on="account_id",how="left").fill_null(0.0)
    del df_amounts,df_al,bc,bt,bp,ae,df_sr; gc.collect()
    amt_feature_cols=[c for c in df_amt.columns if c.startswith("amt_") and not c.startswith("_")]
    df_amt_features=df_amt.select(["account_id"]+amt_feature_cols)
else:
    df_amt_features=all_accounts.select("account_id"); amt_feature_cols=[]
for ac in amt_feature_cols:
    df_amt_features=df_amt_features.with_columns(pl.when(pl.col(ac).is_infinite()|pl.col(ac).is_nan()).then(0.0).otherwise(pl.col(ac)).alias(ac))
print(f"  ✅ 금액 분포 {len(amt_feature_cols)}개")
log_memory("전체 노드 피처 완료")

# =============================================================
# 3. GNN 엣지/노드 구성 — 강화된 입력
# =============================================================
print("\n🔗 [Step 3] GNN 엣지/노드 구성 (Enriched Input)...")

# ─── Edge 구성: 시간(5) + 금액(3) + 유형(2) = 10개 ───
df_raw_reload = pl.read_parquet(raw_path)
if df_raw_reload["Timestamp"].dtype == pl.Utf8:
    df_raw_reload = df_raw_reload.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_raw_train_edges = df_raw_reload.filter(pl.col("Timestamp")<train_cutoff); del df_raw_reload; gc.collect()

# Amount 컬럼 확인
edge_amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train_edges.columns: edge_amount_col = cand; break

# 기본 엣지 DataFrame
df_edges = df_raw_train_edges.select(
    ["from_acc","to_acc","Timestamp"] +
    ([edge_amount_col] if edge_amount_col else []) +
    (["Payment Format"] if "Payment Format" in df_raw_train_edges.columns else []) +
    (["Is Laundering"] if "Is Laundering" in df_raw_train_edges.columns else [])
).with_columns([pl.col("from_acc").cast(pl.Utf8), pl.col("to_acc").cast(pl.Utf8)])

df_edges = df_edges.join(all_accounts.rename({"account_id":"from_acc","node_id":"src_id"}),on="from_acc",how="left")
df_edges = df_edges.join(all_accounts.rename({"account_id":"to_acc","node_id":"dst_id"}),on="to_acc",how="left")

# --- Edge 피처 1: 시간 (기존 5개) ---
ts = df_edges["Timestamp"]; mn_ts=ts.min(); mx_ts=ts.max(); ts_rng=(mx_ts-mn_ts).total_seconds()
etn = ((ts-mn_ts).dt.total_microseconds().cast(pl.Float64)/1e6/max(ts_rng,1.0)).to_numpy().astype(np.float32)
hrs = df_edges["Timestamp"].dt.hour().to_numpy().astype(np.float32)
dow = df_edges["Timestamp"].dt.weekday().to_numpy().astype(np.float32)
time_feats = np.stack([etn, np.sin(2*np.pi*hrs/24), np.cos(2*np.pi*hrs/24),
                        np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7)], axis=1)  # (E, 5)

# --- Edge 피처 2: 금액 (3개) ---
if edge_amount_col:
    print(f"  📊 Edge 금액 피처 ('{edge_amount_col}')...")
    amounts_raw = df_edges[edge_amount_col].cast(pl.Float64).fill_null(0.0).to_numpy().astype(np.float32)
    # (a) log_amount: 정규화된 로그 금액
    log_amounts = np.log1p(np.maximum(amounts_raw, 0)).astype(np.float32)
    log_amounts = log_amounts / (np.max(log_amounts) + 1e-8)  # 0~1 정규화
    # (b) round_flag: 1000/5000 단위 여부
    round_flag = ((amounts_raw % 1000 == 0) | (amounts_raw % 5000 == 0)).astype(np.float32)
    # (c) relative_amount: 전체 평균 대비 비율 (로그 스케일)
    global_mean_amt = np.mean(amounts_raw[amounts_raw > 0]) if np.any(amounts_raw > 0) else 1.0
    relative_amt = np.log1p(amounts_raw / (global_mean_amt + 1e-8)).astype(np.float32)
    relative_amt = relative_amt / (np.max(relative_amt) + 1e-8)
    amount_feats = np.stack([log_amounts, round_flag, relative_amt], axis=1)  # (E, 3)
else:
    print("  ⚠️ Edge 금액 없음, 0으로 채움")
    amount_feats = np.zeros((len(df_edges), 3), dtype=np.float32)

# --- Edge 피처 3: 거래 유형 (2개) ---
# (a) cross_border: 해외 거래 여부 (from_bank != to_bank의 국가)
# (b) inter_bank: 은행 간 거래 여부
# Payment Format에서 추출 시도
if "Payment Format" in df_edges.columns:
    print("  📊 Edge 유형 피처 (Payment Format)...")
    pf_series = df_edges["Payment Format"].cast(pl.Utf8).fill_null("unknown")
    # wire/SWIFT → 해외 가능성 높음
    is_wire = pf_series.str.contains("(?i)wire|swift|cheque").to_numpy().astype(np.float32)
    # ACH/internal → 국내
    is_internal = pf_series.str.contains("(?i)ach|internal|reinvestment").to_numpy().astype(np.float32)
    type_feats = np.stack([is_wire, is_internal], axis=1)  # (E, 2)
else:
    print("  ⚠️ Payment Format 없음, 0으로 채움")
    type_feats = np.zeros((len(df_edges), 2), dtype=np.float32)

# 전체 Edge 피처 결합
edge_features_all = np.concatenate([time_feats, amount_feats, type_feats], axis=1)  # (E, 10)
EDGE_RAW_DIM = edge_features_all.shape[1]
print(f"  📊 Edge 피처: {EDGE_RAW_DIM}개 (시간 5 + 금액 3 + 유형 2)")

edge_index_train = torch.tensor(df_edges.select(["src_id","dst_id"]).to_numpy().T, dtype=torch.long)
edge_attr_train = torch.tensor(edge_features_all, dtype=torch.float32)
print(f"📊 Train 엣지: {edge_index_train.shape[1]:,}")

del df_raw_train, df_raw_train_edges, df_edges, ts, hrs, dow, etn
del time_feats, amount_feats, type_feats, edge_features_all; gc.collect()

# ─── 노드 피처: V2(38) + 그래프(13) + 조건부(10) + AML(5) + 규칙성(7) + 금액(8) = 81개 ───
print("  📊 노드 피처 구성 (81개 → GNN 입력)...")
df_v2_node = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])
df_node_all = (all_accounts.join(df_v2_node, on="account_id", how="left")
    .join(df_graph_feats, on="account_id", how="left")
    .join(df_cond_features, on="account_id", how="left")
    .join(df_aml_scores, on="account_id", how="left")
    .join(df_regularity, on="account_id", how="left")
    .join(df_amt_features, on="account_id", how="left")
    .fill_null(0.0))

# GNN 입력 피처 목록
gnn_input_cols = gnn_feature_cols_v2 + graph_feature_cols + cond_feature_cols + aml_score_cols + regularity_cols + amt_feature_cols

# inf/NaN 처리
for col in gnn_input_cols:
    if col in df_node_all.columns:
        df_node_all = df_node_all.with_columns(
            pl.when(pl.col(col).is_infinite()|pl.col(col).is_nan()).then(0.0).otherwise(pl.col(col)).alias(col))

X_node = torch.tensor(df_node_all.select(gnn_input_cols).to_numpy(), dtype=torch.float32)
IN_DIM = X_node.shape[1]
print(f"  ✅ GNN 노드 입력: {IN_DIM}차원 = V2({len(gnn_feature_cols_v2)})+그래프({len(graph_feature_cols)})+조건부({len(cond_feature_cols)})+AML({len(aml_score_cols)})+규칙성({len(regularity_cols)})+금액({len(amt_feature_cols)})")

# 레이블 + 마스크
target_labels = df_v2_train.filter(pl.col("is_laundering")==1).select("account_id").unique().with_columns(pl.lit(1).alias("label"))
df_labels = all_accounts.join(target_labels,on="account_id",how="left").fill_null(0)
Y_node = torch.tensor(df_labels["label"].to_numpy(), dtype=torch.long)
pos_node_ids = set(df_labels.filter(pl.col("label")==1)["node_id"].to_list())
active_df = pl.DataFrame({"account_id":df_v2_train["account_id"].unique(),"is_active":True})
mask_df = all_accounts.join(active_df,on="account_id",how="left").fill_null(False)
active_node_ids = mask_df.filter(pl.col("is_active"))["node_id"].to_list()
active_pos = [n for n in active_node_ids if n in pos_node_ids]
active_neg = [n for n in active_node_ids if n not in pos_node_ids]
n_neg_gnn = min(len(active_pos)*NEG_NODE_RATIO_GNN, len(active_neg))
np.random.seed(RANDOM_SEED)
sampled_neg = np.random.choice(active_neg,size=n_neg_gnn,replace=False).tolist()
sampled_nodes = set(active_pos+sampled_neg)
train_mask_np = np.zeros(num_nodes, dtype=bool)
for nid in sampled_nodes: train_mask_np[nid]=True
train_mask = torch.tensor(train_mask_np, dtype=torch.bool)
print(f"📊 GNN 노드: 양성 {len(active_pos):,} + 음성 {n_neg_gnn:,}")
del df_v2_node, df_node_all, mask_df, active_df; gc.collect()

graph_data = Data(x=X_node, edge_index=edge_index_train, edge_attr=edge_attr_train, y=Y_node)
graph_data.train_mask = train_mask; graph_data.num_nodes = num_nodes
del X_node, edge_index_train, edge_attr_train, Y_node, train_mask; gc.collect()
log_memory("GNN 데이터 준비 완료")

# =============================================================
# 4. TGAT 모델 (v4 동일 구조, 입력 차원만 확장)
# =============================================================
print("\n🧠 [Step 4] TGAT v3 모델 (Enriched Input)...")

class EdgeProjection(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, out_dim), nn.GELU(), nn.Linear(out_dim, out_dim))
    def forward(self, ea): return self.proj(ea)

class TGAT_v3(nn.Module):
    def __init__(self, in_dim, hidden_dim, edge_raw_dim, edge_dim, n_heads, dropout=0.2):
        super().__init__()
        self.edge_proj = EdgeProjection(edge_raw_dim, edge_dim)
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.conv1 = TransformerConv(hidden_dim, hidden_dim//n_heads, heads=n_heads,
                                      edge_dim=edge_dim, dropout=dropout, concat=True)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.conv2 = TransformerConv(hidden_dim, hidden_dim//n_heads, heads=n_heads,
                                      edge_dim=edge_dim, dropout=dropout, concat=True)
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(nn.Linear(hidden_dim, hidden_dim//2), nn.GELU(),
                                         nn.Dropout(dropout), nn.Linear(hidden_dim//2, 1))
        self.dropout = dropout
    def forward(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        h=h+F.dropout(self.conv1(self.norm1(h),edge_index,ee),p=self.dropout,training=self.training)
        h=h+F.dropout(self.conv2(self.norm2(h),edge_index,ee),p=self.dropout,training=self.training)
        return self.classifier(self.norm_out(h)).squeeze(-1)
    @torch.no_grad()
    def get_embedding(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        h=h+self.conv1(self.norm1(h),edge_index,ee)
        h=h+self.conv2(self.norm2(h),edge_index,ee)
        return self.norm_out(h)

# =============================================================
# 5. GNN 학습
# =============================================================
print("\n🔥 [Step 5] TGAT 학습 (Enriched: 노드 {IN_DIM}d + Edge {EDGE_RAW_DIM}d)...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print(f"📌 디바이스: {device}")

train_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                               input_nodes=graph_data.train_mask, shuffle=True, num_workers=4)
full_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                              input_nodes=None, shuffle=False, num_workers=4)
model = TGAT_v3(in_dim=IN_DIM, hidden_dim=HIDDEN_DIM, edge_raw_dim=EDGE_RAW_DIM,
                edge_dim=EDGE_DIM, n_heads=N_HEADS, dropout=0.2).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"📊 파라미터: {total_params:,} (v4 대비 input_proj만 확장)")
pw=(len(sampled_nodes)-len(active_pos))/max(len(active_pos),1); print(f"📊 pos_weight: {pw:.1f}")
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw],device=device))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6, weight_decay=1e-4)
def set_lr(opt,ep):
    if ep<WARMUP_EPOCHS: lr=1e-6+(BASE_LR-1e-6)*(ep/WARMUP_EPOCHS)
    else: p=(ep-WARMUP_EPOCHS)/max(N_EPOCHS_GNN-WARMUP_EPOCHS,1); lr=1e-5+(BASE_LR-1e-5)*0.5*(1+np.cos(np.pi*p))
    for pg in opt.param_groups: pg['lr']=lr
    return lr
model.train(); best_loss=float('inf'); patience_counter=0; best_state=None
for epoch in range(N_EPOCHS_GNN):
    clr=set_lr(optimizer,epoch); tl=0.0; nb=0
    for batch in tqdm(train_loader,desc=f"Epoch {epoch+1}/{N_EPOCHS_GNN}",leave=False):
        batch=batch.to(device); optimizer.zero_grad()
        bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
        out=model(batch.x,batch.edge_index,bea)
        loss=criterion(out[:batch.batch_size],batch.y[:batch.batch_size].float())
        if torch.isnan(loss) or torch.isinf(loss): optimizer.zero_grad(); continue
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=0.5)
        optimizer.step(); tl+=loss.item(); nb+=1
    al=tl/max(nb,1)
    if (epoch+1)%5==0 or epoch==0: print(f"  → Epoch {epoch+1:2d} | Loss: {al:.4f} | LR: {clr:.6f}")
    if al<best_loss-0.0005: best_loss=al; patience_counter=0; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
    else: patience_counter+=1
    if patience_counter>=PATIENCE: print(f"  ⏹️ Early stopping at epoch {epoch+1}"); break
if best_state: model.load_state_dict({k:v.to(device) for k,v in best_state.items()}); print(f"  ✅ Best model 복원 (loss: {best_loss:.4f})"); del best_state

# =============================================================
# 6. 임베딩 추출 + Interaction
# =============================================================
print("\n📦 [Step 6] 임베딩 추출...")
model.eval(); all_emb=[]
with torch.no_grad():
    for batch in tqdm(full_loader,desc="Embedding Extraction"):
        batch=batch.to(device)
        bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
        emb=model.get_embedding(batch.x,batch.edge_index,bea)
        all_emb.append(emb[:batch.batch_size].cpu())
final_emb=torch.cat(all_emb,dim=0).numpy()
emb_cols=[f"tgat_emb_{i}" for i in range(HIDDEN_DIM)]
df_tgat=pl.concat([all_accounts.select("account_id"),pl.DataFrame(final_emb,schema=emb_cols)],how="horizontal")
print(f"✅ 임베딩: {final_emb.shape}"); del graph_data,all_emb,final_emb; torch.cuda.empty_cache(); gc.collect()

# Interaction
print("🔗 Interaction 피처...")
df_tgat_with_pr=df_tgat.join(df_graph_feats.select(["account_id","graph_pagerank"]),on="account_id",how="left").fill_null(0.0)
emb_np=df_tgat_with_pr.select(emb_cols).to_numpy(); emb_var=np.var(emb_np,axis=0)
top_idx=np.argsort(emb_var)[::-1][:N_INTERACTION_DIMS]; top_ec=[emb_cols[i] for i in top_idx]
ie=[]; ic=[]
for cn in top_ec: nn2=f"interact_pr_x_{cn}"; ie.append((pl.col(cn)*pl.col("graph_pagerank")*1e6).alias(nn2)); ic.append(nn2)
ie.append((sum(pl.col(c)**2 for c in emb_cols).sqrt()*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_norm")); ic.append("interact_pr_x_emb_norm")
ie.append((sum(pl.col(c) for c in top_ec)*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_topsum")); ic.append("interact_pr_x_emb_topsum")
interaction_cols = ic
df_interactions=df_tgat_with_pr.with_columns(ie).select(["account_id"]+ic)
for c in ic: df_interactions=df_interactions.with_columns(pl.when(pl.col(c).is_infinite()|pl.col(c).is_nan()).then(0.0).otherwise(pl.col(c)).alias(c))
print(f"  ✅ Interaction {len(ic)}개"); del df_tgat_with_pr,emb_np; gc.collect()
log_memory("임베딩 완료")

# =============================================================
# 7. XGBoost 데이터 구성 (v4 피처 구조 + Enriched TGAT)
# =============================================================
print("\n🚀 [Step 7] XGBoost 데이터 구성...")
xgb_exclude_cols=["account_id","time_group","is_laundering","mode_format","currency_mode","date"]
xgb_v2_cols=[c for c in df_v2.columns if c not in xgb_exclude_cols]
xgb_feature_cols = xgb_v2_cols+graph_feature_cols+cond_feature_cols+aml_score_cols+emb_cols+interaction_cols
print(f"📊 XGBoost 피처: {len(xgb_feature_cols)} (v4 동일 구조, TGAT 임베딩만 Enriched)")

def build_xgb_df(db):
    df=db.join(df_graph_feats,on="account_id",how="left").join(df_cond_features,on="account_id",how="left").join(df_aml_scores,on="account_id",how="left").join(df_tgat,on="account_id",how="left").join(df_interactions,on="account_id",how="left").fill_null(0.0).fill_nan(0.0)
    for cn in xgb_feature_cols:
        if cn in df.columns: df=df.with_columns(pl.when(pl.col(cn).is_infinite()).then(0.0).otherwise(pl.col(cn)).alias(cn))
    return df

df_tr=build_xgb_df(df_v2.filter(pl.col("time_group")<train_cutoff))
df_tr_pos=df_tr.filter(pl.col("is_laundering")==1); df_tr_neg=df_tr.filter(pl.col("is_laundering")==0)
n_neg=min(len(df_tr_pos)*NEG_ROW_RATIO_XGB,len(df_tr_neg))
df_tr_s=pl.concat([df_tr_pos,df_tr_neg.sample(n=n_neg,seed=RANDOM_SEED)]).sample(fraction=1.0,seed=RANDOM_SEED)
X_train=df_tr_s.select(xgb_feature_cols).to_pandas(); y_train=df_tr_s["is_laundering"].to_numpy()
print(f"📊 Train: {len(df_tr):,} → {len(df_tr_s):,} ({y_train.mean()*100:.2f}%)")
del df_tr,df_tr_pos,df_tr_neg,df_tr_s; gc.collect()
df_vl=build_xgb_df(df_v2.filter((pl.col("time_group")>=train_cutoff)&(pl.col("time_group")<val_cutoff)))
X_val=df_vl.select(xgb_feature_cols).to_pandas(); y_val=df_vl["is_laundering"].to_numpy(); print(f"📊 Val: {len(X_val):,}"); del df_vl; gc.collect()
df_te=build_xgb_df(df_v2.filter(pl.col("time_group")>=val_cutoff))
df_test_meta=df_te.select(["account_id","time_group","is_laundering"])
X_test=df_te.select(xgb_feature_cols).to_pandas(); y_test=df_te["is_laundering"].to_numpy(); print(f"📊 Test: {len(X_test):,}")
del df_te,df_v2,df_tgat,df_graph_feats,df_cond_features,df_aml_scores,df_interactions,df_regularity,df_amt_features; gc.collect()
log_memory("XGBoost 준비")

# =============================================================
# 8. XGBoost 2-Stage Pruning
# =============================================================
print("\n🔥 [Step 8] 2-Stage Pruning...")
apr=y_train.sum()/len(y_train); spw=max((1-apr)/apr,1.0)

# Stage 1
s1p={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda","random_state":42,"max_depth":8,"learning_rate":0.03,"scale_pos_weight":spw,"subsample":0.8,"colsample_bytree":0.7,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
m_s1=xgb.XGBClassifier(**s1p,n_estimators=1500,early_stopping_rounds=100)
m_s1.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=100)
s1_val=m_s1.best_score; s1_test=average_precision_score(y_test,m_s1.predict_proba(X_test)[:,1])
print(f"\n  Stage1: Val={s1_val:.5f} Test={s1_test:.4f}")
imp_s1=m_s1.feature_importances_; feat_imp=sorted(zip(xgb_feature_cols,imp_s1),key=lambda x:x[1],reverse=True)
del m_s1; gc.collect()

# Stage 2
bp2={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda","random_state":42,"scale_pos_weight":spw,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
configs=[]
for nf in [30,40,50,60]:
    sel=[n for n,_ in feat_imp[:nf]]
    configs.append({"label":f"top{nf}_base","features":sel,"hp":{"max_depth":8,"learning_rate":0.03,"colsample_bytree":0.7,"subsample":0.8,"min_child_weight":5,"n_estimators":1500}})
    configs.append({"label":f"top{nf}_lr02","features":sel,"hp":{"max_depth":8,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"n_estimators":2500}})
configs.append({"label":"all","features":xgb_feature_cols,"hp":{"max_depth":8,"learning_rate":0.03,"colsample_bytree":0.7,"subsample":0.8,"min_child_weight":5,"n_estimators":1500}})

print(f"\n📊 {len(configs)}개 조합 탐색...")
best_s2={"val":-1}
for cfg in configs:
    hp=cfg["hp"].copy(); ne=hp.pop("n_estimators"); sf=cfg["features"]
    m=xgb.XGBClassifier(**{**bp2,**hp},n_estimators=ne,early_stopping_rounds=100)
    m.fit(X_train[sf],y_train,eval_set=[(X_val[sf],y_val)],verbose=0)
    va=m.best_score
    print(f"  {cfg['label']:<18s} ({len(sf):3d}개) Val={va:.5f}")
    if va>best_s2["val"]:
        if best_s2.get("model"): del best_s2["model"]
        best_s2={"label":cfg["label"],"features":sf,"val":va,"iter":m.best_iteration,"model":m}
    else: del m
    gc.collect()

y_prob=best_s2["model"].predict_proba(X_test[best_s2["features"]])[:,1]
best_features=best_s2["features"]
print(f"\n  🏆 Best: {best_s2['label']} ({len(best_features)}개) Val={best_s2['val']:.5f}")

# Importance
imp=best_s2["model"].feature_importances_; tidx=np.argsort(imp)[::-1][:25]
print(f"\n🔍 Feature Importance Top 25")
for i,idx in enumerate(tidx):
    c=best_features[idx]; t=""
    if c.startswith("tgat_emb_"): t=" [TGAT]"
    elif c.startswith("graph_"): t=" [GRAPH]"
    elif c.startswith("cond_"): t=" [COND]"
    elif c.startswith("score_"): t=" [SCORE]"
    elif c.startswith("interact_"): t=" [INTER]"
    print(f"  {i+1:2d}. {c:<40s}: {imp[idx]:.4f}{t}")
ti=imp.sum()
v2i=sum(imp[i] for i,c in enumerate(best_features) if c in xgb_v2_cols)
gi=sum(imp[i] for i,c in enumerate(best_features) if c.startswith("graph_"))
ei=sum(imp[i] for i,c in enumerate(best_features) if c.startswith("tgat_emb_"))
print(f"\n📊 그룹별: V2={v2i/ti*100:.1f}% 그래프={gi/ti*100:.1f}% TGAT(enriched)={ei/ti*100:.1f}%")
del X_train,X_val; gc.collect()

# =============================================================
# 9. 최종 리포트
# =============================================================
print("\n"+"="*60); print("🏆 [TGAT v7c] Input Enrichment + Edge Enhancement 리포트"); print("="*60)
auprc=average_precision_score(y_test,y_prob); print(f"\n  AUPRC: {auprc:.4f}")
best_f1=0; best_thresh=0
for th in np.arange(0.05,0.95,0.01):
    f=f1_score(y_test,(y_prob>=th).astype(int),zero_division=0)
    if f>best_f1: best_f1=f; best_thresh=th
y_pred=(y_prob>=best_thresh).astype(int)
prec=precision_score(y_test,y_pred,zero_division=0); rec=recall_score(y_test,y_pred,zero_division=0)
print(f"\n📌 최적: T={best_thresh:.2f} F1={best_f1:.4f} P={prec:.4f} R={rec:.4f}")
print(f"\n📌 임계값별:")
for t in [0.1,0.3,0.5,0.7,0.8,0.9]:
    yp=(y_prob>=t).astype(int)
    print(f"  T={t:.1f} P={precision_score(y_test,yp,zero_division=0):.4f} R={recall_score(y_test,yp,zero_division=0):.4f} F1={f1_score(y_test,yp,zero_division=0):.4f}")

df_res=df_test_meta.with_columns([pl.col("time_group").dt.date().alias("date"),pl.Series("pred_prob",y_prob)])
df_dist=df_res.sort("pred_prob",descending=True).unique(subset=["account_id"],maintain_order=True)
print(f"\n📌 Top-K:")
for k in [100,500,1000,5000,10000]:
    ck=min(k,len(df_dist)); hits=df_dist.head(ck)["is_laundering"].sum()
    print(f"  Top-{k:5d}: {hits:5d}명 ({hits/ck*100:.2f}%)")

print(f"\n📌 일별 Top-100:")
for d in df_dist["date"].unique(maintain_order=True).sort(descending=True):
    hits=df_dist.filter(pl.col("date")==d).head(100)["is_laundering"].sum()
    print(f"  {d}: {hits}")

tn,fp,fn,tp=confusion_matrix(y_test,y_pred).ravel()
print(f"\n📌 v4→v7c: AUPRC 0.5370→{auprc:.4f} ({'↑' if auprc>0.537 else '↓'}{abs(auprc-0.537):.4f})")
print(f"  F1 0.4940→{best_f1:.4f} | TP:{tp:,} FP:{fp:,} FN:{fn:,} TN:{tn:,}")
print(f"\n📌 Enrichment 상세:")
print(f"  노드 입력: v4 38d → v7c {IN_DIM}d (+{IN_DIM-38} 피처)")
print(f"  Edge 입력: v4 5d → v7c {EDGE_RAW_DIM}d (+{EDGE_RAW_DIM-5} 피처)")
print(f"  모델 구조: 동일 (TransformerConv 2층)")
print(f"\n📌 2-Stage: S1({len(xgb_feature_cols)}개)={s1_val:.5f}/{s1_test:.4f} → S2({best_s2['label']},{len(best_features)}개)={best_s2['val']:.5f}/{auprc:.4f}")

del X_test; gc.collect(); log_memory("최종 완료")
print(f"\n✅ TGAT v7c 완료!")

🛡️ [TGAT v7c] Input Enrichment + Edge Enhancement
   ▸ 노드 입력: V2(38)+그래프(13)+조건부(10)+AML(5)+규칙성(7)+금액(8) = 81개
   ▸ Edge 입력: 시간(5)+금액(3)+유형(2) = 10개
   ▸ 모델 구조: v4 TGAT 동일 (TransformerConv 2층)
   ▸ XGBoost: v4 피처 + Enriched TGAT 임베딩 + 2-Stage Pruning

📂 데이터 로딩 중...
⏱️ Train cutoff: 2022-09-09 20:00:00
⏱️ Val   cutoff: 2022-09-14 05:00:00
📊 전체 노드 수: 2,076,999
  💾 [데이터 로드 완료] RAM: 71.60GB | GPU: 0.02GB

📐 [Step 2] 전체 노드 피처 계산 (81개)...
  📊 그래프 구조 피처...
  📊 PageRank...
  ✅ 그래프 피처 13개
  📊 조건부행동 피처...
  ✅ 조건부행동 10개
  📊 AML 점수...
  ✅ AML 점수 5개
  📊 규칙성 피처...
  ✅ 규칙성 7개
  📊 금액 분포 피처...
  ✅ 금액 분포 8개
  💾 [전체 노드 피처 완료] RAM: 72.13GB | GPU: 0.02GB

🔗 [Step 3] GNN 엣지/노드 구성 (Enriched Input)...
  📊 Edge 금액 피처 ('Amount Received')...
  📊 Edge 유형 피처 (Payment Format)...
  📊 Edge 피처: 10개 (시간 5 + 금액 3 + 유형 2)
📊 Train 엣지: 19,060,343
  📊 노드 피처 구성 (81개 → GNN 입력)...
  ✅ GNN 노드 입력: 81차원 = V2(38)+그래프(13)+조건부(10)+AML(5)+규칙성(7)+금액(8)
📊 GNN 노드: 양성 20,400 + 음성 1,020,000
  💾 [GNN 데이터 준비 완료] RAM: 71.87GB | GPU: 0.02GB



  → Epoch  1 | Loss: 1.3528 | LR: 0.000001


  → Epoch  5 | Loss: 1.1086 | LR: 0.000800


  → Epoch 10 | Loss: 1.0622 | LR: 0.000968


  → Epoch 15 | Loss: 1.0362 | LR: 0.000847


...

  → Epoch 30 | Loss: 1.0087 | LR: 0.000232


  → Epoch 35 | Loss: 1.0124 | LR: 0.000080


  → Epoch 40 | Loss: 1.0002 | LR: 0.000012
  ✅ Best model 복원 (loss: 0.9999)

📦 [Step 6] 임베딩 추출...


Embedding Extraction: 100%|█████████████████████████████████████████████████| 1015/1015 [00:19<00:00, 53.19it/s]


✅ 임베딩: (2076999, 64)
🔗 Interaction 피처...
  ✅ Interaction 10개
  💾 [임베딩 완료] RAM: 71.68GB | GPU: 0.02GB

🚀 [Step 7] XGBoost 데이터 구성...
📊 XGBoost 피처: 140 (v4 동일 구조, TGAT 임베딩만 Enriched)
📊 Train: 17,172,747 → 619,437 (4.76%)
📊 Val: 5,790,644
📊 Test: 5,792,913
  💾 [XGBoost 준비] RAM: 82.55GB | GPU: 0.02GB

🔥 [Step 8] 2-Stage Pruning...
[0]	validation_0-aucpr:0.08094
[100]	validation_0-aucpr:0.32701
[200]	validation_0-aucpr:0.41072
[300]	validation_0-aucpr:0.43000
[400]	validation_0-aucpr:0.44102
[500]	validation_0-aucpr:0.44481
[600]	validation_0-aucpr:0.45336
[700]	validation_0-aucpr:0.45526
[800]	validation_0-aucpr:0.45956
[900]	validation_0-aucpr:0.46321
[1000]	validation_0-aucpr:0.46431
[1100]	validation_0-aucpr:0.46795
[1200]	validation_0-aucpr:0.46977
[1300]	validation_0-aucpr:0.47256
[1400]	validation_0-aucpr:0.47452
[1499]	validation_0-aucpr:0.47460

  Stage1: Val=0.47520 Test=0.5325

📊 9개 조합 탐색...
  top30_base         ( 30개) Val=0.44789
  top30_lr02         ( 30개) Val=0.44963
  top40_ba

In [4]:
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv
from torch_geometric.loader import NeighborLoader
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm
import gc, numpy as np, psutil, os, warnings
warnings.filterwarnings("ignore", category=UserWarning)

def log_memory(tag=""):
    rss = psutil.Process(os.getpid()).memory_info().rss/(1024**3)
    gpu = torch.cuda.memory_allocated()/(1024**3) if torch.cuda.is_available() else 0
    print(f"  💾 [{tag}] RAM: {rss:.2f}GB | GPU: {gpu:.2f}GB")

print("=" * 60)
print("🛡️ [TGAT v7c.1] Input Enrichment + Edge Enhancement + HP Grid")
print("   ▸ 노드 입력: V2(38)+그래프(13)+조건부(10)+AML(5)+규칙성(7)+금액(8) = 81개")
print("   ▸ Edge 입력: 시간(5)+금액(3)+유형(2) = 10개")
print("   ▸ 모델 구조: v4 TGAT 동일 (TransformerConv 2층)")
print("   ▸ XGBoost: 2-Stage Pruning + 확장 HP 그리드 (7×5+1=36조합)")
print("=" * 60)

# ===================== 설정 =====================
NEG_NODE_RATIO_GNN = 50; NEG_ROW_RATIO_XGB = 20; RANDOM_SEED = 42
HIDDEN_DIM = 64; EDGE_DIM = 16; N_HEADS = 4
N_EPOCHS_GNN = 40; BATCH_SIZE = 2048; NUM_NEIGHBORS = [15, 10]
N_INTERACTION_DIMS = 8; PATIENCE = 10; BASE_LR = 0.001; WARMUP_EPOCHS = 5

raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"

# =============================================================
# 1. 데이터 로드
# =============================================================
print("\n📂 데이터 로딩 중...")
df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))
if df_raw["Timestamp"].dtype == pl.Utf8:
    df_raw = df_raw.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_v2 = df_v2.sort("time_group")
total_count = len(df_v2)
train_cutoff = df_v2["time_group"][int(total_count*0.6)]
val_cutoff = df_v2["time_group"][int(total_count*0.8)]
print(f"⏱️ Train cutoff: {train_cutoff}\n⏱️ Val   cutoff: {val_cutoff}")

all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")
num_nodes = len(all_accounts); print(f"📊 전체 노드 수: {num_nodes:,}")
log_memory("데이터 로드 완료")

# =============================================================
# 2. 모든 노드 피처 계산 (GNN 입력용 81개 + XGBoost 겸용)
# =============================================================
print("\n📐 [Step 2] 전체 노드 피처 계산 (81개)...")
df_raw_train = df_raw.filter(pl.col("Timestamp") < train_cutoff)

# ─── 2-1. 그래프 구조 피처 13개 ───
print("  📊 그래프 구조 피처...")
df_from = df_raw_train.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id"))
df_to = df_raw_train.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
out_degree = df_from.group_by("account_id").len().rename({"len":"graph_out_degree"})
in_degree = df_to.group_by("account_id").len().rename({"len":"graph_in_degree"})
unique_out = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("to_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_out_partners"))
unique_in = df_raw_train.select([pl.col("to_acc").cast(pl.Utf8).alias("account_id"),pl.col("from_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_in_partners"))
esf = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("a"),pl.col("to_acc").cast(pl.Utf8).alias("b")]).unique()
est = esf.select([pl.col("b").alias("a"),pl.col("a").alias("b")])
bidir = esf.join(est, on=["a","b"], how="inner")
bidir_count = bidir.select(pl.col("a").alias("account_id")).group_by("account_id").len().rename({"len":"graph_bidir_count"})
del esf, est, bidir; gc.collect()
time_features = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("Timestamp")]).group_by("account_id").agg([
    pl.col("Timestamp").sort().diff().dt.total_seconds().mean().alias("graph_avg_tx_interval"),
    pl.col("Timestamp").sort().diff().dt.total_seconds().std().alias("graph_std_tx_interval"),
    pl.col("Timestamp").count().alias("graph_total_tx_count")])

# PageRank
print("  📊 PageRank...")
account_to_id = dict(zip(all_accounts["account_id"].to_list(), all_accounts["node_id"].to_list()))
src_ids = df_raw_train["from_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
dst_ids = df_raw_train["to_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
vm=(src_ids>=0)&(dst_ids>=0); sv=src_ids[vm]; dv=dst_ids[vm]
oda=np.zeros(num_nodes,dtype=np.float64); np.add.at(oda,sv,1.0); oda=np.maximum(oda,1.0)
pr=np.ones(num_nodes,dtype=np.float64)/num_nodes
for _ in range(10): npr=np.ones(num_nodes,dtype=np.float64)*0.15/num_nodes; np.add.at(npr,dv,0.85*pr[sv]/oda[sv]); pr=npr
pr_df = pl.DataFrame({"node_id":np.arange(num_nodes,dtype=np.uint32),"graph_pagerank":pr.astype(np.float32)})
pr_df = all_accounts.join(pr_df,on="node_id",how="left").select(["account_id","graph_pagerank"])
del src_ids,dst_ids,sv,dv,oda,pr,npr; gc.collect()

df_graph_feats = (all_accounts.select("account_id").join(out_degree,on="account_id",how="left").join(in_degree,on="account_id",how="left")
    .join(unique_out,on="account_id",how="left").join(unique_in,on="account_id",how="left")
    .join(bidir_count,on="account_id",how="left").join(time_features,on="account_id",how="left")
    .join(pr_df,on="account_id",how="left").fill_null(0.0)
    .with_columns([(pl.col("graph_out_degree")+pl.col("graph_in_degree")).alias("graph_total_degree"),
        (pl.col("graph_out_degree")/(pl.col("graph_in_degree")+1)).alias("graph_out_in_ratio"),
        (pl.col("graph_bidir_count")/(pl.col("graph_out_degree")+1)).alias("graph_reciprocity"),
        (pl.col("graph_unique_out_partners")/(pl.col("graph_out_degree")+1)).alias("graph_partner_diversity")]))
graph_feature_cols = [c for c in df_graph_feats.columns if c.startswith("graph_")]
del out_degree,in_degree,unique_out,unique_in,bidir_count,time_features,pr_df,df_from,df_to; gc.collect()
print(f"  ✅ 그래프 피처 {len(graph_feature_cols)}개")

# ─── 2-2. 조건부행동 피처 10개 ───
print("  📊 조건부행동 피처...")
df_v2_train = df_v2.filter(pl.col("time_group")<train_cutoff)
exclude_cols = ["account_id","time_group","is_laundering","mode_format","currency_mode"]
gnn_feature_cols_v2 = [c for c in df_v2.columns if c not in exclude_cols]  # V2 원본 38개
df_v2_agg = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])

df_cond = df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
v2a=set(df_cond.columns)
def sc(n): return pl.col(n) if n in v2a else pl.lit(0.0)
ce=[]
if "cnt_night" in v2a and "cnt_1h" in v2a: ce.append((pl.col("graph_total_degree")*sc("cnt_night")/(sc("cnt_1h")+1)).alias("cond_highdeg_night_intensity"))
if "avg_log_amount" in v2a: ce.append((sc("avg_log_amount")/(pl.col("graph_total_degree").log1p()+1)).alias("cond_lowdeg_high_amount"))
if "ratio_cross_border" in v2a: ce.append((sc("ratio_cross_border")*(1-pl.col("graph_reciprocity"))).alias("cond_oneway_crossborder"))
if "cnt_1h" in v2a: ce.append((pl.col("graph_unique_out_partners")*sc("cnt_1h")/(pl.col("graph_out_degree")+1)).alias("cond_fanout_burst"))
if "cnt_risk_format" in v2a: ce.append((pl.col("graph_pagerank")*sc("cnt_risk_format")).alias("cond_hub_risk_format"))
if "cnt_wire" in v2a: ce.append((pl.col("graph_out_in_ratio")*sc("cnt_wire")).alias("cond_asymmetric_wire"))
if "cnt_inter_bank" in v2a: ce.append((pl.col("graph_std_tx_interval").fill_null(0)/(pl.col("graph_avg_tx_interval").fill_null(1)+1)*sc("cnt_inter_bank")).alias("cond_irregular_interbank"))
if "cnt_currencies" in v2a: ce.append((pl.col("graph_partner_diversity")*sc("cnt_currencies")).alias("cond_diverse_partner_currency"))
if "amount_kurtosis" in v2a: ce.append((pl.col("graph_reciprocity")*sc("amount_kurtosis")).alias("cond_bidir_amount_volatility"))
if "entity_acct_cnt" in v2a and "burst_ratio" in v2a: ce.append((sc("entity_acct_cnt")*sc("burst_ratio")).alias("cond_multi_acct_burst"))
df_cond=df_cond.with_columns(ce); cond_feature_cols=[e.meta.output_name() for e in ce]
for cn in cond_feature_cols: df_cond=df_cond.with_columns(pl.when(pl.col(cn).is_infinite()|pl.col(cn).is_nan()).then(0.0).otherwise(pl.col(cn)).alias(cn))
df_cond_features=df_cond.select(["account_id"]+cond_feature_cols); del df_cond; gc.collect()
print(f"  ✅ 조건부행동 {len(cond_feature_cols)}개")

# ─── 2-3. AML 점수 5개 ───
print("  📊 AML 점수...")
df_sb=df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
se=[((sc("cnt_1h").fill_null(0).clip(0,1e6)+1).log1p()*pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("burst_ratio").fill_null(0).clip(0,1e6)/(sc("avg_log_amount").fill_null(1).clip(0,1e6)+1)).alias("score_smurf"),
    ((1.0/((pl.col("graph_out_in_ratio").clip(0.01,100)-1.0).abs()+0.1))*sc("cnt_wire").fill_null(0).clip(0,1e6).log1p()*(1-pl.col("graph_reciprocity").clip(0,1))/(pl.col("graph_unique_out_partners").clip(0,1e6).log1p()+1)).alias("score_mule"),
    (pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("cnt_currencies").fill_null(0).clip(0,1e6).log1p()*sc("ratio_cross_border").fill_null(0).clip(0,1)/(pl.col("graph_avg_tx_interval").fill_null(3600).clip(0,1e9)/3600+1)).alias("score_layering"),
    (pl.col("graph_in_degree").clip(0,1e6).log1p()*(pl.col("graph_pagerank").clip(0,1)*1e6)*sc("cnt_risk_format").fill_null(0).clip(0,1e6).log1p()/(pl.col("graph_out_degree").clip(0,1e6).log1p()+1)).alias("score_integration")]
df_sb=df_sb.with_columns(se)
sn=["score_smurf","score_mule","score_layering","score_integration"]
for s in sn: df_sb=df_sb.with_columns(pl.when(pl.col(s).is_infinite()|pl.col(s).is_nan()).then(0.0).otherwise(pl.col(s)).alias(s))
for s in sn:
    mn=df_sb[s].min();mx=df_sb[s].max();rng=mx-mn if(mx-mn)>0 else 1.0
    df_sb=df_sb.with_columns(((pl.col(s)-mn)/rng).alias(f"{s}_norm"))
df_sb=df_sb.with_columns((pl.col("score_smurf_norm")*0.3+pl.col("score_mule_norm")*0.25+pl.col("score_layering_norm")*0.3+pl.col("score_integration_norm")*0.15).alias("score_composite"))
aml_score_cols=sn+["score_composite"]
df_aml_scores=df_sb.select(["account_id"]+aml_score_cols)
del df_sb, df_v2_agg; gc.collect()
print(f"  ✅ AML 점수 {len(aml_score_cols)}개")

# ─── 2-4. 규칙성 피처 7개 ───
print("  📊 규칙성 피처...")
df_tx_time = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner"),
    pl.col("Timestamp"),
    pl.col("Timestamp").dt.hour().alias("hour"),
    pl.col("Timestamp").dt.weekday().alias("dow"),
])
reg_agg = df_tx_time.group_by("account_id").agg([
    (pl.col("hour").is_between(9,17).mean()).alias("reg_business_hour_ratio"),
    (pl.col("dow").is_in([5,6]).mean()).alias("reg_weekend_ratio"),
    (pl.col("hour").is_between(0,5).mean()).alias("reg_night_strict_ratio"),
    pl.col("partner").count().alias("_total_tx"),
])
# Hour entropy
hc=df_tx_time.group_by(["account_id","hour"]).len().rename({"len":"h_count"})
hp=hc.pivot(on="hour",index="account_id",values="h_count").fill_null(0)
hcols=[c for c in hp.columns if c!="account_id"]
hmat=hp.select(hcols).to_numpy().astype(np.float64); hsum=np.maximum(hmat.sum(axis=1,keepdims=True),1.0)
hprobs=hmat/hsum; hlog=np.where(hprobs>0,np.log(hprobs+1e-12),0.0)
df_h_ent=pl.DataFrame({"account_id":hp["account_id"],"reg_hour_entropy":-np.sum(hprobs*hlog,axis=1).astype(np.float32)})
del hc,hp,hmat,hprobs,hlog; gc.collect()
# DOW entropy
dc=df_tx_time.group_by(["account_id","dow"]).len().rename({"len":"d_count"})
dp=dc.pivot(on="dow",index="account_id",values="d_count").fill_null(0)
dcols=[c for c in dp.columns if c!="account_id"]
dmat=dp.select(dcols).to_numpy().astype(np.float64); dsum=np.maximum(dmat.sum(axis=1,keepdims=True),1.0)
dprobs=dmat/dsum; dlog=np.where(dprobs>0,np.log(dprobs+1e-12),0.0)
df_d_ent=pl.DataFrame({"account_id":dp["account_id"],"reg_dow_entropy":-np.sum(dprobs*dlog,axis=1).astype(np.float32)})
del dc,dp,dmat,dprobs,dlog; gc.collect()
# Partner consistency + top concentration
pf=df_tx_time.group_by(["account_id","partner"]).len().rename({"len":"p_count"})
rp=pf.filter(pl.col("p_count")>=2).group_by("account_id").len().rename({"len":"repeat_cnt"})
tp2=pf.group_by("account_id").len().rename({"len":"total_cnt"})
df_pc=tp2.join(rp,on="account_id",how="left").fill_null(0).with_columns((pl.col("repeat_cnt")/(pl.col("total_cnt")+1)).alias("reg_consistent_partner_ratio")).select(["account_id","reg_consistent_partner_ratio"])
top_p=pf.sort(["account_id","p_count"],descending=[False,True]).unique(subset=["account_id"],maintain_order=True).rename({"p_count":"top_cnt"}).select(["account_id","top_cnt"])
df_tc=reg_agg.select(["account_id","_total_tx"]).join(top_p,on="account_id",how="left").fill_null(0).with_columns((pl.col("top_cnt")/(pl.col("_total_tx")+1)).alias("reg_top_partner_concentration")).select(["account_id","reg_top_partner_concentration"])
del pf,rp,tp2,top_p,df_tx_time; gc.collect()
df_regularity=(reg_agg.select(["account_id","reg_business_hour_ratio","reg_weekend_ratio","reg_night_strict_ratio"])
    .join(df_h_ent,on="account_id",how="left").join(df_d_ent,on="account_id",how="left")
    .join(df_pc,on="account_id",how="left").join(df_tc,on="account_id",how="left").fill_null(0.0))
regularity_cols=[c for c in df_regularity.columns if c.startswith("reg_")]
for rc in regularity_cols: df_regularity=df_regularity.with_columns(pl.when(pl.col(rc).is_infinite()|pl.col(rc).is_nan()).then(0.0).otherwise(pl.col(rc)).alias(rc))
del reg_agg,df_h_ent,df_d_ent,df_pc,df_tc; gc.collect()
print(f"  ✅ 규칙성 {len(regularity_cols)}개")

# ─── 2-5. 금액 분포 피처 8개 ───
print("  📊 금액 분포 피처...")
amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train.columns: amount_col = cand; break
if amount_col:
    df_amounts=df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col(amount_col).cast(pl.Float64).alias("amount")]).filter(pl.col("amount")>0)
    df_amt=df_amounts.group_by("account_id").agg([
        ((pl.col("amount")%1000==0)|(pl.col("amount")%5000==0)).mean().alias("amt_round_number_ratio"),
        (pl.col("amount").std()/(pl.col("amount").mean()+1)).alias("amt_cv"),
        (pl.col("amount").max()/(pl.col("amount").mean()+1)).alias("amt_max_mean_ratio"),
        (pl.col("amount").median()/(pl.col("amount").mean()+1)).alias("amt_median_mean_ratio"),
        pl.col("amount").quantile(0.9).alias("_q90"), pl.col("amount").sum().alias("_tsum"),
        pl.col("amount").count().alias("_cnt"),
        pl.col("amount").quantile(0.75).alias("_q75"), pl.col("amount").quantile(0.25).alias("_q25"),
        pl.col("amount").median().alias("_med"),
    ]).fill_null(0.0).with_columns([
        (pl.col("_q90")*0.1*pl.col("_cnt")/(pl.col("_tsum")+1)).alias("amt_top10pct_concentration"),
        ((pl.col("_q75")-pl.col("_q25"))/(pl.col("_med")+1)).alias("amt_iqr_median_ratio")])
    gm=df_amounts["amount"].mean(); st=max(gm*0.1,1.0)
    df_sr=df_amounts.with_columns((pl.col("amount")<st).alias("is_small")).group_by("account_id").agg(pl.col("is_small").mean().alias("amt_small_tx_ratio"))
    df_amt=df_amt.join(df_sr,on="account_id",how="left").fill_null(0.0)
    df_al=df_amounts.with_columns((pl.col("amount")+1).log().cast(pl.Int32).alias("log_bucket"))
    bc=df_al.group_by(["account_id","log_bucket"]).len().rename({"len":"b_count"})
    bt=bc.group_by("account_id").agg(pl.col("b_count").sum().alias("b_total"))
    bp=bc.join(bt,on="account_id").with_columns((pl.col("b_count")/pl.col("b_total")).alias("b_prob"))
    ae=bp.group_by("account_id").agg((-1*(pl.col("b_prob")*(pl.col("b_prob")+1e-12).log()).sum()).alias("amt_entropy"))
    df_amt=df_amt.join(ae,on="account_id",how="left").fill_null(0.0)
    del df_amounts,df_al,bc,bt,bp,ae,df_sr; gc.collect()
    amt_feature_cols=[c for c in df_amt.columns if c.startswith("amt_") and not c.startswith("_")]
    df_amt_features=df_amt.select(["account_id"]+amt_feature_cols)
else:
    df_amt_features=all_accounts.select("account_id"); amt_feature_cols=[]
for ac in amt_feature_cols:
    df_amt_features=df_amt_features.with_columns(pl.when(pl.col(ac).is_infinite()|pl.col(ac).is_nan()).then(0.0).otherwise(pl.col(ac)).alias(ac))
print(f"  ✅ 금액 분포 {len(amt_feature_cols)}개")
log_memory("전체 노드 피처 완료")

# =============================================================
# 3. GNN 엣지/노드 구성 — 강화된 입력
# =============================================================
print("\n🔗 [Step 3] GNN 엣지/노드 구성 (Enriched Input)...")

# ─── Edge 구성: 시간(5) + 금액(3) + 유형(2) = 10개 ───
df_raw_reload = pl.read_parquet(raw_path)
if df_raw_reload["Timestamp"].dtype == pl.Utf8:
    df_raw_reload = df_raw_reload.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_raw_train_edges = df_raw_reload.filter(pl.col("Timestamp")<train_cutoff); del df_raw_reload; gc.collect()

# Amount 컬럼 확인
edge_amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train_edges.columns: edge_amount_col = cand; break

# 기본 엣지 DataFrame
df_edges = df_raw_train_edges.select(
    ["from_acc","to_acc","Timestamp"] +
    ([edge_amount_col] if edge_amount_col else []) +
    (["Payment Format"] if "Payment Format" in df_raw_train_edges.columns else []) +
    (["Is Laundering"] if "Is Laundering" in df_raw_train_edges.columns else [])
).with_columns([pl.col("from_acc").cast(pl.Utf8), pl.col("to_acc").cast(pl.Utf8)])

df_edges = df_edges.join(all_accounts.rename({"account_id":"from_acc","node_id":"src_id"}),on="from_acc",how="left")
df_edges = df_edges.join(all_accounts.rename({"account_id":"to_acc","node_id":"dst_id"}),on="to_acc",how="left")

# --- Edge 피처 1: 시간 (기존 5개) ---
ts = df_edges["Timestamp"]; mn_ts=ts.min(); mx_ts=ts.max(); ts_rng=(mx_ts-mn_ts).total_seconds()
etn = ((ts-mn_ts).dt.total_microseconds().cast(pl.Float64)/1e6/max(ts_rng,1.0)).to_numpy().astype(np.float32)
hrs = df_edges["Timestamp"].dt.hour().to_numpy().astype(np.float32)
dow = df_edges["Timestamp"].dt.weekday().to_numpy().astype(np.float32)
time_feats = np.stack([etn, np.sin(2*np.pi*hrs/24), np.cos(2*np.pi*hrs/24),
                        np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7)], axis=1)  # (E, 5)

# --- Edge 피처 2: 금액 (3개) ---
if edge_amount_col:
    print(f"  📊 Edge 금액 피처 ('{edge_amount_col}')...")
    amounts_raw = df_edges[edge_amount_col].cast(pl.Float64).fill_null(0.0).to_numpy().astype(np.float32)
    # (a) log_amount: 정규화된 로그 금액
    log_amounts = np.log1p(np.maximum(amounts_raw, 0)).astype(np.float32)
    log_amounts = log_amounts / (np.max(log_amounts) + 1e-8)  # 0~1 정규화
    # (b) round_flag: 1000/5000 단위 여부
    round_flag = ((amounts_raw % 1000 == 0) | (amounts_raw % 5000 == 0)).astype(np.float32)
    # (c) relative_amount: 전체 평균 대비 비율 (로그 스케일)
    global_mean_amt = np.mean(amounts_raw[amounts_raw > 0]) if np.any(amounts_raw > 0) else 1.0
    relative_amt = np.log1p(amounts_raw / (global_mean_amt + 1e-8)).astype(np.float32)
    relative_amt = relative_amt / (np.max(relative_amt) + 1e-8)
    amount_feats = np.stack([log_amounts, round_flag, relative_amt], axis=1)  # (E, 3)
else:
    print("  ⚠️ Edge 금액 없음, 0으로 채움")
    amount_feats = np.zeros((len(df_edges), 3), dtype=np.float32)

# --- Edge 피처 3: 거래 유형 (2개) ---
# (a) cross_border: 해외 거래 여부 (from_bank != to_bank의 국가)
# (b) inter_bank: 은행 간 거래 여부
# Payment Format에서 추출 시도
if "Payment Format" in df_edges.columns:
    print("  📊 Edge 유형 피처 (Payment Format)...")
    pf_series = df_edges["Payment Format"].cast(pl.Utf8).fill_null("unknown")
    # wire/SWIFT → 해외 가능성 높음
    is_wire = pf_series.str.contains("(?i)wire|swift|cheque").to_numpy().astype(np.float32)
    # ACH/internal → 국내
    is_internal = pf_series.str.contains("(?i)ach|internal|reinvestment").to_numpy().astype(np.float32)
    type_feats = np.stack([is_wire, is_internal], axis=1)  # (E, 2)
else:
    print("  ⚠️ Payment Format 없음, 0으로 채움")
    type_feats = np.zeros((len(df_edges), 2), dtype=np.float32)

# 전체 Edge 피처 결합
edge_features_all = np.concatenate([time_feats, amount_feats, type_feats], axis=1)  # (E, 10)
EDGE_RAW_DIM = edge_features_all.shape[1]
print(f"  📊 Edge 피처: {EDGE_RAW_DIM}개 (시간 5 + 금액 3 + 유형 2)")

edge_index_train = torch.tensor(df_edges.select(["src_id","dst_id"]).to_numpy().T, dtype=torch.long)
edge_attr_train = torch.tensor(edge_features_all, dtype=torch.float32)
print(f"📊 Train 엣지: {edge_index_train.shape[1]:,}")

del df_raw_train, df_raw_train_edges, df_edges, ts, hrs, dow, etn
del time_feats, amount_feats, type_feats, edge_features_all; gc.collect()

# ─── 노드 피처: V2(38) + 그래프(13) + 조건부(10) + AML(5) + 규칙성(7) + 금액(8) = 81개 ───
print("  📊 노드 피처 구성 (81개 → GNN 입력)...")
df_v2_node = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])
df_node_all = (all_accounts.join(df_v2_node, on="account_id", how="left")
    .join(df_graph_feats, on="account_id", how="left")
    .join(df_cond_features, on="account_id", how="left")
    .join(df_aml_scores, on="account_id", how="left")
    .join(df_regularity, on="account_id", how="left")
    .join(df_amt_features, on="account_id", how="left")
    .fill_null(0.0))

# GNN 입력 피처 목록
gnn_input_cols = gnn_feature_cols_v2 + graph_feature_cols + cond_feature_cols + aml_score_cols + regularity_cols + amt_feature_cols

# inf/NaN 처리
for col in gnn_input_cols:
    if col in df_node_all.columns:
        df_node_all = df_node_all.with_columns(
            pl.when(pl.col(col).is_infinite()|pl.col(col).is_nan()).then(0.0).otherwise(pl.col(col)).alias(col))

X_node = torch.tensor(df_node_all.select(gnn_input_cols).to_numpy(), dtype=torch.float32)
IN_DIM = X_node.shape[1]
print(f"  ✅ GNN 노드 입력: {IN_DIM}차원 = V2({len(gnn_feature_cols_v2)})+그래프({len(graph_feature_cols)})+조건부({len(cond_feature_cols)})+AML({len(aml_score_cols)})+규칙성({len(regularity_cols)})+금액({len(amt_feature_cols)})")

# 레이블 + 마스크
target_labels = df_v2_train.filter(pl.col("is_laundering")==1).select("account_id").unique().with_columns(pl.lit(1).alias("label"))
df_labels = all_accounts.join(target_labels,on="account_id",how="left").fill_null(0)
Y_node = torch.tensor(df_labels["label"].to_numpy(), dtype=torch.long)
pos_node_ids = set(df_labels.filter(pl.col("label")==1)["node_id"].to_list())
active_df = pl.DataFrame({"account_id":df_v2_train["account_id"].unique(),"is_active":True})
mask_df = all_accounts.join(active_df,on="account_id",how="left").fill_null(False)
active_node_ids = mask_df.filter(pl.col("is_active"))["node_id"].to_list()
active_pos = [n for n in active_node_ids if n in pos_node_ids]
active_neg = [n for n in active_node_ids if n not in pos_node_ids]
n_neg_gnn = min(len(active_pos)*NEG_NODE_RATIO_GNN, len(active_neg))
np.random.seed(RANDOM_SEED)
sampled_neg = np.random.choice(active_neg,size=n_neg_gnn,replace=False).tolist()
sampled_nodes = set(active_pos+sampled_neg)
train_mask_np = np.zeros(num_nodes, dtype=bool)
for nid in sampled_nodes: train_mask_np[nid]=True
train_mask = torch.tensor(train_mask_np, dtype=torch.bool)
print(f"📊 GNN 노드: 양성 {len(active_pos):,} + 음성 {n_neg_gnn:,}")
del df_v2_node, df_node_all, mask_df, active_df; gc.collect()

graph_data = Data(x=X_node, edge_index=edge_index_train, edge_attr=edge_attr_train, y=Y_node)
graph_data.train_mask = train_mask; graph_data.num_nodes = num_nodes
del X_node, edge_index_train, edge_attr_train, Y_node, train_mask; gc.collect()
log_memory("GNN 데이터 준비 완료")

# =============================================================
# 4. TGAT 모델 (v4 동일 구조, 입력 차원만 확장)
# =============================================================
print("\n🧠 [Step 4] TGAT v3 모델 (Enriched Input)...")

class EdgeProjection(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, out_dim), nn.GELU(), nn.Linear(out_dim, out_dim))
    def forward(self, ea): return self.proj(ea)

class TGAT_v3(nn.Module):
    def __init__(self, in_dim, hidden_dim, edge_raw_dim, edge_dim, n_heads, dropout=0.2):
        super().__init__()
        self.edge_proj = EdgeProjection(edge_raw_dim, edge_dim)
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.conv1 = TransformerConv(hidden_dim, hidden_dim//n_heads, heads=n_heads,
                                      edge_dim=edge_dim, dropout=dropout, concat=True)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.conv2 = TransformerConv(hidden_dim, hidden_dim//n_heads, heads=n_heads,
                                      edge_dim=edge_dim, dropout=dropout, concat=True)
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(nn.Linear(hidden_dim, hidden_dim//2), nn.GELU(),
                                         nn.Dropout(dropout), nn.Linear(hidden_dim//2, 1))
        self.dropout = dropout
    def forward(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        h=h+F.dropout(self.conv1(self.norm1(h),edge_index,ee),p=self.dropout,training=self.training)
        h=h+F.dropout(self.conv2(self.norm2(h),edge_index,ee),p=self.dropout,training=self.training)
        return self.classifier(self.norm_out(h)).squeeze(-1)
    @torch.no_grad()
    def get_embedding(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        h=h+self.conv1(self.norm1(h),edge_index,ee)
        h=h+self.conv2(self.norm2(h),edge_index,ee)
        return self.norm_out(h)

# =============================================================
# 5. GNN 학습
# =============================================================
print("\n🔥 [Step 5] TGAT 학습 (Enriched: 노드 {IN_DIM}d + Edge {EDGE_RAW_DIM}d)...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print(f"📌 디바이스: {device}")

train_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                               input_nodes=graph_data.train_mask, shuffle=True, num_workers=4)
full_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                              input_nodes=None, shuffle=False, num_workers=4)
model = TGAT_v3(in_dim=IN_DIM, hidden_dim=HIDDEN_DIM, edge_raw_dim=EDGE_RAW_DIM,
                edge_dim=EDGE_DIM, n_heads=N_HEADS, dropout=0.2).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"📊 파라미터: {total_params:,} (v4 대비 input_proj만 확장)")
pw=(len(sampled_nodes)-len(active_pos))/max(len(active_pos),1); print(f"📊 pos_weight: {pw:.1f}")
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw],device=device))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6, weight_decay=1e-4)
def set_lr(opt,ep):
    if ep<WARMUP_EPOCHS: lr=1e-6+(BASE_LR-1e-6)*(ep/WARMUP_EPOCHS)
    else: p=(ep-WARMUP_EPOCHS)/max(N_EPOCHS_GNN-WARMUP_EPOCHS,1); lr=1e-5+(BASE_LR-1e-5)*0.5*(1+np.cos(np.pi*p))
    for pg in opt.param_groups: pg['lr']=lr
    return lr
model.train(); best_loss=float('inf'); patience_counter=0; best_state=None
for epoch in range(N_EPOCHS_GNN):
    clr=set_lr(optimizer,epoch); tl=0.0; nb=0
    for batch in tqdm(train_loader,desc=f"Epoch {epoch+1}/{N_EPOCHS_GNN}",leave=False):
        batch=batch.to(device); optimizer.zero_grad()
        bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
        out=model(batch.x,batch.edge_index,bea)
        loss=criterion(out[:batch.batch_size],batch.y[:batch.batch_size].float())
        if torch.isnan(loss) or torch.isinf(loss): optimizer.zero_grad(); continue
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=0.5)
        optimizer.step(); tl+=loss.item(); nb+=1
    al=tl/max(nb,1)
    if (epoch+1)%5==0 or epoch==0: print(f"  → Epoch {epoch+1:2d} | Loss: {al:.4f} | LR: {clr:.6f}")
    if al<best_loss-0.0005: best_loss=al; patience_counter=0; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
    else: patience_counter+=1
    if patience_counter>=PATIENCE: print(f"  ⏹️ Early stopping at epoch {epoch+1}"); break
if best_state: model.load_state_dict({k:v.to(device) for k,v in best_state.items()}); print(f"  ✅ Best model 복원 (loss: {best_loss:.4f})"); del best_state

# =============================================================
# 6. 임베딩 추출 + Interaction
# =============================================================
print("\n📦 [Step 6] 임베딩 추출...")
model.eval(); all_emb=[]
with torch.no_grad():
    for batch in tqdm(full_loader,desc="Embedding Extraction"):
        batch=batch.to(device)
        bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
        emb=model.get_embedding(batch.x,batch.edge_index,bea)
        all_emb.append(emb[:batch.batch_size].cpu())
final_emb=torch.cat(all_emb,dim=0).numpy()
emb_cols=[f"tgat_emb_{i}" for i in range(HIDDEN_DIM)]
df_tgat=pl.concat([all_accounts.select("account_id"),pl.DataFrame(final_emb,schema=emb_cols)],how="horizontal")
print(f"✅ 임베딩: {final_emb.shape}"); del graph_data,all_emb,final_emb; torch.cuda.empty_cache(); gc.collect()

# Interaction
print("🔗 Interaction 피처...")
df_tgat_with_pr=df_tgat.join(df_graph_feats.select(["account_id","graph_pagerank"]),on="account_id",how="left").fill_null(0.0)
emb_np=df_tgat_with_pr.select(emb_cols).to_numpy(); emb_var=np.var(emb_np,axis=0)
top_idx=np.argsort(emb_var)[::-1][:N_INTERACTION_DIMS]; top_ec=[emb_cols[i] for i in top_idx]
ie=[]; ic=[]
for cn in top_ec: nn2=f"interact_pr_x_{cn}"; ie.append((pl.col(cn)*pl.col("graph_pagerank")*1e6).alias(nn2)); ic.append(nn2)
ie.append((sum(pl.col(c)**2 for c in emb_cols).sqrt()*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_norm")); ic.append("interact_pr_x_emb_norm")
ie.append((sum(pl.col(c) for c in top_ec)*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_topsum")); ic.append("interact_pr_x_emb_topsum")
interaction_cols = ic
df_interactions=df_tgat_with_pr.with_columns(ie).select(["account_id"]+ic)
for c in ic: df_interactions=df_interactions.with_columns(pl.when(pl.col(c).is_infinite()|pl.col(c).is_nan()).then(0.0).otherwise(pl.col(c)).alias(c))
print(f"  ✅ Interaction {len(ic)}개"); del df_tgat_with_pr,emb_np; gc.collect()
log_memory("임베딩 완료")

# =============================================================
# 7. XGBoost 데이터 구성 (v4 피처 구조 + Enriched TGAT)
# =============================================================
print("\n🚀 [Step 7] XGBoost 데이터 구성...")
xgb_exclude_cols=["account_id","time_group","is_laundering","mode_format","currency_mode","date"]
xgb_v2_cols=[c for c in df_v2.columns if c not in xgb_exclude_cols]
xgb_feature_cols = xgb_v2_cols+graph_feature_cols+cond_feature_cols+aml_score_cols+emb_cols+interaction_cols
print(f"📊 XGBoost 피처: {len(xgb_feature_cols)} (v4 동일 구조, TGAT 임베딩만 Enriched)")

def build_xgb_df(db):
    df=db.join(df_graph_feats,on="account_id",how="left").join(df_cond_features,on="account_id",how="left").join(df_aml_scores,on="account_id",how="left").join(df_tgat,on="account_id",how="left").join(df_interactions,on="account_id",how="left").fill_null(0.0).fill_nan(0.0)
    for cn in xgb_feature_cols:
        if cn in df.columns: df=df.with_columns(pl.when(pl.col(cn).is_infinite()).then(0.0).otherwise(pl.col(cn)).alias(cn))
    return df

df_tr=build_xgb_df(df_v2.filter(pl.col("time_group")<train_cutoff))
df_tr_pos=df_tr.filter(pl.col("is_laundering")==1); df_tr_neg=df_tr.filter(pl.col("is_laundering")==0)
n_neg=min(len(df_tr_pos)*NEG_ROW_RATIO_XGB,len(df_tr_neg))
df_tr_s=pl.concat([df_tr_pos,df_tr_neg.sample(n=n_neg,seed=RANDOM_SEED)]).sample(fraction=1.0,seed=RANDOM_SEED)
X_train=df_tr_s.select(xgb_feature_cols).to_pandas(); y_train=df_tr_s["is_laundering"].to_numpy()
print(f"📊 Train: {len(df_tr):,} → {len(df_tr_s):,} ({y_train.mean()*100:.2f}%)")
del df_tr,df_tr_pos,df_tr_neg,df_tr_s; gc.collect()
df_vl=build_xgb_df(df_v2.filter((pl.col("time_group")>=train_cutoff)&(pl.col("time_group")<val_cutoff)))
X_val=df_vl.select(xgb_feature_cols).to_pandas(); y_val=df_vl["is_laundering"].to_numpy(); print(f"📊 Val: {len(X_val):,}"); del df_vl; gc.collect()
df_te=build_xgb_df(df_v2.filter(pl.col("time_group")>=val_cutoff))
df_test_meta=df_te.select(["account_id","time_group","is_laundering"])
X_test=df_te.select(xgb_feature_cols).to_pandas(); y_test=df_te["is_laundering"].to_numpy(); print(f"📊 Test: {len(X_test):,}")
del df_te,df_v2,df_tgat,df_graph_feats,df_cond_features,df_aml_scores,df_interactions,df_regularity,df_amt_features; gc.collect()
log_memory("XGBoost 준비")

# =============================================================
# 8. XGBoost 2-Stage Pruning
# =============================================================
print("\n🔥 [Step 8] 2-Stage Pruning...")
apr=y_train.sum()/len(y_train); spw=max((1-apr)/apr,1.0)

# Stage 1
s1p={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda","random_state":42,"max_depth":8,"learning_rate":0.03,"scale_pos_weight":spw,"subsample":0.8,"colsample_bytree":0.7,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
m_s1=xgb.XGBClassifier(**s1p,n_estimators=1500,early_stopping_rounds=100)
m_s1.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=100)
s1_val=m_s1.best_score; s1_test=average_precision_score(y_test,m_s1.predict_proba(X_test)[:,1])
print(f"\n  Stage1: Val={s1_val:.5f} Test={s1_test:.4f}")
imp_s1=m_s1.feature_importances_; feat_imp=sorted(zip(xgb_feature_cols,imp_s1),key=lambda x:x[1],reverse=True)
del m_s1; gc.collect()

# Stage 2 — 확장된 HP 그리드
bp2={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda","random_state":42,"scale_pos_weight":spw}
configs=[]

# v7c에서 top60이 최적이었으므로 50~70 범위를 더 세밀하게 + 다양한 HP
for nf in [45, 50, 55, 60, 65, 70, 80]:
    sel=[n for n,_ in feat_imp[:nf]]

    # (A) v4 기본
    configs.append({"label":f"top{nf}_d8_c07","features":sel,
        "hp":{"max_depth":8,"learning_rate":0.03,"colsample_bytree":0.7,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":1500}})

    # (B) 느린 LR + colsample 높임 (피처 적으니 더 많이 사용)
    configs.append({"label":f"top{nf}_lr02_c08","features":sel,
        "hp":{"max_depth":8,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}})

    # (C) colsample 0.9 (피처 줄었으니 거의 다 쓰기)
    configs.append({"label":f"top{nf}_lr02_c09","features":sel,
        "hp":{"max_depth":8,"learning_rate":0.02,"colsample_bytree":0.9,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}})

    # (D) depth 9 (더 깊은 트리 — enriched 임베딩이 복잡한 패턴 포착)
    configs.append({"label":f"top{nf}_d9_lr02","features":sel,
        "hp":{"max_depth":9,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}})

    # (E) 강한 정규화 (reg_alpha/lambda 증가)
    configs.append({"label":f"top{nf}_reg_strong","features":sel,
        "hp":{"max_depth":8,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.75,"min_child_weight":8,"gamma":0.2,"reg_alpha":0.5,"reg_lambda":2.0,"n_estimators":2500}})

# 전체 피처 baseline
configs.append({"label":"all_140","features":xgb_feature_cols,
    "hp":{"max_depth":8,"learning_rate":0.03,"colsample_bytree":0.7,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":1500}})

print(f"\n📊 {len(configs)}개 조합 탐색...")
print(f"  {'#':>2s} {'Label':<22s} | {'#f':>3s} {'d':>2s} {'LR':>5s} {'col':>4s} {'mcw':>3s} | {'ValAUCPR':>9s} {'it':>5s}")
print("  "+"-"*70)

best_s2={"val":-1}
for i, cfg in enumerate(configs):
    hp=cfg["hp"].copy(); ne=hp.pop("n_estimators"); sf=cfg["features"]
    m=xgb.XGBClassifier(**{**bp2,**hp},n_estimators=ne,early_stopping_rounds=100)
    m.fit(X_train[sf],y_train,eval_set=[(X_val[sf],y_val)],verbose=0)
    va=m.best_score; bi=m.best_iteration
    print(f"  {i+1:2d} {cfg['label']:<22s} | {len(sf):3d} {hp.get('max_depth',8):2d} {hp.get('learning_rate',0.03):5.3f} {hp.get('colsample_bytree',0.7):4.2f} {hp.get('min_child_weight',5):3d} | {va:9.5f} {bi:5d}")
    if va>best_s2["val"]:
        if best_s2.get("model"): del best_s2["model"]
        best_s2={"label":cfg["label"],"features":sf,"val":va,"iter":bi,"model":m,"hp":hp}
    else: del m
    gc.collect()

y_prob=best_s2["model"].predict_proba(X_test[best_s2["features"]])[:,1]
best_features=best_s2["features"]
print(f"\n  🏆 Best: {best_s2['label']} ({len(best_features)}개) Val={best_s2['val']:.5f} iter={best_s2['iter']}")
print(f"     HP: {best_s2.get('hp',{})}")

# Importance
imp=best_s2["model"].feature_importances_; tidx=np.argsort(imp)[::-1][:25]
print(f"\n🔍 Feature Importance Top 25")
for i,idx in enumerate(tidx):
    c=best_features[idx]; t=""
    if c.startswith("tgat_emb_"): t=" [TGAT]"
    elif c.startswith("graph_"): t=" [GRAPH]"
    elif c.startswith("cond_"): t=" [COND]"
    elif c.startswith("score_"): t=" [SCORE]"
    elif c.startswith("interact_"): t=" [INTER]"
    print(f"  {i+1:2d}. {c:<40s}: {imp[idx]:.4f}{t}")
ti=imp.sum()
v2i=sum(imp[i] for i,c in enumerate(best_features) if c in xgb_v2_cols)
gi=sum(imp[i] for i,c in enumerate(best_features) if c.startswith("graph_"))
ei=sum(imp[i] for i,c in enumerate(best_features) if c.startswith("tgat_emb_"))
print(f"\n📊 그룹별: V2={v2i/ti*100:.1f}% 그래프={gi/ti*100:.1f}% TGAT(enriched)={ei/ti*100:.1f}%")
del X_train,X_val; gc.collect()

# =============================================================
# 9. 최종 리포트
# =============================================================
print("\n"+"="*60); print("🏆 [TGAT v7c.1] Input Enrichment + HP Grid 리포트"); print("="*60)
auprc=average_precision_score(y_test,y_prob); print(f"\n  AUPRC: {auprc:.4f}")
best_f1=0; best_thresh=0
for th in np.arange(0.05,0.95,0.01):
    f=f1_score(y_test,(y_prob>=th).astype(int),zero_division=0)
    if f>best_f1: best_f1=f; best_thresh=th
y_pred=(y_prob>=best_thresh).astype(int)
prec=precision_score(y_test,y_pred,zero_division=0); rec=recall_score(y_test,y_pred,zero_division=0)
print(f"\n📌 최적: T={best_thresh:.2f} F1={best_f1:.4f} P={prec:.4f} R={rec:.4f}")
print(f"\n📌 임계값별:")
for t in [0.1,0.3,0.5,0.7,0.8,0.9]:
    yp=(y_prob>=t).astype(int)
    print(f"  T={t:.1f} P={precision_score(y_test,yp,zero_division=0):.4f} R={recall_score(y_test,yp,zero_division=0):.4f} F1={f1_score(y_test,yp,zero_division=0):.4f}")

df_res=df_test_meta.with_columns([pl.col("time_group").dt.date().alias("date"),pl.Series("pred_prob",y_prob)])
df_dist=df_res.sort("pred_prob",descending=True).unique(subset=["account_id"],maintain_order=True)
print(f"\n📌 Top-K:")
for k in [100,500,1000,5000,10000]:
    ck=min(k,len(df_dist)); hits=df_dist.head(ck)["is_laundering"].sum()
    print(f"  Top-{k:5d}: {hits:5d}명 ({hits/ck*100:.2f}%)")

print(f"\n📌 일별 Top-100:")
for d in df_dist["date"].unique(maintain_order=True).sort(descending=True):
    hits=df_dist.filter(pl.col("date")==d).head(100)["is_laundering"].sum()
    print(f"  {d}: {hits}")

tn,fp,fn,tp=confusion_matrix(y_test,y_pred).ravel()
print(f"\n📌 v4→v7c: AUPRC 0.5370→{auprc:.4f} ({'↑' if auprc>0.537 else '↓'}{abs(auprc-0.537):.4f})")
print(f"  F1 0.4940→{best_f1:.4f} | TP:{tp:,} FP:{fp:,} FN:{fn:,} TN:{tn:,}")
print(f"\n📌 Enrichment 상세:")
print(f"  노드 입력: v4 38d → v7c {IN_DIM}d (+{IN_DIM-38} 피처)")
print(f"  Edge 입력: v4 5d → v7c {EDGE_RAW_DIM}d (+{EDGE_RAW_DIM-5} 피처)")
print(f"  모델 구조: 동일 (TransformerConv 2층)")
print(f"\n📌 2-Stage: S1({len(xgb_feature_cols)}개)={s1_val:.5f}/{s1_test:.4f} → S2({best_s2['label']},{len(best_features)}개)={best_s2['val']:.5f}/{auprc:.4f}")

del X_test; gc.collect(); log_memory("최종 완료")
print(f"\n✅ TGAT v7c 완료!")

🛡️ [TGAT v7c.1] Input Enrichment + Edge Enhancement + HP Grid
   ▸ 노드 입력: V2(38)+그래프(13)+조건부(10)+AML(5)+규칙성(7)+금액(8) = 81개
   ▸ Edge 입력: 시간(5)+금액(3)+유형(2) = 10개
   ▸ 모델 구조: v4 TGAT 동일 (TransformerConv 2층)
   ▸ XGBoost: 2-Stage Pruning + 확장 HP 그리드 (7×5+1=36조합)

📂 데이터 로딩 중...
⏱️ Train cutoff: 2022-09-09 20:00:00
⏱️ Val   cutoff: 2022-09-14 05:00:00
📊 전체 노드 수: 2,076,999
  💾 [데이터 로드 완료] RAM: 73.25GB | GPU: 0.02GB

📐 [Step 2] 전체 노드 피처 계산 (81개)...
  📊 그래프 구조 피처...
  📊 PageRank...
  ✅ 그래프 피처 13개
  📊 조건부행동 피처...
  ✅ 조건부행동 10개
  📊 AML 점수...
  ✅ AML 점수 5개
  📊 규칙성 피처...
  ✅ 규칙성 7개
  📊 금액 분포 피처...
  ✅ 금액 분포 8개
  💾 [전체 노드 피처 완료] RAM: 73.25GB | GPU: 0.02GB

🔗 [Step 3] GNN 엣지/노드 구성 (Enriched Input)...
  📊 Edge 금액 피처 ('Amount Received')...
  📊 Edge 유형 피처 (Payment Format)...
  📊 Edge 피처: 10개 (시간 5 + 금액 3 + 유형 2)
📊 Train 엣지: 19,060,343
  📊 노드 피처 구성 (81개 → GNN 입력)...
  ✅ GNN 노드 입력: 81차원 = V2(38)+그래프(13)+조건부(10)+AML(5)+규칙성(7)+금액(8)
📊 GNN 노드: 양성 20,400 + 음성 1,020,000
  💾 [GNN 데이터 준비 완료] RAM: 74.91GB | GPU:

  → Epoch  1 | Loss: 1.3629 | LR: 0.000001


  → Epoch  5 | Loss: 1.1089 | LR: 0.000800


  → Epoch 10 | Loss: 1.0643 | LR: 0.000968


  → Epoch 15 | Loss: 1.0399 | LR: 0.000847


...

  → Epoch 30 | Loss: 1.0095 | LR: 0.000232


  → Epoch 35 | Loss: 1.0032 | LR: 0.000080


  → Epoch 40 | Loss: 1.0042 | LR: 0.000012
  ✅ Best model 복원 (loss: 1.0032)

📦 [Step 6] 임베딩 추출...


Embedding Extraction: 100%|█████████████████████████████████████████████████| 1015/1015 [00:18<00:00, 54.79it/s]


✅ 임베딩: (2076999, 64)
🔗 Interaction 피처...
  ✅ Interaction 10개
  💾 [임베딩 완료] RAM: 73.97GB | GPU: 0.02GB

🚀 [Step 7] XGBoost 데이터 구성...
📊 XGBoost 피처: 140 (v4 동일 구조, TGAT 임베딩만 Enriched)
📊 Train: 17,172,747 → 619,437 (4.76%)
📊 Val: 5,790,644
📊 Test: 5,792,913
  💾 [XGBoost 준비] RAM: 83.61GB | GPU: 0.02GB

🔥 [Step 8] 2-Stage Pruning...
[0]	validation_0-aucpr:0.11696
[100]	validation_0-aucpr:0.32623
[200]	validation_0-aucpr:0.41234
[300]	validation_0-aucpr:0.43617
[400]	validation_0-aucpr:0.44567
[500]	validation_0-aucpr:0.45160
[600]	validation_0-aucpr:0.45503
[700]	validation_0-aucpr:0.46003
[800]	validation_0-aucpr:0.46308
[900]	validation_0-aucpr:0.46488
[1000]	validation_0-aucpr:0.46689
[1100]	validation_0-aucpr:0.47039
[1176]	validation_0-aucpr:0.47025

  Stage1: Val=0.47063 Test=0.5324

📊 36개 조합 탐색...
   # Label                  |  #f  d    LR  col mcw |  ValAUCPR    it
  ----------------------------------------------------------------------
   1 top45_d8_c07           |  45  8 0.030 0.70 

In [5]:
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv
from torch_geometric.loader import NeighborLoader
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm
import gc, numpy as np, psutil, os, warnings
warnings.filterwarnings("ignore", category=UserWarning)

def log_memory(tag=""):
    rss = psutil.Process(os.getpid()).memory_info().rss/(1024**3)
    gpu = torch.cuda.memory_allocated()/(1024**3) if torch.cuda.is_available() else 0
    print(f"  💾 [{tag}] RAM: {rss:.2f}GB | GPU: {gpu:.2f}GB")

print("=" * 60)
print("🛡️ [TGAT v7c.2] Input+Edge Enrichment + GNN 탐색 + HP Grid")
print("   ▸ 노드 81d + Edge 10d (v7c 동일)")
print("   ▸ GNN 탐색: hidden 64/128 × layer 2/3 = 4개 설정")
print("   ▸ XGBoost: 확장 HP 그리드 (7×5+1=36조합)")
print("=" * 60)

# ===================== 설정 =====================
NEG_NODE_RATIO_GNN = 50; NEG_ROW_RATIO_XGB = 20; RANDOM_SEED = 42
HIDDEN_DIM = 64; EDGE_DIM = 16; N_HEADS = 4
N_EPOCHS_GNN = 40; BATCH_SIZE = 2048; NUM_NEIGHBORS = [15, 10]
N_INTERACTION_DIMS = 8; PATIENCE = 10; BASE_LR = 0.001; WARMUP_EPOCHS = 5

raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"

# =============================================================
# 1. 데이터 로드
# =============================================================
print("\n📂 데이터 로딩 중...")
df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))
if df_raw["Timestamp"].dtype == pl.Utf8:
    df_raw = df_raw.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_v2 = df_v2.sort("time_group")
total_count = len(df_v2)
train_cutoff = df_v2["time_group"][int(total_count*0.6)]
val_cutoff = df_v2["time_group"][int(total_count*0.8)]
print(f"⏱️ Train cutoff: {train_cutoff}\n⏱️ Val   cutoff: {val_cutoff}")

all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")
num_nodes = len(all_accounts); print(f"📊 전체 노드 수: {num_nodes:,}")
log_memory("데이터 로드 완료")

# =============================================================
# 2. 모든 노드 피처 계산 (GNN 입력용 81개 + XGBoost 겸용)
# =============================================================
print("\n📐 [Step 2] 전체 노드 피처 계산 (81개)...")
df_raw_train = df_raw.filter(pl.col("Timestamp") < train_cutoff)

# ─── 2-1. 그래프 구조 피처 13개 ───
print("  📊 그래프 구조 피처...")
df_from = df_raw_train.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id"))
df_to = df_raw_train.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
out_degree = df_from.group_by("account_id").len().rename({"len":"graph_out_degree"})
in_degree = df_to.group_by("account_id").len().rename({"len":"graph_in_degree"})
unique_out = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("to_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_out_partners"))
unique_in = df_raw_train.select([pl.col("to_acc").cast(pl.Utf8).alias("account_id"),pl.col("from_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_in_partners"))
esf = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("a"),pl.col("to_acc").cast(pl.Utf8).alias("b")]).unique()
est = esf.select([pl.col("b").alias("a"),pl.col("a").alias("b")])
bidir = esf.join(est, on=["a","b"], how="inner")
bidir_count = bidir.select(pl.col("a").alias("account_id")).group_by("account_id").len().rename({"len":"graph_bidir_count"})
del esf, est, bidir; gc.collect()
time_features = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("Timestamp")]).group_by("account_id").agg([
    pl.col("Timestamp").sort().diff().dt.total_seconds().mean().alias("graph_avg_tx_interval"),
    pl.col("Timestamp").sort().diff().dt.total_seconds().std().alias("graph_std_tx_interval"),
    pl.col("Timestamp").count().alias("graph_total_tx_count")])

# PageRank
print("  📊 PageRank...")
account_to_id = dict(zip(all_accounts["account_id"].to_list(), all_accounts["node_id"].to_list()))
src_ids = df_raw_train["from_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
dst_ids = df_raw_train["to_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
vm=(src_ids>=0)&(dst_ids>=0); sv=src_ids[vm]; dv=dst_ids[vm]
oda=np.zeros(num_nodes,dtype=np.float64); np.add.at(oda,sv,1.0); oda=np.maximum(oda,1.0)
pr=np.ones(num_nodes,dtype=np.float64)/num_nodes
for _ in range(10): npr=np.ones(num_nodes,dtype=np.float64)*0.15/num_nodes; np.add.at(npr,dv,0.85*pr[sv]/oda[sv]); pr=npr
pr_df = pl.DataFrame({"node_id":np.arange(num_nodes,dtype=np.uint32),"graph_pagerank":pr.astype(np.float32)})
pr_df = all_accounts.join(pr_df,on="node_id",how="left").select(["account_id","graph_pagerank"])
del src_ids,dst_ids,sv,dv,oda,pr,npr; gc.collect()

df_graph_feats = (all_accounts.select("account_id").join(out_degree,on="account_id",how="left").join(in_degree,on="account_id",how="left")
    .join(unique_out,on="account_id",how="left").join(unique_in,on="account_id",how="left")
    .join(bidir_count,on="account_id",how="left").join(time_features,on="account_id",how="left")
    .join(pr_df,on="account_id",how="left").fill_null(0.0)
    .with_columns([(pl.col("graph_out_degree")+pl.col("graph_in_degree")).alias("graph_total_degree"),
        (pl.col("graph_out_degree")/(pl.col("graph_in_degree")+1)).alias("graph_out_in_ratio"),
        (pl.col("graph_bidir_count")/(pl.col("graph_out_degree")+1)).alias("graph_reciprocity"),
        (pl.col("graph_unique_out_partners")/(pl.col("graph_out_degree")+1)).alias("graph_partner_diversity")]))
graph_feature_cols = [c for c in df_graph_feats.columns if c.startswith("graph_")]
del out_degree,in_degree,unique_out,unique_in,bidir_count,time_features,pr_df,df_from,df_to; gc.collect()
print(f"  ✅ 그래프 피처 {len(graph_feature_cols)}개")

# ─── 2-2. 조건부행동 피처 10개 ───
print("  📊 조건부행동 피처...")
df_v2_train = df_v2.filter(pl.col("time_group")<train_cutoff)
exclude_cols = ["account_id","time_group","is_laundering","mode_format","currency_mode"]
gnn_feature_cols_v2 = [c for c in df_v2.columns if c not in exclude_cols]  # V2 원본 38개
df_v2_agg = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])

df_cond = df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
v2a=set(df_cond.columns)
def sc(n): return pl.col(n) if n in v2a else pl.lit(0.0)
ce=[]
if "cnt_night" in v2a and "cnt_1h" in v2a: ce.append((pl.col("graph_total_degree")*sc("cnt_night")/(sc("cnt_1h")+1)).alias("cond_highdeg_night_intensity"))
if "avg_log_amount" in v2a: ce.append((sc("avg_log_amount")/(pl.col("graph_total_degree").log1p()+1)).alias("cond_lowdeg_high_amount"))
if "ratio_cross_border" in v2a: ce.append((sc("ratio_cross_border")*(1-pl.col("graph_reciprocity"))).alias("cond_oneway_crossborder"))
if "cnt_1h" in v2a: ce.append((pl.col("graph_unique_out_partners")*sc("cnt_1h")/(pl.col("graph_out_degree")+1)).alias("cond_fanout_burst"))
if "cnt_risk_format" in v2a: ce.append((pl.col("graph_pagerank")*sc("cnt_risk_format")).alias("cond_hub_risk_format"))
if "cnt_wire" in v2a: ce.append((pl.col("graph_out_in_ratio")*sc("cnt_wire")).alias("cond_asymmetric_wire"))
if "cnt_inter_bank" in v2a: ce.append((pl.col("graph_std_tx_interval").fill_null(0)/(pl.col("graph_avg_tx_interval").fill_null(1)+1)*sc("cnt_inter_bank")).alias("cond_irregular_interbank"))
if "cnt_currencies" in v2a: ce.append((pl.col("graph_partner_diversity")*sc("cnt_currencies")).alias("cond_diverse_partner_currency"))
if "amount_kurtosis" in v2a: ce.append((pl.col("graph_reciprocity")*sc("amount_kurtosis")).alias("cond_bidir_amount_volatility"))
if "entity_acct_cnt" in v2a and "burst_ratio" in v2a: ce.append((sc("entity_acct_cnt")*sc("burst_ratio")).alias("cond_multi_acct_burst"))
df_cond=df_cond.with_columns(ce); cond_feature_cols=[e.meta.output_name() for e in ce]
for cn in cond_feature_cols: df_cond=df_cond.with_columns(pl.when(pl.col(cn).is_infinite()|pl.col(cn).is_nan()).then(0.0).otherwise(pl.col(cn)).alias(cn))
df_cond_features=df_cond.select(["account_id"]+cond_feature_cols); del df_cond; gc.collect()
print(f"  ✅ 조건부행동 {len(cond_feature_cols)}개")

# ─── 2-3. AML 점수 5개 ───
print("  📊 AML 점수...")
df_sb=df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
se=[((sc("cnt_1h").fill_null(0).clip(0,1e6)+1).log1p()*pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("burst_ratio").fill_null(0).clip(0,1e6)/(sc("avg_log_amount").fill_null(1).clip(0,1e6)+1)).alias("score_smurf"),
    ((1.0/((pl.col("graph_out_in_ratio").clip(0.01,100)-1.0).abs()+0.1))*sc("cnt_wire").fill_null(0).clip(0,1e6).log1p()*(1-pl.col("graph_reciprocity").clip(0,1))/(pl.col("graph_unique_out_partners").clip(0,1e6).log1p()+1)).alias("score_mule"),
    (pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("cnt_currencies").fill_null(0).clip(0,1e6).log1p()*sc("ratio_cross_border").fill_null(0).clip(0,1)/(pl.col("graph_avg_tx_interval").fill_null(3600).clip(0,1e9)/3600+1)).alias("score_layering"),
    (pl.col("graph_in_degree").clip(0,1e6).log1p()*(pl.col("graph_pagerank").clip(0,1)*1e6)*sc("cnt_risk_format").fill_null(0).clip(0,1e6).log1p()/(pl.col("graph_out_degree").clip(0,1e6).log1p()+1)).alias("score_integration")]
df_sb=df_sb.with_columns(se)
sn=["score_smurf","score_mule","score_layering","score_integration"]
for s in sn: df_sb=df_sb.with_columns(pl.when(pl.col(s).is_infinite()|pl.col(s).is_nan()).then(0.0).otherwise(pl.col(s)).alias(s))
for s in sn:
    mn=df_sb[s].min();mx=df_sb[s].max();rng=mx-mn if(mx-mn)>0 else 1.0
    df_sb=df_sb.with_columns(((pl.col(s)-mn)/rng).alias(f"{s}_norm"))
df_sb=df_sb.with_columns((pl.col("score_smurf_norm")*0.3+pl.col("score_mule_norm")*0.25+pl.col("score_layering_norm")*0.3+pl.col("score_integration_norm")*0.15).alias("score_composite"))
aml_score_cols=sn+["score_composite"]
df_aml_scores=df_sb.select(["account_id"]+aml_score_cols)
del df_sb, df_v2_agg; gc.collect()
print(f"  ✅ AML 점수 {len(aml_score_cols)}개")

# ─── 2-4. 규칙성 피처 7개 ───
print("  📊 규칙성 피처...")
df_tx_time = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner"),
    pl.col("Timestamp"),
    pl.col("Timestamp").dt.hour().alias("hour"),
    pl.col("Timestamp").dt.weekday().alias("dow"),
])
reg_agg = df_tx_time.group_by("account_id").agg([
    (pl.col("hour").is_between(9,17).mean()).alias("reg_business_hour_ratio"),
    (pl.col("dow").is_in([5,6]).mean()).alias("reg_weekend_ratio"),
    (pl.col("hour").is_between(0,5).mean()).alias("reg_night_strict_ratio"),
    pl.col("partner").count().alias("_total_tx"),
])
# Hour entropy
hc=df_tx_time.group_by(["account_id","hour"]).len().rename({"len":"h_count"})
hp=hc.pivot(on="hour",index="account_id",values="h_count").fill_null(0)
hcols=[c for c in hp.columns if c!="account_id"]
hmat=hp.select(hcols).to_numpy().astype(np.float64); hsum=np.maximum(hmat.sum(axis=1,keepdims=True),1.0)
hprobs=hmat/hsum; hlog=np.where(hprobs>0,np.log(hprobs+1e-12),0.0)
df_h_ent=pl.DataFrame({"account_id":hp["account_id"],"reg_hour_entropy":-np.sum(hprobs*hlog,axis=1).astype(np.float32)})
del hc,hp,hmat,hprobs,hlog; gc.collect()
# DOW entropy
dc=df_tx_time.group_by(["account_id","dow"]).len().rename({"len":"d_count"})
dp=dc.pivot(on="dow",index="account_id",values="d_count").fill_null(0)
dcols=[c for c in dp.columns if c!="account_id"]
dmat=dp.select(dcols).to_numpy().astype(np.float64); dsum=np.maximum(dmat.sum(axis=1,keepdims=True),1.0)
dprobs=dmat/dsum; dlog=np.where(dprobs>0,np.log(dprobs+1e-12),0.0)
df_d_ent=pl.DataFrame({"account_id":dp["account_id"],"reg_dow_entropy":-np.sum(dprobs*dlog,axis=1).astype(np.float32)})
del dc,dp,dmat,dprobs,dlog; gc.collect()
# Partner consistency + top concentration
pf=df_tx_time.group_by(["account_id","partner"]).len().rename({"len":"p_count"})
rp=pf.filter(pl.col("p_count")>=2).group_by("account_id").len().rename({"len":"repeat_cnt"})
tp2=pf.group_by("account_id").len().rename({"len":"total_cnt"})
df_pc=tp2.join(rp,on="account_id",how="left").fill_null(0).with_columns((pl.col("repeat_cnt")/(pl.col("total_cnt")+1)).alias("reg_consistent_partner_ratio")).select(["account_id","reg_consistent_partner_ratio"])
top_p=pf.sort(["account_id","p_count"],descending=[False,True]).unique(subset=["account_id"],maintain_order=True).rename({"p_count":"top_cnt"}).select(["account_id","top_cnt"])
df_tc=reg_agg.select(["account_id","_total_tx"]).join(top_p,on="account_id",how="left").fill_null(0).with_columns((pl.col("top_cnt")/(pl.col("_total_tx")+1)).alias("reg_top_partner_concentration")).select(["account_id","reg_top_partner_concentration"])
del pf,rp,tp2,top_p,df_tx_time; gc.collect()
df_regularity=(reg_agg.select(["account_id","reg_business_hour_ratio","reg_weekend_ratio","reg_night_strict_ratio"])
    .join(df_h_ent,on="account_id",how="left").join(df_d_ent,on="account_id",how="left")
    .join(df_pc,on="account_id",how="left").join(df_tc,on="account_id",how="left").fill_null(0.0))
regularity_cols=[c for c in df_regularity.columns if c.startswith("reg_")]
for rc in regularity_cols: df_regularity=df_regularity.with_columns(pl.when(pl.col(rc).is_infinite()|pl.col(rc).is_nan()).then(0.0).otherwise(pl.col(rc)).alias(rc))
del reg_agg,df_h_ent,df_d_ent,df_pc,df_tc; gc.collect()
print(f"  ✅ 규칙성 {len(regularity_cols)}개")

# ─── 2-5. 금액 분포 피처 8개 ───
print("  📊 금액 분포 피처...")
amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train.columns: amount_col = cand; break
if amount_col:
    df_amounts=df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col(amount_col).cast(pl.Float64).alias("amount")]).filter(pl.col("amount")>0)
    df_amt=df_amounts.group_by("account_id").agg([
        ((pl.col("amount")%1000==0)|(pl.col("amount")%5000==0)).mean().alias("amt_round_number_ratio"),
        (pl.col("amount").std()/(pl.col("amount").mean()+1)).alias("amt_cv"),
        (pl.col("amount").max()/(pl.col("amount").mean()+1)).alias("amt_max_mean_ratio"),
        (pl.col("amount").median()/(pl.col("amount").mean()+1)).alias("amt_median_mean_ratio"),
        pl.col("amount").quantile(0.9).alias("_q90"), pl.col("amount").sum().alias("_tsum"),
        pl.col("amount").count().alias("_cnt"),
        pl.col("amount").quantile(0.75).alias("_q75"), pl.col("amount").quantile(0.25).alias("_q25"),
        pl.col("amount").median().alias("_med"),
    ]).fill_null(0.0).with_columns([
        (pl.col("_q90")*0.1*pl.col("_cnt")/(pl.col("_tsum")+1)).alias("amt_top10pct_concentration"),
        ((pl.col("_q75")-pl.col("_q25"))/(pl.col("_med")+1)).alias("amt_iqr_median_ratio")])
    gm=df_amounts["amount"].mean(); st=max(gm*0.1,1.0)
    df_sr=df_amounts.with_columns((pl.col("amount")<st).alias("is_small")).group_by("account_id").agg(pl.col("is_small").mean().alias("amt_small_tx_ratio"))
    df_amt=df_amt.join(df_sr,on="account_id",how="left").fill_null(0.0)
    df_al=df_amounts.with_columns((pl.col("amount")+1).log().cast(pl.Int32).alias("log_bucket"))
    bc=df_al.group_by(["account_id","log_bucket"]).len().rename({"len":"b_count"})
    bt=bc.group_by("account_id").agg(pl.col("b_count").sum().alias("b_total"))
    bp=bc.join(bt,on="account_id").with_columns((pl.col("b_count")/pl.col("b_total")).alias("b_prob"))
    ae=bp.group_by("account_id").agg((-1*(pl.col("b_prob")*(pl.col("b_prob")+1e-12).log()).sum()).alias("amt_entropy"))
    df_amt=df_amt.join(ae,on="account_id",how="left").fill_null(0.0)
    del df_amounts,df_al,bc,bt,bp,ae,df_sr; gc.collect()
    amt_feature_cols=[c for c in df_amt.columns if c.startswith("amt_") and not c.startswith("_")]
    df_amt_features=df_amt.select(["account_id"]+amt_feature_cols)
else:
    df_amt_features=all_accounts.select("account_id"); amt_feature_cols=[]
for ac in amt_feature_cols:
    df_amt_features=df_amt_features.with_columns(pl.when(pl.col(ac).is_infinite()|pl.col(ac).is_nan()).then(0.0).otherwise(pl.col(ac)).alias(ac))
print(f"  ✅ 금액 분포 {len(amt_feature_cols)}개")
log_memory("전체 노드 피처 완료")

# =============================================================
# 3. GNN 엣지/노드 구성 — 강화된 입력
# =============================================================
print("\n🔗 [Step 3] GNN 엣지/노드 구성 (Enriched Input)...")

# ─── Edge 구성: 시간(5) + 금액(3) + 유형(2) = 10개 ───
df_raw_reload = pl.read_parquet(raw_path)
if df_raw_reload["Timestamp"].dtype == pl.Utf8:
    df_raw_reload = df_raw_reload.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_raw_train_edges = df_raw_reload.filter(pl.col("Timestamp")<train_cutoff); del df_raw_reload; gc.collect()

# Amount 컬럼 확인
edge_amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train_edges.columns: edge_amount_col = cand; break

# 기본 엣지 DataFrame
df_edges = df_raw_train_edges.select(
    ["from_acc","to_acc","Timestamp"] +
    ([edge_amount_col] if edge_amount_col else []) +
    (["Payment Format"] if "Payment Format" in df_raw_train_edges.columns else []) +
    (["Is Laundering"] if "Is Laundering" in df_raw_train_edges.columns else [])
).with_columns([pl.col("from_acc").cast(pl.Utf8), pl.col("to_acc").cast(pl.Utf8)])

df_edges = df_edges.join(all_accounts.rename({"account_id":"from_acc","node_id":"src_id"}),on="from_acc",how="left")
df_edges = df_edges.join(all_accounts.rename({"account_id":"to_acc","node_id":"dst_id"}),on="to_acc",how="left")

# --- Edge 피처 1: 시간 (기존 5개) ---
ts = df_edges["Timestamp"]; mn_ts=ts.min(); mx_ts=ts.max(); ts_rng=(mx_ts-mn_ts).total_seconds()
etn = ((ts-mn_ts).dt.total_microseconds().cast(pl.Float64)/1e6/max(ts_rng,1.0)).to_numpy().astype(np.float32)
hrs = df_edges["Timestamp"].dt.hour().to_numpy().astype(np.float32)
dow = df_edges["Timestamp"].dt.weekday().to_numpy().astype(np.float32)
time_feats = np.stack([etn, np.sin(2*np.pi*hrs/24), np.cos(2*np.pi*hrs/24),
                        np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7)], axis=1)  # (E, 5)

# --- Edge 피처 2: 금액 (3개) ---
if edge_amount_col:
    print(f"  📊 Edge 금액 피처 ('{edge_amount_col}')...")
    amounts_raw = df_edges[edge_amount_col].cast(pl.Float64).fill_null(0.0).to_numpy().astype(np.float32)
    # (a) log_amount: 정규화된 로그 금액
    log_amounts = np.log1p(np.maximum(amounts_raw, 0)).astype(np.float32)
    log_amounts = log_amounts / (np.max(log_amounts) + 1e-8)  # 0~1 정규화
    # (b) round_flag: 1000/5000 단위 여부
    round_flag = ((amounts_raw % 1000 == 0) | (amounts_raw % 5000 == 0)).astype(np.float32)
    # (c) relative_amount: 전체 평균 대비 비율 (로그 스케일)
    global_mean_amt = np.mean(amounts_raw[amounts_raw > 0]) if np.any(amounts_raw > 0) else 1.0
    relative_amt = np.log1p(amounts_raw / (global_mean_amt + 1e-8)).astype(np.float32)
    relative_amt = relative_amt / (np.max(relative_amt) + 1e-8)
    amount_feats = np.stack([log_amounts, round_flag, relative_amt], axis=1)  # (E, 3)
else:
    print("  ⚠️ Edge 금액 없음, 0으로 채움")
    amount_feats = np.zeros((len(df_edges), 3), dtype=np.float32)

# --- Edge 피처 3: 거래 유형 (2개) ---
# (a) cross_border: 해외 거래 여부 (from_bank != to_bank의 국가)
# (b) inter_bank: 은행 간 거래 여부
# Payment Format에서 추출 시도
if "Payment Format" in df_edges.columns:
    print("  📊 Edge 유형 피처 (Payment Format)...")
    pf_series = df_edges["Payment Format"].cast(pl.Utf8).fill_null("unknown")
    # wire/SWIFT → 해외 가능성 높음
    is_wire = pf_series.str.contains("(?i)wire|swift|cheque").to_numpy().astype(np.float32)
    # ACH/internal → 국내
    is_internal = pf_series.str.contains("(?i)ach|internal|reinvestment").to_numpy().astype(np.float32)
    type_feats = np.stack([is_wire, is_internal], axis=1)  # (E, 2)
else:
    print("  ⚠️ Payment Format 없음, 0으로 채움")
    type_feats = np.zeros((len(df_edges), 2), dtype=np.float32)

# 전체 Edge 피처 결합
edge_features_all = np.concatenate([time_feats, amount_feats, type_feats], axis=1)  # (E, 10)
EDGE_RAW_DIM = edge_features_all.shape[1]
print(f"  📊 Edge 피처: {EDGE_RAW_DIM}개 (시간 5 + 금액 3 + 유형 2)")

edge_index_train = torch.tensor(df_edges.select(["src_id","dst_id"]).to_numpy().T, dtype=torch.long)
edge_attr_train = torch.tensor(edge_features_all, dtype=torch.float32)
print(f"📊 Train 엣지: {edge_index_train.shape[1]:,}")

del df_raw_train, df_raw_train_edges, df_edges, ts, hrs, dow, etn
del time_feats, amount_feats, type_feats, edge_features_all; gc.collect()

# ─── 노드 피처: V2(38) + 그래프(13) + 조건부(10) + AML(5) + 규칙성(7) + 금액(8) = 81개 ───
print("  📊 노드 피처 구성 (81개 → GNN 입력)...")
df_v2_node = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])
df_node_all = (all_accounts.join(df_v2_node, on="account_id", how="left")
    .join(df_graph_feats, on="account_id", how="left")
    .join(df_cond_features, on="account_id", how="left")
    .join(df_aml_scores, on="account_id", how="left")
    .join(df_regularity, on="account_id", how="left")
    .join(df_amt_features, on="account_id", how="left")
    .fill_null(0.0))

# GNN 입력 피처 목록
gnn_input_cols = gnn_feature_cols_v2 + graph_feature_cols + cond_feature_cols + aml_score_cols + regularity_cols + amt_feature_cols

# inf/NaN 처리
for col in gnn_input_cols:
    if col in df_node_all.columns:
        df_node_all = df_node_all.with_columns(
            pl.when(pl.col(col).is_infinite()|pl.col(col).is_nan()).then(0.0).otherwise(pl.col(col)).alias(col))

X_node = torch.tensor(df_node_all.select(gnn_input_cols).to_numpy(), dtype=torch.float32)
IN_DIM = X_node.shape[1]
print(f"  ✅ GNN 노드 입력: {IN_DIM}차원 = V2({len(gnn_feature_cols_v2)})+그래프({len(graph_feature_cols)})+조건부({len(cond_feature_cols)})+AML({len(aml_score_cols)})+규칙성({len(regularity_cols)})+금액({len(amt_feature_cols)})")

# 레이블 + 마스크
target_labels = df_v2_train.filter(pl.col("is_laundering")==1).select("account_id").unique().with_columns(pl.lit(1).alias("label"))
df_labels = all_accounts.join(target_labels,on="account_id",how="left").fill_null(0)
Y_node = torch.tensor(df_labels["label"].to_numpy(), dtype=torch.long)
pos_node_ids = set(df_labels.filter(pl.col("label")==1)["node_id"].to_list())
active_df = pl.DataFrame({"account_id":df_v2_train["account_id"].unique(),"is_active":True})
mask_df = all_accounts.join(active_df,on="account_id",how="left").fill_null(False)
active_node_ids = mask_df.filter(pl.col("is_active"))["node_id"].to_list()
active_pos = [n for n in active_node_ids if n in pos_node_ids]
active_neg = [n for n in active_node_ids if n not in pos_node_ids]
n_neg_gnn = min(len(active_pos)*NEG_NODE_RATIO_GNN, len(active_neg))
np.random.seed(RANDOM_SEED)
sampled_neg = np.random.choice(active_neg,size=n_neg_gnn,replace=False).tolist()
sampled_nodes = set(active_pos+sampled_neg)
train_mask_np = np.zeros(num_nodes, dtype=bool)
for nid in sampled_nodes: train_mask_np[nid]=True
train_mask = torch.tensor(train_mask_np, dtype=torch.bool)
print(f"📊 GNN 노드: 양성 {len(active_pos):,} + 음성 {n_neg_gnn:,}")
del df_v2_node, df_node_all, mask_df, active_df; gc.collect()

graph_data = Data(x=X_node, edge_index=edge_index_train, edge_attr=edge_attr_train, y=Y_node)
graph_data.train_mask = train_mask; graph_data.num_nodes = num_nodes
del X_node, edge_index_train, edge_attr_train, Y_node, train_mask; gc.collect()
log_memory("GNN 데이터 준비 완료")

# =============================================================
# 4-6. Multi-Config GNN 탐색 → 최적 임베딩 선택
# =============================================================
print("\n🧠 [Step 4-6] Multi-Config GNN 탐색...")

class EdgeProjection(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, out_dim), nn.GELU(), nn.Linear(out_dim, out_dim))
    def forward(self, ea): return self.proj(ea)

class TGAT_Flex(nn.Module):
    """가변 hidden_dim + 2~3 layer"""
    def __init__(self, in_dim, hidden_dim, edge_raw_dim, edge_dim, n_heads, n_layers=2, dropout=0.2):
        super().__init__()
        self.edge_proj = EdgeProjection(edge_raw_dim, edge_dim)
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.n_layers = n_layers
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_layers)])
        self.convs = nn.ModuleList([
            TransformerConv(hidden_dim, hidden_dim//n_heads, heads=n_heads,
                            edge_dim=edge_dim, dropout=dropout, concat=True)
            for _ in range(n_layers)])
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim//2, 1))
        self.dropout = dropout; self.hidden_dim = hidden_dim

    def forward(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        for i in range(self.n_layers):
            h = h + F.dropout(self.convs[i](self.norms[i](h), edge_index, ee), p=self.dropout, training=self.training)
        return self.classifier(self.norm_out(h)).squeeze(-1)

    @torch.no_grad()
    def get_embedding(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        for i in range(self.n_layers):
            h = h + self.convs[i](self.norms[i](h), edge_index, ee)
        return self.norm_out(h)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"📌 디바이스: {device}")
pw = (len(sampled_nodes)-len(active_pos))/max(len(active_pos),1)
print(f"📊 pos_weight: {pw:.1f}")

def make_lr(opt, ep, n_ep):
    if ep < WARMUP_EPOCHS: lr = 1e-6+(BASE_LR-1e-6)*(ep/WARMUP_EPOCHS)
    else: p=(ep-WARMUP_EPOCHS)/max(n_ep-WARMUP_EPOCHS,1); lr=1e-5+(BASE_LR-1e-5)*0.5*(1+np.cos(np.pi*p))
    for pg in opt.param_groups: pg['lr']=lr
    return lr

GNN_CONFIGS = [
    {"label":"h64_L2",  "hidden_dim":64,  "n_layers":2, "n_heads":4, "edge_dim":16},
    {"label":"h128_L2", "hidden_dim":128, "n_layers":2, "n_heads":4, "edge_dim":32},
    {"label":"h64_L3",  "hidden_dim":64,  "n_layers":3, "n_heads":4, "edge_dim":16},
    {"label":"h128_L3", "hidden_dim":128, "n_layers":3, "n_heads":4, "edge_dim":32},
]

# Quick eval용 XGBoost (각 GNN 비교)
xgb_exclude_cols=["account_id","time_group","is_laundering","mode_format","currency_mode","date"]
xgb_v2_cols=[c for c in df_v2.columns if c not in xgb_exclude_cols]
graph_feature_cols=[c for c in df_graph_feats.columns if c.startswith("graph_")]
base_feature_cols = xgb_v2_cols + graph_feature_cols + cond_feature_cols + aml_score_cols
apr_train = df_v2_train.filter(pl.col("is_laundering")==1).height / max(df_v2_train.height,1)
spw = max((1-apr_train)/apr_train, 1.0) if apr_train > 0 else 20.0

def build_eval_df(db, df_emb_local, emb_col_names):
    """df_v2 기반 데이터에 그래프/조건부/AML/임베딩을 join"""
    feature_cols = base_feature_cols + emb_col_names
    df = (db.join(df_graph_feats,on="account_id",how="left")
          .join(df_cond_features,on="account_id",how="left")
          .join(df_aml_scores,on="account_id",how="left")
          .join(df_emb_local,on="account_id",how="left")
          .fill_null(0.0).fill_nan(0.0))
    for cn in feature_cols:
        if cn in df.columns:
            df = df.with_columns(pl.when(pl.col(cn).is_infinite()).then(0.0).otherwise(pl.col(cn)).alias(cn))
    return df, feature_cols

def quick_xgb_eval(df_emb_local, emb_col_names, label):
    """빠른 XGBoost 평가: Stage 1만"""
    df_tr, fcols = build_eval_df(df_v2.filter(pl.col("time_group")<train_cutoff), df_emb_local, emb_col_names)
    df_tr_pos=df_tr.filter(pl.col("is_laundering")==1); df_tr_neg=df_tr.filter(pl.col("is_laundering")==0)
    n_neg=min(len(df_tr_pos)*NEG_ROW_RATIO_XGB,len(df_tr_neg))
    df_tr_s=pl.concat([df_tr_pos,df_tr_neg.sample(n=n_neg,seed=RANDOM_SEED)]).sample(fraction=1.0,seed=RANDOM_SEED)
    X_tr=df_tr_s.select(fcols).to_pandas(); y_tr=df_tr_s["is_laundering"].to_numpy()
    del df_tr,df_tr_pos,df_tr_neg,df_tr_s; gc.collect()

    df_vl, _ = build_eval_df(df_v2.filter((pl.col("time_group")>=train_cutoff)&(pl.col("time_group")<val_cutoff)), df_emb_local, emb_col_names)
    X_vl=df_vl.select(fcols).to_pandas(); y_vl=df_vl["is_laundering"].to_numpy(); del df_vl; gc.collect()

    df_te, _ = build_eval_df(df_v2.filter(pl.col("time_group")>=val_cutoff), df_emb_local, emb_col_names)
    X_te=df_te.select(fcols).to_pandas(); y_te=df_te["is_laundering"].to_numpy(); del df_te; gc.collect()

    qp={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
        "random_state":42,"max_depth":8,"learning_rate":0.03,"scale_pos_weight":spw,
        "subsample":0.8,"colsample_bytree":0.7,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
    m=xgb.XGBClassifier(**qp,n_estimators=1500,early_stopping_rounds=100)
    m.fit(X_tr,y_tr,eval_set=[(X_vl,y_vl)],verbose=0)
    q_val=m.best_score; q_test=average_precision_score(y_te,m.predict_proba(X_te)[:,1])
    del m,X_tr,X_vl,X_te,y_tr,y_vl,y_te; gc.collect()
    return q_val, q_test

print(f"\n📊 {len(GNN_CONFIGS)}개 GNN 설정 탐색...")
print(f"  {'#':>2s} {'Label':<12s} | {'hidden':>6s} {'layers':>6s} {'edim':>4s} | {'Loss':>8s} {'Params':>8s} | {'QVal':>8s} {'QTest':>8s}")
print("  "+"-"*80)

gnn_results = []
for gi, gcfg in enumerate(GNN_CONFIGS):
    gl = gcfg["label"]; hd=gcfg["hidden_dim"]; nl=gcfg["n_layers"]
    nh=gcfg["n_heads"]; ed=gcfg["edge_dim"]

    train_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                                   input_nodes=graph_data.train_mask, shuffle=True, num_workers=4)
    full_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                                  input_nodes=None, shuffle=False, num_workers=4)
    model = TGAT_Flex(in_dim=IN_DIM, hidden_dim=hd, edge_raw_dim=EDGE_RAW_DIM,
                      edge_dim=ed, n_heads=nh, n_layers=nl, dropout=0.2).to(device)
    np_count = sum(p.numel() for p in model.parameters())
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw],device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6, weight_decay=1e-4)

    model.train(); bl=float('inf'); pc2=0; bs=None
    for epoch in range(N_EPOCHS_GNN):
        clr=make_lr(optimizer,epoch,N_EPOCHS_GNN); tl=0.0; nb2=0
        for batch in tqdm(train_loader,desc=f"[{gl}] Ep{epoch+1}",leave=False):
            batch=batch.to(device); optimizer.zero_grad()
            bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
            out=model(batch.x,batch.edge_index,bea)
            loss=criterion(out[:batch.batch_size],batch.y[:batch.batch_size].float())
            if torch.isnan(loss) or torch.isinf(loss): optimizer.zero_grad(); continue
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=0.5)
            optimizer.step(); tl+=loss.item(); nb2+=1
        al=tl/max(nb2,1)
        if (epoch+1)%10==0 or epoch==0: print(f"    [{gl}] Ep{epoch+1:2d} Loss={al:.4f}")
        if al<bl-0.0005: bl=al; pc2=0; bs={k:v.cpu().clone() for k,v in model.state_dict().items()}
        else: pc2+=1
        if pc2>=PATIENCE: break
    if bs: model.load_state_dict({k:v.to(device) for k,v in bs.items()}); del bs

    # 임베딩
    model.eval(); ae=[]
    with torch.no_grad():
        for batch in tqdm(full_loader,desc=f"[{gl}] Embed",leave=False):
            batch=batch.to(device)
            bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
            emb=model.get_embedding(batch.x,batch.edge_index,bea)
            ae.append(emb[:batch.batch_size].cpu())
    femb=torch.cat(ae,dim=0).numpy()
    del model,ae,train_loader,full_loader; torch.cuda.empty_cache(); gc.collect()

    ecn=[f"tgat_emb_{i}" for i in range(hd)]
    df_emb=pl.concat([all_accounts.select("account_id"),pl.DataFrame(femb,schema=ecn)],how="horizontal")

    # Quick XGBoost eval
    qv,qt = quick_xgb_eval(df_emb, ecn, gl)
    print(f"  {gi+1:2d} {gl:<12s} | {hd:6d} {nl:6d} {ed:4d} | {bl:8.4f} {np_count:8,} | {qv:8.5f} {qt:8.4f}")

    gnn_results.append({"label":gl,"hidden_dim":hd,"n_layers":nl,"loss":bl,"params":np_count,
                         "q_val":qv,"q_test":qt,"embedding":femb,"ecn":ecn,"df_emb":df_emb})
    log_memory(f"{gl}")

# 최적 GNN 선택
best_gnn = max(gnn_results, key=lambda x: x["q_val"])
print(f"\n  🏆 Best GNN: {best_gnn['label']} (QVal={best_gnn['q_val']:.5f} QTest={best_gnn['q_test']:.4f})")

# 메모리 정리
for r in gnn_results:
    if r is not best_gnn: del r["embedding"], r["df_emb"]
gc.collect()

emb_cols = best_gnn["ecn"]
df_tgat = best_gnn["df_emb"]
FINAL_HIDDEN_DIM = best_gnn["hidden_dim"]

# Interaction 피처
print(f"\n🔗 Interaction 피처 ({best_gnn['label']})...")
df_tgat_with_pr=df_tgat.join(df_graph_feats.select(["account_id","graph_pagerank"]),on="account_id",how="left").fill_null(0.0)
emb_np=df_tgat_with_pr.select(emb_cols).to_numpy(); emb_var=np.var(emb_np,axis=0)
top_idx=np.argsort(emb_var)[::-1][:N_INTERACTION_DIMS]; top_ec=[emb_cols[i] for i in top_idx]
ie_l=[]; ic_l=[]
for cn in top_ec: nn2=f"interact_pr_x_{cn}"; ie_l.append((pl.col(cn)*pl.col("graph_pagerank")*1e6).alias(nn2)); ic_l.append(nn2)
ie_l.append((sum(pl.col(c)**2 for c in emb_cols).sqrt()*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_norm")); ic_l.append("interact_pr_x_emb_norm")
ie_l.append((sum(pl.col(c) for c in top_ec)*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_topsum")); ic_l.append("interact_pr_x_emb_topsum")
interaction_cols=ic_l
df_interactions=df_tgat_with_pr.with_columns(ie_l).select(["account_id"]+ic_l)
for c in ic_l: df_interactions=df_interactions.with_columns(pl.when(pl.col(c).is_infinite()|pl.col(c).is_nan()).then(0.0).otherwise(pl.col(c)).alias(c))
print(f"  ✅ Interaction {len(ic_l)}개")
del df_tgat_with_pr,emb_np,best_gnn["embedding"]; gc.collect()
del graph_data; torch.cuda.empty_cache(); gc.collect()
log_memory("GNN 탐색 완료")

# =============================================================
# 7. XGBoost 데이터 구성
# =============================================================
print("\n🚀 [Step 7] XGBoost 데이터 구성...")
xgb_feature_cols = xgb_v2_cols+graph_feature_cols+cond_feature_cols+aml_score_cols+emb_cols+interaction_cols
print(f"📊 XGBoost 피처: {len(xgb_feature_cols)} (V2:{len(xgb_v2_cols)} G:{len(graph_feature_cols)} C:{len(cond_feature_cols)} A:{len(aml_score_cols)} E:{len(emb_cols)} I:{len(interaction_cols)})")

def build_xgb_df(db):
    df=db.join(df_graph_feats,on="account_id",how="left").join(df_cond_features,on="account_id",how="left").join(df_aml_scores,on="account_id",how="left").join(df_tgat,on="account_id",how="left").join(df_interactions,on="account_id",how="left").fill_null(0.0).fill_nan(0.0)
    for cn in xgb_feature_cols:
        if cn in df.columns: df=df.with_columns(pl.when(pl.col(cn).is_infinite()).then(0.0).otherwise(pl.col(cn)).alias(cn))
    return df

df_tr=build_xgb_df(df_v2.filter(pl.col("time_group")<train_cutoff))
df_tr_pos=df_tr.filter(pl.col("is_laundering")==1); df_tr_neg=df_tr.filter(pl.col("is_laundering")==0)
n_neg=min(len(df_tr_pos)*NEG_ROW_RATIO_XGB,len(df_tr_neg))
df_tr_s=pl.concat([df_tr_pos,df_tr_neg.sample(n=n_neg,seed=RANDOM_SEED)]).sample(fraction=1.0,seed=RANDOM_SEED)
X_train=df_tr_s.select(xgb_feature_cols).to_pandas(); y_train=df_tr_s["is_laundering"].to_numpy()
print(f"📊 Train: {len(df_tr):,} → {len(df_tr_s):,} ({y_train.mean()*100:.2f}%)")
del df_tr,df_tr_pos,df_tr_neg,df_tr_s; gc.collect()
df_vl=build_xgb_df(df_v2.filter((pl.col("time_group")>=train_cutoff)&(pl.col("time_group")<val_cutoff)))
X_val=df_vl.select(xgb_feature_cols).to_pandas(); y_val=df_vl["is_laundering"].to_numpy(); print(f"📊 Val: {len(X_val):,}"); del df_vl; gc.collect()
df_te=build_xgb_df(df_v2.filter(pl.col("time_group")>=val_cutoff))
df_test_meta=df_te.select(["account_id","time_group","is_laundering"])
X_test=df_te.select(xgb_feature_cols).to_pandas(); y_test=df_te["is_laundering"].to_numpy(); print(f"📊 Test: {len(X_test):,}")
del df_te,df_v2,df_tgat,df_graph_feats,df_cond_features,df_aml_scores,df_interactions,df_regularity,df_amt_features; gc.collect()
log_memory("XGBoost 준비")

# =============================================================
# 8. XGBoost 2-Stage + 확장 HP 그리드
# =============================================================
print("\n🔥 [Step 8] 2-Stage Pruning + 확장 HP 그리드...")
apr=y_train.sum()/len(y_train); spw=max((1-apr)/apr,1.0)

# Stage 1
print("\n── Stage 1 ──")
s1p={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda","random_state":42,"max_depth":8,"learning_rate":0.03,"scale_pos_weight":spw,"subsample":0.8,"colsample_bytree":0.7,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
m_s1=xgb.XGBClassifier(**s1p,n_estimators=1500,early_stopping_rounds=100)
m_s1.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=100)
s1_val=m_s1.best_score; s1_test=average_precision_score(y_test,m_s1.predict_proba(X_test)[:,1])
print(f"\n  Stage1: Val={s1_val:.5f} Test={s1_test:.4f}")
imp_s1=m_s1.feature_importances_; feat_imp=sorted(zip(xgb_feature_cols,imp_s1),key=lambda x:x[1],reverse=True)
del m_s1; gc.collect()

# Stage 2 — 확장 HP 그리드
print("\n── Stage 2: 확장 HP 그리드 ──")
bp2={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda","random_state":42,"scale_pos_weight":spw}
configs=[]
for nf in [45,50,55,60,65,70,80]:
    sel=[n for n,_ in feat_imp[:nf]]
    configs.append({"label":f"top{nf}_d8_c07","features":sel,"hp":{"max_depth":8,"learning_rate":0.03,"colsample_bytree":0.7,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":1500}})
    configs.append({"label":f"top{nf}_lr02_c08","features":sel,"hp":{"max_depth":8,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}})
    configs.append({"label":f"top{nf}_lr02_c09","features":sel,"hp":{"max_depth":8,"learning_rate":0.02,"colsample_bytree":0.9,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}})
    configs.append({"label":f"top{nf}_d9_lr02","features":sel,"hp":{"max_depth":9,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}})
    configs.append({"label":f"top{nf}_reg","features":sel,"hp":{"max_depth":8,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.75,"min_child_weight":8,"gamma":0.2,"reg_alpha":0.5,"reg_lambda":2.0,"n_estimators":2500}})
configs.append({"label":"all","features":xgb_feature_cols,"hp":{"max_depth":8,"learning_rate":0.03,"colsample_bytree":0.7,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":1500}})

print(f"\n📊 {len(configs)}개 조합 탐색...")
print(f"  {'#':>2s} {'Label':<22s} | {'#f':>3s} {'d':>2s} {'LR':>5s} {'col':>4s} | {'Val':>9s} {'it':>5s}")
print("  "+"-"*65)
best_s2={"val":-1}
for i,cfg in enumerate(configs):
    hp=cfg["hp"].copy(); ne=hp.pop("n_estimators"); sf=cfg["features"]
    m=xgb.XGBClassifier(**{**bp2,**hp},n_estimators=ne,early_stopping_rounds=100)
    m.fit(X_train[sf],y_train,eval_set=[(X_val[sf],y_val)],verbose=0)
    va=m.best_score; bi=m.best_iteration
    print(f"  {i+1:2d} {cfg['label']:<22s} | {len(sf):3d} {hp.get('max_depth',8):2d} {hp.get('learning_rate',0.03):5.3f} {hp.get('colsample_bytree',0.7):4.2f} | {va:9.5f} {bi:5d}")
    if va>best_s2["val"]:
        if best_s2.get("model"): del best_s2["model"]
        best_s2={"label":cfg["label"],"features":sf,"val":va,"iter":bi,"model":m,"hp":hp}
    else: del m
    gc.collect()

y_prob=best_s2["model"].predict_proba(X_test[best_s2["features"]])[:,1]
best_features=best_s2["features"]
print(f"\n  🏆 Best: {best_s2['label']} ({len(best_features)}개) Val={best_s2['val']:.5f} iter={best_s2['iter']}")

# Importance
imp=best_s2["model"].feature_importances_; tidx=np.argsort(imp)[::-1][:25]
print(f"\n🔍 Feature Importance Top 25")
for i,idx in enumerate(tidx):
    c=best_features[idx]; t=""
    if c.startswith("tgat_emb_"): t=" [TGAT]"
    elif c.startswith("graph_"): t=" [GRAPH]"
    elif c.startswith("cond_"): t=" [COND]"
    elif c.startswith("score_"): t=" [SCORE]"
    elif c.startswith("interact_"): t=" [INTER]"
    print(f"  {i+1:2d}. {c:<40s}: {imp[idx]:.4f}{t}")
ti=imp.sum()
v2i=sum(imp[i] for i,c in enumerate(best_features) if c in xgb_v2_cols)
gi=sum(imp[i] for i,c in enumerate(best_features) if c.startswith("graph_"))
ei=sum(imp[i] for i,c in enumerate(best_features) if c.startswith("tgat_emb_"))
ii=sum(imp[i] for i,c in enumerate(best_features) if c.startswith("interact_"))
print(f"\n📊 그룹별: V2={v2i/ti*100:.1f}% Graph={gi/ti*100:.1f}% TGAT={ei/ti*100:.1f}% Inter={ii/ti*100:.1f}%")
del X_train,X_val; gc.collect()

# =============================================================
# 9. 최종 리포트
# =============================================================
print("\n"+"="*60); print("🏆 [TGAT v7c.2] 최종 리포트"); print("="*60)
auprc=average_precision_score(y_test,y_prob); print(f"\n  AUPRC: {auprc:.4f}")
best_f1=0; best_thresh=0
for th in np.arange(0.05,0.95,0.01):
    f=f1_score(y_test,(y_prob>=th).astype(int),zero_division=0)
    if f>best_f1: best_f1=f; best_thresh=th
y_pred=(y_prob>=best_thresh).astype(int)
prec=precision_score(y_test,y_pred,zero_division=0); rec=recall_score(y_test,y_pred,zero_division=0)
print(f"\n📌 최적: T={best_thresh:.2f} F1={best_f1:.4f} P={prec:.4f} R={rec:.4f}")
print(f"\n📌 임계값별:")
for t in [0.1,0.3,0.5,0.7,0.8,0.9]:
    yp=(y_prob>=t).astype(int)
    print(f"  T={t:.1f} P={precision_score(y_test,yp,zero_division=0):.4f} R={recall_score(y_test,yp,zero_division=0):.4f} F1={f1_score(y_test,yp,zero_division=0):.4f}")

df_res=df_test_meta.with_columns([pl.col("time_group").dt.date().alias("date"),pl.Series("pred_prob",y_prob)])
df_dist=df_res.sort("pred_prob",descending=True).unique(subset=["account_id"],maintain_order=True)
print(f"\n📌 Top-K:")
for k in [100,500,1000,5000,10000]:
    ck=min(k,len(df_dist)); hits=df_dist.head(ck)["is_laundering"].sum()
    print(f"  Top-{k:5d}: {hits:5d}명 ({hits/ck*100:.2f}%)")
print(f"\n📌 일별 Top-100:")
for d in df_dist["date"].unique(maintain_order=True).sort(descending=True):
    hits=df_dist.filter(pl.col("date")==d).head(100)["is_laundering"].sum(); print(f"  {d}: {hits}")

tn,fp,fn,tp=confusion_matrix(y_test,y_pred).ravel()
print(f"\n📌 v4→v7c.2: AUPRC 0.5370→{auprc:.4f} ({'↑' if auprc>0.537 else '↓'}{abs(auprc-0.537):.4f})")
print(f"  F1 0.4940→{best_f1:.4f} | TP:{tp:,} FP:{fp:,} FN:{fn:,} TN:{tn:,}")

# GNN 탐색 결과 요약
print(f"\n📌 GNN 탐색 결과:")
for r in gnn_results:
    tag = " ★" if r["label"]==best_gnn["label"] else ""
    print(f"  {r['label']:<12s}: QVal={r['q_val']:.5f} QTest={r['q_test']:.4f} Loss={r['loss']:.4f} Params={r['params']:,}{tag}")

print(f"\n📌 2-Stage: S1({len(xgb_feature_cols)}개)={s1_val:.5f}/{s1_test:.4f} → S2({best_s2['label']},{len(best_features)}개)={best_s2['val']:.5f}/{auprc:.4f}")
del X_test; gc.collect(); log_memory("최종 완료")
print(f"\n✅ TGAT v7c.2 완료!")

🛡️ [TGAT v7c.2] Input+Edge Enrichment + GNN 탐색 + HP Grid
   ▸ 노드 81d + Edge 10d (v7c 동일)
   ▸ GNN 탐색: hidden 64/128 × layer 2/3 = 4개 설정
   ▸ XGBoost: 확장 HP 그리드 (7×5+1=36조합)

📂 데이터 로딩 중...
⏱️ Train cutoff: 2022-09-09 20:00:00
⏱️ Val   cutoff: 2022-09-14 05:00:00
📊 전체 노드 수: 2,076,999
  💾 [데이터 로드 완료] RAM: 74.35GB | GPU: 0.02GB

📐 [Step 2] 전체 노드 피처 계산 (81개)...
  📊 그래프 구조 피처...
  📊 PageRank...
  ✅ 그래프 피처 13개
  📊 조건부행동 피처...
  ✅ 조건부행동 10개
  📊 AML 점수...
  ✅ AML 점수 5개
  📊 규칙성 피처...
  ✅ 규칙성 7개
  📊 금액 분포 피처...
  ✅ 금액 분포 8개
  💾 [전체 노드 피처 완료] RAM: 74.30GB | GPU: 0.02GB

🔗 [Step 3] GNN 엣지/노드 구성 (Enriched Input)...
  📊 Edge 금액 피처 ('Amount Received')...
  📊 Edge 유형 피처 (Payment Format)...
  📊 Edge 피처: 10개 (시간 5 + 금액 3 + 유형 2)
📊 Train 엣지: 19,060,343
  📊 노드 피처 구성 (81개 → GNN 입력)...
  ✅ GNN 노드 입력: 81차원 = V2(38)+그래프(13)+조건부(10)+AML(5)+규칙성(7)+금액(8)
📊 GNN 노드: 양성 20,400 + 음성 1,020,000
  💾 [GNN 데이터 준비 완료] RAM: 75.93GB | GPU: 0.02GB

🧠 [Step 4-6] Multi-Config GNN 탐색...
📌 디바이스: cuda
📊 pos_weight: 50.0

📊 4개 GNN 

    [h64_L2] Ep 1 Loss=1.3710


    [h64_L2] Ep10 Loss=1.0629


    [h64_L2] Ep20 Loss=1.0314


    [h64_L2] Ep30 Loss=1.0110


...

    [h128_L3] Ep10 Loss=1.0415


    [h128_L3] Ep20 Loss=1.0076


    [h128_L3] Ep30 Loss=0.9858


    [h128_L3] Ep40 Loss=0.9779


   4 h128_L3      |    128      3   32 |   0.9748  231,681 |  0.34684   0.4144
  💾 [h128_L3] RAM: 73.42GB | GPU: 0.02GB

  🏆 Best GNN: h128_L2 (QVal=0.42064 QTest=0.4656)

🔗 Interaction 피처 (h128_L2)...
  ✅ Interaction 10개
  💾 [GNN 탐색 완료] RAM: 69.88GB | GPU: 0.02GB

🚀 [Step 7] XGBoost 데이터 구성...
📊 XGBoost 피처: 204 (V2:38 G:13 C:10 A:5 E:128 I:10)
📊 Train: 17,172,747 → 619,437 (4.76%)
📊 Val: 5,790,644
📊 Test: 5,792,913
  💾 [XGBoost 준비] RAM: 91.49GB | GPU: 0.02GB

🔥 [Step 8] 2-Stage Pruning + 확장 HP 그리드...

── Stage 1 ──
[0]	validation_0-aucpr:0.11693
[100]	validation_0-aucpr:0.33235
[200]	validation_0-aucpr:0.41622
[300]	validation_0-aucpr:0.43264
[400]	validation_0-aucpr:0.44503
[500]	validation_0-aucpr:0.44784
[600]	validation_0-aucpr:0.45379
[700]	validation_0-aucpr:0.45569
[800]	validation_0-aucpr:0.45788
[900]	validation_0-aucpr:0.46074
[1000]	validation_0-aucpr:0.46213
[1100]	validation_0-aucpr:0.46460
[1200]	validation_0-aucpr:0.46670
[1300]	validation_0-aucpr:0.46976
[1400]	validati

In [1]:
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv
from torch_geometric.loader import NeighborLoader
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm
import gc, numpy as np, psutil, os, warnings
warnings.filterwarnings("ignore", category=UserWarning)

def log_memory(tag=""):
    rss = psutil.Process(os.getpid()).memory_info().rss/(1024**3)
    gpu = torch.cuda.memory_allocated()/(1024**3) if torch.cuda.is_available() else 0
    print(f"  💾 [{tag}] RAM: {rss:.2f}GB | GPU: {gpu:.2f}GB")

print("=" * 60)
print("🛡️ [TGAT v7c.4] FP/FN 보완 피처 추가 (trend_* 제거)")
print("   ▸ v7c.2 대비 추가 피처 4그룹:")
print("     ① Cross-border 정밀화 (cb_*, 4개)")
print("     ② 거래량 대비 정규화 개선 (vol_*, 5개)")
print("     ③ 시간 윈도우별 행동 변화 (ws_*, 4개)")
print("     ④ 입금/출금 독립 피처 (inout_*, 6개)")
print("   ▸ GNN 탐색: hidden 64/128 × layer 2/3 = 4개")
print("   ▸ XGBoost: 확장 HP 그리드 (7×5+1=36조합)")
print("=" * 60)

# ===================== 설정 =====================
NEG_NODE_RATIO_GNN = 50; NEG_ROW_RATIO_XGB = 20; RANDOM_SEED = 42
HIDDEN_DIM = 64; EDGE_DIM = 16; N_HEADS = 4
N_EPOCHS_GNN = 40; BATCH_SIZE = 2048; NUM_NEIGHBORS = [15, 10]
N_INTERACTION_DIMS = 8; PATIENCE = 10; BASE_LR = 0.001; WARMUP_EPOCHS = 5

raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"

# =============================================================
# 1. 데이터 로드
# =============================================================
print("\n📂 데이터 로딩 중...")
df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))
if df_raw["Timestamp"].dtype == pl.Utf8:
    df_raw = df_raw.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_v2 = df_v2.sort("time_group")
total_count = len(df_v2)
train_cutoff = df_v2["time_group"][int(total_count*0.6)]
val_cutoff = df_v2["time_group"][int(total_count*0.8)]
print(f"⏱️ Train cutoff: {train_cutoff}\n⏱️ Val   cutoff: {val_cutoff}")

all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")
num_nodes = len(all_accounts); print(f"📊 전체 노드 수: {num_nodes:,}")
log_memory("데이터 로드 완료")

# =============================================================
# 2. 모든 노드 피처 계산 (GNN 입력용 + XGBoost 겸용)
# =============================================================
print("\n📐 [Step 2] 전체 노드 피처 계산...")
df_raw_train = df_raw.filter(pl.col("Timestamp") < train_cutoff)

# ─── 2-1. 그래프 구조 피처 13개 ───
print("  📊 그래프 구조 피처...")
df_from = df_raw_train.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id"))
df_to = df_raw_train.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
out_degree = df_from.group_by("account_id").len().rename({"len":"graph_out_degree"})
in_degree = df_to.group_by("account_id").len().rename({"len":"graph_in_degree"})
unique_out = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("to_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_out_partners"))
unique_in = df_raw_train.select([pl.col("to_acc").cast(pl.Utf8).alias("account_id"),pl.col("from_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_in_partners"))
esf = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("a"),pl.col("to_acc").cast(pl.Utf8).alias("b")]).unique()
est = esf.select([pl.col("b").alias("a"),pl.col("a").alias("b")])
bidir = esf.join(est, on=["a","b"], how="inner")
bidir_count = bidir.select(pl.col("a").alias("account_id")).group_by("account_id").len().rename({"len":"graph_bidir_count"})
del esf, est, bidir; gc.collect()
time_features = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("Timestamp")]).group_by("account_id").agg([
    pl.col("Timestamp").sort().diff().dt.total_seconds().mean().alias("graph_avg_tx_interval"),
    pl.col("Timestamp").sort().diff().dt.total_seconds().std().alias("graph_std_tx_interval"),
    pl.col("Timestamp").count().alias("graph_total_tx_count")])

# PageRank
print("  📊 PageRank...")
account_to_id = dict(zip(all_accounts["account_id"].to_list(), all_accounts["node_id"].to_list()))
src_ids = df_raw_train["from_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
dst_ids = df_raw_train["to_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
vm=(src_ids>=0)&(dst_ids>=0); sv=src_ids[vm]; dv=dst_ids[vm]
oda=np.zeros(num_nodes,dtype=np.float64); np.add.at(oda,sv,1.0); oda=np.maximum(oda,1.0)
pr=np.ones(num_nodes,dtype=np.float64)/num_nodes
for _ in range(10): npr=np.ones(num_nodes,dtype=np.float64)*0.15/num_nodes; np.add.at(npr,dv,0.85*pr[sv]/oda[sv]); pr=npr
pr_df = pl.DataFrame({"node_id":np.arange(num_nodes,dtype=np.uint32),"graph_pagerank":pr.astype(np.float32)})
pr_df = all_accounts.join(pr_df,on="node_id",how="left").select(["account_id","graph_pagerank"])
del src_ids,dst_ids,sv,dv,oda,pr,npr; gc.collect()

df_graph_feats = (all_accounts.select("account_id").join(out_degree,on="account_id",how="left").join(in_degree,on="account_id",how="left")
    .join(unique_out,on="account_id",how="left").join(unique_in,on="account_id",how="left")
    .join(bidir_count,on="account_id",how="left").join(time_features,on="account_id",how="left")
    .join(pr_df,on="account_id",how="left").fill_null(0.0)
    .with_columns([(pl.col("graph_out_degree")+pl.col("graph_in_degree")).alias("graph_total_degree"),
        (pl.col("graph_out_degree")/(pl.col("graph_in_degree")+1)).alias("graph_out_in_ratio"),
        (pl.col("graph_bidir_count")/(pl.col("graph_out_degree")+1)).alias("graph_reciprocity"),
        (pl.col("graph_unique_out_partners")/(pl.col("graph_out_degree")+1)).alias("graph_partner_diversity")]))
graph_feature_cols = [c for c in df_graph_feats.columns if c.startswith("graph_")]
del out_degree,in_degree,unique_out,unique_in,bidir_count,time_features,pr_df,df_from,df_to; gc.collect()
print(f"  ✅ 그래프 피처 {len(graph_feature_cols)}개")

# ─── 2-2. 조건부행동 피처 10개 ───
print("  📊 조건부행동 피처...")
df_v2_train = df_v2.filter(pl.col("time_group")<train_cutoff)
exclude_cols = ["account_id","time_group","is_laundering","mode_format","currency_mode"]
gnn_feature_cols_v2 = [c for c in df_v2.columns if c not in exclude_cols]
df_v2_agg = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])

df_cond = df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
v2a=set(df_cond.columns)
def sc(n): return pl.col(n) if n in v2a else pl.lit(0.0)
ce=[]
if "cnt_night" in v2a and "cnt_1h" in v2a: ce.append((pl.col("graph_total_degree")*sc("cnt_night")/(sc("cnt_1h")+1)).alias("cond_highdeg_night_intensity"))
if "avg_log_amount" in v2a: ce.append((sc("avg_log_amount")/(pl.col("graph_total_degree").log1p()+1)).alias("cond_lowdeg_high_amount"))
if "ratio_cross_border" in v2a: ce.append((sc("ratio_cross_border")*(1-pl.col("graph_reciprocity"))).alias("cond_oneway_crossborder"))
if "cnt_1h" in v2a: ce.append((pl.col("graph_unique_out_partners")*sc("cnt_1h")/(pl.col("graph_out_degree")+1)).alias("cond_fanout_burst"))
if "cnt_risk_format" in v2a: ce.append((pl.col("graph_pagerank")*sc("cnt_risk_format")).alias("cond_hub_risk_format"))
if "cnt_wire" in v2a: ce.append((pl.col("graph_out_in_ratio")*sc("cnt_wire")).alias("cond_asymmetric_wire"))
if "cnt_inter_bank" in v2a: ce.append((pl.col("graph_std_tx_interval").fill_null(0)/(pl.col("graph_avg_tx_interval").fill_null(1)+1)*sc("cnt_inter_bank")).alias("cond_irregular_interbank"))
if "cnt_currencies" in v2a: ce.append((pl.col("graph_partner_diversity")*sc("cnt_currencies")).alias("cond_diverse_partner_currency"))
if "amount_kurtosis" in v2a: ce.append((pl.col("graph_reciprocity")*sc("amount_kurtosis")).alias("cond_bidir_amount_volatility"))
if "entity_acct_cnt" in v2a and "burst_ratio" in v2a: ce.append((sc("entity_acct_cnt")*sc("burst_ratio")).alias("cond_multi_acct_burst"))
df_cond=df_cond.with_columns(ce); cond_feature_cols=[e.meta.output_name() for e in ce]
for cn in cond_feature_cols: df_cond=df_cond.with_columns(pl.when(pl.col(cn).is_infinite()|pl.col(cn).is_nan()).then(0.0).otherwise(pl.col(cn)).alias(cn))
df_cond_features=df_cond.select(["account_id"]+cond_feature_cols); del df_cond; gc.collect()
print(f"  ✅ 조건부행동 {len(cond_feature_cols)}개")

# ─── 2-3. AML 점수 5개 ───
print("  📊 AML 점수...")
df_sb=df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
se=[((sc("cnt_1h").fill_null(0).clip(0,1e6)+1).log1p()*pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("burst_ratio").fill_null(0).clip(0,1e6)/(sc("avg_log_amount").fill_null(1).clip(0,1e6)+1)).alias("score_smurf"),
    ((1.0/((pl.col("graph_out_in_ratio").clip(0.01,100)-1.0).abs()+0.1))*sc("cnt_wire").fill_null(0).clip(0,1e6).log1p()*(1-pl.col("graph_reciprocity").clip(0,1))/(pl.col("graph_unique_out_partners").clip(0,1e6).log1p()+1)).alias("score_mule"),
    (pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("cnt_currencies").fill_null(0).clip(0,1e6).log1p()*sc("ratio_cross_border").fill_null(0).clip(0,1)/(pl.col("graph_avg_tx_interval").fill_null(3600).clip(0,1e9)/3600+1)).alias("score_layering"),
    (pl.col("graph_in_degree").clip(0,1e6).log1p()*(pl.col("graph_pagerank").clip(0,1)*1e6)*sc("cnt_risk_format").fill_null(0).clip(0,1e6).log1p()/(pl.col("graph_out_degree").clip(0,1e6).log1p()+1)).alias("score_integration")]
df_sb=df_sb.with_columns(se)
sn=["score_smurf","score_mule","score_layering","score_integration"]
for s in sn: df_sb=df_sb.with_columns(pl.when(pl.col(s).is_infinite()|pl.col(s).is_nan()).then(0.0).otherwise(pl.col(s)).alias(s))
for s in sn:
    mn=df_sb[s].min();mx=df_sb[s].max();rng=mx-mn if(mx-mn)>0 else 1.0
    df_sb=df_sb.with_columns(((pl.col(s)-mn)/rng).alias(f"{s}_norm"))
df_sb=df_sb.with_columns((pl.col("score_smurf_norm")*0.3+pl.col("score_mule_norm")*0.25+pl.col("score_layering_norm")*0.3+pl.col("score_integration_norm")*0.15).alias("score_composite"))
aml_score_cols=sn+["score_composite"]
df_aml_scores=df_sb.select(["account_id"]+aml_score_cols)
del df_sb, df_v2_agg; gc.collect()
print(f"  ✅ AML 점수 {len(aml_score_cols)}개")

# ─── 2-4. 규칙성 피처 7개 ───
print("  📊 규칙성 피처...")
df_tx_time = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner"),
    pl.col("Timestamp"),
    pl.col("Timestamp").dt.hour().alias("hour"),
    pl.col("Timestamp").dt.weekday().alias("dow"),
])
reg_agg = df_tx_time.group_by("account_id").agg([
    (pl.col("hour").is_between(9,17).mean()).alias("reg_business_hour_ratio"),
    (pl.col("dow").is_in([5,6]).mean()).alias("reg_weekend_ratio"),
    (pl.col("hour").is_between(0,5).mean()).alias("reg_night_strict_ratio"),
    pl.col("partner").count().alias("_total_tx"),
])
hc=df_tx_time.group_by(["account_id","hour"]).len().rename({"len":"h_count"})
hp=hc.pivot(on="hour",index="account_id",values="h_count").fill_null(0)
hcols=[c for c in hp.columns if c!="account_id"]
hmat=hp.select(hcols).to_numpy().astype(np.float64); hsum=np.maximum(hmat.sum(axis=1,keepdims=True),1.0)
hprobs=hmat/hsum; hlog=np.where(hprobs>0,np.log(hprobs+1e-12),0.0)
df_h_ent=pl.DataFrame({"account_id":hp["account_id"],"reg_hour_entropy":-np.sum(hprobs*hlog,axis=1).astype(np.float32)})
del hc,hp,hmat,hprobs,hlog; gc.collect()
dc=df_tx_time.group_by(["account_id","dow"]).len().rename({"len":"d_count"})
dp=dc.pivot(on="dow",index="account_id",values="d_count").fill_null(0)
dcols=[c for c in dp.columns if c!="account_id"]
dmat=dp.select(dcols).to_numpy().astype(np.float64); dsum=np.maximum(dmat.sum(axis=1,keepdims=True),1.0)
dprobs=dmat/dsum; dlog=np.where(dprobs>0,np.log(dprobs+1e-12),0.0)
df_d_ent=pl.DataFrame({"account_id":dp["account_id"],"reg_dow_entropy":-np.sum(dprobs*dlog,axis=1).astype(np.float32)})
del dc,dp,dmat,dprobs,dlog; gc.collect()
pf=df_tx_time.group_by(["account_id","partner"]).len().rename({"len":"p_count"})
rp=pf.filter(pl.col("p_count")>=2).group_by("account_id").len().rename({"len":"repeat_cnt"})
tp2=pf.group_by("account_id").len().rename({"len":"total_cnt"})
df_pc=tp2.join(rp,on="account_id",how="left").fill_null(0).with_columns((pl.col("repeat_cnt")/(pl.col("total_cnt")+1)).alias("reg_consistent_partner_ratio")).select(["account_id","reg_consistent_partner_ratio"])
top_p=pf.sort(["account_id","p_count"],descending=[False,True]).unique(subset=["account_id"],maintain_order=True).rename({"p_count":"top_cnt"}).select(["account_id","top_cnt"])
df_tc=reg_agg.select(["account_id","_total_tx"]).join(top_p,on="account_id",how="left").fill_null(0).with_columns((pl.col("top_cnt")/(pl.col("_total_tx")+1)).alias("reg_top_partner_concentration")).select(["account_id","reg_top_partner_concentration"])
del pf,rp,tp2,top_p; gc.collect()
df_regularity=(reg_agg.select(["account_id","reg_business_hour_ratio","reg_weekend_ratio","reg_night_strict_ratio"])
    .join(df_h_ent,on="account_id",how="left").join(df_d_ent,on="account_id",how="left")
    .join(df_pc,on="account_id",how="left").join(df_tc,on="account_id",how="left").fill_null(0.0))
regularity_cols=[c for c in df_regularity.columns if c.startswith("reg_")]
for rc in regularity_cols: df_regularity=df_regularity.with_columns(pl.when(pl.col(rc).is_infinite()|pl.col(rc).is_nan()).then(0.0).otherwise(pl.col(rc)).alias(rc))
del reg_agg,df_h_ent,df_d_ent,df_pc,df_tc; gc.collect()
print(f"  ✅ 규칙성 {len(regularity_cols)}개")

# ─── 2-5. 금액 분포 피처 8개 ───
print("  📊 금액 분포 피처...")
amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train.columns: amount_col = cand; break
if amount_col:
    df_amounts=df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col(amount_col).cast(pl.Float64).alias("amount")]).filter(pl.col("amount")>0)
    df_amt=df_amounts.group_by("account_id").agg([
        ((pl.col("amount")%1000==0)|(pl.col("amount")%5000==0)).mean().alias("amt_round_number_ratio"),
        (pl.col("amount").std()/(pl.col("amount").mean()+1)).alias("amt_cv"),
        (pl.col("amount").max()/(pl.col("amount").mean()+1)).alias("amt_max_mean_ratio"),
        (pl.col("amount").median()/(pl.col("amount").mean()+1)).alias("amt_median_mean_ratio"),
        pl.col("amount").quantile(0.9).alias("_q90"), pl.col("amount").sum().alias("_tsum"),
        pl.col("amount").count().alias("_cnt"),
        pl.col("amount").quantile(0.75).alias("_q75"), pl.col("amount").quantile(0.25).alias("_q25"),
        pl.col("amount").median().alias("_med"),
    ]).fill_null(0.0).with_columns([
        (pl.col("_q90")*0.1*pl.col("_cnt")/(pl.col("_tsum")+1)).alias("amt_top10pct_concentration"),
        ((pl.col("_q75")-pl.col("_q25"))/(pl.col("_med")+1)).alias("amt_iqr_median_ratio")])
    gm=df_amounts["amount"].mean(); st=max(gm*0.1,1.0)
    df_sr=df_amounts.with_columns((pl.col("amount")<st).alias("is_small")).group_by("account_id").agg(pl.col("is_small").mean().alias("amt_small_tx_ratio"))
    df_amt=df_amt.join(df_sr,on="account_id",how="left").fill_null(0.0)
    df_al=df_amounts.with_columns((pl.col("amount")+1).log().cast(pl.Int32).alias("log_bucket"))
    bc=df_al.group_by(["account_id","log_bucket"]).len().rename({"len":"b_count"})
    bt=bc.group_by("account_id").agg(pl.col("b_count").sum().alias("b_total"))
    bp=bc.join(bt,on="account_id").with_columns((pl.col("b_count")/pl.col("b_total")).alias("b_prob"))
    ae=bp.group_by("account_id").agg((-1*(pl.col("b_prob")*(pl.col("b_prob")+1e-12).log()).sum()).alias("amt_entropy"))
    df_amt=df_amt.join(ae,on="account_id",how="left").fill_null(0.0)
    del df_amounts,df_al,bc,bt,bp,ae,df_sr; gc.collect()
    amt_feature_cols=[c for c in df_amt.columns if c.startswith("amt_") and not c.startswith("_")]
    df_amt_features=df_amt.select(["account_id"]+amt_feature_cols)
else:
    df_amt_features=all_accounts.select("account_id"); amt_feature_cols=[]
for ac in amt_feature_cols:
    df_amt_features=df_amt_features.with_columns(pl.when(pl.col(ac).is_infinite()|pl.col(ac).is_nan()).then(0.0).otherwise(pl.col(ac)).alias(ac))
print(f"  ✅ 금액 분포 {len(amt_feature_cols)}개")

# =============================================================
# ★ NEW: 2-6. Cross-border 정밀화 피처 4개
# =============================================================
# 핵심 아이디어: ratio_cross_border 단독이 아니라
#   "cross-border × 새로운 상대방 비율 × 방향 비대칭"을 결합
print("  📊 ★ Cross-border 정밀화 피처 (신규)...")
cb_feature_cols = []

# 데이터에 to_acc country/bank 정보가 있을 경우를 가정
# → 없으면 graph_partner_diversity 와 ratio_cross_border 조합으로 근사
df_v2_agg_cb = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2 if c in df_v2_train.columns
])
df_cb = df_v2_agg_cb.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)
v2cb = set(df_cb.columns)

cb_exprs = []
# ① cb_new_partner_ratio: cross-border이면서 새로운 상대방 비율
#    = ratio_cross_border × (unique_out_partners / (total_tx_count+1))
#    → 새로운 상대에게 해외 송금하는 패턴
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("graph_unique_out_partners") /
         (pl.col("graph_total_tx_count") + 1)
        ).alias("cb_new_partner_ratio")
    )
    cb_feature_cols.append("cb_new_partner_ratio")

# ② cb_oneway_new: cross-border × 비양방향 × 새 상대방
#    → 양방향 거래 없이 새로운 상대에게만 해외 송금 (전형적 분산 송금)
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         (1.0 - pl.col("graph_reciprocity")) *
         (pl.col("graph_unique_out_partners") / (pl.col("graph_out_degree") + 1))
        ).alias("cb_oneway_new")
    )
    cb_feature_cols.append("cb_oneway_new")

# ③ cb_amount_asymmetry: 해외 송금 비율 × 금액 비대칭 (out >> in)
#    = ratio_cross_border × out_in_ratio (clip 해서 폭발 방지)
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("graph_out_in_ratio").clip(0, 20)
        ).alias("cb_amount_asymmetry")
    )
    cb_feature_cols.append("cb_amount_asymmetry")

# ④ cb_burst_cross: cross-border이 짧은 시간 내 집중 (burst_ratio 결합)
if "ratio_cross_border" in v2cb and "burst_ratio" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("burst_ratio").clip(0, 100)
        ).alias("cb_burst_cross")
    )
    cb_feature_cols.append("cb_burst_cross")

if cb_exprs:
    df_cb = df_cb.with_columns(cb_exprs)
    for cn in cb_feature_cols:
        df_cb = df_cb.with_columns(
            pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
            .then(0.0).otherwise(pl.col(cn)).alias(cn))
    df_cb_features = df_cb.select(["account_id"] + cb_feature_cols)
else:
    df_cb_features = all_accounts.select("account_id"); cb_feature_cols = []
del df_cb, df_v2_agg_cb; gc.collect()
print(f"  ✅ Cross-border 정밀화 {len(cb_feature_cols)}개: {cb_feature_cols}")

# trend_* 피처 제거 (ws_* 와 중복) — v7c.4
trend_feature_cols = []

# =============================================================
# ★ NEW: 2-8. 거래량 대비 상대 정규화 피처 5개
# =============================================================
# 핵심 아이디어: cond_lowdeg_high_amount가 FN에서 TP보다 오히려 높아 역전됨
#   → "전체 네트워크 대비 비정상적으로 높은 금액"을 더 정밀하게 포착
print("  📊 ★ 거래량 대비 상대 정규화 피처 (신규)...")
vol_norm_feature_cols = []

df_v2_agg_vn = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2 if c in df_v2_train.columns
])
df_vn = df_v2_agg_vn.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)

# 네트워크 전체 평균값 계산 (정규화 기준)
if "avg_log_amount" in set(df_vn.columns):
    global_avg_log_amount = df_vn["avg_log_amount"].mean() or 1.0
    global_avg_degree = df_vn["graph_total_degree"].mean() or 1.0
    global_avg_cnt_1h = df_vn["cnt_1h"].mean() if "cnt_1h" in df_vn.columns else 1.0
    global_avg_cnt_1h = global_avg_cnt_1h or 1.0

    vn_exprs = []
    # ① vol_amount_per_peer: 건당 금액 / (파트너 수 × 전체 평균)
    #    → degree가 낮은데 금액이 높은 패턴을 전체 대비 상대적으로 포착
    vn_exprs.append(
        (pl.col("avg_log_amount") / (float(global_avg_log_amount) + 1e-8) /
         (pl.col("graph_unique_out_partners").log1p() + 1)
        ).alias("vol_amount_per_peer")
    )
    vol_norm_feature_cols.append("vol_amount_per_peer")

    # ② vol_concentrated_amount: 소수 파트너에게 집중 송금
    #    = avg_log_amount × (1 - partner_diversity)
    #    → diversity가 낮을수록 (=한 곳에 집중) × 금액이 높을수록 높아짐
    vn_exprs.append(
        (pl.col("avg_log_amount") * (1.0 - pl.col("graph_partner_diversity").clip(0, 1))
        ).alias("vol_concentrated_amount")
    )
    vol_norm_feature_cols.append("vol_concentrated_amount")

    # ③ vol_low_freq_high_amt: 낮은 빈도이지만 건당 금액이 전체 평균 대비 높음
    #    = (avg_log_amount / global_mean) / log1p(cnt_1h) → 드문데 고액
    if "cnt_1h" in set(df_vn.columns):
        vn_exprs.append(
            (pl.col("avg_log_amount") / float(global_avg_log_amount) /
             (pl.col("cnt_1h").clip(0, 1e6).log1p() + 1)
            ).alias("vol_low_freq_high_amt")
        )
        vol_norm_feature_cols.append("vol_low_freq_high_amt")

    # ④ vol_degree_normalized_amount: 디그리로 나눈 상대 금액 (올바른 역산 버전)
    #    기존 cond_lowdeg_high_amount 대체 — log 스케일 적용 + 전체 평균으로 나눔
    vn_exprs.append(
        ((pl.col("avg_log_amount") - float(global_avg_log_amount)) /
         (pl.col("graph_total_degree").clip(1, 1e6).log1p())
        ).alias("vol_degree_normalized_amount")
    )
    vol_norm_feature_cols.append("vol_degree_normalized_amount")

    # ⑤ vol_outlier_score: z-score 방식의 금액 이상 점수
    #    = (avg_log_amount - global_mean) → 양수면 전체 평균 초과
    vn_exprs.append(
        (pl.col("avg_log_amount") - float(global_avg_log_amount)).alias("vol_outlier_score")
    )
    vol_norm_feature_cols.append("vol_outlier_score")

    df_vn = df_vn.with_columns(vn_exprs)
    for cn in vol_norm_feature_cols:
        df_vn = df_vn.with_columns(
            pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
            .then(0.0).otherwise(pl.col(cn)).alias(cn))
    df_vol_norm_features = df_vn.select(["account_id"] + vol_norm_feature_cols)
else:
    df_vol_norm_features = all_accounts.select("account_id"); vol_norm_feature_cols = []

del df_vn, df_v2_agg_vn; gc.collect()
print(f"  ✅ 거래량 대비 정규화 {len(vol_norm_feature_cols)}개: {vol_norm_feature_cols}")

# =============================================================
# ★ NEW: 2-9. 시간 윈도우별 행동 변화 피처 4개
# =============================================================
# 핵심 아이디어: 미탐의 71%가 테스트 첫 3일 집중
#   → train 말미(~recent window)에서의 행동 변화를 포착
print("  📊 ★ 시간 윈도우별 행동 변화 피처 (신규)...")
win_shift_feature_cols = []

# train 마지막 7일 vs 나머지 기간 비교
last_7d_cutoff = train_cutoff - pl.duration(days=7)
df_raw_recent = df_raw_train.filter(pl.col("Timestamp") >= last_7d_cutoff)
df_raw_older  = df_raw_train.filter(pl.col("Timestamp") < last_7d_cutoff)

recent_cnt = df_raw_recent.select(
    pl.col("from_acc").cast(pl.Utf8).alias("account_id")
).group_by("account_id").len().rename({"len": "ws_recent_cnt"})
older_cnt = df_raw_older.select(
    pl.col("from_acc").cast(pl.Utf8).alias("account_id")
).group_by("account_id").len().rename({"len": "ws_older_cnt"})

df_ws = all_accounts.select("account_id").join(recent_cnt, on="account_id", how="left").join(older_cnt, on="account_id", how="left").fill_null(0.0)

ws_exprs = []
# ① ws_recent_surge: 최근 7일 / 이전 기간 (정규화) → 갑작스러운 증가
ws_exprs.append(
    (pl.col("ws_recent_cnt") / (pl.col("ws_older_cnt") / 30.0 + 1)).alias("ws_recent_surge")
)
win_shift_feature_cols.append("ws_recent_surge")

# ② ws_recent_only: 최근 7일에만 거래가 있는 계좌 (0/1)
#    → 갑자기 나타난 계좌
ws_exprs.append(
    ((pl.col("ws_recent_cnt") > 0) & (pl.col("ws_older_cnt") == 0)).cast(pl.Float64).alias("ws_recent_only")
)
win_shift_feature_cols.append("ws_recent_only")

df_ws = df_ws.with_columns(ws_exprs)

# ③ 최근 7일 고유 파트너 급증
recent_partners = df_raw_recent.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner")
]).group_by("account_id").agg(pl.col("partner").n_unique().alias("ws_recent_partners"))
older_partners = df_raw_older.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner")
]).group_by("account_id").agg(pl.col("partner").n_unique().alias("ws_older_partners"))

df_ws = df_ws.join(recent_partners, on="account_id", how="left").join(older_partners, on="account_id", how="left").fill_null(0.0)

# ④ ws_partner_explosion: 최근 7일 파트너가 이전 대비 급증
df_ws = df_ws.with_columns(
    (pl.col("ws_recent_partners") / (pl.col("ws_older_partners") + 1)).alias("ws_partner_explosion")
)
win_shift_feature_cols.append("ws_partner_explosion")

for cn in win_shift_feature_cols:
    df_ws = df_ws.with_columns(
        pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
        .then(0.0).otherwise(pl.col(cn)).alias(cn))

df_win_shift_features = df_ws.select(["account_id"] + win_shift_feature_cols)
del df_raw_recent, df_raw_older, df_ws, recent_cnt, older_cnt
del recent_partners, older_partners; gc.collect()
print(f"  ✅ 시간 윈도우 변화 {len(win_shift_feature_cols)}개: {win_shift_feature_cols}")

# =============================================================
# ★ NEW: 2-10. 입금/출금 독립 피처 6개
# =============================================================
# 핵심 아이디어: V2에 sum_in_1h/sum_out_1h는 있지만
#   AML 점수/조건부 피처에서 in/out 독립적 분석이 부족
print("  📊 ★ 입금/출금 독립 피처 (신규)...")
inout_feature_cols = []

# 출금(from) 집계
df_out_tx = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col(amount_col).cast(pl.Float64).alias("amount") if amount_col else pl.lit(1.0).alias("amount")
]).filter(pl.col("amount") > 0)

# 입금(to) 집계
df_in_tx = df_raw_train.select([
    pl.col("to_acc").cast(pl.Utf8).alias("account_id"),
    pl.col(amount_col).cast(pl.Float64).alias("amount") if amount_col else pl.lit(1.0).alias("amount")
]).filter(pl.col("amount") > 0)

out_stats = df_out_tx.group_by("account_id").agg([
    pl.col("amount").sum().alias("inout_total_out_amount"),
    pl.col("amount").mean().alias("inout_avg_out_amount"),
    pl.col("amount").count().alias("inout_out_cnt"),
])
in_stats = df_in_tx.group_by("account_id").agg([
    pl.col("amount").sum().alias("inout_total_in_amount"),
    pl.col("amount").mean().alias("inout_avg_in_amount"),
    pl.col("amount").count().alias("inout_in_cnt"),
])
df_inout = all_accounts.select("account_id").join(out_stats, on="account_id", how="left").join(in_stats, on="account_id", how="left").fill_null(0.0)

inout_exprs = []
# ① inout_out_amount_ratio: 출금액 / (입금액 + 출금액) → 1이면 순수 송금자
inout_exprs.append(
    (pl.col("inout_total_out_amount") /
     (pl.col("inout_total_out_amount") + pl.col("inout_total_in_amount") + 1)
    ).alias("inout_out_amount_ratio")
)
inout_feature_cols.append("inout_out_amount_ratio")

# ② inout_avg_out_vs_in: 평균 출금액 / 평균 입금액 → 1 초과면 출금이 더 큰 계좌
inout_exprs.append(
    (pl.col("inout_avg_out_amount") / (pl.col("inout_avg_in_amount") + 1)
    ).alias("inout_avg_out_vs_in")
)
inout_feature_cols.append("inout_avg_out_vs_in")

# ③ inout_cnt_imbalance: (출금 건수 - 입금 건수) / 전체 건수
#    → 양수면 보내는 쪽이 훨씬 많은 계좌 (smurf 패턴)
inout_exprs.append(
    ((pl.col("inout_out_cnt") - pl.col("inout_in_cnt")) /
     (pl.col("inout_out_cnt") + pl.col("inout_in_cnt") + 1)
    ).alias("inout_cnt_imbalance")
)
inout_feature_cols.append("inout_cnt_imbalance")

# ④ inout_fan_out_score: 출금 건수 × 고유 파트너 다양성 → 여러 곳에 분산 송금
df_inout = df_inout.join(df_graph_feats.select(["account_id", "graph_partner_diversity", "graph_unique_out_partners"]), on="account_id", how="left").fill_null(0.0)
inout_exprs.append(
    (pl.col("inout_out_cnt").log1p() * pl.col("graph_partner_diversity")
    ).alias("inout_fan_out_score")
)
inout_feature_cols.append("inout_fan_out_score")

# ⑤ inout_in_concentration: 입금이 소수에서만 들어옴 (mule 패턴)
#    = avg_in_amount × (1 - partner_diversity) → 소수 소스에서 집중 수취
inout_exprs.append(
    (pl.col("inout_avg_in_amount").log1p() * (1.0 - pl.col("graph_partner_diversity").clip(0, 1))
    ).alias("inout_in_concentration")
)
inout_feature_cols.append("inout_in_concentration")

# ⑥ inout_net_flow: log(출금합) - log(입금합) → 순 흐름 방향성
inout_exprs.append(
    (pl.col("inout_total_out_amount").log1p() - pl.col("inout_total_in_amount").log1p()
    ).alias("inout_net_flow")
)
inout_feature_cols.append("inout_net_flow")

df_inout = df_inout.with_columns(inout_exprs)
for cn in inout_feature_cols:
    df_inout = df_inout.with_columns(
        pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
        .then(0.0).otherwise(pl.col(cn)).alias(cn))
df_inout_features = df_inout.select(["account_id"] + inout_feature_cols)
del df_out_tx, df_in_tx, out_stats, in_stats, df_inout; gc.collect()
print(f"  ✅ 입금/출금 독립 {len(inout_feature_cols)}개: {inout_feature_cols}")
log_memory("전체 노드 피처 완료")

# =============================================================
# 3. GNN 엣지/노드 구성 — 강화된 입력
# =============================================================
print("\n🔗 [Step 3] GNN 엣지/노드 구성 (Enriched Input)...")

df_raw_reload = pl.read_parquet(raw_path)
if df_raw_reload["Timestamp"].dtype == pl.Utf8:
    df_raw_reload = df_raw_reload.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_raw_train_edges = df_raw_reload.filter(pl.col("Timestamp")<train_cutoff); del df_raw_reload; gc.collect()

edge_amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train_edges.columns: edge_amount_col = cand; break

df_edges = df_raw_train_edges.select(
    ["from_acc","to_acc","Timestamp"] +
    ([edge_amount_col] if edge_amount_col else []) +
    (["Payment Format"] if "Payment Format" in df_raw_train_edges.columns else []) +
    (["Is Laundering"] if "Is Laundering" in df_raw_train_edges.columns else [])
).with_columns([pl.col("from_acc").cast(pl.Utf8), pl.col("to_acc").cast(pl.Utf8)])

df_edges = df_edges.join(all_accounts.rename({"account_id":"from_acc","node_id":"src_id"}),on="from_acc",how="left")
df_edges = df_edges.join(all_accounts.rename({"account_id":"to_acc","node_id":"dst_id"}),on="to_acc",how="left")

ts = df_edges["Timestamp"]; mn_ts=ts.min(); mx_ts=ts.max(); ts_rng=(mx_ts-mn_ts).total_seconds()
etn = ((ts-mn_ts).dt.total_microseconds().cast(pl.Float64)/1e6/max(ts_rng,1.0)).to_numpy().astype(np.float32)
hrs = df_edges["Timestamp"].dt.hour().to_numpy().astype(np.float32)
dow = df_edges["Timestamp"].dt.weekday().to_numpy().astype(np.float32)
time_feats = np.stack([etn, np.sin(2*np.pi*hrs/24), np.cos(2*np.pi*hrs/24),
                        np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7)], axis=1)

if edge_amount_col:
    amounts_raw = df_edges[edge_amount_col].cast(pl.Float64).fill_null(0.0).to_numpy().astype(np.float32)
    log_amounts = np.log1p(np.maximum(amounts_raw, 0)).astype(np.float32)
    log_amounts = log_amounts / (np.max(log_amounts) + 1e-8)
    round_flag = ((amounts_raw % 1000 == 0) | (amounts_raw % 5000 == 0)).astype(np.float32)
    global_mean_amt = np.mean(amounts_raw[amounts_raw > 0]) if np.any(amounts_raw > 0) else 1.0
    relative_amt = np.log1p(amounts_raw / (global_mean_amt + 1e-8)).astype(np.float32)
    relative_amt = relative_amt / (np.max(relative_amt) + 1e-8)
    amount_feats = np.stack([log_amounts, round_flag, relative_amt], axis=1)
else:
    amount_feats = np.zeros((len(df_edges), 3), dtype=np.float32)

if "Payment Format" in df_edges.columns:
    pf_series = df_edges["Payment Format"].cast(pl.Utf8).fill_null("unknown")
    is_wire = pf_series.str.contains("(?i)wire|swift|cheque").to_numpy().astype(np.float32)
    is_internal = pf_series.str.contains("(?i)ach|internal|reinvestment").to_numpy().astype(np.float32)
    type_feats = np.stack([is_wire, is_internal], axis=1)
else:
    type_feats = np.zeros((len(df_edges), 2), dtype=np.float32)

edge_features_all = np.concatenate([time_feats, amount_feats, type_feats], axis=1)
EDGE_RAW_DIM = edge_features_all.shape[1]
print(f"  📊 Edge 피처: {EDGE_RAW_DIM}개")

edge_index_train = torch.tensor(df_edges.select(["src_id","dst_id"]).to_numpy().T, dtype=torch.long)
edge_attr_train = torch.tensor(edge_features_all, dtype=torch.float32)
print(f"📊 Train 엣지: {edge_index_train.shape[1]:,}")

del df_raw_train, df_raw_train_edges, df_edges, ts, hrs, dow, etn
del time_feats, amount_feats, type_feats, edge_features_all; gc.collect()

# ─── 노드 피처: V2(38) + 그래프(13) + 조건부(10) + AML(5)
#              + 규칙성(7) + 금액(8) + CB(4) + 정규화(5) + 윈도우(4) + 입출금(6) ───
print("  📊 노드 피처 구성 (신규 피처 포함)...")
df_v2_node = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])
df_node_all = (all_accounts.join(df_v2_node, on="account_id", how="left")
    .join(df_graph_feats, on="account_id", how="left")
    .join(df_cond_features, on="account_id", how="left")
    .join(df_aml_scores, on="account_id", how="left")
    .join(df_regularity, on="account_id", how="left")
    .join(df_amt_features, on="account_id", how="left")
    .join(df_cb_features, on="account_id", how="left")          # ★ 신규
    .join(df_vol_norm_features, on="account_id", how="left")    # ★ 신규
    .join(df_win_shift_features, on="account_id", how="left")   # ★ 신규
    .join(df_inout_features, on="account_id", how="left")       # ★ 신규
    .fill_null(0.0))

gnn_input_cols = (gnn_feature_cols_v2 + graph_feature_cols + cond_feature_cols +
                  aml_score_cols + regularity_cols + amt_feature_cols +
                  cb_feature_cols + vol_norm_feature_cols +
                  win_shift_feature_cols + inout_feature_cols)

for col in gnn_input_cols:
    if col in df_node_all.columns:
        df_node_all = df_node_all.with_columns(
            pl.when(pl.col(col).is_infinite()|pl.col(col).is_nan()).then(0.0).otherwise(pl.col(col)).alias(col))

X_node = torch.tensor(df_node_all.select(gnn_input_cols).to_numpy(), dtype=torch.float32)
IN_DIM = X_node.shape[1]
print(f"  ✅ GNN 노드 입력: {IN_DIM}차원")
print(f"     = V2({len(gnn_feature_cols_v2)}) + 그래프({len(graph_feature_cols)}) + 조건부({len(cond_feature_cols)})")
print(f"       + AML({len(aml_score_cols)}) + 규칙성({len(regularity_cols)}) + 금액({len(amt_feature_cols)})")
print(f"       + CB정밀화({len(cb_feature_cols)}) + 정규화({len(vol_norm_feature_cols)})")
print(f"       + 윈도우({len(win_shift_feature_cols)}) + 입출금({len(inout_feature_cols)})")

target_labels = df_v2_train.filter(pl.col("is_laundering")==1).select("account_id").unique().with_columns(pl.lit(1).alias("label"))
df_labels = all_accounts.join(target_labels,on="account_id",how="left").fill_null(0)
Y_node = torch.tensor(df_labels["label"].to_numpy(), dtype=torch.long)
pos_node_ids = set(df_labels.filter(pl.col("label")==1)["node_id"].to_list())
active_df = pl.DataFrame({"account_id":df_v2_train["account_id"].unique(),"is_active":True})
mask_df = all_accounts.join(active_df,on="account_id",how="left").fill_null(False)
active_node_ids = mask_df.filter(pl.col("is_active"))["node_id"].to_list()
active_pos = [n for n in active_node_ids if n in pos_node_ids]
active_neg = [n for n in active_node_ids if n not in pos_node_ids]
n_neg_gnn = min(len(active_pos)*NEG_NODE_RATIO_GNN, len(active_neg))
np.random.seed(RANDOM_SEED)
sampled_neg = np.random.choice(active_neg,size=n_neg_gnn,replace=False).tolist()
sampled_nodes = set(active_pos+sampled_neg)
train_mask_np = np.zeros(num_nodes, dtype=bool)
for nid in sampled_nodes: train_mask_np[nid]=True
train_mask = torch.tensor(train_mask_np, dtype=torch.bool)
print(f"📊 GNN 노드: 양성 {len(active_pos):,} + 음성 {n_neg_gnn:,}")
del df_v2_node, df_node_all, mask_df, active_df; gc.collect()

graph_data = Data(x=X_node, edge_index=edge_index_train, edge_attr=edge_attr_train, y=Y_node)
graph_data.train_mask = train_mask; graph_data.num_nodes = num_nodes
del X_node, edge_index_train, edge_attr_train, Y_node, train_mask; gc.collect()
log_memory("GNN 데이터 준비 완료")

# =============================================================
# 4-6. Multi-Config GNN 탐색 → 최적 임베딩 선택
# =============================================================
print("\n🧠 [Step 4-6] Multi-Config GNN 탐색...")

class EdgeProjection(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, out_dim), nn.GELU(), nn.Linear(out_dim, out_dim))
    def forward(self, ea): return self.proj(ea)

class TGAT_Flex(nn.Module):
    def __init__(self, in_dim, hidden_dim, edge_raw_dim, edge_dim, n_heads, n_layers=2, dropout=0.2):
        super().__init__()
        self.edge_proj = EdgeProjection(edge_raw_dim, edge_dim)
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.n_layers = n_layers
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_layers)])
        self.convs = nn.ModuleList([
            TransformerConv(hidden_dim, hidden_dim//n_heads, heads=n_heads,
                            edge_dim=edge_dim, dropout=dropout, concat=True)
            for _ in range(n_layers)])
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim//2, 1))
        self.dropout = dropout; self.hidden_dim = hidden_dim

    def forward(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        for i in range(self.n_layers):
            h = h + F.dropout(self.convs[i](self.norms[i](h), edge_index, ee), p=self.dropout, training=self.training)
        return self.classifier(self.norm_out(h)).squeeze(-1)

    @torch.no_grad()
    def get_embedding(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        for i in range(self.n_layers):
            h = h + self.convs[i](self.norms[i](h), edge_index, ee)
        return self.norm_out(h)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"📌 디바이스: {device}")
pw = (len(sampled_nodes)-len(active_pos))/max(len(active_pos),1)
print(f"📊 pos_weight: {pw:.1f}")

def make_lr(opt, ep, n_ep):
    if ep < WARMUP_EPOCHS: lr = 1e-6+(BASE_LR-1e-6)*(ep/WARMUP_EPOCHS)
    else: p=(ep-WARMUP_EPOCHS)/max(n_ep-WARMUP_EPOCHS,1); lr=1e-5+(BASE_LR-1e-5)*0.5*(1+np.cos(np.pi*p))
    for pg in opt.param_groups: pg['lr']=lr
    return lr

GNN_CONFIGS = [
    {"label":"h64_L2",  "hidden_dim":64,  "n_layers":2, "n_heads":4, "edge_dim":16},
    {"label":"h128_L2", "hidden_dim":128, "n_layers":2, "n_heads":4, "edge_dim":32},
    {"label":"h64_L3",  "hidden_dim":64,  "n_layers":3, "n_heads":4, "edge_dim":16},
    {"label":"h128_L3", "hidden_dim":128, "n_layers":3, "n_heads":4, "edge_dim":32},
]

xgb_exclude_cols=["account_id","time_group","is_laundering","mode_format","currency_mode","date"]
xgb_v2_cols=[c for c in df_v2.columns if c not in xgb_exclude_cols]
apr_train = df_v2_train.filter(pl.col("is_laundering")==1).height / max(df_v2_train.height,1)
spw = max((1-apr_train)/apr_train, 1.0) if apr_train > 0 else 20.0

# ★ 신규 피처들을 base_feature_cols에 포함
base_feature_cols = (xgb_v2_cols + graph_feature_cols + cond_feature_cols + aml_score_cols +
                     cb_feature_cols + vol_norm_feature_cols +
                     win_shift_feature_cols + inout_feature_cols)

def build_eval_df(db, df_emb_local, emb_col_names):
    feature_cols = base_feature_cols + emb_col_names
    df = (db.join(df_graph_feats,on="account_id",how="left")
          .join(df_cond_features,on="account_id",how="left")
          .join(df_aml_scores,on="account_id",how="left")
          .join(df_cb_features,on="account_id",how="left")            # ★
          .join(df_vol_norm_features,on="account_id",how="left")      # ★
          .join(df_win_shift_features,on="account_id",how="left")     # ★
          .join(df_inout_features,on="account_id",how="left")         # ★
          .join(df_emb_local,on="account_id",how="left")
          .fill_null(0.0).fill_nan(0.0))
    for cn in feature_cols:
        if cn in df.columns:
            df = df.with_columns(pl.when(pl.col(cn).is_infinite()).then(0.0).otherwise(pl.col(cn)).alias(cn))
    return df, feature_cols

def quick_xgb_eval(df_emb_local, emb_col_names, label):
    df_tr, fcols = build_eval_df(df_v2.filter(pl.col("time_group")<train_cutoff), df_emb_local, emb_col_names)
    df_tr_pos=df_tr.filter(pl.col("is_laundering")==1); df_tr_neg=df_tr.filter(pl.col("is_laundering")==0)
    n_neg=min(len(df_tr_pos)*NEG_ROW_RATIO_XGB,len(df_tr_neg))
    df_tr_s=pl.concat([df_tr_pos,df_tr_neg.sample(n=n_neg,seed=RANDOM_SEED)]).sample(fraction=1.0,seed=RANDOM_SEED)
    X_tr=df_tr_s.select(fcols).to_pandas(); y_tr=df_tr_s["is_laundering"].to_numpy()
    del df_tr,df_tr_pos,df_tr_neg,df_tr_s; gc.collect()
    df_vl, _ = build_eval_df(df_v2.filter((pl.col("time_group")>=train_cutoff)&(pl.col("time_group")<val_cutoff)), df_emb_local, emb_col_names)
    X_vl=df_vl.select(fcols).to_pandas(); y_vl=df_vl["is_laundering"].to_numpy(); del df_vl; gc.collect()
    df_te, _ = build_eval_df(df_v2.filter(pl.col("time_group")>=val_cutoff), df_emb_local, emb_col_names)
    X_te=df_te.select(fcols).to_pandas(); y_te=df_te["is_laundering"].to_numpy(); del df_te; gc.collect()
    qp={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
        "random_state":42,"max_depth":8,"learning_rate":0.03,"scale_pos_weight":spw,
        "subsample":0.8,"colsample_bytree":0.7,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
    m=xgb.XGBClassifier(**qp,n_estimators=1500,early_stopping_rounds=100)
    m.fit(X_tr,y_tr,eval_set=[(X_vl,y_vl)],verbose=0)
    q_val=m.best_score; q_test=average_precision_score(y_te,m.predict_proba(X_te)[:,1])
    del m,X_tr,X_vl,X_te,y_tr,y_vl,y_te; gc.collect()
    return q_val, q_test

print(f"\n📊 {len(GNN_CONFIGS)}개 GNN 설정 탐색...")
print(f"  {'#':>2s} {'Label':<12s} | {'hidden':>6s} {'layers':>6s} {'edim':>4s} | {'Loss':>8s} {'Params':>8s} | {'QVal':>8s} {'QTest':>8s}")
print("  "+"-"*80)

gnn_results = []
for gi, gcfg in enumerate(GNN_CONFIGS):
    gl = gcfg["label"]; hd=gcfg["hidden_dim"]; nl=gcfg["n_layers"]
    nh=gcfg["n_heads"]; ed=gcfg["edge_dim"]
    train_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                                   input_nodes=graph_data.train_mask, shuffle=True, num_workers=4)
    full_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                                  input_nodes=None, shuffle=False, num_workers=4)
    model = TGAT_Flex(in_dim=IN_DIM, hidden_dim=hd, edge_raw_dim=EDGE_RAW_DIM,
                      edge_dim=ed, n_heads=nh, n_layers=nl, dropout=0.2).to(device)
    np_count = sum(p.numel() for p in model.parameters())
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw],device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6, weight_decay=1e-4)
    model.train(); bl=float('inf'); pc2=0; bs=None
    for epoch in range(N_EPOCHS_GNN):
        clr=make_lr(optimizer,epoch,N_EPOCHS_GNN); tl=0.0; nb2=0
        for batch in tqdm(train_loader,desc=f"[{gl}] Ep{epoch+1}",leave=False):
            batch=batch.to(device); optimizer.zero_grad()
            bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
            out=model(batch.x,batch.edge_index,bea)
            loss=criterion(out[:batch.batch_size],batch.y[:batch.batch_size].float())
            if torch.isnan(loss) or torch.isinf(loss): optimizer.zero_grad(); continue
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=0.5)
            optimizer.step(); tl+=loss.item(); nb2+=1
        al=tl/max(nb2,1)
        if (epoch+1)%10==0 or epoch==0: print(f"    [{gl}] Ep{epoch+1:2d} Loss={al:.4f}")
        if al<bl-0.0005: bl=al; pc2=0; bs={k:v.cpu().clone() for k,v in model.state_dict().items()}
        else: pc2+=1
        if pc2>=PATIENCE: break
    if bs: model.load_state_dict({k:v.to(device) for k,v in bs.items()}); del bs
    model.eval(); ae=[]
    with torch.no_grad():
        for batch in tqdm(full_loader,desc=f"[{gl}] Embed",leave=False):
            batch=batch.to(device)
            bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
            emb=model.get_embedding(batch.x,batch.edge_index,bea)
            ae.append(emb[:batch.batch_size].cpu())
    femb=torch.cat(ae,dim=0).numpy()
    del model,ae,train_loader,full_loader; torch.cuda.empty_cache(); gc.collect()
    ecn=[f"tgat_emb_{i}" for i in range(hd)]
    df_emb=pl.concat([all_accounts.select("account_id"),pl.DataFrame(femb,schema=ecn)],how="horizontal")
    qv,qt = quick_xgb_eval(df_emb, ecn, gl)
    print(f"  {gi+1:2d} {gl:<12s} | {hd:6d} {nl:6d} {ed:4d} | {bl:8.4f} {np_count:8,} | {qv:8.5f} {qt:8.4f}")
    gnn_results.append({"label":gl,"hidden_dim":hd,"n_layers":nl,"loss":bl,"params":np_count,
                         "q_val":qv,"q_test":qt,"embedding":femb,"ecn":ecn,"df_emb":df_emb})
    log_memory(f"{gl}")

best_gnn = max(gnn_results, key=lambda x: x["q_val"])
print(f"\n  🏆 Best GNN: {best_gnn['label']} (QVal={best_gnn['q_val']:.5f} QTest={best_gnn['q_test']:.4f})")
for r in gnn_results:
    if r is not best_gnn: del r["embedding"], r["df_emb"]
gc.collect()

emb_cols = best_gnn["ecn"]
df_tgat = best_gnn["df_emb"]
FINAL_HIDDEN_DIM = best_gnn["hidden_dim"]

print(f"\n🔗 Interaction 피처 ({best_gnn['label']})...")
df_tgat_with_pr=df_tgat.join(df_graph_feats.select(["account_id","graph_pagerank"]),on="account_id",how="left").fill_null(0.0)
emb_np=df_tgat_with_pr.select(emb_cols).to_numpy(); emb_var=np.var(emb_np,axis=0)
top_idx=np.argsort(emb_var)[::-1][:N_INTERACTION_DIMS]; top_ec=[emb_cols[i] for i in top_idx]
ie_l=[]; ic_l=[]
for cn in top_ec: nn2=f"interact_pr_x_{cn}"; ie_l.append((pl.col(cn)*pl.col("graph_pagerank")*1e6).alias(nn2)); ic_l.append(nn2)
ie_l.append((sum(pl.col(c)**2 for c in emb_cols).sqrt()*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_norm")); ic_l.append("interact_pr_x_emb_norm")
ie_l.append((sum(pl.col(c) for c in top_ec)*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_topsum")); ic_l.append("interact_pr_x_emb_topsum")
interaction_cols=ic_l
df_interactions=df_tgat_with_pr.with_columns(ie_l).select(["account_id"]+ic_l)
for c in ic_l: df_interactions=df_interactions.with_columns(pl.when(pl.col(c).is_infinite()|pl.col(c).is_nan()).then(0.0).otherwise(pl.col(c)).alias(c))
print(f"  ✅ Interaction {len(ic_l)}개")
del df_tgat_with_pr,emb_np,best_gnn["embedding"]; gc.collect()
del graph_data; torch.cuda.empty_cache(); gc.collect()
log_memory("GNN 탐색 완료")

# =============================================================
# 7. XGBoost 데이터 구성
# =============================================================
print("\n🚀 [Step 7] XGBoost 데이터 구성...")
xgb_feature_cols = (xgb_v2_cols + graph_feature_cols + cond_feature_cols + aml_score_cols +
                    cb_feature_cols + vol_norm_feature_cols +
                    win_shift_feature_cols + inout_feature_cols +
                    emb_cols + interaction_cols)
print(f"📊 XGBoost 총 피처: {len(xgb_feature_cols)}")
print(f"   V2:{len(xgb_v2_cols)} G:{len(graph_feature_cols)} Cond:{len(cond_feature_cols)} AML:{len(aml_score_cols)}")
print(f"   CB:{len(cb_feature_cols)} VolNorm:{len(vol_norm_feature_cols)}")
print(f"   WinShift:{len(win_shift_feature_cols)} InOut:{len(inout_feature_cols)}")
print(f"   Emb:{len(emb_cols)} Inter:{len(interaction_cols)}")

def build_xgb_df(db):
    df=(db.join(df_graph_feats,on="account_id",how="left")
          .join(df_cond_features,on="account_id",how="left")
          .join(df_aml_scores,on="account_id",how="left")
          .join(df_cb_features,on="account_id",how="left")
          .join(df_vol_norm_features,on="account_id",how="left")
          .join(df_win_shift_features,on="account_id",how="left")
          .join(df_inout_features,on="account_id",how="left")
          .join(df_tgat,on="account_id",how="left")
          .join(df_interactions,on="account_id",how="left")
          .fill_null(0.0).fill_nan(0.0))
    for cn in xgb_feature_cols:
        if cn in df.columns: df=df.with_columns(pl.when(pl.col(cn).is_infinite()).then(0.0).otherwise(pl.col(cn)).alias(cn))
    return df

df_tr=build_xgb_df(df_v2.filter(pl.col("time_group")<train_cutoff))
df_tr_pos=df_tr.filter(pl.col("is_laundering")==1); df_tr_neg=df_tr.filter(pl.col("is_laundering")==0)
n_neg=min(len(df_tr_pos)*NEG_ROW_RATIO_XGB,len(df_tr_neg))
df_tr_s=pl.concat([df_tr_pos,df_tr_neg.sample(n=n_neg,seed=RANDOM_SEED)]).sample(fraction=1.0,seed=RANDOM_SEED)
X_train=df_tr_s.select(xgb_feature_cols).to_pandas(); y_train=df_tr_s["is_laundering"].to_numpy()
print(f"📊 Train: {len(df_tr):,} → {len(df_tr_s):,} ({y_train.mean()*100:.2f}%)")
del df_tr,df_tr_pos,df_tr_neg,df_tr_s; gc.collect()
df_vl=build_xgb_df(df_v2.filter((pl.col("time_group")>=train_cutoff)&(pl.col("time_group")<val_cutoff)))
X_val=df_vl.select(xgb_feature_cols).to_pandas(); y_val=df_vl["is_laundering"].to_numpy(); print(f"📊 Val: {len(X_val):,}"); del df_vl; gc.collect()
df_te=build_xgb_df(df_v2.filter(pl.col("time_group")>=val_cutoff))
df_test_meta=df_te.select(["account_id","time_group","is_laundering"])
X_test=df_te.select(xgb_feature_cols).to_pandas(); y_test=df_te["is_laundering"].to_numpy(); print(f"📊 Test: {len(X_test):,}")
del df_te,df_v2,df_tgat,df_graph_feats,df_cond_features,df_aml_scores
del df_cb_features,df_vol_norm_features,df_win_shift_features,df_inout_features
del df_interactions,df_regularity,df_amt_features; gc.collect()
log_memory("XGBoost 준비")

# =============================================================
# 8. XGBoost 2-Stage + 확장 HP 그리드
# =============================================================
print("\n🔥 [Step 8] 2-Stage Pruning + 확장 HP 그리드...")
apr=y_train.sum()/len(y_train); spw=max((1-apr)/apr,1.0)

print("\n── Stage 1 ──")
s1p={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda","random_state":42,"max_depth":8,"learning_rate":0.03,"scale_pos_weight":spw,"subsample":0.8,"colsample_bytree":0.7,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
m_s1=xgb.XGBClassifier(**s1p,n_estimators=1500,early_stopping_rounds=100)
m_s1.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=100)
s1_val=m_s1.best_score; s1_test=average_precision_score(y_test,m_s1.predict_proba(X_test)[:,1])
print(f"\n  Stage1: Val={s1_val:.5f} Test={s1_test:.4f}")
imp_s1=m_s1.feature_importances_; feat_imp=sorted(zip(xgb_feature_cols,imp_s1),key=lambda x:x[1],reverse=True)
del m_s1; gc.collect()

print("\n── Stage 2: 확장 HP 그리드 ──")
bp2={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda","random_state":42,"scale_pos_weight":spw}
configs=[]
for nf in [45,50,55,60,65,70,80]:
    sel=[n for n,_ in feat_imp[:nf]]
    configs.append({"label":f"top{nf}_d8_c07","features":sel,"hp":{"max_depth":8,"learning_rate":0.03,"colsample_bytree":0.7,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":1500}})
    configs.append({"label":f"top{nf}_lr02_c08","features":sel,"hp":{"max_depth":8,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}})
    configs.append({"label":f"top{nf}_lr02_c09","features":sel,"hp":{"max_depth":8,"learning_rate":0.02,"colsample_bytree":0.9,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}})
    configs.append({"label":f"top{nf}_d9_lr02","features":sel,"hp":{"max_depth":9,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}})
    configs.append({"label":f"top{nf}_reg","features":sel,"hp":{"max_depth":8,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.75,"min_child_weight":8,"gamma":0.2,"reg_alpha":0.5,"reg_lambda":2.0,"n_estimators":2500}})
configs.append({"label":"all","features":xgb_feature_cols,"hp":{"max_depth":8,"learning_rate":0.03,"colsample_bytree":0.7,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":1500}})

print(f"\n📊 {len(configs)}개 조합 탐색...")
print(f"  {'#':>2s} {'Label':<22s} | {'#f':>3s} {'d':>2s} {'LR':>5s} {'col':>4s} | {'Val':>9s} {'it':>5s}")
print("  "+"-"*65)
best_s2={"val":-1}
for i,cfg in enumerate(configs):
    hp=cfg["hp"].copy(); ne=hp.pop("n_estimators"); sf=cfg["features"]
    m=xgb.XGBClassifier(**{**bp2,**hp},n_estimators=ne,early_stopping_rounds=100)
    m.fit(X_train[sf],y_train,eval_set=[(X_val[sf],y_val)],verbose=0)
    va=m.best_score; bi=m.best_iteration
    print(f"  {i+1:2d} {cfg['label']:<22s} | {len(sf):3d} {hp.get('max_depth',8):2d} {hp.get('learning_rate',0.03):5.3f} {hp.get('colsample_bytree',0.7):4.2f} | {va:9.5f} {bi:5d}")
    if va>best_s2["val"]:
        if best_s2.get("model"): del best_s2["model"]
        best_s2={"label":cfg["label"],"features":sf,"val":va,"iter":bi,"model":m,"hp":hp}
    else: del m
    gc.collect()

y_prob=best_s2["model"].predict_proba(X_test[best_s2["features"]])[:,1]
best_features=best_s2["features"]
print(f"\n  🏆 Best: {best_s2['label']} ({len(best_features)}개) Val={best_s2['val']:.5f} iter={best_s2['iter']}")

imp=best_s2["model"].feature_importances_; tidx=np.argsort(imp)[::-1][:30]
print(f"\n🔍 Feature Importance Top 30")
for i,idx in enumerate(tidx):
    c=best_features[idx]; t=""
    if c.startswith("tgat_emb_"): t=" [TGAT]"
    elif c.startswith("graph_"): t=" [GRAPH]"
    elif c.startswith("cond_"): t=" [COND]"
    elif c.startswith("score_"): t=" [SCORE]"
    elif c.startswith("interact_"): t=" [INTER]"
    elif c.startswith("cb_"): t=" [★CB]"
    elif c.startswith("vol_"): t=" [★VOL]"
    elif c.startswith("ws_"): t=" [★WIN]"
    elif c.startswith("inout_"): t=" [★INOUT]"
    print(f"  {i+1:2d}. {c:<45s}: {imp[idx]:.4f}{t}")

ti=imp.sum()
def group_imp(prefix): return sum(imp[i] for i,c in enumerate(best_features) if c.startswith(prefix))
print(f"\n📊 그룹별 중요도:")
print(f"   V2        : {group_imp('') / ti * 100:.1f}% (기존)")
print(f"   Graph     : {group_imp('graph_') / ti * 100:.1f}%")
print(f"   TGAT      : {group_imp('tgat_emb_') / ti * 100:.1f}%")
print(f"   Interact  : {group_imp('interact_') / ti * 100:.1f}%")
print(f"   ★ CB      : {group_imp('cb_') / ti * 100:.1f}%")
print(f"   ★ VolNorm : {group_imp('vol_') / ti * 100:.1f}%")
print(f"   ★ WinShift: {group_imp('ws_') / ti * 100:.1f}%")
print(f"   ★ InOut   : {group_imp('inout_') / ti * 100:.1f}%")
del X_train,X_val; gc.collect()

# =============================================================
# 9. 최종 리포트
# =============================================================
print("\n"+"="*60); print("🏆 [TGAT v7c.4] 최종 리포트"); print("="*60)
auprc=average_precision_score(y_test,y_prob); print(f"\n  AUPRC: {auprc:.4f}")
best_f1=0; best_thresh=0
for th in np.arange(0.05,0.95,0.01):
    f=f1_score(y_test,(y_prob>=th).astype(int),zero_division=0)
    if f>best_f1: best_f1=f; best_thresh=th
y_pred=(y_prob>=best_thresh).astype(int)
prec=precision_score(y_test,y_pred,zero_division=0); rec=recall_score(y_test,y_pred,zero_division=0)
print(f"\n📌 최적: T={best_thresh:.2f} F1={best_f1:.4f} P={prec:.4f} R={rec:.4f}")
print(f"\n📌 임계값별:")
for t in [0.1,0.3,0.5,0.7,0.8,0.9]:
    yp=(y_prob>=t).astype(int)
    print(f"  T={t:.1f} P={precision_score(y_test,yp,zero_division=0):.4f} R={recall_score(y_test,yp,zero_division=0):.4f} F1={f1_score(y_test,yp,zero_division=0):.4f}")

df_res=df_test_meta.with_columns([pl.col("time_group").dt.date().alias("date"),pl.Series("pred_prob",y_prob)])
df_dist=df_res.sort("pred_prob",descending=True).unique(subset=["account_id"],maintain_order=True)
print(f"\n📌 Top-K:")
for k in [100,500,1000,5000,10000]:
    ck=min(k,len(df_dist)); hits=df_dist.head(ck)["is_laundering"].sum()
    print(f"  Top-{k:5d}: {hits:5d}명 ({hits/ck*100:.2f}%)")
print(f"\n📌 일별 Top-100:")
for d in df_dist["date"].unique(maintain_order=True).sort(descending=True):
    hits=df_dist.filter(pl.col("date")==d).head(100)["is_laundering"].sum(); print(f"  {d}: {hits}")

tn,fp,fn,tp=confusion_matrix(y_test,y_pred).ravel()
print(f"\n📌 v7c.2→v7c.4: AUPRC {auprc:.4f} | F1 {best_f1:.4f}")
print(f"  TP:{tp:,} FP:{fp:,} FN:{fn:,} TN:{tn:,}")
print(f"  Precision:{prec:.4f} Recall:{rec:.4f}")

print(f"\n📌 GNN 탐색 결과:")
for r in gnn_results:
    tag = " ★" if r["label"]==best_gnn["label"] else ""
    print(f"  {r['label']:<12s}: QVal={r['q_val']:.5f} QTest={r['q_test']:.4f} Loss={r['loss']:.4f} Params={r['params']:,}{tag}")

print(f"\n📌 2-Stage 결과:")
print(f"  S1({len(xgb_feature_cols)}개): Val={s1_val:.5f} Test={s1_test:.4f}")
print(f"  S2({best_s2['label']},{len(best_features)}개): Val={best_s2['val']:.5f} Test={auprc:.4f}")

print(f"\n✨ 신규 피처 그룹 추가 요약 (trend_* 제거 후):")
print(f"  ① CB 정밀화  : {len(cb_feature_cols)}개 → {cb_feature_cols}")
print(f"  ② 거래량 정규화: {len(vol_norm_feature_cols)}개 → {vol_norm_feature_cols}")
print(f"  ③ 윈도우 변화: {len(win_shift_feature_cols)}개 → {win_shift_feature_cols}")
print(f"  ④ 입출금 독립: {len(inout_feature_cols)}개 → {inout_feature_cols}")

del X_test; gc.collect(); log_memory("최종 완료")
print(f"\n✅ TGAT v7c.4 완료!")

🛡️ [TGAT v7c.4] FP/FN 보완 피처 추가 (trend_* 제거)
   ▸ v7c.2 대비 추가 피처 4그룹:
     ① Cross-border 정밀화 (cb_*, 4개)
     ② 거래량 대비 정규화 개선 (vol_*, 5개)
     ③ 시간 윈도우별 행동 변화 (ws_*, 4개)
     ④ 입금/출금 독립 피처 (inout_*, 6개)
   ▸ GNN 탐색: hidden 64/128 × layer 2/3 = 4개
   ▸ XGBoost: 확장 HP 그리드 (7×5+1=36조합)

📂 데이터 로딩 중...
⏱️ Train cutoff: 2022-09-09 20:00:00
⏱️ Val   cutoff: 2022-09-14 05:00:00
📊 전체 노드 수: 2,076,999
  💾 [데이터 로드 완료] RAM: 33.67GB | GPU: 0.00GB

📐 [Step 2] 전체 노드 피처 계산...
  📊 그래프 구조 피처...
  📊 PageRank...
  ✅ 그래프 피처 13개
  📊 조건부행동 피처...
  ✅ 조건부행동 10개
  📊 AML 점수...
  ✅ AML 점수 5개
  📊 규칙성 피처...
  ✅ 규칙성 7개
  📊 금액 분포 피처...
  ✅ 금액 분포 8개
  📊 ★ Cross-border 정밀화 피처 (신규)...
  ✅ Cross-border 정밀화 4개: ['cb_new_partner_ratio', 'cb_oneway_new', 'cb_amount_asymmetry', 'cb_burst_cross']
  📊 ★ 거래량 대비 상대 정규화 피처 (신규)...
  ✅ 거래량 대비 정규화 5개: ['vol_amount_per_peer', 'vol_concentrated_amount', 'vol_low_freq_high_amt', 'vol_degree_normalized_amount', 'vol_outlier_score']
  📊 ★ 시간 윈도우별 행동 변화 피처 (신규)...
  ✅ 시간 윈도우 변화 3개: ['ws_re

    [h64_L2] Ep 1 Loss=1.3596


    [h64_L2] Ep10 Loss=1.0558


    [h64_L2] Ep20 Loss=1.0255


    [h64_L2] Ep30 Loss=1.0032


...

    [h128_L3] Ep10 Loss=1.0391


    [h128_L3] Ep20 Loss=1.0037


    [h128_L3] Ep30 Loss=0.9812


    [h128_L3] Ep40 Loss=0.9731


   4 h128_L3      |    128      3   32 |   0.9705  233,985 |  0.38210   0.4093
  💾 [h128_L3] RAM: 92.04GB | GPU: 0.02GB

  🏆 Best GNN: h64_L2 (QVal=0.38589 QTest=0.4112)

🔗 Interaction 피처 (h64_L2)...
  ✅ Interaction 10개
  💾 [GNN 탐색 완료] RAM: 88.72GB | GPU: 0.02GB

🚀 [Step 7] XGBoost 데이터 구성...
📊 XGBoost 총 피처: 158
   V2:38 G:13 Cond:10 AML:5
   CB:4 VolNorm:5
   WinShift:3 InOut:6
   Emb:64 Inter:10
📊 Train: 17,172,747 → 619,437 (4.76%)
📊 Val: 5,790,644
📊 Test: 5,792,913
  💾 [XGBoost 준비] RAM: 104.18GB | GPU: 0.02GB

🔥 [Step 8] 2-Stage Pruning + 확장 HP 그리드...

── Stage 1 ──
[0]	validation_0-aucpr:0.11835
[100]	validation_0-aucpr:0.28368
[200]	validation_0-aucpr:0.37949
[300]	validation_0-aucpr:0.39957
[400]	validation_0-aucpr:0.41179
[500]	validation_0-aucpr:0.41683
[600]	validation_0-aucpr:0.42134
[700]	validation_0-aucpr:0.42666
[800]	validation_0-aucpr:0.43039
[900]	validation_0-aucpr:0.43290
[1000]	validation_0-aucpr:0.43625
[1100]	validation_0-aucpr:0.44014
[1200]	validation_0-aucpr:0.

In [2]:
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv
from torch_geometric.loader import NeighborLoader
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm
import gc, numpy as np, psutil, os, warnings
warnings.filterwarnings("ignore", category=UserWarning)

def log_memory(tag=""):
    rss = psutil.Process(os.getpid()).memory_info().rss/(1024**3)
    gpu = torch.cuda.memory_allocated()/(1024**3) if torch.cuda.is_available() else 0
    print(f"  💾 [{tag}] RAM: {rss:.2f}GB | GPU: {gpu:.2f}GB")

print("=" * 60)
print("🛡️ [TGAT v7c.5] GNN v7c.2 복원 + XGBoost 선별 신규 피처")
print("   ▸ GNN 입력: v7c.2 동일 81차원 (신규 피처 노이즈 제거)")
print("   ▸ XGBoost 추가 피처 (기여도 상위만):")
print("     ① ws_partner_explosion, ws_recent_only (ws_* 상위 2개)")
print("     ② inout_fan_out_score, inout_net_flow  (inout_* 상위 2개)")
print("   ▸ GNN 탐색: hidden 64/128 × layer 2/3 = 4개")
print("   ▸ XGBoost: 확장 HP 그리드 — top범위 45~100, 42조합")
print("=" * 60)

# ===================== 설정 =====================
NEG_NODE_RATIO_GNN = 50; NEG_ROW_RATIO_XGB = 20; RANDOM_SEED = 42
HIDDEN_DIM = 64; EDGE_DIM = 16; N_HEADS = 4
N_EPOCHS_GNN = 40; BATCH_SIZE = 2048; NUM_NEIGHBORS = [15, 10]
N_INTERACTION_DIMS = 8; PATIENCE = 10; BASE_LR = 0.001; WARMUP_EPOCHS = 5

raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"

# =============================================================
# 1. 데이터 로드
# =============================================================
print("\n📂 데이터 로딩 중...")
df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))
if df_raw["Timestamp"].dtype == pl.Utf8:
    df_raw = df_raw.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_v2 = df_v2.sort("time_group")
total_count = len(df_v2)
train_cutoff = df_v2["time_group"][int(total_count*0.6)]
val_cutoff = df_v2["time_group"][int(total_count*0.8)]
print(f"⏱️ Train cutoff: {train_cutoff}\n⏱️ Val   cutoff: {val_cutoff}")

all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")
num_nodes = len(all_accounts); print(f"📊 전체 노드 수: {num_nodes:,}")
log_memory("데이터 로드 완료")

# =============================================================
# 2. 모든 노드 피처 계산 (GNN 입력용 + XGBoost 겸용)
# =============================================================
print("\n📐 [Step 2] 전체 노드 피처 계산...")
df_raw_train = df_raw.filter(pl.col("Timestamp") < train_cutoff)

# ─── 2-1. 그래프 구조 피처 13개 ───
print("  📊 그래프 구조 피처...")
df_from = df_raw_train.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id"))
df_to = df_raw_train.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
out_degree = df_from.group_by("account_id").len().rename({"len":"graph_out_degree"})
in_degree = df_to.group_by("account_id").len().rename({"len":"graph_in_degree"})
unique_out = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("to_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_out_partners"))
unique_in = df_raw_train.select([pl.col("to_acc").cast(pl.Utf8).alias("account_id"),pl.col("from_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_in_partners"))
esf = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("a"),pl.col("to_acc").cast(pl.Utf8).alias("b")]).unique()
est = esf.select([pl.col("b").alias("a"),pl.col("a").alias("b")])
bidir = esf.join(est, on=["a","b"], how="inner")
bidir_count = bidir.select(pl.col("a").alias("account_id")).group_by("account_id").len().rename({"len":"graph_bidir_count"})
del esf, est, bidir; gc.collect()
time_features = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("Timestamp")]).group_by("account_id").agg([
    pl.col("Timestamp").sort().diff().dt.total_seconds().mean().alias("graph_avg_tx_interval"),
    pl.col("Timestamp").sort().diff().dt.total_seconds().std().alias("graph_std_tx_interval"),
    pl.col("Timestamp").count().alias("graph_total_tx_count")])

# PageRank
print("  📊 PageRank...")
account_to_id = dict(zip(all_accounts["account_id"].to_list(), all_accounts["node_id"].to_list()))
src_ids = df_raw_train["from_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
dst_ids = df_raw_train["to_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
vm=(src_ids>=0)&(dst_ids>=0); sv=src_ids[vm]; dv=dst_ids[vm]
oda=np.zeros(num_nodes,dtype=np.float64); np.add.at(oda,sv,1.0); oda=np.maximum(oda,1.0)
pr=np.ones(num_nodes,dtype=np.float64)/num_nodes
for _ in range(10): npr=np.ones(num_nodes,dtype=np.float64)*0.15/num_nodes; np.add.at(npr,dv,0.85*pr[sv]/oda[sv]); pr=npr
pr_df = pl.DataFrame({"node_id":np.arange(num_nodes,dtype=np.uint32),"graph_pagerank":pr.astype(np.float32)})
pr_df = all_accounts.join(pr_df,on="node_id",how="left").select(["account_id","graph_pagerank"])
del src_ids,dst_ids,sv,dv,oda,pr,npr; gc.collect()

df_graph_feats = (all_accounts.select("account_id").join(out_degree,on="account_id",how="left").join(in_degree,on="account_id",how="left")
    .join(unique_out,on="account_id",how="left").join(unique_in,on="account_id",how="left")
    .join(bidir_count,on="account_id",how="left").join(time_features,on="account_id",how="left")
    .join(pr_df,on="account_id",how="left").fill_null(0.0)
    .with_columns([(pl.col("graph_out_degree")+pl.col("graph_in_degree")).alias("graph_total_degree"),
        (pl.col("graph_out_degree")/(pl.col("graph_in_degree")+1)).alias("graph_out_in_ratio"),
        (pl.col("graph_bidir_count")/(pl.col("graph_out_degree")+1)).alias("graph_reciprocity"),
        (pl.col("graph_unique_out_partners")/(pl.col("graph_out_degree")+1)).alias("graph_partner_diversity")]))
graph_feature_cols = [c for c in df_graph_feats.columns if c.startswith("graph_")]
del out_degree,in_degree,unique_out,unique_in,bidir_count,time_features,pr_df,df_from,df_to; gc.collect()
print(f"  ✅ 그래프 피처 {len(graph_feature_cols)}개")

# ─── 2-2. 조건부행동 피처 10개 ───
print("  📊 조건부행동 피처...")
df_v2_train = df_v2.filter(pl.col("time_group")<train_cutoff)
exclude_cols = ["account_id","time_group","is_laundering","mode_format","currency_mode"]
gnn_feature_cols_v2 = [c for c in df_v2.columns if c not in exclude_cols]
df_v2_agg = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])

df_cond = df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
v2a=set(df_cond.columns)
def sc(n): return pl.col(n) if n in v2a else pl.lit(0.0)
ce=[]
if "cnt_night" in v2a and "cnt_1h" in v2a: ce.append((pl.col("graph_total_degree")*sc("cnt_night")/(sc("cnt_1h")+1)).alias("cond_highdeg_night_intensity"))
if "avg_log_amount" in v2a: ce.append((sc("avg_log_amount")/(pl.col("graph_total_degree").log1p()+1)).alias("cond_lowdeg_high_amount"))
if "ratio_cross_border" in v2a: ce.append((sc("ratio_cross_border")*(1-pl.col("graph_reciprocity"))).alias("cond_oneway_crossborder"))
if "cnt_1h" in v2a: ce.append((pl.col("graph_unique_out_partners")*sc("cnt_1h")/(pl.col("graph_out_degree")+1)).alias("cond_fanout_burst"))
if "cnt_risk_format" in v2a: ce.append((pl.col("graph_pagerank")*sc("cnt_risk_format")).alias("cond_hub_risk_format"))
if "cnt_wire" in v2a: ce.append((pl.col("graph_out_in_ratio")*sc("cnt_wire")).alias("cond_asymmetric_wire"))
if "cnt_inter_bank" in v2a: ce.append((pl.col("graph_std_tx_interval").fill_null(0)/(pl.col("graph_avg_tx_interval").fill_null(1)+1)*sc("cnt_inter_bank")).alias("cond_irregular_interbank"))
if "cnt_currencies" in v2a: ce.append((pl.col("graph_partner_diversity")*sc("cnt_currencies")).alias("cond_diverse_partner_currency"))
if "amount_kurtosis" in v2a: ce.append((pl.col("graph_reciprocity")*sc("amount_kurtosis")).alias("cond_bidir_amount_volatility"))
if "entity_acct_cnt" in v2a and "burst_ratio" in v2a: ce.append((sc("entity_acct_cnt")*sc("burst_ratio")).alias("cond_multi_acct_burst"))
df_cond=df_cond.with_columns(ce); cond_feature_cols=[e.meta.output_name() for e in ce]
for cn in cond_feature_cols: df_cond=df_cond.with_columns(pl.when(pl.col(cn).is_infinite()|pl.col(cn).is_nan()).then(0.0).otherwise(pl.col(cn)).alias(cn))
df_cond_features=df_cond.select(["account_id"]+cond_feature_cols); del df_cond; gc.collect()
print(f"  ✅ 조건부행동 {len(cond_feature_cols)}개")

# ─── 2-3. AML 점수 5개 ───
print("  📊 AML 점수...")
df_sb=df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
se=[((sc("cnt_1h").fill_null(0).clip(0,1e6)+1).log1p()*pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("burst_ratio").fill_null(0).clip(0,1e6)/(sc("avg_log_amount").fill_null(1).clip(0,1e6)+1)).alias("score_smurf"),
    ((1.0/((pl.col("graph_out_in_ratio").clip(0.01,100)-1.0).abs()+0.1))*sc("cnt_wire").fill_null(0).clip(0,1e6).log1p()*(1-pl.col("graph_reciprocity").clip(0,1))/(pl.col("graph_unique_out_partners").clip(0,1e6).log1p()+1)).alias("score_mule"),
    (pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("cnt_currencies").fill_null(0).clip(0,1e6).log1p()*sc("ratio_cross_border").fill_null(0).clip(0,1)/(pl.col("graph_avg_tx_interval").fill_null(3600).clip(0,1e9)/3600+1)).alias("score_layering"),
    (pl.col("graph_in_degree").clip(0,1e6).log1p()*(pl.col("graph_pagerank").clip(0,1)*1e6)*sc("cnt_risk_format").fill_null(0).clip(0,1e6).log1p()/(pl.col("graph_out_degree").clip(0,1e6).log1p()+1)).alias("score_integration")]
df_sb=df_sb.with_columns(se)
sn=["score_smurf","score_mule","score_layering","score_integration"]
for s in sn: df_sb=df_sb.with_columns(pl.when(pl.col(s).is_infinite()|pl.col(s).is_nan()).then(0.0).otherwise(pl.col(s)).alias(s))
for s in sn:
    mn=df_sb[s].min();mx=df_sb[s].max();rng=mx-mn if(mx-mn)>0 else 1.0
    df_sb=df_sb.with_columns(((pl.col(s)-mn)/rng).alias(f"{s}_norm"))
df_sb=df_sb.with_columns((pl.col("score_smurf_norm")*0.3+pl.col("score_mule_norm")*0.25+pl.col("score_layering_norm")*0.3+pl.col("score_integration_norm")*0.15).alias("score_composite"))
aml_score_cols=sn+["score_composite"]
df_aml_scores=df_sb.select(["account_id"]+aml_score_cols)
del df_sb, df_v2_agg; gc.collect()
print(f"  ✅ AML 점수 {len(aml_score_cols)}개")

# ─── 2-4. 규칙성 피처 7개 ───
print("  📊 규칙성 피처...")
df_tx_time = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner"),
    pl.col("Timestamp"),
    pl.col("Timestamp").dt.hour().alias("hour"),
    pl.col("Timestamp").dt.weekday().alias("dow"),
])
reg_agg = df_tx_time.group_by("account_id").agg([
    (pl.col("hour").is_between(9,17).mean()).alias("reg_business_hour_ratio"),
    (pl.col("dow").is_in([5,6]).mean()).alias("reg_weekend_ratio"),
    (pl.col("hour").is_between(0,5).mean()).alias("reg_night_strict_ratio"),
    pl.col("partner").count().alias("_total_tx"),
])
hc=df_tx_time.group_by(["account_id","hour"]).len().rename({"len":"h_count"})
hp=hc.pivot(on="hour",index="account_id",values="h_count").fill_null(0)
hcols=[c for c in hp.columns if c!="account_id"]
hmat=hp.select(hcols).to_numpy().astype(np.float64); hsum=np.maximum(hmat.sum(axis=1,keepdims=True),1.0)
hprobs=hmat/hsum; hlog=np.where(hprobs>0,np.log(hprobs+1e-12),0.0)
df_h_ent=pl.DataFrame({"account_id":hp["account_id"],"reg_hour_entropy":-np.sum(hprobs*hlog,axis=1).astype(np.float32)})
del hc,hp,hmat,hprobs,hlog; gc.collect()
dc=df_tx_time.group_by(["account_id","dow"]).len().rename({"len":"d_count"})
dp=dc.pivot(on="dow",index="account_id",values="d_count").fill_null(0)
dcols=[c for c in dp.columns if c!="account_id"]
dmat=dp.select(dcols).to_numpy().astype(np.float64); dsum=np.maximum(dmat.sum(axis=1,keepdims=True),1.0)
dprobs=dmat/dsum; dlog=np.where(dprobs>0,np.log(dprobs+1e-12),0.0)
df_d_ent=pl.DataFrame({"account_id":dp["account_id"],"reg_dow_entropy":-np.sum(dprobs*dlog,axis=1).astype(np.float32)})
del dc,dp,dmat,dprobs,dlog; gc.collect()
pf=df_tx_time.group_by(["account_id","partner"]).len().rename({"len":"p_count"})
rp=pf.filter(pl.col("p_count")>=2).group_by("account_id").len().rename({"len":"repeat_cnt"})
tp2=pf.group_by("account_id").len().rename({"len":"total_cnt"})
df_pc=tp2.join(rp,on="account_id",how="left").fill_null(0).with_columns((pl.col("repeat_cnt")/(pl.col("total_cnt")+1)).alias("reg_consistent_partner_ratio")).select(["account_id","reg_consistent_partner_ratio"])
top_p=pf.sort(["account_id","p_count"],descending=[False,True]).unique(subset=["account_id"],maintain_order=True).rename({"p_count":"top_cnt"}).select(["account_id","top_cnt"])
df_tc=reg_agg.select(["account_id","_total_tx"]).join(top_p,on="account_id",how="left").fill_null(0).with_columns((pl.col("top_cnt")/(pl.col("_total_tx")+1)).alias("reg_top_partner_concentration")).select(["account_id","reg_top_partner_concentration"])
del pf,rp,tp2,top_p; gc.collect()
df_regularity=(reg_agg.select(["account_id","reg_business_hour_ratio","reg_weekend_ratio","reg_night_strict_ratio"])
    .join(df_h_ent,on="account_id",how="left").join(df_d_ent,on="account_id",how="left")
    .join(df_pc,on="account_id",how="left").join(df_tc,on="account_id",how="left").fill_null(0.0))
regularity_cols=[c for c in df_regularity.columns if c.startswith("reg_")]
for rc in regularity_cols: df_regularity=df_regularity.with_columns(pl.when(pl.col(rc).is_infinite()|pl.col(rc).is_nan()).then(0.0).otherwise(pl.col(rc)).alias(rc))
del reg_agg,df_h_ent,df_d_ent,df_pc,df_tc; gc.collect()
print(f"  ✅ 규칙성 {len(regularity_cols)}개")

# ─── 2-5. 금액 분포 피처 8개 ───
print("  📊 금액 분포 피처...")
amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train.columns: amount_col = cand; break
if amount_col:
    df_amounts=df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col(amount_col).cast(pl.Float64).alias("amount")]).filter(pl.col("amount")>0)
    df_amt=df_amounts.group_by("account_id").agg([
        ((pl.col("amount")%1000==0)|(pl.col("amount")%5000==0)).mean().alias("amt_round_number_ratio"),
        (pl.col("amount").std()/(pl.col("amount").mean()+1)).alias("amt_cv"),
        (pl.col("amount").max()/(pl.col("amount").mean()+1)).alias("amt_max_mean_ratio"),
        (pl.col("amount").median()/(pl.col("amount").mean()+1)).alias("amt_median_mean_ratio"),
        pl.col("amount").quantile(0.9).alias("_q90"), pl.col("amount").sum().alias("_tsum"),
        pl.col("amount").count().alias("_cnt"),
        pl.col("amount").quantile(0.75).alias("_q75"), pl.col("amount").quantile(0.25).alias("_q25"),
        pl.col("amount").median().alias("_med"),
    ]).fill_null(0.0).with_columns([
        (pl.col("_q90")*0.1*pl.col("_cnt")/(pl.col("_tsum")+1)).alias("amt_top10pct_concentration"),
        ((pl.col("_q75")-pl.col("_q25"))/(pl.col("_med")+1)).alias("amt_iqr_median_ratio")])
    gm=df_amounts["amount"].mean(); st=max(gm*0.1,1.0)
    df_sr=df_amounts.with_columns((pl.col("amount")<st).alias("is_small")).group_by("account_id").agg(pl.col("is_small").mean().alias("amt_small_tx_ratio"))
    df_amt=df_amt.join(df_sr,on="account_id",how="left").fill_null(0.0)
    df_al=df_amounts.with_columns((pl.col("amount")+1).log().cast(pl.Int32).alias("log_bucket"))
    bc=df_al.group_by(["account_id","log_bucket"]).len().rename({"len":"b_count"})
    bt=bc.group_by("account_id").agg(pl.col("b_count").sum().alias("b_total"))
    bp=bc.join(bt,on="account_id").with_columns((pl.col("b_count")/pl.col("b_total")).alias("b_prob"))
    ae=bp.group_by("account_id").agg((-1*(pl.col("b_prob")*(pl.col("b_prob")+1e-12).log()).sum()).alias("amt_entropy"))
    df_amt=df_amt.join(ae,on="account_id",how="left").fill_null(0.0)
    del df_amounts,df_al,bc,bt,bp,ae,df_sr; gc.collect()
    amt_feature_cols=[c for c in df_amt.columns if c.startswith("amt_") and not c.startswith("_")]
    df_amt_features=df_amt.select(["account_id"]+amt_feature_cols)
else:
    df_amt_features=all_accounts.select("account_id"); amt_feature_cols=[]
for ac in amt_feature_cols:
    df_amt_features=df_amt_features.with_columns(pl.when(pl.col(ac).is_infinite()|pl.col(ac).is_nan()).then(0.0).otherwise(pl.col(ac)).alias(ac))
print(f"  ✅ 금액 분포 {len(amt_feature_cols)}개")

# =============================================================
# ★ NEW: 2-6. Cross-border 정밀화 피처 4개
# =============================================================
# 핵심 아이디어: ratio_cross_border 단독이 아니라
#   "cross-border × 새로운 상대방 비율 × 방향 비대칭"을 결합
print("  📊 ★ Cross-border 정밀화 피처 (신규)...")
cb_feature_cols = []

# 데이터에 to_acc country/bank 정보가 있을 경우를 가정
# → 없으면 graph_partner_diversity 와 ratio_cross_border 조합으로 근사
df_v2_agg_cb = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2 if c in df_v2_train.columns
])
df_cb = df_v2_agg_cb.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)
v2cb = set(df_cb.columns)

cb_exprs = []
# ① cb_new_partner_ratio: cross-border이면서 새로운 상대방 비율
#    = ratio_cross_border × (unique_out_partners / (total_tx_count+1))
#    → 새로운 상대에게 해외 송금하는 패턴
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("graph_unique_out_partners") /
         (pl.col("graph_total_tx_count") + 1)
        ).alias("cb_new_partner_ratio")
    )
    cb_feature_cols.append("cb_new_partner_ratio")

# ② cb_oneway_new: cross-border × 비양방향 × 새 상대방
#    → 양방향 거래 없이 새로운 상대에게만 해외 송금 (전형적 분산 송금)
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         (1.0 - pl.col("graph_reciprocity")) *
         (pl.col("graph_unique_out_partners") / (pl.col("graph_out_degree") + 1))
        ).alias("cb_oneway_new")
    )
    cb_feature_cols.append("cb_oneway_new")

# ③ cb_amount_asymmetry: 해외 송금 비율 × 금액 비대칭 (out >> in)
#    = ratio_cross_border × out_in_ratio (clip 해서 폭발 방지)
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("graph_out_in_ratio").clip(0, 20)
        ).alias("cb_amount_asymmetry")
    )
    cb_feature_cols.append("cb_amount_asymmetry")

# ④ cb_burst_cross: cross-border이 짧은 시간 내 집중 (burst_ratio 결합)
if "ratio_cross_border" in v2cb and "burst_ratio" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("burst_ratio").clip(0, 100)
        ).alias("cb_burst_cross")
    )
    cb_feature_cols.append("cb_burst_cross")

if cb_exprs:
    df_cb = df_cb.with_columns(cb_exprs)
    for cn in cb_feature_cols:
        df_cb = df_cb.with_columns(
            pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
            .then(0.0).otherwise(pl.col(cn)).alias(cn))
    df_cb_features = df_cb.select(["account_id"] + cb_feature_cols)
else:
    df_cb_features = all_accounts.select("account_id"); cb_feature_cols = []
del df_cb, df_v2_agg_cb; gc.collect()
print(f"  ✅ Cross-border 정밀화 {len(cb_feature_cols)}개: {cb_feature_cols}")

# trend_* 피처 제거 (ws_* 와 중복) — v7c.4
trend_feature_cols = []

# =============================================================
# ★ NEW: 2-8. 거래량 대비 상대 정규화 피처 5개
# =============================================================
# 핵심 아이디어: cond_lowdeg_high_amount가 FN에서 TP보다 오히려 높아 역전됨
#   → "전체 네트워크 대비 비정상적으로 높은 금액"을 더 정밀하게 포착
print("  📊 ★ 거래량 대비 상대 정규화 피처 (신규)...")
vol_norm_feature_cols = []

df_v2_agg_vn = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2 if c in df_v2_train.columns
])
df_vn = df_v2_agg_vn.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)

# 네트워크 전체 평균값 계산 (정규화 기준)
if "avg_log_amount" in set(df_vn.columns):
    global_avg_log_amount = df_vn["avg_log_amount"].mean() or 1.0
    global_avg_degree = df_vn["graph_total_degree"].mean() or 1.0
    global_avg_cnt_1h = df_vn["cnt_1h"].mean() if "cnt_1h" in df_vn.columns else 1.0
    global_avg_cnt_1h = global_avg_cnt_1h or 1.0

    vn_exprs = []
    # ① vol_amount_per_peer: 건당 금액 / (파트너 수 × 전체 평균)
    #    → degree가 낮은데 금액이 높은 패턴을 전체 대비 상대적으로 포착
    vn_exprs.append(
        (pl.col("avg_log_amount") / (float(global_avg_log_amount) + 1e-8) /
         (pl.col("graph_unique_out_partners").log1p() + 1)
        ).alias("vol_amount_per_peer")
    )
    vol_norm_feature_cols.append("vol_amount_per_peer")

    # ② vol_concentrated_amount: 소수 파트너에게 집중 송금
    #    = avg_log_amount × (1 - partner_diversity)
    #    → diversity가 낮을수록 (=한 곳에 집중) × 금액이 높을수록 높아짐
    vn_exprs.append(
        (pl.col("avg_log_amount") * (1.0 - pl.col("graph_partner_diversity").clip(0, 1))
        ).alias("vol_concentrated_amount")
    )
    vol_norm_feature_cols.append("vol_concentrated_amount")

    # ③ vol_low_freq_high_amt: 낮은 빈도이지만 건당 금액이 전체 평균 대비 높음
    #    = (avg_log_amount / global_mean) / log1p(cnt_1h) → 드문데 고액
    if "cnt_1h" in set(df_vn.columns):
        vn_exprs.append(
            (pl.col("avg_log_amount") / float(global_avg_log_amount) /
             (pl.col("cnt_1h").clip(0, 1e6).log1p() + 1)
            ).alias("vol_low_freq_high_amt")
        )
        vol_norm_feature_cols.append("vol_low_freq_high_amt")

    # ④ vol_degree_normalized_amount: 디그리로 나눈 상대 금액 (올바른 역산 버전)
    #    기존 cond_lowdeg_high_amount 대체 — log 스케일 적용 + 전체 평균으로 나눔
    vn_exprs.append(
        ((pl.col("avg_log_amount") - float(global_avg_log_amount)) /
         (pl.col("graph_total_degree").clip(1, 1e6).log1p())
        ).alias("vol_degree_normalized_amount")
    )
    vol_norm_feature_cols.append("vol_degree_normalized_amount")

    # ⑤ vol_outlier_score: z-score 방식의 금액 이상 점수
    #    = (avg_log_amount - global_mean) → 양수면 전체 평균 초과
    vn_exprs.append(
        (pl.col("avg_log_amount") - float(global_avg_log_amount)).alias("vol_outlier_score")
    )
    vol_norm_feature_cols.append("vol_outlier_score")

    df_vn = df_vn.with_columns(vn_exprs)
    for cn in vol_norm_feature_cols:
        df_vn = df_vn.with_columns(
            pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
            .then(0.0).otherwise(pl.col(cn)).alias(cn))
    df_vol_norm_features = df_vn.select(["account_id"] + vol_norm_feature_cols)
else:
    df_vol_norm_features = all_accounts.select("account_id"); vol_norm_feature_cols = []

del df_vn, df_v2_agg_vn; gc.collect()
print(f"  ✅ 거래량 대비 정규화 {len(vol_norm_feature_cols)}개: {vol_norm_feature_cols}")

# =============================================================
# ★ NEW: 2-9. 시간 윈도우별 행동 변화 피처 4개
# =============================================================
# 핵심 아이디어: 미탐의 71%가 테스트 첫 3일 집중
#   → train 말미(~recent window)에서의 행동 변화를 포착
print("  📊 ★ 시간 윈도우별 행동 변화 피처 (신규)...")
win_shift_feature_cols = []

# train 마지막 7일 vs 나머지 기간 비교
last_7d_cutoff = train_cutoff - pl.duration(days=7)
df_raw_recent = df_raw_train.filter(pl.col("Timestamp") >= last_7d_cutoff)
df_raw_older  = df_raw_train.filter(pl.col("Timestamp") < last_7d_cutoff)

recent_cnt = df_raw_recent.select(
    pl.col("from_acc").cast(pl.Utf8).alias("account_id")
).group_by("account_id").len().rename({"len": "ws_recent_cnt"})
older_cnt = df_raw_older.select(
    pl.col("from_acc").cast(pl.Utf8).alias("account_id")
).group_by("account_id").len().rename({"len": "ws_older_cnt"})

df_ws = all_accounts.select("account_id").join(recent_cnt, on="account_id", how="left").join(older_cnt, on="account_id", how="left").fill_null(0.0)

ws_exprs = []
# ① ws_recent_surge: 최근 7일 / 이전 기간 (정규화) → 갑작스러운 증가
ws_exprs.append(
    (pl.col("ws_recent_cnt") / (pl.col("ws_older_cnt") / 30.0 + 1)).alias("ws_recent_surge")
)
win_shift_feature_cols.append("ws_recent_surge")

# ② ws_recent_only: 최근 7일에만 거래가 있는 계좌 (0/1)
#    → 갑자기 나타난 계좌
ws_exprs.append(
    ((pl.col("ws_recent_cnt") > 0) & (pl.col("ws_older_cnt") == 0)).cast(pl.Float64).alias("ws_recent_only")
)
win_shift_feature_cols.append("ws_recent_only")

df_ws = df_ws.with_columns(ws_exprs)

# ③ 최근 7일 고유 파트너 급증
recent_partners = df_raw_recent.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner")
]).group_by("account_id").agg(pl.col("partner").n_unique().alias("ws_recent_partners"))
older_partners = df_raw_older.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner")
]).group_by("account_id").agg(pl.col("partner").n_unique().alias("ws_older_partners"))

df_ws = df_ws.join(recent_partners, on="account_id", how="left").join(older_partners, on="account_id", how="left").fill_null(0.0)

# ④ ws_partner_explosion: 최근 7일 파트너가 이전 대비 급증
df_ws = df_ws.with_columns(
    (pl.col("ws_recent_partners") / (pl.col("ws_older_partners") + 1)).alias("ws_partner_explosion")
)
win_shift_feature_cols.append("ws_partner_explosion")

for cn in win_shift_feature_cols:
    df_ws = df_ws.with_columns(
        pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
        .then(0.0).otherwise(pl.col(cn)).alias(cn))

df_win_shift_features = df_ws.select(["account_id"] + win_shift_feature_cols)
del df_raw_recent, df_raw_older, df_ws, recent_cnt, older_cnt
del recent_partners, older_partners; gc.collect()
print(f"  ✅ 시간 윈도우 변화 {len(win_shift_feature_cols)}개: {win_shift_feature_cols}")

# =============================================================
# ★ NEW: 2-10. 입금/출금 독립 피처 6개
# =============================================================
# 핵심 아이디어: V2에 sum_in_1h/sum_out_1h는 있지만
#   AML 점수/조건부 피처에서 in/out 독립적 분석이 부족
print("  📊 ★ 입금/출금 독립 피처 (신규)...")
inout_feature_cols = []

# 출금(from) 집계
df_out_tx = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col(amount_col).cast(pl.Float64).alias("amount") if amount_col else pl.lit(1.0).alias("amount")
]).filter(pl.col("amount") > 0)

# 입금(to) 집계
df_in_tx = df_raw_train.select([
    pl.col("to_acc").cast(pl.Utf8).alias("account_id"),
    pl.col(amount_col).cast(pl.Float64).alias("amount") if amount_col else pl.lit(1.0).alias("amount")
]).filter(pl.col("amount") > 0)

out_stats = df_out_tx.group_by("account_id").agg([
    pl.col("amount").sum().alias("inout_total_out_amount"),
    pl.col("amount").mean().alias("inout_avg_out_amount"),
    pl.col("amount").count().alias("inout_out_cnt"),
])
in_stats = df_in_tx.group_by("account_id").agg([
    pl.col("amount").sum().alias("inout_total_in_amount"),
    pl.col("amount").mean().alias("inout_avg_in_amount"),
    pl.col("amount").count().alias("inout_in_cnt"),
])
df_inout = all_accounts.select("account_id").join(out_stats, on="account_id", how="left").join(in_stats, on="account_id", how="left").fill_null(0.0)

inout_exprs = []
# ① inout_out_amount_ratio: 출금액 / (입금액 + 출금액) → 1이면 순수 송금자
inout_exprs.append(
    (pl.col("inout_total_out_amount") /
     (pl.col("inout_total_out_amount") + pl.col("inout_total_in_amount") + 1)
    ).alias("inout_out_amount_ratio")
)
inout_feature_cols.append("inout_out_amount_ratio")

# ② inout_avg_out_vs_in: 평균 출금액 / 평균 입금액 → 1 초과면 출금이 더 큰 계좌
inout_exprs.append(
    (pl.col("inout_avg_out_amount") / (pl.col("inout_avg_in_amount") + 1)
    ).alias("inout_avg_out_vs_in")
)
inout_feature_cols.append("inout_avg_out_vs_in")

# ③ inout_cnt_imbalance: (출금 건수 - 입금 건수) / 전체 건수
#    → 양수면 보내는 쪽이 훨씬 많은 계좌 (smurf 패턴)
inout_exprs.append(
    ((pl.col("inout_out_cnt") - pl.col("inout_in_cnt")) /
     (pl.col("inout_out_cnt") + pl.col("inout_in_cnt") + 1)
    ).alias("inout_cnt_imbalance")
)
inout_feature_cols.append("inout_cnt_imbalance")

# ④ inout_fan_out_score: 출금 건수 × 고유 파트너 다양성 → 여러 곳에 분산 송금
df_inout = df_inout.join(df_graph_feats.select(["account_id", "graph_partner_diversity", "graph_unique_out_partners"]), on="account_id", how="left").fill_null(0.0)
inout_exprs.append(
    (pl.col("inout_out_cnt").log1p() * pl.col("graph_partner_diversity")
    ).alias("inout_fan_out_score")
)
inout_feature_cols.append("inout_fan_out_score")

# ⑤ inout_in_concentration: 입금이 소수에서만 들어옴 (mule 패턴)
#    = avg_in_amount × (1 - partner_diversity) → 소수 소스에서 집중 수취
inout_exprs.append(
    (pl.col("inout_avg_in_amount").log1p() * (1.0 - pl.col("graph_partner_diversity").clip(0, 1))
    ).alias("inout_in_concentration")
)
inout_feature_cols.append("inout_in_concentration")

# ⑥ inout_net_flow: log(출금합) - log(입금합) → 순 흐름 방향성
inout_exprs.append(
    (pl.col("inout_total_out_amount").log1p() - pl.col("inout_total_in_amount").log1p()
    ).alias("inout_net_flow")
)
inout_feature_cols.append("inout_net_flow")

df_inout = df_inout.with_columns(inout_exprs)
for cn in inout_feature_cols:
    df_inout = df_inout.with_columns(
        pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
        .then(0.0).otherwise(pl.col(cn)).alias(cn))
df_inout_features = df_inout.select(["account_id"] + inout_feature_cols)
del df_out_tx, df_in_tx, out_stats, in_stats, df_inout; gc.collect()
print(f"  ✅ 입금/출금 독립 {len(inout_feature_cols)}개: {inout_feature_cols}")
log_memory("전체 노드 피처 완료")

# =============================================================
# 3. GNN 엣지/노드 구성 — 강화된 입력
# =============================================================
print("\n🔗 [Step 3] GNN 엣지/노드 구성 (Enriched Input)...")

df_raw_reload = pl.read_parquet(raw_path)
if df_raw_reload["Timestamp"].dtype == pl.Utf8:
    df_raw_reload = df_raw_reload.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_raw_train_edges = df_raw_reload.filter(pl.col("Timestamp")<train_cutoff); del df_raw_reload; gc.collect()

edge_amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train_edges.columns: edge_amount_col = cand; break

df_edges = df_raw_train_edges.select(
    ["from_acc","to_acc","Timestamp"] +
    ([edge_amount_col] if edge_amount_col else []) +
    (["Payment Format"] if "Payment Format" in df_raw_train_edges.columns else []) +
    (["Is Laundering"] if "Is Laundering" in df_raw_train_edges.columns else [])
).with_columns([pl.col("from_acc").cast(pl.Utf8), pl.col("to_acc").cast(pl.Utf8)])

df_edges = df_edges.join(all_accounts.rename({"account_id":"from_acc","node_id":"src_id"}),on="from_acc",how="left")
df_edges = df_edges.join(all_accounts.rename({"account_id":"to_acc","node_id":"dst_id"}),on="to_acc",how="left")

ts = df_edges["Timestamp"]; mn_ts=ts.min(); mx_ts=ts.max(); ts_rng=(mx_ts-mn_ts).total_seconds()
etn = ((ts-mn_ts).dt.total_microseconds().cast(pl.Float64)/1e6/max(ts_rng,1.0)).to_numpy().astype(np.float32)
hrs = df_edges["Timestamp"].dt.hour().to_numpy().astype(np.float32)
dow = df_edges["Timestamp"].dt.weekday().to_numpy().astype(np.float32)
time_feats = np.stack([etn, np.sin(2*np.pi*hrs/24), np.cos(2*np.pi*hrs/24),
                        np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7)], axis=1)

if edge_amount_col:
    amounts_raw = df_edges[edge_amount_col].cast(pl.Float64).fill_null(0.0).to_numpy().astype(np.float32)
    log_amounts = np.log1p(np.maximum(amounts_raw, 0)).astype(np.float32)
    log_amounts = log_amounts / (np.max(log_amounts) + 1e-8)
    round_flag = ((amounts_raw % 1000 == 0) | (amounts_raw % 5000 == 0)).astype(np.float32)
    global_mean_amt = np.mean(amounts_raw[amounts_raw > 0]) if np.any(amounts_raw > 0) else 1.0
    relative_amt = np.log1p(amounts_raw / (global_mean_amt + 1e-8)).astype(np.float32)
    relative_amt = relative_amt / (np.max(relative_amt) + 1e-8)
    amount_feats = np.stack([log_amounts, round_flag, relative_amt], axis=1)
else:
    amount_feats = np.zeros((len(df_edges), 3), dtype=np.float32)

if "Payment Format" in df_edges.columns:
    pf_series = df_edges["Payment Format"].cast(pl.Utf8).fill_null("unknown")
    is_wire = pf_series.str.contains("(?i)wire|swift|cheque").to_numpy().astype(np.float32)
    is_internal = pf_series.str.contains("(?i)ach|internal|reinvestment").to_numpy().astype(np.float32)
    type_feats = np.stack([is_wire, is_internal], axis=1)
else:
    type_feats = np.zeros((len(df_edges), 2), dtype=np.float32)

edge_features_all = np.concatenate([time_feats, amount_feats, type_feats], axis=1)
EDGE_RAW_DIM = edge_features_all.shape[1]
print(f"  📊 Edge 피처: {EDGE_RAW_DIM}개")

edge_index_train = torch.tensor(df_edges.select(["src_id","dst_id"]).to_numpy().T, dtype=torch.long)
edge_attr_train = torch.tensor(edge_features_all, dtype=torch.float32)
print(f"📊 Train 엣지: {edge_index_train.shape[1]:,}")

del df_raw_train, df_raw_train_edges, df_edges, ts, hrs, dow, etn
del time_feats, amount_feats, type_feats, edge_features_all; gc.collect()

# ─── 노드 피처: V2(38) + 그래프(13) + 조건부(10) + AML(5) + 규칙성(7) + 금액(8) = 81개
# ★ v7c.5: GNN 입력은 v7c.2와 동일하게 유지 (신규 피처는 XGBoost에만 투입)
print("  📊 노드 피처 구성 (GNN: v7c.2 동일 81차원)...")
df_v2_node = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])
df_node_all = (all_accounts.join(df_v2_node, on="account_id", how="left")
    .join(df_graph_feats, on="account_id", how="left")
    .join(df_cond_features, on="account_id", how="left")
    .join(df_aml_scores, on="account_id", how="left")
    .join(df_regularity, on="account_id", how="left")
    .join(df_amt_features, on="account_id", how="left")
    .fill_null(0.0))

# GNN 입력은 v7c.2 피처만 (81개) — 신규 피처는 XGBoost 단계에만 추가
gnn_input_cols = (gnn_feature_cols_v2 + graph_feature_cols + cond_feature_cols +
                  aml_score_cols + regularity_cols + amt_feature_cols)

for col in gnn_input_cols:
    if col in df_node_all.columns:
        df_node_all = df_node_all.with_columns(
            pl.when(pl.col(col).is_infinite()|pl.col(col).is_nan()).then(0.0).otherwise(pl.col(col)).alias(col))

X_node = torch.tensor(df_node_all.select(gnn_input_cols).to_numpy(), dtype=torch.float32)
IN_DIM = X_node.shape[1]
print(f"  ✅ GNN 노드 입력: {IN_DIM}차원 (v7c.2 동일)")
print(f"     = V2({len(gnn_feature_cols_v2)}) + 그래프({len(graph_feature_cols)}) + 조건부({len(cond_feature_cols)})")
print(f"       + AML({len(aml_score_cols)}) + 규칙성({len(regularity_cols)}) + 금액({len(amt_feature_cols)})")

target_labels = df_v2_train.filter(pl.col("is_laundering")==1).select("account_id").unique().with_columns(pl.lit(1).alias("label"))
df_labels = all_accounts.join(target_labels,on="account_id",how="left").fill_null(0)
Y_node = torch.tensor(df_labels["label"].to_numpy(), dtype=torch.long)
pos_node_ids = set(df_labels.filter(pl.col("label")==1)["node_id"].to_list())
active_df = pl.DataFrame({"account_id":df_v2_train["account_id"].unique(),"is_active":True})
mask_df = all_accounts.join(active_df,on="account_id",how="left").fill_null(False)
active_node_ids = mask_df.filter(pl.col("is_active"))["node_id"].to_list()
active_pos = [n for n in active_node_ids if n in pos_node_ids]
active_neg = [n for n in active_node_ids if n not in pos_node_ids]
n_neg_gnn = min(len(active_pos)*NEG_NODE_RATIO_GNN, len(active_neg))
np.random.seed(RANDOM_SEED)
sampled_neg = np.random.choice(active_neg,size=n_neg_gnn,replace=False).tolist()
sampled_nodes = set(active_pos+sampled_neg)
train_mask_np = np.zeros(num_nodes, dtype=bool)
for nid in sampled_nodes: train_mask_np[nid]=True
train_mask = torch.tensor(train_mask_np, dtype=torch.bool)
print(f"📊 GNN 노드: 양성 {len(active_pos):,} + 음성 {n_neg_gnn:,}")
del df_v2_node, df_node_all, mask_df, active_df; gc.collect()

graph_data = Data(x=X_node, edge_index=edge_index_train, edge_attr=edge_attr_train, y=Y_node)
graph_data.train_mask = train_mask; graph_data.num_nodes = num_nodes
del X_node, edge_index_train, edge_attr_train, Y_node, train_mask; gc.collect()
log_memory("GNN 데이터 준비 완료")

# =============================================================
# 4-6. Multi-Config GNN 탐색 → 최적 임베딩 선택
# =============================================================
print("\n🧠 [Step 4-6] Multi-Config GNN 탐색...")

class EdgeProjection(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, out_dim), nn.GELU(), nn.Linear(out_dim, out_dim))
    def forward(self, ea): return self.proj(ea)

class TGAT_Flex(nn.Module):
    def __init__(self, in_dim, hidden_dim, edge_raw_dim, edge_dim, n_heads, n_layers=2, dropout=0.2):
        super().__init__()
        self.edge_proj = EdgeProjection(edge_raw_dim, edge_dim)
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.n_layers = n_layers
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_layers)])
        self.convs = nn.ModuleList([
            TransformerConv(hidden_dim, hidden_dim//n_heads, heads=n_heads,
                            edge_dim=edge_dim, dropout=dropout, concat=True)
            for _ in range(n_layers)])
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim//2, 1))
        self.dropout = dropout; self.hidden_dim = hidden_dim

    def forward(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        for i in range(self.n_layers):
            h = h + F.dropout(self.convs[i](self.norms[i](h), edge_index, ee), p=self.dropout, training=self.training)
        return self.classifier(self.norm_out(h)).squeeze(-1)

    @torch.no_grad()
    def get_embedding(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        for i in range(self.n_layers):
            h = h + self.convs[i](self.norms[i](h), edge_index, ee)
        return self.norm_out(h)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"📌 디바이스: {device}")
pw = (len(sampled_nodes)-len(active_pos))/max(len(active_pos),1)
print(f"📊 pos_weight: {pw:.1f}")

def make_lr(opt, ep, n_ep):
    if ep < WARMUP_EPOCHS: lr = 1e-6+(BASE_LR-1e-6)*(ep/WARMUP_EPOCHS)
    else: p=(ep-WARMUP_EPOCHS)/max(n_ep-WARMUP_EPOCHS,1); lr=1e-5+(BASE_LR-1e-5)*0.5*(1+np.cos(np.pi*p))
    for pg in opt.param_groups: pg['lr']=lr
    return lr

GNN_CONFIGS = [
    {"label":"h64_L2",  "hidden_dim":64,  "n_layers":2, "n_heads":4, "edge_dim":16},
    {"label":"h128_L2", "hidden_dim":128, "n_layers":2, "n_heads":4, "edge_dim":32},
    {"label":"h64_L3",  "hidden_dim":64,  "n_layers":3, "n_heads":4, "edge_dim":16},
    {"label":"h128_L3", "hidden_dim":128, "n_layers":3, "n_heads":4, "edge_dim":32},
]

xgb_exclude_cols=["account_id","time_group","is_laundering","mode_format","currency_mode","date"]
xgb_v2_cols=[c for c in df_v2.columns if c not in xgb_exclude_cols]
apr_train = df_v2_train.filter(pl.col("is_laundering")==1).height / max(df_v2_train.height,1)
spw = max((1-apr_train)/apr_train, 1.0) if apr_train > 0 else 20.0

# ★ 신규 피처들을 base_feature_cols에 포함
base_feature_cols = (xgb_v2_cols + graph_feature_cols + cond_feature_cols + aml_score_cols +
                     cb_feature_cols + vol_norm_feature_cols +
                     win_shift_feature_cols + inout_feature_cols)

def build_eval_df(db, df_emb_local, emb_col_names):
    feature_cols = base_feature_cols + emb_col_names
    df = (db.join(df_graph_feats,on="account_id",how="left")
          .join(df_cond_features,on="account_id",how="left")
          .join(df_aml_scores,on="account_id",how="left")
          .join(df_cb_features,on="account_id",how="left")            # ★
          .join(df_vol_norm_features,on="account_id",how="left")      # ★
          .join(df_win_shift_features,on="account_id",how="left")     # ★
          .join(df_inout_features,on="account_id",how="left")         # ★
          .join(df_emb_local,on="account_id",how="left")
          .fill_null(0.0).fill_nan(0.0))
    for cn in feature_cols:
        if cn in df.columns:
            df = df.with_columns(pl.when(pl.col(cn).is_infinite()).then(0.0).otherwise(pl.col(cn)).alias(cn))
    return df, feature_cols

def quick_xgb_eval(df_emb_local, emb_col_names, label):
    df_tr, fcols = build_eval_df(df_v2.filter(pl.col("time_group")<train_cutoff), df_emb_local, emb_col_names)
    df_tr_pos=df_tr.filter(pl.col("is_laundering")==1); df_tr_neg=df_tr.filter(pl.col("is_laundering")==0)
    n_neg=min(len(df_tr_pos)*NEG_ROW_RATIO_XGB,len(df_tr_neg))
    df_tr_s=pl.concat([df_tr_pos,df_tr_neg.sample(n=n_neg,seed=RANDOM_SEED)]).sample(fraction=1.0,seed=RANDOM_SEED)
    X_tr=df_tr_s.select(fcols).to_pandas(); y_tr=df_tr_s["is_laundering"].to_numpy()
    del df_tr,df_tr_pos,df_tr_neg,df_tr_s; gc.collect()
    df_vl, _ = build_eval_df(df_v2.filter((pl.col("time_group")>=train_cutoff)&(pl.col("time_group")<val_cutoff)), df_emb_local, emb_col_names)
    X_vl=df_vl.select(fcols).to_pandas(); y_vl=df_vl["is_laundering"].to_numpy(); del df_vl; gc.collect()
    df_te, _ = build_eval_df(df_v2.filter(pl.col("time_group")>=val_cutoff), df_emb_local, emb_col_names)
    X_te=df_te.select(fcols).to_pandas(); y_te=df_te["is_laundering"].to_numpy(); del df_te; gc.collect()
    qp={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
        "random_state":42,"max_depth":8,"learning_rate":0.03,"scale_pos_weight":spw,
        "subsample":0.8,"colsample_bytree":0.7,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
    m=xgb.XGBClassifier(**qp,n_estimators=1500,early_stopping_rounds=100)
    m.fit(X_tr,y_tr,eval_set=[(X_vl,y_vl)],verbose=0)
    q_val=m.best_score; q_test=average_precision_score(y_te,m.predict_proba(X_te)[:,1])
    del m,X_tr,X_vl,X_te,y_tr,y_vl,y_te; gc.collect()
    return q_val, q_test

print(f"\n📊 {len(GNN_CONFIGS)}개 GNN 설정 탐색...")
print(f"  {'#':>2s} {'Label':<12s} | {'hidden':>6s} {'layers':>6s} {'edim':>4s} | {'Loss':>8s} {'Params':>8s} | {'QVal':>8s} {'QTest':>8s}")
print("  "+"-"*80)

gnn_results = []
for gi, gcfg in enumerate(GNN_CONFIGS):
    gl = gcfg["label"]; hd=gcfg["hidden_dim"]; nl=gcfg["n_layers"]
    nh=gcfg["n_heads"]; ed=gcfg["edge_dim"]
    train_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                                   input_nodes=graph_data.train_mask, shuffle=True, num_workers=4)
    full_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                                  input_nodes=None, shuffle=False, num_workers=4)
    model = TGAT_Flex(in_dim=IN_DIM, hidden_dim=hd, edge_raw_dim=EDGE_RAW_DIM,
                      edge_dim=ed, n_heads=nh, n_layers=nl, dropout=0.2).to(device)
    np_count = sum(p.numel() for p in model.parameters())
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw],device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6, weight_decay=1e-4)
    model.train(); bl=float('inf'); pc2=0; bs=None
    for epoch in range(N_EPOCHS_GNN):
        clr=make_lr(optimizer,epoch,N_EPOCHS_GNN); tl=0.0; nb2=0
        for batch in tqdm(train_loader,desc=f"[{gl}] Ep{epoch+1}",leave=False):
            batch=batch.to(device); optimizer.zero_grad()
            bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
            out=model(batch.x,batch.edge_index,bea)
            loss=criterion(out[:batch.batch_size],batch.y[:batch.batch_size].float())
            if torch.isnan(loss) or torch.isinf(loss): optimizer.zero_grad(); continue
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=0.5)
            optimizer.step(); tl+=loss.item(); nb2+=1
        al=tl/max(nb2,1)
        if (epoch+1)%10==0 or epoch==0: print(f"    [{gl}] Ep{epoch+1:2d} Loss={al:.4f}")
        if al<bl-0.0005: bl=al; pc2=0; bs={k:v.cpu().clone() for k,v in model.state_dict().items()}
        else: pc2+=1
        if pc2>=PATIENCE: break
    if bs: model.load_state_dict({k:v.to(device) for k,v in bs.items()}); del bs
    model.eval(); ae=[]
    with torch.no_grad():
        for batch in tqdm(full_loader,desc=f"[{gl}] Embed",leave=False):
            batch=batch.to(device)
            bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
            emb=model.get_embedding(batch.x,batch.edge_index,bea)
            ae.append(emb[:batch.batch_size].cpu())
    femb=torch.cat(ae,dim=0).numpy()
    del model,ae,train_loader,full_loader; torch.cuda.empty_cache(); gc.collect()
    ecn=[f"tgat_emb_{i}" for i in range(hd)]
    df_emb=pl.concat([all_accounts.select("account_id"),pl.DataFrame(femb,schema=ecn)],how="horizontal")
    qv,qt = quick_xgb_eval(df_emb, ecn, gl)
    print(f"  {gi+1:2d} {gl:<12s} | {hd:6d} {nl:6d} {ed:4d} | {bl:8.4f} {np_count:8,} | {qv:8.5f} {qt:8.4f}")
    gnn_results.append({"label":gl,"hidden_dim":hd,"n_layers":nl,"loss":bl,"params":np_count,
                         "q_val":qv,"q_test":qt,"embedding":femb,"ecn":ecn,"df_emb":df_emb})
    log_memory(f"{gl}")

best_gnn = max(gnn_results, key=lambda x: x["q_val"])
print(f"\n  🏆 Best GNN: {best_gnn['label']} (QVal={best_gnn['q_val']:.5f} QTest={best_gnn['q_test']:.4f})")
for r in gnn_results:
    if r is not best_gnn: del r["embedding"], r["df_emb"]
gc.collect()

emb_cols = best_gnn["ecn"]
df_tgat = best_gnn["df_emb"]
FINAL_HIDDEN_DIM = best_gnn["hidden_dim"]

print(f"\n🔗 Interaction 피처 ({best_gnn['label']})...")
df_tgat_with_pr=df_tgat.join(df_graph_feats.select(["account_id","graph_pagerank"]),on="account_id",how="left").fill_null(0.0)
emb_np=df_tgat_with_pr.select(emb_cols).to_numpy(); emb_var=np.var(emb_np,axis=0)
top_idx=np.argsort(emb_var)[::-1][:N_INTERACTION_DIMS]; top_ec=[emb_cols[i] for i in top_idx]
ie_l=[]; ic_l=[]
for cn in top_ec: nn2=f"interact_pr_x_{cn}"; ie_l.append((pl.col(cn)*pl.col("graph_pagerank")*1e6).alias(nn2)); ic_l.append(nn2)
ie_l.append((sum(pl.col(c)**2 for c in emb_cols).sqrt()*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_norm")); ic_l.append("interact_pr_x_emb_norm")
ie_l.append((sum(pl.col(c) for c in top_ec)*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_topsum")); ic_l.append("interact_pr_x_emb_topsum")
interaction_cols=ic_l
df_interactions=df_tgat_with_pr.with_columns(ie_l).select(["account_id"]+ic_l)
for c in ic_l: df_interactions=df_interactions.with_columns(pl.when(pl.col(c).is_infinite()|pl.col(c).is_nan()).then(0.0).otherwise(pl.col(c)).alias(c))
print(f"  ✅ Interaction {len(ic_l)}개")
del df_tgat_with_pr,emb_np,best_gnn["embedding"]; gc.collect()
del graph_data; torch.cuda.empty_cache(); gc.collect()
log_memory("GNN 탐색 완료")

# =============================================================
# 7. XGBoost 데이터 구성
# =============================================================
print("\n🚀 [Step 7] XGBoost 데이터 구성...")

# ★ v7c.5 핵심: XGBoost에만 선별 신규 피처 4개 추가
# v7c.4 기여도 결과: ws_partner_explosion(0.71%), ws_recent_only(0.57%)
#                   inout_fan_out_score(0.35%), inout_net_flow(추정)
# cb_*, vol_* 는 기여도 0.5% 미만으로 제외
selected_new_cols = [
    "ws_partner_explosion",   # 기여도 1위 신규 피처
    "ws_recent_only",         # 기여도 2위 신규 피처
    "inout_fan_out_score",    # smurf 핵심 패턴
    "inout_net_flow",         # 순 자금 흐름 방향
]
# 실제 존재하는 것만 사용
selected_new_cols = [c for c in selected_new_cols
                     if c in win_shift_feature_cols or c in inout_feature_cols]

xgb_feature_cols = (xgb_v2_cols + graph_feature_cols + cond_feature_cols + aml_score_cols +
                    selected_new_cols +
                    emb_cols + interaction_cols)
print(f"📊 XGBoost 총 피처: {len(xgb_feature_cols)}")
print(f"   V2:{len(xgb_v2_cols)} G:{len(graph_feature_cols)} Cond:{len(cond_feature_cols)} AML:{len(aml_score_cols)}")
print(f"   ★선별 신규:{len(selected_new_cols)}개 → {selected_new_cols}")
print(f"   Emb:{len(emb_cols)} Inter:{len(interaction_cols)}")

def build_xgb_df(db):
    df=(db.join(df_graph_feats,on="account_id",how="left")
          .join(df_cond_features,on="account_id",how="left")
          .join(df_aml_scores,on="account_id",how="left")
          .join(df_win_shift_features,on="account_id",how="left")
          .join(df_inout_features,on="account_id",how="left")
          .join(df_tgat,on="account_id",how="left")
          .join(df_interactions,on="account_id",how="left")
          .fill_null(0.0).fill_nan(0.0))
    for cn in xgb_feature_cols:
        if cn in df.columns: df=df.with_columns(pl.when(pl.col(cn).is_infinite()).then(0.0).otherwise(pl.col(cn)).alias(cn))
    return df

df_tr=build_xgb_df(df_v2.filter(pl.col("time_group")<train_cutoff))
df_tr_pos=df_tr.filter(pl.col("is_laundering")==1); df_tr_neg=df_tr.filter(pl.col("is_laundering")==0)
n_neg=min(len(df_tr_pos)*NEG_ROW_RATIO_XGB,len(df_tr_neg))
df_tr_s=pl.concat([df_tr_pos,df_tr_neg.sample(n=n_neg,seed=RANDOM_SEED)]).sample(fraction=1.0,seed=RANDOM_SEED)
X_train=df_tr_s.select(xgb_feature_cols).to_pandas(); y_train=df_tr_s["is_laundering"].to_numpy()
print(f"📊 Train: {len(df_tr):,} → {len(df_tr_s):,} ({y_train.mean()*100:.2f}%)")
del df_tr,df_tr_pos,df_tr_neg,df_tr_s; gc.collect()
df_vl=build_xgb_df(df_v2.filter((pl.col("time_group")>=train_cutoff)&(pl.col("time_group")<val_cutoff)))
X_val=df_vl.select(xgb_feature_cols).to_pandas(); y_val=df_vl["is_laundering"].to_numpy(); print(f"📊 Val: {len(X_val):,}"); del df_vl; gc.collect()
df_te=build_xgb_df(df_v2.filter(pl.col("time_group")>=val_cutoff))
df_test_meta=df_te.select(["account_id","time_group","is_laundering"])
X_test=df_te.select(xgb_feature_cols).to_pandas(); y_test=df_te["is_laundering"].to_numpy(); print(f"📊 Test: {len(X_test):,}")
del df_te,df_v2,df_tgat,df_graph_feats,df_cond_features,df_aml_scores
del df_cb_features,df_vol_norm_features,df_win_shift_features,df_inout_features
del df_interactions,df_regularity,df_amt_features; gc.collect()
log_memory("XGBoost 준비")

# =============================================================
# 8. XGBoost 2-Stage + 확장 HP 그리드
# =============================================================
print("\n🔥 [Step 8] 2-Stage Pruning + 확장 HP 그리드...")
apr=y_train.sum()/len(y_train); spw=max((1-apr)/apr,1.0)

print("\n── Stage 1 ──")
s1p={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda","random_state":42,"max_depth":8,"learning_rate":0.03,"scale_pos_weight":spw,"subsample":0.8,"colsample_bytree":0.7,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
m_s1=xgb.XGBClassifier(**s1p,n_estimators=1500,early_stopping_rounds=100)
m_s1.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=100)
s1_val=m_s1.best_score; s1_test=average_precision_score(y_test,m_s1.predict_proba(X_test)[:,1])
print(f"\n  Stage1: Val={s1_val:.5f} Test={s1_test:.4f}")
imp_s1=m_s1.feature_importances_; feat_imp=sorted(zip(xgb_feature_cols,imp_s1),key=lambda x:x[1],reverse=True)
del m_s1; gc.collect()

print("\n── Stage 2: 확장 HP 그리드 ──")
bp2={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda","random_state":42,"scale_pos_weight":spw}
configs=[]
# ★ v7c.5: top 범위를 45~100으로 확장 (v7c.4 best=top70 근처를 촘촘하게)
for nf in [45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]:
    sel=[n for n,_ in feat_imp[:nf]]
    # 기본 4종 HP
    configs.append({"label":f"top{nf}_d8_c07","features":sel,"hp":{"max_depth":8,"learning_rate":0.03,"colsample_bytree":0.7,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":1500}})
    configs.append({"label":f"top{nf}_lr02_c08","features":sel,"hp":{"max_depth":8,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}})
    configs.append({"label":f"top{nf}_d9_lr02","features":sel,"hp":{"max_depth":9,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}})
    configs.append({"label":f"top{nf}_d9_c09","features":sel,"hp":{"max_depth":9,"learning_rate":0.02,"colsample_bytree":0.9,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}})
    # ★ 신규: depth 10 탐색 (v7c.4에서 d9가 best였으므로 한 단계 더)
    configs.append({"label":f"top{nf}_d10_lr02","features":sel,"hp":{"max_depth":10,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}})
configs.append({"label":"all","features":xgb_feature_cols,"hp":{"max_depth":9,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}})

print(f"\n📊 {len(configs)}개 조합 탐색...")
print(f"  {'#':>2s} {'Label':<22s} | {'#f':>3s} {'d':>2s} {'LR':>5s} {'col':>4s} | {'Val':>9s} {'it':>5s}")
print("  "+"-"*65)
best_s2={"val":-1}
for i,cfg in enumerate(configs):
    hp=cfg["hp"].copy(); ne=hp.pop("n_estimators"); sf=cfg["features"]
    m=xgb.XGBClassifier(**{**bp2,**hp},n_estimators=ne,early_stopping_rounds=100)
    m.fit(X_train[sf],y_train,eval_set=[(X_val[sf],y_val)],verbose=0)
    va=m.best_score; bi=m.best_iteration
    print(f"  {i+1:2d} {cfg['label']:<22s} | {len(sf):3d} {hp.get('max_depth',8):2d} {hp.get('learning_rate',0.03):5.3f} {hp.get('colsample_bytree',0.7):4.2f} | {va:9.5f} {bi:5d}")
    if va>best_s2["val"]:
        if best_s2.get("model"): del best_s2["model"]
        best_s2={"label":cfg["label"],"features":sf,"val":va,"iter":bi,"model":m,"hp":hp}
    else: del m
    gc.collect()

y_prob=best_s2["model"].predict_proba(X_test[best_s2["features"]])[:,1]
best_features=best_s2["features"]
print(f"\n  🏆 Best: {best_s2['label']} ({len(best_features)}개) Val={best_s2['val']:.5f} iter={best_s2['iter']}")

imp=best_s2["model"].feature_importances_; tidx=np.argsort(imp)[::-1][:30]
print(f"\n🔍 Feature Importance Top 30")
for i,idx in enumerate(tidx):
    c=best_features[idx]; t=""
    if c.startswith("tgat_emb_"): t=" [TGAT]"
    elif c.startswith("graph_"): t=" [GRAPH]"
    elif c.startswith("cond_"): t=" [COND]"
    elif c.startswith("score_"): t=" [SCORE]"
    elif c.startswith("interact_"): t=" [INTER]"
    elif c.startswith("cb_"): t=" [★CB]"
    elif c.startswith("vol_"): t=" [★VOL]"
    elif c.startswith("ws_"): t=" [★WIN]"
    elif c.startswith("inout_"): t=" [★INOUT]"
    print(f"  {i+1:2d}. {c:<45s}: {imp[idx]:.4f}{t}")

ti=imp.sum()
def group_imp(prefix): return sum(imp[i] for i,c in enumerate(best_features) if c.startswith(prefix))
print(f"\n📊 그룹별 중요도:")
print(f"   V2        : {group_imp('') / ti * 100:.1f}% (기존)")
print(f"   Graph     : {group_imp('graph_') / ti * 100:.1f}%")
print(f"   TGAT      : {group_imp('tgat_emb_') / ti * 100:.1f}%")
print(f"   Interact  : {group_imp('interact_') / ti * 100:.1f}%")
print(f"   ★ CB      : {group_imp('cb_') / ti * 100:.1f}%")
print(f"   ★ VolNorm : {group_imp('vol_') / ti * 100:.1f}%")
print(f"   ★ WinShift: {group_imp('ws_') / ti * 100:.1f}%")
print(f"   ★ InOut   : {group_imp('inout_') / ti * 100:.1f}%")
del X_train,X_val; gc.collect()

# =============================================================
# 9. 최종 리포트
# =============================================================
print("\n"+"="*60); print("🏆 [TGAT v7c.5] 최종 리포트"); print("="*60)
auprc=average_precision_score(y_test,y_prob); print(f"\n  AUPRC: {auprc:.4f}")
best_f1=0; best_thresh=0
for th in np.arange(0.05,0.95,0.01):
    f=f1_score(y_test,(y_prob>=th).astype(int),zero_division=0)
    if f>best_f1: best_f1=f; best_thresh=th
y_pred=(y_prob>=best_thresh).astype(int)
prec=precision_score(y_test,y_pred,zero_division=0); rec=recall_score(y_test,y_pred,zero_division=0)
print(f"\n📌 최적: T={best_thresh:.2f} F1={best_f1:.4f} P={prec:.4f} R={rec:.4f}")
print(f"\n📌 임계값별:")
for t in [0.1,0.3,0.5,0.7,0.8,0.9]:
    yp=(y_prob>=t).astype(int)
    print(f"  T={t:.1f} P={precision_score(y_test,yp,zero_division=0):.4f} R={recall_score(y_test,yp,zero_division=0):.4f} F1={f1_score(y_test,yp,zero_division=0):.4f}")

df_res=df_test_meta.with_columns([pl.col("time_group").dt.date().alias("date"),pl.Series("pred_prob",y_prob)])
df_dist=df_res.sort("pred_prob",descending=True).unique(subset=["account_id"],maintain_order=True)
print(f"\n📌 Top-K:")
for k in [100,500,1000,5000,10000]:
    ck=min(k,len(df_dist)); hits=df_dist.head(ck)["is_laundering"].sum()
    print(f"  Top-{k:5d}: {hits:5d}명 ({hits/ck*100:.2f}%)")
print(f"\n📌 일별 Top-100:")
for d in df_dist["date"].unique(maintain_order=True).sort(descending=True):
    hits=df_dist.filter(pl.col("date")==d).head(100)["is_laundering"].sum(); print(f"  {d}: {hits}")

tn,fp,fn,tp=confusion_matrix(y_test,y_pred).ravel()
print(f"\n📌 v7c.4→v7c.5: AUPRC {auprc:.4f} | F1 {best_f1:.4f}")
print(f"  TP:{tp:,} FP:{fp:,} FN:{fn:,} TN:{tn:,}")
print(f"  Precision:{prec:.4f} Recall:{rec:.4f}")

print(f"\n📌 GNN 탐색 결과:")
for r in gnn_results:
    tag = " ★" if r["label"]==best_gnn["label"] else ""
    print(f"  {r['label']:<12s}: QVal={r['q_val']:.5f} QTest={r['q_test']:.4f} Loss={r['loss']:.4f} Params={r['params']:,}{tag}")

print(f"\n📌 2-Stage 결과:")
print(f"  S1({len(xgb_feature_cols)}개): Val={s1_val:.5f} Test={s1_test:.4f}")
print(f"  S2({best_s2['label']},{len(best_features)}개): Val={best_s2['val']:.5f} Test={auprc:.4f}")

print(f"\n✨ v7c.5 변경 요약:")
print(f"  GNN 입력   : 99차원(v7c.4) → 81차원(v7c.2 복원)")
print(f"  XGB 신규   : 19개(v7c.4) → {len(selected_new_cols)}개 선별 → {selected_new_cols}")
print(f"  Stage2 범위: top45~80(v7c.4) → top45~100 + depth10 추가")
print(f"  총 XGB 피처: {len(xgb_feature_cols)}개")

del X_test; gc.collect(); log_memory("최종 완료")
print(f"\n✅ TGAT v7c.5 완료!")

🛡️ [TGAT v7c.5] GNN v7c.2 복원 + XGBoost 선별 신규 피처
   ▸ GNN 입력: v7c.2 동일 81차원 (신규 피처 노이즈 제거)
   ▸ XGBoost 추가 피처 (기여도 상위만):
     ① ws_partner_explosion, ws_recent_only (ws_* 상위 2개)
     ② inout_fan_out_score, inout_net_flow  (inout_* 상위 2개)
   ▸ GNN 탐색: hidden 64/128 × layer 2/3 = 4개
   ▸ XGBoost: 확장 HP 그리드 — top범위 45~100, 42조합

📂 데이터 로딩 중...
⏱️ Train cutoff: 2022-09-09 20:00:00
⏱️ Val   cutoff: 2022-09-14 05:00:00
📊 전체 노드 수: 2,076,999
  💾 [데이터 로드 완료] RAM: 102.43GB | GPU: 0.02GB

📐 [Step 2] 전체 노드 피처 계산...
  📊 그래프 구조 피처...
  📊 PageRank...
  ✅ 그래프 피처 13개
  📊 조건부행동 피처...
  ✅ 조건부행동 10개
  📊 AML 점수...
  ✅ AML 점수 5개
  📊 규칙성 피처...
  ✅ 규칙성 7개
  📊 금액 분포 피처...
  ✅ 금액 분포 8개
  📊 ★ Cross-border 정밀화 피처 (신규)...
  ✅ Cross-border 정밀화 4개: ['cb_new_partner_ratio', 'cb_oneway_new', 'cb_amount_asymmetry', 'cb_burst_cross']
  📊 ★ 거래량 대비 상대 정규화 피처 (신규)...
  ✅ 거래량 대비 정규화 5개: ['vol_amount_per_peer', 'vol_concentrated_amount', 'vol_low_freq_high_amt', 'vol_degree_normalized_amount', 'vol_outlier_score']
  📊 ★ 시간 윈도우

    [h64_L2] Ep 1 Loss=1.3763


    [h64_L2] Ep10 Loss=1.0584


    [h64_L2] Ep20 Loss=1.0274


    [h64_L2] Ep30 Loss=1.0075


...

    [h128_L3] Ep10 Loss=1.0482


    [h128_L3] Ep20 Loss=1.0111


    [h128_L3] Ep30 Loss=0.9848


    [h128_L3] Ep40 Loss=0.9760


   4 h128_L3      |    128      3   32 |   0.9729  231,681 |  0.38212   0.4073
  💾 [h128_L3] RAM: 86.00GB | GPU: 0.02GB

  🏆 Best GNN: h64_L2 (QVal=0.38912 QTest=0.4199)

🔗 Interaction 피처 (h64_L2)...
  ✅ Interaction 10개
  💾 [GNN 탐색 완료] RAM: 82.90GB | GPU: 0.02GB

🚀 [Step 7] XGBoost 데이터 구성...
📊 XGBoost 총 피처: 144
   V2:38 G:13 Cond:10 AML:5
   ★선별 신규:4개 → ['ws_partner_explosion', 'ws_recent_only', 'inout_fan_out_score', 'inout_net_flow']
   Emb:64 Inter:10
📊 Train: 17,172,747 → 619,437 (4.76%)
📊 Val: 5,790,644
📊 Test: 5,792,913
  💾 [XGBoost 준비] RAM: 99.23GB | GPU: 0.02GB

🔥 [Step 8] 2-Stage Pruning + 확장 HP 그리드...

── Stage 1 ──
[0]	validation_0-aucpr:0.11880
[100]	validation_0-aucpr:0.27184
[200]	validation_0-aucpr:0.36481
[300]	validation_0-aucpr:0.39128
[400]	validation_0-aucpr:0.40461
[500]	validation_0-aucpr:0.40979
[600]	validation_0-aucpr:0.41608
[700]	validation_0-aucpr:0.41784
[800]	validation_0-aucpr:0.41885
[900]	validation_0-aucpr:0.42282
[1000]	validation_0-aucpr:0.42620
[110

In [3]:
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv
from torch_geometric.loader import NeighborLoader
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm
import gc, numpy as np, psutil, os, warnings
warnings.filterwarnings("ignore", category=UserWarning)

def log_memory(tag=""):
    rss = psutil.Process(os.getpid()).memory_info().rss/(1024**3)
    gpu = torch.cuda.memory_allocated()/(1024**3) if torch.cuda.is_available() else 0
    print(f"  💾 [{tag}] RAM: {rss:.2f}GB | GPU: {gpu:.2f}GB")

print("=" * 60)
print("🛡️ [TGAT v7c.6] Forward Selection 피처 선택")
print("   ▸ GNN 입력: v7c.2 동일 81차원 유지")
print("   ▸ XGBoost 피처 선택 방식 개선:")
print("     기존: Stage1 importance top-K 일괄 선택")
print("     변경: Forward Selection — 하나씩 추가, Val AUCPR")
print("           실제 개선 시만 채택 (중복/노이즈 자동 탈락)")
print("   ▸ XGBoost 입력 후보: 모든 신규 피처 포함 (ws_*, inout_*,")
print("     cb_*, vol_*) → FS가 유효한 것만 선별")
print("   ▸ Stage 3: FS 채택 피처 고정 후 HP 그리드 7종")
print("=" * 60)

# ===================== 설정 =====================
NEG_NODE_RATIO_GNN = 50; NEG_ROW_RATIO_XGB = 20; RANDOM_SEED = 42
HIDDEN_DIM = 64; EDGE_DIM = 16; N_HEADS = 4
N_EPOCHS_GNN = 40; BATCH_SIZE = 2048; NUM_NEIGHBORS = [15, 10]
N_INTERACTION_DIMS = 8; PATIENCE = 10; BASE_LR = 0.001; WARMUP_EPOCHS = 5

raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"

# =============================================================
# 1. 데이터 로드
# =============================================================
print("\n📂 데이터 로딩 중...")
df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))
if df_raw["Timestamp"].dtype == pl.Utf8:
    df_raw = df_raw.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_v2 = df_v2.sort("time_group")
total_count = len(df_v2)
train_cutoff = df_v2["time_group"][int(total_count*0.6)]
val_cutoff = df_v2["time_group"][int(total_count*0.8)]
print(f"⏱️ Train cutoff: {train_cutoff}\n⏱️ Val   cutoff: {val_cutoff}")

all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")
num_nodes = len(all_accounts); print(f"📊 전체 노드 수: {num_nodes:,}")
log_memory("데이터 로드 완료")

# =============================================================
# 2. 모든 노드 피처 계산 (GNN 입력용 + XGBoost 겸용)
# =============================================================
print("\n📐 [Step 2] 전체 노드 피처 계산...")
df_raw_train = df_raw.filter(pl.col("Timestamp") < train_cutoff)

# ─── 2-1. 그래프 구조 피처 13개 ───
print("  📊 그래프 구조 피처...")
df_from = df_raw_train.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id"))
df_to = df_raw_train.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
out_degree = df_from.group_by("account_id").len().rename({"len":"graph_out_degree"})
in_degree = df_to.group_by("account_id").len().rename({"len":"graph_in_degree"})
unique_out = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("to_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_out_partners"))
unique_in = df_raw_train.select([pl.col("to_acc").cast(pl.Utf8).alias("account_id"),pl.col("from_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_in_partners"))
esf = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("a"),pl.col("to_acc").cast(pl.Utf8).alias("b")]).unique()
est = esf.select([pl.col("b").alias("a"),pl.col("a").alias("b")])
bidir = esf.join(est, on=["a","b"], how="inner")
bidir_count = bidir.select(pl.col("a").alias("account_id")).group_by("account_id").len().rename({"len":"graph_bidir_count"})
del esf, est, bidir; gc.collect()
time_features = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("Timestamp")]).group_by("account_id").agg([
    pl.col("Timestamp").sort().diff().dt.total_seconds().mean().alias("graph_avg_tx_interval"),
    pl.col("Timestamp").sort().diff().dt.total_seconds().std().alias("graph_std_tx_interval"),
    pl.col("Timestamp").count().alias("graph_total_tx_count")])

# PageRank
print("  📊 PageRank...")
account_to_id = dict(zip(all_accounts["account_id"].to_list(), all_accounts["node_id"].to_list()))
src_ids = df_raw_train["from_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
dst_ids = df_raw_train["to_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
vm=(src_ids>=0)&(dst_ids>=0); sv=src_ids[vm]; dv=dst_ids[vm]
oda=np.zeros(num_nodes,dtype=np.float64); np.add.at(oda,sv,1.0); oda=np.maximum(oda,1.0)
pr=np.ones(num_nodes,dtype=np.float64)/num_nodes
for _ in range(10): npr=np.ones(num_nodes,dtype=np.float64)*0.15/num_nodes; np.add.at(npr,dv,0.85*pr[sv]/oda[sv]); pr=npr
pr_df = pl.DataFrame({"node_id":np.arange(num_nodes,dtype=np.uint32),"graph_pagerank":pr.astype(np.float32)})
pr_df = all_accounts.join(pr_df,on="node_id",how="left").select(["account_id","graph_pagerank"])
del src_ids,dst_ids,sv,dv,oda,pr,npr; gc.collect()

df_graph_feats = (all_accounts.select("account_id").join(out_degree,on="account_id",how="left").join(in_degree,on="account_id",how="left")
    .join(unique_out,on="account_id",how="left").join(unique_in,on="account_id",how="left")
    .join(bidir_count,on="account_id",how="left").join(time_features,on="account_id",how="left")
    .join(pr_df,on="account_id",how="left").fill_null(0.0)
    .with_columns([(pl.col("graph_out_degree")+pl.col("graph_in_degree")).alias("graph_total_degree"),
        (pl.col("graph_out_degree")/(pl.col("graph_in_degree")+1)).alias("graph_out_in_ratio"),
        (pl.col("graph_bidir_count")/(pl.col("graph_out_degree")+1)).alias("graph_reciprocity"),
        (pl.col("graph_unique_out_partners")/(pl.col("graph_out_degree")+1)).alias("graph_partner_diversity")]))
graph_feature_cols = [c for c in df_graph_feats.columns if c.startswith("graph_")]
del out_degree,in_degree,unique_out,unique_in,bidir_count,time_features,pr_df,df_from,df_to; gc.collect()
print(f"  ✅ 그래프 피처 {len(graph_feature_cols)}개")

# ─── 2-2. 조건부행동 피처 10개 ───
print("  📊 조건부행동 피처...")
df_v2_train = df_v2.filter(pl.col("time_group")<train_cutoff)
exclude_cols = ["account_id","time_group","is_laundering","mode_format","currency_mode"]
gnn_feature_cols_v2 = [c for c in df_v2.columns if c not in exclude_cols]
df_v2_agg = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])

df_cond = df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
v2a=set(df_cond.columns)
def sc(n): return pl.col(n) if n in v2a else pl.lit(0.0)
ce=[]
if "cnt_night" in v2a and "cnt_1h" in v2a: ce.append((pl.col("graph_total_degree")*sc("cnt_night")/(sc("cnt_1h")+1)).alias("cond_highdeg_night_intensity"))
if "avg_log_amount" in v2a: ce.append((sc("avg_log_amount")/(pl.col("graph_total_degree").log1p()+1)).alias("cond_lowdeg_high_amount"))
if "ratio_cross_border" in v2a: ce.append((sc("ratio_cross_border")*(1-pl.col("graph_reciprocity"))).alias("cond_oneway_crossborder"))
if "cnt_1h" in v2a: ce.append((pl.col("graph_unique_out_partners")*sc("cnt_1h")/(pl.col("graph_out_degree")+1)).alias("cond_fanout_burst"))
if "cnt_risk_format" in v2a: ce.append((pl.col("graph_pagerank")*sc("cnt_risk_format")).alias("cond_hub_risk_format"))
if "cnt_wire" in v2a: ce.append((pl.col("graph_out_in_ratio")*sc("cnt_wire")).alias("cond_asymmetric_wire"))
if "cnt_inter_bank" in v2a: ce.append((pl.col("graph_std_tx_interval").fill_null(0)/(pl.col("graph_avg_tx_interval").fill_null(1)+1)*sc("cnt_inter_bank")).alias("cond_irregular_interbank"))
if "cnt_currencies" in v2a: ce.append((pl.col("graph_partner_diversity")*sc("cnt_currencies")).alias("cond_diverse_partner_currency"))
if "amount_kurtosis" in v2a: ce.append((pl.col("graph_reciprocity")*sc("amount_kurtosis")).alias("cond_bidir_amount_volatility"))
if "entity_acct_cnt" in v2a and "burst_ratio" in v2a: ce.append((sc("entity_acct_cnt")*sc("burst_ratio")).alias("cond_multi_acct_burst"))
df_cond=df_cond.with_columns(ce); cond_feature_cols=[e.meta.output_name() for e in ce]
for cn in cond_feature_cols: df_cond=df_cond.with_columns(pl.when(pl.col(cn).is_infinite()|pl.col(cn).is_nan()).then(0.0).otherwise(pl.col(cn)).alias(cn))
df_cond_features=df_cond.select(["account_id"]+cond_feature_cols); del df_cond; gc.collect()
print(f"  ✅ 조건부행동 {len(cond_feature_cols)}개")

# ─── 2-3. AML 점수 5개 ───
print("  📊 AML 점수...")
df_sb=df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
se=[((sc("cnt_1h").fill_null(0).clip(0,1e6)+1).log1p()*pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("burst_ratio").fill_null(0).clip(0,1e6)/(sc("avg_log_amount").fill_null(1).clip(0,1e6)+1)).alias("score_smurf"),
    ((1.0/((pl.col("graph_out_in_ratio").clip(0.01,100)-1.0).abs()+0.1))*sc("cnt_wire").fill_null(0).clip(0,1e6).log1p()*(1-pl.col("graph_reciprocity").clip(0,1))/(pl.col("graph_unique_out_partners").clip(0,1e6).log1p()+1)).alias("score_mule"),
    (pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("cnt_currencies").fill_null(0).clip(0,1e6).log1p()*sc("ratio_cross_border").fill_null(0).clip(0,1)/(pl.col("graph_avg_tx_interval").fill_null(3600).clip(0,1e9)/3600+1)).alias("score_layering"),
    (pl.col("graph_in_degree").clip(0,1e6).log1p()*(pl.col("graph_pagerank").clip(0,1)*1e6)*sc("cnt_risk_format").fill_null(0).clip(0,1e6).log1p()/(pl.col("graph_out_degree").clip(0,1e6).log1p()+1)).alias("score_integration")]
df_sb=df_sb.with_columns(se)
sn=["score_smurf","score_mule","score_layering","score_integration"]
for s in sn: df_sb=df_sb.with_columns(pl.when(pl.col(s).is_infinite()|pl.col(s).is_nan()).then(0.0).otherwise(pl.col(s)).alias(s))
for s in sn:
    mn=df_sb[s].min();mx=df_sb[s].max();rng=mx-mn if(mx-mn)>0 else 1.0
    df_sb=df_sb.with_columns(((pl.col(s)-mn)/rng).alias(f"{s}_norm"))
df_sb=df_sb.with_columns((pl.col("score_smurf_norm")*0.3+pl.col("score_mule_norm")*0.25+pl.col("score_layering_norm")*0.3+pl.col("score_integration_norm")*0.15).alias("score_composite"))
aml_score_cols=sn+["score_composite"]
df_aml_scores=df_sb.select(["account_id"]+aml_score_cols)
del df_sb, df_v2_agg; gc.collect()
print(f"  ✅ AML 점수 {len(aml_score_cols)}개")

# ─── 2-4. 규칙성 피처 7개 ───
print("  📊 규칙성 피처...")
df_tx_time = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner"),
    pl.col("Timestamp"),
    pl.col("Timestamp").dt.hour().alias("hour"),
    pl.col("Timestamp").dt.weekday().alias("dow"),
])
reg_agg = df_tx_time.group_by("account_id").agg([
    (pl.col("hour").is_between(9,17).mean()).alias("reg_business_hour_ratio"),
    (pl.col("dow").is_in([5,6]).mean()).alias("reg_weekend_ratio"),
    (pl.col("hour").is_between(0,5).mean()).alias("reg_night_strict_ratio"),
    pl.col("partner").count().alias("_total_tx"),
])
hc=df_tx_time.group_by(["account_id","hour"]).len().rename({"len":"h_count"})
hp=hc.pivot(on="hour",index="account_id",values="h_count").fill_null(0)
hcols=[c for c in hp.columns if c!="account_id"]
hmat=hp.select(hcols).to_numpy().astype(np.float64); hsum=np.maximum(hmat.sum(axis=1,keepdims=True),1.0)
hprobs=hmat/hsum; hlog=np.where(hprobs>0,np.log(hprobs+1e-12),0.0)
df_h_ent=pl.DataFrame({"account_id":hp["account_id"],"reg_hour_entropy":-np.sum(hprobs*hlog,axis=1).astype(np.float32)})
del hc,hp,hmat,hprobs,hlog; gc.collect()
dc=df_tx_time.group_by(["account_id","dow"]).len().rename({"len":"d_count"})
dp=dc.pivot(on="dow",index="account_id",values="d_count").fill_null(0)
dcols=[c for c in dp.columns if c!="account_id"]
dmat=dp.select(dcols).to_numpy().astype(np.float64); dsum=np.maximum(dmat.sum(axis=1,keepdims=True),1.0)
dprobs=dmat/dsum; dlog=np.where(dprobs>0,np.log(dprobs+1e-12),0.0)
df_d_ent=pl.DataFrame({"account_id":dp["account_id"],"reg_dow_entropy":-np.sum(dprobs*dlog,axis=1).astype(np.float32)})
del dc,dp,dmat,dprobs,dlog; gc.collect()
pf=df_tx_time.group_by(["account_id","partner"]).len().rename({"len":"p_count"})
rp=pf.filter(pl.col("p_count")>=2).group_by("account_id").len().rename({"len":"repeat_cnt"})
tp2=pf.group_by("account_id").len().rename({"len":"total_cnt"})
df_pc=tp2.join(rp,on="account_id",how="left").fill_null(0).with_columns((pl.col("repeat_cnt")/(pl.col("total_cnt")+1)).alias("reg_consistent_partner_ratio")).select(["account_id","reg_consistent_partner_ratio"])
top_p=pf.sort(["account_id","p_count"],descending=[False,True]).unique(subset=["account_id"],maintain_order=True).rename({"p_count":"top_cnt"}).select(["account_id","top_cnt"])
df_tc=reg_agg.select(["account_id","_total_tx"]).join(top_p,on="account_id",how="left").fill_null(0).with_columns((pl.col("top_cnt")/(pl.col("_total_tx")+1)).alias("reg_top_partner_concentration")).select(["account_id","reg_top_partner_concentration"])
del pf,rp,tp2,top_p; gc.collect()
df_regularity=(reg_agg.select(["account_id","reg_business_hour_ratio","reg_weekend_ratio","reg_night_strict_ratio"])
    .join(df_h_ent,on="account_id",how="left").join(df_d_ent,on="account_id",how="left")
    .join(df_pc,on="account_id",how="left").join(df_tc,on="account_id",how="left").fill_null(0.0))
regularity_cols=[c for c in df_regularity.columns if c.startswith("reg_")]
for rc in regularity_cols: df_regularity=df_regularity.with_columns(pl.when(pl.col(rc).is_infinite()|pl.col(rc).is_nan()).then(0.0).otherwise(pl.col(rc)).alias(rc))
del reg_agg,df_h_ent,df_d_ent,df_pc,df_tc; gc.collect()
print(f"  ✅ 규칙성 {len(regularity_cols)}개")

# ─── 2-5. 금액 분포 피처 8개 ───
print("  📊 금액 분포 피처...")
amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train.columns: amount_col = cand; break
if amount_col:
    df_amounts=df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col(amount_col).cast(pl.Float64).alias("amount")]).filter(pl.col("amount")>0)
    df_amt=df_amounts.group_by("account_id").agg([
        ((pl.col("amount")%1000==0)|(pl.col("amount")%5000==0)).mean().alias("amt_round_number_ratio"),
        (pl.col("amount").std()/(pl.col("amount").mean()+1)).alias("amt_cv"),
        (pl.col("amount").max()/(pl.col("amount").mean()+1)).alias("amt_max_mean_ratio"),
        (pl.col("amount").median()/(pl.col("amount").mean()+1)).alias("amt_median_mean_ratio"),
        pl.col("amount").quantile(0.9).alias("_q90"), pl.col("amount").sum().alias("_tsum"),
        pl.col("amount").count().alias("_cnt"),
        pl.col("amount").quantile(0.75).alias("_q75"), pl.col("amount").quantile(0.25).alias("_q25"),
        pl.col("amount").median().alias("_med"),
    ]).fill_null(0.0).with_columns([
        (pl.col("_q90")*0.1*pl.col("_cnt")/(pl.col("_tsum")+1)).alias("amt_top10pct_concentration"),
        ((pl.col("_q75")-pl.col("_q25"))/(pl.col("_med")+1)).alias("amt_iqr_median_ratio")])
    gm=df_amounts["amount"].mean(); st=max(gm*0.1,1.0)
    df_sr=df_amounts.with_columns((pl.col("amount")<st).alias("is_small")).group_by("account_id").agg(pl.col("is_small").mean().alias("amt_small_tx_ratio"))
    df_amt=df_amt.join(df_sr,on="account_id",how="left").fill_null(0.0)
    df_al=df_amounts.with_columns((pl.col("amount")+1).log().cast(pl.Int32).alias("log_bucket"))
    bc=df_al.group_by(["account_id","log_bucket"]).len().rename({"len":"b_count"})
    bt=bc.group_by("account_id").agg(pl.col("b_count").sum().alias("b_total"))
    bp=bc.join(bt,on="account_id").with_columns((pl.col("b_count")/pl.col("b_total")).alias("b_prob"))
    ae=bp.group_by("account_id").agg((-1*(pl.col("b_prob")*(pl.col("b_prob")+1e-12).log()).sum()).alias("amt_entropy"))
    df_amt=df_amt.join(ae,on="account_id",how="left").fill_null(0.0)
    del df_amounts,df_al,bc,bt,bp,ae,df_sr; gc.collect()
    amt_feature_cols=[c for c in df_amt.columns if c.startswith("amt_") and not c.startswith("_")]
    df_amt_features=df_amt.select(["account_id"]+amt_feature_cols)
else:
    df_amt_features=all_accounts.select("account_id"); amt_feature_cols=[]
for ac in amt_feature_cols:
    df_amt_features=df_amt_features.with_columns(pl.when(pl.col(ac).is_infinite()|pl.col(ac).is_nan()).then(0.0).otherwise(pl.col(ac)).alias(ac))
print(f"  ✅ 금액 분포 {len(amt_feature_cols)}개")

# =============================================================
# ★ NEW: 2-6. Cross-border 정밀화 피처 4개
# =============================================================
# 핵심 아이디어: ratio_cross_border 단독이 아니라
#   "cross-border × 새로운 상대방 비율 × 방향 비대칭"을 결합
print("  📊 ★ Cross-border 정밀화 피처 (신규)...")
cb_feature_cols = []

# 데이터에 to_acc country/bank 정보가 있을 경우를 가정
# → 없으면 graph_partner_diversity 와 ratio_cross_border 조합으로 근사
df_v2_agg_cb = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2 if c in df_v2_train.columns
])
df_cb = df_v2_agg_cb.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)
v2cb = set(df_cb.columns)

cb_exprs = []
# ① cb_new_partner_ratio: cross-border이면서 새로운 상대방 비율
#    = ratio_cross_border × (unique_out_partners / (total_tx_count+1))
#    → 새로운 상대에게 해외 송금하는 패턴
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("graph_unique_out_partners") /
         (pl.col("graph_total_tx_count") + 1)
        ).alias("cb_new_partner_ratio")
    )
    cb_feature_cols.append("cb_new_partner_ratio")

# ② cb_oneway_new: cross-border × 비양방향 × 새 상대방
#    → 양방향 거래 없이 새로운 상대에게만 해외 송금 (전형적 분산 송금)
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         (1.0 - pl.col("graph_reciprocity")) *
         (pl.col("graph_unique_out_partners") / (pl.col("graph_out_degree") + 1))
        ).alias("cb_oneway_new")
    )
    cb_feature_cols.append("cb_oneway_new")

# ③ cb_amount_asymmetry: 해외 송금 비율 × 금액 비대칭 (out >> in)
#    = ratio_cross_border × out_in_ratio (clip 해서 폭발 방지)
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("graph_out_in_ratio").clip(0, 20)
        ).alias("cb_amount_asymmetry")
    )
    cb_feature_cols.append("cb_amount_asymmetry")

# ④ cb_burst_cross: cross-border이 짧은 시간 내 집중 (burst_ratio 결합)
if "ratio_cross_border" in v2cb and "burst_ratio" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("burst_ratio").clip(0, 100)
        ).alias("cb_burst_cross")
    )
    cb_feature_cols.append("cb_burst_cross")

if cb_exprs:
    df_cb = df_cb.with_columns(cb_exprs)
    for cn in cb_feature_cols:
        df_cb = df_cb.with_columns(
            pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
            .then(0.0).otherwise(pl.col(cn)).alias(cn))
    df_cb_features = df_cb.select(["account_id"] + cb_feature_cols)
else:
    df_cb_features = all_accounts.select("account_id"); cb_feature_cols = []
del df_cb, df_v2_agg_cb; gc.collect()
print(f"  ✅ Cross-border 정밀화 {len(cb_feature_cols)}개: {cb_feature_cols}")

# trend_* 피처 제거 (ws_* 와 중복) — v7c.4
trend_feature_cols = []

# =============================================================
# ★ NEW: 2-8. 거래량 대비 상대 정규화 피처 5개
# =============================================================
# 핵심 아이디어: cond_lowdeg_high_amount가 FN에서 TP보다 오히려 높아 역전됨
#   → "전체 네트워크 대비 비정상적으로 높은 금액"을 더 정밀하게 포착
print("  📊 ★ 거래량 대비 상대 정규화 피처 (신규)...")
vol_norm_feature_cols = []

df_v2_agg_vn = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2 if c in df_v2_train.columns
])
df_vn = df_v2_agg_vn.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)

# 네트워크 전체 평균값 계산 (정규화 기준)
if "avg_log_amount" in set(df_vn.columns):
    global_avg_log_amount = df_vn["avg_log_amount"].mean() or 1.0
    global_avg_degree = df_vn["graph_total_degree"].mean() or 1.0
    global_avg_cnt_1h = df_vn["cnt_1h"].mean() if "cnt_1h" in df_vn.columns else 1.0
    global_avg_cnt_1h = global_avg_cnt_1h or 1.0

    vn_exprs = []
    # ① vol_amount_per_peer: 건당 금액 / (파트너 수 × 전체 평균)
    #    → degree가 낮은데 금액이 높은 패턴을 전체 대비 상대적으로 포착
    vn_exprs.append(
        (pl.col("avg_log_amount") / (float(global_avg_log_amount) + 1e-8) /
         (pl.col("graph_unique_out_partners").log1p() + 1)
        ).alias("vol_amount_per_peer")
    )
    vol_norm_feature_cols.append("vol_amount_per_peer")

    # ② vol_concentrated_amount: 소수 파트너에게 집중 송금
    #    = avg_log_amount × (1 - partner_diversity)
    #    → diversity가 낮을수록 (=한 곳에 집중) × 금액이 높을수록 높아짐
    vn_exprs.append(
        (pl.col("avg_log_amount") * (1.0 - pl.col("graph_partner_diversity").clip(0, 1))
        ).alias("vol_concentrated_amount")
    )
    vol_norm_feature_cols.append("vol_concentrated_amount")

    # ③ vol_low_freq_high_amt: 낮은 빈도이지만 건당 금액이 전체 평균 대비 높음
    #    = (avg_log_amount / global_mean) / log1p(cnt_1h) → 드문데 고액
    if "cnt_1h" in set(df_vn.columns):
        vn_exprs.append(
            (pl.col("avg_log_amount") / float(global_avg_log_amount) /
             (pl.col("cnt_1h").clip(0, 1e6).log1p() + 1)
            ).alias("vol_low_freq_high_amt")
        )
        vol_norm_feature_cols.append("vol_low_freq_high_amt")

    # ④ vol_degree_normalized_amount: 디그리로 나눈 상대 금액 (올바른 역산 버전)
    #    기존 cond_lowdeg_high_amount 대체 — log 스케일 적용 + 전체 평균으로 나눔
    vn_exprs.append(
        ((pl.col("avg_log_amount") - float(global_avg_log_amount)) /
         (pl.col("graph_total_degree").clip(1, 1e6).log1p())
        ).alias("vol_degree_normalized_amount")
    )
    vol_norm_feature_cols.append("vol_degree_normalized_amount")

    # ⑤ vol_outlier_score: z-score 방식의 금액 이상 점수
    #    = (avg_log_amount - global_mean) → 양수면 전체 평균 초과
    vn_exprs.append(
        (pl.col("avg_log_amount") - float(global_avg_log_amount)).alias("vol_outlier_score")
    )
    vol_norm_feature_cols.append("vol_outlier_score")

    df_vn = df_vn.with_columns(vn_exprs)
    for cn in vol_norm_feature_cols:
        df_vn = df_vn.with_columns(
            pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
            .then(0.0).otherwise(pl.col(cn)).alias(cn))
    df_vol_norm_features = df_vn.select(["account_id"] + vol_norm_feature_cols)
else:
    df_vol_norm_features = all_accounts.select("account_id"); vol_norm_feature_cols = []

del df_vn, df_v2_agg_vn; gc.collect()
print(f"  ✅ 거래량 대비 정규화 {len(vol_norm_feature_cols)}개: {vol_norm_feature_cols}")

# =============================================================
# ★ NEW: 2-9. 시간 윈도우별 행동 변화 피처 4개
# =============================================================
# 핵심 아이디어: 미탐의 71%가 테스트 첫 3일 집중
#   → train 말미(~recent window)에서의 행동 변화를 포착
print("  📊 ★ 시간 윈도우별 행동 변화 피처 (신규)...")
win_shift_feature_cols = []

# train 마지막 7일 vs 나머지 기간 비교
last_7d_cutoff = train_cutoff - pl.duration(days=7)
df_raw_recent = df_raw_train.filter(pl.col("Timestamp") >= last_7d_cutoff)
df_raw_older  = df_raw_train.filter(pl.col("Timestamp") < last_7d_cutoff)

recent_cnt = df_raw_recent.select(
    pl.col("from_acc").cast(pl.Utf8).alias("account_id")
).group_by("account_id").len().rename({"len": "ws_recent_cnt"})
older_cnt = df_raw_older.select(
    pl.col("from_acc").cast(pl.Utf8).alias("account_id")
).group_by("account_id").len().rename({"len": "ws_older_cnt"})

df_ws = all_accounts.select("account_id").join(recent_cnt, on="account_id", how="left").join(older_cnt, on="account_id", how="left").fill_null(0.0)

ws_exprs = []
# ① ws_recent_surge: 최근 7일 / 이전 기간 (정규화) → 갑작스러운 증가
ws_exprs.append(
    (pl.col("ws_recent_cnt") / (pl.col("ws_older_cnt") / 30.0 + 1)).alias("ws_recent_surge")
)
win_shift_feature_cols.append("ws_recent_surge")

# ② ws_recent_only: 최근 7일에만 거래가 있는 계좌 (0/1)
#    → 갑자기 나타난 계좌
ws_exprs.append(
    ((pl.col("ws_recent_cnt") > 0) & (pl.col("ws_older_cnt") == 0)).cast(pl.Float64).alias("ws_recent_only")
)
win_shift_feature_cols.append("ws_recent_only")

df_ws = df_ws.with_columns(ws_exprs)

# ③ 최근 7일 고유 파트너 급증
recent_partners = df_raw_recent.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner")
]).group_by("account_id").agg(pl.col("partner").n_unique().alias("ws_recent_partners"))
older_partners = df_raw_older.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner")
]).group_by("account_id").agg(pl.col("partner").n_unique().alias("ws_older_partners"))

df_ws = df_ws.join(recent_partners, on="account_id", how="left").join(older_partners, on="account_id", how="left").fill_null(0.0)

# ④ ws_partner_explosion: 최근 7일 파트너가 이전 대비 급증
df_ws = df_ws.with_columns(
    (pl.col("ws_recent_partners") / (pl.col("ws_older_partners") + 1)).alias("ws_partner_explosion")
)
win_shift_feature_cols.append("ws_partner_explosion")

for cn in win_shift_feature_cols:
    df_ws = df_ws.with_columns(
        pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
        .then(0.0).otherwise(pl.col(cn)).alias(cn))

df_win_shift_features = df_ws.select(["account_id"] + win_shift_feature_cols)
del df_raw_recent, df_raw_older, df_ws, recent_cnt, older_cnt
del recent_partners, older_partners; gc.collect()
print(f"  ✅ 시간 윈도우 변화 {len(win_shift_feature_cols)}개: {win_shift_feature_cols}")

# =============================================================
# ★ NEW: 2-10. 입금/출금 독립 피처 6개
# =============================================================
# 핵심 아이디어: V2에 sum_in_1h/sum_out_1h는 있지만
#   AML 점수/조건부 피처에서 in/out 독립적 분석이 부족
print("  📊 ★ 입금/출금 독립 피처 (신규)...")
inout_feature_cols = []

# 출금(from) 집계
df_out_tx = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col(amount_col).cast(pl.Float64).alias("amount") if amount_col else pl.lit(1.0).alias("amount")
]).filter(pl.col("amount") > 0)

# 입금(to) 집계
df_in_tx = df_raw_train.select([
    pl.col("to_acc").cast(pl.Utf8).alias("account_id"),
    pl.col(amount_col).cast(pl.Float64).alias("amount") if amount_col else pl.lit(1.0).alias("amount")
]).filter(pl.col("amount") > 0)

out_stats = df_out_tx.group_by("account_id").agg([
    pl.col("amount").sum().alias("inout_total_out_amount"),
    pl.col("amount").mean().alias("inout_avg_out_amount"),
    pl.col("amount").count().alias("inout_out_cnt"),
])
in_stats = df_in_tx.group_by("account_id").agg([
    pl.col("amount").sum().alias("inout_total_in_amount"),
    pl.col("amount").mean().alias("inout_avg_in_amount"),
    pl.col("amount").count().alias("inout_in_cnt"),
])
df_inout = all_accounts.select("account_id").join(out_stats, on="account_id", how="left").join(in_stats, on="account_id", how="left").fill_null(0.0)

inout_exprs = []
# ① inout_out_amount_ratio: 출금액 / (입금액 + 출금액) → 1이면 순수 송금자
inout_exprs.append(
    (pl.col("inout_total_out_amount") /
     (pl.col("inout_total_out_amount") + pl.col("inout_total_in_amount") + 1)
    ).alias("inout_out_amount_ratio")
)
inout_feature_cols.append("inout_out_amount_ratio")

# ② inout_avg_out_vs_in: 평균 출금액 / 평균 입금액 → 1 초과면 출금이 더 큰 계좌
inout_exprs.append(
    (pl.col("inout_avg_out_amount") / (pl.col("inout_avg_in_amount") + 1)
    ).alias("inout_avg_out_vs_in")
)
inout_feature_cols.append("inout_avg_out_vs_in")

# ③ inout_cnt_imbalance: (출금 건수 - 입금 건수) / 전체 건수
#    → 양수면 보내는 쪽이 훨씬 많은 계좌 (smurf 패턴)
inout_exprs.append(
    ((pl.col("inout_out_cnt") - pl.col("inout_in_cnt")) /
     (pl.col("inout_out_cnt") + pl.col("inout_in_cnt") + 1)
    ).alias("inout_cnt_imbalance")
)
inout_feature_cols.append("inout_cnt_imbalance")

# ④ inout_fan_out_score: 출금 건수 × 고유 파트너 다양성 → 여러 곳에 분산 송금
df_inout = df_inout.join(df_graph_feats.select(["account_id", "graph_partner_diversity", "graph_unique_out_partners"]), on="account_id", how="left").fill_null(0.0)
inout_exprs.append(
    (pl.col("inout_out_cnt").log1p() * pl.col("graph_partner_diversity")
    ).alias("inout_fan_out_score")
)
inout_feature_cols.append("inout_fan_out_score")

# ⑤ inout_in_concentration: 입금이 소수에서만 들어옴 (mule 패턴)
#    = avg_in_amount × (1 - partner_diversity) → 소수 소스에서 집중 수취
inout_exprs.append(
    (pl.col("inout_avg_in_amount").log1p() * (1.0 - pl.col("graph_partner_diversity").clip(0, 1))
    ).alias("inout_in_concentration")
)
inout_feature_cols.append("inout_in_concentration")

# ⑥ inout_net_flow: log(출금합) - log(입금합) → 순 흐름 방향성
inout_exprs.append(
    (pl.col("inout_total_out_amount").log1p() - pl.col("inout_total_in_amount").log1p()
    ).alias("inout_net_flow")
)
inout_feature_cols.append("inout_net_flow")

df_inout = df_inout.with_columns(inout_exprs)
for cn in inout_feature_cols:
    df_inout = df_inout.with_columns(
        pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
        .then(0.0).otherwise(pl.col(cn)).alias(cn))
df_inout_features = df_inout.select(["account_id"] + inout_feature_cols)
del df_out_tx, df_in_tx, out_stats, in_stats, df_inout; gc.collect()
print(f"  ✅ 입금/출금 독립 {len(inout_feature_cols)}개: {inout_feature_cols}")
log_memory("전체 노드 피처 완료")

# =============================================================
# 3. GNN 엣지/노드 구성 — 강화된 입력
# =============================================================
print("\n🔗 [Step 3] GNN 엣지/노드 구성 (Enriched Input)...")

df_raw_reload = pl.read_parquet(raw_path)
if df_raw_reload["Timestamp"].dtype == pl.Utf8:
    df_raw_reload = df_raw_reload.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_raw_train_edges = df_raw_reload.filter(pl.col("Timestamp")<train_cutoff); del df_raw_reload; gc.collect()

edge_amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train_edges.columns: edge_amount_col = cand; break

df_edges = df_raw_train_edges.select(
    ["from_acc","to_acc","Timestamp"] +
    ([edge_amount_col] if edge_amount_col else []) +
    (["Payment Format"] if "Payment Format" in df_raw_train_edges.columns else []) +
    (["Is Laundering"] if "Is Laundering" in df_raw_train_edges.columns else [])
).with_columns([pl.col("from_acc").cast(pl.Utf8), pl.col("to_acc").cast(pl.Utf8)])

df_edges = df_edges.join(all_accounts.rename({"account_id":"from_acc","node_id":"src_id"}),on="from_acc",how="left")
df_edges = df_edges.join(all_accounts.rename({"account_id":"to_acc","node_id":"dst_id"}),on="to_acc",how="left")

ts = df_edges["Timestamp"]; mn_ts=ts.min(); mx_ts=ts.max(); ts_rng=(mx_ts-mn_ts).total_seconds()
etn = ((ts-mn_ts).dt.total_microseconds().cast(pl.Float64)/1e6/max(ts_rng,1.0)).to_numpy().astype(np.float32)
hrs = df_edges["Timestamp"].dt.hour().to_numpy().astype(np.float32)
dow = df_edges["Timestamp"].dt.weekday().to_numpy().astype(np.float32)
time_feats = np.stack([etn, np.sin(2*np.pi*hrs/24), np.cos(2*np.pi*hrs/24),
                        np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7)], axis=1)

if edge_amount_col:
    amounts_raw = df_edges[edge_amount_col].cast(pl.Float64).fill_null(0.0).to_numpy().astype(np.float32)
    log_amounts = np.log1p(np.maximum(amounts_raw, 0)).astype(np.float32)
    log_amounts = log_amounts / (np.max(log_amounts) + 1e-8)
    round_flag = ((amounts_raw % 1000 == 0) | (amounts_raw % 5000 == 0)).astype(np.float32)
    global_mean_amt = np.mean(amounts_raw[amounts_raw > 0]) if np.any(amounts_raw > 0) else 1.0
    relative_amt = np.log1p(amounts_raw / (global_mean_amt + 1e-8)).astype(np.float32)
    relative_amt = relative_amt / (np.max(relative_amt) + 1e-8)
    amount_feats = np.stack([log_amounts, round_flag, relative_amt], axis=1)
else:
    amount_feats = np.zeros((len(df_edges), 3), dtype=np.float32)

if "Payment Format" in df_edges.columns:
    pf_series = df_edges["Payment Format"].cast(pl.Utf8).fill_null("unknown")
    is_wire = pf_series.str.contains("(?i)wire|swift|cheque").to_numpy().astype(np.float32)
    is_internal = pf_series.str.contains("(?i)ach|internal|reinvestment").to_numpy().astype(np.float32)
    type_feats = np.stack([is_wire, is_internal], axis=1)
else:
    type_feats = np.zeros((len(df_edges), 2), dtype=np.float32)

edge_features_all = np.concatenate([time_feats, amount_feats, type_feats], axis=1)
EDGE_RAW_DIM = edge_features_all.shape[1]
print(f"  📊 Edge 피처: {EDGE_RAW_DIM}개")

edge_index_train = torch.tensor(df_edges.select(["src_id","dst_id"]).to_numpy().T, dtype=torch.long)
edge_attr_train = torch.tensor(edge_features_all, dtype=torch.float32)
print(f"📊 Train 엣지: {edge_index_train.shape[1]:,}")

del df_raw_train, df_raw_train_edges, df_edges, ts, hrs, dow, etn
del time_feats, amount_feats, type_feats, edge_features_all; gc.collect()

# ─── 노드 피처: V2(38) + 그래프(13) + 조건부(10) + AML(5) + 규칙성(7) + 금액(8) = 81개
# ★ v7c.5: GNN 입력은 v7c.2와 동일하게 유지 (신규 피처는 XGBoost에만 투입)
print("  📊 노드 피처 구성 (GNN: v7c.2 동일 81차원)...")
df_v2_node = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])
df_node_all = (all_accounts.join(df_v2_node, on="account_id", how="left")
    .join(df_graph_feats, on="account_id", how="left")
    .join(df_cond_features, on="account_id", how="left")
    .join(df_aml_scores, on="account_id", how="left")
    .join(df_regularity, on="account_id", how="left")
    .join(df_amt_features, on="account_id", how="left")
    .fill_null(0.0))

# GNN 입력은 v7c.2 피처만 (81개) — 신규 피처는 XGBoost 단계에만 추가
gnn_input_cols = (gnn_feature_cols_v2 + graph_feature_cols + cond_feature_cols +
                  aml_score_cols + regularity_cols + amt_feature_cols)

for col in gnn_input_cols:
    if col in df_node_all.columns:
        df_node_all = df_node_all.with_columns(
            pl.when(pl.col(col).is_infinite()|pl.col(col).is_nan()).then(0.0).otherwise(pl.col(col)).alias(col))

X_node = torch.tensor(df_node_all.select(gnn_input_cols).to_numpy(), dtype=torch.float32)
IN_DIM = X_node.shape[1]
print(f"  ✅ GNN 노드 입력: {IN_DIM}차원 (v7c.2 동일)")
print(f"     = V2({len(gnn_feature_cols_v2)}) + 그래프({len(graph_feature_cols)}) + 조건부({len(cond_feature_cols)})")
print(f"       + AML({len(aml_score_cols)}) + 규칙성({len(regularity_cols)}) + 금액({len(amt_feature_cols)})")

target_labels = df_v2_train.filter(pl.col("is_laundering")==1).select("account_id").unique().with_columns(pl.lit(1).alias("label"))
df_labels = all_accounts.join(target_labels,on="account_id",how="left").fill_null(0)
Y_node = torch.tensor(df_labels["label"].to_numpy(), dtype=torch.long)
pos_node_ids = set(df_labels.filter(pl.col("label")==1)["node_id"].to_list())
active_df = pl.DataFrame({"account_id":df_v2_train["account_id"].unique(),"is_active":True})
mask_df = all_accounts.join(active_df,on="account_id",how="left").fill_null(False)
active_node_ids = mask_df.filter(pl.col("is_active"))["node_id"].to_list()
active_pos = [n for n in active_node_ids if n in pos_node_ids]
active_neg = [n for n in active_node_ids if n not in pos_node_ids]
n_neg_gnn = min(len(active_pos)*NEG_NODE_RATIO_GNN, len(active_neg))
np.random.seed(RANDOM_SEED)
sampled_neg = np.random.choice(active_neg,size=n_neg_gnn,replace=False).tolist()
sampled_nodes = set(active_pos+sampled_neg)
train_mask_np = np.zeros(num_nodes, dtype=bool)
for nid in sampled_nodes: train_mask_np[nid]=True
train_mask = torch.tensor(train_mask_np, dtype=torch.bool)
print(f"📊 GNN 노드: 양성 {len(active_pos):,} + 음성 {n_neg_gnn:,}")
del df_v2_node, df_node_all, mask_df, active_df; gc.collect()

graph_data = Data(x=X_node, edge_index=edge_index_train, edge_attr=edge_attr_train, y=Y_node)
graph_data.train_mask = train_mask; graph_data.num_nodes = num_nodes
del X_node, edge_index_train, edge_attr_train, Y_node, train_mask; gc.collect()
log_memory("GNN 데이터 준비 완료")

# =============================================================
# 4-6. Multi-Config GNN 탐색 → 최적 임베딩 선택
# =============================================================
print("\n🧠 [Step 4-6] Multi-Config GNN 탐색...")

class EdgeProjection(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, out_dim), nn.GELU(), nn.Linear(out_dim, out_dim))
    def forward(self, ea): return self.proj(ea)

class TGAT_Flex(nn.Module):
    def __init__(self, in_dim, hidden_dim, edge_raw_dim, edge_dim, n_heads, n_layers=2, dropout=0.2):
        super().__init__()
        self.edge_proj = EdgeProjection(edge_raw_dim, edge_dim)
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.n_layers = n_layers
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_layers)])
        self.convs = nn.ModuleList([
            TransformerConv(hidden_dim, hidden_dim//n_heads, heads=n_heads,
                            edge_dim=edge_dim, dropout=dropout, concat=True)
            for _ in range(n_layers)])
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim//2, 1))
        self.dropout = dropout; self.hidden_dim = hidden_dim

    def forward(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        for i in range(self.n_layers):
            h = h + F.dropout(self.convs[i](self.norms[i](h), edge_index, ee), p=self.dropout, training=self.training)
        return self.classifier(self.norm_out(h)).squeeze(-1)

    @torch.no_grad()
    def get_embedding(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        for i in range(self.n_layers):
            h = h + self.convs[i](self.norms[i](h), edge_index, ee)
        return self.norm_out(h)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"📌 디바이스: {device}")
pw = (len(sampled_nodes)-len(active_pos))/max(len(active_pos),1)
print(f"📊 pos_weight: {pw:.1f}")

def make_lr(opt, ep, n_ep):
    if ep < WARMUP_EPOCHS: lr = 1e-6+(BASE_LR-1e-6)*(ep/WARMUP_EPOCHS)
    else: p=(ep-WARMUP_EPOCHS)/max(n_ep-WARMUP_EPOCHS,1); lr=1e-5+(BASE_LR-1e-5)*0.5*(1+np.cos(np.pi*p))
    for pg in opt.param_groups: pg['lr']=lr
    return lr

GNN_CONFIGS = [
    {"label":"h64_L2",  "hidden_dim":64,  "n_layers":2, "n_heads":4, "edge_dim":16},
    {"label":"h128_L2", "hidden_dim":128, "n_layers":2, "n_heads":4, "edge_dim":32},
    {"label":"h64_L3",  "hidden_dim":64,  "n_layers":3, "n_heads":4, "edge_dim":16},
    {"label":"h128_L3", "hidden_dim":128, "n_layers":3, "n_heads":4, "edge_dim":32},
]

xgb_exclude_cols=["account_id","time_group","is_laundering","mode_format","currency_mode","date"]
xgb_v2_cols=[c for c in df_v2.columns if c not in xgb_exclude_cols]
apr_train = df_v2_train.filter(pl.col("is_laundering")==1).height / max(df_v2_train.height,1)
spw = max((1-apr_train)/apr_train, 1.0) if apr_train > 0 else 20.0

# ★ 신규 피처들을 base_feature_cols에 포함
base_feature_cols = (xgb_v2_cols + graph_feature_cols + cond_feature_cols + aml_score_cols +
                     cb_feature_cols + vol_norm_feature_cols +
                     win_shift_feature_cols + inout_feature_cols)

def build_eval_df(db, df_emb_local, emb_col_names):
    feature_cols = base_feature_cols + emb_col_names
    df = (db.join(df_graph_feats,on="account_id",how="left")
          .join(df_cond_features,on="account_id",how="left")
          .join(df_aml_scores,on="account_id",how="left")
          .join(df_cb_features,on="account_id",how="left")            # ★
          .join(df_vol_norm_features,on="account_id",how="left")      # ★
          .join(df_win_shift_features,on="account_id",how="left")     # ★
          .join(df_inout_features,on="account_id",how="left")         # ★
          .join(df_emb_local,on="account_id",how="left")
          .fill_null(0.0).fill_nan(0.0))
    for cn in feature_cols:
        if cn in df.columns:
            df = df.with_columns(pl.when(pl.col(cn).is_infinite()).then(0.0).otherwise(pl.col(cn)).alias(cn))
    return df, feature_cols

def quick_xgb_eval(df_emb_local, emb_col_names, label):
    df_tr, fcols = build_eval_df(df_v2.filter(pl.col("time_group")<train_cutoff), df_emb_local, emb_col_names)
    df_tr_pos=df_tr.filter(pl.col("is_laundering")==1); df_tr_neg=df_tr.filter(pl.col("is_laundering")==0)
    n_neg=min(len(df_tr_pos)*NEG_ROW_RATIO_XGB,len(df_tr_neg))
    df_tr_s=pl.concat([df_tr_pos,df_tr_neg.sample(n=n_neg,seed=RANDOM_SEED)]).sample(fraction=1.0,seed=RANDOM_SEED)
    X_tr=df_tr_s.select(fcols).to_pandas(); y_tr=df_tr_s["is_laundering"].to_numpy()
    del df_tr,df_tr_pos,df_tr_neg,df_tr_s; gc.collect()
    df_vl, _ = build_eval_df(df_v2.filter((pl.col("time_group")>=train_cutoff)&(pl.col("time_group")<val_cutoff)), df_emb_local, emb_col_names)
    X_vl=df_vl.select(fcols).to_pandas(); y_vl=df_vl["is_laundering"].to_numpy(); del df_vl; gc.collect()
    df_te, _ = build_eval_df(df_v2.filter(pl.col("time_group")>=val_cutoff), df_emb_local, emb_col_names)
    X_te=df_te.select(fcols).to_pandas(); y_te=df_te["is_laundering"].to_numpy(); del df_te; gc.collect()
    qp={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
        "random_state":42,"max_depth":8,"learning_rate":0.03,"scale_pos_weight":spw,
        "subsample":0.8,"colsample_bytree":0.7,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
    m=xgb.XGBClassifier(**qp,n_estimators=1500,early_stopping_rounds=100)
    m.fit(X_tr,y_tr,eval_set=[(X_vl,y_vl)],verbose=0)
    q_val=m.best_score; q_test=average_precision_score(y_te,m.predict_proba(X_te)[:,1])
    del m,X_tr,X_vl,X_te,y_tr,y_vl,y_te; gc.collect()
    return q_val, q_test

print(f"\n📊 {len(GNN_CONFIGS)}개 GNN 설정 탐색...")
print(f"  {'#':>2s} {'Label':<12s} | {'hidden':>6s} {'layers':>6s} {'edim':>4s} | {'Loss':>8s} {'Params':>8s} | {'QVal':>8s} {'QTest':>8s}")
print("  "+"-"*80)

gnn_results = []
for gi, gcfg in enumerate(GNN_CONFIGS):
    gl = gcfg["label"]; hd=gcfg["hidden_dim"]; nl=gcfg["n_layers"]
    nh=gcfg["n_heads"]; ed=gcfg["edge_dim"]
    train_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                                   input_nodes=graph_data.train_mask, shuffle=True, num_workers=4)
    full_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                                  input_nodes=None, shuffle=False, num_workers=4)
    model = TGAT_Flex(in_dim=IN_DIM, hidden_dim=hd, edge_raw_dim=EDGE_RAW_DIM,
                      edge_dim=ed, n_heads=nh, n_layers=nl, dropout=0.2).to(device)
    np_count = sum(p.numel() for p in model.parameters())
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw],device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6, weight_decay=1e-4)
    model.train(); bl=float('inf'); pc2=0; bs=None
    for epoch in range(N_EPOCHS_GNN):
        clr=make_lr(optimizer,epoch,N_EPOCHS_GNN); tl=0.0; nb2=0
        for batch in tqdm(train_loader,desc=f"[{gl}] Ep{epoch+1}",leave=False):
            batch=batch.to(device); optimizer.zero_grad()
            bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
            out=model(batch.x,batch.edge_index,bea)
            loss=criterion(out[:batch.batch_size],batch.y[:batch.batch_size].float())
            if torch.isnan(loss) or torch.isinf(loss): optimizer.zero_grad(); continue
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=0.5)
            optimizer.step(); tl+=loss.item(); nb2+=1
        al=tl/max(nb2,1)
        if (epoch+1)%10==0 or epoch==0: print(f"    [{gl}] Ep{epoch+1:2d} Loss={al:.4f}")
        if al<bl-0.0005: bl=al; pc2=0; bs={k:v.cpu().clone() for k,v in model.state_dict().items()}
        else: pc2+=1
        if pc2>=PATIENCE: break
    if bs: model.load_state_dict({k:v.to(device) for k,v in bs.items()}); del bs
    model.eval(); ae=[]
    with torch.no_grad():
        for batch in tqdm(full_loader,desc=f"[{gl}] Embed",leave=False):
            batch=batch.to(device)
            bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
            emb=model.get_embedding(batch.x,batch.edge_index,bea)
            ae.append(emb[:batch.batch_size].cpu())
    femb=torch.cat(ae,dim=0).numpy()
    del model,ae,train_loader,full_loader; torch.cuda.empty_cache(); gc.collect()
    ecn=[f"tgat_emb_{i}" for i in range(hd)]
    df_emb=pl.concat([all_accounts.select("account_id"),pl.DataFrame(femb,schema=ecn)],how="horizontal")
    qv,qt = quick_xgb_eval(df_emb, ecn, gl)
    print(f"  {gi+1:2d} {gl:<12s} | {hd:6d} {nl:6d} {ed:4d} | {bl:8.4f} {np_count:8,} | {qv:8.5f} {qt:8.4f}")
    gnn_results.append({"label":gl,"hidden_dim":hd,"n_layers":nl,"loss":bl,"params":np_count,
                         "q_val":qv,"q_test":qt,"embedding":femb,"ecn":ecn,"df_emb":df_emb})
    log_memory(f"{gl}")

best_gnn = max(gnn_results, key=lambda x: x["q_val"])
print(f"\n  🏆 Best GNN: {best_gnn['label']} (QVal={best_gnn['q_val']:.5f} QTest={best_gnn['q_test']:.4f})")
for r in gnn_results:
    if r is not best_gnn: del r["embedding"], r["df_emb"]
gc.collect()

emb_cols = best_gnn["ecn"]
df_tgat = best_gnn["df_emb"]
FINAL_HIDDEN_DIM = best_gnn["hidden_dim"]

print(f"\n🔗 Interaction 피처 ({best_gnn['label']})...")
df_tgat_with_pr=df_tgat.join(df_graph_feats.select(["account_id","graph_pagerank"]),on="account_id",how="left").fill_null(0.0)
emb_np=df_tgat_with_pr.select(emb_cols).to_numpy(); emb_var=np.var(emb_np,axis=0)
top_idx=np.argsort(emb_var)[::-1][:N_INTERACTION_DIMS]; top_ec=[emb_cols[i] for i in top_idx]
ie_l=[]; ic_l=[]
for cn in top_ec: nn2=f"interact_pr_x_{cn}"; ie_l.append((pl.col(cn)*pl.col("graph_pagerank")*1e6).alias(nn2)); ic_l.append(nn2)
ie_l.append((sum(pl.col(c)**2 for c in emb_cols).sqrt()*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_norm")); ic_l.append("interact_pr_x_emb_norm")
ie_l.append((sum(pl.col(c) for c in top_ec)*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_topsum")); ic_l.append("interact_pr_x_emb_topsum")
interaction_cols=ic_l
df_interactions=df_tgat_with_pr.with_columns(ie_l).select(["account_id"]+ic_l)
for c in ic_l: df_interactions=df_interactions.with_columns(pl.when(pl.col(c).is_infinite()|pl.col(c).is_nan()).then(0.0).otherwise(pl.col(c)).alias(c))
print(f"  ✅ Interaction {len(ic_l)}개")
del df_tgat_with_pr,emb_np,best_gnn["embedding"]; gc.collect()
del graph_data; torch.cuda.empty_cache(); gc.collect()
log_memory("GNN 탐색 완료")

# =============================================================
# 7. XGBoost 데이터 구성
# =============================================================
print("\n🚀 [Step 7] XGBoost 데이터 구성...")

# ★ v7c.6: 모든 신규 피처를 후보로 포함 — Forward Selection이 유효한 것만 걸러냄
# v7c.5에서 수동으로 4개만 선별했던 것을 FS 자동 선별로 교체
xgb_feature_cols = (xgb_v2_cols + graph_feature_cols + cond_feature_cols + aml_score_cols +
                    cb_feature_cols + vol_norm_feature_cols +
                    win_shift_feature_cols + inout_feature_cols +
                    emb_cols + interaction_cols)
selected_new_cols = cb_feature_cols + vol_norm_feature_cols + win_shift_feature_cols + inout_feature_cols
print(f"📊 XGBoost 총 후보 피처: {len(xgb_feature_cols)}")
print(f"   V2:{len(xgb_v2_cols)} G:{len(graph_feature_cols)} Cond:{len(cond_feature_cols)} AML:{len(aml_score_cols)}")
print(f"   CB:{len(cb_feature_cols)} VolNorm:{len(vol_norm_feature_cols)}")
print(f"   WinShift:{len(win_shift_feature_cols)} InOut:{len(inout_feature_cols)}")
print(f"   Emb:{len(emb_cols)} Inter:{len(interaction_cols)}")
print(f"   ★ Forward Selection이 {len(selected_new_cols)}개 신규 피처 중 유효한 것 선별")

def build_xgb_df(db):
    df=(db.join(df_graph_feats,on="account_id",how="left")
          .join(df_cond_features,on="account_id",how="left")
          .join(df_aml_scores,on="account_id",how="left")
          .join(df_cb_features,on="account_id",how="left")
          .join(df_vol_norm_features,on="account_id",how="left")
          .join(df_win_shift_features,on="account_id",how="left")
          .join(df_inout_features,on="account_id",how="left")
          .join(df_tgat,on="account_id",how="left")
          .join(df_interactions,on="account_id",how="left")
          .fill_null(0.0).fill_nan(0.0))
    for cn in xgb_feature_cols:
        if cn in df.columns: df=df.with_columns(pl.when(pl.col(cn).is_infinite()).then(0.0).otherwise(pl.col(cn)).alias(cn))
    return df

df_tr=build_xgb_df(df_v2.filter(pl.col("time_group")<train_cutoff))
df_tr_pos=df_tr.filter(pl.col("is_laundering")==1); df_tr_neg=df_tr.filter(pl.col("is_laundering")==0)
n_neg=min(len(df_tr_pos)*NEG_ROW_RATIO_XGB,len(df_tr_neg))
df_tr_s=pl.concat([df_tr_pos,df_tr_neg.sample(n=n_neg,seed=RANDOM_SEED)]).sample(fraction=1.0,seed=RANDOM_SEED)
X_train=df_tr_s.select(xgb_feature_cols).to_pandas(); y_train=df_tr_s["is_laundering"].to_numpy()
print(f"📊 Train: {len(df_tr):,} → {len(df_tr_s):,} ({y_train.mean()*100:.2f}%)")
del df_tr,df_tr_pos,df_tr_neg,df_tr_s; gc.collect()
df_vl=build_xgb_df(df_v2.filter((pl.col("time_group")>=train_cutoff)&(pl.col("time_group")<val_cutoff)))
X_val=df_vl.select(xgb_feature_cols).to_pandas(); y_val=df_vl["is_laundering"].to_numpy(); print(f"📊 Val: {len(X_val):,}"); del df_vl; gc.collect()
df_te=build_xgb_df(df_v2.filter(pl.col("time_group")>=val_cutoff))
df_test_meta=df_te.select(["account_id","time_group","is_laundering"])
X_test=df_te.select(xgb_feature_cols).to_pandas(); y_test=df_te["is_laundering"].to_numpy(); print(f"📊 Test: {len(X_test):,}")
del df_te,df_v2,df_tgat,df_graph_feats,df_cond_features,df_aml_scores
del df_cb_features,df_vol_norm_features,df_win_shift_features,df_inout_features
del df_interactions,df_regularity,df_amt_features; gc.collect()
log_memory("XGBoost 준비")

# =============================================================
# 8. XGBoost 3-Stage: Full → Forward Selection → HP Grid
# =============================================================
# ★ v7c.6 핵심 변경: Stage 1 importance 기반 top-K 선택의 한계 해결
#   문제: 상관된 피처끼리 importance를 나눠 가져서 유용한 피처가 순위 밖으로 밀림
#   해결: Stage 2를 Forward Selection으로 교체
#     → 피처를 하나씩 추가하며 Val AUCPR이 실제로 오를 때만 채택
#     → 중복/노이즈 피처는 자동으로 탈락
print("\n🔥 [Step 8] 3-Stage XGBoost (Full → Forward Selection → HP Grid)...")
apr=y_train.sum()/len(y_train); spw=max((1-apr)/apr,1.0)

# ── Stage 1: 전체 피처로 학습 → importance 순위 산출 ──────────────────
print("\n── Stage 1: 전체 피처 학습 (importance 순위 산출) ──")
s1p={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
     "random_state":42,"max_depth":8,"learning_rate":0.03,"scale_pos_weight":spw,
     "subsample":0.8,"colsample_bytree":0.7,"min_child_weight":5,
     "gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
m_s1=xgb.XGBClassifier(**s1p,n_estimators=1500,early_stopping_rounds=100)
m_s1.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=100)
s1_val=m_s1.best_score
s1_test=average_precision_score(y_test,m_s1.predict_proba(X_test)[:,1])
print(f"\n  Stage1: Val={s1_val:.5f} Test={s1_test:.4f}")
imp_s1=m_s1.feature_importances_
# importance 순위 정렬 (Stage 2 Forward Selection의 탐색 순서로 사용)
feat_imp=sorted(zip(xgb_feature_cols,imp_s1),key=lambda x:x[1],reverse=True)
del m_s1; gc.collect()

# ── Stage 2: Forward Selection ─────────────────────────────────────────
# importance 상위 순서대로 피처를 하나씩 추가하며
# Val AUCPR이 오를 때만 채택, 연속 N_PATIENCE_FS번 개선 없으면 조기 종료
print("\n── Stage 2: Forward Selection ──")
print("  피처를 importance 순서대로 하나씩 추가 → Val 개선 시 채택")

N_PATIENCE_FS = 8   # 연속 N번 개선 없으면 탐색 종료
MIN_DELTA_FS  = 5e-5  # 최소 개선량 (노이즈 수준 변동 무시)
MAX_FEATURES_FS = 100  # 탐색할 최대 피처 수 (전체 중 상위 100개만 탐색)

fs_hp={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
       "random_state":42,"max_depth":9,"learning_rate":0.02,"scale_pos_weight":spw,
       "subsample":0.8,"colsample_bytree":0.8,"min_child_weight":5,
       "gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}

selected_fs = []          # 채택된 피처 누적
best_fs_val = -1.0        # 현재까지 best Val AUCPR
no_improve  = 0           # 연속 미개선 횟수
fs_log      = []          # 로그 기록

candidates = [f for f,_ in feat_imp[:MAX_FEATURES_FS]]

print(f"  {'#':>3s} {'Feature':<45s} | {'Val':>9s} {'Δ':>8s} {'채택':>4s}")
print("  " + "-" * 75)

for fi, feat in enumerate(candidates):
    probe = selected_fs + [feat]
    m_fs = xgb.XGBClassifier(**fs_hp, n_estimators=800, early_stopping_rounds=50)
    m_fs.fit(X_train[probe], y_train, eval_set=[(X_val[probe], y_val)], verbose=False)
    v = m_fs.best_score
    delta = v - best_fs_val
    adopted = delta > MIN_DELTA_FS

    tag = "✅" if adopted else "  "
    print(f"  {fi+1:3d} {feat:<45s} | {v:9.5f} {delta:+8.5f} {tag}")
    fs_log.append({"feat":feat,"val":v,"delta":delta,"adopted":adopted})

    if adopted:
        selected_fs.append(feat)
        best_fs_val = v
        no_improve = 0
    else:
        no_improve += 1

    del m_fs; gc.collect()

    if no_improve >= N_PATIENCE_FS:
        print(f"\n  ⏹ 연속 {N_PATIENCE_FS}회 미개선 → 조기 종료 (탐색 {fi+1}/{len(candidates)})")
        break

print(f"\n  ✅ Forward Selection 완료: {len(selected_fs)}개 채택")
print(f"  채택 피처: {selected_fs}")
print(f"  Best Val AUCPR: {best_fs_val:.5f}  (Stage1: {s1_val:.5f}  Δ={best_fs_val-s1_val:+.5f})")

# ── Stage 3: HP Grid (Forward Selection 채택 피처 고정) ───────────────
print("\n── Stage 3: HP Grid (Forward Selection 피처 고정) ──")
# Forward Selection이 고른 피처로 HP만 탐색
bp3={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
     "random_state":42,"scale_pos_weight":spw}
hp_grid=[
    {"label":"d8_lr03_c07","hp":{"max_depth":8,"learning_rate":0.03,"colsample_bytree":0.7,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":1500}},
    {"label":"d8_lr02_c08","hp":{"max_depth":8,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d9_lr02_c08","hp":{"max_depth":9,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d9_lr02_c09","hp":{"max_depth":9,"learning_rate":0.02,"colsample_bytree":0.9,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d10_lr02_c08","hp":{"max_depth":10,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d10_lr01_c08","hp":{"max_depth":10,"learning_rate":0.01,"colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":4000}},
    {"label":"d9_reg","hp":{"max_depth":9,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.75,"min_child_weight":8,"gamma":0.2,"reg_alpha":0.5,"reg_lambda":2.0,"n_estimators":2500}},
]

print(f"\n📊 {len(hp_grid)}개 HP 조합 탐색 (피처 {len(selected_fs)}개 고정)...")
print(f"  {'#':>2s} {'Label':<20s} | {'d':>2s} {'LR':>5s} {'col':>4s} | {'Val':>9s} {'it':>5s}")
print("  "+"-"*55)
best_s2={"val":-1}
for i,cfg in enumerate(hp_grid):
    hp=cfg["hp"].copy(); ne=hp.pop("n_estimators"); sf=selected_fs
    m=xgb.XGBClassifier(**{**bp3,**hp},n_estimators=ne,early_stopping_rounds=100)
    m.fit(X_train[sf],y_train,eval_set=[(X_val[sf],y_val)],verbose=False)
    va=m.best_score; bi=m.best_iteration
    print(f"  {i+1:2d} {cfg['label']:<20s} | {hp.get('max_depth',8):2d} {hp.get('learning_rate',0.03):5.3f} {hp.get('colsample_bytree',0.7):4.2f} | {va:9.5f} {bi:5d}")
    if va>best_s2["val"]:
        if best_s2.get("model"): del best_s2["model"]
        best_s2={"label":cfg["label"],"features":sf,"val":va,"iter":bi,"model":m,"hp":hp}
    else: del m
    gc.collect()

y_prob=best_s2["model"].predict_proba(X_test[best_s2["features"]])[:,1]
best_features=best_s2["features"]
print(f"\n  🏆 Best: {best_s2['label']} ({len(best_features)}개) Val={best_s2['val']:.5f} iter={best_s2['iter']}")

imp=best_s2["model"].feature_importances_; tidx=np.argsort(imp)[::-1][:30]
print(f"\n🔍 Feature Importance Top 30")
for i,idx in enumerate(tidx):
    c=best_features[idx]; t=""
    if c.startswith("tgat_emb_"): t=" [TGAT]"
    elif c.startswith("graph_"): t=" [GRAPH]"
    elif c.startswith("cond_"): t=" [COND]"
    elif c.startswith("score_"): t=" [SCORE]"
    elif c.startswith("interact_"): t=" [INTER]"
    elif c.startswith("cb_"): t=" [★CB]"
    elif c.startswith("vol_"): t=" [★VOL]"
    elif c.startswith("ws_"): t=" [★WIN]"
    elif c.startswith("inout_"): t=" [★INOUT]"
    print(f"  {i+1:2d}. {c:<45s}: {imp[idx]:.4f}{t}")

ti=imp.sum()
def group_imp(prefix): return sum(imp[i] for i,c in enumerate(best_features) if c.startswith(prefix))
print(f"\n📊 그룹별 중요도:")
print(f"   V2        : {group_imp('') / ti * 100:.1f}% (기존)")
print(f"   Graph     : {group_imp('graph_') / ti * 100:.1f}%")
print(f"   TGAT      : {group_imp('tgat_emb_') / ti * 100:.1f}%")
print(f"   Interact  : {group_imp('interact_') / ti * 100:.1f}%")
print(f"   ★ CB      : {group_imp('cb_') / ti * 100:.1f}%")
print(f"   ★ VolNorm : {group_imp('vol_') / ti * 100:.1f}%")
print(f"   ★ WinShift: {group_imp('ws_') / ti * 100:.1f}%")
print(f"   ★ InOut   : {group_imp('inout_') / ti * 100:.1f}%")
del X_train,X_val; gc.collect()

# =============================================================
# 9. 최종 리포트
# =============================================================
print("\n"+"="*60); print("🏆 [TGAT v7c.6] 최종 리포트"); print("="*60)
auprc=average_precision_score(y_test,y_prob); print(f"\n  AUPRC: {auprc:.4f}")
best_f1=0; best_thresh=0
for th in np.arange(0.05,0.95,0.01):
    f=f1_score(y_test,(y_prob>=th).astype(int),zero_division=0)
    if f>best_f1: best_f1=f; best_thresh=th
y_pred=(y_prob>=best_thresh).astype(int)
prec=precision_score(y_test,y_pred,zero_division=0); rec=recall_score(y_test,y_pred,zero_division=0)
print(f"\n📌 최적: T={best_thresh:.2f} F1={best_f1:.4f} P={prec:.4f} R={rec:.4f}")
print(f"\n📌 임계값별:")
for t in [0.1,0.3,0.5,0.7,0.8,0.9]:
    yp=(y_prob>=t).astype(int)
    print(f"  T={t:.1f} P={precision_score(y_test,yp,zero_division=0):.4f} R={recall_score(y_test,yp,zero_division=0):.4f} F1={f1_score(y_test,yp,zero_division=0):.4f}")

df_res=df_test_meta.with_columns([pl.col("time_group").dt.date().alias("date"),pl.Series("pred_prob",y_prob)])
df_dist=df_res.sort("pred_prob",descending=True).unique(subset=["account_id"],maintain_order=True)
print(f"\n📌 Top-K:")
for k in [100,500,1000,5000,10000]:
    ck=min(k,len(df_dist)); hits=df_dist.head(ck)["is_laundering"].sum()
    print(f"  Top-{k:5d}: {hits:5d}명 ({hits/ck*100:.2f}%)")
print(f"\n📌 일별 Top-100:")
for d in df_dist["date"].unique(maintain_order=True).sort(descending=True):
    hits=df_dist.filter(pl.col("date")==d).head(100)["is_laundering"].sum(); print(f"  {d}: {hits}")

tn,fp,fn,tp=confusion_matrix(y_test,y_pred).ravel()
print(f"\n📌 v7c.5→v7c.6: AUPRC {auprc:.4f} | F1 {best_f1:.4f}")
print(f"  TP:{tp:,} FP:{fp:,} FN:{fn:,} TN:{tn:,}")
print(f"  Precision:{prec:.4f} Recall:{rec:.4f}")

print(f"\n📌 GNN 탐색 결과:")
for r in gnn_results:
    tag = " ★" if r["label"]==best_gnn["label"] else ""
    print(f"  {r['label']:<12s}: QVal={r['q_val']:.5f} QTest={r['q_test']:.4f} Loss={r['loss']:.4f} Params={r['params']:,}{tag}")

print(f"\n📌 3-Stage 결과:")
print(f"  S1 전체({len(xgb_feature_cols)}개): Val={s1_val:.5f} Test={s1_test:.4f}")
print(f"  S2 Forward Selection: {len(selected_fs)}개 채택, Val={best_fs_val:.5f}")
print(f"  S3 HP Grid ({best_s2['label']}): Val={best_s2['val']:.5f} Test={auprc:.4f}")

print(f"\n✨ v7c.6 변경 요약:")
print(f"  GNN 입력   : 81차원 (v7c.2 동일, v7c.5 유지)")
print(f"  피처 선택  : top-K 일괄 → Forward Selection 자동 선별")
print(f"  채택 피처  : {len(selected_fs)}개 → {selected_fs}")
print(f"  신규 피처 중 FS 채택: {[c for c in selected_fs if c in selected_new_cols]}")

del X_test; gc.collect(); log_memory("최종 완료")
print(f"\n✅ TGAT v7c.6 완료!")

🛡️ [TGAT v7c.6] Forward Selection 피처 선택
   ▸ GNN 입력: v7c.2 동일 81차원 유지
   ▸ XGBoost 피처 선택 방식 개선:
     기존: Stage1 importance top-K 일괄 선택
     변경: Forward Selection — 하나씩 추가, Val AUCPR
           실제 개선 시만 채택 (중복/노이즈 자동 탈락)
   ▸ XGBoost 입력 후보: 모든 신규 피처 포함 (ws_*, inout_*,
     cb_*, vol_*) → FS가 유효한 것만 선별
   ▸ Stage 3: FS 채택 피처 고정 후 HP 그리드 7종

📂 데이터 로딩 중...
⏱️ Train cutoff: 2022-09-09 20:00:00
⏱️ Val   cutoff: 2022-09-14 05:00:00
📊 전체 노드 수: 2,076,999
  💾 [데이터 로드 완료] RAM: 50.78GB | GPU: 0.02GB

📐 [Step 2] 전체 노드 피처 계산...
  📊 그래프 구조 피처...
  📊 PageRank...
  ✅ 그래프 피처 13개
  📊 조건부행동 피처...
  ✅ 조건부행동 10개
  📊 AML 점수...
  ✅ AML 점수 5개
  📊 규칙성 피처...
  ✅ 규칙성 7개
  📊 금액 분포 피처...
  ✅ 금액 분포 8개
  📊 ★ Cross-border 정밀화 피처 (신규)...
  ✅ Cross-border 정밀화 4개: ['cb_new_partner_ratio', 'cb_oneway_new', 'cb_amount_asymmetry', 'cb_burst_cross']
  📊 ★ 거래량 대비 상대 정규화 피처 (신규)...
  ✅ 거래량 대비 정규화 5개: ['vol_amount_per_peer', 'vol_concentrated_amount', 'vol_low_freq_high_amt', 'vol_degree_normalized_amount', 'vol_outlier_score']

    [h64_L2] Ep 1 Loss=1.3807


    [h64_L2] Ep10 Loss=1.0616


    [h64_L2] Ep20 Loss=1.0273


    [h64_L2] Ep30 Loss=1.0198


...

    [h128_L3] Ep10 Loss=1.0440


    [h128_L3] Ep20 Loss=1.0061


    [h128_L3] Ep30 Loss=0.9866


    [h128_L3] Ep40 Loss=0.9740


   4 h128_L3      |    128      3   32 |   0.9740  231,681 |  0.38221   0.4080
  💾 [h128_L3] RAM: 89.86GB | GPU: 0.02GB

  🏆 Best GNN: h128_L2 (QVal=0.38937 QTest=0.4214)

🔗 Interaction 피처 (h128_L2)...
  ✅ Interaction 10개
  💾 [GNN 탐색 완료] RAM: 87.77GB | GPU: 0.02GB

🚀 [Step 7] XGBoost 데이터 구성...
📊 XGBoost 총 후보 피처: 222
   V2:38 G:13 Cond:10 AML:5
   CB:4 VolNorm:5
   WinShift:3 InOut:6
   Emb:128 Inter:10
   ★ Forward Selection이 18개 신규 피처 중 유효한 것 선별
📊 Train: 17,172,747 → 619,437 (4.76%)
📊 Val: 5,790,644
📊 Test: 5,792,913
  💾 [XGBoost 준비] RAM: 108.91GB | GPU: 0.02GB

🔥 [Step 8] 3-Stage XGBoost (Full → Forward Selection → HP Grid)...

── Stage 1: 전체 피처 학습 (importance 순위 산출) ──
[0]	validation_0-aucpr:0.11731
[100]	validation_0-aucpr:0.28729
[200]	validation_0-aucpr:0.37914
[300]	validation_0-aucpr:0.39674
[400]	validation_0-aucpr:0.40658
[500]	validation_0-aucpr:0.41155
[600]	validation_0-aucpr:0.41866
[700]	validation_0-aucpr:0.42170
[800]	validation_0-aucpr:0.42656
[900]	validation_0-aucpr

In [5]:
import polars as pl

df = pl.read_parquet(
    "/home/tracerofjageum/HI-Medium_Master.parquet",
    columns=["Payment Format", "Is Laundering"]
)

result = (
    df.group_by("Payment Format")
    .agg([
        pl.len().alias("count"),
        pl.col("Is Laundering").mean().alias("aml_rate")
    ])
    .sort("count", descending=True)
)

total = len(df)
print(f"총 거래: {total:,}건 | 고유 포맷: {result.height}종\n")
print(f"{'Payment Format':<30} {'건수':>10} {'비율':>7} {'세탁율':>7}")
print("-" * 58)
for row in result.iter_rows(named=True):
    fmt = str(row["Payment Format"]) if row["Payment Format"] else "None"
    cnt = row["count"]
    rate = row["aml_rate"] or 0.0
    print(f"{fmt:<30} {cnt:>10,} {cnt/total*100:>6.2f}% {rate*100:>6.2f}%")

총 거래: 31,898,669건 | 고유 포맷: 7종

Payment Format                         건수      비율     세탁율
----------------------------------------------------------
Cheque                         12,280,121  38.50%   0.02%
Credit Card                     8,778,042  27.52%   0.02%
ACH                             3,868,513  12.13%   0.79%
Cash                            3,217,537  10.09%   0.02%
Reinvestment                    1,945,644   6.10%   0.00%
Wire                            1,119,774   3.51%   0.00%
Bitcoin                           689,038   2.16%   0.04%


In [7]:
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv
from torch_geometric.loader import NeighborLoader
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm
import gc, numpy as np, psutil, os, warnings
warnings.filterwarnings("ignore", category=UserWarning)

def log_memory(tag=""):
    rss = psutil.Process(os.getpid()).memory_info().rss/(1024**3)
    gpu = torch.cuda.memory_allocated()/(1024**3) if torch.cuda.is_available() else 0
    print(f"  💾 [{tag}] RAM: {rss:.2f}GB | GPU: {gpu:.2f}GB")

print("=" * 60)
print("🛡️ [TGAT v7c.10] 튜닝 개선 + 엣지 피처 정밀화")
print("   ▸ GNN: epochs 40→60, early_stop delta 0.0005→0.001")
print("   ▸ GNN: edge_dim 16→32 (h64 병목 해소)")
print("   ▸ GNN: NEG_NODE_RATIO 50→25 (이중 불균형 보정 완화)")
print("   ▸ FS:  n_estimators 800→1500 (피처 평가 수렴 강화)")
print("   ▸ 엣지: 10→17차원 (7종 원-핫+위험도, 금액 분위수 추가)")
print("=" * 60)

# ===================== 설정 =====================
NEG_NODE_RATIO_GNN = 25   # ★ 50→25: 이중 불균형 보정 완화, 더 다양한 음성 패턴 학습
NEG_ROW_RATIO_XGB = 20; RANDOM_SEED = 42
HIDDEN_DIM = 64; EDGE_DIM = 32   # ★ 16→32: 엣지 17차원 → 병목 해소
N_HEADS = 4
N_EPOCHS_GNN = 60             # ★ 40→60: 충분한 수렴 시간 확보
BATCH_SIZE = 2048; NUM_NEIGHBORS = [15, 10]
N_INTERACTION_DIMS = 8
PATIENCE = 10
EARLY_STOP_DELTA = 0.001      # ★ 0.0005→0.001: 실질적 early stopping 작동
BASE_LR = 0.001; WARMUP_EPOCHS = 5

raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"

# =============================================================
# 1. 데이터 로드
# =============================================================
print("\n📂 데이터 로딩 중...")
df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))
if df_raw["Timestamp"].dtype == pl.Utf8:
    df_raw = df_raw.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_v2 = df_v2.sort("time_group")
total_count = len(df_v2)
train_cutoff = df_v2["time_group"][int(total_count*0.6)]
val_cutoff = df_v2["time_group"][int(total_count*0.8)]
print(f"⏱️ Train cutoff: {train_cutoff}\n⏱️ Val   cutoff: {val_cutoff}")

all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")
num_nodes = len(all_accounts); print(f"📊 전체 노드 수: {num_nodes:,}")
log_memory("데이터 로드 완료")

# =============================================================
# 2. 모든 노드 피처 계산 (GNN 입력용 + XGBoost 겸용)
# =============================================================
print("\n📐 [Step 2] 전체 노드 피처 계산...")
df_raw_train = df_raw.filter(pl.col("Timestamp") < train_cutoff)

# ─── 2-1. 그래프 구조 피처 13개 ───
print("  📊 그래프 구조 피처...")
df_from = df_raw_train.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id"))
df_to = df_raw_train.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
out_degree = df_from.group_by("account_id").len().rename({"len":"graph_out_degree"})
in_degree = df_to.group_by("account_id").len().rename({"len":"graph_in_degree"})
unique_out = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("to_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_out_partners"))
unique_in = df_raw_train.select([pl.col("to_acc").cast(pl.Utf8).alias("account_id"),pl.col("from_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_in_partners"))
esf = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("a"),pl.col("to_acc").cast(pl.Utf8).alias("b")]).unique()
est = esf.select([pl.col("b").alias("a"),pl.col("a").alias("b")])
bidir = esf.join(est, on=["a","b"], how="inner")
bidir_count = bidir.select(pl.col("a").alias("account_id")).group_by("account_id").len().rename({"len":"graph_bidir_count"})
del esf, est, bidir; gc.collect()
time_features = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("Timestamp")]).group_by("account_id").agg([
    pl.col("Timestamp").sort().diff().dt.total_seconds().mean().alias("graph_avg_tx_interval"),
    pl.col("Timestamp").sort().diff().dt.total_seconds().std().alias("graph_std_tx_interval"),
    pl.col("Timestamp").count().alias("graph_total_tx_count")])

# PageRank
print("  📊 PageRank...")
account_to_id = dict(zip(all_accounts["account_id"].to_list(), all_accounts["node_id"].to_list()))
src_ids = df_raw_train["from_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
dst_ids = df_raw_train["to_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
vm=(src_ids>=0)&(dst_ids>=0); sv=src_ids[vm]; dv=dst_ids[vm]
oda=np.zeros(num_nodes,dtype=np.float64); np.add.at(oda,sv,1.0); oda=np.maximum(oda,1.0)
pr=np.ones(num_nodes,dtype=np.float64)/num_nodes
for _ in range(10): npr=np.ones(num_nodes,dtype=np.float64)*0.15/num_nodes; np.add.at(npr,dv,0.85*pr[sv]/oda[sv]); pr=npr
pr_df = pl.DataFrame({"node_id":np.arange(num_nodes,dtype=np.uint32),"graph_pagerank":pr.astype(np.float32)})
pr_df = all_accounts.join(pr_df,on="node_id",how="left").select(["account_id","graph_pagerank"])
del src_ids,dst_ids,sv,dv,oda,pr,npr; gc.collect()

df_graph_feats = (all_accounts.select("account_id").join(out_degree,on="account_id",how="left").join(in_degree,on="account_id",how="left")
    .join(unique_out,on="account_id",how="left").join(unique_in,on="account_id",how="left")
    .join(bidir_count,on="account_id",how="left").join(time_features,on="account_id",how="left")
    .join(pr_df,on="account_id",how="left").fill_null(0.0)
    .with_columns([(pl.col("graph_out_degree")+pl.col("graph_in_degree")).alias("graph_total_degree"),
        (pl.col("graph_out_degree")/(pl.col("graph_in_degree")+1)).alias("graph_out_in_ratio"),
        (pl.col("graph_bidir_count")/(pl.col("graph_out_degree")+1)).alias("graph_reciprocity"),
        (pl.col("graph_unique_out_partners")/(pl.col("graph_out_degree")+1)).alias("graph_partner_diversity")]))
graph_feature_cols = [c for c in df_graph_feats.columns if c.startswith("graph_")]
del out_degree,in_degree,unique_out,unique_in,bidir_count,time_features,pr_df,df_from,df_to; gc.collect()
print(f"  ✅ 그래프 피처 {len(graph_feature_cols)}개")

# ─── 2-2. 조건부행동 피처 10개 ───
print("  📊 조건부행동 피처...")
df_v2_train = df_v2.filter(pl.col("time_group")<train_cutoff)
exclude_cols = ["account_id","time_group","is_laundering","mode_format","currency_mode"]
gnn_feature_cols_v2 = [c for c in df_v2.columns if c not in exclude_cols]
df_v2_agg = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])

df_cond = df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
v2a=set(df_cond.columns)
def sc(n): return pl.col(n) if n in v2a else pl.lit(0.0)
ce=[]
if "cnt_night" in v2a and "cnt_1h" in v2a: ce.append((pl.col("graph_total_degree")*sc("cnt_night")/(sc("cnt_1h")+1)).alias("cond_highdeg_night_intensity"))
if "avg_log_amount" in v2a: ce.append((sc("avg_log_amount")/(pl.col("graph_total_degree").log1p()+1)).alias("cond_lowdeg_high_amount"))
if "ratio_cross_border" in v2a: ce.append((sc("ratio_cross_border")*(1-pl.col("graph_reciprocity"))).alias("cond_oneway_crossborder"))
if "cnt_1h" in v2a: ce.append((pl.col("graph_unique_out_partners")*sc("cnt_1h")/(pl.col("graph_out_degree")+1)).alias("cond_fanout_burst"))
if "cnt_risk_format" in v2a: ce.append((pl.col("graph_pagerank")*sc("cnt_risk_format")).alias("cond_hub_risk_format"))
if "cnt_wire" in v2a: ce.append((pl.col("graph_out_in_ratio")*sc("cnt_wire")).alias("cond_asymmetric_wire"))
if "cnt_inter_bank" in v2a: ce.append((pl.col("graph_std_tx_interval").fill_null(0)/(pl.col("graph_avg_tx_interval").fill_null(1)+1)*sc("cnt_inter_bank")).alias("cond_irregular_interbank"))
if "cnt_currencies" in v2a: ce.append((pl.col("graph_partner_diversity")*sc("cnt_currencies")).alias("cond_diverse_partner_currency"))
if "amount_kurtosis" in v2a: ce.append((pl.col("graph_reciprocity")*sc("amount_kurtosis")).alias("cond_bidir_amount_volatility"))
if "entity_acct_cnt" in v2a and "burst_ratio" in v2a: ce.append((sc("entity_acct_cnt")*sc("burst_ratio")).alias("cond_multi_acct_burst"))
df_cond=df_cond.with_columns(ce); cond_feature_cols=[e.meta.output_name() for e in ce]
for cn in cond_feature_cols: df_cond=df_cond.with_columns(pl.when(pl.col(cn).is_infinite()|pl.col(cn).is_nan()).then(0.0).otherwise(pl.col(cn)).alias(cn))
df_cond_features=df_cond.select(["account_id"]+cond_feature_cols); del df_cond; gc.collect()
print(f"  ✅ 조건부행동 {len(cond_feature_cols)}개")

# ─── 2-3. AML 점수 5개 ───
print("  📊 AML 점수...")
df_sb=df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
se=[((sc("cnt_1h").fill_null(0).clip(0,1e6)+1).log1p()*pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("burst_ratio").fill_null(0).clip(0,1e6)/(sc("avg_log_amount").fill_null(1).clip(0,1e6)+1)).alias("score_smurf"),
    ((1.0/((pl.col("graph_out_in_ratio").clip(0.01,100)-1.0).abs()+0.1))*sc("cnt_wire").fill_null(0).clip(0,1e6).log1p()*(1-pl.col("graph_reciprocity").clip(0,1))/(pl.col("graph_unique_out_partners").clip(0,1e6).log1p()+1)).alias("score_mule"),
    (pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("cnt_currencies").fill_null(0).clip(0,1e6).log1p()*sc("ratio_cross_border").fill_null(0).clip(0,1)/(pl.col("graph_avg_tx_interval").fill_null(3600).clip(0,1e9)/3600+1)).alias("score_layering"),
    (pl.col("graph_in_degree").clip(0,1e6).log1p()*(pl.col("graph_pagerank").clip(0,1)*1e6)*sc("cnt_risk_format").fill_null(0).clip(0,1e6).log1p()/(pl.col("graph_out_degree").clip(0,1e6).log1p()+1)).alias("score_integration")]
df_sb=df_sb.with_columns(se)
sn=["score_smurf","score_mule","score_layering","score_integration"]
for s in sn: df_sb=df_sb.with_columns(pl.when(pl.col(s).is_infinite()|pl.col(s).is_nan()).then(0.0).otherwise(pl.col(s)).alias(s))
for s in sn:
    mn=df_sb[s].min();mx=df_sb[s].max();rng=mx-mn if(mx-mn)>0 else 1.0
    df_sb=df_sb.with_columns(((pl.col(s)-mn)/rng).alias(f"{s}_norm"))
df_sb=df_sb.with_columns((pl.col("score_smurf_norm")*0.3+pl.col("score_mule_norm")*0.25+pl.col("score_layering_norm")*0.3+pl.col("score_integration_norm")*0.15).alias("score_composite"))
aml_score_cols=sn+["score_composite"]
df_aml_scores=df_sb.select(["account_id"]+aml_score_cols)
del df_sb, df_v2_agg; gc.collect()
print(f"  ✅ AML 점수 {len(aml_score_cols)}개")

# ─── 2-4. 규칙성 피처 7개 ───
print("  📊 규칙성 피처...")
df_tx_time = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner"),
    pl.col("Timestamp"),
    pl.col("Timestamp").dt.hour().alias("hour"),
    pl.col("Timestamp").dt.weekday().alias("dow"),
])
reg_agg = df_tx_time.group_by("account_id").agg([
    (pl.col("hour").is_between(9,17).mean()).alias("reg_business_hour_ratio"),
    (pl.col("dow").is_in([5,6]).mean()).alias("reg_weekend_ratio"),
    (pl.col("hour").is_between(0,5).mean()).alias("reg_night_strict_ratio"),
    pl.col("partner").count().alias("_total_tx"),
])
hc=df_tx_time.group_by(["account_id","hour"]).len().rename({"len":"h_count"})
hp=hc.pivot(on="hour",index="account_id",values="h_count").fill_null(0)
hcols=[c for c in hp.columns if c!="account_id"]
hmat=hp.select(hcols).to_numpy().astype(np.float64); hsum=np.maximum(hmat.sum(axis=1,keepdims=True),1.0)
hprobs=hmat/hsum; hlog=np.where(hprobs>0,np.log(hprobs+1e-12),0.0)
df_h_ent=pl.DataFrame({"account_id":hp["account_id"],"reg_hour_entropy":-np.sum(hprobs*hlog,axis=1).astype(np.float32)})
del hc,hp,hmat,hprobs,hlog; gc.collect()
dc=df_tx_time.group_by(["account_id","dow"]).len().rename({"len":"d_count"})
dp=dc.pivot(on="dow",index="account_id",values="d_count").fill_null(0)
dcols=[c for c in dp.columns if c!="account_id"]
dmat=dp.select(dcols).to_numpy().astype(np.float64); dsum=np.maximum(dmat.sum(axis=1,keepdims=True),1.0)
dprobs=dmat/dsum; dlog=np.where(dprobs>0,np.log(dprobs+1e-12),0.0)
df_d_ent=pl.DataFrame({"account_id":dp["account_id"],"reg_dow_entropy":-np.sum(dprobs*dlog,axis=1).astype(np.float32)})
del dc,dp,dmat,dprobs,dlog; gc.collect()
pf=df_tx_time.group_by(["account_id","partner"]).len().rename({"len":"p_count"})
rp=pf.filter(pl.col("p_count")>=2).group_by("account_id").len().rename({"len":"repeat_cnt"})
tp2=pf.group_by("account_id").len().rename({"len":"total_cnt"})
df_pc=tp2.join(rp,on="account_id",how="left").fill_null(0).with_columns((pl.col("repeat_cnt")/(pl.col("total_cnt")+1)).alias("reg_consistent_partner_ratio")).select(["account_id","reg_consistent_partner_ratio"])
top_p=pf.sort(["account_id","p_count"],descending=[False,True]).unique(subset=["account_id"],maintain_order=True).rename({"p_count":"top_cnt"}).select(["account_id","top_cnt"])
df_tc=reg_agg.select(["account_id","_total_tx"]).join(top_p,on="account_id",how="left").fill_null(0).with_columns((pl.col("top_cnt")/(pl.col("_total_tx")+1)).alias("reg_top_partner_concentration")).select(["account_id","reg_top_partner_concentration"])
del pf,rp,tp2,top_p; gc.collect()
df_regularity=(reg_agg.select(["account_id","reg_business_hour_ratio","reg_weekend_ratio","reg_night_strict_ratio"])
    .join(df_h_ent,on="account_id",how="left").join(df_d_ent,on="account_id",how="left")
    .join(df_pc,on="account_id",how="left").join(df_tc,on="account_id",how="left").fill_null(0.0))
regularity_cols=[c for c in df_regularity.columns if c.startswith("reg_")]
for rc in regularity_cols: df_regularity=df_regularity.with_columns(pl.when(pl.col(rc).is_infinite()|pl.col(rc).is_nan()).then(0.0).otherwise(pl.col(rc)).alias(rc))
del reg_agg,df_h_ent,df_d_ent,df_pc,df_tc; gc.collect()
print(f"  ✅ 규칙성 {len(regularity_cols)}개")

# ─── 2-5. 금액 분포 피처 8개 ───
print("  📊 금액 분포 피처...")
amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train.columns: amount_col = cand; break
if amount_col:
    df_amounts=df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col(amount_col).cast(pl.Float64).alias("amount")]).filter(pl.col("amount")>0)
    df_amt=df_amounts.group_by("account_id").agg([
        ((pl.col("amount")%1000==0)|(pl.col("amount")%5000==0)).mean().alias("amt_round_number_ratio"),
        (pl.col("amount").std()/(pl.col("amount").mean()+1)).alias("amt_cv"),
        (pl.col("amount").max()/(pl.col("amount").mean()+1)).alias("amt_max_mean_ratio"),
        (pl.col("amount").median()/(pl.col("amount").mean()+1)).alias("amt_median_mean_ratio"),
        pl.col("amount").quantile(0.9).alias("_q90"), pl.col("amount").sum().alias("_tsum"),
        pl.col("amount").count().alias("_cnt"),
        pl.col("amount").quantile(0.75).alias("_q75"), pl.col("amount").quantile(0.25).alias("_q25"),
        pl.col("amount").median().alias("_med"),
    ]).fill_null(0.0).with_columns([
        (pl.col("_q90")*0.1*pl.col("_cnt")/(pl.col("_tsum")+1)).alias("amt_top10pct_concentration"),
        ((pl.col("_q75")-pl.col("_q25"))/(pl.col("_med")+1)).alias("amt_iqr_median_ratio")])
    gm=df_amounts["amount"].mean(); st=max(gm*0.1,1.0)
    df_sr=df_amounts.with_columns((pl.col("amount")<st).alias("is_small")).group_by("account_id").agg(pl.col("is_small").mean().alias("amt_small_tx_ratio"))
    df_amt=df_amt.join(df_sr,on="account_id",how="left").fill_null(0.0)
    df_al=df_amounts.with_columns((pl.col("amount")+1).log().cast(pl.Int32).alias("log_bucket"))
    bc=df_al.group_by(["account_id","log_bucket"]).len().rename({"len":"b_count"})
    bt=bc.group_by("account_id").agg(pl.col("b_count").sum().alias("b_total"))
    bp=bc.join(bt,on="account_id").with_columns((pl.col("b_count")/pl.col("b_total")).alias("b_prob"))
    ae=bp.group_by("account_id").agg((-1*(pl.col("b_prob")*(pl.col("b_prob")+1e-12).log()).sum()).alias("amt_entropy"))
    df_amt=df_amt.join(ae,on="account_id",how="left").fill_null(0.0)
    del df_amounts,df_al,bc,bt,bp,ae,df_sr; gc.collect()
    amt_feature_cols=[c for c in df_amt.columns if c.startswith("amt_") and not c.startswith("_")]
    df_amt_features=df_amt.select(["account_id"]+amt_feature_cols)
else:
    df_amt_features=all_accounts.select("account_id"); amt_feature_cols=[]
for ac in amt_feature_cols:
    df_amt_features=df_amt_features.with_columns(pl.when(pl.col(ac).is_infinite()|pl.col(ac).is_nan()).then(0.0).otherwise(pl.col(ac)).alias(ac))
print(f"  ✅ 금액 분포 {len(amt_feature_cols)}개")

# =============================================================
# ★ NEW: 2-6. Cross-border 정밀화 피처 4개
# =============================================================
# 핵심 아이디어: ratio_cross_border 단독이 아니라
#   "cross-border × 새로운 상대방 비율 × 방향 비대칭"을 결합
print("  📊 ★ Cross-border 정밀화 피처 (신규)...")
cb_feature_cols = []

# 데이터에 to_acc country/bank 정보가 있을 경우를 가정
# → 없으면 graph_partner_diversity 와 ratio_cross_border 조합으로 근사
df_v2_agg_cb = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2 if c in df_v2_train.columns
])
df_cb = df_v2_agg_cb.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)
v2cb = set(df_cb.columns)

cb_exprs = []
# ① cb_new_partner_ratio: cross-border이면서 새로운 상대방 비율
#    = ratio_cross_border × (unique_out_partners / (total_tx_count+1))
#    → 새로운 상대에게 해외 송금하는 패턴
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("graph_unique_out_partners") /
         (pl.col("graph_total_tx_count") + 1)
        ).alias("cb_new_partner_ratio")
    )
    cb_feature_cols.append("cb_new_partner_ratio")

# ② cb_oneway_new: cross-border × 비양방향 × 새 상대방
#    → 양방향 거래 없이 새로운 상대에게만 해외 송금 (전형적 분산 송금)
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         (1.0 - pl.col("graph_reciprocity")) *
         (pl.col("graph_unique_out_partners") / (pl.col("graph_out_degree") + 1))
        ).alias("cb_oneway_new")
    )
    cb_feature_cols.append("cb_oneway_new")

# ③ cb_amount_asymmetry: 해외 송금 비율 × 금액 비대칭 (out >> in)
#    = ratio_cross_border × out_in_ratio (clip 해서 폭발 방지)
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("graph_out_in_ratio").clip(0, 20)
        ).alias("cb_amount_asymmetry")
    )
    cb_feature_cols.append("cb_amount_asymmetry")

# ④ cb_burst_cross: cross-border이 짧은 시간 내 집중 (burst_ratio 결합)
if "ratio_cross_border" in v2cb and "burst_ratio" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("burst_ratio").clip(0, 100)
        ).alias("cb_burst_cross")
    )
    cb_feature_cols.append("cb_burst_cross")

if cb_exprs:
    df_cb = df_cb.with_columns(cb_exprs)
    for cn in cb_feature_cols:
        df_cb = df_cb.with_columns(
            pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
            .then(0.0).otherwise(pl.col(cn)).alias(cn))
    df_cb_features = df_cb.select(["account_id"] + cb_feature_cols)
else:
    df_cb_features = all_accounts.select("account_id"); cb_feature_cols = []
del df_cb, df_v2_agg_cb; gc.collect()
print(f"  ✅ Cross-border 정밀화 {len(cb_feature_cols)}개: {cb_feature_cols}")

# trend_* 피처 제거 (ws_* 와 중복) — v7c.4
trend_feature_cols = []

# =============================================================
# ★ NEW: 2-8. 거래량 대비 상대 정규화 피처 5개
# =============================================================
# 핵심 아이디어: cond_lowdeg_high_amount가 FN에서 TP보다 오히려 높아 역전됨
#   → "전체 네트워크 대비 비정상적으로 높은 금액"을 더 정밀하게 포착
print("  📊 ★ 거래량 대비 상대 정규화 피처 (신규)...")
vol_norm_feature_cols = []

df_v2_agg_vn = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2 if c in df_v2_train.columns
])
df_vn = df_v2_agg_vn.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)

# 네트워크 전체 평균값 계산 (정규화 기준)
if "avg_log_amount" in set(df_vn.columns):
    global_avg_log_amount = df_vn["avg_log_amount"].mean() or 1.0
    global_avg_degree = df_vn["graph_total_degree"].mean() or 1.0
    global_avg_cnt_1h = df_vn["cnt_1h"].mean() if "cnt_1h" in df_vn.columns else 1.0
    global_avg_cnt_1h = global_avg_cnt_1h or 1.0

    vn_exprs = []
    # ① vol_amount_per_peer: 건당 금액 / (파트너 수 × 전체 평균)
    #    → degree가 낮은데 금액이 높은 패턴을 전체 대비 상대적으로 포착
    vn_exprs.append(
        (pl.col("avg_log_amount") / (float(global_avg_log_amount) + 1e-8) /
         (pl.col("graph_unique_out_partners").log1p() + 1)
        ).alias("vol_amount_per_peer")
    )
    vol_norm_feature_cols.append("vol_amount_per_peer")

    # ② vol_concentrated_amount: 소수 파트너에게 집중 송금
    #    = avg_log_amount × (1 - partner_diversity)
    #    → diversity가 낮을수록 (=한 곳에 집중) × 금액이 높을수록 높아짐
    vn_exprs.append(
        (pl.col("avg_log_amount") * (1.0 - pl.col("graph_partner_diversity").clip(0, 1))
        ).alias("vol_concentrated_amount")
    )
    vol_norm_feature_cols.append("vol_concentrated_amount")

    # ③ vol_low_freq_high_amt: 낮은 빈도이지만 건당 금액이 전체 평균 대비 높음
    #    = (avg_log_amount / global_mean) / log1p(cnt_1h) → 드문데 고액
    if "cnt_1h" in set(df_vn.columns):
        vn_exprs.append(
            (pl.col("avg_log_amount") / float(global_avg_log_amount) /
             (pl.col("cnt_1h").clip(0, 1e6).log1p() + 1)
            ).alias("vol_low_freq_high_amt")
        )
        vol_norm_feature_cols.append("vol_low_freq_high_amt")

    # ④ vol_degree_normalized_amount: 디그리로 나눈 상대 금액 (올바른 역산 버전)
    #    기존 cond_lowdeg_high_amount 대체 — log 스케일 적용 + 전체 평균으로 나눔
    vn_exprs.append(
        ((pl.col("avg_log_amount") - float(global_avg_log_amount)) /
         (pl.col("graph_total_degree").clip(1, 1e6).log1p())
        ).alias("vol_degree_normalized_amount")
    )
    vol_norm_feature_cols.append("vol_degree_normalized_amount")

    # ⑤ vol_outlier_score: z-score 방식의 금액 이상 점수
    #    = (avg_log_amount - global_mean) → 양수면 전체 평균 초과
    vn_exprs.append(
        (pl.col("avg_log_amount") - float(global_avg_log_amount)).alias("vol_outlier_score")
    )
    vol_norm_feature_cols.append("vol_outlier_score")

    df_vn = df_vn.with_columns(vn_exprs)
    for cn in vol_norm_feature_cols:
        df_vn = df_vn.with_columns(
            pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
            .then(0.0).otherwise(pl.col(cn)).alias(cn))
    df_vol_norm_features = df_vn.select(["account_id"] + vol_norm_feature_cols)
else:
    df_vol_norm_features = all_accounts.select("account_id"); vol_norm_feature_cols = []

del df_vn, df_v2_agg_vn; gc.collect()
print(f"  ✅ 거래량 대비 정규화 {len(vol_norm_feature_cols)}개: {vol_norm_feature_cols}")

# =============================================================
# ★ NEW: 2-9. 시간 윈도우별 행동 변화 피처 4개
# =============================================================
# 핵심 아이디어: 미탐의 71%가 테스트 첫 3일 집중
#   → train 말미(~recent window)에서의 행동 변화를 포착
print("  📊 ★ 시간 윈도우별 행동 변화 피처 (신규)...")
win_shift_feature_cols = []

# train 마지막 7일 vs 나머지 기간 비교
last_7d_cutoff = train_cutoff - pl.duration(days=7)
df_raw_recent = df_raw_train.filter(pl.col("Timestamp") >= last_7d_cutoff)
df_raw_older  = df_raw_train.filter(pl.col("Timestamp") < last_7d_cutoff)

recent_cnt = df_raw_recent.select(
    pl.col("from_acc").cast(pl.Utf8).alias("account_id")
).group_by("account_id").len().rename({"len": "ws_recent_cnt"})
older_cnt = df_raw_older.select(
    pl.col("from_acc").cast(pl.Utf8).alias("account_id")
).group_by("account_id").len().rename({"len": "ws_older_cnt"})

df_ws = all_accounts.select("account_id").join(recent_cnt, on="account_id", how="left").join(older_cnt, on="account_id", how="left").fill_null(0.0)

ws_exprs = []
# ① ws_recent_surge: 최근 7일 / 이전 기간 (정규화) → 갑작스러운 증가
ws_exprs.append(
    (pl.col("ws_recent_cnt") / (pl.col("ws_older_cnt") / 30.0 + 1)).alias("ws_recent_surge")
)
win_shift_feature_cols.append("ws_recent_surge")

# ② ws_recent_only: 최근 7일에만 거래가 있는 계좌 (0/1)
#    → 갑자기 나타난 계좌
ws_exprs.append(
    ((pl.col("ws_recent_cnt") > 0) & (pl.col("ws_older_cnt") == 0)).cast(pl.Float64).alias("ws_recent_only")
)
win_shift_feature_cols.append("ws_recent_only")

df_ws = df_ws.with_columns(ws_exprs)

# ③ 최근 7일 고유 파트너 급증
recent_partners = df_raw_recent.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner")
]).group_by("account_id").agg(pl.col("partner").n_unique().alias("ws_recent_partners"))
older_partners = df_raw_older.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner")
]).group_by("account_id").agg(pl.col("partner").n_unique().alias("ws_older_partners"))

df_ws = df_ws.join(recent_partners, on="account_id", how="left").join(older_partners, on="account_id", how="left").fill_null(0.0)

# ④ ws_partner_explosion: 최근 7일 파트너가 이전 대비 급증
df_ws = df_ws.with_columns(
    (pl.col("ws_recent_partners") / (pl.col("ws_older_partners") + 1)).alias("ws_partner_explosion")
)
win_shift_feature_cols.append("ws_partner_explosion")

for cn in win_shift_feature_cols:
    df_ws = df_ws.with_columns(
        pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
        .then(0.0).otherwise(pl.col(cn)).alias(cn))

df_win_shift_features = df_ws.select(["account_id"] + win_shift_feature_cols)
del df_raw_recent, df_raw_older, df_ws, recent_cnt, older_cnt
del recent_partners, older_partners; gc.collect()
print(f"  ✅ 시간 윈도우 변화 {len(win_shift_feature_cols)}개: {win_shift_feature_cols}")

# =============================================================
# ★ NEW: 2-10. 입금/출금 독립 피처 6개
# =============================================================
# 핵심 아이디어: V2에 sum_in_1h/sum_out_1h는 있지만
#   AML 점수/조건부 피처에서 in/out 독립적 분석이 부족
print("  📊 ★ 입금/출금 독립 피처 (신규)...")
inout_feature_cols = []

# 출금(from) 집계
df_out_tx = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col(amount_col).cast(pl.Float64).alias("amount") if amount_col else pl.lit(1.0).alias("amount")
]).filter(pl.col("amount") > 0)

# 입금(to) 집계
df_in_tx = df_raw_train.select([
    pl.col("to_acc").cast(pl.Utf8).alias("account_id"),
    pl.col(amount_col).cast(pl.Float64).alias("amount") if amount_col else pl.lit(1.0).alias("amount")
]).filter(pl.col("amount") > 0)

out_stats = df_out_tx.group_by("account_id").agg([
    pl.col("amount").sum().alias("inout_total_out_amount"),
    pl.col("amount").mean().alias("inout_avg_out_amount"),
    pl.col("amount").count().alias("inout_out_cnt"),
])
in_stats = df_in_tx.group_by("account_id").agg([
    pl.col("amount").sum().alias("inout_total_in_amount"),
    pl.col("amount").mean().alias("inout_avg_in_amount"),
    pl.col("amount").count().alias("inout_in_cnt"),
])
df_inout = all_accounts.select("account_id").join(out_stats, on="account_id", how="left").join(in_stats, on="account_id", how="left").fill_null(0.0)

inout_exprs = []
# ① inout_out_amount_ratio: 출금액 / (입금액 + 출금액) → 1이면 순수 송금자
inout_exprs.append(
    (pl.col("inout_total_out_amount") /
     (pl.col("inout_total_out_amount") + pl.col("inout_total_in_amount") + 1)
    ).alias("inout_out_amount_ratio")
)
inout_feature_cols.append("inout_out_amount_ratio")

# ② inout_avg_out_vs_in: 평균 출금액 / 평균 입금액 → 1 초과면 출금이 더 큰 계좌
inout_exprs.append(
    (pl.col("inout_avg_out_amount") / (pl.col("inout_avg_in_amount") + 1)
    ).alias("inout_avg_out_vs_in")
)
inout_feature_cols.append("inout_avg_out_vs_in")

# ③ inout_cnt_imbalance: (출금 건수 - 입금 건수) / 전체 건수
#    → 양수면 보내는 쪽이 훨씬 많은 계좌 (smurf 패턴)
inout_exprs.append(
    ((pl.col("inout_out_cnt") - pl.col("inout_in_cnt")) /
     (pl.col("inout_out_cnt") + pl.col("inout_in_cnt") + 1)
    ).alias("inout_cnt_imbalance")
)
inout_feature_cols.append("inout_cnt_imbalance")

# ④ inout_fan_out_score: 출금 건수 × 고유 파트너 다양성 → 여러 곳에 분산 송금
df_inout = df_inout.join(df_graph_feats.select(["account_id", "graph_partner_diversity", "graph_unique_out_partners"]), on="account_id", how="left").fill_null(0.0)
inout_exprs.append(
    (pl.col("inout_out_cnt").log1p() * pl.col("graph_partner_diversity")
    ).alias("inout_fan_out_score")
)
inout_feature_cols.append("inout_fan_out_score")

# ⑤ inout_in_concentration: 입금이 소수에서만 들어옴 (mule 패턴)
#    = avg_in_amount × (1 - partner_diversity) → 소수 소스에서 집중 수취
inout_exprs.append(
    (pl.col("inout_avg_in_amount").log1p() * (1.0 - pl.col("graph_partner_diversity").clip(0, 1))
    ).alias("inout_in_concentration")
)
inout_feature_cols.append("inout_in_concentration")

# ⑥ inout_net_flow: log(출금합) - log(입금합) → 순 흐름 방향성
inout_exprs.append(
    (pl.col("inout_total_out_amount").log1p() - pl.col("inout_total_in_amount").log1p()
    ).alias("inout_net_flow")
)
inout_feature_cols.append("inout_net_flow")

df_inout = df_inout.with_columns(inout_exprs)
for cn in inout_feature_cols:
    df_inout = df_inout.with_columns(
        pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
        .then(0.0).otherwise(pl.col(cn)).alias(cn))
df_inout_features = df_inout.select(["account_id"] + inout_feature_cols)
del df_out_tx, df_in_tx, out_stats, in_stats, df_inout; gc.collect()
print(f"  ✅ 입금/출금 독립 {len(inout_feature_cols)}개: {inout_feature_cols}")
log_memory("전체 노드 피처 완료")

# =============================================================
# 3. GNN 엣지/노드 구성 — 강화된 입력
# =============================================================
print("\n🔗 [Step 3] GNN 엣지/노드 구성 (Enriched Input)...")

df_raw_reload = pl.read_parquet(raw_path)
if df_raw_reload["Timestamp"].dtype == pl.Utf8:
    df_raw_reload = df_raw_reload.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_raw_train_edges = df_raw_reload.filter(pl.col("Timestamp")<train_cutoff); del df_raw_reload; gc.collect()

edge_amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train_edges.columns: edge_amount_col = cand; break

df_edges = df_raw_train_edges.select(
    ["from_acc","to_acc","Timestamp"] +
    ([edge_amount_col] if edge_amount_col else []) +
    (["Payment Format"] if "Payment Format" in df_raw_train_edges.columns else []) +
    (["Is Laundering"] if "Is Laundering" in df_raw_train_edges.columns else [])
).with_columns([pl.col("from_acc").cast(pl.Utf8), pl.col("to_acc").cast(pl.Utf8)])

df_edges = df_edges.join(all_accounts.rename({"account_id":"from_acc","node_id":"src_id"}),on="from_acc",how="left")
df_edges = df_edges.join(all_accounts.rename({"account_id":"to_acc","node_id":"dst_id"}),on="to_acc",how="left")

# ── 시간 피처 (5개, 기존 유지) ──────────────────────────────────────────
ts = df_edges["Timestamp"]; mn_ts=ts.min(); mx_ts=ts.max(); ts_rng=(mx_ts-mn_ts).total_seconds()
etn = ((ts-mn_ts).dt.total_microseconds().cast(pl.Float64)/1e6/max(ts_rng,1.0)).to_numpy().astype(np.float32)
hrs = df_edges["Timestamp"].dt.hour().to_numpy().astype(np.float32)
dow = df_edges["Timestamp"].dt.weekday().to_numpy().astype(np.float32)
time_feats = np.stack([etn, np.sin(2*np.pi*hrs/24), np.cos(2*np.pi*hrs/24),
                        np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7)], axis=1)

# ── 금액 피처 (3개 → 4개) ─────────────────────────────────────────────
# ★ 변경: round_flag를 1000/5000 → 100단위로 확장 (소액 스머핑 감지)
# ★ 추가: amount_pct — 전체 거래 중 분위수 위치 (극단적 고액 거래 포착)
if edge_amount_col:
    amounts_raw = df_edges[edge_amount_col].cast(pl.Float64).fill_null(0.0).to_numpy().astype(np.float32)
    log_amounts  = np.log1p(np.maximum(amounts_raw, 0)).astype(np.float32)
    log_amounts  = log_amounts / (np.max(log_amounts) + 1e-8)
    round_flag   = ((amounts_raw % 100 == 0) & (amounts_raw > 0)).astype(np.float32)
    global_mean_amt = np.mean(amounts_raw[amounts_raw > 0]) if np.any(amounts_raw > 0) else 1.0
    relative_amt = np.log1p(amounts_raw / (global_mean_amt + 1e-8)).astype(np.float32)
    relative_amt = relative_amt / (np.max(relative_amt) + 1e-8)
    nonzero_amt  = amounts_raw[amounts_raw > 0]
    pct_vals     = np.percentile(nonzero_amt, np.linspace(0,100,1001)) if len(nonzero_amt)>0 else np.zeros(1001)
    amount_pct   = np.clip(np.searchsorted(pct_vals, amounts_raw, side='right').astype(np.float32)/1000.0, 0.0, 1.0)
    amount_feats = np.stack([log_amounts, round_flag, relative_amt, amount_pct], axis=1)
else:
    amount_feats = np.zeros((len(df_edges), 4), dtype=np.float32)

# ── Payment Format 피처 (이진 정규식 2개 → 정밀 인코딩 9개) ──────────
# ★ 실제 분포 확인 결과 (HI-Medium, 7종):
#   Cheque      38.50%  세탁율 0.02%   │ ACH      12.13%  세탁율 0.79% ← 압도적 고위험
#   Credit Card 27.52%  세탁율 0.02%   │ Cash     10.09%  세탁율 0.02%
#   Reinvestment 6.10%  세탁율 0.00%   │ Wire      3.51%  세탁율 0.00%
#   Bitcoin      2.16%  세탁율 0.04%
#
# 설계 결정:
#   ① 7종 전부 원-핫 (포맷 수가 적어 "기타" 불필요)
#   ② AML 위험도 연속값 1개 (ACH=0.0079, 나머지≈0 → 강한 분리 신호)
#   ③ 기존 is_wire(정규식) 완전 제거 — Wire 세탁율 0.00%로 노이즈였음
PAYMENT_FORMATS = ["Cheque", "Credit Card", "ACH", "Cash", "Reinvestment", "Wire", "Bitcoin"]
# 포맷별 전체 데이터 기준 AML 위험도 (사전 계산값, 학습 데이터 누수 방지)
FORMAT_RISK_MAP  = {"Cheque":0.0002, "Credit Card":0.0002, "ACH":0.0079,
                    "Cash":0.0002, "Reinvestment":0.0000, "Wire":0.0000, "Bitcoin":0.0004}

if "Payment Format" in df_edges.columns:
    pf_arr = df_edges["Payment Format"].cast(pl.Utf8).fill_null("Unknown").to_numpy()

    # ① 7종 원-핫 (7차원)
    onehot_feats = np.zeros((len(pf_arr), len(PAYMENT_FORMATS)), dtype=np.float32)
    for j, fmt in enumerate(PAYMENT_FORMATS):
        onehot_feats[:, j] = (pf_arr == fmt).astype(np.float32)

    # ② AML 위험도 연속값 (1차원) — ACH가 다른 포맷 대비 ~40배 높아 강한 신호
    fmt_risk = np.array([FORMAT_RISK_MAP.get(f, 0.0002) for f in pf_arr], dtype=np.float32)

    type_feats   = np.concatenate([onehot_feats, fmt_risk.reshape(-1,1)], axis=1)  # 8차원
    N_TYPE_FEATS = type_feats.shape[1]
else:
    N_TYPE_FEATS = len(PAYMENT_FORMATS) + 1
    type_feats   = np.zeros((len(df_edges), N_TYPE_FEATS), dtype=np.float32)

edge_features_all = np.concatenate([time_feats, amount_feats, type_feats], axis=1)
EDGE_RAW_DIM = edge_features_all.shape[1]
print(f"  📊 Edge 피처: {EDGE_RAW_DIM}개")
print(f"     시간(5) + 금액(4) + 포맷({N_TYPE_FEATS}: 원-핫7+위험도1)")
print(f"     ※ 기존 is_wire/is_internal 정규식 → 7종 원-핫+위험도로 교체")

edge_index_train = torch.tensor(df_edges.select(["src_id","dst_id"]).to_numpy().T, dtype=torch.long)
edge_attr_train = torch.tensor(edge_features_all, dtype=torch.float32)
print(f"📊 Train 엣지: {edge_index_train.shape[1]:,}")

del df_raw_train, df_raw_train_edges, df_edges, ts, hrs, dow, etn
del time_feats, amount_feats, type_feats, edge_features_all; gc.collect()

# ─── 노드 피처: V2(38) + 그래프(13) + 조건부(10) + AML(5) + 규칙성(7) + 금액(8) = 81개
# ★ v7c.5: GNN 입력은 v7c.2와 동일하게 유지 (신규 피처는 XGBoost에만 투입)
print("  📊 노드 피처 구성 (GNN: v7c.2 동일 81차원)...")
df_v2_node = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])
df_node_all = (all_accounts.join(df_v2_node, on="account_id", how="left")
    .join(df_graph_feats, on="account_id", how="left")
    .join(df_cond_features, on="account_id", how="left")
    .join(df_aml_scores, on="account_id", how="left")
    .join(df_regularity, on="account_id", how="left")
    .join(df_amt_features, on="account_id", how="left")
    .fill_null(0.0))

# GNN 입력은 v7c.2 피처만 (81개) — 신규 피처는 XGBoost 단계에만 추가
gnn_input_cols = (gnn_feature_cols_v2 + graph_feature_cols + cond_feature_cols +
                  aml_score_cols + regularity_cols + amt_feature_cols)

for col in gnn_input_cols:
    if col in df_node_all.columns:
        df_node_all = df_node_all.with_columns(
            pl.when(pl.col(col).is_infinite()|pl.col(col).is_nan()).then(0.0).otherwise(pl.col(col)).alias(col))

X_node = torch.tensor(df_node_all.select(gnn_input_cols).to_numpy(), dtype=torch.float32)
IN_DIM = X_node.shape[1]
print(f"  ✅ GNN 노드 입력: {IN_DIM}차원 (v7c.2 동일)")
print(f"     = V2({len(gnn_feature_cols_v2)}) + 그래프({len(graph_feature_cols)}) + 조건부({len(cond_feature_cols)})")
print(f"       + AML({len(aml_score_cols)}) + 규칙성({len(regularity_cols)}) + 금액({len(amt_feature_cols)})")

target_labels = df_v2_train.filter(pl.col("is_laundering")==1).select("account_id").unique().with_columns(pl.lit(1).alias("label"))
df_labels = all_accounts.join(target_labels,on="account_id",how="left").fill_null(0)
Y_node = torch.tensor(df_labels["label"].to_numpy(), dtype=torch.long)
pos_node_ids = set(df_labels.filter(pl.col("label")==1)["node_id"].to_list())
active_df = pl.DataFrame({"account_id":df_v2_train["account_id"].unique(),"is_active":True})
mask_df = all_accounts.join(active_df,on="account_id",how="left").fill_null(False)
active_node_ids = mask_df.filter(pl.col("is_active"))["node_id"].to_list()
active_pos = [n for n in active_node_ids if n in pos_node_ids]
active_neg = [n for n in active_node_ids if n not in pos_node_ids]
n_neg_gnn = min(len(active_pos)*NEG_NODE_RATIO_GNN, len(active_neg))
np.random.seed(RANDOM_SEED)
sampled_neg = np.random.choice(active_neg,size=n_neg_gnn,replace=False).tolist()
sampled_nodes = set(active_pos+sampled_neg)
train_mask_np = np.zeros(num_nodes, dtype=bool)
for nid in sampled_nodes: train_mask_np[nid]=True
train_mask = torch.tensor(train_mask_np, dtype=torch.bool)
print(f"📊 GNN 노드: 양성 {len(active_pos):,} + 음성 {n_neg_gnn:,}")
del df_v2_node, df_node_all, mask_df, active_df; gc.collect()

graph_data = Data(x=X_node, edge_index=edge_index_train, edge_attr=edge_attr_train, y=Y_node)
graph_data.train_mask = train_mask; graph_data.num_nodes = num_nodes
del X_node, edge_index_train, edge_attr_train, Y_node, train_mask; gc.collect()
log_memory("GNN 데이터 준비 완료")

# =============================================================
# 4-6. Multi-Config GNN 탐색 → 최적 임베딩 선택
# =============================================================
print("\n🧠 [Step 4-6] Multi-Config GNN 탐색...")

class EdgeProjection(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, out_dim), nn.GELU(), nn.Linear(out_dim, out_dim))
    def forward(self, ea): return self.proj(ea)

class TGAT_Flex(nn.Module):
    def __init__(self, in_dim, hidden_dim, edge_raw_dim, edge_dim, n_heads, n_layers=2, dropout=0.2):
        super().__init__()
        self.edge_proj = EdgeProjection(edge_raw_dim, edge_dim)
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.n_layers = n_layers
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_layers)])
        self.convs = nn.ModuleList([
            TransformerConv(hidden_dim, hidden_dim//n_heads, heads=n_heads,
                            edge_dim=edge_dim, dropout=dropout, concat=True)
            for _ in range(n_layers)])
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim//2, 1))
        self.dropout = dropout; self.hidden_dim = hidden_dim

    def forward(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        for i in range(self.n_layers):
            h = h + F.dropout(self.convs[i](self.norms[i](h), edge_index, ee), p=self.dropout, training=self.training)
        return self.classifier(self.norm_out(h)).squeeze(-1)

    @torch.no_grad()
    def get_embedding(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        for i in range(self.n_layers):
            h = h + self.convs[i](self.norms[i](h), edge_index, ee)
        return self.norm_out(h)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"📌 디바이스: {device}")
pw = (len(sampled_nodes)-len(active_pos))/max(len(active_pos),1)
print(f"📊 pos_weight: {pw:.1f}")

def make_lr(opt, ep, n_ep):
    if ep < WARMUP_EPOCHS: lr = 1e-6+(BASE_LR-1e-6)*(ep/WARMUP_EPOCHS)
    else: p=(ep-WARMUP_EPOCHS)/max(n_ep-WARMUP_EPOCHS,1); lr=1e-5+(BASE_LR-1e-5)*0.5*(1+np.cos(np.pi*p))
    for pg in opt.param_groups: pg['lr']=lr
    return lr

GNN_CONFIGS = [
    {"label":"h64_L2",  "hidden_dim":64,  "n_layers":2, "n_heads":4, "edge_dim":32},  # ★ 16→32
    {"label":"h128_L2", "hidden_dim":128, "n_layers":2, "n_heads":4, "edge_dim":32},
    {"label":"h64_L3",  "hidden_dim":64,  "n_layers":3, "n_heads":4, "edge_dim":32},  # ★ 16→32
    {"label":"h128_L3", "hidden_dim":128, "n_layers":3, "n_heads":4, "edge_dim":32},
]

xgb_exclude_cols=["account_id","time_group","is_laundering","mode_format","currency_mode","date"]
xgb_v2_cols=[c for c in df_v2.columns if c not in xgb_exclude_cols]
apr_train = df_v2_train.filter(pl.col("is_laundering")==1).height / max(df_v2_train.height,1)
spw = max((1-apr_train)/apr_train, 1.0) if apr_train > 0 else 20.0

# ★ 신규 피처들을 base_feature_cols에 포함
base_feature_cols = (xgb_v2_cols + graph_feature_cols + cond_feature_cols + aml_score_cols +
                     cb_feature_cols + vol_norm_feature_cols +
                     win_shift_feature_cols + inout_feature_cols)

def build_eval_df(db, df_emb_local, emb_col_names):
    feature_cols = base_feature_cols + emb_col_names
    df = (db.join(df_graph_feats,on="account_id",how="left")
          .join(df_cond_features,on="account_id",how="left")
          .join(df_aml_scores,on="account_id",how="left")
          .join(df_cb_features,on="account_id",how="left")            # ★
          .join(df_vol_norm_features,on="account_id",how="left")      # ★
          .join(df_win_shift_features,on="account_id",how="left")     # ★
          .join(df_inout_features,on="account_id",how="left")         # ★
          .join(df_emb_local,on="account_id",how="left")
          .fill_null(0.0).fill_nan(0.0))
    for cn in feature_cols:
        if cn in df.columns:
            df = df.with_columns(pl.when(pl.col(cn).is_infinite()).then(0.0).otherwise(pl.col(cn)).alias(cn))
    return df, feature_cols

def quick_xgb_eval(df_emb_local, emb_col_names, label):
    df_tr, fcols = build_eval_df(df_v2.filter(pl.col("time_group")<train_cutoff), df_emb_local, emb_col_names)
    df_tr_pos=df_tr.filter(pl.col("is_laundering")==1); df_tr_neg=df_tr.filter(pl.col("is_laundering")==0)
    n_neg=min(len(df_tr_pos)*NEG_ROW_RATIO_XGB,len(df_tr_neg))
    df_tr_s=pl.concat([df_tr_pos,df_tr_neg.sample(n=n_neg,seed=RANDOM_SEED)]).sample(fraction=1.0,seed=RANDOM_SEED)
    X_tr=df_tr_s.select(fcols).to_pandas(); y_tr=df_tr_s["is_laundering"].to_numpy()
    del df_tr,df_tr_pos,df_tr_neg,df_tr_s; gc.collect()
    df_vl, _ = build_eval_df(df_v2.filter((pl.col("time_group")>=train_cutoff)&(pl.col("time_group")<val_cutoff)), df_emb_local, emb_col_names)
    X_vl=df_vl.select(fcols).to_pandas(); y_vl=df_vl["is_laundering"].to_numpy(); del df_vl; gc.collect()
    df_te, _ = build_eval_df(df_v2.filter(pl.col("time_group")>=val_cutoff), df_emb_local, emb_col_names)
    X_te=df_te.select(fcols).to_pandas(); y_te=df_te["is_laundering"].to_numpy(); del df_te; gc.collect()
    qp={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
        "random_state":42,"max_depth":8,"learning_rate":0.03,"scale_pos_weight":spw,
        "subsample":0.8,"colsample_bytree":0.7,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
    m=xgb.XGBClassifier(**qp,n_estimators=1500,early_stopping_rounds=100)
    m.fit(X_tr,y_tr,eval_set=[(X_vl,y_vl)],verbose=0)
    q_val=m.best_score; q_test=average_precision_score(y_te,m.predict_proba(X_te)[:,1])
    del m,X_tr,X_vl,X_te,y_tr,y_vl,y_te; gc.collect()
    return q_val, q_test

print(f"\n📊 {len(GNN_CONFIGS)}개 GNN 설정 탐색...")
print(f"  {'#':>2s} {'Label':<12s} | {'hidden':>6s} {'layers':>6s} {'edim':>4s} | {'Loss':>8s} {'Params':>8s} | {'QVal':>8s} {'QTest':>8s}")
print("  "+"-"*80)

gnn_results = []
for gi, gcfg in enumerate(GNN_CONFIGS):
    gl = gcfg["label"]; hd=gcfg["hidden_dim"]; nl=gcfg["n_layers"]
    nh=gcfg["n_heads"]; ed=gcfg["edge_dim"]
    train_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                                   input_nodes=graph_data.train_mask, shuffle=True, num_workers=4)
    full_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                                  input_nodes=None, shuffle=False, num_workers=4)
    model = TGAT_Flex(in_dim=IN_DIM, hidden_dim=hd, edge_raw_dim=EDGE_RAW_DIM,
                      edge_dim=ed, n_heads=nh, n_layers=nl, dropout=0.2).to(device)
    np_count = sum(p.numel() for p in model.parameters())
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw],device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6, weight_decay=1e-4)
    model.train(); bl=float('inf'); pc2=0; bs=None
    for epoch in range(N_EPOCHS_GNN):
        clr=make_lr(optimizer,epoch,N_EPOCHS_GNN); tl=0.0; nb2=0
        for batch in tqdm(train_loader,desc=f"[{gl}] Ep{epoch+1}",leave=False):
            batch=batch.to(device); optimizer.zero_grad()
            bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
            out=model(batch.x,batch.edge_index,bea)
            loss=criterion(out[:batch.batch_size],batch.y[:batch.batch_size].float())
            if torch.isnan(loss) or torch.isinf(loss): optimizer.zero_grad(); continue
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=0.5)
            optimizer.step(); tl+=loss.item(); nb2+=1
        al=tl/max(nb2,1)
        if (epoch+1)%10==0 or epoch==0: print(f"    [{gl}] Ep{epoch+1:2d} Loss={al:.4f}")
        if al<bl-EARLY_STOP_DELTA: bl=al; pc2=0; bs={k:v.cpu().clone() for k,v in model.state_dict().items()}
        else: pc2+=1
        if pc2>=PATIENCE: break
    if bs: model.load_state_dict({k:v.to(device) for k,v in bs.items()}); del bs
    model.eval(); ae=[]
    with torch.no_grad():
        for batch in tqdm(full_loader,desc=f"[{gl}] Embed",leave=False):
            batch=batch.to(device)
            bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
            emb=model.get_embedding(batch.x,batch.edge_index,bea)
            ae.append(emb[:batch.batch_size].cpu())
    femb=torch.cat(ae,dim=0).numpy()
    del model,ae,train_loader,full_loader; torch.cuda.empty_cache(); gc.collect()
    ecn=[f"tgat_emb_{i}" for i in range(hd)]
    df_emb=pl.concat([all_accounts.select("account_id"),pl.DataFrame(femb,schema=ecn)],how="horizontal")
    qv,qt = quick_xgb_eval(df_emb, ecn, gl)
    print(f"  {gi+1:2d} {gl:<12s} | {hd:6d} {nl:6d} {ed:4d} | {bl:8.4f} {np_count:8,} | {qv:8.5f} {qt:8.4f}")
    gnn_results.append({"label":gl,"hidden_dim":hd,"n_layers":nl,"loss":bl,"params":np_count,
                         "q_val":qv,"q_test":qt,"embedding":femb,"ecn":ecn,"df_emb":df_emb})
    log_memory(f"{gl}")

best_gnn = max(gnn_results, key=lambda x: x["q_val"])
print(f"\n  🏆 Best GNN: {best_gnn['label']} (QVal={best_gnn['q_val']:.5f} QTest={best_gnn['q_test']:.4f})")
for r in gnn_results:
    if r is not best_gnn: del r["embedding"], r["df_emb"]
gc.collect()

emb_cols = best_gnn["ecn"]
df_tgat = best_gnn["df_emb"]
FINAL_HIDDEN_DIM = best_gnn["hidden_dim"]

print(f"\n🔗 Interaction 피처 ({best_gnn['label']})...")
df_tgat_with_pr=df_tgat.join(df_graph_feats.select(["account_id","graph_pagerank"]),on="account_id",how="left").fill_null(0.0)
emb_np=df_tgat_with_pr.select(emb_cols).to_numpy(); emb_var=np.var(emb_np,axis=0)
top_idx=np.argsort(emb_var)[::-1][:N_INTERACTION_DIMS]; top_ec=[emb_cols[i] for i in top_idx]
ie_l=[]; ic_l=[]
for cn in top_ec: nn2=f"interact_pr_x_{cn}"; ie_l.append((pl.col(cn)*pl.col("graph_pagerank")*1e6).alias(nn2)); ic_l.append(nn2)
ie_l.append((sum(pl.col(c)**2 for c in emb_cols).sqrt()*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_norm")); ic_l.append("interact_pr_x_emb_norm")
ie_l.append((sum(pl.col(c) for c in top_ec)*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_topsum")); ic_l.append("interact_pr_x_emb_topsum")
interaction_cols=ic_l
df_interactions=df_tgat_with_pr.with_columns(ie_l).select(["account_id"]+ic_l)
for c in ic_l: df_interactions=df_interactions.with_columns(pl.when(pl.col(c).is_infinite()|pl.col(c).is_nan()).then(0.0).otherwise(pl.col(c)).alias(c))
print(f"  ✅ Interaction {len(ic_l)}개")
del df_tgat_with_pr,emb_np,best_gnn["embedding"]; gc.collect()
del graph_data; torch.cuda.empty_cache(); gc.collect()
log_memory("GNN 탐색 완료")

# =============================================================
# 7. XGBoost 데이터 구성
# =============================================================
print("\n🚀 [Step 7] XGBoost 데이터 구성...")

# ★ v7c.7: regularity(7) + amt(8) = 15개를 XGBoost 후보에 추가 (v7c.6 누락 수정)
# GNN 입력에는 이미 포함돼 있었지만 xgb_feature_cols에서 빠져 있었음
xgb_feature_cols = (xgb_v2_cols + graph_feature_cols + cond_feature_cols + aml_score_cols +
                    regularity_cols + amt_feature_cols +          # ★ 신규 추가
                    cb_feature_cols + vol_norm_feature_cols +
                    win_shift_feature_cols + inout_feature_cols +
                    emb_cols + interaction_cols)
selected_new_cols = cb_feature_cols + vol_norm_feature_cols + win_shift_feature_cols + inout_feature_cols
print(f"📊 XGBoost 총 후보 피처: {len(xgb_feature_cols)}")
print(f"   V2:{len(xgb_v2_cols)} G:{len(graph_feature_cols)} Cond:{len(cond_feature_cols)} AML:{len(aml_score_cols)}")
print(f"   ★추가 Reg:{len(regularity_cols)} Amt:{len(amt_feature_cols)}  (v7c.6 누락 수정)")
print(f"   CB:{len(cb_feature_cols)} VolNorm:{len(vol_norm_feature_cols)}")
print(f"   WinShift:{len(win_shift_feature_cols)} InOut:{len(inout_feature_cols)}")
print(f"   Emb:{len(emb_cols)} Inter:{len(interaction_cols)}")
print(f"   ★ Forward Selection이 전체 후보에서 유효한 것 선별")

def build_xgb_df(db):
    df=(db.join(df_graph_feats,on="account_id",how="left")
          .join(df_cond_features,on="account_id",how="left")
          .join(df_aml_scores,on="account_id",how="left")
          .join(df_regularity,on="account_id",how="left")        # ★ 신규 추가
          .join(df_amt_features,on="account_id",how="left")      # ★ 신규 추가
          .join(df_cb_features,on="account_id",how="left")
          .join(df_vol_norm_features,on="account_id",how="left")
          .join(df_win_shift_features,on="account_id",how="left")
          .join(df_inout_features,on="account_id",how="left")
          .join(df_tgat,on="account_id",how="left")
          .join(df_interactions,on="account_id",how="left")
          .fill_null(0.0).fill_nan(0.0))
    for cn in xgb_feature_cols:
        if cn in df.columns: df=df.with_columns(pl.when(pl.col(cn).is_infinite()).then(0.0).otherwise(pl.col(cn)).alias(cn))
    return df

df_tr=build_xgb_df(df_v2.filter(pl.col("time_group")<train_cutoff))
df_tr_pos=df_tr.filter(pl.col("is_laundering")==1); df_tr_neg=df_tr.filter(pl.col("is_laundering")==0)
n_neg=min(len(df_tr_pos)*NEG_ROW_RATIO_XGB,len(df_tr_neg))
df_tr_s=pl.concat([df_tr_pos,df_tr_neg.sample(n=n_neg,seed=RANDOM_SEED)]).sample(fraction=1.0,seed=RANDOM_SEED)
X_train=df_tr_s.select(xgb_feature_cols).to_pandas(); y_train=df_tr_s["is_laundering"].to_numpy()
print(f"📊 Train: {len(df_tr):,} → {len(df_tr_s):,} ({y_train.mean()*100:.2f}%)")
del df_tr,df_tr_pos,df_tr_neg,df_tr_s; gc.collect()
df_vl=build_xgb_df(df_v2.filter((pl.col("time_group")>=train_cutoff)&(pl.col("time_group")<val_cutoff)))
X_val=df_vl.select(xgb_feature_cols).to_pandas(); y_val=df_vl["is_laundering"].to_numpy(); print(f"📊 Val: {len(X_val):,}"); del df_vl; gc.collect()
df_te=build_xgb_df(df_v2.filter(pl.col("time_group")>=val_cutoff))
df_test_meta=df_te.select(["account_id","time_group","is_laundering"])
X_test=df_te.select(xgb_feature_cols).to_pandas(); y_test=df_te["is_laundering"].to_numpy(); print(f"📊 Test: {len(X_test):,}")
del df_te,df_v2,df_tgat,df_graph_feats,df_cond_features,df_aml_scores
del df_cb_features,df_vol_norm_features,df_win_shift_features,df_inout_features
del df_interactions,df_regularity,df_amt_features; gc.collect()
log_memory("XGBoost 준비")

# =============================================================
# 8. XGBoost 3-Stage: Full → Forward Selection → HP Grid
# =============================================================
# ★ v7c.6 핵심 변경: Stage 1 importance 기반 top-K 선택의 한계 해결
#   문제: 상관된 피처끼리 importance를 나눠 가져서 유용한 피처가 순위 밖으로 밀림
#   해결: Stage 2를 Forward Selection으로 교체
#     → 피처를 하나씩 추가하며 Val AUCPR이 실제로 오를 때만 채택
#     → 중복/노이즈 피처는 자동으로 탈락
print("\n🔥 [Step 8] 3-Stage XGBoost (Full → Forward Selection → HP Grid)...")
apr=y_train.sum()/len(y_train); spw=max((1-apr)/apr,1.0)

# ── Stage 1: 전체 피처로 학습 → importance 순위 산출 ──────────────────
print("\n── Stage 1: 전체 피처 학습 (importance 순위 산출) ──")
s1p={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
     "random_state":42,"max_depth":8,"learning_rate":0.03,"scale_pos_weight":spw,
     "subsample":0.8,"colsample_bytree":0.7,"min_child_weight":5,
     "gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
m_s1=xgb.XGBClassifier(**s1p,n_estimators=1500,early_stopping_rounds=100)
m_s1.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=100)
s1_val=m_s1.best_score
s1_test=average_precision_score(y_test,m_s1.predict_proba(X_test)[:,1])
print(f"\n  Stage1: Val={s1_val:.5f} Test={s1_test:.4f}")
imp_s1=m_s1.feature_importances_
# importance 순위 정렬 (Stage 2 Forward Selection의 탐색 순서로 사용)
feat_imp=sorted(zip(xgb_feature_cols,imp_s1),key=lambda x:x[1],reverse=True)
del m_s1; gc.collect()

# ── Stage 2: Forward Selection ─────────────────────────────────────────
# importance 상위 순서대로 피처를 하나씩 추가하며
# Val AUCPR이 오를 때만 채택, 연속 N_PATIENCE_FS번 개선 없으면 조기 종료
print("\n── Stage 2: Forward Selection ──")
print("  피처를 importance 순서대로 하나씩 추가 → Val 개선 시 채택")

N_PATIENCE_FS = 8   # 연속 N번 개선 없으면 탐색 종료
MIN_DELTA_FS  = 5e-5  # 최소 개선량 (노이즈 수준 변동 무시)
MAX_FEATURES_FS = 100  # 탐색할 최대 피처 수 (전체 중 상위 100개만 탐색)

fs_hp={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
       "random_state":42,"max_depth":9,"learning_rate":0.02,"scale_pos_weight":spw,
       "subsample":0.8,"colsample_bytree":0.8,"min_child_weight":5,
       "gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}

selected_fs = []          # 채택된 피처 누적
best_fs_val = -1.0        # 현재까지 best Val AUCPR
no_improve  = 0           # 연속 미개선 횟수
fs_log      = []          # 로그 기록

candidates = [f for f,_ in feat_imp[:MAX_FEATURES_FS]]

print(f"  {'#':>3s} {'Feature':<45s} | {'Val':>9s} {'Δ':>8s} {'채택':>4s}")
print("  " + "-" * 75)

for fi, feat in enumerate(candidates):
    probe = selected_fs + [feat]
    m_fs = xgb.XGBClassifier(**fs_hp, n_estimators=1500, early_stopping_rounds=100)  # ★ 800→1500
    m_fs.fit(X_train[probe], y_train, eval_set=[(X_val[probe], y_val)], verbose=False)
    v = m_fs.best_score
    delta = v - best_fs_val
    adopted = delta > MIN_DELTA_FS

    tag = "✅" if adopted else "  "
    print(f"  {fi+1:3d} {feat:<45s} | {v:9.5f} {delta:+8.5f} {tag}")
    fs_log.append({"feat":feat,"val":v,"delta":delta,"adopted":adopted})

    if adopted:
        selected_fs.append(feat)
        best_fs_val = v
        no_improve = 0
    else:
        no_improve += 1

    del m_fs; gc.collect()

    if no_improve >= N_PATIENCE_FS:
        print(f"\n  ⏹ 연속 {N_PATIENCE_FS}회 미개선 → 조기 종료 (탐색 {fi+1}/{len(candidates)})")
        break

print(f"\n  ✅ Forward Selection 완료: {len(selected_fs)}개 채택")
print(f"  채택 피처: {selected_fs}")
print(f"  Best Val AUCPR: {best_fs_val:.5f}  (Stage1: {s1_val:.5f}  Δ={best_fs_val-s1_val:+.5f})")

# ── Stage 3: HP Grid (Forward Selection 채택 피처 고정) ───────────────
print("\n── Stage 3: HP Grid (Forward Selection 피처 고정) ──")
# Forward Selection이 고른 피처로 HP만 탐색
bp3={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
     "random_state":42,"scale_pos_weight":spw}
hp_grid=[
    {"label":"d8_lr03_c07","hp":{"max_depth":8,"learning_rate":0.03,"colsample_bytree":0.7,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":1500}},
    {"label":"d8_lr02_c08","hp":{"max_depth":8,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d9_lr02_c08","hp":{"max_depth":9,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d9_lr02_c09","hp":{"max_depth":9,"learning_rate":0.02,"colsample_bytree":0.9,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d10_lr02_c08","hp":{"max_depth":10,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d10_lr01_c08","hp":{"max_depth":10,"learning_rate":0.01,"colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":4000}},
    {"label":"d9_reg","hp":{"max_depth":9,"learning_rate":0.02,"colsample_bytree":0.8,"subsample":0.75,"min_child_weight":8,"gamma":0.2,"reg_alpha":0.5,"reg_lambda":2.0,"n_estimators":2500}},
]

print(f"\n📊 {len(hp_grid)}개 HP 조합 탐색 (피처 {len(selected_fs)}개 고정)...")
print(f"  {'#':>2s} {'Label':<20s} | {'d':>2s} {'LR':>5s} {'col':>4s} | {'Val':>9s} {'it':>5s}")
print("  "+"-"*55)
best_s2={"val":-1}
for i,cfg in enumerate(hp_grid):
    hp=cfg["hp"].copy(); ne=hp.pop("n_estimators"); sf=selected_fs
    m=xgb.XGBClassifier(**{**bp3,**hp},n_estimators=ne,early_stopping_rounds=100)
    m.fit(X_train[sf],y_train,eval_set=[(X_val[sf],y_val)],verbose=False)
    va=m.best_score; bi=m.best_iteration
    print(f"  {i+1:2d} {cfg['label']:<20s} | {hp.get('max_depth',8):2d} {hp.get('learning_rate',0.03):5.3f} {hp.get('colsample_bytree',0.7):4.2f} | {va:9.5f} {bi:5d}")
    if va>best_s2["val"]:
        if best_s2.get("model"): del best_s2["model"]
        best_s2={"label":cfg["label"],"features":sf,"val":va,"iter":bi,"model":m,"hp":hp}
    else: del m
    gc.collect()

y_prob=best_s2["model"].predict_proba(X_test[best_s2["features"]])[:,1]
best_features=best_s2["features"]
print(f"\n  🏆 Best: {best_s2['label']} ({len(best_features)}개) Val={best_s2['val']:.5f} iter={best_s2['iter']}")

imp=best_s2["model"].feature_importances_; tidx=np.argsort(imp)[::-1][:30]
print(f"\n🔍 Feature Importance Top 30")
for i,idx in enumerate(tidx):
    c=best_features[idx]; t=""
    if c.startswith("tgat_emb_"): t=" [TGAT]"
    elif c.startswith("graph_"): t=" [GRAPH]"
    elif c.startswith("cond_"): t=" [COND]"
    elif c.startswith("score_"): t=" [SCORE]"
    elif c.startswith("interact_"): t=" [INTER]"
    elif c.startswith("cb_"): t=" [★CB]"
    elif c.startswith("vol_"): t=" [★VOL]"
    elif c.startswith("ws_"): t=" [★WIN]"
    elif c.startswith("inout_"): t=" [★INOUT]"
    print(f"  {i+1:2d}. {c:<45s}: {imp[idx]:.4f}{t}")

ti=imp.sum()
def group_imp(prefix): return sum(imp[i] for i,c in enumerate(best_features) if c.startswith(prefix))
print(f"\n📊 그룹별 중요도:")
print(f"   V2        : {group_imp('') / ti * 100:.1f}% (기존)")
print(f"   Graph     : {group_imp('graph_') / ti * 100:.1f}%")
print(f"   TGAT      : {group_imp('tgat_emb_') / ti * 100:.1f}%")
print(f"   Interact  : {group_imp('interact_') / ti * 100:.1f}%")
print(f"   ★ CB      : {group_imp('cb_') / ti * 100:.1f}%")
print(f"   ★ VolNorm : {group_imp('vol_') / ti * 100:.1f}%")
print(f"   ★ WinShift: {group_imp('ws_') / ti * 100:.1f}%")
print(f"   ★ InOut   : {group_imp('inout_') / ti * 100:.1f}%")
del X_train,X_val; gc.collect()

# =============================================================
# 9. 최종 리포트
# =============================================================
print("\n"+"="*60); print("🏆 [TGAT v7c.10] 최종 리포트"); print("="*60)
auprc=average_precision_score(y_test,y_prob); print(f"\n  AUPRC: {auprc:.4f}")
best_f1=0; best_thresh=0
for th in np.arange(0.05,0.95,0.01):
    f=f1_score(y_test,(y_prob>=th).astype(int),zero_division=0)
    if f>best_f1: best_f1=f; best_thresh=th
y_pred=(y_prob>=best_thresh).astype(int)
prec=precision_score(y_test,y_pred,zero_division=0); rec=recall_score(y_test,y_pred,zero_division=0)
print(f"\n📌 최적: T={best_thresh:.2f} F1={best_f1:.4f} P={prec:.4f} R={rec:.4f}")
print(f"\n📌 임계값별:")
for t in [0.1,0.3,0.5,0.7,0.8,0.9]:
    yp=(y_prob>=t).astype(int)
    print(f"  T={t:.1f} P={precision_score(y_test,yp,zero_division=0):.4f} R={recall_score(y_test,yp,zero_division=0):.4f} F1={f1_score(y_test,yp,zero_division=0):.4f}")

df_res=df_test_meta.with_columns([pl.col("time_group").dt.date().alias("date"),pl.Series("pred_prob",y_prob)])
df_dist=df_res.sort("pred_prob",descending=True).unique(subset=["account_id"],maintain_order=True)
print(f"\n📌 Top-K:")
for k in [100,500,1000,5000,10000]:
    ck=min(k,len(df_dist)); hits=df_dist.head(ck)["is_laundering"].sum()
    print(f"  Top-{k:5d}: {hits:5d}명 ({hits/ck*100:.2f}%)")
print(f"\n📌 일별 Top-100:")
for d in df_dist["date"].unique(maintain_order=True).sort(descending=True):
    hits=df_dist.filter(pl.col("date")==d).head(100)["is_laundering"].sum(); print(f"  {d}: {hits}")

tn,fp,fn,tp=confusion_matrix(y_test,y_pred).ravel()
print(f"\n📌 v7c.9→v7c.10: AUPRC {auprc:.4f} | F1 {best_f1:.4f}")
print(f"  TP:{tp:,} FP:{fp:,} FN:{fn:,} TN:{tn:,}")
print(f"  Precision:{prec:.4f} Recall:{rec:.4f}")

print(f"\n📌 GNN 탐색 결과:")
for r in gnn_results:
    tag = " ★" if r["label"]==best_gnn["label"] else ""
    print(f"  {r['label']:<12s}: QVal={r['q_val']:.5f} QTest={r['q_test']:.4f} Loss={r['loss']:.4f} Params={r['params']:,}{tag}")

print(f"\n📌 3-Stage 결과:")
print(f"  S1 전체({len(xgb_feature_cols)}개): Val={s1_val:.5f} Test={s1_test:.4f}")
print(f"  S2 Forward Selection: {len(selected_fs)}개 채택, Val={best_fs_val:.5f}")
print(f"  S3 HP Grid ({best_s2['label']}): Val={best_s2['val']:.5f} Test={auprc:.4f}")

print(f"\n✨ v7c.10 변경 요약 (v7c.6 대비 누적):")
print(f"  GNN epochs    : 40 → {N_EPOCHS_GNN}")
print(f"  GNN edge_dim  : 16 → 32 (h64)")
print(f"  GNN NEG_RATIO : 50 → {NEG_NODE_RATIO_GNN}")
print(f"  FS n_est      : 800 → 1500")
print(f"  엣지 차원     : 10 → {EDGE_RAW_DIM}")
print(f"  XGB 후보      : {len(xgb_feature_cols)}개 (regularity+amt 포함)")
print(f"  FS 채택       : {len(selected_fs)}개")
print(f"  신규 피처 FS 채택: {[c for c in selected_fs if c in selected_new_cols]}")
print(f"  규칙성/금액 FS 채택: {[c for c in selected_fs if c in regularity_cols+amt_feature_cols]}")

del X_test; gc.collect(); log_memory("최종 완료")
print(f"\n✅ TGAT v7c.10 완료!")

🛡️ [TGAT v7c.10] 튜닝 개선 + 엣지 피처 정밀화
   ▸ GNN: epochs 40→60, early_stop delta 0.0005→0.001
   ▸ GNN: edge_dim 16→32 (h64 병목 해소)
   ▸ GNN: NEG_NODE_RATIO 50→25 (이중 불균형 보정 완화)
   ▸ FS:  n_estimators 800→1500 (피처 평가 수렴 강화)
   ▸ 엣지: 10→17차원 (7종 원-핫+위험도, 금액 분위수 추가)

📂 데이터 로딩 중...
⏱️ Train cutoff: 2022-09-09 20:00:00
⏱️ Val   cutoff: 2022-09-14 05:00:00
📊 전체 노드 수: 2,076,999
  💾 [데이터 로드 완료] RAM: 98.29GB | GPU: 0.02GB

📐 [Step 2] 전체 노드 피처 계산...
  📊 그래프 구조 피처...
  📊 PageRank...
  ✅ 그래프 피처 13개
  📊 조건부행동 피처...
  ✅ 조건부행동 10개
  📊 AML 점수...
  ✅ AML 점수 5개
  📊 규칙성 피처...
  ✅ 규칙성 7개
  📊 금액 분포 피처...
  ✅ 금액 분포 8개
  📊 ★ Cross-border 정밀화 피처 (신규)...
  ✅ Cross-border 정밀화 4개: ['cb_new_partner_ratio', 'cb_oneway_new', 'cb_amount_asymmetry', 'cb_burst_cross']
  📊 ★ 거래량 대비 상대 정규화 피처 (신규)...
  ✅ 거래량 대비 정규화 5개: ['vol_amount_per_peer', 'vol_concentrated_amount', 'vol_low_freq_high_amt', 'vol_degree_normalized_amount', 'vol_outlier_score']
  📊 ★ 시간 윈도우별 행동 변화 피처 (신규)...
  ✅ 시간 윈도우 변화 3개: ['ws_recent_surge', 'ws_recent_

    [h64_L2] Ep 1 Loss=1.3256


    [h64_L2] Ep10 Loss=1.0429


    [h64_L2] Ep20 Loss=1.0115


    [h64_L2] Ep30 Loss=0.9923


...

    [h128_L3] Ep30 Loss=0.9772


    [h128_L3] Ep40 Loss=0.9654


    [h128_L3] Ep50 Loss=0.9580


    [h128_L3] Ep60 Loss=0.9517


   4 h128_L3      |    128      3   32 |   0.9518  231,905 |  0.38330   0.4085
  💾 [h128_L3] RAM: 86.73GB | GPU: 0.02GB

  🏆 Best GNN: h64_L3 (QVal=0.38687 QTest=0.4117)

🔗 Interaction 피처 (h64_L3)...
  ✅ Interaction 10개
  💾 [GNN 탐색 완료] RAM: 83.47GB | GPU: 0.02GB

🚀 [Step 7] XGBoost 데이터 구성...
📊 XGBoost 총 후보 피처: 173
   V2:38 G:13 Cond:10 AML:5
   ★추가 Reg:7 Amt:8  (v7c.6 누락 수정)
   CB:4 VolNorm:5
   WinShift:3 InOut:6
   Emb:64 Inter:10
   ★ Forward Selection이 전체 후보에서 유효한 것 선별
📊 Train: 17,172,747 → 619,437 (4.76%)
📊 Val: 5,790,644
📊 Test: 5,792,913
  💾 [XGBoost 준비] RAM: 108.83GB | GPU: 0.02GB

🔥 [Step 8] 3-Stage XGBoost (Full → Forward Selection → HP Grid)...

── Stage 1: 전체 피처 학습 (importance 순위 산출) ──
[0]	validation_0-aucpr:0.11827
[100]	validation_0-aucpr:0.28521
[200]	validation_0-aucpr:0.38209
[300]	validation_0-aucpr:0.39981
[400]	validation_0-aucpr:0.41013
[500]	validation_0-aucpr:0.41501
[600]	validation_0-aucpr:0.42127
[700]	validation_0-aucpr:0.42349
[800]	validation_0-aucpr:0.428

In [8]:
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv
from torch_geometric.loader import NeighborLoader
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm
import gc, numpy as np, psutil, os, warnings
warnings.filterwarnings("ignore", category=UserWarning)

def log_memory(tag=""):
    rss = psutil.Process(os.getpid()).memory_info().rss/(1024**3)
    gpu = torch.cuda.memory_allocated()/(1024**3) if torch.cuda.is_available() else 0
    print(f"  💾 [{tag}] RAM: {rss:.2f}GB | GPU: {gpu:.2f}GB")

print("=" * 60)
print("🛡️ [TGAT v7c.11] GNN 수렴 강화 + FS 탐색 확대 + XGB 미세조정")
print("   ▸ GNN: epochs 60→100, LR 0.001→0.003 (수렴 속도 개선)")
print("   ▸ GNN: NUM_NEIGHBORS [15,10]→[25,15] (이웃 샘플 확대)")
print("   ▸ FS:  patience 8→12, MAX_FEATURES 100→150 (후반 탐색 강화)")
print("   ▸ XGB: d10_lr005 추가, d11_lr01 추가 (iter 한도 해소)")
print("=" * 60)

# ===================== 설정 =====================
NEG_NODE_RATIO_GNN = 25   # ★ 50→25: 이중 불균형 보정 완화, 더 다양한 음성 패턴 학습
NEG_ROW_RATIO_XGB = 20; RANDOM_SEED = 42
HIDDEN_DIM = 64; EDGE_DIM = 32   # ★ 16→32: 엣지 17차원 → 병목 해소
N_HEADS = 4
N_EPOCHS_GNN = 100            # ★ 60→100: Loss 아직 수렴 중 (Ep60=0.965)
BATCH_SIZE = 2048
NUM_NEIGHBORS = [25, 15]      # ★ [15,10]→[25,15]: 이웃 샘플 확대, 임베딩 품질 향상
N_INTERACTION_DIMS = 8
PATIENCE = 10
EARLY_STOP_DELTA = 0.001      # ★ 0.0005→0.001: 실질적 early stopping 작동
BASE_LR = 0.003               # ★ 0.001→0.003: 초기 수렴 속도 개선
WARMUP_EPOCHS = 5

raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"

# =============================================================
# 1. 데이터 로드
# =============================================================
print("\n📂 데이터 로딩 중...")
df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))
if df_raw["Timestamp"].dtype == pl.Utf8:
    df_raw = df_raw.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_v2 = df_v2.sort("time_group")
total_count = len(df_v2)
train_cutoff = df_v2["time_group"][int(total_count*0.6)]
val_cutoff = df_v2["time_group"][int(total_count*0.8)]
print(f"⏱️ Train cutoff: {train_cutoff}\n⏱️ Val   cutoff: {val_cutoff}")

all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")
num_nodes = len(all_accounts); print(f"📊 전체 노드 수: {num_nodes:,}")
log_memory("데이터 로드 완료")

# =============================================================
# 2. 모든 노드 피처 계산 (GNN 입력용 + XGBoost 겸용)
# =============================================================
print("\n📐 [Step 2] 전체 노드 피처 계산...")
df_raw_train = df_raw.filter(pl.col("Timestamp") < train_cutoff)

# ─── 2-1. 그래프 구조 피처 13개 ───
print("  📊 그래프 구조 피처...")
df_from = df_raw_train.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id"))
df_to = df_raw_train.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
out_degree = df_from.group_by("account_id").len().rename({"len":"graph_out_degree"})
in_degree = df_to.group_by("account_id").len().rename({"len":"graph_in_degree"})
unique_out = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("to_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_out_partners"))
unique_in = df_raw_train.select([pl.col("to_acc").cast(pl.Utf8).alias("account_id"),pl.col("from_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_in_partners"))
esf = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("a"),pl.col("to_acc").cast(pl.Utf8).alias("b")]).unique()
est = esf.select([pl.col("b").alias("a"),pl.col("a").alias("b")])
bidir = esf.join(est, on=["a","b"], how="inner")
bidir_count = bidir.select(pl.col("a").alias("account_id")).group_by("account_id").len().rename({"len":"graph_bidir_count"})
del esf, est, bidir; gc.collect()
time_features = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("Timestamp")]).group_by("account_id").agg([
    pl.col("Timestamp").sort().diff().dt.total_seconds().mean().alias("graph_avg_tx_interval"),
    pl.col("Timestamp").sort().diff().dt.total_seconds().std().alias("graph_std_tx_interval"),
    pl.col("Timestamp").count().alias("graph_total_tx_count")])

# PageRank
print("  📊 PageRank...")
account_to_id = dict(zip(all_accounts["account_id"].to_list(), all_accounts["node_id"].to_list()))
src_ids = df_raw_train["from_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
dst_ids = df_raw_train["to_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
vm=(src_ids>=0)&(dst_ids>=0); sv=src_ids[vm]; dv=dst_ids[vm]
oda=np.zeros(num_nodes,dtype=np.float64); np.add.at(oda,sv,1.0); oda=np.maximum(oda,1.0)
pr=np.ones(num_nodes,dtype=np.float64)/num_nodes
for _ in range(10): npr=np.ones(num_nodes,dtype=np.float64)*0.15/num_nodes; np.add.at(npr,dv,0.85*pr[sv]/oda[sv]); pr=npr
pr_df = pl.DataFrame({"node_id":np.arange(num_nodes,dtype=np.uint32),"graph_pagerank":pr.astype(np.float32)})
pr_df = all_accounts.join(pr_df,on="node_id",how="left").select(["account_id","graph_pagerank"])
del src_ids,dst_ids,sv,dv,oda,pr,npr; gc.collect()

df_graph_feats = (all_accounts.select("account_id").join(out_degree,on="account_id",how="left").join(in_degree,on="account_id",how="left")
    .join(unique_out,on="account_id",how="left").join(unique_in,on="account_id",how="left")
    .join(bidir_count,on="account_id",how="left").join(time_features,on="account_id",how="left")
    .join(pr_df,on="account_id",how="left").fill_null(0.0)
    .with_columns([(pl.col("graph_out_degree")+pl.col("graph_in_degree")).alias("graph_total_degree"),
        (pl.col("graph_out_degree")/(pl.col("graph_in_degree")+1)).alias("graph_out_in_ratio"),
        (pl.col("graph_bidir_count")/(pl.col("graph_out_degree")+1)).alias("graph_reciprocity"),
        (pl.col("graph_unique_out_partners")/(pl.col("graph_out_degree")+1)).alias("graph_partner_diversity")]))
graph_feature_cols = [c for c in df_graph_feats.columns if c.startswith("graph_")]
del out_degree,in_degree,unique_out,unique_in,bidir_count,time_features,pr_df,df_from,df_to; gc.collect()
print(f"  ✅ 그래프 피처 {len(graph_feature_cols)}개")

# ─── 2-2. 조건부행동 피처 10개 ───
print("  📊 조건부행동 피처...")
df_v2_train = df_v2.filter(pl.col("time_group")<train_cutoff)
exclude_cols = ["account_id","time_group","is_laundering","mode_format","currency_mode"]
gnn_feature_cols_v2 = [c for c in df_v2.columns if c not in exclude_cols]
df_v2_agg = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])

df_cond = df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
v2a=set(df_cond.columns)
def sc(n): return pl.col(n) if n in v2a else pl.lit(0.0)
ce=[]
if "cnt_night" in v2a and "cnt_1h" in v2a: ce.append((pl.col("graph_total_degree")*sc("cnt_night")/(sc("cnt_1h")+1)).alias("cond_highdeg_night_intensity"))
if "avg_log_amount" in v2a: ce.append((sc("avg_log_amount")/(pl.col("graph_total_degree").log1p()+1)).alias("cond_lowdeg_high_amount"))
if "ratio_cross_border" in v2a: ce.append((sc("ratio_cross_border")*(1-pl.col("graph_reciprocity"))).alias("cond_oneway_crossborder"))
if "cnt_1h" in v2a: ce.append((pl.col("graph_unique_out_partners")*sc("cnt_1h")/(pl.col("graph_out_degree")+1)).alias("cond_fanout_burst"))
if "cnt_risk_format" in v2a: ce.append((pl.col("graph_pagerank")*sc("cnt_risk_format")).alias("cond_hub_risk_format"))
if "cnt_wire" in v2a: ce.append((pl.col("graph_out_in_ratio")*sc("cnt_wire")).alias("cond_asymmetric_wire"))
if "cnt_inter_bank" in v2a: ce.append((pl.col("graph_std_tx_interval").fill_null(0)/(pl.col("graph_avg_tx_interval").fill_null(1)+1)*sc("cnt_inter_bank")).alias("cond_irregular_interbank"))
if "cnt_currencies" in v2a: ce.append((pl.col("graph_partner_diversity")*sc("cnt_currencies")).alias("cond_diverse_partner_currency"))
if "amount_kurtosis" in v2a: ce.append((pl.col("graph_reciprocity")*sc("amount_kurtosis")).alias("cond_bidir_amount_volatility"))
if "entity_acct_cnt" in v2a and "burst_ratio" in v2a: ce.append((sc("entity_acct_cnt")*sc("burst_ratio")).alias("cond_multi_acct_burst"))
df_cond=df_cond.with_columns(ce); cond_feature_cols=[e.meta.output_name() for e in ce]
for cn in cond_feature_cols: df_cond=df_cond.with_columns(pl.when(pl.col(cn).is_infinite()|pl.col(cn).is_nan()).then(0.0).otherwise(pl.col(cn)).alias(cn))
df_cond_features=df_cond.select(["account_id"]+cond_feature_cols); del df_cond; gc.collect()
print(f"  ✅ 조건부행동 {len(cond_feature_cols)}개")

# ─── 2-3. AML 점수 5개 ───
print("  📊 AML 점수...")
df_sb=df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
se=[((sc("cnt_1h").fill_null(0).clip(0,1e6)+1).log1p()*pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("burst_ratio").fill_null(0).clip(0,1e6)/(sc("avg_log_amount").fill_null(1).clip(0,1e6)+1)).alias("score_smurf"),
    ((1.0/((pl.col("graph_out_in_ratio").clip(0.01,100)-1.0).abs()+0.1))*sc("cnt_wire").fill_null(0).clip(0,1e6).log1p()*(1-pl.col("graph_reciprocity").clip(0,1))/(pl.col("graph_unique_out_partners").clip(0,1e6).log1p()+1)).alias("score_mule"),
    (pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("cnt_currencies").fill_null(0).clip(0,1e6).log1p()*sc("ratio_cross_border").fill_null(0).clip(0,1)/(pl.col("graph_avg_tx_interval").fill_null(3600).clip(0,1e9)/3600+1)).alias("score_layering"),
    (pl.col("graph_in_degree").clip(0,1e6).log1p()*(pl.col("graph_pagerank").clip(0,1)*1e6)*sc("cnt_risk_format").fill_null(0).clip(0,1e6).log1p()/(pl.col("graph_out_degree").clip(0,1e6).log1p()+1)).alias("score_integration")]
df_sb=df_sb.with_columns(se)
sn=["score_smurf","score_mule","score_layering","score_integration"]
for s in sn: df_sb=df_sb.with_columns(pl.when(pl.col(s).is_infinite()|pl.col(s).is_nan()).then(0.0).otherwise(pl.col(s)).alias(s))
for s in sn:
    mn=df_sb[s].min();mx=df_sb[s].max();rng=mx-mn if(mx-mn)>0 else 1.0
    df_sb=df_sb.with_columns(((pl.col(s)-mn)/rng).alias(f"{s}_norm"))
df_sb=df_sb.with_columns((pl.col("score_smurf_norm")*0.3+pl.col("score_mule_norm")*0.25+pl.col("score_layering_norm")*0.3+pl.col("score_integration_norm")*0.15).alias("score_composite"))
aml_score_cols=sn+["score_composite"]
df_aml_scores=df_sb.select(["account_id"]+aml_score_cols)
del df_sb, df_v2_agg; gc.collect()
print(f"  ✅ AML 점수 {len(aml_score_cols)}개")

# ─── 2-4. 규칙성 피처 7개 ───
print("  📊 규칙성 피처...")
df_tx_time = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner"),
    pl.col("Timestamp"),
    pl.col("Timestamp").dt.hour().alias("hour"),
    pl.col("Timestamp").dt.weekday().alias("dow"),
])
reg_agg = df_tx_time.group_by("account_id").agg([
    (pl.col("hour").is_between(9,17).mean()).alias("reg_business_hour_ratio"),
    (pl.col("dow").is_in([5,6]).mean()).alias("reg_weekend_ratio"),
    (pl.col("hour").is_between(0,5).mean()).alias("reg_night_strict_ratio"),
    pl.col("partner").count().alias("_total_tx"),
])
hc=df_tx_time.group_by(["account_id","hour"]).len().rename({"len":"h_count"})
hp=hc.pivot(on="hour",index="account_id",values="h_count").fill_null(0)
hcols=[c for c in hp.columns if c!="account_id"]
hmat=hp.select(hcols).to_numpy().astype(np.float64); hsum=np.maximum(hmat.sum(axis=1,keepdims=True),1.0)
hprobs=hmat/hsum; hlog=np.where(hprobs>0,np.log(hprobs+1e-12),0.0)
df_h_ent=pl.DataFrame({"account_id":hp["account_id"],"reg_hour_entropy":-np.sum(hprobs*hlog,axis=1).astype(np.float32)})
del hc,hp,hmat,hprobs,hlog; gc.collect()
dc=df_tx_time.group_by(["account_id","dow"]).len().rename({"len":"d_count"})
dp=dc.pivot(on="dow",index="account_id",values="d_count").fill_null(0)
dcols=[c for c in dp.columns if c!="account_id"]
dmat=dp.select(dcols).to_numpy().astype(np.float64); dsum=np.maximum(dmat.sum(axis=1,keepdims=True),1.0)
dprobs=dmat/dsum; dlog=np.where(dprobs>0,np.log(dprobs+1e-12),0.0)
df_d_ent=pl.DataFrame({"account_id":dp["account_id"],"reg_dow_entropy":-np.sum(dprobs*dlog,axis=1).astype(np.float32)})
del dc,dp,dmat,dprobs,dlog; gc.collect()
pf=df_tx_time.group_by(["account_id","partner"]).len().rename({"len":"p_count"})
rp=pf.filter(pl.col("p_count")>=2).group_by("account_id").len().rename({"len":"repeat_cnt"})
tp2=pf.group_by("account_id").len().rename({"len":"total_cnt"})
df_pc=tp2.join(rp,on="account_id",how="left").fill_null(0).with_columns((pl.col("repeat_cnt")/(pl.col("total_cnt")+1)).alias("reg_consistent_partner_ratio")).select(["account_id","reg_consistent_partner_ratio"])
top_p=pf.sort(["account_id","p_count"],descending=[False,True]).unique(subset=["account_id"],maintain_order=True).rename({"p_count":"top_cnt"}).select(["account_id","top_cnt"])
df_tc=reg_agg.select(["account_id","_total_tx"]).join(top_p,on="account_id",how="left").fill_null(0).with_columns((pl.col("top_cnt")/(pl.col("_total_tx")+1)).alias("reg_top_partner_concentration")).select(["account_id","reg_top_partner_concentration"])
del pf,rp,tp2,top_p; gc.collect()
df_regularity=(reg_agg.select(["account_id","reg_business_hour_ratio","reg_weekend_ratio","reg_night_strict_ratio"])
    .join(df_h_ent,on="account_id",how="left").join(df_d_ent,on="account_id",how="left")
    .join(df_pc,on="account_id",how="left").join(df_tc,on="account_id",how="left").fill_null(0.0))
regularity_cols=[c for c in df_regularity.columns if c.startswith("reg_")]
for rc in regularity_cols: df_regularity=df_regularity.with_columns(pl.when(pl.col(rc).is_infinite()|pl.col(rc).is_nan()).then(0.0).otherwise(pl.col(rc)).alias(rc))
del reg_agg,df_h_ent,df_d_ent,df_pc,df_tc; gc.collect()
print(f"  ✅ 규칙성 {len(regularity_cols)}개")

# ─── 2-5. 금액 분포 피처 8개 ───
print("  📊 금액 분포 피처...")
amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train.columns: amount_col = cand; break
if amount_col:
    df_amounts=df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col(amount_col).cast(pl.Float64).alias("amount")]).filter(pl.col("amount")>0)
    df_amt=df_amounts.group_by("account_id").agg([
        ((pl.col("amount")%1000==0)|(pl.col("amount")%5000==0)).mean().alias("amt_round_number_ratio"),
        (pl.col("amount").std()/(pl.col("amount").mean()+1)).alias("amt_cv"),
        (pl.col("amount").max()/(pl.col("amount").mean()+1)).alias("amt_max_mean_ratio"),
        (pl.col("amount").median()/(pl.col("amount").mean()+1)).alias("amt_median_mean_ratio"),
        pl.col("amount").quantile(0.9).alias("_q90"), pl.col("amount").sum().alias("_tsum"),
        pl.col("amount").count().alias("_cnt"),
        pl.col("amount").quantile(0.75).alias("_q75"), pl.col("amount").quantile(0.25).alias("_q25"),
        pl.col("amount").median().alias("_med"),
    ]).fill_null(0.0).with_columns([
        (pl.col("_q90")*0.1*pl.col("_cnt")/(pl.col("_tsum")+1)).alias("amt_top10pct_concentration"),
        ((pl.col("_q75")-pl.col("_q25"))/(pl.col("_med")+1)).alias("amt_iqr_median_ratio")])
    gm=df_amounts["amount"].mean(); st=max(gm*0.1,1.0)
    df_sr=df_amounts.with_columns((pl.col("amount")<st).alias("is_small")).group_by("account_id").agg(pl.col("is_small").mean().alias("amt_small_tx_ratio"))
    df_amt=df_amt.join(df_sr,on="account_id",how="left").fill_null(0.0)
    df_al=df_amounts.with_columns((pl.col("amount")+1).log().cast(pl.Int32).alias("log_bucket"))
    bc=df_al.group_by(["account_id","log_bucket"]).len().rename({"len":"b_count"})
    bt=bc.group_by("account_id").agg(pl.col("b_count").sum().alias("b_total"))
    bp=bc.join(bt,on="account_id").with_columns((pl.col("b_count")/pl.col("b_total")).alias("b_prob"))
    ae=bp.group_by("account_id").agg((-1*(pl.col("b_prob")*(pl.col("b_prob")+1e-12).log()).sum()).alias("amt_entropy"))
    df_amt=df_amt.join(ae,on="account_id",how="left").fill_null(0.0)
    del df_amounts,df_al,bc,bt,bp,ae,df_sr; gc.collect()
    amt_feature_cols=[c for c in df_amt.columns if c.startswith("amt_") and not c.startswith("_")]
    df_amt_features=df_amt.select(["account_id"]+amt_feature_cols)
else:
    df_amt_features=all_accounts.select("account_id"); amt_feature_cols=[]
for ac in amt_feature_cols:
    df_amt_features=df_amt_features.with_columns(pl.when(pl.col(ac).is_infinite()|pl.col(ac).is_nan()).then(0.0).otherwise(pl.col(ac)).alias(ac))
print(f"  ✅ 금액 분포 {len(amt_feature_cols)}개")

# =============================================================
# ★ NEW: 2-6. Cross-border 정밀화 피처 4개
# =============================================================
# 핵심 아이디어: ratio_cross_border 단독이 아니라
#   "cross-border × 새로운 상대방 비율 × 방향 비대칭"을 결합
print("  📊 ★ Cross-border 정밀화 피처 (신규)...")
cb_feature_cols = []

# 데이터에 to_acc country/bank 정보가 있을 경우를 가정
# → 없으면 graph_partner_diversity 와 ratio_cross_border 조합으로 근사
df_v2_agg_cb = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2 if c in df_v2_train.columns
])
df_cb = df_v2_agg_cb.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)
v2cb = set(df_cb.columns)

cb_exprs = []
# ① cb_new_partner_ratio: cross-border이면서 새로운 상대방 비율
#    = ratio_cross_border × (unique_out_partners / (total_tx_count+1))
#    → 새로운 상대에게 해외 송금하는 패턴
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("graph_unique_out_partners") /
         (pl.col("graph_total_tx_count") + 1)
        ).alias("cb_new_partner_ratio")
    )
    cb_feature_cols.append("cb_new_partner_ratio")

# ② cb_oneway_new: cross-border × 비양방향 × 새 상대방
#    → 양방향 거래 없이 새로운 상대에게만 해외 송금 (전형적 분산 송금)
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         (1.0 - pl.col("graph_reciprocity")) *
         (pl.col("graph_unique_out_partners") / (pl.col("graph_out_degree") + 1))
        ).alias("cb_oneway_new")
    )
    cb_feature_cols.append("cb_oneway_new")

# ③ cb_amount_asymmetry: 해외 송금 비율 × 금액 비대칭 (out >> in)
#    = ratio_cross_border × out_in_ratio (clip 해서 폭발 방지)
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("graph_out_in_ratio").clip(0, 20)
        ).alias("cb_amount_asymmetry")
    )
    cb_feature_cols.append("cb_amount_asymmetry")

# ④ cb_burst_cross: cross-border이 짧은 시간 내 집중 (burst_ratio 결합)
if "ratio_cross_border" in v2cb and "burst_ratio" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("burst_ratio").clip(0, 100)
        ).alias("cb_burst_cross")
    )
    cb_feature_cols.append("cb_burst_cross")

if cb_exprs:
    df_cb = df_cb.with_columns(cb_exprs)
    for cn in cb_feature_cols:
        df_cb = df_cb.with_columns(
            pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
            .then(0.0).otherwise(pl.col(cn)).alias(cn))
    df_cb_features = df_cb.select(["account_id"] + cb_feature_cols)
else:
    df_cb_features = all_accounts.select("account_id"); cb_feature_cols = []
del df_cb, df_v2_agg_cb; gc.collect()
print(f"  ✅ Cross-border 정밀화 {len(cb_feature_cols)}개: {cb_feature_cols}")

# trend_* 피처 제거 (ws_* 와 중복) — v7c.4
trend_feature_cols = []

# =============================================================
# ★ NEW: 2-8. 거래량 대비 상대 정규화 피처 5개
# =============================================================
# 핵심 아이디어: cond_lowdeg_high_amount가 FN에서 TP보다 오히려 높아 역전됨
#   → "전체 네트워크 대비 비정상적으로 높은 금액"을 더 정밀하게 포착
print("  📊 ★ 거래량 대비 상대 정규화 피처 (신규)...")
vol_norm_feature_cols = []

df_v2_agg_vn = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2 if c in df_v2_train.columns
])
df_vn = df_v2_agg_vn.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)

# 네트워크 전체 평균값 계산 (정규화 기준)
if "avg_log_amount" in set(df_vn.columns):
    global_avg_log_amount = df_vn["avg_log_amount"].mean() or 1.0
    global_avg_degree = df_vn["graph_total_degree"].mean() or 1.0
    global_avg_cnt_1h = df_vn["cnt_1h"].mean() if "cnt_1h" in df_vn.columns else 1.0
    global_avg_cnt_1h = global_avg_cnt_1h or 1.0

    vn_exprs = []
    # ① vol_amount_per_peer: 건당 금액 / (파트너 수 × 전체 평균)
    #    → degree가 낮은데 금액이 높은 패턴을 전체 대비 상대적으로 포착
    vn_exprs.append(
        (pl.col("avg_log_amount") / (float(global_avg_log_amount) + 1e-8) /
         (pl.col("graph_unique_out_partners").log1p() + 1)
        ).alias("vol_amount_per_peer")
    )
    vol_norm_feature_cols.append("vol_amount_per_peer")

    # ② vol_concentrated_amount: 소수 파트너에게 집중 송금
    #    = avg_log_amount × (1 - partner_diversity)
    #    → diversity가 낮을수록 (=한 곳에 집중) × 금액이 높을수록 높아짐
    vn_exprs.append(
        (pl.col("avg_log_amount") * (1.0 - pl.col("graph_partner_diversity").clip(0, 1))
        ).alias("vol_concentrated_amount")
    )
    vol_norm_feature_cols.append("vol_concentrated_amount")

    # ③ vol_low_freq_high_amt: 낮은 빈도이지만 건당 금액이 전체 평균 대비 높음
    #    = (avg_log_amount / global_mean) / log1p(cnt_1h) → 드문데 고액
    if "cnt_1h" in set(df_vn.columns):
        vn_exprs.append(
            (pl.col("avg_log_amount") / float(global_avg_log_amount) /
             (pl.col("cnt_1h").clip(0, 1e6).log1p() + 1)
            ).alias("vol_low_freq_high_amt")
        )
        vol_norm_feature_cols.append("vol_low_freq_high_amt")

    # ④ vol_degree_normalized_amount: 디그리로 나눈 상대 금액 (올바른 역산 버전)
    #    기존 cond_lowdeg_high_amount 대체 — log 스케일 적용 + 전체 평균으로 나눔
    vn_exprs.append(
        ((pl.col("avg_log_amount") - float(global_avg_log_amount)) /
         (pl.col("graph_total_degree").clip(1, 1e6).log1p())
        ).alias("vol_degree_normalized_amount")
    )
    vol_norm_feature_cols.append("vol_degree_normalized_amount")

    # ⑤ vol_outlier_score: z-score 방식의 금액 이상 점수
    #    = (avg_log_amount - global_mean) → 양수면 전체 평균 초과
    vn_exprs.append(
        (pl.col("avg_log_amount") - float(global_avg_log_amount)).alias("vol_outlier_score")
    )
    vol_norm_feature_cols.append("vol_outlier_score")

    df_vn = df_vn.with_columns(vn_exprs)
    for cn in vol_norm_feature_cols:
        df_vn = df_vn.with_columns(
            pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
            .then(0.0).otherwise(pl.col(cn)).alias(cn))
    df_vol_norm_features = df_vn.select(["account_id"] + vol_norm_feature_cols)
else:
    df_vol_norm_features = all_accounts.select("account_id"); vol_norm_feature_cols = []

del df_vn, df_v2_agg_vn; gc.collect()
print(f"  ✅ 거래량 대비 정규화 {len(vol_norm_feature_cols)}개: {vol_norm_feature_cols}")

# =============================================================
# ★ NEW: 2-9. 시간 윈도우별 행동 변화 피처 4개
# =============================================================
# 핵심 아이디어: 미탐의 71%가 테스트 첫 3일 집중
#   → train 말미(~recent window)에서의 행동 변화를 포착
print("  📊 ★ 시간 윈도우별 행동 변화 피처 (신규)...")
win_shift_feature_cols = []

# train 마지막 7일 vs 나머지 기간 비교
last_7d_cutoff = train_cutoff - pl.duration(days=7)
df_raw_recent = df_raw_train.filter(pl.col("Timestamp") >= last_7d_cutoff)
df_raw_older  = df_raw_train.filter(pl.col("Timestamp") < last_7d_cutoff)

recent_cnt = df_raw_recent.select(
    pl.col("from_acc").cast(pl.Utf8).alias("account_id")
).group_by("account_id").len().rename({"len": "ws_recent_cnt"})
older_cnt = df_raw_older.select(
    pl.col("from_acc").cast(pl.Utf8).alias("account_id")
).group_by("account_id").len().rename({"len": "ws_older_cnt"})

df_ws = all_accounts.select("account_id").join(recent_cnt, on="account_id", how="left").join(older_cnt, on="account_id", how="left").fill_null(0.0)

ws_exprs = []
# ① ws_recent_surge: 최근 7일 / 이전 기간 (정규화) → 갑작스러운 증가
ws_exprs.append(
    (pl.col("ws_recent_cnt") / (pl.col("ws_older_cnt") / 30.0 + 1)).alias("ws_recent_surge")
)
win_shift_feature_cols.append("ws_recent_surge")

# ② ws_recent_only: 최근 7일에만 거래가 있는 계좌 (0/1)
#    → 갑자기 나타난 계좌
ws_exprs.append(
    ((pl.col("ws_recent_cnt") > 0) & (pl.col("ws_older_cnt") == 0)).cast(pl.Float64).alias("ws_recent_only")
)
win_shift_feature_cols.append("ws_recent_only")

df_ws = df_ws.with_columns(ws_exprs)

# ③ 최근 7일 고유 파트너 급증
recent_partners = df_raw_recent.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner")
]).group_by("account_id").agg(pl.col("partner").n_unique().alias("ws_recent_partners"))
older_partners = df_raw_older.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner")
]).group_by("account_id").agg(pl.col("partner").n_unique().alias("ws_older_partners"))

df_ws = df_ws.join(recent_partners, on="account_id", how="left").join(older_partners, on="account_id", how="left").fill_null(0.0)

# ④ ws_partner_explosion: 최근 7일 파트너가 이전 대비 급증
df_ws = df_ws.with_columns(
    (pl.col("ws_recent_partners") / (pl.col("ws_older_partners") + 1)).alias("ws_partner_explosion")
)
win_shift_feature_cols.append("ws_partner_explosion")

for cn in win_shift_feature_cols:
    df_ws = df_ws.with_columns(
        pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
        .then(0.0).otherwise(pl.col(cn)).alias(cn))

df_win_shift_features = df_ws.select(["account_id"] + win_shift_feature_cols)
del df_raw_recent, df_raw_older, df_ws, recent_cnt, older_cnt
del recent_partners, older_partners; gc.collect()
print(f"  ✅ 시간 윈도우 변화 {len(win_shift_feature_cols)}개: {win_shift_feature_cols}")

# =============================================================
# ★ NEW: 2-10. 입금/출금 독립 피처 6개
# =============================================================
# 핵심 아이디어: V2에 sum_in_1h/sum_out_1h는 있지만
#   AML 점수/조건부 피처에서 in/out 독립적 분석이 부족
print("  📊 ★ 입금/출금 독립 피처 (신규)...")
inout_feature_cols = []

# 출금(from) 집계
df_out_tx = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col(amount_col).cast(pl.Float64).alias("amount") if amount_col else pl.lit(1.0).alias("amount")
]).filter(pl.col("amount") > 0)

# 입금(to) 집계
df_in_tx = df_raw_train.select([
    pl.col("to_acc").cast(pl.Utf8).alias("account_id"),
    pl.col(amount_col).cast(pl.Float64).alias("amount") if amount_col else pl.lit(1.0).alias("amount")
]).filter(pl.col("amount") > 0)

out_stats = df_out_tx.group_by("account_id").agg([
    pl.col("amount").sum().alias("inout_total_out_amount"),
    pl.col("amount").mean().alias("inout_avg_out_amount"),
    pl.col("amount").count().alias("inout_out_cnt"),
])
in_stats = df_in_tx.group_by("account_id").agg([
    pl.col("amount").sum().alias("inout_total_in_amount"),
    pl.col("amount").mean().alias("inout_avg_in_amount"),
    pl.col("amount").count().alias("inout_in_cnt"),
])
df_inout = all_accounts.select("account_id").join(out_stats, on="account_id", how="left").join(in_stats, on="account_id", how="left").fill_null(0.0)

inout_exprs = []
# ① inout_out_amount_ratio: 출금액 / (입금액 + 출금액) → 1이면 순수 송금자
inout_exprs.append(
    (pl.col("inout_total_out_amount") /
     (pl.col("inout_total_out_amount") + pl.col("inout_total_in_amount") + 1)
    ).alias("inout_out_amount_ratio")
)
inout_feature_cols.append("inout_out_amount_ratio")

# ② inout_avg_out_vs_in: 평균 출금액 / 평균 입금액 → 1 초과면 출금이 더 큰 계좌
inout_exprs.append(
    (pl.col("inout_avg_out_amount") / (pl.col("inout_avg_in_amount") + 1)
    ).alias("inout_avg_out_vs_in")
)
inout_feature_cols.append("inout_avg_out_vs_in")

# ③ inout_cnt_imbalance: (출금 건수 - 입금 건수) / 전체 건수
#    → 양수면 보내는 쪽이 훨씬 많은 계좌 (smurf 패턴)
inout_exprs.append(
    ((pl.col("inout_out_cnt") - pl.col("inout_in_cnt")) /
     (pl.col("inout_out_cnt") + pl.col("inout_in_cnt") + 1)
    ).alias("inout_cnt_imbalance")
)
inout_feature_cols.append("inout_cnt_imbalance")

# ④ inout_fan_out_score: 출금 건수 × 고유 파트너 다양성 → 여러 곳에 분산 송금
df_inout = df_inout.join(df_graph_feats.select(["account_id", "graph_partner_diversity", "graph_unique_out_partners"]), on="account_id", how="left").fill_null(0.0)
inout_exprs.append(
    (pl.col("inout_out_cnt").log1p() * pl.col("graph_partner_diversity")
    ).alias("inout_fan_out_score")
)
inout_feature_cols.append("inout_fan_out_score")

# ⑤ inout_in_concentration: 입금이 소수에서만 들어옴 (mule 패턴)
#    = avg_in_amount × (1 - partner_diversity) → 소수 소스에서 집중 수취
inout_exprs.append(
    (pl.col("inout_avg_in_amount").log1p() * (1.0 - pl.col("graph_partner_diversity").clip(0, 1))
    ).alias("inout_in_concentration")
)
inout_feature_cols.append("inout_in_concentration")

# ⑥ inout_net_flow: log(출금합) - log(입금합) → 순 흐름 방향성
inout_exprs.append(
    (pl.col("inout_total_out_amount").log1p() - pl.col("inout_total_in_amount").log1p()
    ).alias("inout_net_flow")
)
inout_feature_cols.append("inout_net_flow")

df_inout = df_inout.with_columns(inout_exprs)
for cn in inout_feature_cols:
    df_inout = df_inout.with_columns(
        pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
        .then(0.0).otherwise(pl.col(cn)).alias(cn))
df_inout_features = df_inout.select(["account_id"] + inout_feature_cols)
del df_out_tx, df_in_tx, out_stats, in_stats, df_inout; gc.collect()
print(f"  ✅ 입금/출금 독립 {len(inout_feature_cols)}개: {inout_feature_cols}")
log_memory("전체 노드 피처 완료")

# =============================================================
# 3. GNN 엣지/노드 구성 — 강화된 입력
# =============================================================
print("\n🔗 [Step 3] GNN 엣지/노드 구성 (Enriched Input)...")

df_raw_reload = pl.read_parquet(raw_path)
if df_raw_reload["Timestamp"].dtype == pl.Utf8:
    df_raw_reload = df_raw_reload.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_raw_train_edges = df_raw_reload.filter(pl.col("Timestamp")<train_cutoff); del df_raw_reload; gc.collect()

edge_amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train_edges.columns: edge_amount_col = cand; break

df_edges = df_raw_train_edges.select(
    ["from_acc","to_acc","Timestamp"] +
    ([edge_amount_col] if edge_amount_col else []) +
    (["Payment Format"] if "Payment Format" in df_raw_train_edges.columns else []) +
    (["Is Laundering"] if "Is Laundering" in df_raw_train_edges.columns else [])
).with_columns([pl.col("from_acc").cast(pl.Utf8), pl.col("to_acc").cast(pl.Utf8)])

df_edges = df_edges.join(all_accounts.rename({"account_id":"from_acc","node_id":"src_id"}),on="from_acc",how="left")
df_edges = df_edges.join(all_accounts.rename({"account_id":"to_acc","node_id":"dst_id"}),on="to_acc",how="left")

# ── 시간 피처 (5개, 기존 유지) ──────────────────────────────────────────
ts = df_edges["Timestamp"]; mn_ts=ts.min(); mx_ts=ts.max(); ts_rng=(mx_ts-mn_ts).total_seconds()
etn = ((ts-mn_ts).dt.total_microseconds().cast(pl.Float64)/1e6/max(ts_rng,1.0)).to_numpy().astype(np.float32)
hrs = df_edges["Timestamp"].dt.hour().to_numpy().astype(np.float32)
dow = df_edges["Timestamp"].dt.weekday().to_numpy().astype(np.float32)
time_feats = np.stack([etn, np.sin(2*np.pi*hrs/24), np.cos(2*np.pi*hrs/24),
                        np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7)], axis=1)

# ── 금액 피처 (3개 → 4개) ─────────────────────────────────────────────
# ★ 변경: round_flag를 1000/5000 → 100단위로 확장 (소액 스머핑 감지)
# ★ 추가: amount_pct — 전체 거래 중 분위수 위치 (극단적 고액 거래 포착)
if edge_amount_col:
    amounts_raw = df_edges[edge_amount_col].cast(pl.Float64).fill_null(0.0).to_numpy().astype(np.float32)
    log_amounts  = np.log1p(np.maximum(amounts_raw, 0)).astype(np.float32)
    log_amounts  = log_amounts / (np.max(log_amounts) + 1e-8)
    round_flag   = ((amounts_raw % 100 == 0) & (amounts_raw > 0)).astype(np.float32)
    global_mean_amt = np.mean(amounts_raw[amounts_raw > 0]) if np.any(amounts_raw > 0) else 1.0
    relative_amt = np.log1p(amounts_raw / (global_mean_amt + 1e-8)).astype(np.float32)
    relative_amt = relative_amt / (np.max(relative_amt) + 1e-8)
    nonzero_amt  = amounts_raw[amounts_raw > 0]
    pct_vals     = np.percentile(nonzero_amt, np.linspace(0,100,1001)) if len(nonzero_amt)>0 else np.zeros(1001)
    amount_pct   = np.clip(np.searchsorted(pct_vals, amounts_raw, side='right').astype(np.float32)/1000.0, 0.0, 1.0)
    amount_feats = np.stack([log_amounts, round_flag, relative_amt, amount_pct], axis=1)
else:
    amount_feats = np.zeros((len(df_edges), 4), dtype=np.float32)

# ── Payment Format 피처 (이진 정규식 2개 → 정밀 인코딩 9개) ──────────
# ★ 실제 분포 확인 결과 (HI-Medium, 7종):
#   Cheque      38.50%  세탁율 0.02%   │ ACH      12.13%  세탁율 0.79% ← 압도적 고위험
#   Credit Card 27.52%  세탁율 0.02%   │ Cash     10.09%  세탁율 0.02%
#   Reinvestment 6.10%  세탁율 0.00%   │ Wire      3.51%  세탁율 0.00%
#   Bitcoin      2.16%  세탁율 0.04%
#
# 설계 결정:
#   ① 7종 전부 원-핫 (포맷 수가 적어 "기타" 불필요)
#   ② AML 위험도 연속값 1개 (ACH=0.0079, 나머지≈0 → 강한 분리 신호)
#   ③ 기존 is_wire(정규식) 완전 제거 — Wire 세탁율 0.00%로 노이즈였음
PAYMENT_FORMATS = ["Cheque", "Credit Card", "ACH", "Cash", "Reinvestment", "Wire", "Bitcoin"]
# 포맷별 전체 데이터 기준 AML 위험도 (사전 계산값, 학습 데이터 누수 방지)
FORMAT_RISK_MAP  = {"Cheque":0.0002, "Credit Card":0.0002, "ACH":0.0079,
                    "Cash":0.0002, "Reinvestment":0.0000, "Wire":0.0000, "Bitcoin":0.0004}

if "Payment Format" in df_edges.columns:
    pf_arr = df_edges["Payment Format"].cast(pl.Utf8).fill_null("Unknown").to_numpy()

    # ① 7종 원-핫 (7차원)
    onehot_feats = np.zeros((len(pf_arr), len(PAYMENT_FORMATS)), dtype=np.float32)
    for j, fmt in enumerate(PAYMENT_FORMATS):
        onehot_feats[:, j] = (pf_arr == fmt).astype(np.float32)

    # ② AML 위험도 연속값 (1차원) — ACH가 다른 포맷 대비 ~40배 높아 강한 신호
    fmt_risk = np.array([FORMAT_RISK_MAP.get(f, 0.0002) for f in pf_arr], dtype=np.float32)

    type_feats   = np.concatenate([onehot_feats, fmt_risk.reshape(-1,1)], axis=1)  # 8차원
    N_TYPE_FEATS = type_feats.shape[1]
else:
    N_TYPE_FEATS = len(PAYMENT_FORMATS) + 1
    type_feats   = np.zeros((len(df_edges), N_TYPE_FEATS), dtype=np.float32)

edge_features_all = np.concatenate([time_feats, amount_feats, type_feats], axis=1)
EDGE_RAW_DIM = edge_features_all.shape[1]
print(f"  📊 Edge 피처: {EDGE_RAW_DIM}개")
print(f"     시간(5) + 금액(4) + 포맷({N_TYPE_FEATS}: 원-핫7+위험도1)")
print(f"     ※ 기존 is_wire/is_internal 정규식 → 7종 원-핫+위험도로 교체")

edge_index_train = torch.tensor(df_edges.select(["src_id","dst_id"]).to_numpy().T, dtype=torch.long)
edge_attr_train = torch.tensor(edge_features_all, dtype=torch.float32)
print(f"📊 Train 엣지: {edge_index_train.shape[1]:,}")

del df_raw_train, df_raw_train_edges, df_edges, ts, hrs, dow, etn
del time_feats, amount_feats, type_feats, edge_features_all; gc.collect()

# ─── 노드 피처: V2(38) + 그래프(13) + 조건부(10) + AML(5) + 규칙성(7) + 금액(8) = 81개
# ★ v7c.5: GNN 입력은 v7c.2와 동일하게 유지 (신규 피처는 XGBoost에만 투입)
print("  📊 노드 피처 구성 (GNN: v7c.2 동일 81차원)...")
df_v2_node = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])
df_node_all = (all_accounts.join(df_v2_node, on="account_id", how="left")
    .join(df_graph_feats, on="account_id", how="left")
    .join(df_cond_features, on="account_id", how="left")
    .join(df_aml_scores, on="account_id", how="left")
    .join(df_regularity, on="account_id", how="left")
    .join(df_amt_features, on="account_id", how="left")
    .fill_null(0.0))

# GNN 입력은 v7c.2 피처만 (81개) — 신규 피처는 XGBoost 단계에만 추가
gnn_input_cols = (gnn_feature_cols_v2 + graph_feature_cols + cond_feature_cols +
                  aml_score_cols + regularity_cols + amt_feature_cols)

for col in gnn_input_cols:
    if col in df_node_all.columns:
        df_node_all = df_node_all.with_columns(
            pl.when(pl.col(col).is_infinite()|pl.col(col).is_nan()).then(0.0).otherwise(pl.col(col)).alias(col))

X_node = torch.tensor(df_node_all.select(gnn_input_cols).to_numpy(), dtype=torch.float32)
IN_DIM = X_node.shape[1]
print(f"  ✅ GNN 노드 입력: {IN_DIM}차원 (v7c.2 동일)")
print(f"     = V2({len(gnn_feature_cols_v2)}) + 그래프({len(graph_feature_cols)}) + 조건부({len(cond_feature_cols)})")
print(f"       + AML({len(aml_score_cols)}) + 규칙성({len(regularity_cols)}) + 금액({len(amt_feature_cols)})")

target_labels = df_v2_train.filter(pl.col("is_laundering")==1).select("account_id").unique().with_columns(pl.lit(1).alias("label"))
df_labels = all_accounts.join(target_labels,on="account_id",how="left").fill_null(0)
Y_node = torch.tensor(df_labels["label"].to_numpy(), dtype=torch.long)
pos_node_ids = set(df_labels.filter(pl.col("label")==1)["node_id"].to_list())
active_df = pl.DataFrame({"account_id":df_v2_train["account_id"].unique(),"is_active":True})
mask_df = all_accounts.join(active_df,on="account_id",how="left").fill_null(False)
active_node_ids = mask_df.filter(pl.col("is_active"))["node_id"].to_list()
active_pos = [n for n in active_node_ids if n in pos_node_ids]
active_neg = [n for n in active_node_ids if n not in pos_node_ids]
n_neg_gnn = min(len(active_pos)*NEG_NODE_RATIO_GNN, len(active_neg))
np.random.seed(RANDOM_SEED)
sampled_neg = np.random.choice(active_neg,size=n_neg_gnn,replace=False).tolist()
sampled_nodes = set(active_pos+sampled_neg)
train_mask_np = np.zeros(num_nodes, dtype=bool)
for nid in sampled_nodes: train_mask_np[nid]=True
train_mask = torch.tensor(train_mask_np, dtype=torch.bool)
print(f"📊 GNN 노드: 양성 {len(active_pos):,} + 음성 {n_neg_gnn:,}")
del df_v2_node, df_node_all, mask_df, active_df; gc.collect()

graph_data = Data(x=X_node, edge_index=edge_index_train, edge_attr=edge_attr_train, y=Y_node)
graph_data.train_mask = train_mask; graph_data.num_nodes = num_nodes
del X_node, edge_index_train, edge_attr_train, Y_node, train_mask; gc.collect()
log_memory("GNN 데이터 준비 완료")

# =============================================================
# 4-6. Multi-Config GNN 탐색 → 최적 임베딩 선택
# =============================================================
print("\n🧠 [Step 4-6] Multi-Config GNN 탐색...")

class EdgeProjection(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, out_dim), nn.GELU(), nn.Linear(out_dim, out_dim))
    def forward(self, ea): return self.proj(ea)

class TGAT_Flex(nn.Module):
    def __init__(self, in_dim, hidden_dim, edge_raw_dim, edge_dim, n_heads, n_layers=2, dropout=0.2):
        super().__init__()
        self.edge_proj = EdgeProjection(edge_raw_dim, edge_dim)
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.n_layers = n_layers
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_layers)])
        self.convs = nn.ModuleList([
            TransformerConv(hidden_dim, hidden_dim//n_heads, heads=n_heads,
                            edge_dim=edge_dim, dropout=dropout, concat=True)
            for _ in range(n_layers)])
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim//2, 1))
        self.dropout = dropout; self.hidden_dim = hidden_dim

    def forward(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        for i in range(self.n_layers):
            h = h + F.dropout(self.convs[i](self.norms[i](h), edge_index, ee), p=self.dropout, training=self.training)
        return self.classifier(self.norm_out(h)).squeeze(-1)

    @torch.no_grad()
    def get_embedding(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        for i in range(self.n_layers):
            h = h + self.convs[i](self.norms[i](h), edge_index, ee)
        return self.norm_out(h)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"📌 디바이스: {device}")
pw = (len(sampled_nodes)-len(active_pos))/max(len(active_pos),1)
print(f"📊 pos_weight: {pw:.1f}")

def make_lr(opt, ep, n_ep):
    if ep < WARMUP_EPOCHS: lr = 1e-6+(BASE_LR-1e-6)*(ep/WARMUP_EPOCHS)
    else: p=(ep-WARMUP_EPOCHS)/max(n_ep-WARMUP_EPOCHS,1); lr=1e-5+(BASE_LR-1e-5)*0.5*(1+np.cos(np.pi*p))
    for pg in opt.param_groups: pg['lr']=lr
    return lr

GNN_CONFIGS = [
    {"label":"h64_L2",  "hidden_dim":64,  "n_layers":2, "n_heads":4, "edge_dim":32},  # ★ 16→32
    {"label":"h128_L2", "hidden_dim":128, "n_layers":2, "n_heads":4, "edge_dim":32},
    {"label":"h64_L3",  "hidden_dim":64,  "n_layers":3, "n_heads":4, "edge_dim":32},  # ★ 16→32
    {"label":"h128_L3", "hidden_dim":128, "n_layers":3, "n_heads":4, "edge_dim":32},
]

xgb_exclude_cols=["account_id","time_group","is_laundering","mode_format","currency_mode","date"]
xgb_v2_cols=[c for c in df_v2.columns if c not in xgb_exclude_cols]
apr_train = df_v2_train.filter(pl.col("is_laundering")==1).height / max(df_v2_train.height,1)
spw = max((1-apr_train)/apr_train, 1.0) if apr_train > 0 else 20.0

# ★ 신규 피처들을 base_feature_cols에 포함
base_feature_cols = (xgb_v2_cols + graph_feature_cols + cond_feature_cols + aml_score_cols +
                     cb_feature_cols + vol_norm_feature_cols +
                     win_shift_feature_cols + inout_feature_cols)

def build_eval_df(db, df_emb_local, emb_col_names):
    feature_cols = base_feature_cols + emb_col_names
    df = (db.join(df_graph_feats,on="account_id",how="left")
          .join(df_cond_features,on="account_id",how="left")
          .join(df_aml_scores,on="account_id",how="left")
          .join(df_cb_features,on="account_id",how="left")            # ★
          .join(df_vol_norm_features,on="account_id",how="left")      # ★
          .join(df_win_shift_features,on="account_id",how="left")     # ★
          .join(df_inout_features,on="account_id",how="left")         # ★
          .join(df_emb_local,on="account_id",how="left")
          .fill_null(0.0).fill_nan(0.0))
    for cn in feature_cols:
        if cn in df.columns:
            df = df.with_columns(pl.when(pl.col(cn).is_infinite()).then(0.0).otherwise(pl.col(cn)).alias(cn))
    return df, feature_cols

def quick_xgb_eval(df_emb_local, emb_col_names, label):
    df_tr, fcols = build_eval_df(df_v2.filter(pl.col("time_group")<train_cutoff), df_emb_local, emb_col_names)
    df_tr_pos=df_tr.filter(pl.col("is_laundering")==1); df_tr_neg=df_tr.filter(pl.col("is_laundering")==0)
    n_neg=min(len(df_tr_pos)*NEG_ROW_RATIO_XGB,len(df_tr_neg))
    df_tr_s=pl.concat([df_tr_pos,df_tr_neg.sample(n=n_neg,seed=RANDOM_SEED)]).sample(fraction=1.0,seed=RANDOM_SEED)
    X_tr=df_tr_s.select(fcols).to_pandas(); y_tr=df_tr_s["is_laundering"].to_numpy()
    del df_tr,df_tr_pos,df_tr_neg,df_tr_s; gc.collect()
    df_vl, _ = build_eval_df(df_v2.filter((pl.col("time_group")>=train_cutoff)&(pl.col("time_group")<val_cutoff)), df_emb_local, emb_col_names)
    X_vl=df_vl.select(fcols).to_pandas(); y_vl=df_vl["is_laundering"].to_numpy(); del df_vl; gc.collect()
    df_te, _ = build_eval_df(df_v2.filter(pl.col("time_group")>=val_cutoff), df_emb_local, emb_col_names)
    X_te=df_te.select(fcols).to_pandas(); y_te=df_te["is_laundering"].to_numpy(); del df_te; gc.collect()
    qp={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
        "random_state":42,"max_depth":8,"learning_rate":0.03,"scale_pos_weight":spw,
        "subsample":0.8,"colsample_bytree":0.7,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
    m=xgb.XGBClassifier(**qp,n_estimators=1500,early_stopping_rounds=100)
    m.fit(X_tr,y_tr,eval_set=[(X_vl,y_vl)],verbose=0)
    q_val=m.best_score; q_test=average_precision_score(y_te,m.predict_proba(X_te)[:,1])
    del m,X_tr,X_vl,X_te,y_tr,y_vl,y_te; gc.collect()
    return q_val, q_test

print(f"\n📊 {len(GNN_CONFIGS)}개 GNN 설정 탐색...")
print(f"  {'#':>2s} {'Label':<12s} | {'hidden':>6s} {'layers':>6s} {'edim':>4s} | {'Loss':>8s} {'Params':>8s} | {'QVal':>8s} {'QTest':>8s}")
print("  "+"-"*80)

gnn_results = []
for gi, gcfg in enumerate(GNN_CONFIGS):
    gl = gcfg["label"]; hd=gcfg["hidden_dim"]; nl=gcfg["n_layers"]
    nh=gcfg["n_heads"]; ed=gcfg["edge_dim"]
    train_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                                   input_nodes=graph_data.train_mask, shuffle=True, num_workers=4)
    full_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                                  input_nodes=None, shuffle=False, num_workers=4)
    model = TGAT_Flex(in_dim=IN_DIM, hidden_dim=hd, edge_raw_dim=EDGE_RAW_DIM,
                      edge_dim=ed, n_heads=nh, n_layers=nl, dropout=0.2).to(device)
    np_count = sum(p.numel() for p in model.parameters())
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw],device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6, weight_decay=1e-4)
    model.train(); bl=float('inf'); pc2=0; bs=None
    for epoch in range(N_EPOCHS_GNN):
        clr=make_lr(optimizer,epoch,N_EPOCHS_GNN); tl=0.0; nb2=0
        for batch in tqdm(train_loader,desc=f"[{gl}] Ep{epoch+1}",leave=False):
            batch=batch.to(device); optimizer.zero_grad()
            bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
            out=model(batch.x,batch.edge_index,bea)
            loss=criterion(out[:batch.batch_size],batch.y[:batch.batch_size].float())
            if torch.isnan(loss) or torch.isinf(loss): optimizer.zero_grad(); continue
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=0.5)
            optimizer.step(); tl+=loss.item(); nb2+=1
        al=tl/max(nb2,1)
        if (epoch+1)%10==0 or epoch==0: print(f"    [{gl}] Ep{epoch+1:2d} Loss={al:.4f}")
        if al<bl-EARLY_STOP_DELTA: bl=al; pc2=0; bs={k:v.cpu().clone() for k,v in model.state_dict().items()}
        else: pc2+=1
        if pc2>=PATIENCE: break
    if bs: model.load_state_dict({k:v.to(device) for k,v in bs.items()}); del bs
    model.eval(); ae=[]
    with torch.no_grad():
        for batch in tqdm(full_loader,desc=f"[{gl}] Embed",leave=False):
            batch=batch.to(device)
            bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
            emb=model.get_embedding(batch.x,batch.edge_index,bea)
            ae.append(emb[:batch.batch_size].cpu())
    femb=torch.cat(ae,dim=0).numpy()
    del model,ae,train_loader,full_loader; torch.cuda.empty_cache(); gc.collect()
    ecn=[f"tgat_emb_{i}" for i in range(hd)]
    df_emb=pl.concat([all_accounts.select("account_id"),pl.DataFrame(femb,schema=ecn)],how="horizontal")
    qv,qt = quick_xgb_eval(df_emb, ecn, gl)
    print(f"  {gi+1:2d} {gl:<12s} | {hd:6d} {nl:6d} {ed:4d} | {bl:8.4f} {np_count:8,} | {qv:8.5f} {qt:8.4f}")
    gnn_results.append({"label":gl,"hidden_dim":hd,"n_layers":nl,"loss":bl,"params":np_count,
                         "q_val":qv,"q_test":qt,"embedding":femb,"ecn":ecn,"df_emb":df_emb})
    log_memory(f"{gl}")

best_gnn = max(gnn_results, key=lambda x: x["q_val"])
print(f"\n  🏆 Best GNN: {best_gnn['label']} (QVal={best_gnn['q_val']:.5f} QTest={best_gnn['q_test']:.4f})")
for r in gnn_results:
    if r is not best_gnn: del r["embedding"], r["df_emb"]
gc.collect()

emb_cols = best_gnn["ecn"]
df_tgat = best_gnn["df_emb"]
FINAL_HIDDEN_DIM = best_gnn["hidden_dim"]

print(f"\n🔗 Interaction 피처 ({best_gnn['label']})...")
df_tgat_with_pr=df_tgat.join(df_graph_feats.select(["account_id","graph_pagerank"]),on="account_id",how="left").fill_null(0.0)
emb_np=df_tgat_with_pr.select(emb_cols).to_numpy(); emb_var=np.var(emb_np,axis=0)
top_idx=np.argsort(emb_var)[::-1][:N_INTERACTION_DIMS]; top_ec=[emb_cols[i] for i in top_idx]
ie_l=[]; ic_l=[]
for cn in top_ec: nn2=f"interact_pr_x_{cn}"; ie_l.append((pl.col(cn)*pl.col("graph_pagerank")*1e6).alias(nn2)); ic_l.append(nn2)
ie_l.append((sum(pl.col(c)**2 for c in emb_cols).sqrt()*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_norm")); ic_l.append("interact_pr_x_emb_norm")
ie_l.append((sum(pl.col(c) for c in top_ec)*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_topsum")); ic_l.append("interact_pr_x_emb_topsum")
interaction_cols=ic_l
df_interactions=df_tgat_with_pr.with_columns(ie_l).select(["account_id"]+ic_l)
for c in ic_l: df_interactions=df_interactions.with_columns(pl.when(pl.col(c).is_infinite()|pl.col(c).is_nan()).then(0.0).otherwise(pl.col(c)).alias(c))
print(f"  ✅ Interaction {len(ic_l)}개")
del df_tgat_with_pr,emb_np,best_gnn["embedding"]; gc.collect()
del graph_data; torch.cuda.empty_cache(); gc.collect()
log_memory("GNN 탐색 완료")

# =============================================================
# 7. XGBoost 데이터 구성
# =============================================================
print("\n🚀 [Step 7] XGBoost 데이터 구성...")

# ★ v7c.7: regularity(7) + amt(8) = 15개를 XGBoost 후보에 추가 (v7c.6 누락 수정)
# GNN 입력에는 이미 포함돼 있었지만 xgb_feature_cols에서 빠져 있었음
xgb_feature_cols = (xgb_v2_cols + graph_feature_cols + cond_feature_cols + aml_score_cols +
                    regularity_cols + amt_feature_cols +          # ★ 신규 추가
                    cb_feature_cols + vol_norm_feature_cols +
                    win_shift_feature_cols + inout_feature_cols +
                    emb_cols + interaction_cols)
selected_new_cols = cb_feature_cols + vol_norm_feature_cols + win_shift_feature_cols + inout_feature_cols
print(f"📊 XGBoost 총 후보 피처: {len(xgb_feature_cols)}")
print(f"   V2:{len(xgb_v2_cols)} G:{len(graph_feature_cols)} Cond:{len(cond_feature_cols)} AML:{len(aml_score_cols)}")
print(f"   ★추가 Reg:{len(regularity_cols)} Amt:{len(amt_feature_cols)}  (v7c.6 누락 수정)")
print(f"   CB:{len(cb_feature_cols)} VolNorm:{len(vol_norm_feature_cols)}")
print(f"   WinShift:{len(win_shift_feature_cols)} InOut:{len(inout_feature_cols)}")
print(f"   Emb:{len(emb_cols)} Inter:{len(interaction_cols)}")
print(f"   ★ Forward Selection이 전체 후보에서 유효한 것 선별")

def build_xgb_df(db):
    df=(db.join(df_graph_feats,on="account_id",how="left")
          .join(df_cond_features,on="account_id",how="left")
          .join(df_aml_scores,on="account_id",how="left")
          .join(df_regularity,on="account_id",how="left")        # ★ 신규 추가
          .join(df_amt_features,on="account_id",how="left")      # ★ 신규 추가
          .join(df_cb_features,on="account_id",how="left")
          .join(df_vol_norm_features,on="account_id",how="left")
          .join(df_win_shift_features,on="account_id",how="left")
          .join(df_inout_features,on="account_id",how="left")
          .join(df_tgat,on="account_id",how="left")
          .join(df_interactions,on="account_id",how="left")
          .fill_null(0.0).fill_nan(0.0))
    for cn in xgb_feature_cols:
        if cn in df.columns: df=df.with_columns(pl.when(pl.col(cn).is_infinite()).then(0.0).otherwise(pl.col(cn)).alias(cn))
    return df

df_tr=build_xgb_df(df_v2.filter(pl.col("time_group")<train_cutoff))
df_tr_pos=df_tr.filter(pl.col("is_laundering")==1); df_tr_neg=df_tr.filter(pl.col("is_laundering")==0)
n_neg=min(len(df_tr_pos)*NEG_ROW_RATIO_XGB,len(df_tr_neg))
df_tr_s=pl.concat([df_tr_pos,df_tr_neg.sample(n=n_neg,seed=RANDOM_SEED)]).sample(fraction=1.0,seed=RANDOM_SEED)
X_train=df_tr_s.select(xgb_feature_cols).to_pandas(); y_train=df_tr_s["is_laundering"].to_numpy()
print(f"📊 Train: {len(df_tr):,} → {len(df_tr_s):,} ({y_train.mean()*100:.2f}%)")
del df_tr,df_tr_pos,df_tr_neg,df_tr_s; gc.collect()
df_vl=build_xgb_df(df_v2.filter((pl.col("time_group")>=train_cutoff)&(pl.col("time_group")<val_cutoff)))
X_val=df_vl.select(xgb_feature_cols).to_pandas(); y_val=df_vl["is_laundering"].to_numpy(); print(f"📊 Val: {len(X_val):,}"); del df_vl; gc.collect()
df_te=build_xgb_df(df_v2.filter(pl.col("time_group")>=val_cutoff))
df_test_meta=df_te.select(["account_id","time_group","is_laundering"])
X_test=df_te.select(xgb_feature_cols).to_pandas(); y_test=df_te["is_laundering"].to_numpy(); print(f"📊 Test: {len(X_test):,}")
del df_te,df_v2,df_tgat,df_graph_feats,df_cond_features,df_aml_scores
del df_cb_features,df_vol_norm_features,df_win_shift_features,df_inout_features
del df_interactions,df_regularity,df_amt_features; gc.collect()
log_memory("XGBoost 준비")

# =============================================================
# 8. XGBoost 3-Stage: Full → Forward Selection → HP Grid
# =============================================================
# ★ v7c.6 핵심 변경: Stage 1 importance 기반 top-K 선택의 한계 해결
#   문제: 상관된 피처끼리 importance를 나눠 가져서 유용한 피처가 순위 밖으로 밀림
#   해결: Stage 2를 Forward Selection으로 교체
#     → 피처를 하나씩 추가하며 Val AUCPR이 실제로 오를 때만 채택
#     → 중복/노이즈 피처는 자동으로 탈락
print("\n🔥 [Step 8] 3-Stage XGBoost (Full → Forward Selection → HP Grid)...")
apr=y_train.sum()/len(y_train); spw=max((1-apr)/apr,1.0)

# ── Stage 1: 전체 피처로 학습 → importance 순위 산출 ──────────────────
print("\n── Stage 1: 전체 피처 학습 (importance 순위 산출) ──")
s1p={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
     "random_state":42,"max_depth":8,"learning_rate":0.03,"scale_pos_weight":spw,
     "subsample":0.8,"colsample_bytree":0.7,"min_child_weight":5,
     "gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
m_s1=xgb.XGBClassifier(**s1p,n_estimators=1500,early_stopping_rounds=100)
m_s1.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=100)
s1_val=m_s1.best_score
s1_test=average_precision_score(y_test,m_s1.predict_proba(X_test)[:,1])
print(f"\n  Stage1: Val={s1_val:.5f} Test={s1_test:.4f}")
imp_s1=m_s1.feature_importances_
# importance 순위 정렬 (Stage 2 Forward Selection의 탐색 순서로 사용)
feat_imp=sorted(zip(xgb_feature_cols,imp_s1),key=lambda x:x[1],reverse=True)
del m_s1; gc.collect()

# ── Stage 2: Forward Selection ─────────────────────────────────────────
# importance 상위 순서대로 피처를 하나씩 추가하며
# Val AUCPR이 오를 때만 채택, 연속 N_PATIENCE_FS번 개선 없으면 조기 종료
print("\n── Stage 2: Forward Selection ──")
print("  피처를 importance 순서대로 하나씩 추가 → Val 개선 시 채택")

N_PATIENCE_FS = 12          # ★ 8→12: 후반 피처 탐색 여유 확보
MIN_DELTA_FS  = 5e-5        # 최소 개선량 (노이즈 수준 변동 무시)
MAX_FEATURES_FS = 150       # ★ 100→150: v7c.10에서 98/100 조기 종료됨

fs_hp={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
       "random_state":42,"max_depth":9,"learning_rate":0.02,"scale_pos_weight":spw,
       "subsample":0.8,"colsample_bytree":0.8,"min_child_weight":5,
       "gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}

selected_fs = []          # 채택된 피처 누적
best_fs_val = -1.0        # 현재까지 best Val AUCPR
no_improve  = 0           # 연속 미개선 횟수
fs_log      = []          # 로그 기록

candidates = [f for f,_ in feat_imp[:MAX_FEATURES_FS]]

print(f"  {'#':>3s} {'Feature':<45s} | {'Val':>9s} {'Δ':>8s} {'채택':>4s}")
print("  " + "-" * 75)

for fi, feat in enumerate(candidates):
    probe = selected_fs + [feat]
    m_fs = xgb.XGBClassifier(**fs_hp, n_estimators=1500, early_stopping_rounds=100)  # ★ 800→1500
    m_fs.fit(X_train[probe], y_train, eval_set=[(X_val[probe], y_val)], verbose=False)
    v = m_fs.best_score
    delta = v - best_fs_val
    adopted = delta > MIN_DELTA_FS

    tag = "✅" if adopted else "  "
    print(f"  {fi+1:3d} {feat:<45s} | {v:9.5f} {delta:+8.5f} {tag}")
    fs_log.append({"feat":feat,"val":v,"delta":delta,"adopted":adopted})

    if adopted:
        selected_fs.append(feat)
        best_fs_val = v
        no_improve = 0
    else:
        no_improve += 1

    del m_fs; gc.collect()

    if no_improve >= N_PATIENCE_FS:
        print(f"\n  ⏹ 연속 {N_PATIENCE_FS}회 미개선 → 조기 종료 (탐색 {fi+1}/{len(candidates)})")
        break

print(f"\n  ✅ Forward Selection 완료: {len(selected_fs)}개 채택")
print(f"  채택 피처: {selected_fs}")
print(f"  Best Val AUCPR: {best_fs_val:.5f}  (Stage1: {s1_val:.5f}  Δ={best_fs_val-s1_val:+.5f})")

# ── Stage 3: HP Grid (Forward Selection 채택 피처 고정) ───────────────
print("\n── Stage 3: HP Grid (Forward Selection 피처 고정) ──")
# Forward Selection이 고른 피처로 HP만 탐색
bp3={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
     "random_state":42,"scale_pos_weight":spw}
hp_grid=[
    {"label":"d8_lr03_c07",  "hp":{"max_depth":8, "learning_rate":0.03, "colsample_bytree":0.7,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":1500}},
    {"label":"d8_lr02_c08",  "hp":{"max_depth":8, "learning_rate":0.02, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d9_lr02_c08",  "hp":{"max_depth":9, "learning_rate":0.02, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d9_lr02_c09",  "hp":{"max_depth":9, "learning_rate":0.02, "colsample_bytree":0.9,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d10_lr02_c08", "hp":{"max_depth":10,"learning_rate":0.02, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d10_lr01_c08", "hp":{"max_depth":10,"learning_rate":0.01, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":4000}},
    {"label":"d10_lr005_c08","hp":{"max_depth":10,"learning_rate":0.005,"colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":8000}},  # ★ 신규: v7c.10 iter=3996 한도 도달
    {"label":"d11_lr01_c08", "hp":{"max_depth":11,"learning_rate":0.01, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":5000}},  # ★ 신규: depth 확장
    {"label":"d9_reg",       "hp":{"max_depth":9, "learning_rate":0.02, "colsample_bytree":0.8,"subsample":0.75,"min_child_weight":8,"gamma":0.2,"reg_alpha":0.5,"reg_lambda":2.0,"n_estimators":2500}},
]

print(f"\n📊 {len(hp_grid)}개 HP 조합 탐색 (피처 {len(selected_fs)}개 고정)...")
print(f"  {'#':>2s} {'Label':<20s} | {'d':>2s} {'LR':>5s} {'col':>4s} | {'Val':>9s} {'it':>5s}")
print("  "+"-"*55)
best_s2={"val":-1}
for i,cfg in enumerate(hp_grid):
    hp=cfg["hp"].copy(); ne=hp.pop("n_estimators"); sf=selected_fs
    m=xgb.XGBClassifier(**{**bp3,**hp},n_estimators=ne,early_stopping_rounds=100)
    m.fit(X_train[sf],y_train,eval_set=[(X_val[sf],y_val)],verbose=False)
    va=m.best_score; bi=m.best_iteration
    print(f"  {i+1:2d} {cfg['label']:<20s} | {hp.get('max_depth',8):2d} {hp.get('learning_rate',0.03):5.3f} {hp.get('colsample_bytree',0.7):4.2f} | {va:9.5f} {bi:5d}")
    if va>best_s2["val"]:
        if best_s2.get("model"): del best_s2["model"]
        best_s2={"label":cfg["label"],"features":sf,"val":va,"iter":bi,"model":m,"hp":hp}
    else: del m
    gc.collect()

y_prob=best_s2["model"].predict_proba(X_test[best_s2["features"]])[:,1]
best_features=best_s2["features"]
print(f"\n  🏆 Best: {best_s2['label']} ({len(best_features)}개) Val={best_s2['val']:.5f} iter={best_s2['iter']}")

imp=best_s2["model"].feature_importances_; tidx=np.argsort(imp)[::-1][:30]
print(f"\n🔍 Feature Importance Top 30")
for i,idx in enumerate(tidx):
    c=best_features[idx]; t=""
    if c.startswith("tgat_emb_"): t=" [TGAT]"
    elif c.startswith("graph_"): t=" [GRAPH]"
    elif c.startswith("cond_"): t=" [COND]"
    elif c.startswith("score_"): t=" [SCORE]"
    elif c.startswith("interact_"): t=" [INTER]"
    elif c.startswith("cb_"): t=" [★CB]"
    elif c.startswith("vol_"): t=" [★VOL]"
    elif c.startswith("ws_"): t=" [★WIN]"
    elif c.startswith("inout_"): t=" [★INOUT]"
    print(f"  {i+1:2d}. {c:<45s}: {imp[idx]:.4f}{t}")

ti=imp.sum()
def group_imp(prefix): return sum(imp[i] for i,c in enumerate(best_features) if c.startswith(prefix))
print(f"\n📊 그룹별 중요도:")
print(f"   V2        : {group_imp('') / ti * 100:.1f}% (기존)")
print(f"   Graph     : {group_imp('graph_') / ti * 100:.1f}%")
print(f"   TGAT      : {group_imp('tgat_emb_') / ti * 100:.1f}%")
print(f"   Interact  : {group_imp('interact_') / ti * 100:.1f}%")
print(f"   ★ CB      : {group_imp('cb_') / ti * 100:.1f}%")
print(f"   ★ VolNorm : {group_imp('vol_') / ti * 100:.1f}%")
print(f"   ★ WinShift: {group_imp('ws_') / ti * 100:.1f}%")
print(f"   ★ InOut   : {group_imp('inout_') / ti * 100:.1f}%")
del X_train,X_val; gc.collect()

# =============================================================
# 9. 최종 리포트
# =============================================================
print("\n"+"="*60); print("🏆 [TGAT v7c.11] 최종 리포트"); print("="*60)
auprc=average_precision_score(y_test,y_prob); print(f"\n  AUPRC: {auprc:.4f}")
best_f1=0; best_thresh=0
for th in np.arange(0.05,0.95,0.01):
    f=f1_score(y_test,(y_prob>=th).astype(int),zero_division=0)
    if f>best_f1: best_f1=f; best_thresh=th
y_pred=(y_prob>=best_thresh).astype(int)
prec=precision_score(y_test,y_pred,zero_division=0); rec=recall_score(y_test,y_pred,zero_division=0)
print(f"\n📌 최적: T={best_thresh:.2f} F1={best_f1:.4f} P={prec:.4f} R={rec:.4f}")
print(f"\n📌 임계값별:")
for t in [0.1,0.3,0.5,0.7,0.8,0.9]:
    yp=(y_prob>=t).astype(int)
    print(f"  T={t:.1f} P={precision_score(y_test,yp,zero_division=0):.4f} R={recall_score(y_test,yp,zero_division=0):.4f} F1={f1_score(y_test,yp,zero_division=0):.4f}")

df_res=df_test_meta.with_columns([pl.col("time_group").dt.date().alias("date"),pl.Series("pred_prob",y_prob)])
df_dist=df_res.sort("pred_prob",descending=True).unique(subset=["account_id"],maintain_order=True)
print(f"\n📌 Top-K:")
for k in [100,500,1000,5000,10000]:
    ck=min(k,len(df_dist)); hits=df_dist.head(ck)["is_laundering"].sum()
    print(f"  Top-{k:5d}: {hits:5d}명 ({hits/ck*100:.2f}%)")
print(f"\n📌 일별 Top-100:")
for d in df_dist["date"].unique(maintain_order=True).sort(descending=True):
    hits=df_dist.filter(pl.col("date")==d).head(100)["is_laundering"].sum(); print(f"  {d}: {hits}")

tn,fp,fn,tp=confusion_matrix(y_test,y_pred).ravel()
print(f"\n📌 v7c.10→v7c.11: AUPRC {auprc:.4f} | F1 {best_f1:.4f}")
print(f"  TP:{tp:,} FP:{fp:,} FN:{fn:,} TN:{tn:,}")
print(f"  Precision:{prec:.4f} Recall:{rec:.4f}")

print(f"\n📌 GNN 탐색 결과:")
for r in gnn_results:
    tag = " ★" if r["label"]==best_gnn["label"] else ""
    print(f"  {r['label']:<12s}: QVal={r['q_val']:.5f} QTest={r['q_test']:.4f} Loss={r['loss']:.4f} Params={r['params']:,}{tag}")

print(f"\n📌 3-Stage 결과:")
print(f"  S1 전체({len(xgb_feature_cols)}개): Val={s1_val:.5f} Test={s1_test:.4f}")
print(f"  S2 Forward Selection: {len(selected_fs)}개 채택, Val={best_fs_val:.5f}")
print(f"  S3 HP Grid ({best_s2['label']}): Val={best_s2['val']:.5f} Test={auprc:.4f}")

print(f"\n✨ v7c.11 변경 요약 (v7c.10 대비):")
print(f"  GNN epochs     : 60 → {N_EPOCHS_GNN}")
print(f"  GNN BASE_LR    : 0.001 → {BASE_LR}")
print(f"  GNN neighbors  : [15,10] → {NUM_NEIGHBORS}")
print(f"  FS patience    : 8 → {N_PATIENCE_FS}")
print(f"  FS MAX_FEATURES: 100 → {MAX_FEATURES_FS}")
print(f"  HP grid        : 7종 → {len(hp_grid)}종 (d10_lr005, d11_lr01 추가)")
print(f"  XGB 후보       : {len(xgb_feature_cols)}개")
print(f"  FS 채택        : {len(selected_fs)}개")
print(f"  신규 피처 FS 채택: {[c for c in selected_fs if c in selected_new_cols]}")
print(f"  규칙성/금액 FS 채택: {[c for c in selected_fs if c in regularity_cols+amt_feature_cols]}")

del X_test; gc.collect(); log_memory("최종 완료")
print(f"\n✅ TGAT v7c.11 완료!")

🛡️ [TGAT v7c.11] GNN 수렴 강화 + FS 탐색 확대 + XGB 미세조정
   ▸ GNN: epochs 60→100, LR 0.001→0.003 (수렴 속도 개선)
   ▸ GNN: NUM_NEIGHBORS [15,10]→[25,15] (이웃 샘플 확대)
   ▸ FS:  patience 8→12, MAX_FEATURES 100→150 (후반 탐색 강화)
   ▸ XGB: d10_lr005 추가, d11_lr01 추가 (iter 한도 해소)

📂 데이터 로딩 중...
⏱️ Train cutoff: 2022-09-09 20:00:00
⏱️ Val   cutoff: 2022-09-14 05:00:00
📊 전체 노드 수: 2,076,999
  💾 [데이터 로드 완료] RAM: 98.28GB | GPU: 0.02GB

📐 [Step 2] 전체 노드 피처 계산...
  📊 그래프 구조 피처...
  📊 PageRank...
  ✅ 그래프 피처 13개
  📊 조건부행동 피처...
  ✅ 조건부행동 10개
  📊 AML 점수...
  ✅ AML 점수 5개
  📊 규칙성 피처...
  ✅ 규칙성 7개
  📊 금액 분포 피처...
  ✅ 금액 분포 8개
  📊 ★ Cross-border 정밀화 피처 (신규)...
  ✅ Cross-border 정밀화 4개: ['cb_new_partner_ratio', 'cb_oneway_new', 'cb_amount_asymmetry', 'cb_burst_cross']
  📊 ★ 거래량 대비 상대 정규화 피처 (신규)...
  ✅ 거래량 대비 정규화 5개: ['vol_amount_per_peer', 'vol_concentrated_amount', 'vol_low_freq_high_amt', 'vol_degree_normalized_amount', 'vol_outlier_score']
  📊 ★ 시간 윈도우별 행동 변화 피처 (신규)...
  ✅ 시간 윈도우 변화 3개: ['ws_recent_surge', 'ws_recent_on

    [h64_L2] Ep 1 Loss=1.3675


    [h64_L2] Ep10 Loss=1.0372


    [h64_L2] Ep20 Loss=1.0002


    [h64_L2] Ep30 Loss=0.9813


...

    [h128_L3] Ep60 Loss=0.9263


    [h128_L3] Ep70 Loss=0.9180


    [h128_L3] Ep80 Loss=0.9130


    [h128_L3] Ep90 Loss=0.9080


   4 h128_L3      |    128      3   32 |   0.9058  231,905 |  0.38575   0.4086
  💾 [h128_L3] RAM: 85.49GB | GPU: 0.02GB

  🏆 Best GNN: h128_L3 (QVal=0.38575 QTest=0.4086)

🔗 Interaction 피처 (h128_L3)...
  ✅ Interaction 10개
  💾 [GNN 탐색 완료] RAM: 82.85GB | GPU: 0.02GB

🚀 [Step 7] XGBoost 데이터 구성...
📊 XGBoost 총 후보 피처: 237
   V2:38 G:13 Cond:10 AML:5
   ★추가 Reg:7 Amt:8  (v7c.6 누락 수정)
   CB:4 VolNorm:5
   WinShift:3 InOut:6
   Emb:128 Inter:10
   ★ Forward Selection이 전체 후보에서 유효한 것 선별
📊 Train: 17,172,747 → 619,437 (4.76%)
📊 Val: 5,790,644
📊 Test: 5,792,913
  💾 [XGBoost 준비] RAM: 108.88GB | GPU: 0.02GB

🔥 [Step 8] 3-Stage XGBoost (Full → Forward Selection → HP Grid)...

── Stage 1: 전체 피처 학습 (importance 순위 산출) ──
[0]	validation_0-aucpr:0.11796
[100]	validation_0-aucpr:0.28269
[200]	validation_0-aucpr:0.37299
[300]	validation_0-aucpr:0.38716
[400]	validation_0-aucpr:0.40045
[500]	validation_0-aucpr:0.40512
[600]	validation_0-aucpr:0.41309
[700]	validation_0-aucpr:0.41993
[800]	validation_0-aucpr:0.

In [1]:
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv
from torch_geometric.loader import NeighborLoader
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm
import gc, numpy as np, psutil, os, warnings
warnings.filterwarnings("ignore", category=UserWarning)

def log_memory(tag=""):
    rss = psutil.Process(os.getpid()).memory_info().rss/(1024**3)
    gpu = torch.cuda.memory_allocated()/(1024**3) if torch.cuda.is_available() else 0
    print(f"  💾 [{tag}] RAM: {rss:.2f}GB | GPU: {gpu:.2f}GB")

print("=" * 60)
print("🛡️ [TGAT v7c.12] GNN 과적합 교정 + XGB HP 정제 + FS 확대")
print("   ▸ GNN: LR 0.003→0.001 복구 (과적합 억제)")
print("   ▸ GNN: dropout 0.2→0.3, epochs 100→80")
print("   ▸ XGB: d10_lr005 제거, d11_c09/d12_lr01 추가")
print("   ▸ FS:  patience 12→15 (규칙성/금액 피처 재탐색)")
print("=" * 60)

# ===================== 설정 =====================
NEG_NODE_RATIO_GNN = 25
NEG_ROW_RATIO_XGB = 20; RANDOM_SEED = 42
HIDDEN_DIM = 64; EDGE_DIM = 32
N_HEADS = 4
N_EPOCHS_GNN = 80             # ★ 100→80: 과학습 전 종료
BATCH_SIZE = 2048
NUM_NEIGHBORS = [25, 15]
N_INTERACTION_DIMS = 8
PATIENCE = 10
EARLY_STOP_DELTA = 0.001
BASE_LR = 0.001               # ★ 0.003→0.001: LR 복구 (0.003은 QVal 하락 유발)
WARMUP_EPOCHS = 5
GNN_DROPOUT = 0.3             # ★ 0.2→0.3: 과적합 억제

raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"

# =============================================================
# 1. 데이터 로드
# =============================================================
print("\n📂 데이터 로딩 중...")
df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))
if df_raw["Timestamp"].dtype == pl.Utf8:
    df_raw = df_raw.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_v2 = df_v2.sort("time_group")
total_count = len(df_v2)
train_cutoff = df_v2["time_group"][int(total_count*0.6)]
val_cutoff = df_v2["time_group"][int(total_count*0.8)]
print(f"⏱️ Train cutoff: {train_cutoff}\n⏱️ Val   cutoff: {val_cutoff}")

all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")
num_nodes = len(all_accounts); print(f"📊 전체 노드 수: {num_nodes:,}")
log_memory("데이터 로드 완료")

# =============================================================
# 2. 모든 노드 피처 계산 (GNN 입력용 + XGBoost 겸용)
# =============================================================
print("\n📐 [Step 2] 전체 노드 피처 계산...")
df_raw_train = df_raw.filter(pl.col("Timestamp") < train_cutoff)

# ─── 2-1. 그래프 구조 피처 13개 ───
print("  📊 그래프 구조 피처...")
df_from = df_raw_train.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id"))
df_to = df_raw_train.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
out_degree = df_from.group_by("account_id").len().rename({"len":"graph_out_degree"})
in_degree = df_to.group_by("account_id").len().rename({"len":"graph_in_degree"})
unique_out = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("to_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_out_partners"))
unique_in = df_raw_train.select([pl.col("to_acc").cast(pl.Utf8).alias("account_id"),pl.col("from_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_in_partners"))
esf = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("a"),pl.col("to_acc").cast(pl.Utf8).alias("b")]).unique()
est = esf.select([pl.col("b").alias("a"),pl.col("a").alias("b")])
bidir = esf.join(est, on=["a","b"], how="inner")
bidir_count = bidir.select(pl.col("a").alias("account_id")).group_by("account_id").len().rename({"len":"graph_bidir_count"})
del esf, est, bidir; gc.collect()
time_features = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("Timestamp")]).group_by("account_id").agg([
    pl.col("Timestamp").sort().diff().dt.total_seconds().mean().alias("graph_avg_tx_interval"),
    pl.col("Timestamp").sort().diff().dt.total_seconds().std().alias("graph_std_tx_interval"),
    pl.col("Timestamp").count().alias("graph_total_tx_count")])

# PageRank
print("  📊 PageRank...")
account_to_id = dict(zip(all_accounts["account_id"].to_list(), all_accounts["node_id"].to_list()))
src_ids = df_raw_train["from_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
dst_ids = df_raw_train["to_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
vm=(src_ids>=0)&(dst_ids>=0); sv=src_ids[vm]; dv=dst_ids[vm]
oda=np.zeros(num_nodes,dtype=np.float64); np.add.at(oda,sv,1.0); oda=np.maximum(oda,1.0)
pr=np.ones(num_nodes,dtype=np.float64)/num_nodes
for _ in range(10): npr=np.ones(num_nodes,dtype=np.float64)*0.15/num_nodes; np.add.at(npr,dv,0.85*pr[sv]/oda[sv]); pr=npr
pr_df = pl.DataFrame({"node_id":np.arange(num_nodes,dtype=np.uint32),"graph_pagerank":pr.astype(np.float32)})
pr_df = all_accounts.join(pr_df,on="node_id",how="left").select(["account_id","graph_pagerank"])
del src_ids,dst_ids,sv,dv,oda,pr,npr; gc.collect()

df_graph_feats = (all_accounts.select("account_id").join(out_degree,on="account_id",how="left").join(in_degree,on="account_id",how="left")
    .join(unique_out,on="account_id",how="left").join(unique_in,on="account_id",how="left")
    .join(bidir_count,on="account_id",how="left").join(time_features,on="account_id",how="left")
    .join(pr_df,on="account_id",how="left").fill_null(0.0)
    .with_columns([(pl.col("graph_out_degree")+pl.col("graph_in_degree")).alias("graph_total_degree"),
        (pl.col("graph_out_degree")/(pl.col("graph_in_degree")+1)).alias("graph_out_in_ratio"),
        (pl.col("graph_bidir_count")/(pl.col("graph_out_degree")+1)).alias("graph_reciprocity"),
        (pl.col("graph_unique_out_partners")/(pl.col("graph_out_degree")+1)).alias("graph_partner_diversity")]))
graph_feature_cols = [c for c in df_graph_feats.columns if c.startswith("graph_")]
del out_degree,in_degree,unique_out,unique_in,bidir_count,time_features,pr_df,df_from,df_to; gc.collect()
print(f"  ✅ 그래프 피처 {len(graph_feature_cols)}개")

# ─── 2-2. 조건부행동 피처 10개 ───
print("  📊 조건부행동 피처...")
df_v2_train = df_v2.filter(pl.col("time_group")<train_cutoff)
exclude_cols = ["account_id","time_group","is_laundering","mode_format","currency_mode"]
gnn_feature_cols_v2 = [c for c in df_v2.columns if c not in exclude_cols]
df_v2_agg = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])

df_cond = df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
v2a=set(df_cond.columns)
def sc(n): return pl.col(n) if n in v2a else pl.lit(0.0)
ce=[]
if "cnt_night" in v2a and "cnt_1h" in v2a: ce.append((pl.col("graph_total_degree")*sc("cnt_night")/(sc("cnt_1h")+1)).alias("cond_highdeg_night_intensity"))
if "avg_log_amount" in v2a: ce.append((sc("avg_log_amount")/(pl.col("graph_total_degree").log1p()+1)).alias("cond_lowdeg_high_amount"))
if "ratio_cross_border" in v2a: ce.append((sc("ratio_cross_border")*(1-pl.col("graph_reciprocity"))).alias("cond_oneway_crossborder"))
if "cnt_1h" in v2a: ce.append((pl.col("graph_unique_out_partners")*sc("cnt_1h")/(pl.col("graph_out_degree")+1)).alias("cond_fanout_burst"))
if "cnt_risk_format" in v2a: ce.append((pl.col("graph_pagerank")*sc("cnt_risk_format")).alias("cond_hub_risk_format"))
if "cnt_wire" in v2a: ce.append((pl.col("graph_out_in_ratio")*sc("cnt_wire")).alias("cond_asymmetric_wire"))
if "cnt_inter_bank" in v2a: ce.append((pl.col("graph_std_tx_interval").fill_null(0)/(pl.col("graph_avg_tx_interval").fill_null(1)+1)*sc("cnt_inter_bank")).alias("cond_irregular_interbank"))
if "cnt_currencies" in v2a: ce.append((pl.col("graph_partner_diversity")*sc("cnt_currencies")).alias("cond_diverse_partner_currency"))
if "amount_kurtosis" in v2a: ce.append((pl.col("graph_reciprocity")*sc("amount_kurtosis")).alias("cond_bidir_amount_volatility"))
if "entity_acct_cnt" in v2a and "burst_ratio" in v2a: ce.append((sc("entity_acct_cnt")*sc("burst_ratio")).alias("cond_multi_acct_burst"))
df_cond=df_cond.with_columns(ce); cond_feature_cols=[e.meta.output_name() for e in ce]
for cn in cond_feature_cols: df_cond=df_cond.with_columns(pl.when(pl.col(cn).is_infinite()|pl.col(cn).is_nan()).then(0.0).otherwise(pl.col(cn)).alias(cn))
df_cond_features=df_cond.select(["account_id"]+cond_feature_cols); del df_cond; gc.collect()
print(f"  ✅ 조건부행동 {len(cond_feature_cols)}개")

# ─── 2-3. AML 점수 5개 ───
print("  📊 AML 점수...")
df_sb=df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
se=[((sc("cnt_1h").fill_null(0).clip(0,1e6)+1).log1p()*pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("burst_ratio").fill_null(0).clip(0,1e6)/(sc("avg_log_amount").fill_null(1).clip(0,1e6)+1)).alias("score_smurf"),
    ((1.0/((pl.col("graph_out_in_ratio").clip(0.01,100)-1.0).abs()+0.1))*sc("cnt_wire").fill_null(0).clip(0,1e6).log1p()*(1-pl.col("graph_reciprocity").clip(0,1))/(pl.col("graph_unique_out_partners").clip(0,1e6).log1p()+1)).alias("score_mule"),
    (pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("cnt_currencies").fill_null(0).clip(0,1e6).log1p()*sc("ratio_cross_border").fill_null(0).clip(0,1)/(pl.col("graph_avg_tx_interval").fill_null(3600).clip(0,1e9)/3600+1)).alias("score_layering"),
    (pl.col("graph_in_degree").clip(0,1e6).log1p()*(pl.col("graph_pagerank").clip(0,1)*1e6)*sc("cnt_risk_format").fill_null(0).clip(0,1e6).log1p()/(pl.col("graph_out_degree").clip(0,1e6).log1p()+1)).alias("score_integration")]
df_sb=df_sb.with_columns(se)
sn=["score_smurf","score_mule","score_layering","score_integration"]
for s in sn: df_sb=df_sb.with_columns(pl.when(pl.col(s).is_infinite()|pl.col(s).is_nan()).then(0.0).otherwise(pl.col(s)).alias(s))
for s in sn:
    mn=df_sb[s].min();mx=df_sb[s].max();rng=mx-mn if(mx-mn)>0 else 1.0
    df_sb=df_sb.with_columns(((pl.col(s)-mn)/rng).alias(f"{s}_norm"))
df_sb=df_sb.with_columns((pl.col("score_smurf_norm")*0.3+pl.col("score_mule_norm")*0.25+pl.col("score_layering_norm")*0.3+pl.col("score_integration_norm")*0.15).alias("score_composite"))
aml_score_cols=sn+["score_composite"]
df_aml_scores=df_sb.select(["account_id"]+aml_score_cols)
del df_sb, df_v2_agg; gc.collect()
print(f"  ✅ AML 점수 {len(aml_score_cols)}개")

# ─── 2-4. 규칙성 피처 7개 ───
print("  📊 규칙성 피처...")
df_tx_time = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner"),
    pl.col("Timestamp"),
    pl.col("Timestamp").dt.hour().alias("hour"),
    pl.col("Timestamp").dt.weekday().alias("dow"),
])
reg_agg = df_tx_time.group_by("account_id").agg([
    (pl.col("hour").is_between(9,17).mean()).alias("reg_business_hour_ratio"),
    (pl.col("dow").is_in([5,6]).mean()).alias("reg_weekend_ratio"),
    (pl.col("hour").is_between(0,5).mean()).alias("reg_night_strict_ratio"),
    pl.col("partner").count().alias("_total_tx"),
])
hc=df_tx_time.group_by(["account_id","hour"]).len().rename({"len":"h_count"})
hp=hc.pivot(on="hour",index="account_id",values="h_count").fill_null(0)
hcols=[c for c in hp.columns if c!="account_id"]
hmat=hp.select(hcols).to_numpy().astype(np.float64); hsum=np.maximum(hmat.sum(axis=1,keepdims=True),1.0)
hprobs=hmat/hsum; hlog=np.where(hprobs>0,np.log(hprobs+1e-12),0.0)
df_h_ent=pl.DataFrame({"account_id":hp["account_id"],"reg_hour_entropy":-np.sum(hprobs*hlog,axis=1).astype(np.float32)})
del hc,hp,hmat,hprobs,hlog; gc.collect()
dc=df_tx_time.group_by(["account_id","dow"]).len().rename({"len":"d_count"})
dp=dc.pivot(on="dow",index="account_id",values="d_count").fill_null(0)
dcols=[c for c in dp.columns if c!="account_id"]
dmat=dp.select(dcols).to_numpy().astype(np.float64); dsum=np.maximum(dmat.sum(axis=1,keepdims=True),1.0)
dprobs=dmat/dsum; dlog=np.where(dprobs>0,np.log(dprobs+1e-12),0.0)
df_d_ent=pl.DataFrame({"account_id":dp["account_id"],"reg_dow_entropy":-np.sum(dprobs*dlog,axis=1).astype(np.float32)})
del dc,dp,dmat,dprobs,dlog; gc.collect()
pf=df_tx_time.group_by(["account_id","partner"]).len().rename({"len":"p_count"})
rp=pf.filter(pl.col("p_count")>=2).group_by("account_id").len().rename({"len":"repeat_cnt"})
tp2=pf.group_by("account_id").len().rename({"len":"total_cnt"})
df_pc=tp2.join(rp,on="account_id",how="left").fill_null(0).with_columns((pl.col("repeat_cnt")/(pl.col("total_cnt")+1)).alias("reg_consistent_partner_ratio")).select(["account_id","reg_consistent_partner_ratio"])
top_p=pf.sort(["account_id","p_count"],descending=[False,True]).unique(subset=["account_id"],maintain_order=True).rename({"p_count":"top_cnt"}).select(["account_id","top_cnt"])
df_tc=reg_agg.select(["account_id","_total_tx"]).join(top_p,on="account_id",how="left").fill_null(0).with_columns((pl.col("top_cnt")/(pl.col("_total_tx")+1)).alias("reg_top_partner_concentration")).select(["account_id","reg_top_partner_concentration"])
del pf,rp,tp2,top_p; gc.collect()
df_regularity=(reg_agg.select(["account_id","reg_business_hour_ratio","reg_weekend_ratio","reg_night_strict_ratio"])
    .join(df_h_ent,on="account_id",how="left").join(df_d_ent,on="account_id",how="left")
    .join(df_pc,on="account_id",how="left").join(df_tc,on="account_id",how="left").fill_null(0.0))
regularity_cols=[c for c in df_regularity.columns if c.startswith("reg_")]
for rc in regularity_cols: df_regularity=df_regularity.with_columns(pl.when(pl.col(rc).is_infinite()|pl.col(rc).is_nan()).then(0.0).otherwise(pl.col(rc)).alias(rc))
del reg_agg,df_h_ent,df_d_ent,df_pc,df_tc; gc.collect()
print(f"  ✅ 규칙성 {len(regularity_cols)}개")

# ─── 2-5. 금액 분포 피처 8개 ───
print("  📊 금액 분포 피처...")
amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train.columns: amount_col = cand; break
if amount_col:
    df_amounts=df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col(amount_col).cast(pl.Float64).alias("amount")]).filter(pl.col("amount")>0)
    df_amt=df_amounts.group_by("account_id").agg([
        ((pl.col("amount")%1000==0)|(pl.col("amount")%5000==0)).mean().alias("amt_round_number_ratio"),
        (pl.col("amount").std()/(pl.col("amount").mean()+1)).alias("amt_cv"),
        (pl.col("amount").max()/(pl.col("amount").mean()+1)).alias("amt_max_mean_ratio"),
        (pl.col("amount").median()/(pl.col("amount").mean()+1)).alias("amt_median_mean_ratio"),
        pl.col("amount").quantile(0.9).alias("_q90"), pl.col("amount").sum().alias("_tsum"),
        pl.col("amount").count().alias("_cnt"),
        pl.col("amount").quantile(0.75).alias("_q75"), pl.col("amount").quantile(0.25).alias("_q25"),
        pl.col("amount").median().alias("_med"),
    ]).fill_null(0.0).with_columns([
        (pl.col("_q90")*0.1*pl.col("_cnt")/(pl.col("_tsum")+1)).alias("amt_top10pct_concentration"),
        ((pl.col("_q75")-pl.col("_q25"))/(pl.col("_med")+1)).alias("amt_iqr_median_ratio")])
    gm=df_amounts["amount"].mean(); st=max(gm*0.1,1.0)
    df_sr=df_amounts.with_columns((pl.col("amount")<st).alias("is_small")).group_by("account_id").agg(pl.col("is_small").mean().alias("amt_small_tx_ratio"))
    df_amt=df_amt.join(df_sr,on="account_id",how="left").fill_null(0.0)
    df_al=df_amounts.with_columns((pl.col("amount")+1).log().cast(pl.Int32).alias("log_bucket"))
    bc=df_al.group_by(["account_id","log_bucket"]).len().rename({"len":"b_count"})
    bt=bc.group_by("account_id").agg(pl.col("b_count").sum().alias("b_total"))
    bp=bc.join(bt,on="account_id").with_columns((pl.col("b_count")/pl.col("b_total")).alias("b_prob"))
    ae=bp.group_by("account_id").agg((-1*(pl.col("b_prob")*(pl.col("b_prob")+1e-12).log()).sum()).alias("amt_entropy"))
    df_amt=df_amt.join(ae,on="account_id",how="left").fill_null(0.0)
    del df_amounts,df_al,bc,bt,bp,ae,df_sr; gc.collect()
    amt_feature_cols=[c for c in df_amt.columns if c.startswith("amt_") and not c.startswith("_")]
    df_amt_features=df_amt.select(["account_id"]+amt_feature_cols)
else:
    df_amt_features=all_accounts.select("account_id"); amt_feature_cols=[]
for ac in amt_feature_cols:
    df_amt_features=df_amt_features.with_columns(pl.when(pl.col(ac).is_infinite()|pl.col(ac).is_nan()).then(0.0).otherwise(pl.col(ac)).alias(ac))
print(f"  ✅ 금액 분포 {len(amt_feature_cols)}개")

# =============================================================
# ★ NEW: 2-6. Cross-border 정밀화 피처 4개
# =============================================================
# 핵심 아이디어: ratio_cross_border 단독이 아니라
#   "cross-border × 새로운 상대방 비율 × 방향 비대칭"을 결합
print("  📊 ★ Cross-border 정밀화 피처 (신규)...")
cb_feature_cols = []

# 데이터에 to_acc country/bank 정보가 있을 경우를 가정
# → 없으면 graph_partner_diversity 와 ratio_cross_border 조합으로 근사
df_v2_agg_cb = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2 if c in df_v2_train.columns
])
df_cb = df_v2_agg_cb.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)
v2cb = set(df_cb.columns)

cb_exprs = []
# ① cb_new_partner_ratio: cross-border이면서 새로운 상대방 비율
#    = ratio_cross_border × (unique_out_partners / (total_tx_count+1))
#    → 새로운 상대에게 해외 송금하는 패턴
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("graph_unique_out_partners") /
         (pl.col("graph_total_tx_count") + 1)
        ).alias("cb_new_partner_ratio")
    )
    cb_feature_cols.append("cb_new_partner_ratio")

# ② cb_oneway_new: cross-border × 비양방향 × 새 상대방
#    → 양방향 거래 없이 새로운 상대에게만 해외 송금 (전형적 분산 송금)
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         (1.0 - pl.col("graph_reciprocity")) *
         (pl.col("graph_unique_out_partners") / (pl.col("graph_out_degree") + 1))
        ).alias("cb_oneway_new")
    )
    cb_feature_cols.append("cb_oneway_new")

# ③ cb_amount_asymmetry: 해외 송금 비율 × 금액 비대칭 (out >> in)
#    = ratio_cross_border × out_in_ratio (clip 해서 폭발 방지)
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("graph_out_in_ratio").clip(0, 20)
        ).alias("cb_amount_asymmetry")
    )
    cb_feature_cols.append("cb_amount_asymmetry")

# ④ cb_burst_cross: cross-border이 짧은 시간 내 집중 (burst_ratio 결합)
if "ratio_cross_border" in v2cb and "burst_ratio" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("burst_ratio").clip(0, 100)
        ).alias("cb_burst_cross")
    )
    cb_feature_cols.append("cb_burst_cross")

if cb_exprs:
    df_cb = df_cb.with_columns(cb_exprs)
    for cn in cb_feature_cols:
        df_cb = df_cb.with_columns(
            pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
            .then(0.0).otherwise(pl.col(cn)).alias(cn))
    df_cb_features = df_cb.select(["account_id"] + cb_feature_cols)
else:
    df_cb_features = all_accounts.select("account_id"); cb_feature_cols = []
del df_cb, df_v2_agg_cb; gc.collect()
print(f"  ✅ Cross-border 정밀화 {len(cb_feature_cols)}개: {cb_feature_cols}")

# trend_* 피처 제거 (ws_* 와 중복) — v7c.4
trend_feature_cols = []

# =============================================================
# ★ NEW: 2-8. 거래량 대비 상대 정규화 피처 5개
# =============================================================
# 핵심 아이디어: cond_lowdeg_high_amount가 FN에서 TP보다 오히려 높아 역전됨
#   → "전체 네트워크 대비 비정상적으로 높은 금액"을 더 정밀하게 포착
print("  📊 ★ 거래량 대비 상대 정규화 피처 (신규)...")
vol_norm_feature_cols = []

df_v2_agg_vn = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2 if c in df_v2_train.columns
])
df_vn = df_v2_agg_vn.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)

# 네트워크 전체 평균값 계산 (정규화 기준)
if "avg_log_amount" in set(df_vn.columns):
    global_avg_log_amount = df_vn["avg_log_amount"].mean() or 1.0
    global_avg_degree = df_vn["graph_total_degree"].mean() or 1.0
    global_avg_cnt_1h = df_vn["cnt_1h"].mean() if "cnt_1h" in df_vn.columns else 1.0
    global_avg_cnt_1h = global_avg_cnt_1h or 1.0

    vn_exprs = []
    # ① vol_amount_per_peer: 건당 금액 / (파트너 수 × 전체 평균)
    #    → degree가 낮은데 금액이 높은 패턴을 전체 대비 상대적으로 포착
    vn_exprs.append(
        (pl.col("avg_log_amount") / (float(global_avg_log_amount) + 1e-8) /
         (pl.col("graph_unique_out_partners").log1p() + 1)
        ).alias("vol_amount_per_peer")
    )
    vol_norm_feature_cols.append("vol_amount_per_peer")

    # ② vol_concentrated_amount: 소수 파트너에게 집중 송금
    #    = avg_log_amount × (1 - partner_diversity)
    #    → diversity가 낮을수록 (=한 곳에 집중) × 금액이 높을수록 높아짐
    vn_exprs.append(
        (pl.col("avg_log_amount") * (1.0 - pl.col("graph_partner_diversity").clip(0, 1))
        ).alias("vol_concentrated_amount")
    )
    vol_norm_feature_cols.append("vol_concentrated_amount")

    # ③ vol_low_freq_high_amt: 낮은 빈도이지만 건당 금액이 전체 평균 대비 높음
    #    = (avg_log_amount / global_mean) / log1p(cnt_1h) → 드문데 고액
    if "cnt_1h" in set(df_vn.columns):
        vn_exprs.append(
            (pl.col("avg_log_amount") / float(global_avg_log_amount) /
             (pl.col("cnt_1h").clip(0, 1e6).log1p() + 1)
            ).alias("vol_low_freq_high_amt")
        )
        vol_norm_feature_cols.append("vol_low_freq_high_amt")

    # ④ vol_degree_normalized_amount: 디그리로 나눈 상대 금액 (올바른 역산 버전)
    #    기존 cond_lowdeg_high_amount 대체 — log 스케일 적용 + 전체 평균으로 나눔
    vn_exprs.append(
        ((pl.col("avg_log_amount") - float(global_avg_log_amount)) /
         (pl.col("graph_total_degree").clip(1, 1e6).log1p())
        ).alias("vol_degree_normalized_amount")
    )
    vol_norm_feature_cols.append("vol_degree_normalized_amount")

    # ⑤ vol_outlier_score: z-score 방식의 금액 이상 점수
    #    = (avg_log_amount - global_mean) → 양수면 전체 평균 초과
    vn_exprs.append(
        (pl.col("avg_log_amount") - float(global_avg_log_amount)).alias("vol_outlier_score")
    )
    vol_norm_feature_cols.append("vol_outlier_score")

    df_vn = df_vn.with_columns(vn_exprs)
    for cn in vol_norm_feature_cols:
        df_vn = df_vn.with_columns(
            pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
            .then(0.0).otherwise(pl.col(cn)).alias(cn))
    df_vol_norm_features = df_vn.select(["account_id"] + vol_norm_feature_cols)
else:
    df_vol_norm_features = all_accounts.select("account_id"); vol_norm_feature_cols = []

del df_vn, df_v2_agg_vn; gc.collect()
print(f"  ✅ 거래량 대비 정규화 {len(vol_norm_feature_cols)}개: {vol_norm_feature_cols}")

# =============================================================
# ★ NEW: 2-9. 시간 윈도우별 행동 변화 피처 4개
# =============================================================
# 핵심 아이디어: 미탐의 71%가 테스트 첫 3일 집중
#   → train 말미(~recent window)에서의 행동 변화를 포착
print("  📊 ★ 시간 윈도우별 행동 변화 피처 (신규)...")
win_shift_feature_cols = []

# train 마지막 7일 vs 나머지 기간 비교
last_7d_cutoff = train_cutoff - pl.duration(days=7)
df_raw_recent = df_raw_train.filter(pl.col("Timestamp") >= last_7d_cutoff)
df_raw_older  = df_raw_train.filter(pl.col("Timestamp") < last_7d_cutoff)

recent_cnt = df_raw_recent.select(
    pl.col("from_acc").cast(pl.Utf8).alias("account_id")
).group_by("account_id").len().rename({"len": "ws_recent_cnt"})
older_cnt = df_raw_older.select(
    pl.col("from_acc").cast(pl.Utf8).alias("account_id")
).group_by("account_id").len().rename({"len": "ws_older_cnt"})

df_ws = all_accounts.select("account_id").join(recent_cnt, on="account_id", how="left").join(older_cnt, on="account_id", how="left").fill_null(0.0)

ws_exprs = []
# ① ws_recent_surge: 최근 7일 / 이전 기간 (정규화) → 갑작스러운 증가
ws_exprs.append(
    (pl.col("ws_recent_cnt") / (pl.col("ws_older_cnt") / 30.0 + 1)).alias("ws_recent_surge")
)
win_shift_feature_cols.append("ws_recent_surge")

# ② ws_recent_only: 최근 7일에만 거래가 있는 계좌 (0/1)
#    → 갑자기 나타난 계좌
ws_exprs.append(
    ((pl.col("ws_recent_cnt") > 0) & (pl.col("ws_older_cnt") == 0)).cast(pl.Float64).alias("ws_recent_only")
)
win_shift_feature_cols.append("ws_recent_only")

df_ws = df_ws.with_columns(ws_exprs)

# ③ 최근 7일 고유 파트너 급증
recent_partners = df_raw_recent.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner")
]).group_by("account_id").agg(pl.col("partner").n_unique().alias("ws_recent_partners"))
older_partners = df_raw_older.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner")
]).group_by("account_id").agg(pl.col("partner").n_unique().alias("ws_older_partners"))

df_ws = df_ws.join(recent_partners, on="account_id", how="left").join(older_partners, on="account_id", how="left").fill_null(0.0)

# ④ ws_partner_explosion: 최근 7일 파트너가 이전 대비 급증
df_ws = df_ws.with_columns(
    (pl.col("ws_recent_partners") / (pl.col("ws_older_partners") + 1)).alias("ws_partner_explosion")
)
win_shift_feature_cols.append("ws_partner_explosion")

for cn in win_shift_feature_cols:
    df_ws = df_ws.with_columns(
        pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
        .then(0.0).otherwise(pl.col(cn)).alias(cn))

df_win_shift_features = df_ws.select(["account_id"] + win_shift_feature_cols)
del df_raw_recent, df_raw_older, df_ws, recent_cnt, older_cnt
del recent_partners, older_partners; gc.collect()
print(f"  ✅ 시간 윈도우 변화 {len(win_shift_feature_cols)}개: {win_shift_feature_cols}")

# =============================================================
# ★ NEW: 2-10. 입금/출금 독립 피처 6개
# =============================================================
# 핵심 아이디어: V2에 sum_in_1h/sum_out_1h는 있지만
#   AML 점수/조건부 피처에서 in/out 독립적 분석이 부족
print("  📊 ★ 입금/출금 독립 피처 (신규)...")
inout_feature_cols = []

# 출금(from) 집계
df_out_tx = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col(amount_col).cast(pl.Float64).alias("amount") if amount_col else pl.lit(1.0).alias("amount")
]).filter(pl.col("amount") > 0)

# 입금(to) 집계
df_in_tx = df_raw_train.select([
    pl.col("to_acc").cast(pl.Utf8).alias("account_id"),
    pl.col(amount_col).cast(pl.Float64).alias("amount") if amount_col else pl.lit(1.0).alias("amount")
]).filter(pl.col("amount") > 0)

out_stats = df_out_tx.group_by("account_id").agg([
    pl.col("amount").sum().alias("inout_total_out_amount"),
    pl.col("amount").mean().alias("inout_avg_out_amount"),
    pl.col("amount").count().alias("inout_out_cnt"),
])
in_stats = df_in_tx.group_by("account_id").agg([
    pl.col("amount").sum().alias("inout_total_in_amount"),
    pl.col("amount").mean().alias("inout_avg_in_amount"),
    pl.col("amount").count().alias("inout_in_cnt"),
])
df_inout = all_accounts.select("account_id").join(out_stats, on="account_id", how="left").join(in_stats, on="account_id", how="left").fill_null(0.0)

inout_exprs = []
# ① inout_out_amount_ratio: 출금액 / (입금액 + 출금액) → 1이면 순수 송금자
inout_exprs.append(
    (pl.col("inout_total_out_amount") /
     (pl.col("inout_total_out_amount") + pl.col("inout_total_in_amount") + 1)
    ).alias("inout_out_amount_ratio")
)
inout_feature_cols.append("inout_out_amount_ratio")

# ② inout_avg_out_vs_in: 평균 출금액 / 평균 입금액 → 1 초과면 출금이 더 큰 계좌
inout_exprs.append(
    (pl.col("inout_avg_out_amount") / (pl.col("inout_avg_in_amount") + 1)
    ).alias("inout_avg_out_vs_in")
)
inout_feature_cols.append("inout_avg_out_vs_in")

# ③ inout_cnt_imbalance: (출금 건수 - 입금 건수) / 전체 건수
#    → 양수면 보내는 쪽이 훨씬 많은 계좌 (smurf 패턴)
inout_exprs.append(
    ((pl.col("inout_out_cnt") - pl.col("inout_in_cnt")) /
     (pl.col("inout_out_cnt") + pl.col("inout_in_cnt") + 1)
    ).alias("inout_cnt_imbalance")
)
inout_feature_cols.append("inout_cnt_imbalance")

# ④ inout_fan_out_score: 출금 건수 × 고유 파트너 다양성 → 여러 곳에 분산 송금
df_inout = df_inout.join(df_graph_feats.select(["account_id", "graph_partner_diversity", "graph_unique_out_partners"]), on="account_id", how="left").fill_null(0.0)
inout_exprs.append(
    (pl.col("inout_out_cnt").log1p() * pl.col("graph_partner_diversity")
    ).alias("inout_fan_out_score")
)
inout_feature_cols.append("inout_fan_out_score")

# ⑤ inout_in_concentration: 입금이 소수에서만 들어옴 (mule 패턴)
#    = avg_in_amount × (1 - partner_diversity) → 소수 소스에서 집중 수취
inout_exprs.append(
    (pl.col("inout_avg_in_amount").log1p() * (1.0 - pl.col("graph_partner_diversity").clip(0, 1))
    ).alias("inout_in_concentration")
)
inout_feature_cols.append("inout_in_concentration")

# ⑥ inout_net_flow: log(출금합) - log(입금합) → 순 흐름 방향성
inout_exprs.append(
    (pl.col("inout_total_out_amount").log1p() - pl.col("inout_total_in_amount").log1p()
    ).alias("inout_net_flow")
)
inout_feature_cols.append("inout_net_flow")

df_inout = df_inout.with_columns(inout_exprs)
for cn in inout_feature_cols:
    df_inout = df_inout.with_columns(
        pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
        .then(0.0).otherwise(pl.col(cn)).alias(cn))
df_inout_features = df_inout.select(["account_id"] + inout_feature_cols)
del df_out_tx, df_in_tx, out_stats, in_stats, df_inout; gc.collect()
print(f"  ✅ 입금/출금 독립 {len(inout_feature_cols)}개: {inout_feature_cols}")
log_memory("전체 노드 피처 완료")

# =============================================================
# 3. GNN 엣지/노드 구성 — 강화된 입력
# =============================================================
print("\n🔗 [Step 3] GNN 엣지/노드 구성 (Enriched Input)...")

df_raw_reload = pl.read_parquet(raw_path)
if df_raw_reload["Timestamp"].dtype == pl.Utf8:
    df_raw_reload = df_raw_reload.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_raw_train_edges = df_raw_reload.filter(pl.col("Timestamp")<train_cutoff); del df_raw_reload; gc.collect()

edge_amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train_edges.columns: edge_amount_col = cand; break

df_edges = df_raw_train_edges.select(
    ["from_acc","to_acc","Timestamp"] +
    ([edge_amount_col] if edge_amount_col else []) +
    (["Payment Format"] if "Payment Format" in df_raw_train_edges.columns else []) +
    (["Is Laundering"] if "Is Laundering" in df_raw_train_edges.columns else [])
).with_columns([pl.col("from_acc").cast(pl.Utf8), pl.col("to_acc").cast(pl.Utf8)])

df_edges = df_edges.join(all_accounts.rename({"account_id":"from_acc","node_id":"src_id"}),on="from_acc",how="left")
df_edges = df_edges.join(all_accounts.rename({"account_id":"to_acc","node_id":"dst_id"}),on="to_acc",how="left")

# ── 시간 피처 (5개, 기존 유지) ──────────────────────────────────────────
ts = df_edges["Timestamp"]; mn_ts=ts.min(); mx_ts=ts.max(); ts_rng=(mx_ts-mn_ts).total_seconds()
etn = ((ts-mn_ts).dt.total_microseconds().cast(pl.Float64)/1e6/max(ts_rng,1.0)).to_numpy().astype(np.float32)
hrs = df_edges["Timestamp"].dt.hour().to_numpy().astype(np.float32)
dow = df_edges["Timestamp"].dt.weekday().to_numpy().astype(np.float32)
time_feats = np.stack([etn, np.sin(2*np.pi*hrs/24), np.cos(2*np.pi*hrs/24),
                        np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7)], axis=1)

# ── 금액 피처 (3개 → 4개) ─────────────────────────────────────────────
# ★ 변경: round_flag를 1000/5000 → 100단위로 확장 (소액 스머핑 감지)
# ★ 추가: amount_pct — 전체 거래 중 분위수 위치 (극단적 고액 거래 포착)
if edge_amount_col:
    amounts_raw = df_edges[edge_amount_col].cast(pl.Float64).fill_null(0.0).to_numpy().astype(np.float32)
    log_amounts  = np.log1p(np.maximum(amounts_raw, 0)).astype(np.float32)
    log_amounts  = log_amounts / (np.max(log_amounts) + 1e-8)
    round_flag   = ((amounts_raw % 100 == 0) & (amounts_raw > 0)).astype(np.float32)
    global_mean_amt = np.mean(amounts_raw[amounts_raw > 0]) if np.any(amounts_raw > 0) else 1.0
    relative_amt = np.log1p(amounts_raw / (global_mean_amt + 1e-8)).astype(np.float32)
    relative_amt = relative_amt / (np.max(relative_amt) + 1e-8)
    nonzero_amt  = amounts_raw[amounts_raw > 0]
    pct_vals     = np.percentile(nonzero_amt, np.linspace(0,100,1001)) if len(nonzero_amt)>0 else np.zeros(1001)
    amount_pct   = np.clip(np.searchsorted(pct_vals, amounts_raw, side='right').astype(np.float32)/1000.0, 0.0, 1.0)
    amount_feats = np.stack([log_amounts, round_flag, relative_amt, amount_pct], axis=1)
else:
    amount_feats = np.zeros((len(df_edges), 4), dtype=np.float32)

# ── Payment Format 피처 (이진 정규식 2개 → 정밀 인코딩 9개) ──────────
# ★ 실제 분포 확인 결과 (HI-Medium, 7종):
#   Cheque      38.50%  세탁율 0.02%   │ ACH      12.13%  세탁율 0.79% ← 압도적 고위험
#   Credit Card 27.52%  세탁율 0.02%   │ Cash     10.09%  세탁율 0.02%
#   Reinvestment 6.10%  세탁율 0.00%   │ Wire      3.51%  세탁율 0.00%
#   Bitcoin      2.16%  세탁율 0.04%
#
# 설계 결정:
#   ① 7종 전부 원-핫 (포맷 수가 적어 "기타" 불필요)
#   ② AML 위험도 연속값 1개 (ACH=0.0079, 나머지≈0 → 강한 분리 신호)
#   ③ 기존 is_wire(정규식) 완전 제거 — Wire 세탁율 0.00%로 노이즈였음
PAYMENT_FORMATS = ["Cheque", "Credit Card", "ACH", "Cash", "Reinvestment", "Wire", "Bitcoin"]
# 포맷별 전체 데이터 기준 AML 위험도 (사전 계산값, 학습 데이터 누수 방지)
FORMAT_RISK_MAP  = {"Cheque":0.0002, "Credit Card":0.0002, "ACH":0.0079,
                    "Cash":0.0002, "Reinvestment":0.0000, "Wire":0.0000, "Bitcoin":0.0004}

if "Payment Format" in df_edges.columns:
    pf_arr = df_edges["Payment Format"].cast(pl.Utf8).fill_null("Unknown").to_numpy()

    # ① 7종 원-핫 (7차원)
    onehot_feats = np.zeros((len(pf_arr), len(PAYMENT_FORMATS)), dtype=np.float32)
    for j, fmt in enumerate(PAYMENT_FORMATS):
        onehot_feats[:, j] = (pf_arr == fmt).astype(np.float32)

    # ② AML 위험도 연속값 (1차원) — ACH가 다른 포맷 대비 ~40배 높아 강한 신호
    fmt_risk = np.array([FORMAT_RISK_MAP.get(f, 0.0002) for f in pf_arr], dtype=np.float32)

    type_feats   = np.concatenate([onehot_feats, fmt_risk.reshape(-1,1)], axis=1)  # 8차원
    N_TYPE_FEATS = type_feats.shape[1]
else:
    N_TYPE_FEATS = len(PAYMENT_FORMATS) + 1
    type_feats   = np.zeros((len(df_edges), N_TYPE_FEATS), dtype=np.float32)

edge_features_all = np.concatenate([time_feats, amount_feats, type_feats], axis=1)
EDGE_RAW_DIM = edge_features_all.shape[1]
print(f"  📊 Edge 피처: {EDGE_RAW_DIM}개")
print(f"     시간(5) + 금액(4) + 포맷({N_TYPE_FEATS}: 원-핫7+위험도1)")
print(f"     ※ 기존 is_wire/is_internal 정규식 → 7종 원-핫+위험도로 교체")

edge_index_train = torch.tensor(df_edges.select(["src_id","dst_id"]).to_numpy().T, dtype=torch.long)
edge_attr_train = torch.tensor(edge_features_all, dtype=torch.float32)
print(f"📊 Train 엣지: {edge_index_train.shape[1]:,}")

del df_raw_train, df_raw_train_edges, df_edges, ts, hrs, dow, etn
del time_feats, amount_feats, type_feats, edge_features_all; gc.collect()

# ─── 노드 피처: V2(38) + 그래프(13) + 조건부(10) + AML(5) + 규칙성(7) + 금액(8) = 81개
# ★ v7c.5: GNN 입력은 v7c.2와 동일하게 유지 (신규 피처는 XGBoost에만 투입)
print("  📊 노드 피처 구성 (GNN: v7c.2 동일 81차원)...")
df_v2_node = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])
df_node_all = (all_accounts.join(df_v2_node, on="account_id", how="left")
    .join(df_graph_feats, on="account_id", how="left")
    .join(df_cond_features, on="account_id", how="left")
    .join(df_aml_scores, on="account_id", how="left")
    .join(df_regularity, on="account_id", how="left")
    .join(df_amt_features, on="account_id", how="left")
    .fill_null(0.0))

# GNN 입력은 v7c.2 피처만 (81개) — 신규 피처는 XGBoost 단계에만 추가
gnn_input_cols = (gnn_feature_cols_v2 + graph_feature_cols + cond_feature_cols +
                  aml_score_cols + regularity_cols + amt_feature_cols)

for col in gnn_input_cols:
    if col in df_node_all.columns:
        df_node_all = df_node_all.with_columns(
            pl.when(pl.col(col).is_infinite()|pl.col(col).is_nan()).then(0.0).otherwise(pl.col(col)).alias(col))

X_node = torch.tensor(df_node_all.select(gnn_input_cols).to_numpy(), dtype=torch.float32)
IN_DIM = X_node.shape[1]
print(f"  ✅ GNN 노드 입력: {IN_DIM}차원 (v7c.2 동일)")
print(f"     = V2({len(gnn_feature_cols_v2)}) + 그래프({len(graph_feature_cols)}) + 조건부({len(cond_feature_cols)})")
print(f"       + AML({len(aml_score_cols)}) + 규칙성({len(regularity_cols)}) + 금액({len(amt_feature_cols)})")

target_labels = df_v2_train.filter(pl.col("is_laundering")==1).select("account_id").unique().with_columns(pl.lit(1).alias("label"))
df_labels = all_accounts.join(target_labels,on="account_id",how="left").fill_null(0)
Y_node = torch.tensor(df_labels["label"].to_numpy(), dtype=torch.long)
pos_node_ids = set(df_labels.filter(pl.col("label")==1)["node_id"].to_list())
active_df = pl.DataFrame({"account_id":df_v2_train["account_id"].unique(),"is_active":True})
mask_df = all_accounts.join(active_df,on="account_id",how="left").fill_null(False)
active_node_ids = mask_df.filter(pl.col("is_active"))["node_id"].to_list()
active_pos = [n for n in active_node_ids if n in pos_node_ids]
active_neg = [n for n in active_node_ids if n not in pos_node_ids]
n_neg_gnn = min(len(active_pos)*NEG_NODE_RATIO_GNN, len(active_neg))
np.random.seed(RANDOM_SEED)
sampled_neg = np.random.choice(active_neg,size=n_neg_gnn,replace=False).tolist()
sampled_nodes = set(active_pos+sampled_neg)
train_mask_np = np.zeros(num_nodes, dtype=bool)
for nid in sampled_nodes: train_mask_np[nid]=True
train_mask = torch.tensor(train_mask_np, dtype=torch.bool)
print(f"📊 GNN 노드: 양성 {len(active_pos):,} + 음성 {n_neg_gnn:,}")
del df_v2_node, df_node_all, mask_df, active_df; gc.collect()

graph_data = Data(x=X_node, edge_index=edge_index_train, edge_attr=edge_attr_train, y=Y_node)
graph_data.train_mask = train_mask; graph_data.num_nodes = num_nodes
del X_node, edge_index_train, edge_attr_train, Y_node, train_mask; gc.collect()
log_memory("GNN 데이터 준비 완료")

# =============================================================
# 4-6. Multi-Config GNN 탐색 → 최적 임베딩 선택
# =============================================================
print("\n🧠 [Step 4-6] Multi-Config GNN 탐색...")

class EdgeProjection(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, out_dim), nn.GELU(), nn.Linear(out_dim, out_dim))
    def forward(self, ea): return self.proj(ea)

class TGAT_Flex(nn.Module):
    def __init__(self, in_dim, hidden_dim, edge_raw_dim, edge_dim, n_heads, n_layers=2, dropout=0.2):
        super().__init__()
        self.edge_proj = EdgeProjection(edge_raw_dim, edge_dim)
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.n_layers = n_layers
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_layers)])
        self.convs = nn.ModuleList([
            TransformerConv(hidden_dim, hidden_dim//n_heads, heads=n_heads,
                            edge_dim=edge_dim, dropout=dropout, concat=True)
            for _ in range(n_layers)])
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim//2, 1))
        self.dropout = dropout; self.hidden_dim = hidden_dim

    def forward(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        for i in range(self.n_layers):
            h = h + F.dropout(self.convs[i](self.norms[i](h), edge_index, ee), p=self.dropout, training=self.training)
        return self.classifier(self.norm_out(h)).squeeze(-1)

    @torch.no_grad()
    def get_embedding(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        for i in range(self.n_layers):
            h = h + self.convs[i](self.norms[i](h), edge_index, ee)
        return self.norm_out(h)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"📌 디바이스: {device}")
pw = (len(sampled_nodes)-len(active_pos))/max(len(active_pos),1)
print(f"📊 pos_weight: {pw:.1f}")

def make_lr(opt, ep, n_ep):
    if ep < WARMUP_EPOCHS: lr = 1e-6+(BASE_LR-1e-6)*(ep/WARMUP_EPOCHS)
    else: p=(ep-WARMUP_EPOCHS)/max(n_ep-WARMUP_EPOCHS,1); lr=1e-5+(BASE_LR-1e-5)*0.5*(1+np.cos(np.pi*p))
    for pg in opt.param_groups: pg['lr']=lr
    return lr

GNN_CONFIGS = [
    {"label":"h64_L2",  "hidden_dim":64,  "n_layers":2, "n_heads":4, "edge_dim":32},  # ★ 16→32
    {"label":"h128_L2", "hidden_dim":128, "n_layers":2, "n_heads":4, "edge_dim":32},
    {"label":"h64_L3",  "hidden_dim":64,  "n_layers":3, "n_heads":4, "edge_dim":32},  # ★ 16→32
    {"label":"h128_L3", "hidden_dim":128, "n_layers":3, "n_heads":4, "edge_dim":32},
]

xgb_exclude_cols=["account_id","time_group","is_laundering","mode_format","currency_mode","date"]
xgb_v2_cols=[c for c in df_v2.columns if c not in xgb_exclude_cols]
apr_train = df_v2_train.filter(pl.col("is_laundering")==1).height / max(df_v2_train.height,1)
spw = max((1-apr_train)/apr_train, 1.0) if apr_train > 0 else 20.0

# ★ 신규 피처들을 base_feature_cols에 포함
base_feature_cols = (xgb_v2_cols + graph_feature_cols + cond_feature_cols + aml_score_cols +
                     cb_feature_cols + vol_norm_feature_cols +
                     win_shift_feature_cols + inout_feature_cols)

def build_eval_df(db, df_emb_local, emb_col_names):
    feature_cols = base_feature_cols + emb_col_names
    df = (db.join(df_graph_feats,on="account_id",how="left")
          .join(df_cond_features,on="account_id",how="left")
          .join(df_aml_scores,on="account_id",how="left")
          .join(df_cb_features,on="account_id",how="left")            # ★
          .join(df_vol_norm_features,on="account_id",how="left")      # ★
          .join(df_win_shift_features,on="account_id",how="left")     # ★
          .join(df_inout_features,on="account_id",how="left")         # ★
          .join(df_emb_local,on="account_id",how="left")
          .fill_null(0.0).fill_nan(0.0))
    for cn in feature_cols:
        if cn in df.columns:
            df = df.with_columns(pl.when(pl.col(cn).is_infinite()).then(0.0).otherwise(pl.col(cn)).alias(cn))
    return df, feature_cols

def quick_xgb_eval(df_emb_local, emb_col_names, label):
    df_tr, fcols = build_eval_df(df_v2.filter(pl.col("time_group")<train_cutoff), df_emb_local, emb_col_names)
    df_tr_pos=df_tr.filter(pl.col("is_laundering")==1); df_tr_neg=df_tr.filter(pl.col("is_laundering")==0)
    n_neg=min(len(df_tr_pos)*NEG_ROW_RATIO_XGB,len(df_tr_neg))
    df_tr_s=pl.concat([df_tr_pos,df_tr_neg.sample(n=n_neg,seed=RANDOM_SEED)]).sample(fraction=1.0,seed=RANDOM_SEED)
    X_tr=df_tr_s.select(fcols).to_pandas(); y_tr=df_tr_s["is_laundering"].to_numpy()
    del df_tr,df_tr_pos,df_tr_neg,df_tr_s; gc.collect()
    df_vl, _ = build_eval_df(df_v2.filter((pl.col("time_group")>=train_cutoff)&(pl.col("time_group")<val_cutoff)), df_emb_local, emb_col_names)
    X_vl=df_vl.select(fcols).to_pandas(); y_vl=df_vl["is_laundering"].to_numpy(); del df_vl; gc.collect()
    df_te, _ = build_eval_df(df_v2.filter(pl.col("time_group")>=val_cutoff), df_emb_local, emb_col_names)
    X_te=df_te.select(fcols).to_pandas(); y_te=df_te["is_laundering"].to_numpy(); del df_te; gc.collect()
    qp={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
        "random_state":42,"max_depth":8,"learning_rate":0.03,"scale_pos_weight":spw,
        "subsample":0.8,"colsample_bytree":0.7,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
    m=xgb.XGBClassifier(**qp,n_estimators=1500,early_stopping_rounds=100)
    m.fit(X_tr,y_tr,eval_set=[(X_vl,y_vl)],verbose=0)
    q_val=m.best_score; q_test=average_precision_score(y_te,m.predict_proba(X_te)[:,1])
    del m,X_tr,X_vl,X_te,y_tr,y_vl,y_te; gc.collect()
    return q_val, q_test

print(f"\n📊 {len(GNN_CONFIGS)}개 GNN 설정 탐색...")
print(f"  {'#':>2s} {'Label':<12s} | {'hidden':>6s} {'layers':>6s} {'edim':>4s} | {'Loss':>8s} {'Params':>8s} | {'QVal':>8s} {'QTest':>8s}")
print("  "+"-"*80)

gnn_results = []
for gi, gcfg in enumerate(GNN_CONFIGS):
    gl = gcfg["label"]; hd=gcfg["hidden_dim"]; nl=gcfg["n_layers"]
    nh=gcfg["n_heads"]; ed=gcfg["edge_dim"]
    train_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                                   input_nodes=graph_data.train_mask, shuffle=True, num_workers=4)
    full_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                                  input_nodes=None, shuffle=False, num_workers=4)
    model = TGAT_Flex(in_dim=IN_DIM, hidden_dim=hd, edge_raw_dim=EDGE_RAW_DIM,
                      edge_dim=ed, n_heads=nh, n_layers=nl, dropout=GNN_DROPOUT).to(device)
    np_count = sum(p.numel() for p in model.parameters())
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw],device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6, weight_decay=1e-4)
    model.train(); bl=float('inf'); pc2=0; bs=None
    for epoch in range(N_EPOCHS_GNN):
        clr=make_lr(optimizer,epoch,N_EPOCHS_GNN); tl=0.0; nb2=0
        for batch in tqdm(train_loader,desc=f"[{gl}] Ep{epoch+1}",leave=False):
            batch=batch.to(device); optimizer.zero_grad()
            bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
            out=model(batch.x,batch.edge_index,bea)
            loss=criterion(out[:batch.batch_size],batch.y[:batch.batch_size].float())
            if torch.isnan(loss) or torch.isinf(loss): optimizer.zero_grad(); continue
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=0.5)
            optimizer.step(); tl+=loss.item(); nb2+=1
        al=tl/max(nb2,1)
        if (epoch+1)%10==0 or epoch==0: print(f"    [{gl}] Ep{epoch+1:2d} Loss={al:.4f}")
        if al<bl-EARLY_STOP_DELTA: bl=al; pc2=0; bs={k:v.cpu().clone() for k,v in model.state_dict().items()}
        else: pc2+=1
        if pc2>=PATIENCE: break
    if bs: model.load_state_dict({k:v.to(device) for k,v in bs.items()}); del bs
    model.eval(); ae=[]
    with torch.no_grad():
        for batch in tqdm(full_loader,desc=f"[{gl}] Embed",leave=False):
            batch=batch.to(device)
            bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
            emb=model.get_embedding(batch.x,batch.edge_index,bea)
            ae.append(emb[:batch.batch_size].cpu())
    femb=torch.cat(ae,dim=0).numpy()
    del model,ae,train_loader,full_loader; torch.cuda.empty_cache(); gc.collect()
    ecn=[f"tgat_emb_{i}" for i in range(hd)]
    df_emb=pl.concat([all_accounts.select("account_id"),pl.DataFrame(femb,schema=ecn)],how="horizontal")
    qv,qt = quick_xgb_eval(df_emb, ecn, gl)
    print(f"  {gi+1:2d} {gl:<12s} | {hd:6d} {nl:6d} {ed:4d} | {bl:8.4f} {np_count:8,} | {qv:8.5f} {qt:8.4f}")
    gnn_results.append({"label":gl,"hidden_dim":hd,"n_layers":nl,"loss":bl,"params":np_count,
                         "q_val":qv,"q_test":qt,"embedding":femb,"ecn":ecn,"df_emb":df_emb})
    log_memory(f"{gl}")

best_gnn = max(gnn_results, key=lambda x: x["q_val"])
print(f"\n  🏆 Best GNN: {best_gnn['label']} (QVal={best_gnn['q_val']:.5f} QTest={best_gnn['q_test']:.4f})")
for r in gnn_results:
    if r is not best_gnn: del r["embedding"], r["df_emb"]
gc.collect()

emb_cols = best_gnn["ecn"]
df_tgat = best_gnn["df_emb"]
FINAL_HIDDEN_DIM = best_gnn["hidden_dim"]

print(f"\n🔗 Interaction 피처 ({best_gnn['label']})...")
df_tgat_with_pr=df_tgat.join(df_graph_feats.select(["account_id","graph_pagerank"]),on="account_id",how="left").fill_null(0.0)
emb_np=df_tgat_with_pr.select(emb_cols).to_numpy(); emb_var=np.var(emb_np,axis=0)
top_idx=np.argsort(emb_var)[::-1][:N_INTERACTION_DIMS]; top_ec=[emb_cols[i] for i in top_idx]
ie_l=[]; ic_l=[]
for cn in top_ec: nn2=f"interact_pr_x_{cn}"; ie_l.append((pl.col(cn)*pl.col("graph_pagerank")*1e6).alias(nn2)); ic_l.append(nn2)
ie_l.append((sum(pl.col(c)**2 for c in emb_cols).sqrt()*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_norm")); ic_l.append("interact_pr_x_emb_norm")
ie_l.append((sum(pl.col(c) for c in top_ec)*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_topsum")); ic_l.append("interact_pr_x_emb_topsum")
interaction_cols=ic_l
df_interactions=df_tgat_with_pr.with_columns(ie_l).select(["account_id"]+ic_l)
for c in ic_l: df_interactions=df_interactions.with_columns(pl.when(pl.col(c).is_infinite()|pl.col(c).is_nan()).then(0.0).otherwise(pl.col(c)).alias(c))
print(f"  ✅ Interaction {len(ic_l)}개")
del df_tgat_with_pr,emb_np,best_gnn["embedding"]; gc.collect()
del graph_data; torch.cuda.empty_cache(); gc.collect()
log_memory("GNN 탐색 완료")

# =============================================================
# 7. XGBoost 데이터 구성
# =============================================================
print("\n🚀 [Step 7] XGBoost 데이터 구성...")

# ★ v7c.7: regularity(7) + amt(8) = 15개를 XGBoost 후보에 추가 (v7c.6 누락 수정)
# GNN 입력에는 이미 포함돼 있었지만 xgb_feature_cols에서 빠져 있었음
xgb_feature_cols = (xgb_v2_cols + graph_feature_cols + cond_feature_cols + aml_score_cols +
                    regularity_cols + amt_feature_cols +          # ★ 신규 추가
                    cb_feature_cols + vol_norm_feature_cols +
                    win_shift_feature_cols + inout_feature_cols +
                    emb_cols + interaction_cols)
selected_new_cols = cb_feature_cols + vol_norm_feature_cols + win_shift_feature_cols + inout_feature_cols
print(f"📊 XGBoost 총 후보 피처: {len(xgb_feature_cols)}")
print(f"   V2:{len(xgb_v2_cols)} G:{len(graph_feature_cols)} Cond:{len(cond_feature_cols)} AML:{len(aml_score_cols)}")
print(f"   ★추가 Reg:{len(regularity_cols)} Amt:{len(amt_feature_cols)}  (v7c.6 누락 수정)")
print(f"   CB:{len(cb_feature_cols)} VolNorm:{len(vol_norm_feature_cols)}")
print(f"   WinShift:{len(win_shift_feature_cols)} InOut:{len(inout_feature_cols)}")
print(f"   Emb:{len(emb_cols)} Inter:{len(interaction_cols)}")
print(f"   ★ Forward Selection이 전체 후보에서 유효한 것 선별")

def build_xgb_df(db):
    df=(db.join(df_graph_feats,on="account_id",how="left")
          .join(df_cond_features,on="account_id",how="left")
          .join(df_aml_scores,on="account_id",how="left")
          .join(df_regularity,on="account_id",how="left")        # ★ 신규 추가
          .join(df_amt_features,on="account_id",how="left")      # ★ 신규 추가
          .join(df_cb_features,on="account_id",how="left")
          .join(df_vol_norm_features,on="account_id",how="left")
          .join(df_win_shift_features,on="account_id",how="left")
          .join(df_inout_features,on="account_id",how="left")
          .join(df_tgat,on="account_id",how="left")
          .join(df_interactions,on="account_id",how="left")
          .fill_null(0.0).fill_nan(0.0))
    for cn in xgb_feature_cols:
        if cn in df.columns: df=df.with_columns(pl.when(pl.col(cn).is_infinite()).then(0.0).otherwise(pl.col(cn)).alias(cn))
    return df

df_tr=build_xgb_df(df_v2.filter(pl.col("time_group")<train_cutoff))
df_tr_pos=df_tr.filter(pl.col("is_laundering")==1); df_tr_neg=df_tr.filter(pl.col("is_laundering")==0)
n_neg=min(len(df_tr_pos)*NEG_ROW_RATIO_XGB,len(df_tr_neg))
df_tr_s=pl.concat([df_tr_pos,df_tr_neg.sample(n=n_neg,seed=RANDOM_SEED)]).sample(fraction=1.0,seed=RANDOM_SEED)
X_train=df_tr_s.select(xgb_feature_cols).to_pandas(); y_train=df_tr_s["is_laundering"].to_numpy()
print(f"📊 Train: {len(df_tr):,} → {len(df_tr_s):,} ({y_train.mean()*100:.2f}%)")
del df_tr,df_tr_pos,df_tr_neg,df_tr_s; gc.collect()
df_vl=build_xgb_df(df_v2.filter((pl.col("time_group")>=train_cutoff)&(pl.col("time_group")<val_cutoff)))
X_val=df_vl.select(xgb_feature_cols).to_pandas(); y_val=df_vl["is_laundering"].to_numpy(); print(f"📊 Val: {len(X_val):,}"); del df_vl; gc.collect()
df_te=build_xgb_df(df_v2.filter(pl.col("time_group")>=val_cutoff))
df_test_meta=df_te.select(["account_id","time_group","is_laundering"])
X_test=df_te.select(xgb_feature_cols).to_pandas(); y_test=df_te["is_laundering"].to_numpy(); print(f"📊 Test: {len(X_test):,}")
del df_te,df_v2,df_tgat,df_graph_feats,df_cond_features,df_aml_scores
del df_cb_features,df_vol_norm_features,df_win_shift_features,df_inout_features
del df_interactions,df_regularity,df_amt_features; gc.collect()
log_memory("XGBoost 준비")

# =============================================================
# 8. XGBoost 3-Stage: Full → Forward Selection → HP Grid
# =============================================================
# ★ v7c.6 핵심 변경: Stage 1 importance 기반 top-K 선택의 한계 해결
#   문제: 상관된 피처끼리 importance를 나눠 가져서 유용한 피처가 순위 밖으로 밀림
#   해결: Stage 2를 Forward Selection으로 교체
#     → 피처를 하나씩 추가하며 Val AUCPR이 실제로 오를 때만 채택
#     → 중복/노이즈 피처는 자동으로 탈락
print("\n🔥 [Step 8] 3-Stage XGBoost (Full → Forward Selection → HP Grid)...")
apr=y_train.sum()/len(y_train); spw=max((1-apr)/apr,1.0)

# ── Stage 1: 전체 피처로 학습 → importance 순위 산출 ──────────────────
print("\n── Stage 1: 전체 피처 학습 (importance 순위 산출) ──")
s1p={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
     "random_state":42,"max_depth":8,"learning_rate":0.03,"scale_pos_weight":spw,
     "subsample":0.8,"colsample_bytree":0.7,"min_child_weight":5,
     "gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
m_s1=xgb.XGBClassifier(**s1p,n_estimators=1500,early_stopping_rounds=100)
m_s1.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=100)
s1_val=m_s1.best_score
s1_test=average_precision_score(y_test,m_s1.predict_proba(X_test)[:,1])
print(f"\n  Stage1: Val={s1_val:.5f} Test={s1_test:.4f}")
imp_s1=m_s1.feature_importances_
# importance 순위 정렬 (Stage 2 Forward Selection의 탐색 순서로 사용)
feat_imp=sorted(zip(xgb_feature_cols,imp_s1),key=lambda x:x[1],reverse=True)
del m_s1; gc.collect()

# ── Stage 2: Forward Selection ─────────────────────────────────────────
# importance 상위 순서대로 피처를 하나씩 추가하며
# Val AUCPR이 오를 때만 채택, 연속 N_PATIENCE_FS번 개선 없으면 조기 종료
print("\n── Stage 2: Forward Selection ──")
print("  피처를 importance 순서대로 하나씩 추가 → Val 개선 시 채택")

N_PATIENCE_FS = 15          # ★ 12→15: 규칙성/금액 피처 재탐색 여지
MIN_DELTA_FS  = 5e-5
MAX_FEATURES_FS = 150

fs_hp={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
       "random_state":42,"max_depth":9,"learning_rate":0.02,"scale_pos_weight":spw,
       "subsample":0.8,"colsample_bytree":0.8,"min_child_weight":5,
       "gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}

selected_fs = []          # 채택된 피처 누적
best_fs_val = -1.0        # 현재까지 best Val AUCPR
no_improve  = 0           # 연속 미개선 횟수
fs_log      = []          # 로그 기록

candidates = [f for f,_ in feat_imp[:MAX_FEATURES_FS]]

print(f"  {'#':>3s} {'Feature':<45s} | {'Val':>9s} {'Δ':>8s} {'채택':>4s}")
print("  " + "-" * 75)

for fi, feat in enumerate(candidates):
    probe = selected_fs + [feat]
    m_fs = xgb.XGBClassifier(**fs_hp, n_estimators=1500, early_stopping_rounds=100)  # ★ 800→1500
    m_fs.fit(X_train[probe], y_train, eval_set=[(X_val[probe], y_val)], verbose=False)
    v = m_fs.best_score
    delta = v - best_fs_val
    adopted = delta > MIN_DELTA_FS

    tag = "✅" if adopted else "  "
    print(f"  {fi+1:3d} {feat:<45s} | {v:9.5f} {delta:+8.5f} {tag}")
    fs_log.append({"feat":feat,"val":v,"delta":delta,"adopted":adopted})

    if adopted:
        selected_fs.append(feat)
        best_fs_val = v
        no_improve = 0
    else:
        no_improve += 1

    del m_fs; gc.collect()

    if no_improve >= N_PATIENCE_FS:
        print(f"\n  ⏹ 연속 {N_PATIENCE_FS}회 미개선 → 조기 종료 (탐색 {fi+1}/{len(candidates)})")
        break

print(f"\n  ✅ Forward Selection 완료: {len(selected_fs)}개 채택")
print(f"  채택 피처: {selected_fs}")
print(f"  Best Val AUCPR: {best_fs_val:.5f}  (Stage1: {s1_val:.5f}  Δ={best_fs_val-s1_val:+.5f})")

# ── Stage 3: HP Grid (Forward Selection 채택 피처 고정) ───────────────
print("\n── Stage 3: HP Grid (Forward Selection 피처 고정) ──")
# Forward Selection이 고른 피처로 HP만 탐색
bp3={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
     "random_state":42,"scale_pos_weight":spw}
hp_grid=[
    {"label":"d8_lr03_c07",  "hp":{"max_depth":8, "learning_rate":0.03, "colsample_bytree":0.7,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":1500}},
    {"label":"d8_lr02_c08",  "hp":{"max_depth":8, "learning_rate":0.02, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d9_lr02_c08",  "hp":{"max_depth":9, "learning_rate":0.02, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d9_lr02_c09",  "hp":{"max_depth":9, "learning_rate":0.02, "colsample_bytree":0.9,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d10_lr02_c08", "hp":{"max_depth":10,"learning_rate":0.02, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d10_lr01_c08", "hp":{"max_depth":10,"learning_rate":0.01, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":4000}},
    {"label":"d11_lr01_c08", "hp":{"max_depth":11,"learning_rate":0.01, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":5000}},  # v7c.11 best
    {"label":"d11_lr01_c09", "hp":{"max_depth":11,"learning_rate":0.01, "colsample_bytree":0.9,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":5000}},  # ★ 신규: d11 colsample 확대
    {"label":"d12_lr01_c08", "hp":{"max_depth":12,"learning_rate":0.01, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":5000}},  # ★ 신규: depth 한계 탐색
    {"label":"d9_reg",       "hp":{"max_depth":9, "learning_rate":0.02, "colsample_bytree":0.8,"subsample":0.75,"min_child_weight":8,"gamma":0.2,"reg_alpha":0.5,"reg_lambda":2.0,"n_estimators":2500}},
]

print(f"\n📊 {len(hp_grid)}개 HP 조합 탐색 (피처 {len(selected_fs)}개 고정)...")
print(f"  {'#':>2s} {'Label':<20s} | {'d':>2s} {'LR':>5s} {'col':>4s} | {'Val':>9s} {'it':>5s}")
print("  "+"-"*55)
best_s2={"val":-1}
for i,cfg in enumerate(hp_grid):
    hp=cfg["hp"].copy(); ne=hp.pop("n_estimators"); sf=selected_fs
    m=xgb.XGBClassifier(**{**bp3,**hp},n_estimators=ne,early_stopping_rounds=100)
    m.fit(X_train[sf],y_train,eval_set=[(X_val[sf],y_val)],verbose=False)
    va=m.best_score; bi=m.best_iteration
    print(f"  {i+1:2d} {cfg['label']:<20s} | {hp.get('max_depth',8):2d} {hp.get('learning_rate',0.03):5.3f} {hp.get('colsample_bytree',0.7):4.2f} | {va:9.5f} {bi:5d}")
    if va>best_s2["val"]:
        if best_s2.get("model"): del best_s2["model"]
        best_s2={"label":cfg["label"],"features":sf,"val":va,"iter":bi,"model":m,"hp":hp}
    else: del m
    gc.collect()

y_prob=best_s2["model"].predict_proba(X_test[best_s2["features"]])[:,1]
best_features=best_s2["features"]
print(f"\n  🏆 Best: {best_s2['label']} ({len(best_features)}개) Val={best_s2['val']:.5f} iter={best_s2['iter']}")

imp=best_s2["model"].feature_importances_; tidx=np.argsort(imp)[::-1][:30]
print(f"\n🔍 Feature Importance Top 30")
for i,idx in enumerate(tidx):
    c=best_features[idx]; t=""
    if c.startswith("tgat_emb_"): t=" [TGAT]"
    elif c.startswith("graph_"): t=" [GRAPH]"
    elif c.startswith("cond_"): t=" [COND]"
    elif c.startswith("score_"): t=" [SCORE]"
    elif c.startswith("interact_"): t=" [INTER]"
    elif c.startswith("cb_"): t=" [★CB]"
    elif c.startswith("vol_"): t=" [★VOL]"
    elif c.startswith("ws_"): t=" [★WIN]"
    elif c.startswith("inout_"): t=" [★INOUT]"
    print(f"  {i+1:2d}. {c:<45s}: {imp[idx]:.4f}{t}")

ti=imp.sum()
def group_imp(prefix): return sum(imp[i] for i,c in enumerate(best_features) if c.startswith(prefix))
print(f"\n📊 그룹별 중요도:")
print(f"   V2        : {group_imp('') / ti * 100:.1f}% (기존)")
print(f"   Graph     : {group_imp('graph_') / ti * 100:.1f}%")
print(f"   TGAT      : {group_imp('tgat_emb_') / ti * 100:.1f}%")
print(f"   Interact  : {group_imp('interact_') / ti * 100:.1f}%")
print(f"   ★ CB      : {group_imp('cb_') / ti * 100:.1f}%")
print(f"   ★ VolNorm : {group_imp('vol_') / ti * 100:.1f}%")
print(f"   ★ WinShift: {group_imp('ws_') / ti * 100:.1f}%")
print(f"   ★ InOut   : {group_imp('inout_') / ti * 100:.1f}%")
del X_train,X_val; gc.collect()

# =============================================================
# 9. 최종 리포트
# =============================================================
print("\n"+"="*60); print("🏆 [TGAT v7c.12] 최종 리포트"); print("="*60)
auprc=average_precision_score(y_test,y_prob); print(f"\n  AUPRC: {auprc:.4f}")
best_f1=0; best_thresh=0
for th in np.arange(0.05,0.95,0.01):
    f=f1_score(y_test,(y_prob>=th).astype(int),zero_division=0)
    if f>best_f1: best_f1=f; best_thresh=th
y_pred=(y_prob>=best_thresh).astype(int)
prec=precision_score(y_test,y_pred,zero_division=0); rec=recall_score(y_test,y_pred,zero_division=0)
print(f"\n📌 최적: T={best_thresh:.2f} F1={best_f1:.4f} P={prec:.4f} R={rec:.4f}")
print(f"\n📌 임계값별:")
for t in [0.1,0.3,0.5,0.7,0.8,0.9]:
    yp=(y_prob>=t).astype(int)
    print(f"  T={t:.1f} P={precision_score(y_test,yp,zero_division=0):.4f} R={recall_score(y_test,yp,zero_division=0):.4f} F1={f1_score(y_test,yp,zero_division=0):.4f}")

df_res=df_test_meta.with_columns([pl.col("time_group").dt.date().alias("date"),pl.Series("pred_prob",y_prob)])
df_dist=df_res.sort("pred_prob",descending=True).unique(subset=["account_id"],maintain_order=True)
print(f"\n📌 Top-K:")
for k in [100,500,1000,5000,10000]:
    ck=min(k,len(df_dist)); hits=df_dist.head(ck)["is_laundering"].sum()
    print(f"  Top-{k:5d}: {hits:5d}명 ({hits/ck*100:.2f}%)")
print(f"\n📌 일별 Top-100:")
for d in df_dist["date"].unique(maintain_order=True).sort(descending=True):
    hits=df_dist.filter(pl.col("date")==d).head(100)["is_laundering"].sum(); print(f"  {d}: {hits}")

tn,fp,fn,tp=confusion_matrix(y_test,y_pred).ravel()
print(f"\n📌 v7c.11→v7c.12: AUPRC {auprc:.4f} | F1 {best_f1:.4f}")
print(f"  TP:{tp:,} FP:{fp:,} FN:{fn:,} TN:{tn:,}")
print(f"  Precision:{prec:.4f} Recall:{rec:.4f}")

print(f"\n📌 GNN 탐색 결과:")
for r in gnn_results:
    tag = " ★" if r["label"]==best_gnn["label"] else ""
    print(f"  {r['label']:<12s}: QVal={r['q_val']:.5f} QTest={r['q_test']:.4f} Loss={r['loss']:.4f} Params={r['params']:,}{tag}")

print(f"\n📌 3-Stage 결과:")
print(f"  S1 전체({len(xgb_feature_cols)}개): Val={s1_val:.5f} Test={s1_test:.4f}")
print(f"  S2 Forward Selection: {len(selected_fs)}개 채택, Val={best_fs_val:.5f}")
print(f"  S3 HP Grid ({best_s2['label']}): Val={best_s2['val']:.5f} Test={auprc:.4f}")

print(f"\n✨ v7c.12 변경 요약 (v7c.11 대비):")
print(f"  GNN BASE_LR    : 0.003 → {BASE_LR} (과적합 억제)")
print(f"  GNN dropout    : 0.2 → {GNN_DROPOUT}")
print(f"  GNN epochs     : 100 → {N_EPOCHS_GNN}")
print(f"  FS patience    : 12 → {N_PATIENCE_FS}")
print(f"  HP grid        : 9종 → {len(hp_grid)}종 (d10_lr005 제거, d11_c09/d12 추가)")
print(f"  XGB 후보       : {len(xgb_feature_cols)}개")
print(f"  FS 채택        : {len(selected_fs)}개")
print(f"  신규 피처 FS 채택: {[c for c in selected_fs if c in selected_new_cols]}")
print(f"  규칙성/금액 FS 채택: {[c for c in selected_fs if c in regularity_cols+amt_feature_cols]}")

del X_test; gc.collect(); log_memory("최종 완료")
print(f"\n✅ TGAT v7c.12 완료!")

🛡️ [TGAT v7c.12] GNN 과적합 교정 + XGB HP 정제 + FS 확대
   ▸ GNN: LR 0.003→0.001 복구 (과적합 억제)
   ▸ GNN: dropout 0.2→0.3, epochs 100→80
   ▸ XGB: d10_lr005 제거, d11_c09/d12_lr01 추가
   ▸ FS:  patience 12→15 (규칙성/금액 피처 재탐색)

📂 데이터 로딩 중...
⏱️ Train cutoff: 2022-09-09 20:00:00
⏱️ Val   cutoff: 2022-09-14 05:00:00
📊 전체 노드 수: 2,076,999
  💾 [데이터 로드 완료] RAM: 33.85GB | GPU: 0.00GB

📐 [Step 2] 전체 노드 피처 계산...
  📊 그래프 구조 피처...
  📊 PageRank...
  ✅ 그래프 피처 13개
  📊 조건부행동 피처...
  ✅ 조건부행동 10개
  📊 AML 점수...
  ✅ AML 점수 5개
  📊 규칙성 피처...
  ✅ 규칙성 7개
  📊 금액 분포 피처...
  ✅ 금액 분포 8개
  📊 ★ Cross-border 정밀화 피처 (신규)...
  ✅ Cross-border 정밀화 4개: ['cb_new_partner_ratio', 'cb_oneway_new', 'cb_amount_asymmetry', 'cb_burst_cross']
  📊 ★ 거래량 대비 상대 정규화 피처 (신규)...
  ✅ 거래량 대비 정규화 5개: ['vol_amount_per_peer', 'vol_concentrated_amount', 'vol_low_freq_high_amt', 'vol_degree_normalized_amount', 'vol_outlier_score']
  📊 ★ 시간 윈도우별 행동 변화 피처 (신규)...
  ✅ 시간 윈도우 변화 3개: ['ws_recent_surge', 'ws_recent_only', 'ws_partner_explosion']
  📊 ★ 입금/출금 독립 피처

    [h64_L2] Ep 1 Loss=1.3456


    [h64_L2] Ep10 Loss=1.0493


    [h64_L2] Ep20 Loss=1.0199


    [h64_L2] Ep30 Loss=1.0095


...

    [h128_L3] Ep50 Loss=0.9690


    [h128_L3] Ep60 Loss=0.9616


    [h128_L3] Ep70 Loss=0.9552


    [h128_L3] Ep80 Loss=0.9515


   4 h128_L3      |    128      3   32 |   0.9516  231,905 |  0.38515   0.4131
  💾 [h128_L3] RAM: 72.85GB | GPU: 0.02GB

  🏆 Best GNN: h128_L3 (QVal=0.38515 QTest=0.4131)

🔗 Interaction 피처 (h128_L3)...
  ✅ Interaction 10개
  💾 [GNN 탐색 완료] RAM: 68.99GB | GPU: 0.02GB

🚀 [Step 7] XGBoost 데이터 구성...
📊 XGBoost 총 후보 피처: 237
   V2:38 G:13 Cond:10 AML:5
   ★추가 Reg:7 Amt:8  (v7c.6 누락 수정)
   CB:4 VolNorm:5
   WinShift:3 InOut:6
   Emb:128 Inter:10
   ★ Forward Selection이 전체 후보에서 유효한 것 선별
📊 Train: 17,172,747 → 619,437 (4.76%)
📊 Val: 5,790,644
📊 Test: 5,792,913
  💾 [XGBoost 준비] RAM: 96.57GB | GPU: 0.02GB

🔥 [Step 8] 3-Stage XGBoost (Full → Forward Selection → HP Grid)...

── Stage 1: 전체 피처 학습 (importance 순위 산출) ──
[0]	validation_0-aucpr:0.11791
[100]	validation_0-aucpr:0.28412
[200]	validation_0-aucpr:0.37658
[300]	validation_0-aucpr:0.39657
[400]	validation_0-aucpr:0.40532
[500]	validation_0-aucpr:0.41008
[600]	validation_0-aucpr:0.41793
[700]	validation_0-aucpr:0.42192
[800]	validation_0-aucpr:0.4

In [4]:
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv
from torch_geometric.loader import NeighborLoader
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm
import gc, numpy as np, psutil, os, warnings
warnings.filterwarnings("ignore", category=UserWarning)

def log_memory(tag=""):
    rss = psutil.Process(os.getpid()).memory_info().rss/(1024**3)
    gpu = torch.cuda.memory_allocated()/(1024**3) if torch.cuda.is_available() else 0
    print(f"  💾 [{tag}] RAM: {rss:.2f}GB | GPU: {gpu:.2f}GB")

print("=" * 60)
print("🛡️ [TGAT v7c.18] 엣지 피처 정밀화")
print("   ▸ relative_amt → is_large_tx (amount > 5×global_mean)")
print("     log_amount와 중복 제거 → 비선형 임계값 신호로 교체")
print("=" * 60)

# ===================== 설정 =====================
NEG_NODE_RATIO_GNN = 25
NEG_ROW_RATIO_XGB = 20; RANDOM_SEED = 42
EDGE_DIM = 32             # ★ HIDDEN_DIM 전역 상수 제거 (GNN_CONFIGS에서 직접 지정)
N_HEADS = 4
N_EPOCHS_GNN = 80             # ★ 100→80: 과학습 전 종료
BATCH_SIZE = 2048
NUM_NEIGHBORS = [25, 15]
N_INTERACTION_DIMS = 8
PATIENCE = 10
EARLY_STOP_DELTA = 0.001
BASE_LR = 0.001               # ★ 0.003→0.001: LR 복구 (0.003은 QVal 하락 유발)
WARMUP_EPOCHS = 5
GNN_DROPOUT = 0.3             # ★ 0.2→0.3: 과적합 억제

raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"

# =============================================================
# 1. 데이터 로드
# =============================================================
print("\n📂 데이터 로딩 중...")
df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))
if df_raw["Timestamp"].dtype == pl.Utf8:
    df_raw = df_raw.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_v2 = df_v2.sort("time_group")
total_count = len(df_v2)
train_cutoff = df_v2["time_group"][int(total_count*0.6)]
val_cutoff = df_v2["time_group"][int(total_count*0.8)]
print(f"⏱️ Train cutoff: {train_cutoff}\n⏱️ Val   cutoff: {val_cutoff}")

all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")
num_nodes = len(all_accounts); print(f"📊 전체 노드 수: {num_nodes:,}")
log_memory("데이터 로드 완료")

# =============================================================
# 2. 모든 노드 피처 계산 (GNN 입력용 + XGBoost 겸용)
# =============================================================
print("\n📐 [Step 2] 전체 노드 피처 계산...")
df_raw_train = df_raw.filter(pl.col("Timestamp") < train_cutoff)

# ─── 2-1. 그래프 구조 피처 13개 ───
print("  📊 그래프 구조 피처...")
df_from = df_raw_train.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id"))
df_to = df_raw_train.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
out_degree = df_from.group_by("account_id").len().rename({"len":"graph_out_degree"})
in_degree = df_to.group_by("account_id").len().rename({"len":"graph_in_degree"})
unique_out = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("to_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_out_partners"))
unique_in = df_raw_train.select([pl.col("to_acc").cast(pl.Utf8).alias("account_id"),pl.col("from_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_in_partners"))
esf = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("a"),pl.col("to_acc").cast(pl.Utf8).alias("b")]).unique()
est = esf.select([pl.col("b").alias("a"),pl.col("a").alias("b")])
bidir = esf.join(est, on=["a","b"], how="inner")
bidir_count = bidir.select(pl.col("a").alias("account_id")).group_by("account_id").len().rename({"len":"graph_bidir_count"})
del esf, est, bidir; gc.collect()
time_features = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("Timestamp")]).group_by("account_id").agg([
    pl.col("Timestamp").sort().diff().dt.total_seconds().mean().alias("graph_avg_tx_interval"),
    pl.col("Timestamp").sort().diff().dt.total_seconds().std().alias("graph_std_tx_interval"),
    pl.col("Timestamp").count().alias("graph_total_tx_count")])

# PageRank
print("  📊 PageRank...")
account_to_id = dict(zip(all_accounts["account_id"].to_list(), all_accounts["node_id"].to_list()))
src_ids = df_raw_train["from_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
dst_ids = df_raw_train["to_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
vm=(src_ids>=0)&(dst_ids>=0); sv=src_ids[vm]; dv=dst_ids[vm]
oda=np.zeros(num_nodes,dtype=np.float64); np.add.at(oda,sv,1.0); oda=np.maximum(oda,1.0)
pr=np.ones(num_nodes,dtype=np.float64)/num_nodes
for _ in range(10): npr=np.ones(num_nodes,dtype=np.float64)*0.15/num_nodes; np.add.at(npr,dv,0.85*pr[sv]/oda[sv]); pr=npr
pr_df = pl.DataFrame({"node_id":np.arange(num_nodes,dtype=np.uint32),"graph_pagerank":pr.astype(np.float32)})
pr_df = all_accounts.join(pr_df,on="node_id",how="left").select(["account_id","graph_pagerank"])
del src_ids,dst_ids,sv,dv,oda,pr,npr; gc.collect()

df_graph_feats = (all_accounts.select("account_id").join(out_degree,on="account_id",how="left").join(in_degree,on="account_id",how="left")
    .join(unique_out,on="account_id",how="left").join(unique_in,on="account_id",how="left")
    .join(bidir_count,on="account_id",how="left").join(time_features,on="account_id",how="left")
    .join(pr_df,on="account_id",how="left").fill_null(0.0)
    .with_columns([(pl.col("graph_out_degree")+pl.col("graph_in_degree")).alias("graph_total_degree"),
        (pl.col("graph_out_degree")/(pl.col("graph_in_degree")+1)).alias("graph_out_in_ratio"),
        (pl.col("graph_bidir_count")/(pl.col("graph_out_degree")+1)).alias("graph_reciprocity"),
        (pl.col("graph_unique_out_partners")/(pl.col("graph_out_degree")+1)).alias("graph_partner_diversity")]))
graph_feature_cols = [c for c in df_graph_feats.columns if c.startswith("graph_")]
del out_degree,in_degree,unique_out,unique_in,bidir_count,time_features,pr_df,df_from,df_to; gc.collect()
print(f"  ✅ 그래프 피처 {len(graph_feature_cols)}개")

# ─── 2-2. 조건부행동 피처 10개 ───
print("  📊 조건부행동 피처...")
df_v2_train = df_v2.filter(pl.col("time_group")<train_cutoff)
exclude_cols = ["account_id","time_group","is_laundering","mode_format","currency_mode"]
gnn_feature_cols_v2 = [c for c in df_v2.columns if c not in exclude_cols]
df_v2_agg = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])

df_cond = df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
v2a=set(df_cond.columns)
def sc(n): return pl.col(n) if n in v2a else pl.lit(0.0)
ce=[]
if "cnt_night" in v2a and "cnt_1h" in v2a: ce.append((pl.col("graph_total_degree")*sc("cnt_night")/(sc("cnt_1h")+1)).alias("cond_highdeg_night_intensity"))
if "avg_log_amount" in v2a: ce.append((sc("avg_log_amount")/(pl.col("graph_total_degree").log1p()+1)).alias("cond_lowdeg_high_amount"))
if "ratio_cross_border" in v2a: ce.append((sc("ratio_cross_border")*(1-pl.col("graph_reciprocity"))).alias("cond_oneway_crossborder"))
if "cnt_1h" in v2a: ce.append((pl.col("graph_unique_out_partners")*sc("cnt_1h")/(pl.col("graph_out_degree")+1)).alias("cond_fanout_burst"))
if "cnt_risk_format" in v2a: ce.append((pl.col("graph_pagerank")*sc("cnt_risk_format")).alias("cond_hub_risk_format"))
if "cnt_wire" in v2a: ce.append((pl.col("graph_out_in_ratio")*sc("cnt_wire")).alias("cond_asymmetric_wire"))
if "cnt_inter_bank" in v2a: ce.append((pl.col("graph_std_tx_interval").fill_null(0)/(pl.col("graph_avg_tx_interval").fill_null(1)+1)*sc("cnt_inter_bank")).alias("cond_irregular_interbank"))
if "cnt_currencies" in v2a: ce.append((pl.col("graph_partner_diversity")*sc("cnt_currencies")).alias("cond_diverse_partner_currency"))
if "amount_kurtosis" in v2a: ce.append((pl.col("graph_reciprocity")*sc("amount_kurtosis")).alias("cond_bidir_amount_volatility"))
if "entity_acct_cnt" in v2a and "burst_ratio" in v2a: ce.append((sc("entity_acct_cnt")*sc("burst_ratio")).alias("cond_multi_acct_burst"))
df_cond=df_cond.with_columns(ce); cond_feature_cols=[e.meta.output_name() for e in ce]
for cn in cond_feature_cols: df_cond=df_cond.with_columns(pl.when(pl.col(cn).is_infinite()|pl.col(cn).is_nan()).then(0.0).otherwise(pl.col(cn)).alias(cn))
df_cond_features=df_cond.select(["account_id"]+cond_feature_cols); del df_cond; gc.collect()
print(f"  ✅ 조건부행동 {len(cond_feature_cols)}개")

# ─── 2-3. AML 점수 5개 ───
print("  📊 AML 점수...")
df_sb=df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
se=[((sc("cnt_1h").fill_null(0).clip(0,1e6)+1).log1p()*pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("burst_ratio").fill_null(0).clip(0,1e6)/(sc("avg_log_amount").fill_null(1).clip(0,1e6)+1)).alias("score_smurf"),
    ((1.0/((pl.col("graph_out_in_ratio").clip(0.01,100)-1.0).abs()+0.1))*sc("cnt_wire").fill_null(0).clip(0,1e6).log1p()*(1-pl.col("graph_reciprocity").clip(0,1))/(pl.col("graph_unique_out_partners").clip(0,1e6).log1p()+1)).alias("score_mule"),
    (pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("cnt_currencies").fill_null(0).clip(0,1e6).log1p()*sc("ratio_cross_border").fill_null(0).clip(0,1)/(pl.col("graph_avg_tx_interval").fill_null(3600).clip(0,1e9)/3600+1)).alias("score_layering"),
    (pl.col("graph_in_degree").clip(0,1e6).log1p()*(pl.col("graph_pagerank").clip(0,1)*1e6)*sc("cnt_risk_format").fill_null(0).clip(0,1e6).log1p()/(pl.col("graph_out_degree").clip(0,1e6).log1p()+1)).alias("score_integration")]
df_sb=df_sb.with_columns(se)
sn=["score_smurf","score_mule","score_layering","score_integration"]
for s in sn: df_sb=df_sb.with_columns(pl.when(pl.col(s).is_infinite()|pl.col(s).is_nan()).then(0.0).otherwise(pl.col(s)).alias(s))
for s in sn:
    mn=df_sb[s].min();mx=df_sb[s].max();rng=mx-mn if(mx-mn)>0 else 1.0
    df_sb=df_sb.with_columns(((pl.col(s)-mn)/rng).alias(f"{s}_norm"))
df_sb=df_sb.with_columns((pl.col("score_smurf_norm")*0.3+pl.col("score_mule_norm")*0.25+pl.col("score_layering_norm")*0.3+pl.col("score_integration_norm")*0.15).alias("score_composite"))
aml_score_cols=sn+["score_composite"]
df_aml_scores=df_sb.select(["account_id"]+aml_score_cols)
del df_sb, df_v2_agg; gc.collect()
print(f"  ✅ AML 점수 {len(aml_score_cols)}개")

# ─── 2-4. 규칙성 피처 7개 ───
print("  📊 규칙성 피처...")
df_tx_time = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner"),
    pl.col("Timestamp"),
    pl.col("Timestamp").dt.hour().alias("hour"),
    pl.col("Timestamp").dt.weekday().alias("dow"),
])
reg_agg = df_tx_time.group_by("account_id").agg([
    (pl.col("hour").is_between(9,17).mean()).alias("reg_business_hour_ratio"),
    (pl.col("dow").is_in([5,6]).mean()).alias("reg_weekend_ratio"),
    (pl.col("hour").is_between(0,5).mean()).alias("reg_night_strict_ratio"),
    pl.col("partner").count().alias("_total_tx"),
])
hc=df_tx_time.group_by(["account_id","hour"]).len().rename({"len":"h_count"})
hp=hc.pivot(on="hour",index="account_id",values="h_count").fill_null(0)
hcols=[c for c in hp.columns if c!="account_id"]
hmat=hp.select(hcols).to_numpy().astype(np.float64); hsum=np.maximum(hmat.sum(axis=1,keepdims=True),1.0)
hprobs=hmat/hsum; hlog=np.where(hprobs>0,np.log(hprobs+1e-12),0.0)
df_h_ent=pl.DataFrame({"account_id":hp["account_id"],"reg_hour_entropy":-np.sum(hprobs*hlog,axis=1).astype(np.float32)})
del hc,hp,hmat,hprobs,hlog; gc.collect()
dc=df_tx_time.group_by(["account_id","dow"]).len().rename({"len":"d_count"})
dp=dc.pivot(on="dow",index="account_id",values="d_count").fill_null(0)
dcols=[c for c in dp.columns if c!="account_id"]
dmat=dp.select(dcols).to_numpy().astype(np.float64); dsum=np.maximum(dmat.sum(axis=1,keepdims=True),1.0)
dprobs=dmat/dsum; dlog=np.where(dprobs>0,np.log(dprobs+1e-12),0.0)
df_d_ent=pl.DataFrame({"account_id":dp["account_id"],"reg_dow_entropy":-np.sum(dprobs*dlog,axis=1).astype(np.float32)})
del dc,dp,dmat,dprobs,dlog; gc.collect()
pf=df_tx_time.group_by(["account_id","partner"]).len().rename({"len":"p_count"})
rp=pf.filter(pl.col("p_count")>=2).group_by("account_id").len().rename({"len":"repeat_cnt"})
tp2=pf.group_by("account_id").len().rename({"len":"total_cnt"})
df_pc=tp2.join(rp,on="account_id",how="left").fill_null(0).with_columns((pl.col("repeat_cnt")/(pl.col("total_cnt")+1)).alias("reg_consistent_partner_ratio")).select(["account_id","reg_consistent_partner_ratio"])
top_p=pf.sort(["account_id","p_count"],descending=[False,True]).unique(subset=["account_id"],maintain_order=True).rename({"p_count":"top_cnt"}).select(["account_id","top_cnt"])
df_tc=reg_agg.select(["account_id","_total_tx"]).join(top_p,on="account_id",how="left").fill_null(0).with_columns((pl.col("top_cnt")/(pl.col("_total_tx")+1)).alias("reg_top_partner_concentration")).select(["account_id","reg_top_partner_concentration"])
del pf,rp,tp2,top_p; gc.collect()
df_regularity=(reg_agg.select(["account_id","reg_business_hour_ratio","reg_weekend_ratio","reg_night_strict_ratio"])
    .join(df_h_ent,on="account_id",how="left").join(df_d_ent,on="account_id",how="left")
    .join(df_pc,on="account_id",how="left").join(df_tc,on="account_id",how="left").fill_null(0.0))
regularity_cols=[c for c in df_regularity.columns if c.startswith("reg_")]
for rc in regularity_cols: df_regularity=df_regularity.with_columns(pl.when(pl.col(rc).is_infinite()|pl.col(rc).is_nan()).then(0.0).otherwise(pl.col(rc)).alias(rc))
del reg_agg,df_h_ent,df_d_ent,df_pc,df_tc; gc.collect()
print(f"  ✅ 규칙성 {len(regularity_cols)}개")

# ─── 2-5. 금액 분포 피처 8개 ───
print("  📊 금액 분포 피처...")
amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train.columns: amount_col = cand; break
if amount_col:
    df_amounts=df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col(amount_col).cast(pl.Float64).alias("amount")]).filter(pl.col("amount")>0)
    df_amt=df_amounts.group_by("account_id").agg([
        ((pl.col("amount")%1000==0)|(pl.col("amount")%5000==0)).mean().alias("amt_round_number_ratio"),
        (pl.col("amount").std()/(pl.col("amount").mean()+1)).alias("amt_cv"),
        (pl.col("amount").max()/(pl.col("amount").mean()+1)).alias("amt_max_mean_ratio"),
        (pl.col("amount").median()/(pl.col("amount").mean()+1)).alias("amt_median_mean_ratio"),
        pl.col("amount").quantile(0.9).alias("_q90"), pl.col("amount").sum().alias("_tsum"),
        pl.col("amount").count().alias("_cnt"),
        pl.col("amount").quantile(0.75).alias("_q75"), pl.col("amount").quantile(0.25).alias("_q25"),
        pl.col("amount").median().alias("_med"),
    ]).fill_null(0.0).with_columns([
        (pl.col("_q90")*0.1*pl.col("_cnt")/(pl.col("_tsum")+1)).alias("amt_top10pct_concentration"),
        ((pl.col("_q75")-pl.col("_q25"))/(pl.col("_med")+1)).alias("amt_iqr_median_ratio")])
    gm=df_amounts["amount"].mean(); st=max(gm*0.1,1.0)
    df_sr=df_amounts.with_columns((pl.col("amount")<st).alias("is_small")).group_by("account_id").agg(pl.col("is_small").mean().alias("amt_small_tx_ratio"))
    df_amt=df_amt.join(df_sr,on="account_id",how="left").fill_null(0.0)
    df_al=df_amounts.with_columns((pl.col("amount")+1).log().cast(pl.Int32).alias("log_bucket"))
    bc=df_al.group_by(["account_id","log_bucket"]).len().rename({"len":"b_count"})
    bt=bc.group_by("account_id").agg(pl.col("b_count").sum().alias("b_total"))
    bp=bc.join(bt,on="account_id").with_columns((pl.col("b_count")/pl.col("b_total")).alias("b_prob"))
    ae=bp.group_by("account_id").agg((-1*(pl.col("b_prob")*(pl.col("b_prob")+1e-12).log()).sum()).alias("amt_entropy"))
    df_amt=df_amt.join(ae,on="account_id",how="left").fill_null(0.0)
    del df_amounts,df_al,bc,bt,bp,ae,df_sr; gc.collect()
    amt_feature_cols=[c for c in df_amt.columns if c.startswith("amt_") and not c.startswith("_")]
    df_amt_features=df_amt.select(["account_id"]+amt_feature_cols)
else:
    df_amt_features=all_accounts.select("account_id"); amt_feature_cols=[]
for ac in amt_feature_cols:
    df_amt_features=df_amt_features.with_columns(pl.when(pl.col(ac).is_infinite()|pl.col(ac).is_nan()).then(0.0).otherwise(pl.col(ac)).alias(ac))
print(f"  ✅ 금액 분포 {len(amt_feature_cols)}개")

# =============================================================
# ★ NEW: 2-6. Cross-border 정밀화 피처 4개
# =============================================================
# 핵심 아이디어: ratio_cross_border 단독이 아니라
#   "cross-border × 새로운 상대방 비율 × 방향 비대칭"을 결합
print("  📊 ★ Cross-border 정밀화 피처 (신규)...")
cb_feature_cols = []

# 데이터에 to_acc country/bank 정보가 있을 경우를 가정
# → 없으면 graph_partner_diversity 와 ratio_cross_border 조합으로 근사
df_v2_agg_cb = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2 if c in df_v2_train.columns
])
df_cb = df_v2_agg_cb.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)
v2cb = set(df_cb.columns)

cb_exprs = []
# ① cb_new_partner_ratio: cross-border이면서 새로운 상대방 비율
#    = ratio_cross_border × (unique_out_partners / (total_tx_count+1))
#    → 새로운 상대에게 해외 송금하는 패턴
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("graph_unique_out_partners") /
         (pl.col("graph_total_tx_count") + 1)
        ).alias("cb_new_partner_ratio")
    )
    cb_feature_cols.append("cb_new_partner_ratio")

# ② cb_oneway_new: cross-border × 비양방향 × 새 상대방
#    → 양방향 거래 없이 새로운 상대에게만 해외 송금 (전형적 분산 송금)
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         (1.0 - pl.col("graph_reciprocity")) *
         (pl.col("graph_unique_out_partners") / (pl.col("graph_out_degree") + 1))
        ).alias("cb_oneway_new")
    )
    cb_feature_cols.append("cb_oneway_new")

# ③ cb_amount_asymmetry: 해외 송금 비율 × 금액 비대칭 (out >> in)
#    = ratio_cross_border × out_in_ratio (clip 해서 폭발 방지)
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("graph_out_in_ratio").clip(0, 20)
        ).alias("cb_amount_asymmetry")
    )
    cb_feature_cols.append("cb_amount_asymmetry")

# ④ cb_burst_cross: cross-border이 짧은 시간 내 집중 (burst_ratio 결합)
if "ratio_cross_border" in v2cb and "burst_ratio" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("burst_ratio").clip(0, 100)
        ).alias("cb_burst_cross")
    )
    cb_feature_cols.append("cb_burst_cross")

if cb_exprs:
    df_cb = df_cb.with_columns(cb_exprs)
    for cn in cb_feature_cols:
        df_cb = df_cb.with_columns(
            pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
            .then(0.0).otherwise(pl.col(cn)).alias(cn))
    df_cb_features = df_cb.select(["account_id"] + cb_feature_cols)
else:
    df_cb_features = all_accounts.select("account_id"); cb_feature_cols = []
del df_cb, df_v2_agg_cb; gc.collect()
print(f"  ✅ Cross-border 정밀화 {len(cb_feature_cols)}개: {cb_feature_cols}")

# trend_* 피처 제거 (ws_* 와 중복) — v7c.4
trend_feature_cols = []

# =============================================================
# ★ NEW: 2-8. 거래량 대비 상대 정규화 피처 5개
# =============================================================
# 핵심 아이디어: cond_lowdeg_high_amount가 FN에서 TP보다 오히려 높아 역전됨
#   → "전체 네트워크 대비 비정상적으로 높은 금액"을 더 정밀하게 포착
print("  📊 ★ 거래량 대비 상대 정규화 피처 (신규)...")
vol_norm_feature_cols = []

df_v2_agg_vn = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2 if c in df_v2_train.columns
])
df_vn = df_v2_agg_vn.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)

# 네트워크 전체 평균값 계산 (정규화 기준)
if "avg_log_amount" in set(df_vn.columns):
    global_avg_log_amount = df_vn["avg_log_amount"].mean() or 1.0
    global_avg_degree = df_vn["graph_total_degree"].mean() or 1.0
    global_avg_cnt_1h = df_vn["cnt_1h"].mean() if "cnt_1h" in df_vn.columns else 1.0
    global_avg_cnt_1h = global_avg_cnt_1h or 1.0

    vn_exprs = []
    # ① vol_amount_per_peer: 건당 금액 / (파트너 수 × 전체 평균)
    #    → degree가 낮은데 금액이 높은 패턴을 전체 대비 상대적으로 포착
    vn_exprs.append(
        (pl.col("avg_log_amount") / (float(global_avg_log_amount) + 1e-8) /
         (pl.col("graph_unique_out_partners").log1p() + 1)
        ).alias("vol_amount_per_peer")
    )
    vol_norm_feature_cols.append("vol_amount_per_peer")

    # ② vol_concentrated_amount: 소수 파트너에게 집중 송금
    #    = avg_log_amount × (1 - partner_diversity)
    #    → diversity가 낮을수록 (=한 곳에 집중) × 금액이 높을수록 높아짐
    vn_exprs.append(
        (pl.col("avg_log_amount") * (1.0 - pl.col("graph_partner_diversity").clip(0, 1))
        ).alias("vol_concentrated_amount")
    )
    vol_norm_feature_cols.append("vol_concentrated_amount")

    # ③ vol_low_freq_high_amt: 낮은 빈도이지만 건당 금액이 전체 평균 대비 높음
    #    = (avg_log_amount / global_mean) / log1p(cnt_1h) → 드문데 고액
    if "cnt_1h" in set(df_vn.columns):
        vn_exprs.append(
            (pl.col("avg_log_amount") / float(global_avg_log_amount) /
             (pl.col("cnt_1h").clip(0, 1e6).log1p() + 1)
            ).alias("vol_low_freq_high_amt")
        )
        vol_norm_feature_cols.append("vol_low_freq_high_amt")

    # ④ vol_degree_normalized_amount: 디그리로 나눈 상대 금액 (올바른 역산 버전)
    #    기존 cond_lowdeg_high_amount 대체 — log 스케일 적용 + 전체 평균으로 나눔
    vn_exprs.append(
        ((pl.col("avg_log_amount") - float(global_avg_log_amount)) /
         (pl.col("graph_total_degree").clip(1, 1e6).log1p())
        ).alias("vol_degree_normalized_amount")
    )
    vol_norm_feature_cols.append("vol_degree_normalized_amount")

    # ⑤ vol_outlier_score: z-score 방식의 금액 이상 점수
    #    = (avg_log_amount - global_mean) → 양수면 전체 평균 초과
    vn_exprs.append(
        (pl.col("avg_log_amount") - float(global_avg_log_amount)).alias("vol_outlier_score")
    )
    vol_norm_feature_cols.append("vol_outlier_score")

    df_vn = df_vn.with_columns(vn_exprs)
    for cn in vol_norm_feature_cols:
        df_vn = df_vn.with_columns(
            pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
            .then(0.0).otherwise(pl.col(cn)).alias(cn))
    df_vol_norm_features = df_vn.select(["account_id"] + vol_norm_feature_cols)
else:
    df_vol_norm_features = all_accounts.select("account_id"); vol_norm_feature_cols = []

del df_vn, df_v2_agg_vn; gc.collect()
print(f"  ✅ 거래량 대비 정규화 {len(vol_norm_feature_cols)}개: {vol_norm_feature_cols}")

# =============================================================
# ★ NEW: 2-9. 시간 윈도우별 행동 변화 피처 4개
# =============================================================
# 핵심 아이디어: 미탐의 71%가 테스트 첫 3일 집중
#   → train 말미(~recent window)에서의 행동 변화를 포착
print("  📊 ★ 시간 윈도우별 행동 변화 피처 (신규)...")
win_shift_feature_cols = []

# train 마지막 7일 vs 나머지 기간 비교
last_7d_cutoff = train_cutoff - pl.duration(days=7)
df_raw_recent = df_raw_train.filter(pl.col("Timestamp") >= last_7d_cutoff)
df_raw_older  = df_raw_train.filter(pl.col("Timestamp") < last_7d_cutoff)

recent_cnt = df_raw_recent.select(
    pl.col("from_acc").cast(pl.Utf8).alias("account_id")
).group_by("account_id").len().rename({"len": "ws_recent_cnt"})
older_cnt = df_raw_older.select(
    pl.col("from_acc").cast(pl.Utf8).alias("account_id")
).group_by("account_id").len().rename({"len": "ws_older_cnt"})

df_ws = all_accounts.select("account_id").join(recent_cnt, on="account_id", how="left").join(older_cnt, on="account_id", how="left").fill_null(0.0)

ws_exprs = []
# ① ws_recent_surge: 최근 7일 / 이전 기간 (정규화) → 갑작스러운 증가
ws_exprs.append(
    (pl.col("ws_recent_cnt") / (pl.col("ws_older_cnt") / 30.0 + 1)).alias("ws_recent_surge")
)
win_shift_feature_cols.append("ws_recent_surge")

# ② ws_recent_only: 최근 7일에만 거래가 있는 계좌 (0/1)
#    → 갑자기 나타난 계좌
ws_exprs.append(
    ((pl.col("ws_recent_cnt") > 0) & (pl.col("ws_older_cnt") == 0)).cast(pl.Float64).alias("ws_recent_only")
)
win_shift_feature_cols.append("ws_recent_only")

df_ws = df_ws.with_columns(ws_exprs)

# ③ 최근 7일 고유 파트너 급증
recent_partners = df_raw_recent.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner")
]).group_by("account_id").agg(pl.col("partner").n_unique().alias("ws_recent_partners"))
older_partners = df_raw_older.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner")
]).group_by("account_id").agg(pl.col("partner").n_unique().alias("ws_older_partners"))

df_ws = df_ws.join(recent_partners, on="account_id", how="left").join(older_partners, on="account_id", how="left").fill_null(0.0)

# ④ ws_partner_explosion: 최근 7일 파트너가 이전 대비 급증
df_ws = df_ws.with_columns(
    (pl.col("ws_recent_partners") / (pl.col("ws_older_partners") + 1)).alias("ws_partner_explosion")
)
win_shift_feature_cols.append("ws_partner_explosion")

for cn in win_shift_feature_cols:
    df_ws = df_ws.with_columns(
        pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
        .then(0.0).otherwise(pl.col(cn)).alias(cn))

df_win_shift_features = df_ws.select(["account_id"] + win_shift_feature_cols)
del df_raw_recent, df_raw_older, df_ws, recent_cnt, older_cnt
del recent_partners, older_partners; gc.collect()
print(f"  ✅ 시간 윈도우 변화 {len(win_shift_feature_cols)}개: {win_shift_feature_cols}")

# =============================================================
# ★ NEW: 2-10. 입금/출금 독립 피처 6개
# =============================================================
# 핵심 아이디어: V2에 sum_in_1h/sum_out_1h는 있지만
#   AML 점수/조건부 피처에서 in/out 독립적 분석이 부족
print("  📊 ★ 입금/출금 독립 피처 (신규)...")
inout_feature_cols = []

# 출금(from) 집계
df_out_tx = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col(amount_col).cast(pl.Float64).alias("amount") if amount_col else pl.lit(1.0).alias("amount")
]).filter(pl.col("amount") > 0)

# 입금(to) 집계
df_in_tx = df_raw_train.select([
    pl.col("to_acc").cast(pl.Utf8).alias("account_id"),
    pl.col(amount_col).cast(pl.Float64).alias("amount") if amount_col else pl.lit(1.0).alias("amount")
]).filter(pl.col("amount") > 0)

out_stats = df_out_tx.group_by("account_id").agg([
    pl.col("amount").sum().alias("inout_total_out_amount"),
    pl.col("amount").mean().alias("inout_avg_out_amount"),
    pl.col("amount").count().alias("inout_out_cnt"),
])
in_stats = df_in_tx.group_by("account_id").agg([
    pl.col("amount").sum().alias("inout_total_in_amount"),
    pl.col("amount").mean().alias("inout_avg_in_amount"),
    pl.col("amount").count().alias("inout_in_cnt"),
])
df_inout = all_accounts.select("account_id").join(out_stats, on="account_id", how="left").join(in_stats, on="account_id", how="left").fill_null(0.0)

inout_exprs = []
# ① inout_out_amount_ratio: 출금액 / (입금액 + 출금액) → 1이면 순수 송금자
inout_exprs.append(
    (pl.col("inout_total_out_amount") /
     (pl.col("inout_total_out_amount") + pl.col("inout_total_in_amount") + 1)
    ).alias("inout_out_amount_ratio")
)
inout_feature_cols.append("inout_out_amount_ratio")

# ② inout_avg_out_vs_in: 평균 출금액 / 평균 입금액 → 1 초과면 출금이 더 큰 계좌
inout_exprs.append(
    (pl.col("inout_avg_out_amount") / (pl.col("inout_avg_in_amount") + 1)
    ).alias("inout_avg_out_vs_in")
)
inout_feature_cols.append("inout_avg_out_vs_in")

# ③ inout_cnt_imbalance: (출금 건수 - 입금 건수) / 전체 건수
#    → 양수면 보내는 쪽이 훨씬 많은 계좌 (smurf 패턴)
inout_exprs.append(
    ((pl.col("inout_out_cnt") - pl.col("inout_in_cnt")) /
     (pl.col("inout_out_cnt") + pl.col("inout_in_cnt") + 1)
    ).alias("inout_cnt_imbalance")
)
inout_feature_cols.append("inout_cnt_imbalance")

# ④ inout_fan_out_score: 출금 건수 × 고유 파트너 다양성 → 여러 곳에 분산 송금
df_inout = df_inout.join(df_graph_feats.select(["account_id", "graph_partner_diversity", "graph_unique_out_partners"]), on="account_id", how="left").fill_null(0.0)
inout_exprs.append(
    (pl.col("inout_out_cnt").log1p() * pl.col("graph_partner_diversity")
    ).alias("inout_fan_out_score")
)
inout_feature_cols.append("inout_fan_out_score")

# ⑤ inout_in_concentration: 입금이 소수에서만 들어옴 (mule 패턴)
#    = avg_in_amount × (1 - partner_diversity) → 소수 소스에서 집중 수취
inout_exprs.append(
    (pl.col("inout_avg_in_amount").log1p() * (1.0 - pl.col("graph_partner_diversity").clip(0, 1))
    ).alias("inout_in_concentration")
)
inout_feature_cols.append("inout_in_concentration")

# ⑥ inout_net_flow: log(출금합) - log(입금합) → 순 흐름 방향성
inout_exprs.append(
    (pl.col("inout_total_out_amount").log1p() - pl.col("inout_total_in_amount").log1p()
    ).alias("inout_net_flow")
)
inout_feature_cols.append("inout_net_flow")

df_inout = df_inout.with_columns(inout_exprs)
for cn in inout_feature_cols:
    df_inout = df_inout.with_columns(
        pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
        .then(0.0).otherwise(pl.col(cn)).alias(cn))
df_inout_features = df_inout.select(["account_id"] + inout_feature_cols)
del df_out_tx, df_in_tx, out_stats, in_stats, df_inout; gc.collect()
print(f"  ✅ 입금/출금 독립 {len(inout_feature_cols)}개: {inout_feature_cols}")
log_memory("전체 노드 피처 완료")

# =============================================================
# 3. GNN 엣지/노드 구성 — 강화된 입력
# =============================================================
print("\n🔗 [Step 3] GNN 엣지/노드 구성 (Enriched Input)...")

df_raw_reload = pl.read_parquet(raw_path)
if df_raw_reload["Timestamp"].dtype == pl.Utf8:
    df_raw_reload = df_raw_reload.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_raw_train_edges = df_raw_reload.filter(pl.col("Timestamp")<train_cutoff); del df_raw_reload; gc.collect()

edge_amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train_edges.columns: edge_amount_col = cand; break

df_edges = df_raw_train_edges.select(
    ["from_acc","to_acc","Timestamp"] +
    ([edge_amount_col] if edge_amount_col else []) +
    (["Payment Format"] if "Payment Format" in df_raw_train_edges.columns else []) +
    (["Is Laundering"] if "Is Laundering" in df_raw_train_edges.columns else [])
).with_columns([pl.col("from_acc").cast(pl.Utf8), pl.col("to_acc").cast(pl.Utf8)])

df_edges = df_edges.join(all_accounts.rename({"account_id":"from_acc","node_id":"src_id"}),on="from_acc",how="left")
df_edges = df_edges.join(all_accounts.rename({"account_id":"to_acc","node_id":"dst_id"}),on="to_acc",how="left")

# ── 시간 피처 (5개, 기존 유지) ──────────────────────────────────────────
ts = df_edges["Timestamp"]; mn_ts=ts.min(); mx_ts=ts.max(); ts_rng=(mx_ts-mn_ts).total_seconds()
etn = ((ts-mn_ts).dt.total_microseconds().cast(pl.Float64)/1e6/max(ts_rng,1.0)).to_numpy().astype(np.float32)
hrs = df_edges["Timestamp"].dt.hour().to_numpy().astype(np.float32)
dow = df_edges["Timestamp"].dt.weekday().to_numpy().astype(np.float32)
time_feats = np.stack([etn, np.sin(2*np.pi*hrs/24), np.cos(2*np.pi*hrs/24),
                        np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7)], axis=1)

# ── 금액 피처 (4차원) ─────────────────────────────────────────────
# ★ v7c.18: relative_amt → is_large_tx (amount > 5×global_mean, 0/1)
#   근거: relative_amt ≈ log_amount 단조 변환 (중복) → 비선형 임계값 신호로 교체
if edge_amount_col:
    amounts_raw = df_edges[edge_amount_col].cast(pl.Float64).fill_null(0.0).to_numpy().astype(np.float32)
    log_amounts  = np.log1p(np.maximum(amounts_raw, 0)).astype(np.float32)
    log_amounts  = log_amounts / (np.max(log_amounts) + 1e-8)
    round_flag   = ((amounts_raw % 100 == 0) & (amounts_raw > 0)).astype(np.float32)
    global_mean_amt = np.mean(amounts_raw[amounts_raw > 0]) if np.any(amounts_raw > 0) else 1.0
    # ★ v7c.18: relative_amt 제거 → is_large_tx로 교체
    # 근거: relative_amt = log1p(amt/global_mean) ≈ log_amount - const → log_amount와 중복
    #       is_large_tx는 비선형 임계값 신호로 독립적 정보 제공
    is_large_tx  = (amounts_raw > global_mean_amt * 5).astype(np.float32)
    nonzero_amt  = amounts_raw[amounts_raw > 0]
    pct_vals     = np.percentile(nonzero_amt, np.linspace(0,100,1001)) if len(nonzero_amt)>0 else np.zeros(1001)
    amount_pct   = np.clip(np.searchsorted(pct_vals, amounts_raw, side='right').astype(np.float32)/1000.0, 0.0, 1.0)
    amount_feats = np.stack([log_amounts, round_flag, is_large_tx, amount_pct], axis=1)
else:
    amount_feats = np.zeros((len(df_edges), 4), dtype=np.float32)

# ── Payment Format 피처 (이진 정규식 2개 → 정밀 인코딩 9개) ──────────
# ★ 실제 분포 확인 결과 (HI-Medium, 7종):
#   Cheque      38.50%  세탁율 0.02%   │ ACH      12.13%  세탁율 0.79% ← 압도적 고위험
#   Credit Card 27.52%  세탁율 0.02%   │ Cash     10.09%  세탁율 0.02%
#   Reinvestment 6.10%  세탁율 0.00%   │ Wire      3.51%  세탁율 0.00%
#   Bitcoin      2.16%  세탁율 0.04%
#
# 설계 결정:
#   ① 7종 전부 원-핫 (포맷 수가 적어 "기타" 불필요)
#   ② AML 위험도 연속값 1개 (ACH=0.0079, 나머지≈0 → 강한 분리 신호)
#   ③ 기존 is_wire(정규식) 완전 제거 — Wire 세탁율 0.00%로 노이즈였음
PAYMENT_FORMATS = ["Cheque", "Credit Card", "ACH", "Cash", "Reinvestment", "Wire", "Bitcoin"]
# 포맷별 전체 데이터 기준 AML 위험도 (사전 계산값, 학습 데이터 누수 방지)
FORMAT_RISK_MAP  = {"Cheque":0.0002, "Credit Card":0.0002, "ACH":0.0079,
                    "Cash":0.0002, "Reinvestment":0.0000, "Wire":0.0000, "Bitcoin":0.0004}

if "Payment Format" in df_edges.columns:
    pf_arr = df_edges["Payment Format"].cast(pl.Utf8).fill_null("Unknown").to_numpy()

    # ① 7종 원-핫 (7차원)
    onehot_feats = np.zeros((len(pf_arr), len(PAYMENT_FORMATS)), dtype=np.float32)
    for j, fmt in enumerate(PAYMENT_FORMATS):
        onehot_feats[:, j] = (pf_arr == fmt).astype(np.float32)

    # ② AML 위험도 연속값 (1차원) — ACH가 다른 포맷 대비 ~40배 높아 강한 신호
    fmt_risk = np.array([FORMAT_RISK_MAP.get(f, 0.0002) for f in pf_arr], dtype=np.float32)

    type_feats   = np.concatenate([onehot_feats, fmt_risk.reshape(-1,1)], axis=1)  # 8차원
    N_TYPE_FEATS = type_feats.shape[1]
else:
    N_TYPE_FEATS = len(PAYMENT_FORMATS) + 1
    type_feats   = np.zeros((len(df_edges), N_TYPE_FEATS), dtype=np.float32)

edge_features_all = np.concatenate([time_feats, amount_feats, type_feats], axis=1)
EDGE_RAW_DIM = edge_features_all.shape[1]
print(f"  📊 Edge 피처: {EDGE_RAW_DIM}개")
print(f"     시간(5) + 금액(4: log/round/is_large_tx/pct) + 포맷({N_TYPE_FEATS}: 원-핫7+위험도1)")
print(f"     ★ relative_amt → is_large_tx (5×global_mean 임계값)")

edge_index_train = torch.tensor(df_edges.select(["src_id","dst_id"]).to_numpy().T, dtype=torch.long)
edge_attr_train = torch.tensor(edge_features_all, dtype=torch.float32)
print(f"📊 Train 엣지: {edge_index_train.shape[1]:,}")

del df_raw_train, df_raw_train_edges, df_edges, ts, hrs, dow, etn
del time_feats, amount_feats, type_feats, edge_features_all; gc.collect()

# ─── 노드 피처: V2(38) + 그래프(13) + 조건부(10) + AML(5) + 규칙성(7) + 금액(8) = 81개
# ★ v7c.5: GNN 입력은 v7c.2와 동일하게 유지 (신규 피처는 XGBoost에만 투입)
print("  📊 노드 피처 구성 (GNN: v7c.2 동일 81차원)...")
df_v2_node = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])
df_node_all = (all_accounts.join(df_v2_node, on="account_id", how="left")
    .join(df_graph_feats, on="account_id", how="left")
    .join(df_cond_features, on="account_id", how="left")
    .join(df_cb_features, on="account_id", how="left")           # ★ v7c.15: GNN 이동
    .join(df_vol_norm_features, on="account_id", how="left")     # ★ v7c.15: GNN 이동
    .join(df_win_shift_features, on="account_id", how="left")    # ★ v7c.15: GNN 이동
    .join(df_inout_features, on="account_id", how="left")        # ★ v7c.15: GNN 이동
    .join(df_regularity, on="account_id", how="left")            # ★ v7c.16: reg 4개 이동
    .join(df_amt_features, on="account_id", how="left")          # ★ v7c.16: amt 2개 이동
    .fill_null(0.0))

# GNN 입력 피처 결정 (역할 기반 설계)
# ─────────────────────────────────────────────────────────────
# GNN 유지 기준: 이웃 전파로 의미가 강화되는 피처
#   = 계좌 간 관계 패턴, 네트워크 구조 결합값, 방향성 흐름
#
# ★ v7c.15: XGBoost 전용 피처 중 GNN 적합 14개 이동
#
# cb_*(4개) 전부: cross-border × graph 교차항
#   → 이웃도 cb_burst_cross 높은지 전파하면 클러스터 송금 패턴 포착
#
# vol_*(4개): graph_degree/diversity 결합값
#   vol_amount_per_peer, vol_concentrated_amount,
#   vol_low_freq_high_amt, vol_degree_normalized_amount
#   → vol_outlier_score만 제외 (전체 평균 편차, 이웃 전파 의미 없음)
#
# ws_*(2개): 시간 윈도우 행동 변화
#   ws_recent_surge, ws_partner_explosion
#   → ws_recent_only 제외 (0/1 플래그, 계좌 자체 속성)
#
# inout_*(4개): 입출금 방향성 + graph 결합값
#   inout_cnt_imbalance, inout_out_amount_ratio,
#   inout_fan_out_score, inout_in_concentration
#   → inout_avg_out_vs_in, inout_net_flow 제외 (단순 비율/차이값)

# GNN 이동 대상 피처 목록 (명시적 선언)
gnn_cb_cols   = [c for c in cb_feature_cols]                          # 4개 전부
gnn_vol_cols  = [c for c in vol_norm_feature_cols
                 if c != "vol_outlier_score"]                          # 4개 (outlier 제외)
gnn_ws_cols   = [c for c in win_shift_feature_cols
                 if c in ("ws_recent_surge","ws_partner_explosion")]   # 2개
gnn_inout_cols= [c for c in inout_feature_cols
                 if c in ("inout_cnt_imbalance","inout_out_amount_ratio",
                          "inout_fan_out_score","inout_in_concentration")]  # 4개

# ★ v7c.16: 규칙성/금액 중 이웃 전파 적합한 피처 추가 이동
# reg_hour_entropy, reg_dow_entropy: 거래 시간 분포 무질서도 → 이웃 클러스터 동시 활성 시간대 포착
# reg_consistent_partner_ratio: 반복 거래 상대 비율 → ring/loop 구조 탐지
# reg_top_partner_concentration: 상위 파트너 집중도 → 허브 구조 강화
# amt_round_number_ratio: round number 동시 사용 (smurfing) → 이웃 전파로 포착
# amt_entropy: 금액 분포 다양성 → 이웃이 단조로운 금액 쓰는지 (integration 패턴)
gnn_reg_cols  = [c for c in regularity_cols
                 if c in ("reg_hour_entropy","reg_dow_entropy",
                          "reg_consistent_partner_ratio","reg_top_partner_concentration")]  # 4개
gnn_amt_cols  = [c for c in amt_feature_cols
                 if c in ("amt_round_number_ratio","amt_entropy")]     # 2개

gnn_input_cols = (gnn_feature_cols_v2 + graph_feature_cols + cond_feature_cols +
                  gnn_cb_cols + gnn_vol_cols + gnn_ws_cols + gnn_inout_cols +
                  gnn_reg_cols + gnn_amt_cols)

for col in gnn_input_cols:
    if col in df_node_all.columns:
        df_node_all = df_node_all.with_columns(
            pl.when(pl.col(col).is_infinite()|pl.col(col).is_nan()).then(0.0).otherwise(pl.col(col)).alias(col))

X_node = torch.tensor(df_node_all.select(gnn_input_cols).to_numpy(), dtype=torch.float32)
IN_DIM = X_node.shape[1]
print(f"  ✅ GNN 노드 입력: {IN_DIM}차원")
print(f"     = V2({len(gnn_feature_cols_v2)}) + 그래프({len(graph_feature_cols)}) + 조건부({len(cond_feature_cols)})")
print(f"       + CB({len(gnn_cb_cols)}) + Vol({len(gnn_vol_cols)}) + WS({len(gnn_ws_cols)}) + InOut({len(gnn_inout_cols)})")
print(f"       + Reg({len(gnn_reg_cols)}) + Amt({len(gnn_amt_cols)})  ★ v7c.16 추가")

target_labels = df_v2_train.filter(pl.col("is_laundering")==1).select("account_id").unique().with_columns(pl.lit(1).alias("label"))
df_labels = all_accounts.join(target_labels,on="account_id",how="left").fill_null(0)
Y_node = torch.tensor(df_labels["label"].to_numpy(), dtype=torch.long)
pos_node_ids = set(df_labels.filter(pl.col("label")==1)["node_id"].to_list())
active_df = pl.DataFrame({"account_id":df_v2_train["account_id"].unique(),"is_active":True})
mask_df = all_accounts.join(active_df,on="account_id",how="left").fill_null(False)
active_node_ids = mask_df.filter(pl.col("is_active"))["node_id"].to_list()
active_pos = [n for n in active_node_ids if n in pos_node_ids]
active_neg = [n for n in active_node_ids if n not in pos_node_ids]
n_neg_gnn = min(len(active_pos)*NEG_NODE_RATIO_GNN, len(active_neg))
np.random.seed(RANDOM_SEED)
sampled_neg = np.random.choice(active_neg,size=n_neg_gnn,replace=False).tolist()
sampled_nodes = set(active_pos+sampled_neg)
train_mask_np = np.zeros(num_nodes, dtype=bool)
for nid in sampled_nodes: train_mask_np[nid]=True
train_mask = torch.tensor(train_mask_np, dtype=torch.bool)
print(f"📊 GNN 노드: 양성 {len(active_pos):,} + 음성 {n_neg_gnn:,}")
del df_v2_node, df_node_all, mask_df, active_df; gc.collect()

graph_data = Data(x=X_node, edge_index=edge_index_train, edge_attr=edge_attr_train, y=Y_node)
graph_data.train_mask = train_mask; graph_data.num_nodes = num_nodes
del X_node, edge_index_train, edge_attr_train, Y_node, train_mask; gc.collect()
log_memory("GNN 데이터 준비 완료")

# =============================================================
# 4-6. Multi-Config GNN 탐색 → 최적 임베딩 선택
# =============================================================
print("\n🧠 [Step 4-6] Multi-Config GNN 탐색...")

class EdgeProjection(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, out_dim), nn.GELU(), nn.Linear(out_dim, out_dim))
    def forward(self, ea): return self.proj(ea)

class TGAT_Flex(nn.Module):
    def __init__(self, in_dim, hidden_dim, edge_raw_dim, edge_dim, n_heads, n_layers=2, dropout=0.2):
        super().__init__()
        self.edge_proj = EdgeProjection(edge_raw_dim, edge_dim)
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.n_layers = n_layers
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_layers)])
        self.convs = nn.ModuleList([
            TransformerConv(hidden_dim, hidden_dim//n_heads, heads=n_heads,
                            edge_dim=edge_dim, dropout=dropout, concat=True)
            for _ in range(n_layers)])
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim//2, 1))
        self.dropout = dropout; self.hidden_dim = hidden_dim

    def forward(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        for i in range(self.n_layers):
            h = h + F.dropout(self.convs[i](self.norms[i](h), edge_index, ee), p=self.dropout, training=self.training)
        return self.classifier(self.norm_out(h)).squeeze(-1)

    @torch.no_grad()
    def get_embedding(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        for i in range(self.n_layers):
            h = h + self.convs[i](self.norms[i](h), edge_index, ee)
        return self.norm_out(h)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"📌 디바이스: {device}")
# ★ v7c.14: pos_weight를 NEG_NODE_RATIO_GNN과 일치 (이중 보정 해소)
# 기존: 실제 비율(1:50) 기준 pw 계산 → 샘플링(1:25) 후에도 pw=50 유지 → 이중 보정
# 수정: 샘플링 비율과 동일하게 pw=NEG_NODE_RATIO_GNN=25
pw = float(NEG_NODE_RATIO_GNN)
print(f"📊 pos_weight: {pw:.1f}  (NEG_NODE_RATIO {NEG_NODE_RATIO_GNN}과 일치)")

def make_lr(opt, ep, n_ep):
    if ep < WARMUP_EPOCHS: lr = 1e-6+(BASE_LR-1e-6)*(ep/WARMUP_EPOCHS)
    else: p=(ep-WARMUP_EPOCHS)/max(n_ep-WARMUP_EPOCHS,1); lr=1e-5+(BASE_LR-1e-5)*0.5*(1+np.cos(np.pi*p))
    for pg in opt.param_groups: pg['lr']=lr
    return lr

GNN_CONFIGS = [
    {"label":"h96_L2",  "hidden_dim":96,  "n_layers":2, "n_heads":4, "edge_dim":32},  # ★ h64→h96: 입력61 대비 hidden64는 병목
    {"label":"h128_L2", "hidden_dim":128, "n_layers":2, "n_heads":4, "edge_dim":32},
    {"label":"h96_L3",  "hidden_dim":96,  "n_layers":3, "n_heads":4, "edge_dim":32},  # ★ h64→h96
    {"label":"h128_L3", "hidden_dim":128, "n_layers":3, "n_heads":4, "edge_dim":32},
]

xgb_exclude_cols=["account_id","time_group","is_laundering","mode_format","currency_mode","date"]
xgb_v2_cols=[c for c in df_v2.columns if c not in xgb_exclude_cols]
apr_train = df_v2_train.filter(pl.col("is_laundering")==1).height / max(df_v2_train.height,1)
spw = max((1-apr_train)/apr_train, 1.0) if apr_train > 0 else 20.0

# ★ 신규 피처들을 base_feature_cols에 포함
base_feature_cols = (xgb_v2_cols + graph_feature_cols + cond_feature_cols + aml_score_cols +
                     cb_feature_cols + vol_norm_feature_cols +
                     win_shift_feature_cols + inout_feature_cols)

def build_eval_df(db, df_emb_local, emb_col_names):
    feature_cols = base_feature_cols + emb_col_names
    df = (db.join(df_graph_feats,on="account_id",how="left")
          .join(df_cond_features,on="account_id",how="left")
          .join(df_aml_scores,on="account_id",how="left")
          .join(df_cb_features,on="account_id",how="left")            # ★
          .join(df_vol_norm_features,on="account_id",how="left")      # ★
          .join(df_win_shift_features,on="account_id",how="left")     # ★
          .join(df_inout_features,on="account_id",how="left")         # ★
          .join(df_emb_local,on="account_id",how="left")
          .fill_null(0.0).fill_nan(0.0))
    for cn in feature_cols:
        if cn in df.columns:
            df = df.with_columns(pl.when(pl.col(cn).is_infinite()).then(0.0).otherwise(pl.col(cn)).alias(cn))
    return df, feature_cols

def quick_xgb_eval(df_emb_local, emb_col_names, label):
    df_tr, fcols = build_eval_df(df_v2.filter(pl.col("time_group")<train_cutoff), df_emb_local, emb_col_names)
    df_tr_pos=df_tr.filter(pl.col("is_laundering")==1); df_tr_neg=df_tr.filter(pl.col("is_laundering")==0)
    n_neg=min(len(df_tr_pos)*NEG_ROW_RATIO_XGB,len(df_tr_neg))
    df_tr_s=pl.concat([df_tr_pos,df_tr_neg.sample(n=n_neg,seed=RANDOM_SEED)]).sample(fraction=1.0,seed=RANDOM_SEED)
    X_tr=df_tr_s.select(fcols).to_pandas(); y_tr=df_tr_s["is_laundering"].to_numpy()
    del df_tr,df_tr_pos,df_tr_neg,df_tr_s; gc.collect()
    df_vl, _ = build_eval_df(df_v2.filter((pl.col("time_group")>=train_cutoff)&(pl.col("time_group")<val_cutoff)), df_emb_local, emb_col_names)
    X_vl=df_vl.select(fcols).to_pandas(); y_vl=df_vl["is_laundering"].to_numpy(); del df_vl; gc.collect()
    df_te, _ = build_eval_df(df_v2.filter(pl.col("time_group")>=val_cutoff), df_emb_local, emb_col_names)
    X_te=df_te.select(fcols).to_pandas(); y_te=df_te["is_laundering"].to_numpy(); del df_te; gc.collect()
    # ★ v7c.14: Stage 3 best(d11_lr01) 근방 HP로 통일 → GNN 선택 기준 정합성 확보
    # 기존: max_depth=8, lr=0.03 → Stage 3 best(d11_lr01)와 불일치, 잘못된 GNN 선택 가능
    qp={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
        "random_state":42,"max_depth":10,"learning_rate":0.01,"scale_pos_weight":spw,
        "subsample":0.8,"colsample_bytree":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
    m=xgb.XGBClassifier(**qp,n_estimators=4000,early_stopping_rounds=100)
    m.fit(X_tr,y_tr,eval_set=[(X_vl,y_vl)],verbose=0)
    q_val=m.best_score; q_test=average_precision_score(y_te,m.predict_proba(X_te)[:,1])
    del m,X_tr,X_vl,X_te,y_tr,y_vl,y_te; gc.collect()
    return q_val, q_test

print(f"\n📊 {len(GNN_CONFIGS)}개 GNN 설정 탐색...")
print(f"  {'#':>2s} {'Label':<12s} | {'hidden':>6s} {'layers':>6s} {'edim':>4s} | {'Loss':>8s} {'Params':>8s} | {'QVal':>8s} {'QTest':>8s}")
print("  "+"-"*80)

gnn_results = []
for gi, gcfg in enumerate(GNN_CONFIGS):
    gl = gcfg["label"]; hd=gcfg["hidden_dim"]; nl=gcfg["n_layers"]
    nh=gcfg["n_heads"]; ed=gcfg["edge_dim"]
    train_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                                   input_nodes=graph_data.train_mask, shuffle=True, num_workers=4)
    full_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                                  input_nodes=None, shuffle=False, num_workers=4)
    model = TGAT_Flex(in_dim=IN_DIM, hidden_dim=hd, edge_raw_dim=EDGE_RAW_DIM,
                      edge_dim=ed, n_heads=nh, n_layers=nl, dropout=GNN_DROPOUT).to(device)
    np_count = sum(p.numel() for p in model.parameters())
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw],device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6, weight_decay=1e-4)
    model.train(); bl=float('inf'); pc2=0; bs=None
    for epoch in range(N_EPOCHS_GNN):
        clr=make_lr(optimizer,epoch,N_EPOCHS_GNN); tl=0.0; nb2=0
        for batch in tqdm(train_loader,desc=f"[{gl}] Ep{epoch+1}",leave=False):
            batch=batch.to(device); optimizer.zero_grad()
            bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
            out=model(batch.x,batch.edge_index,bea)
            loss=criterion(out[:batch.batch_size],batch.y[:batch.batch_size].float())
            if torch.isnan(loss) or torch.isinf(loss): optimizer.zero_grad(); continue
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=0.5)
            optimizer.step(); tl+=loss.item(); nb2+=1
        al=tl/max(nb2,1)
        if (epoch+1)%10==0 or epoch==0: print(f"    [{gl}] Ep{epoch+1:2d} Loss={al:.4f}")
        if al<bl-EARLY_STOP_DELTA: bl=al; pc2=0; bs={k:v.cpu().clone() for k,v in model.state_dict().items()}
        else: pc2+=1
        if pc2>=PATIENCE: break
    if bs: model.load_state_dict({k:v.to(device) for k,v in bs.items()}); del bs
    model.eval(); ae=[]
    with torch.no_grad():
        for batch in tqdm(full_loader,desc=f"[{gl}] Embed",leave=False):
            batch=batch.to(device)
            bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
            emb=model.get_embedding(batch.x,batch.edge_index,bea)
            ae.append(emb[:batch.batch_size].cpu())
    femb=torch.cat(ae,dim=0).numpy()
    del model,ae,train_loader,full_loader; torch.cuda.empty_cache(); gc.collect()
    ecn=[f"tgat_emb_{i}" for i in range(hd)]
    df_emb=pl.concat([all_accounts.select("account_id"),pl.DataFrame(femb,schema=ecn)],how="horizontal")
    qv,qt = quick_xgb_eval(df_emb, ecn, gl)
    print(f"  {gi+1:2d} {gl:<12s} | {hd:6d} {nl:6d} {ed:4d} | {bl:8.4f} {np_count:8,} | {qv:8.5f} {qt:8.4f}")
    gnn_results.append({"label":gl,"hidden_dim":hd,"n_layers":nl,"loss":bl,"params":np_count,
                         "q_val":qv,"q_test":qt,"embedding":femb,"ecn":ecn,"df_emb":df_emb})
    log_memory(f"{gl}")

best_gnn = max(gnn_results, key=lambda x: x["q_val"])
print(f"\n  🏆 Best GNN: {best_gnn['label']} (QVal={best_gnn['q_val']:.5f} QTest={best_gnn['q_test']:.4f})")
for r in gnn_results:
    if r is not best_gnn: del r["embedding"], r["df_emb"]
gc.collect()

emb_cols = best_gnn["ecn"]
df_tgat = best_gnn["df_emb"]
FINAL_HIDDEN_DIM = best_gnn["hidden_dim"]

print(f"\n🔗 Interaction 피처 ({best_gnn['label']})...")
df_tgat_with_pr=df_tgat.join(df_graph_feats.select(["account_id","graph_pagerank"]),on="account_id",how="left").fill_null(0.0)
emb_np=df_tgat_with_pr.select(emb_cols).to_numpy(); emb_var=np.var(emb_np,axis=0)
top_idx=np.argsort(emb_var)[::-1][:N_INTERACTION_DIMS]; top_ec=[emb_cols[i] for i in top_idx]
ie_l=[]; ic_l=[]
for cn in top_ec: nn2=f"interact_pr_x_{cn}"; ie_l.append((pl.col(cn)*pl.col("graph_pagerank")*1e6).alias(nn2)); ic_l.append(nn2)
ie_l.append((sum(pl.col(c)**2 for c in emb_cols).sqrt()*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_norm")); ic_l.append("interact_pr_x_emb_norm")
ie_l.append((sum(pl.col(c) for c in top_ec)*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_topsum")); ic_l.append("interact_pr_x_emb_topsum")
interaction_cols=ic_l
df_interactions=df_tgat_with_pr.with_columns(ie_l).select(["account_id"]+ic_l)
for c in ic_l: df_interactions=df_interactions.with_columns(pl.when(pl.col(c).is_infinite()|pl.col(c).is_nan()).then(0.0).otherwise(pl.col(c)).alias(c))
print(f"  ✅ Interaction {len(ic_l)}개")
del df_tgat_with_pr,emb_np,best_gnn["embedding"]; gc.collect()
del graph_data; torch.cuda.empty_cache(); gc.collect()
log_memory("GNN 탐색 완료")

# =============================================================
# 7. XGBoost 데이터 구성
# =============================================================
print("\n🚀 [Step 7] XGBoost 데이터 구성...")

# ★ v7c.7: regularity(7) + amt(8) = 15개를 XGBoost 후보에 추가 (v7c.6 누락 수정)
# GNN 입력에는 이미 포함돼 있었지만 xgb_feature_cols에서 빠져 있었음
xgb_feature_cols = (xgb_v2_cols + graph_feature_cols + cond_feature_cols + aml_score_cols +
                    regularity_cols + amt_feature_cols +          # ★ 신규 추가
                    cb_feature_cols + vol_norm_feature_cols +
                    win_shift_feature_cols + inout_feature_cols +
                    emb_cols + interaction_cols)
selected_new_cols = cb_feature_cols + vol_norm_feature_cols + win_shift_feature_cols + inout_feature_cols
print(f"📊 XGBoost 총 후보 피처: {len(xgb_feature_cols)}")
print(f"   V2:{len(xgb_v2_cols)} G:{len(graph_feature_cols)} Cond:{len(cond_feature_cols)} AML:{len(aml_score_cols)}")
print(f"   ★추가 Reg:{len(regularity_cols)} Amt:{len(amt_feature_cols)}  (v7c.6 누락 수정)")
print(f"   CB:{len(cb_feature_cols)} VolNorm:{len(vol_norm_feature_cols)}")
print(f"   WinShift:{len(win_shift_feature_cols)} InOut:{len(inout_feature_cols)}")
print(f"   Emb:{len(emb_cols)} Inter:{len(interaction_cols)}")
print(f"   ★ Forward Selection이 전체 후보에서 유효한 것 선별")

def build_xgb_df(db):
    df=(db.join(df_graph_feats,on="account_id",how="left")
          .join(df_cond_features,on="account_id",how="left")
          .join(df_aml_scores,on="account_id",how="left")
          .join(df_regularity,on="account_id",how="left")        # ★ 신규 추가
          .join(df_amt_features,on="account_id",how="left")      # ★ 신규 추가
          .join(df_cb_features,on="account_id",how="left")
          .join(df_vol_norm_features,on="account_id",how="left")
          .join(df_win_shift_features,on="account_id",how="left")
          .join(df_inout_features,on="account_id",how="left")
          .join(df_tgat,on="account_id",how="left")
          .join(df_interactions,on="account_id",how="left")
          .fill_null(0.0).fill_nan(0.0))
    for cn in xgb_feature_cols:
        if cn in df.columns: df=df.with_columns(pl.when(pl.col(cn).is_infinite()).then(0.0).otherwise(pl.col(cn)).alias(cn))
    return df

df_tr=build_xgb_df(df_v2.filter(pl.col("time_group")<train_cutoff))
df_tr_pos=df_tr.filter(pl.col("is_laundering")==1); df_tr_neg=df_tr.filter(pl.col("is_laundering")==0)
n_neg=min(len(df_tr_pos)*NEG_ROW_RATIO_XGB,len(df_tr_neg))
df_tr_s=pl.concat([df_tr_pos,df_tr_neg.sample(n=n_neg,seed=RANDOM_SEED)]).sample(fraction=1.0,seed=RANDOM_SEED)
X_train=df_tr_s.select(xgb_feature_cols).to_pandas(); y_train=df_tr_s["is_laundering"].to_numpy()
print(f"📊 Train: {len(df_tr):,} → {len(df_tr_s):,} ({y_train.mean()*100:.2f}%)")
del df_tr,df_tr_pos,df_tr_neg,df_tr_s; gc.collect()
df_vl=build_xgb_df(df_v2.filter((pl.col("time_group")>=train_cutoff)&(pl.col("time_group")<val_cutoff)))
X_val=df_vl.select(xgb_feature_cols).to_pandas(); y_val=df_vl["is_laundering"].to_numpy(); print(f"📊 Val: {len(X_val):,}"); del df_vl; gc.collect()
df_te=build_xgb_df(df_v2.filter(pl.col("time_group")>=val_cutoff))
df_test_meta=df_te.select(["account_id","time_group","is_laundering"])
X_test=df_te.select(xgb_feature_cols).to_pandas(); y_test=df_te["is_laundering"].to_numpy(); print(f"📊 Test: {len(X_test):,}")
del df_te,df_v2,df_tgat,df_graph_feats,df_cond_features,df_aml_scores
del df_cb_features,df_vol_norm_features,df_win_shift_features,df_inout_features
del df_interactions,df_regularity,df_amt_features; gc.collect()
log_memory("XGBoost 준비")

# =============================================================
# 8. XGBoost 3-Stage: Full → Forward Selection → HP Grid
# =============================================================
# ★ v7c.6 핵심 변경: Stage 1 importance 기반 top-K 선택의 한계 해결
#   문제: 상관된 피처끼리 importance를 나눠 가져서 유용한 피처가 순위 밖으로 밀림
#   해결: Stage 2를 Forward Selection으로 교체
#     → 피처를 하나씩 추가하며 Val AUCPR이 실제로 오를 때만 채택
#     → 중복/노이즈 피처는 자동으로 탈락
print("\n🔥 [Step 8] 3-Stage XGBoost (Full → Forward Selection → HP Grid)...")
apr=y_train.sum()/len(y_train); spw=max((1-apr)/apr,1.0)

# ── Stage 1: 전체 피처로 학습 → importance 순위 산출 ──────────────────
print("\n── Stage 1: 전체 피처 학습 (importance 순위 산출) ──")
s1p={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
     "random_state":42,"max_depth":8,"learning_rate":0.03,"scale_pos_weight":spw,
     "subsample":0.8,"colsample_bytree":0.7,"min_child_weight":5,
     "gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
m_s1=xgb.XGBClassifier(**s1p,n_estimators=1500,early_stopping_rounds=100)
m_s1.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=100)
s1_val=m_s1.best_score
s1_test=average_precision_score(y_test,m_s1.predict_proba(X_test)[:,1])
print(f"\n  Stage1: Val={s1_val:.5f} Test={s1_test:.4f}")
imp_s1=m_s1.feature_importances_
# importance 순위 정렬 (Stage 2 Forward Selection의 탐색 순서로 사용)
feat_imp=sorted(zip(xgb_feature_cols,imp_s1),key=lambda x:x[1],reverse=True)
del m_s1; gc.collect()

# ── Stage 2: Forward Selection ─────────────────────────────────────────
# importance 상위 순서대로 피처를 하나씩 추가하며
# Val AUCPR이 오를 때만 채택, 연속 N_PATIENCE_FS번 개선 없으면 조기 종료
print("\n── Stage 2: Forward Selection ──")
print("  피처를 importance 순서대로 하나씩 추가 → Val 개선 시 채택")

N_PATIENCE_FS = 18          # ★ 15→18: 탐색 범위 200 확대에 맞게 조정
MIN_DELTA_FS  = 5e-5
MAX_FEATURES_FS = 200       # ★ 150→200: h128 best 시 ~356개 중 42%→56% 커버리지 확보

fs_hp={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
       "random_state":42,"max_depth":9,"learning_rate":0.02,"scale_pos_weight":spw,
       "subsample":0.8,"colsample_bytree":0.8,"min_child_weight":5,
       "gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}

selected_fs = []          # 채택된 피처 누적
best_fs_val = -1.0        # 현재까지 best Val AUCPR
no_improve  = 0           # 연속 미개선 횟수
fs_log      = []          # 로그 기록

candidates = [f for f,_ in feat_imp[:MAX_FEATURES_FS]]

print(f"  {'#':>3s} {'Feature':<45s} | {'Val':>9s} {'Δ':>8s} {'채택':>4s}")
print("  " + "-" * 75)

for fi, feat in enumerate(candidates):
    probe = selected_fs + [feat]
    m_fs = xgb.XGBClassifier(**fs_hp, n_estimators=1500, early_stopping_rounds=100)  # ★ 800→1500
    m_fs.fit(X_train[probe], y_train, eval_set=[(X_val[probe], y_val)], verbose=False)
    v = m_fs.best_score
    delta = v - best_fs_val
    adopted = delta > MIN_DELTA_FS

    tag = "✅" if adopted else "  "
    print(f"  {fi+1:3d} {feat:<45s} | {v:9.5f} {delta:+8.5f} {tag}")
    fs_log.append({"feat":feat,"val":v,"delta":delta,"adopted":adopted})

    if adopted:
        selected_fs.append(feat)
        best_fs_val = v
        no_improve = 0
    else:
        no_improve += 1

    del m_fs; gc.collect()

    if no_improve >= N_PATIENCE_FS:
        print(f"\n  ⏹ 연속 {N_PATIENCE_FS}회 미개선 → 조기 종료 (탐색 {fi+1}/{len(candidates)})")
        break

print(f"\n  ✅ Forward Selection 완료: {len(selected_fs)}개 채택")
print(f"  채택 피처: {selected_fs}")
print(f"  Best Val AUCPR: {best_fs_val:.5f}  (Stage1: {s1_val:.5f}  Δ={best_fs_val-s1_val:+.5f})")

# ── Stage 3: HP Grid (Forward Selection 채택 피처 고정) ───────────────
print("\n── Stage 3: HP Grid (Forward Selection 피처 고정) ──")
# Forward Selection이 고른 피처로 HP만 탐색
bp3={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
     "random_state":42,"scale_pos_weight":spw}
hp_grid=[
    {"label":"d8_lr03_c07",  "hp":{"max_depth":8, "learning_rate":0.03, "colsample_bytree":0.7,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":1500}},
    {"label":"d8_lr02_c08",  "hp":{"max_depth":8, "learning_rate":0.02, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d9_lr02_c08",  "hp":{"max_depth":9, "learning_rate":0.02, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d9_lr02_c09",  "hp":{"max_depth":9, "learning_rate":0.02, "colsample_bytree":0.9,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d10_lr02_c08", "hp":{"max_depth":10,"learning_rate":0.02, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d10_lr01_c08", "hp":{"max_depth":10,"learning_rate":0.01, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":4000}},
    {"label":"d11_lr01_c08", "hp":{"max_depth":11,"learning_rate":0.01, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":5000}},  # v7c.11 best
    {"label":"d11_lr01_c09", "hp":{"max_depth":11,"learning_rate":0.01, "colsample_bytree":0.9,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":5000}},  # ★ 신규: d11 colsample 확대
    {"label":"d12_lr01_c08", "hp":{"max_depth":12,"learning_rate":0.01, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":5000}},  # ★ 신규: depth 한계 탐색
    {"label":"d9_reg",       "hp":{"max_depth":9, "learning_rate":0.02, "colsample_bytree":0.8,"subsample":0.75,"min_child_weight":8,"gamma":0.2,"reg_alpha":0.5,"reg_lambda":2.0,"n_estimators":2500}},
]

print(f"\n📊 {len(hp_grid)}개 HP 조합 탐색 (피처 {len(selected_fs)}개 고정)...")
print(f"  {'#':>2s} {'Label':<20s} | {'d':>2s} {'LR':>5s} {'col':>4s} | {'Val':>9s} {'it':>5s}")
print("  "+"-"*55)
best_s2={"val":-1}
for i,cfg in enumerate(hp_grid):
    hp=cfg["hp"].copy(); ne=hp.pop("n_estimators"); sf=selected_fs
    m=xgb.XGBClassifier(**{**bp3,**hp},n_estimators=ne,early_stopping_rounds=100)
    m.fit(X_train[sf],y_train,eval_set=[(X_val[sf],y_val)],verbose=False)
    va=m.best_score; bi=m.best_iteration
    print(f"  {i+1:2d} {cfg['label']:<20s} | {hp.get('max_depth',8):2d} {hp.get('learning_rate',0.03):5.3f} {hp.get('colsample_bytree',0.7):4.2f} | {va:9.5f} {bi:5d}")
    if va>best_s2["val"]:
        if best_s2.get("model"): del best_s2["model"]
        best_s2={"label":cfg["label"],"features":sf,"val":va,"iter":bi,"model":m,"hp":hp}
    else: del m
    gc.collect()

y_prob=best_s2["model"].predict_proba(X_test[best_s2["features"]])[:,1]
best_features=best_s2["features"]
print(f"\n  🏆 Best: {best_s2['label']} ({len(best_features)}개) Val={best_s2['val']:.5f} iter={best_s2['iter']}")

imp=best_s2["model"].feature_importances_; tidx=np.argsort(imp)[::-1][:30]
print(f"\n🔍 Feature Importance Top 30")
for i,idx in enumerate(tidx):
    c=best_features[idx]; t=""
    if c.startswith("tgat_emb_"): t=" [TGAT]"
    elif c.startswith("graph_"): t=" [GRAPH]"
    elif c.startswith("cond_"): t=" [COND]"
    elif c.startswith("score_"): t=" [SCORE]"
    elif c.startswith("interact_"): t=" [INTER]"
    elif c.startswith("cb_"): t=" [★CB]"
    elif c.startswith("vol_"): t=" [★VOL]"
    elif c.startswith("ws_"): t=" [★WIN]"
    elif c.startswith("inout_"): t=" [★INOUT]"
    print(f"  {i+1:2d}. {c:<45s}: {imp[idx]:.4f}{t}")

ti=imp.sum()
def group_imp(prefix): return sum(imp[i] for i,c in enumerate(best_features) if c.startswith(prefix))
print(f"\n📊 그룹별 중요도:")
print(f"   V2        : {group_imp('') / ti * 100:.1f}% (기존)")
print(f"   Graph     : {group_imp('graph_') / ti * 100:.1f}%")
print(f"   TGAT      : {group_imp('tgat_emb_') / ti * 100:.1f}%")
print(f"   Interact  : {group_imp('interact_') / ti * 100:.1f}%")
print(f"   ★ CB      : {group_imp('cb_') / ti * 100:.1f}%")
print(f"   ★ VolNorm : {group_imp('vol_') / ti * 100:.1f}%")
print(f"   ★ WinShift: {group_imp('ws_') / ti * 100:.1f}%")
print(f"   ★ InOut   : {group_imp('inout_') / ti * 100:.1f}%")
del X_train,X_val; gc.collect()

# =============================================================
# 9. 최종 리포트
# =============================================================
print("\n"+"="*60); print("🏆 [TGAT v7c.18] 최종 리포트"); print("="*60)
auprc=average_precision_score(y_test,y_prob); print(f"\n  AUPRC: {auprc:.4f}")
best_f1=0; best_thresh=0
for th in np.arange(0.05,0.95,0.01):
    f=f1_score(y_test,(y_prob>=th).astype(int),zero_division=0)
    if f>best_f1: best_f1=f; best_thresh=th
y_pred=(y_prob>=best_thresh).astype(int)
prec=precision_score(y_test,y_pred,zero_division=0); rec=recall_score(y_test,y_pred,zero_division=0)
print(f"\n📌 최적: T={best_thresh:.2f} F1={best_f1:.4f} P={prec:.4f} R={rec:.4f}")
print(f"\n📌 임계값별:")
for t in [0.1,0.3,0.5,0.7,0.8,0.9]:
    yp=(y_prob>=t).astype(int)
    print(f"  T={t:.1f} P={precision_score(y_test,yp,zero_division=0):.4f} R={recall_score(y_test,yp,zero_division=0):.4f} F1={f1_score(y_test,yp,zero_division=0):.4f}")

df_res=df_test_meta.with_columns([pl.col("time_group").dt.date().alias("date"),pl.Series("pred_prob",y_prob)])
df_dist=df_res.sort("pred_prob",descending=True).unique(subset=["account_id"],maintain_order=True)
print(f"\n📌 Top-K:")
for k in [100,500,1000,5000,10000]:
    ck=min(k,len(df_dist)); hits=df_dist.head(ck)["is_laundering"].sum()
    print(f"  Top-{k:5d}: {hits:5d}명 ({hits/ck*100:.2f}%)")
print(f"\n📌 일별 Top-100:")
for d in df_dist["date"].unique(maintain_order=True).sort(descending=True):
    hits=df_dist.filter(pl.col("date")==d).head(100)["is_laundering"].sum(); print(f"  {d}: {hits}")

tn,fp,fn,tp=confusion_matrix(y_test,y_pred).ravel()
print(f"\n📌 v7c.17→v7c.18: AUPRC {auprc:.4f} | F1 {best_f1:.4f}")
print(f"  TP:{tp:,} FP:{fp:,} FN:{fn:,} TN:{tn:,}")
print(f"  Precision:{prec:.4f} Recall:{rec:.4f}")

print(f"\n📌 GNN 탐색 결과:")
for r in gnn_results:
    tag = " ★" if r["label"]==best_gnn["label"] else ""
    print(f"  {r['label']:<12s}: QVal={r['q_val']:.5f} QTest={r['q_test']:.4f} Loss={r['loss']:.4f} Params={r['params']:,}{tag}")

print(f"\n📌 3-Stage 결과:")
print(f"  S1 전체({len(xgb_feature_cols)}개): Val={s1_val:.5f} Test={s1_test:.4f}")
print(f"  S2 Forward Selection: {len(selected_fs)}개 채택, Val={best_fs_val:.5f}")
print(f"  S3 HP Grid ({best_s2['label']}): Val={best_s2['val']:.5f} Test={auprc:.4f}")

print(f"\n✨ v7c.18 변경 요약 (v7c.17 대비):")
print(f"  엣지 피처: relative_amt → is_large_tx (5×global_mean 임계값)")
print(f"  엣지 차원: 17차원 유지 (교체, 축소 아님)")

del X_test; gc.collect(); log_memory("최종 완료")
print(f"\n✅ TGAT v7c.18 완료!")

🛡️ [TGAT v7c.18] 엣지 피처 정밀화
   ▸ relative_amt → is_large_tx (amount > 5×global_mean)
     log_amount와 중복 제거 → 비선형 임계값 신호로 교체

📂 데이터 로딩 중...
⏱️ Train cutoff: 2022-09-09 20:00:00
⏱️ Val   cutoff: 2022-09-14 05:00:00
📊 전체 노드 수: 2,076,999
  💾 [데이터 로드 완료] RAM: 74.47GB | GPU: 0.02GB

📐 [Step 2] 전체 노드 피처 계산...
  📊 그래프 구조 피처...
  📊 PageRank...
  ✅ 그래프 피처 13개
  📊 조건부행동 피처...
  ✅ 조건부행동 10개
  📊 AML 점수...
  ✅ AML 점수 5개
  📊 규칙성 피처...
  ✅ 규칙성 7개
  📊 금액 분포 피처...
  ✅ 금액 분포 8개
  📊 ★ Cross-border 정밀화 피처 (신규)...
  ✅ Cross-border 정밀화 4개: ['cb_new_partner_ratio', 'cb_oneway_new', 'cb_amount_asymmetry', 'cb_burst_cross']
  📊 ★ 거래량 대비 상대 정규화 피처 (신규)...
  ✅ 거래량 대비 정규화 5개: ['vol_amount_per_peer', 'vol_concentrated_amount', 'vol_low_freq_high_amt', 'vol_degree_normalized_amount', 'vol_outlier_score']
  📊 ★ 시간 윈도우별 행동 변화 피처 (신규)...
  ✅ 시간 윈도우 변화 3개: ['ws_recent_surge', 'ws_recent_only', 'ws_partner_explosion']
  📊 ★ 입금/출금 독립 피처 (신규)...
  ✅ 입금/출금 독립 6개: ['inout_out_amount_ratio', 'inout_avg_out_vs_in', 'inout_cnt_

    [h96_L2] Ep 1 Loss=1.3711


    [h96_L2] Ep10 Loss=1.0399


    [h96_L2] Ep20 Loss=1.0150


    [h96_L2] Ep30 Loss=1.0003


...

    [h128_L3] Ep50 Loss=0.9716


    [h128_L3] Ep60 Loss=0.9646


    [h128_L3] Ep70 Loss=0.9588


    [h128_L3] Ep80 Loss=0.9558


   4 h128_L3      |    128      3   32 |   0.9551  231,905 |  0.41332   0.4386
  💾 [h128_L3] RAM: 30.41GB | GPU: 0.02GB

  🏆 Best GNN: h96_L3 (QVal=0.41377 QTest=0.4367)

🔗 Interaction 피처 (h96_L3)...
  ✅ Interaction 10개
  💾 [GNN 탐색 완료] RAM: 27.09GB | GPU: 0.02GB

🚀 [Step 7] XGBoost 데이터 구성...
📊 XGBoost 총 후보 피처: 205
   V2:38 G:13 Cond:10 AML:5
   ★추가 Reg:7 Amt:8  (v7c.6 누락 수정)
   CB:4 VolNorm:5
   WinShift:3 InOut:6
   Emb:96 Inter:10
   ★ Forward Selection이 전체 후보에서 유효한 것 선별
📊 Train: 17,172,747 → 619,437 (4.76%)
📊 Val: 5,790,644
📊 Test: 5,792,913
  💾 [XGBoost 준비] RAM: 94.67GB | GPU: 0.02GB

🔥 [Step 8] 3-Stage XGBoost (Full → Forward Selection → HP Grid)...

── Stage 1: 전체 피처 학습 (importance 순위 산출) ──
[0]	validation_0-aucpr:0.11690
[100]	validation_0-aucpr:0.28114
[200]	validation_0-aucpr:0.37055
[300]	validation_0-aucpr:0.38998
[400]	validation_0-aucpr:0.39865
[500]	validation_0-aucpr:0.40390
[600]	validation_0-aucpr:0.41058
[700]	validation_0-aucpr:0.41270
[800]	validation_0-aucpr:0.4190

In [6]:
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv, GATv2Conv, GINEConv
from torch_geometric.loader import NeighborLoader
import xgboost as xgb
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm import tqdm
import gc, numpy as np, psutil, os, warnings
warnings.filterwarnings("ignore", category=UserWarning)

def log_memory(tag=""):
    rss = psutil.Process(os.getpid()).memory_info().rss/(1024**3)
    gpu = torch.cuda.memory_allocated()/(1024**3) if torch.cuda.is_available() else 0
    print(f"  💾 [{tag}] RAM: {rss:.2f}GB | GPU: {gpu:.2f}GB")

print("=" * 60)
print("🛡️ [TGAT v7c.19] 레이어 비교 실험")
print("   ▸ TransformerConv (기존, SUM+attention) × 4")
print("   ▸ GATv2Conv (동적 어텐션 SUM) × 2")
print("   ▸ NNConv+MAX (엣지 변환행렬×MAX 집계) × 2  ← AML 핵심 실험")
print("   ▸ GINEConv+SUM (WL 이론적 최강 표현력) × 2")
print("   총 10개 config → QVal 기준 best 선택 후 3-Stage")
print("=" * 60)

# ===================== 설정 =====================
NEG_NODE_RATIO_GNN = 25
NEG_ROW_RATIO_XGB = 20; RANDOM_SEED = 42
EDGE_DIM = 32             # ★ HIDDEN_DIM 전역 상수 제거 (GNN_CONFIGS에서 직접 지정)
N_HEADS = 4
N_EPOCHS_GNN = 80             # ★ 100→80: 과학습 전 종료
BATCH_SIZE = 2048
NUM_NEIGHBORS = [25, 15]
N_INTERACTION_DIMS = 8
PATIENCE = 10
EARLY_STOP_DELTA = 0.001
BASE_LR = 0.001               # ★ 0.003→0.001: LR 복구 (0.003은 QVal 하락 유발)
WARMUP_EPOCHS = 5
GNN_DROPOUT = 0.3             # ★ 0.2→0.3: 과적합 억제

raw_path = "/home/tracerofjageum/HI-Medium_Master.parquet"
v2_path = "/home/tracerofjageum/aml_features_v2_integrated_final.parquet"

# =============================================================
# 1. 데이터 로드
# =============================================================
print("\n📂 데이터 로딩 중...")
df_raw = pl.read_parquet(raw_path)
df_v2 = pl.read_parquet(v2_path).with_columns(pl.col("account_id").cast(pl.Utf8))
if df_raw["Timestamp"].dtype == pl.Utf8:
    df_raw = df_raw.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_v2 = df_v2.sort("time_group")
total_count = len(df_v2)
train_cutoff = df_v2["time_group"][int(total_count*0.6)]
val_cutoff = df_v2["time_group"][int(total_count*0.8)]
print(f"⏱️ Train cutoff: {train_cutoff}\n⏱️ Val   cutoff: {val_cutoff}")

all_accounts = pl.concat([
    df_raw.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id")),
    df_raw.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
]).unique().with_row_index("node_id")
num_nodes = len(all_accounts); print(f"📊 전체 노드 수: {num_nodes:,}")
log_memory("데이터 로드 완료")

# =============================================================
# 2. 모든 노드 피처 계산 (GNN 입력용 + XGBoost 겸용)
# =============================================================
print("\n📐 [Step 2] 전체 노드 피처 계산...")
df_raw_train = df_raw.filter(pl.col("Timestamp") < train_cutoff)

# ─── 2-1. 그래프 구조 피처 13개 ───
print("  📊 그래프 구조 피처...")
df_from = df_raw_train.select(pl.col("from_acc").cast(pl.Utf8).alias("account_id"))
df_to = df_raw_train.select(pl.col("to_acc").cast(pl.Utf8).alias("account_id"))
out_degree = df_from.group_by("account_id").len().rename({"len":"graph_out_degree"})
in_degree = df_to.group_by("account_id").len().rename({"len":"graph_in_degree"})
unique_out = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("to_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_out_partners"))
unique_in = df_raw_train.select([pl.col("to_acc").cast(pl.Utf8).alias("account_id"),pl.col("from_acc").cast(pl.Utf8).alias("partner")]).group_by("account_id").agg(pl.col("partner").n_unique().alias("graph_unique_in_partners"))
esf = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("a"),pl.col("to_acc").cast(pl.Utf8).alias("b")]).unique()
est = esf.select([pl.col("b").alias("a"),pl.col("a").alias("b")])
bidir = esf.join(est, on=["a","b"], how="inner")
bidir_count = bidir.select(pl.col("a").alias("account_id")).group_by("account_id").len().rename({"len":"graph_bidir_count"})
del esf, est, bidir; gc.collect()
time_features = df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col("Timestamp")]).group_by("account_id").agg([
    pl.col("Timestamp").sort().diff().dt.total_seconds().mean().alias("graph_avg_tx_interval"),
    pl.col("Timestamp").sort().diff().dt.total_seconds().std().alias("graph_std_tx_interval"),
    pl.col("Timestamp").count().alias("graph_total_tx_count")])

# PageRank
print("  📊 PageRank...")
account_to_id = dict(zip(all_accounts["account_id"].to_list(), all_accounts["node_id"].to_list()))
src_ids = df_raw_train["from_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
dst_ids = df_raw_train["to_acc"].cast(pl.Utf8).map_elements(lambda x: account_to_id.get(x,-1), return_dtype=pl.Int64).to_numpy()
vm=(src_ids>=0)&(dst_ids>=0); sv=src_ids[vm]; dv=dst_ids[vm]
oda=np.zeros(num_nodes,dtype=np.float64); np.add.at(oda,sv,1.0); oda=np.maximum(oda,1.0)
pr=np.ones(num_nodes,dtype=np.float64)/num_nodes
for _ in range(10): npr=np.ones(num_nodes,dtype=np.float64)*0.15/num_nodes; np.add.at(npr,dv,0.85*pr[sv]/oda[sv]); pr=npr
pr_df = pl.DataFrame({"node_id":np.arange(num_nodes,dtype=np.uint32),"graph_pagerank":pr.astype(np.float32)})
pr_df = all_accounts.join(pr_df,on="node_id",how="left").select(["account_id","graph_pagerank"])
del src_ids,dst_ids,sv,dv,oda,pr,npr; gc.collect()

df_graph_feats = (all_accounts.select("account_id").join(out_degree,on="account_id",how="left").join(in_degree,on="account_id",how="left")
    .join(unique_out,on="account_id",how="left").join(unique_in,on="account_id",how="left")
    .join(bidir_count,on="account_id",how="left").join(time_features,on="account_id",how="left")
    .join(pr_df,on="account_id",how="left").fill_null(0.0)
    .with_columns([(pl.col("graph_out_degree")+pl.col("graph_in_degree")).alias("graph_total_degree"),
        (pl.col("graph_out_degree")/(pl.col("graph_in_degree")+1)).alias("graph_out_in_ratio"),
        (pl.col("graph_bidir_count")/(pl.col("graph_out_degree")+1)).alias("graph_reciprocity"),
        (pl.col("graph_unique_out_partners")/(pl.col("graph_out_degree")+1)).alias("graph_partner_diversity")]))
graph_feature_cols = [c for c in df_graph_feats.columns if c.startswith("graph_")]
del out_degree,in_degree,unique_out,unique_in,bidir_count,time_features,pr_df,df_from,df_to; gc.collect()
print(f"  ✅ 그래프 피처 {len(graph_feature_cols)}개")

# ─── 2-2. 조건부행동 피처 10개 ───
print("  📊 조건부행동 피처...")
df_v2_train = df_v2.filter(pl.col("time_group")<train_cutoff)
exclude_cols = ["account_id","time_group","is_laundering","mode_format","currency_mode"]
gnn_feature_cols_v2 = [c for c in df_v2.columns if c not in exclude_cols]
df_v2_agg = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])

df_cond = df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
v2a=set(df_cond.columns)
def sc(n): return pl.col(n) if n in v2a else pl.lit(0.0)
ce=[]
if "cnt_night" in v2a and "cnt_1h" in v2a: ce.append((pl.col("graph_total_degree")*sc("cnt_night")/(sc("cnt_1h")+1)).alias("cond_highdeg_night_intensity"))
if "avg_log_amount" in v2a: ce.append((sc("avg_log_amount")/(pl.col("graph_total_degree").log1p()+1)).alias("cond_lowdeg_high_amount"))
if "ratio_cross_border" in v2a: ce.append((sc("ratio_cross_border")*(1-pl.col("graph_reciprocity"))).alias("cond_oneway_crossborder"))
if "cnt_1h" in v2a: ce.append((pl.col("graph_unique_out_partners")*sc("cnt_1h")/(pl.col("graph_out_degree")+1)).alias("cond_fanout_burst"))
if "cnt_risk_format" in v2a: ce.append((pl.col("graph_pagerank")*sc("cnt_risk_format")).alias("cond_hub_risk_format"))
if "cnt_wire" in v2a: ce.append((pl.col("graph_out_in_ratio")*sc("cnt_wire")).alias("cond_asymmetric_wire"))
if "cnt_inter_bank" in v2a: ce.append((pl.col("graph_std_tx_interval").fill_null(0)/(pl.col("graph_avg_tx_interval").fill_null(1)+1)*sc("cnt_inter_bank")).alias("cond_irregular_interbank"))
if "cnt_currencies" in v2a: ce.append((pl.col("graph_partner_diversity")*sc("cnt_currencies")).alias("cond_diverse_partner_currency"))
if "amount_kurtosis" in v2a: ce.append((pl.col("graph_reciprocity")*sc("amount_kurtosis")).alias("cond_bidir_amount_volatility"))
if "entity_acct_cnt" in v2a and "burst_ratio" in v2a: ce.append((sc("entity_acct_cnt")*sc("burst_ratio")).alias("cond_multi_acct_burst"))
df_cond=df_cond.with_columns(ce); cond_feature_cols=[e.meta.output_name() for e in ce]
for cn in cond_feature_cols: df_cond=df_cond.with_columns(pl.when(pl.col(cn).is_infinite()|pl.col(cn).is_nan()).then(0.0).otherwise(pl.col(cn)).alias(cn))
df_cond_features=df_cond.select(["account_id"]+cond_feature_cols); del df_cond; gc.collect()
print(f"  ✅ 조건부행동 {len(cond_feature_cols)}개")

# ─── 2-3. AML 점수 5개 ───
print("  📊 AML 점수...")
df_sb=df_v2_agg.join(df_graph_feats,on="account_id",how="left").fill_null(0.0)
se=[((sc("cnt_1h").fill_null(0).clip(0,1e6)+1).log1p()*pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("burst_ratio").fill_null(0).clip(0,1e6)/(sc("avg_log_amount").fill_null(1).clip(0,1e6)+1)).alias("score_smurf"),
    ((1.0/((pl.col("graph_out_in_ratio").clip(0.01,100)-1.0).abs()+0.1))*sc("cnt_wire").fill_null(0).clip(0,1e6).log1p()*(1-pl.col("graph_reciprocity").clip(0,1))/(pl.col("graph_unique_out_partners").clip(0,1e6).log1p()+1)).alias("score_mule"),
    (pl.col("graph_unique_out_partners").clip(0,1e6).log1p()*sc("cnt_currencies").fill_null(0).clip(0,1e6).log1p()*sc("ratio_cross_border").fill_null(0).clip(0,1)/(pl.col("graph_avg_tx_interval").fill_null(3600).clip(0,1e9)/3600+1)).alias("score_layering"),
    (pl.col("graph_in_degree").clip(0,1e6).log1p()*(pl.col("graph_pagerank").clip(0,1)*1e6)*sc("cnt_risk_format").fill_null(0).clip(0,1e6).log1p()/(pl.col("graph_out_degree").clip(0,1e6).log1p()+1)).alias("score_integration")]
df_sb=df_sb.with_columns(se)
sn=["score_smurf","score_mule","score_layering","score_integration"]
for s in sn: df_sb=df_sb.with_columns(pl.when(pl.col(s).is_infinite()|pl.col(s).is_nan()).then(0.0).otherwise(pl.col(s)).alias(s))
for s in sn:
    mn=df_sb[s].min();mx=df_sb[s].max();rng=mx-mn if(mx-mn)>0 else 1.0
    df_sb=df_sb.with_columns(((pl.col(s)-mn)/rng).alias(f"{s}_norm"))
df_sb=df_sb.with_columns((pl.col("score_smurf_norm")*0.3+pl.col("score_mule_norm")*0.25+pl.col("score_layering_norm")*0.3+pl.col("score_integration_norm")*0.15).alias("score_composite"))
aml_score_cols=sn+["score_composite"]
df_aml_scores=df_sb.select(["account_id"]+aml_score_cols)
del df_sb, df_v2_agg; gc.collect()
print(f"  ✅ AML 점수 {len(aml_score_cols)}개")

# ─── 2-4. 규칙성 피처 7개 ───
print("  📊 규칙성 피처...")
df_tx_time = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner"),
    pl.col("Timestamp"),
    pl.col("Timestamp").dt.hour().alias("hour"),
    pl.col("Timestamp").dt.weekday().alias("dow"),
])
reg_agg = df_tx_time.group_by("account_id").agg([
    (pl.col("hour").is_between(9,17).mean()).alias("reg_business_hour_ratio"),
    (pl.col("dow").is_in([5,6]).mean()).alias("reg_weekend_ratio"),
    (pl.col("hour").is_between(0,5).mean()).alias("reg_night_strict_ratio"),
    pl.col("partner").count().alias("_total_tx"),
])
hc=df_tx_time.group_by(["account_id","hour"]).len().rename({"len":"h_count"})
hp=hc.pivot(on="hour",index="account_id",values="h_count").fill_null(0)
hcols=[c for c in hp.columns if c!="account_id"]
hmat=hp.select(hcols).to_numpy().astype(np.float64); hsum=np.maximum(hmat.sum(axis=1,keepdims=True),1.0)
hprobs=hmat/hsum; hlog=np.where(hprobs>0,np.log(hprobs+1e-12),0.0)
df_h_ent=pl.DataFrame({"account_id":hp["account_id"],"reg_hour_entropy":-np.sum(hprobs*hlog,axis=1).astype(np.float32)})
del hc,hp,hmat,hprobs,hlog; gc.collect()
dc=df_tx_time.group_by(["account_id","dow"]).len().rename({"len":"d_count"})
dp=dc.pivot(on="dow",index="account_id",values="d_count").fill_null(0)
dcols=[c for c in dp.columns if c!="account_id"]
dmat=dp.select(dcols).to_numpy().astype(np.float64); dsum=np.maximum(dmat.sum(axis=1,keepdims=True),1.0)
dprobs=dmat/dsum; dlog=np.where(dprobs>0,np.log(dprobs+1e-12),0.0)
df_d_ent=pl.DataFrame({"account_id":dp["account_id"],"reg_dow_entropy":-np.sum(dprobs*dlog,axis=1).astype(np.float32)})
del dc,dp,dmat,dprobs,dlog; gc.collect()
pf=df_tx_time.group_by(["account_id","partner"]).len().rename({"len":"p_count"})
rp=pf.filter(pl.col("p_count")>=2).group_by("account_id").len().rename({"len":"repeat_cnt"})
tp2=pf.group_by("account_id").len().rename({"len":"total_cnt"})
df_pc=tp2.join(rp,on="account_id",how="left").fill_null(0).with_columns((pl.col("repeat_cnt")/(pl.col("total_cnt")+1)).alias("reg_consistent_partner_ratio")).select(["account_id","reg_consistent_partner_ratio"])
top_p=pf.sort(["account_id","p_count"],descending=[False,True]).unique(subset=["account_id"],maintain_order=True).rename({"p_count":"top_cnt"}).select(["account_id","top_cnt"])
df_tc=reg_agg.select(["account_id","_total_tx"]).join(top_p,on="account_id",how="left").fill_null(0).with_columns((pl.col("top_cnt")/(pl.col("_total_tx")+1)).alias("reg_top_partner_concentration")).select(["account_id","reg_top_partner_concentration"])
del pf,rp,tp2,top_p; gc.collect()
df_regularity=(reg_agg.select(["account_id","reg_business_hour_ratio","reg_weekend_ratio","reg_night_strict_ratio"])
    .join(df_h_ent,on="account_id",how="left").join(df_d_ent,on="account_id",how="left")
    .join(df_pc,on="account_id",how="left").join(df_tc,on="account_id",how="left").fill_null(0.0))
regularity_cols=[c for c in df_regularity.columns if c.startswith("reg_")]
for rc in regularity_cols: df_regularity=df_regularity.with_columns(pl.when(pl.col(rc).is_infinite()|pl.col(rc).is_nan()).then(0.0).otherwise(pl.col(rc)).alias(rc))
del reg_agg,df_h_ent,df_d_ent,df_pc,df_tc; gc.collect()
print(f"  ✅ 규칙성 {len(regularity_cols)}개")

# ─── 2-5. 금액 분포 피처 8개 ───
print("  📊 금액 분포 피처...")
amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train.columns: amount_col = cand; break
if amount_col:
    df_amounts=df_raw_train.select([pl.col("from_acc").cast(pl.Utf8).alias("account_id"),pl.col(amount_col).cast(pl.Float64).alias("amount")]).filter(pl.col("amount")>0)
    df_amt=df_amounts.group_by("account_id").agg([
        ((pl.col("amount")%1000==0)|(pl.col("amount")%5000==0)).mean().alias("amt_round_number_ratio"),
        (pl.col("amount").std()/(pl.col("amount").mean()+1)).alias("amt_cv"),
        (pl.col("amount").max()/(pl.col("amount").mean()+1)).alias("amt_max_mean_ratio"),
        (pl.col("amount").median()/(pl.col("amount").mean()+1)).alias("amt_median_mean_ratio"),
        pl.col("amount").quantile(0.9).alias("_q90"), pl.col("amount").sum().alias("_tsum"),
        pl.col("amount").count().alias("_cnt"),
        pl.col("amount").quantile(0.75).alias("_q75"), pl.col("amount").quantile(0.25).alias("_q25"),
        pl.col("amount").median().alias("_med"),
    ]).fill_null(0.0).with_columns([
        (pl.col("_q90")*0.1*pl.col("_cnt")/(pl.col("_tsum")+1)).alias("amt_top10pct_concentration"),
        ((pl.col("_q75")-pl.col("_q25"))/(pl.col("_med")+1)).alias("amt_iqr_median_ratio")])
    gm=df_amounts["amount"].mean(); st=max(gm*0.1,1.0)
    df_sr=df_amounts.with_columns((pl.col("amount")<st).alias("is_small")).group_by("account_id").agg(pl.col("is_small").mean().alias("amt_small_tx_ratio"))
    df_amt=df_amt.join(df_sr,on="account_id",how="left").fill_null(0.0)
    df_al=df_amounts.with_columns((pl.col("amount")+1).log().cast(pl.Int32).alias("log_bucket"))
    bc=df_al.group_by(["account_id","log_bucket"]).len().rename({"len":"b_count"})
    bt=bc.group_by("account_id").agg(pl.col("b_count").sum().alias("b_total"))
    bp=bc.join(bt,on="account_id").with_columns((pl.col("b_count")/pl.col("b_total")).alias("b_prob"))
    ae=bp.group_by("account_id").agg((-1*(pl.col("b_prob")*(pl.col("b_prob")+1e-12).log()).sum()).alias("amt_entropy"))
    df_amt=df_amt.join(ae,on="account_id",how="left").fill_null(0.0)
    del df_amounts,df_al,bc,bt,bp,ae,df_sr; gc.collect()
    amt_feature_cols=[c for c in df_amt.columns if c.startswith("amt_") and not c.startswith("_")]
    df_amt_features=df_amt.select(["account_id"]+amt_feature_cols)
else:
    df_amt_features=all_accounts.select("account_id"); amt_feature_cols=[]
for ac in amt_feature_cols:
    df_amt_features=df_amt_features.with_columns(pl.when(pl.col(ac).is_infinite()|pl.col(ac).is_nan()).then(0.0).otherwise(pl.col(ac)).alias(ac))
print(f"  ✅ 금액 분포 {len(amt_feature_cols)}개")

# =============================================================
# ★ NEW: 2-6. Cross-border 정밀화 피처 4개
# =============================================================
# 핵심 아이디어: ratio_cross_border 단독이 아니라
#   "cross-border × 새로운 상대방 비율 × 방향 비대칭"을 결합
print("  📊 ★ Cross-border 정밀화 피처 (신규)...")
cb_feature_cols = []

# 데이터에 to_acc country/bank 정보가 있을 경우를 가정
# → 없으면 graph_partner_diversity 와 ratio_cross_border 조합으로 근사
df_v2_agg_cb = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2 if c in df_v2_train.columns
])
df_cb = df_v2_agg_cb.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)
v2cb = set(df_cb.columns)

cb_exprs = []
# ① cb_new_partner_ratio: cross-border이면서 새로운 상대방 비율
#    = ratio_cross_border × (unique_out_partners / (total_tx_count+1))
#    → 새로운 상대에게 해외 송금하는 패턴
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("graph_unique_out_partners") /
         (pl.col("graph_total_tx_count") + 1)
        ).alias("cb_new_partner_ratio")
    )
    cb_feature_cols.append("cb_new_partner_ratio")

# ② cb_oneway_new: cross-border × 비양방향 × 새 상대방
#    → 양방향 거래 없이 새로운 상대에게만 해외 송금 (전형적 분산 송금)
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         (1.0 - pl.col("graph_reciprocity")) *
         (pl.col("graph_unique_out_partners") / (pl.col("graph_out_degree") + 1))
        ).alias("cb_oneway_new")
    )
    cb_feature_cols.append("cb_oneway_new")

# ③ cb_amount_asymmetry: 해외 송금 비율 × 금액 비대칭 (out >> in)
#    = ratio_cross_border × out_in_ratio (clip 해서 폭발 방지)
if "ratio_cross_border" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("graph_out_in_ratio").clip(0, 20)
        ).alias("cb_amount_asymmetry")
    )
    cb_feature_cols.append("cb_amount_asymmetry")

# ④ cb_burst_cross: cross-border이 짧은 시간 내 집중 (burst_ratio 결합)
if "ratio_cross_border" in v2cb and "burst_ratio" in v2cb:
    cb_exprs.append(
        (pl.col("ratio_cross_border") *
         pl.col("burst_ratio").clip(0, 100)
        ).alias("cb_burst_cross")
    )
    cb_feature_cols.append("cb_burst_cross")

if cb_exprs:
    df_cb = df_cb.with_columns(cb_exprs)
    for cn in cb_feature_cols:
        df_cb = df_cb.with_columns(
            pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
            .then(0.0).otherwise(pl.col(cn)).alias(cn))
    df_cb_features = df_cb.select(["account_id"] + cb_feature_cols)
else:
    df_cb_features = all_accounts.select("account_id"); cb_feature_cols = []
del df_cb, df_v2_agg_cb; gc.collect()
print(f"  ✅ Cross-border 정밀화 {len(cb_feature_cols)}개: {cb_feature_cols}")

# trend_* 피처 제거 (ws_* 와 중복) — v7c.4
trend_feature_cols = []

# =============================================================
# ★ NEW: 2-8. 거래량 대비 상대 정규화 피처 5개
# =============================================================
# 핵심 아이디어: cond_lowdeg_high_amount가 FN에서 TP보다 오히려 높아 역전됨
#   → "전체 네트워크 대비 비정상적으로 높은 금액"을 더 정밀하게 포착
print("  📊 ★ 거래량 대비 상대 정규화 피처 (신규)...")
vol_norm_feature_cols = []

df_v2_agg_vn = df_v2_train.group_by("account_id").agg([
    pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2 if c in df_v2_train.columns
])
df_vn = df_v2_agg_vn.join(df_graph_feats, on="account_id", how="left").fill_null(0.0)

# 네트워크 전체 평균값 계산 (정규화 기준)
if "avg_log_amount" in set(df_vn.columns):
    global_avg_log_amount = df_vn["avg_log_amount"].mean() or 1.0
    global_avg_degree = df_vn["graph_total_degree"].mean() or 1.0
    global_avg_cnt_1h = df_vn["cnt_1h"].mean() if "cnt_1h" in df_vn.columns else 1.0
    global_avg_cnt_1h = global_avg_cnt_1h or 1.0

    vn_exprs = []
    # ① vol_amount_per_peer: 건당 금액 / (파트너 수 × 전체 평균)
    #    → degree가 낮은데 금액이 높은 패턴을 전체 대비 상대적으로 포착
    vn_exprs.append(
        (pl.col("avg_log_amount") / (float(global_avg_log_amount) + 1e-8) /
         (pl.col("graph_unique_out_partners").log1p() + 1)
        ).alias("vol_amount_per_peer")
    )
    vol_norm_feature_cols.append("vol_amount_per_peer")

    # ② vol_concentrated_amount: 소수 파트너에게 집중 송금
    #    = avg_log_amount × (1 - partner_diversity)
    #    → diversity가 낮을수록 (=한 곳에 집중) × 금액이 높을수록 높아짐
    vn_exprs.append(
        (pl.col("avg_log_amount") * (1.0 - pl.col("graph_partner_diversity").clip(0, 1))
        ).alias("vol_concentrated_amount")
    )
    vol_norm_feature_cols.append("vol_concentrated_amount")

    # ③ vol_low_freq_high_amt: 낮은 빈도이지만 건당 금액이 전체 평균 대비 높음
    #    = (avg_log_amount / global_mean) / log1p(cnt_1h) → 드문데 고액
    if "cnt_1h" in set(df_vn.columns):
        vn_exprs.append(
            (pl.col("avg_log_amount") / float(global_avg_log_amount) /
             (pl.col("cnt_1h").clip(0, 1e6).log1p() + 1)
            ).alias("vol_low_freq_high_amt")
        )
        vol_norm_feature_cols.append("vol_low_freq_high_amt")

    # ④ vol_degree_normalized_amount: 디그리로 나눈 상대 금액 (올바른 역산 버전)
    #    기존 cond_lowdeg_high_amount 대체 — log 스케일 적용 + 전체 평균으로 나눔
    vn_exprs.append(
        ((pl.col("avg_log_amount") - float(global_avg_log_amount)) /
         (pl.col("graph_total_degree").clip(1, 1e6).log1p())
        ).alias("vol_degree_normalized_amount")
    )
    vol_norm_feature_cols.append("vol_degree_normalized_amount")

    # ⑤ vol_outlier_score: z-score 방식의 금액 이상 점수
    #    = (avg_log_amount - global_mean) → 양수면 전체 평균 초과
    vn_exprs.append(
        (pl.col("avg_log_amount") - float(global_avg_log_amount)).alias("vol_outlier_score")
    )
    vol_norm_feature_cols.append("vol_outlier_score")

    df_vn = df_vn.with_columns(vn_exprs)
    for cn in vol_norm_feature_cols:
        df_vn = df_vn.with_columns(
            pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
            .then(0.0).otherwise(pl.col(cn)).alias(cn))
    df_vol_norm_features = df_vn.select(["account_id"] + vol_norm_feature_cols)
else:
    df_vol_norm_features = all_accounts.select("account_id"); vol_norm_feature_cols = []

del df_vn, df_v2_agg_vn; gc.collect()
print(f"  ✅ 거래량 대비 정규화 {len(vol_norm_feature_cols)}개: {vol_norm_feature_cols}")

# =============================================================
# ★ NEW: 2-9. 시간 윈도우별 행동 변화 피처 4개
# =============================================================
# 핵심 아이디어: 미탐의 71%가 테스트 첫 3일 집중
#   → train 말미(~recent window)에서의 행동 변화를 포착
print("  📊 ★ 시간 윈도우별 행동 변화 피처 (신규)...")
win_shift_feature_cols = []

# train 마지막 7일 vs 나머지 기간 비교
last_7d_cutoff = train_cutoff - pl.duration(days=7)
df_raw_recent = df_raw_train.filter(pl.col("Timestamp") >= last_7d_cutoff)
df_raw_older  = df_raw_train.filter(pl.col("Timestamp") < last_7d_cutoff)

recent_cnt = df_raw_recent.select(
    pl.col("from_acc").cast(pl.Utf8).alias("account_id")
).group_by("account_id").len().rename({"len": "ws_recent_cnt"})
older_cnt = df_raw_older.select(
    pl.col("from_acc").cast(pl.Utf8).alias("account_id")
).group_by("account_id").len().rename({"len": "ws_older_cnt"})

df_ws = all_accounts.select("account_id").join(recent_cnt, on="account_id", how="left").join(older_cnt, on="account_id", how="left").fill_null(0.0)

ws_exprs = []
# ① ws_recent_surge: 최근 7일 / 이전 기간 (정규화) → 갑작스러운 증가
ws_exprs.append(
    (pl.col("ws_recent_cnt") / (pl.col("ws_older_cnt") / 30.0 + 1)).alias("ws_recent_surge")
)
win_shift_feature_cols.append("ws_recent_surge")

# ② ws_recent_only: 최근 7일에만 거래가 있는 계좌 (0/1)
#    → 갑자기 나타난 계좌
ws_exprs.append(
    ((pl.col("ws_recent_cnt") > 0) & (pl.col("ws_older_cnt") == 0)).cast(pl.Float64).alias("ws_recent_only")
)
win_shift_feature_cols.append("ws_recent_only")

df_ws = df_ws.with_columns(ws_exprs)

# ③ 최근 7일 고유 파트너 급증
recent_partners = df_raw_recent.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner")
]).group_by("account_id").agg(pl.col("partner").n_unique().alias("ws_recent_partners"))
older_partners = df_raw_older.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col("to_acc").cast(pl.Utf8).alias("partner")
]).group_by("account_id").agg(pl.col("partner").n_unique().alias("ws_older_partners"))

df_ws = df_ws.join(recent_partners, on="account_id", how="left").join(older_partners, on="account_id", how="left").fill_null(0.0)

# ④ ws_partner_explosion: 최근 7일 파트너가 이전 대비 급증
df_ws = df_ws.with_columns(
    (pl.col("ws_recent_partners") / (pl.col("ws_older_partners") + 1)).alias("ws_partner_explosion")
)
win_shift_feature_cols.append("ws_partner_explosion")

for cn in win_shift_feature_cols:
    df_ws = df_ws.with_columns(
        pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
        .then(0.0).otherwise(pl.col(cn)).alias(cn))

df_win_shift_features = df_ws.select(["account_id"] + win_shift_feature_cols)
del df_raw_recent, df_raw_older, df_ws, recent_cnt, older_cnt
del recent_partners, older_partners; gc.collect()
print(f"  ✅ 시간 윈도우 변화 {len(win_shift_feature_cols)}개: {win_shift_feature_cols}")

# =============================================================
# ★ NEW: 2-10. 입금/출금 독립 피처 6개
# =============================================================
# 핵심 아이디어: V2에 sum_in_1h/sum_out_1h는 있지만
#   AML 점수/조건부 피처에서 in/out 독립적 분석이 부족
print("  📊 ★ 입금/출금 독립 피처 (신규)...")
inout_feature_cols = []

# 출금(from) 집계
df_out_tx = df_raw_train.select([
    pl.col("from_acc").cast(pl.Utf8).alias("account_id"),
    pl.col(amount_col).cast(pl.Float64).alias("amount") if amount_col else pl.lit(1.0).alias("amount")
]).filter(pl.col("amount") > 0)

# 입금(to) 집계
df_in_tx = df_raw_train.select([
    pl.col("to_acc").cast(pl.Utf8).alias("account_id"),
    pl.col(amount_col).cast(pl.Float64).alias("amount") if amount_col else pl.lit(1.0).alias("amount")
]).filter(pl.col("amount") > 0)

out_stats = df_out_tx.group_by("account_id").agg([
    pl.col("amount").sum().alias("inout_total_out_amount"),
    pl.col("amount").mean().alias("inout_avg_out_amount"),
    pl.col("amount").count().alias("inout_out_cnt"),
])
in_stats = df_in_tx.group_by("account_id").agg([
    pl.col("amount").sum().alias("inout_total_in_amount"),
    pl.col("amount").mean().alias("inout_avg_in_amount"),
    pl.col("amount").count().alias("inout_in_cnt"),
])
df_inout = all_accounts.select("account_id").join(out_stats, on="account_id", how="left").join(in_stats, on="account_id", how="left").fill_null(0.0)

inout_exprs = []
# ① inout_out_amount_ratio: 출금액 / (입금액 + 출금액) → 1이면 순수 송금자
inout_exprs.append(
    (pl.col("inout_total_out_amount") /
     (pl.col("inout_total_out_amount") + pl.col("inout_total_in_amount") + 1)
    ).alias("inout_out_amount_ratio")
)
inout_feature_cols.append("inout_out_amount_ratio")

# ② inout_avg_out_vs_in: 평균 출금액 / 평균 입금액 → 1 초과면 출금이 더 큰 계좌
inout_exprs.append(
    (pl.col("inout_avg_out_amount") / (pl.col("inout_avg_in_amount") + 1)
    ).alias("inout_avg_out_vs_in")
)
inout_feature_cols.append("inout_avg_out_vs_in")

# ③ inout_cnt_imbalance: (출금 건수 - 입금 건수) / 전체 건수
#    → 양수면 보내는 쪽이 훨씬 많은 계좌 (smurf 패턴)
inout_exprs.append(
    ((pl.col("inout_out_cnt") - pl.col("inout_in_cnt")) /
     (pl.col("inout_out_cnt") + pl.col("inout_in_cnt") + 1)
    ).alias("inout_cnt_imbalance")
)
inout_feature_cols.append("inout_cnt_imbalance")

# ④ inout_fan_out_score: 출금 건수 × 고유 파트너 다양성 → 여러 곳에 분산 송금
df_inout = df_inout.join(df_graph_feats.select(["account_id", "graph_partner_diversity", "graph_unique_out_partners"]), on="account_id", how="left").fill_null(0.0)
inout_exprs.append(
    (pl.col("inout_out_cnt").log1p() * pl.col("graph_partner_diversity")
    ).alias("inout_fan_out_score")
)
inout_feature_cols.append("inout_fan_out_score")

# ⑤ inout_in_concentration: 입금이 소수에서만 들어옴 (mule 패턴)
#    = avg_in_amount × (1 - partner_diversity) → 소수 소스에서 집중 수취
inout_exprs.append(
    (pl.col("inout_avg_in_amount").log1p() * (1.0 - pl.col("graph_partner_diversity").clip(0, 1))
    ).alias("inout_in_concentration")
)
inout_feature_cols.append("inout_in_concentration")

# ⑥ inout_net_flow: log(출금합) - log(입금합) → 순 흐름 방향성
inout_exprs.append(
    (pl.col("inout_total_out_amount").log1p() - pl.col("inout_total_in_amount").log1p()
    ).alias("inout_net_flow")
)
inout_feature_cols.append("inout_net_flow")

df_inout = df_inout.with_columns(inout_exprs)
for cn in inout_feature_cols:
    df_inout = df_inout.with_columns(
        pl.when(pl.col(cn).is_infinite() | pl.col(cn).is_nan())
        .then(0.0).otherwise(pl.col(cn)).alias(cn))
df_inout_features = df_inout.select(["account_id"] + inout_feature_cols)
del df_out_tx, df_in_tx, out_stats, in_stats, df_inout; gc.collect()
print(f"  ✅ 입금/출금 독립 {len(inout_feature_cols)}개: {inout_feature_cols}")
log_memory("전체 노드 피처 완료")

# =============================================================
# 3. GNN 엣지/노드 구성 — 강화된 입력
# =============================================================
print("\n🔗 [Step 3] GNN 엣지/노드 구성 (Enriched Input)...")

df_raw_reload = pl.read_parquet(raw_path)
if df_raw_reload["Timestamp"].dtype == pl.Utf8:
    df_raw_reload = df_raw_reload.with_columns(pl.col("Timestamp").str.to_datetime(format="%Y/%m/%d %H:%M", strict=False))
df_raw_train_edges = df_raw_reload.filter(pl.col("Timestamp")<train_cutoff); del df_raw_reload; gc.collect()

edge_amount_col = None
for cand in ["Amount Received","Amount Paid","Amount"]:
    if cand in df_raw_train_edges.columns: edge_amount_col = cand; break

df_edges = df_raw_train_edges.select(
    ["from_acc","to_acc","Timestamp"] +
    ([edge_amount_col] if edge_amount_col else []) +
    (["Payment Format"] if "Payment Format" in df_raw_train_edges.columns else []) +
    (["Is Laundering"] if "Is Laundering" in df_raw_train_edges.columns else [])
).with_columns([pl.col("from_acc").cast(pl.Utf8), pl.col("to_acc").cast(pl.Utf8)])

df_edges = df_edges.join(all_accounts.rename({"account_id":"from_acc","node_id":"src_id"}),on="from_acc",how="left")
df_edges = df_edges.join(all_accounts.rename({"account_id":"to_acc","node_id":"dst_id"}),on="to_acc",how="left")

# ── 시간 피처 (5개, 기존 유지) ──────────────────────────────────────────
ts = df_edges["Timestamp"]; mn_ts=ts.min(); mx_ts=ts.max(); ts_rng=(mx_ts-mn_ts).total_seconds()
etn = ((ts-mn_ts).dt.total_microseconds().cast(pl.Float64)/1e6/max(ts_rng,1.0)).to_numpy().astype(np.float32)
hrs = df_edges["Timestamp"].dt.hour().to_numpy().astype(np.float32)
dow = df_edges["Timestamp"].dt.weekday().to_numpy().astype(np.float32)
time_feats = np.stack([etn, np.sin(2*np.pi*hrs/24), np.cos(2*np.pi*hrs/24),
                        np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7)], axis=1)

# ── 금액 피처 (4차원) ─────────────────────────────────────────────
# ★ v7c.18: relative_amt → is_large_tx (amount > 5×global_mean, 0/1)
#   근거: relative_amt ≈ log_amount 단조 변환 (중복) → 비선형 임계값 신호로 교체
if edge_amount_col:
    amounts_raw = df_edges[edge_amount_col].cast(pl.Float64).fill_null(0.0).to_numpy().astype(np.float32)
    log_amounts  = np.log1p(np.maximum(amounts_raw, 0)).astype(np.float32)
    log_amounts  = log_amounts / (np.max(log_amounts) + 1e-8)
    round_flag   = ((amounts_raw % 100 == 0) & (amounts_raw > 0)).astype(np.float32)
    global_mean_amt = np.mean(amounts_raw[amounts_raw > 0]) if np.any(amounts_raw > 0) else 1.0
    # ★ v7c.18: relative_amt 제거 → is_large_tx로 교체
    # 근거: relative_amt = log1p(amt/global_mean) ≈ log_amount - const → log_amount와 중복
    #       is_large_tx는 비선형 임계값 신호로 독립적 정보 제공
    is_large_tx  = (amounts_raw > global_mean_amt * 5).astype(np.float32)
    nonzero_amt  = amounts_raw[amounts_raw > 0]
    pct_vals     = np.percentile(nonzero_amt, np.linspace(0,100,1001)) if len(nonzero_amt)>0 else np.zeros(1001)
    amount_pct   = np.clip(np.searchsorted(pct_vals, amounts_raw, side='right').astype(np.float32)/1000.0, 0.0, 1.0)
    amount_feats = np.stack([log_amounts, round_flag, is_large_tx, amount_pct], axis=1)
else:
    amount_feats = np.zeros((len(df_edges), 4), dtype=np.float32)

# ── Payment Format 피처 (이진 정규식 2개 → 정밀 인코딩 9개) ──────────
# ★ 실제 분포 확인 결과 (HI-Medium, 7종):
#   Cheque      38.50%  세탁율 0.02%   │ ACH      12.13%  세탁율 0.79% ← 압도적 고위험
#   Credit Card 27.52%  세탁율 0.02%   │ Cash     10.09%  세탁율 0.02%
#   Reinvestment 6.10%  세탁율 0.00%   │ Wire      3.51%  세탁율 0.00%
#   Bitcoin      2.16%  세탁율 0.04%
#
# 설계 결정:
#   ① 7종 전부 원-핫 (포맷 수가 적어 "기타" 불필요)
#   ② AML 위험도 연속값 1개 (ACH=0.0079, 나머지≈0 → 강한 분리 신호)
#   ③ 기존 is_wire(정규식) 완전 제거 — Wire 세탁율 0.00%로 노이즈였음
PAYMENT_FORMATS = ["Cheque", "Credit Card", "ACH", "Cash", "Reinvestment", "Wire", "Bitcoin"]
# 포맷별 전체 데이터 기준 AML 위험도 (사전 계산값, 학습 데이터 누수 방지)
FORMAT_RISK_MAP  = {"Cheque":0.0002, "Credit Card":0.0002, "ACH":0.0079,
                    "Cash":0.0002, "Reinvestment":0.0000, "Wire":0.0000, "Bitcoin":0.0004}

if "Payment Format" in df_edges.columns:
    pf_arr = df_edges["Payment Format"].cast(pl.Utf8).fill_null("Unknown").to_numpy()

    # ① 7종 원-핫 (7차원)
    onehot_feats = np.zeros((len(pf_arr), len(PAYMENT_FORMATS)), dtype=np.float32)
    for j, fmt in enumerate(PAYMENT_FORMATS):
        onehot_feats[:, j] = (pf_arr == fmt).astype(np.float32)

    # ② AML 위험도 연속값 (1차원) — ACH가 다른 포맷 대비 ~40배 높아 강한 신호
    fmt_risk = np.array([FORMAT_RISK_MAP.get(f, 0.0002) for f in pf_arr], dtype=np.float32)

    type_feats   = np.concatenate([onehot_feats, fmt_risk.reshape(-1,1)], axis=1)  # 8차원
    N_TYPE_FEATS = type_feats.shape[1]
else:
    N_TYPE_FEATS = len(PAYMENT_FORMATS) + 1
    type_feats   = np.zeros((len(df_edges), N_TYPE_FEATS), dtype=np.float32)

edge_features_all = np.concatenate([time_feats, amount_feats, type_feats], axis=1)
EDGE_RAW_DIM = edge_features_all.shape[1]
print(f"  📊 Edge 피처: {EDGE_RAW_DIM}개")
print(f"     시간(5) + 금액(4: log/round/is_large_tx/pct) + 포맷({N_TYPE_FEATS}: 원-핫7+위험도1)")
print(f"     ★ relative_amt → is_large_tx (5×global_mean 임계값)")

edge_index_train = torch.tensor(df_edges.select(["src_id","dst_id"]).to_numpy().T, dtype=torch.long)
edge_attr_train = torch.tensor(edge_features_all, dtype=torch.float32)
print(f"📊 Train 엣지: {edge_index_train.shape[1]:,}")

del df_raw_train, df_raw_train_edges, df_edges, ts, hrs, dow, etn
del time_feats, amount_feats, type_feats, edge_features_all; gc.collect()

# ─── 노드 피처: V2(38) + 그래프(13) + 조건부(10) + AML(5) + 규칙성(7) + 금액(8) = 81개
# ★ v7c.5: GNN 입력은 v7c.2와 동일하게 유지 (신규 피처는 XGBoost에만 투입)
print("  📊 노드 피처 구성 (GNN: v7c.2 동일 81차원)...")
df_v2_node = df_v2_train.group_by("account_id").agg([pl.col(c).mean().alias(c) for c in gnn_feature_cols_v2])
df_node_all = (all_accounts.join(df_v2_node, on="account_id", how="left")
    .join(df_graph_feats, on="account_id", how="left")
    .join(df_cond_features, on="account_id", how="left")
    .join(df_cb_features, on="account_id", how="left")           # ★ v7c.15: GNN 이동
    .join(df_vol_norm_features, on="account_id", how="left")     # ★ v7c.15: GNN 이동
    .join(df_win_shift_features, on="account_id", how="left")    # ★ v7c.15: GNN 이동
    .join(df_inout_features, on="account_id", how="left")        # ★ v7c.15: GNN 이동
    .join(df_regularity, on="account_id", how="left")            # ★ v7c.16: reg 4개 이동
    .join(df_amt_features, on="account_id", how="left")          # ★ v7c.16: amt 2개 이동
    .fill_null(0.0))

# GNN 입력 피처 결정 (역할 기반 설계)
# ─────────────────────────────────────────────────────────────
# GNN 유지 기준: 이웃 전파로 의미가 강화되는 피처
#   = 계좌 간 관계 패턴, 네트워크 구조 결합값, 방향성 흐름
#
# ★ v7c.15: XGBoost 전용 피처 중 GNN 적합 14개 이동
#
# cb_*(4개) 전부: cross-border × graph 교차항
#   → 이웃도 cb_burst_cross 높은지 전파하면 클러스터 송금 패턴 포착
#
# vol_*(4개): graph_degree/diversity 결합값
#   vol_amount_per_peer, vol_concentrated_amount,
#   vol_low_freq_high_amt, vol_degree_normalized_amount
#   → vol_outlier_score만 제외 (전체 평균 편차, 이웃 전파 의미 없음)
#
# ws_*(2개): 시간 윈도우 행동 변화
#   ws_recent_surge, ws_partner_explosion
#   → ws_recent_only 제외 (0/1 플래그, 계좌 자체 속성)
#
# inout_*(4개): 입출금 방향성 + graph 결합값
#   inout_cnt_imbalance, inout_out_amount_ratio,
#   inout_fan_out_score, inout_in_concentration
#   → inout_avg_out_vs_in, inout_net_flow 제외 (단순 비율/차이값)

# GNN 이동 대상 피처 목록 (명시적 선언)
gnn_cb_cols   = [c for c in cb_feature_cols]                          # 4개 전부
gnn_vol_cols  = [c for c in vol_norm_feature_cols
                 if c != "vol_outlier_score"]                          # 4개 (outlier 제외)
gnn_ws_cols   = [c for c in win_shift_feature_cols
                 if c in ("ws_recent_surge","ws_partner_explosion")]   # 2개
gnn_inout_cols= [c for c in inout_feature_cols
                 if c in ("inout_cnt_imbalance","inout_out_amount_ratio",
                          "inout_fan_out_score","inout_in_concentration")]  # 4개

# ★ v7c.16: 규칙성/금액 중 이웃 전파 적합한 피처 추가 이동
# reg_hour_entropy, reg_dow_entropy: 거래 시간 분포 무질서도 → 이웃 클러스터 동시 활성 시간대 포착
# reg_consistent_partner_ratio: 반복 거래 상대 비율 → ring/loop 구조 탐지
# reg_top_partner_concentration: 상위 파트너 집중도 → 허브 구조 강화
# amt_round_number_ratio: round number 동시 사용 (smurfing) → 이웃 전파로 포착
# amt_entropy: 금액 분포 다양성 → 이웃이 단조로운 금액 쓰는지 (integration 패턴)
gnn_reg_cols  = [c for c in regularity_cols
                 if c in ("reg_hour_entropy","reg_dow_entropy",
                          "reg_consistent_partner_ratio","reg_top_partner_concentration")]  # 4개
gnn_amt_cols  = [c for c in amt_feature_cols
                 if c in ("amt_round_number_ratio","amt_entropy")]     # 2개

gnn_input_cols = (gnn_feature_cols_v2 + graph_feature_cols + cond_feature_cols +
                  gnn_cb_cols + gnn_vol_cols + gnn_ws_cols + gnn_inout_cols +
                  gnn_reg_cols + gnn_amt_cols)

for col in gnn_input_cols:
    if col in df_node_all.columns:
        df_node_all = df_node_all.with_columns(
            pl.when(pl.col(col).is_infinite()|pl.col(col).is_nan()).then(0.0).otherwise(pl.col(col)).alias(col))

X_node = torch.tensor(df_node_all.select(gnn_input_cols).to_numpy(), dtype=torch.float32)
IN_DIM = X_node.shape[1]
print(f"  ✅ GNN 노드 입력: {IN_DIM}차원")
print(f"     = V2({len(gnn_feature_cols_v2)}) + 그래프({len(graph_feature_cols)}) + 조건부({len(cond_feature_cols)})")
print(f"       + CB({len(gnn_cb_cols)}) + Vol({len(gnn_vol_cols)}) + WS({len(gnn_ws_cols)}) + InOut({len(gnn_inout_cols)})")
print(f"       + Reg({len(gnn_reg_cols)}) + Amt({len(gnn_amt_cols)})  ★ v7c.16 추가")

target_labels = df_v2_train.filter(pl.col("is_laundering")==1).select("account_id").unique().with_columns(pl.lit(1).alias("label"))
df_labels = all_accounts.join(target_labels,on="account_id",how="left").fill_null(0)
Y_node = torch.tensor(df_labels["label"].to_numpy(), dtype=torch.long)
pos_node_ids = set(df_labels.filter(pl.col("label")==1)["node_id"].to_list())
active_df = pl.DataFrame({"account_id":df_v2_train["account_id"].unique(),"is_active":True})
mask_df = all_accounts.join(active_df,on="account_id",how="left").fill_null(False)
active_node_ids = mask_df.filter(pl.col("is_active"))["node_id"].to_list()
active_pos = [n for n in active_node_ids if n in pos_node_ids]
active_neg = [n for n in active_node_ids if n not in pos_node_ids]
n_neg_gnn = min(len(active_pos)*NEG_NODE_RATIO_GNN, len(active_neg))
np.random.seed(RANDOM_SEED)
sampled_neg = np.random.choice(active_neg,size=n_neg_gnn,replace=False).tolist()
sampled_nodes = set(active_pos+sampled_neg)
train_mask_np = np.zeros(num_nodes, dtype=bool)
for nid in sampled_nodes: train_mask_np[nid]=True
train_mask = torch.tensor(train_mask_np, dtype=torch.bool)
print(f"📊 GNN 노드: 양성 {len(active_pos):,} + 음성 {n_neg_gnn:,}")
del df_v2_node, df_node_all, mask_df, active_df; gc.collect()

graph_data = Data(x=X_node, edge_index=edge_index_train, edge_attr=edge_attr_train, y=Y_node)
graph_data.train_mask = train_mask; graph_data.num_nodes = num_nodes
del X_node, edge_index_train, edge_attr_train, Y_node, train_mask; gc.collect()
log_memory("GNN 데이터 준비 완료")

# =============================================================
# 4-6. Multi-Config GNN 탐색 → 최적 임베딩 선택
# =============================================================
print("\n🧠 [Step 4-6] Multi-Config GNN 탐색...")

class EdgeProjection(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, out_dim), nn.GELU(), nn.Linear(out_dim, out_dim))
    def forward(self, ea): return self.proj(ea)

class TGAT_Flex(nn.Module):
    def __init__(self, in_dim, hidden_dim, edge_raw_dim, edge_dim, n_heads, n_layers=2, dropout=0.2):
        super().__init__()
        self.edge_proj = EdgeProjection(edge_raw_dim, edge_dim)
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.n_layers = n_layers
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_layers)])
        self.convs = nn.ModuleList([
            TransformerConv(hidden_dim, hidden_dim//n_heads, heads=n_heads,
                            edge_dim=edge_dim, dropout=dropout, concat=True)
            for _ in range(n_layers)])
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim//2, 1))
        self.dropout = dropout; self.hidden_dim = hidden_dim

    def forward(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        for i in range(self.n_layers):
            h = h + F.dropout(self.convs[i](self.norms[i](h), edge_index, ee), p=self.dropout, training=self.training)
        return self.classifier(self.norm_out(h)).squeeze(-1)

    @torch.no_grad()
    def get_embedding(self, x, edge_index, edge_attr):
        ee=self.edge_proj(edge_attr); h=self.input_proj(x)
        for i in range(self.n_layers):
            h = h + self.convs[i](self.norms[i](h), edge_index, ee)
        return self.norm_out(h)

# ─────────────────────────────────────────────────────────────
# ★ v7c.19: 레이어 비교 실험 — edge_attr 지원 3종 추가
#
# [집계 방식 비교]
#  TransformerConv (현재): attention-weighted SUM
#    → 모든 이웃을 소프트하게 가중합
#    → 이웃이 많은 허브 노드에서 신호 희석 가능
#
#  GATv2Conv: attention-weighted SUM (GATv1 개선)
#    → 동적 어텐션 (쿼리·키를 concat 후 공유 MLP)
#    → TransformerConv보다 표현력 강하지만 집계 방식은 동일
#
#  NNConv + MAX: edge-conditioned MAX pooling
#    → 엣지 피처로 메시지 변환 행렬 생성 → 이웃별 변환 후 MAX 집계
#    → "가장 강한 신호 하나"만 통과 → 희소 세탁 엣지 포착에 유리
#    → AML처럼 극소수 악성 이웃이 존재할 때 max가 mean/sum보다 강인
#
#  GINEConv + SUM: sum-aggregation + MLP (WL test 표현력 최대)
#    → 노드+엣지 합산 후 MLP 적용 → 구조 구별 능력 이론적 최강
#    → 단, sum은 노드 수에 따라 스케일이 달라져 정규화 필요
# ─────────────────────────────────────────────────────────────

class GATv2_Model(nn.Module):
    """GATv2Conv: 동적 어텐션 + edge_attr 지원, SUM 집계"""
    def __init__(self, in_dim, hidden_dim, edge_raw_dim, edge_dim, n_heads, n_layers=2, dropout=0.2):
        super().__init__()
        self.edge_proj = EdgeProjection(edge_raw_dim, edge_dim)
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.n_layers = n_layers
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_layers)])
        self.convs = nn.ModuleList([
            GATv2Conv(hidden_dim, hidden_dim // n_heads, heads=n_heads,
                      edge_dim=edge_dim, dropout=dropout, concat=True)
            for _ in range(n_layers)])
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim // 2, 1))
        self.dropout = dropout; self.hidden_dim = hidden_dim

    def forward(self, x, edge_index, edge_attr):
        ee = self.edge_proj(edge_attr); h = self.input_proj(x)
        for i in range(self.n_layers):
            h = h + F.dropout(self.convs[i](self.norms[i](h), edge_index, ee), p=self.dropout, training=self.training)
        return self.classifier(self.norm_out(h)).squeeze(-1)

    @torch.no_grad()
    def get_embedding(self, x, edge_index, edge_attr):
        ee = self.edge_proj(edge_attr); h = self.input_proj(x)
        for i in range(self.n_layers):
            h = h + self.convs[i](self.norms[i](h), edge_index, ee)
        return self.norm_out(h)


class EdgeCondMaxConv(nn.Module):
    """수동 구현: edge-conditioned MAX aggregation
    NNConv의 NeighborLoader 배치 차원 불일치 문제 우회
    동작: edge_attr → gate 벡터 생성 → 노드 메시지에 곱한 뒤 MAX 집계"""
    def __init__(self, hidden_dim, edge_raw_dim, edge_dim):
        super().__init__()
        # edge_attr → hidden_dim 크기 gate 생성 (sigmoid: 0~1 게이팅)
        self.edge_gate = nn.Sequential(
            nn.Linear(edge_raw_dim, edge_dim), nn.GELU(),
            nn.Linear(edge_dim, hidden_dim), nn.Sigmoid()
        )
        self.msg_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, h, edge_index, edge_attr):
        src, dst = edge_index  # src→dst 방향
        # 이웃(src) 노드 메시지 생성
        msg = self.msg_proj(h[src])                     # [E, hidden]
        gate = self.edge_gate(edge_attr)                # [E, hidden]
        msg = msg * gate                                # edge-conditioned 메시지
        # MAX 집계: 각 dst 노드로 들어오는 메시지 중 최대값
        out = torch.full((h.size(0), h.size(1)), -1e9, device=h.device, dtype=h.dtype)
        out = out.scatter_reduce(0, dst.unsqueeze(1).expand_as(msg), msg, reduce="amax", include_self=True)
        out = torch.where(out == -1e9, torch.zeros_like(out), out)  # 이웃 없는 노드 0처리
        return out


class NNConv_Max_Model(nn.Module):
    """EdgeCondMaxConv: edge-conditioned MAX pooling (수동 구현)
    핵심: 소수의 강한 악성 이웃 신호를 희석 없이 통과 → AML에 이론적으로 유리
    NNConv 대비: NeighborLoader 배치 차원 불일치 문제 없음"""
    def __init__(self, in_dim, hidden_dim, edge_raw_dim, edge_dim, n_heads, n_layers=2, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.n_layers = n_layers
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_layers)])
        self.convs = nn.ModuleList([
            EdgeCondMaxConv(hidden_dim, edge_raw_dim, edge_dim)
            for _ in range(n_layers)])
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim // 2, 1))
        self.dropout = dropout; self.hidden_dim = hidden_dim

    def forward(self, x, edge_index, edge_attr):
        h = self.input_proj(x)
        for i in range(self.n_layers):
            h = h + F.dropout(
                F.gelu(self.norms[i](self.convs[i](h, edge_index, edge_attr))),
                p=self.dropout, training=self.training)
        return self.classifier(self.norm_out(h)).squeeze(-1)

    @torch.no_grad()
    def get_embedding(self, x, edge_index, edge_attr):
        h = self.input_proj(x)
        for i in range(self.n_layers):
            h = h + F.gelu(self.norms[i](self.convs[i](h, edge_index, edge_attr)))
        return self.norm_out(h)


class GINEConv_Model(nn.Module):
    """GINEConv + SUM: WL test 수준 표현력, 노드+엣지 합산 후 MLP
    핵심: 그래프 구조 구별 능력 이론적 최강 — 동형 그래프도 구분 가능"""
    def __init__(self, in_dim, hidden_dim, edge_raw_dim, edge_dim, n_heads, n_layers=2, dropout=0.2):
        super().__init__()
        self.edge_proj = EdgeProjection(edge_raw_dim, edge_dim)
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.edge_align = nn.Linear(edge_dim, hidden_dim)  # GINEConv: edge_dim == hidden_dim 필요
        self.n_layers = n_layers
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(n_layers)])
        self.convs = nn.ModuleList([
            GINEConv(
                nn=nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim * 2), nn.GELU(),
                    nn.Linear(hidden_dim * 2, hidden_dim)
                ),
                train_eps=True   # ε 학습 가능 (자기 자신 기여도 적응)
            )
            for _ in range(n_layers)])
        self.norm_out = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim // 2, 1))
        self.dropout = dropout; self.hidden_dim = hidden_dim

    def forward(self, x, edge_index, edge_attr):
        ee = self.edge_align(self.edge_proj(edge_attr))
        h = self.input_proj(x)
        for i in range(self.n_layers):
            h = h + F.dropout(self.convs[i](self.norms[i](h), edge_index, ee), p=self.dropout, training=self.training)
        return self.classifier(self.norm_out(h)).squeeze(-1)

    @torch.no_grad()
    def get_embedding(self, x, edge_index, edge_attr):
        ee = self.edge_align(self.edge_proj(edge_attr))
        h = self.input_proj(x)
        for i in range(self.n_layers):
            h = h + self.convs[i](self.norms[i](h), edge_index, ee)
        return self.norm_out(h)


def build_model(cfg, in_dim, edge_raw_dim, dropout):
    """cfg의 conv_type에 따라 올바른 모델 클래스 반환"""
    ct = cfg.get("conv_type", "transformer")
    hd = cfg["hidden_dim"]; nl = cfg["n_layers"]
    nh = cfg.get("n_heads", 4); ed = cfg.get("edge_dim", 32)
    if ct == "transformer": return TGAT_Flex(in_dim, hd, edge_raw_dim, ed, nh, nl, dropout)
    elif ct == "gatv2":     return GATv2_Model(in_dim, hd, edge_raw_dim, ed, nh, nl, dropout)
    elif ct == "nnconv_max":return NNConv_Max_Model(in_dim, hd, edge_raw_dim, ed, nh, nl, dropout)
    elif ct == "gine":      return GINEConv_Model(in_dim, hd, edge_raw_dim, ed, nh, nl, dropout)
    else: raise ValueError(f"Unknown conv_type: {ct}")


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"📌 디바이스: {device}")
# ★ v7c.14: pos_weight를 NEG_NODE_RATIO_GNN과 일치 (이중 보정 해소)
# 기존: 실제 비율(1:50) 기준 pw 계산 → 샘플링(1:25) 후에도 pw=50 유지 → 이중 보정
# 수정: 샘플링 비율과 동일하게 pw=NEG_NODE_RATIO_GNN=25
pw = float(NEG_NODE_RATIO_GNN)
print(f"📊 pos_weight: {pw:.1f}  (NEG_NODE_RATIO {NEG_NODE_RATIO_GNN}과 일치)")

def make_lr(opt, ep, n_ep):
    if ep < WARMUP_EPOCHS: lr = 1e-6+(BASE_LR-1e-6)*(ep/WARMUP_EPOCHS)
    else: p=(ep-WARMUP_EPOCHS)/max(n_ep-WARMUP_EPOCHS,1); lr=1e-5+(BASE_LR-1e-5)*0.5*(1+np.cos(np.pi*p))
    for pg in opt.param_groups: pg['lr']=lr
    return lr

GNN_CONFIGS = [
    # ── TransformerConv (기존, attention-weighted SUM) ──────────────────
    {"label":"tc_h96_L2",   "conv_type":"transformer", "hidden_dim":96,  "n_layers":2, "n_heads":4, "edge_dim":32},
    {"label":"tc_h128_L2",  "conv_type":"transformer", "hidden_dim":128, "n_layers":2, "n_heads":4, "edge_dim":32},
    {"label":"tc_h96_L3",   "conv_type":"transformer", "hidden_dim":96,  "n_layers":3, "n_heads":4, "edge_dim":32},
    {"label":"tc_h128_L3",  "conv_type":"transformer", "hidden_dim":128, "n_layers":3, "n_heads":4, "edge_dim":32},
    # ── GATv2Conv (동적 어텐션 SUM) ─────────────────────────────────────
    {"label":"gat2_h96_L2", "conv_type":"gatv2",       "hidden_dim":96,  "n_layers":2, "n_heads":4, "edge_dim":32},
    {"label":"gat2_h128_L2","conv_type":"gatv2",       "hidden_dim":128, "n_layers":2, "n_heads":4, "edge_dim":32},
    # ── NNConv + MAX (엣지 변환행렬 × MAX 집계) ─────────────────────────
    {"label":"nn_max_h96_L2", "conv_type":"nnconv_max","hidden_dim":96,  "n_layers":2, "n_heads":4, "edge_dim":32},
    {"label":"nn_max_h96_L3", "conv_type":"nnconv_max","hidden_dim":96,  "n_layers":3, "n_heads":4, "edge_dim":32},
    # ── GINEConv + SUM (WL 이론적 최강 표현력) ──────────────────────────
    {"label":"gine_h96_L2", "conv_type":"gine",        "hidden_dim":96,  "n_layers":2, "n_heads":4, "edge_dim":32},
    {"label":"gine_h128_L2","conv_type":"gine",        "hidden_dim":128, "n_layers":2, "n_heads":4, "edge_dim":32},
]

xgb_exclude_cols=["account_id","time_group","is_laundering","mode_format","currency_mode","date"]
xgb_v2_cols=[c for c in df_v2.columns if c not in xgb_exclude_cols]
apr_train = df_v2_train.filter(pl.col("is_laundering")==1).height / max(df_v2_train.height,1)
spw = max((1-apr_train)/apr_train, 1.0) if apr_train > 0 else 20.0

# ★ 신규 피처들을 base_feature_cols에 포함
base_feature_cols = (xgb_v2_cols + graph_feature_cols + cond_feature_cols + aml_score_cols +
                     cb_feature_cols + vol_norm_feature_cols +
                     win_shift_feature_cols + inout_feature_cols)

def build_eval_df(db, df_emb_local, emb_col_names):
    feature_cols = base_feature_cols + emb_col_names
    df = (db.join(df_graph_feats,on="account_id",how="left")
          .join(df_cond_features,on="account_id",how="left")
          .join(df_aml_scores,on="account_id",how="left")
          .join(df_cb_features,on="account_id",how="left")            # ★
          .join(df_vol_norm_features,on="account_id",how="left")      # ★
          .join(df_win_shift_features,on="account_id",how="left")     # ★
          .join(df_inout_features,on="account_id",how="left")         # ★
          .join(df_emb_local,on="account_id",how="left")
          .fill_null(0.0).fill_nan(0.0))
    for cn in feature_cols:
        if cn in df.columns:
            df = df.with_columns(pl.when(pl.col(cn).is_infinite()).then(0.0).otherwise(pl.col(cn)).alias(cn))
    return df, feature_cols

def quick_xgb_eval(df_emb_local, emb_col_names, label):
    df_tr, fcols = build_eval_df(df_v2.filter(pl.col("time_group")<train_cutoff), df_emb_local, emb_col_names)
    df_tr_pos=df_tr.filter(pl.col("is_laundering")==1); df_tr_neg=df_tr.filter(pl.col("is_laundering")==0)
    n_neg=min(len(df_tr_pos)*NEG_ROW_RATIO_XGB,len(df_tr_neg))
    df_tr_s=pl.concat([df_tr_pos,df_tr_neg.sample(n=n_neg,seed=RANDOM_SEED)]).sample(fraction=1.0,seed=RANDOM_SEED)
    X_tr=df_tr_s.select(fcols).to_pandas(); y_tr=df_tr_s["is_laundering"].to_numpy()
    del df_tr,df_tr_pos,df_tr_neg,df_tr_s; gc.collect()
    df_vl, _ = build_eval_df(df_v2.filter((pl.col("time_group")>=train_cutoff)&(pl.col("time_group")<val_cutoff)), df_emb_local, emb_col_names)
    X_vl=df_vl.select(fcols).to_pandas(); y_vl=df_vl["is_laundering"].to_numpy(); del df_vl; gc.collect()
    df_te, _ = build_eval_df(df_v2.filter(pl.col("time_group")>=val_cutoff), df_emb_local, emb_col_names)
    X_te=df_te.select(fcols).to_pandas(); y_te=df_te["is_laundering"].to_numpy(); del df_te; gc.collect()
    # ★ v7c.14: Stage 3 best(d11_lr01) 근방 HP로 통일 → GNN 선택 기준 정합성 확보
    # 기존: max_depth=8, lr=0.03 → Stage 3 best(d11_lr01)와 불일치, 잘못된 GNN 선택 가능
    qp={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
        "random_state":42,"max_depth":10,"learning_rate":0.01,"scale_pos_weight":spw,
        "subsample":0.8,"colsample_bytree":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
    m=xgb.XGBClassifier(**qp,n_estimators=4000,early_stopping_rounds=100)
    m.fit(X_tr,y_tr,eval_set=[(X_vl,y_vl)],verbose=0)
    q_val=m.best_score; q_test=average_precision_score(y_te,m.predict_proba(X_te)[:,1])
    del m,X_tr,X_vl,X_te,y_tr,y_vl,y_te; gc.collect()
    return q_val, q_test

print(f"\n📊 {len(GNN_CONFIGS)}개 GNN 설정 탐색 (4종 레이어 비교)...")
print(f"  {'#':>2s} {'Label':<18s} {'Conv':>10s} | {'hidden':>6s} {'layers':>6s} | {'Loss':>8s} {'Params':>8s} | {'QVal':>8s} {'QTest':>8s}")
print("  "+"-"*90)

gnn_results = []
for gi, gcfg in enumerate(GNN_CONFIGS):
    gl = gcfg["label"]; hd=gcfg["hidden_dim"]; nl=gcfg["n_layers"]
    nh=gcfg["n_heads"]; ed=gcfg["edge_dim"]
    train_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                                   input_nodes=graph_data.train_mask, shuffle=True, num_workers=4)
    full_loader = NeighborLoader(graph_data, num_neighbors=NUM_NEIGHBORS, batch_size=BATCH_SIZE,
                                  input_nodes=None, shuffle=False, num_workers=4)
    model = build_model(gcfg, IN_DIM, EDGE_RAW_DIM, GNN_DROPOUT).to(device)
    np_count = sum(p.numel() for p in model.parameters())
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw],device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-6, weight_decay=1e-4)
    model.train(); bl=float('inf'); pc2=0; bs=None
    for epoch in range(N_EPOCHS_GNN):
        clr=make_lr(optimizer,epoch,N_EPOCHS_GNN); tl=0.0; nb2=0
        for batch in tqdm(train_loader,desc=f"[{gl}] Ep{epoch+1}",leave=False):
            batch=batch.to(device); optimizer.zero_grad()
            bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
            out=model(batch.x,batch.edge_index,bea)
            loss=criterion(out[:batch.batch_size],batch.y[:batch.batch_size].float())
            if torch.isnan(loss) or torch.isinf(loss): optimizer.zero_grad(); continue
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=0.5)
            optimizer.step(); tl+=loss.item(); nb2+=1
        al=tl/max(nb2,1)
        if (epoch+1)%10==0 or epoch==0: print(f"    [{gl}] Ep{epoch+1:2d} Loss={al:.4f}")
        if al<bl-EARLY_STOP_DELTA: bl=al; pc2=0; bs={k:v.cpu().clone() for k,v in model.state_dict().items()}
        else: pc2+=1
        if pc2>=PATIENCE: break
    if bs: model.load_state_dict({k:v.to(device) for k,v in bs.items()}); del bs
    model.eval(); ae=[]
    with torch.no_grad():
        for batch in tqdm(full_loader,desc=f"[{gl}] Embed",leave=False):
            batch=batch.to(device)
            bea=batch.edge_attr if (batch.edge_attr is not None and batch.edge_attr.numel()>0) else torch.zeros(batch.edge_index.size(1),EDGE_RAW_DIM,device=device)
            emb=model.get_embedding(batch.x,batch.edge_index,bea)
            ae.append(emb[:batch.batch_size].cpu())
    femb=torch.cat(ae,dim=0).numpy()
    del model,ae,train_loader,full_loader; torch.cuda.empty_cache(); gc.collect()
    ecn=[f"tgat_emb_{i}" for i in range(hd)]
    df_emb=pl.concat([all_accounts.select("account_id"),pl.DataFrame(femb,schema=ecn)],how="horizontal")
    qv,qt = quick_xgb_eval(df_emb, ecn, gl)
    print(f"  {gi+1:2d} {gl:<18s} {gcfg.get('conv_type','transformer'):>10s} | {hd:6d} {nl:6d} | {bl:8.4f} {np_count:8,} | {qv:8.5f} {qt:8.4f}")
    gnn_results.append({"label":gl,"hidden_dim":hd,"n_layers":nl,"loss":bl,"params":np_count,
                         "q_val":qv,"q_test":qt,"embedding":femb,"ecn":ecn,"df_emb":df_emb})
    log_memory(f"{gl}")

best_gnn = max(gnn_results, key=lambda x: x["q_val"])
print(f"\n  🏆 Best GNN: {best_gnn['label']} (QVal={best_gnn['q_val']:.5f} QTest={best_gnn['q_test']:.4f})")
for r in gnn_results:
    if r is not best_gnn: del r["embedding"], r["df_emb"]
gc.collect()

emb_cols = best_gnn["ecn"]
df_tgat = best_gnn["df_emb"]
FINAL_HIDDEN_DIM = best_gnn["hidden_dim"]

print(f"\n🔗 Interaction 피처 ({best_gnn['label']})...")
df_tgat_with_pr=df_tgat.join(df_graph_feats.select(["account_id","graph_pagerank"]),on="account_id",how="left").fill_null(0.0)
emb_np=df_tgat_with_pr.select(emb_cols).to_numpy(); emb_var=np.var(emb_np,axis=0)
top_idx=np.argsort(emb_var)[::-1][:N_INTERACTION_DIMS]; top_ec=[emb_cols[i] for i in top_idx]
ie_l=[]; ic_l=[]
for cn in top_ec: nn2=f"interact_pr_x_{cn}"; ie_l.append((pl.col(cn)*pl.col("graph_pagerank")*1e6).alias(nn2)); ic_l.append(nn2)
ie_l.append((sum(pl.col(c)**2 for c in emb_cols).sqrt()*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_norm")); ic_l.append("interact_pr_x_emb_norm")
ie_l.append((sum(pl.col(c) for c in top_ec)*pl.col("graph_pagerank")*1e6).alias("interact_pr_x_emb_topsum")); ic_l.append("interact_pr_x_emb_topsum")
interaction_cols=ic_l
df_interactions=df_tgat_with_pr.with_columns(ie_l).select(["account_id"]+ic_l)
for c in ic_l: df_interactions=df_interactions.with_columns(pl.when(pl.col(c).is_infinite()|pl.col(c).is_nan()).then(0.0).otherwise(pl.col(c)).alias(c))
print(f"  ✅ Interaction {len(ic_l)}개")
del df_tgat_with_pr,emb_np,best_gnn["embedding"]; gc.collect()
del graph_data; torch.cuda.empty_cache(); gc.collect()
log_memory("GNN 탐색 완료")

# =============================================================
# 7. XGBoost 데이터 구성
# =============================================================
print("\n🚀 [Step 7] XGBoost 데이터 구성...")

# ★ v7c.7: regularity(7) + amt(8) = 15개를 XGBoost 후보에 추가 (v7c.6 누락 수정)
# GNN 입력에는 이미 포함돼 있었지만 xgb_feature_cols에서 빠져 있었음
xgb_feature_cols = (xgb_v2_cols + graph_feature_cols + cond_feature_cols + aml_score_cols +
                    regularity_cols + amt_feature_cols +          # ★ 신규 추가
                    cb_feature_cols + vol_norm_feature_cols +
                    win_shift_feature_cols + inout_feature_cols +
                    emb_cols + interaction_cols)
selected_new_cols = cb_feature_cols + vol_norm_feature_cols + win_shift_feature_cols + inout_feature_cols
print(f"📊 XGBoost 총 후보 피처: {len(xgb_feature_cols)}")
print(f"   V2:{len(xgb_v2_cols)} G:{len(graph_feature_cols)} Cond:{len(cond_feature_cols)} AML:{len(aml_score_cols)}")
print(f"   ★추가 Reg:{len(regularity_cols)} Amt:{len(amt_feature_cols)}  (v7c.6 누락 수정)")
print(f"   CB:{len(cb_feature_cols)} VolNorm:{len(vol_norm_feature_cols)}")
print(f"   WinShift:{len(win_shift_feature_cols)} InOut:{len(inout_feature_cols)}")
print(f"   Emb:{len(emb_cols)} Inter:{len(interaction_cols)}")
print(f"   ★ Forward Selection이 전체 후보에서 유효한 것 선별")

def build_xgb_df(db):
    df=(db.join(df_graph_feats,on="account_id",how="left")
          .join(df_cond_features,on="account_id",how="left")
          .join(df_aml_scores,on="account_id",how="left")
          .join(df_regularity,on="account_id",how="left")        # ★ 신규 추가
          .join(df_amt_features,on="account_id",how="left")      # ★ 신규 추가
          .join(df_cb_features,on="account_id",how="left")
          .join(df_vol_norm_features,on="account_id",how="left")
          .join(df_win_shift_features,on="account_id",how="left")
          .join(df_inout_features,on="account_id",how="left")
          .join(df_tgat,on="account_id",how="left")
          .join(df_interactions,on="account_id",how="left")
          .fill_null(0.0).fill_nan(0.0))
    for cn in xgb_feature_cols:
        if cn in df.columns: df=df.with_columns(pl.when(pl.col(cn).is_infinite()).then(0.0).otherwise(pl.col(cn)).alias(cn))
    return df

df_tr=build_xgb_df(df_v2.filter(pl.col("time_group")<train_cutoff))
df_tr_pos=df_tr.filter(pl.col("is_laundering")==1); df_tr_neg=df_tr.filter(pl.col("is_laundering")==0)
n_neg=min(len(df_tr_pos)*NEG_ROW_RATIO_XGB,len(df_tr_neg))
df_tr_s=pl.concat([df_tr_pos,df_tr_neg.sample(n=n_neg,seed=RANDOM_SEED)]).sample(fraction=1.0,seed=RANDOM_SEED)
X_train=df_tr_s.select(xgb_feature_cols).to_pandas(); y_train=df_tr_s["is_laundering"].to_numpy()
print(f"📊 Train: {len(df_tr):,} → {len(df_tr_s):,} ({y_train.mean()*100:.2f}%)")
del df_tr,df_tr_pos,df_tr_neg,df_tr_s; gc.collect()
df_vl=build_xgb_df(df_v2.filter((pl.col("time_group")>=train_cutoff)&(pl.col("time_group")<val_cutoff)))
X_val=df_vl.select(xgb_feature_cols).to_pandas(); y_val=df_vl["is_laundering"].to_numpy(); print(f"📊 Val: {len(X_val):,}"); del df_vl; gc.collect()
df_te=build_xgb_df(df_v2.filter(pl.col("time_group")>=val_cutoff))
df_test_meta=df_te.select(["account_id","time_group","is_laundering"])
X_test=df_te.select(xgb_feature_cols).to_pandas(); y_test=df_te["is_laundering"].to_numpy(); print(f"📊 Test: {len(X_test):,}")
del df_te,df_v2,df_tgat,df_graph_feats,df_cond_features,df_aml_scores
del df_cb_features,df_vol_norm_features,df_win_shift_features,df_inout_features
del df_interactions,df_regularity,df_amt_features; gc.collect()
log_memory("XGBoost 준비")

# =============================================================
# 8. XGBoost 3-Stage: Full → Forward Selection → HP Grid
# =============================================================
# ★ v7c.6 핵심 변경: Stage 1 importance 기반 top-K 선택의 한계 해결
#   문제: 상관된 피처끼리 importance를 나눠 가져서 유용한 피처가 순위 밖으로 밀림
#   해결: Stage 2를 Forward Selection으로 교체
#     → 피처를 하나씩 추가하며 Val AUCPR이 실제로 오를 때만 채택
#     → 중복/노이즈 피처는 자동으로 탈락
print("\n🔥 [Step 8] 3-Stage XGBoost (Full → Forward Selection → HP Grid)...")
apr=y_train.sum()/len(y_train); spw=max((1-apr)/apr,1.0)

# ── Stage 1: 전체 피처로 학습 → importance 순위 산출 ──────────────────
print("\n── Stage 1: 전체 피처 학습 (importance 순위 산출) ──")
s1p={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
     "random_state":42,"max_depth":8,"learning_rate":0.03,"scale_pos_weight":spw,
     "subsample":0.8,"colsample_bytree":0.7,"min_child_weight":5,
     "gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}
m_s1=xgb.XGBClassifier(**s1p,n_estimators=1500,early_stopping_rounds=100)
m_s1.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=100)
s1_val=m_s1.best_score
s1_test=average_precision_score(y_test,m_s1.predict_proba(X_test)[:,1])
print(f"\n  Stage1: Val={s1_val:.5f} Test={s1_test:.4f}")
imp_s1=m_s1.feature_importances_
# importance 순위 정렬 (Stage 2 Forward Selection의 탐색 순서로 사용)
feat_imp=sorted(zip(xgb_feature_cols,imp_s1),key=lambda x:x[1],reverse=True)
del m_s1; gc.collect()

# ── Stage 2: Forward Selection ─────────────────────────────────────────
# importance 상위 순서대로 피처를 하나씩 추가하며
# Val AUCPR이 오를 때만 채택, 연속 N_PATIENCE_FS번 개선 없으면 조기 종료
print("\n── Stage 2: Forward Selection ──")
print("  피처를 importance 순서대로 하나씩 추가 → Val 개선 시 채택")

N_PATIENCE_FS = 18          # ★ 15→18: 탐색 범위 200 확대에 맞게 조정
MIN_DELTA_FS  = 5e-5
MAX_FEATURES_FS = 200       # ★ 150→200: h128 best 시 ~356개 중 42%→56% 커버리지 확보

fs_hp={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
       "random_state":42,"max_depth":9,"learning_rate":0.02,"scale_pos_weight":spw,
       "subsample":0.8,"colsample_bytree":0.8,"min_child_weight":5,
       "gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0}

selected_fs = []          # 채택된 피처 누적
best_fs_val = -1.0        # 현재까지 best Val AUCPR
no_improve  = 0           # 연속 미개선 횟수
fs_log      = []          # 로그 기록

candidates = [f for f,_ in feat_imp[:MAX_FEATURES_FS]]

print(f"  {'#':>3s} {'Feature':<45s} | {'Val':>9s} {'Δ':>8s} {'채택':>4s}")
print("  " + "-" * 75)

for fi, feat in enumerate(candidates):
    probe = selected_fs + [feat]
    m_fs = xgb.XGBClassifier(**fs_hp, n_estimators=1500, early_stopping_rounds=100)  # ★ 800→1500
    m_fs.fit(X_train[probe], y_train, eval_set=[(X_val[probe], y_val)], verbose=False)
    v = m_fs.best_score
    delta = v - best_fs_val
    adopted = delta > MIN_DELTA_FS

    tag = "✅" if adopted else "  "
    print(f"  {fi+1:3d} {feat:<45s} | {v:9.5f} {delta:+8.5f} {tag}")
    fs_log.append({"feat":feat,"val":v,"delta":delta,"adopted":adopted})

    if adopted:
        selected_fs.append(feat)
        best_fs_val = v
        no_improve = 0
    else:
        no_improve += 1

    del m_fs; gc.collect()

    if no_improve >= N_PATIENCE_FS:
        print(f"\n  ⏹ 연속 {N_PATIENCE_FS}회 미개선 → 조기 종료 (탐색 {fi+1}/{len(candidates)})")
        break

print(f"\n  ✅ Forward Selection 완료: {len(selected_fs)}개 채택")
print(f"  채택 피처: {selected_fs}")
print(f"  Best Val AUCPR: {best_fs_val:.5f}  (Stage1: {s1_val:.5f}  Δ={best_fs_val-s1_val:+.5f})")

# ── Stage 3: HP Grid (Forward Selection 채택 피처 고정) ───────────────
print("\n── Stage 3: HP Grid (Forward Selection 피처 고정) ──")
# Forward Selection이 고른 피처로 HP만 탐색
bp3={"objective":"binary:logistic","eval_metric":"aucpr","tree_method":"hist","device":"cuda",
     "random_state":42,"scale_pos_weight":spw}
hp_grid=[
    {"label":"d8_lr03_c07",  "hp":{"max_depth":8, "learning_rate":0.03, "colsample_bytree":0.7,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":1500}},
    {"label":"d8_lr02_c08",  "hp":{"max_depth":8, "learning_rate":0.02, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d9_lr02_c08",  "hp":{"max_depth":9, "learning_rate":0.02, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d9_lr02_c09",  "hp":{"max_depth":9, "learning_rate":0.02, "colsample_bytree":0.9,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d10_lr02_c08", "hp":{"max_depth":10,"learning_rate":0.02, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":2500}},
    {"label":"d10_lr01_c08", "hp":{"max_depth":10,"learning_rate":0.01, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":4000}},
    {"label":"d11_lr01_c08", "hp":{"max_depth":11,"learning_rate":0.01, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":5000}},  # v7c.11 best
    {"label":"d11_lr01_c09", "hp":{"max_depth":11,"learning_rate":0.01, "colsample_bytree":0.9,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":5000}},  # ★ 신규: d11 colsample 확대
    {"label":"d12_lr01_c08", "hp":{"max_depth":12,"learning_rate":0.01, "colsample_bytree":0.8,"subsample":0.8,"min_child_weight":5,"gamma":0.1,"reg_alpha":0.1,"reg_lambda":1.0,"n_estimators":5000}},  # ★ 신규: depth 한계 탐색
    {"label":"d9_reg",       "hp":{"max_depth":9, "learning_rate":0.02, "colsample_bytree":0.8,"subsample":0.75,"min_child_weight":8,"gamma":0.2,"reg_alpha":0.5,"reg_lambda":2.0,"n_estimators":2500}},
]

print(f"\n📊 {len(hp_grid)}개 HP 조합 탐색 (피처 {len(selected_fs)}개 고정)...")
print(f"  {'#':>2s} {'Label':<20s} | {'d':>2s} {'LR':>5s} {'col':>4s} | {'Val':>9s} {'it':>5s}")
print("  "+"-"*55)
best_s2={"val":-1}
for i,cfg in enumerate(hp_grid):
    hp=cfg["hp"].copy(); ne=hp.pop("n_estimators"); sf=selected_fs
    m=xgb.XGBClassifier(**{**bp3,**hp},n_estimators=ne,early_stopping_rounds=100)
    m.fit(X_train[sf],y_train,eval_set=[(X_val[sf],y_val)],verbose=False)
    va=m.best_score; bi=m.best_iteration
    print(f"  {i+1:2d} {cfg['label']:<20s} | {hp.get('max_depth',8):2d} {hp.get('learning_rate',0.03):5.3f} {hp.get('colsample_bytree',0.7):4.2f} | {va:9.5f} {bi:5d}")
    if va>best_s2["val"]:
        if best_s2.get("model"): del best_s2["model"]
        best_s2={"label":cfg["label"],"features":sf,"val":va,"iter":bi,"model":m,"hp":hp}
    else: del m
    gc.collect()

y_prob=best_s2["model"].predict_proba(X_test[best_s2["features"]])[:,1]
best_features=best_s2["features"]
print(f"\n  🏆 Best: {best_s2['label']} ({len(best_features)}개) Val={best_s2['val']:.5f} iter={best_s2['iter']}")

imp=best_s2["model"].feature_importances_; tidx=np.argsort(imp)[::-1][:30]
print(f"\n🔍 Feature Importance Top 30")
for i,idx in enumerate(tidx):
    c=best_features[idx]; t=""
    if c.startswith("tgat_emb_"): t=" [TGAT]"
    elif c.startswith("graph_"): t=" [GRAPH]"
    elif c.startswith("cond_"): t=" [COND]"
    elif c.startswith("score_"): t=" [SCORE]"
    elif c.startswith("interact_"): t=" [INTER]"
    elif c.startswith("cb_"): t=" [★CB]"
    elif c.startswith("vol_"): t=" [★VOL]"
    elif c.startswith("ws_"): t=" [★WIN]"
    elif c.startswith("inout_"): t=" [★INOUT]"
    print(f"  {i+1:2d}. {c:<45s}: {imp[idx]:.4f}{t}")

ti=imp.sum()
def group_imp(prefix): return sum(imp[i] for i,c in enumerate(best_features) if c.startswith(prefix))
print(f"\n📊 그룹별 중요도:")
print(f"   V2        : {group_imp('') / ti * 100:.1f}% (기존)")
print(f"   Graph     : {group_imp('graph_') / ti * 100:.1f}%")
print(f"   TGAT      : {group_imp('tgat_emb_') / ti * 100:.1f}%")
print(f"   Interact  : {group_imp('interact_') / ti * 100:.1f}%")
print(f"   ★ CB      : {group_imp('cb_') / ti * 100:.1f}%")
print(f"   ★ VolNorm : {group_imp('vol_') / ti * 100:.1f}%")
print(f"   ★ WinShift: {group_imp('ws_') / ti * 100:.1f}%")
print(f"   ★ InOut   : {group_imp('inout_') / ti * 100:.1f}%")
del X_train,X_val; gc.collect()

# =============================================================
# 9. 최종 리포트
# =============================================================
print("\n"+"="*60); print("🏆 [TGAT v7c.19] 최종 리포트"); print("="*60)
auprc=average_precision_score(y_test,y_prob); print(f"\n  AUPRC: {auprc:.4f}")
best_f1=0; best_thresh=0
for th in np.arange(0.05,0.95,0.01):
    f=f1_score(y_test,(y_prob>=th).astype(int),zero_division=0)
    if f>best_f1: best_f1=f; best_thresh=th
y_pred=(y_prob>=best_thresh).astype(int)
prec=precision_score(y_test,y_pred,zero_division=0); rec=recall_score(y_test,y_pred,zero_division=0)
print(f"\n📌 최적: T={best_thresh:.2f} F1={best_f1:.4f} P={prec:.4f} R={rec:.4f}")
print(f"\n📌 임계값별:")
for t in [0.1,0.3,0.5,0.7,0.8,0.9]:
    yp=(y_prob>=t).astype(int)
    print(f"  T={t:.1f} P={precision_score(y_test,yp,zero_division=0):.4f} R={recall_score(y_test,yp,zero_division=0):.4f} F1={f1_score(y_test,yp,zero_division=0):.4f}")

df_res=df_test_meta.with_columns([pl.col("time_group").dt.date().alias("date"),pl.Series("pred_prob",y_prob)])
df_dist=df_res.sort("pred_prob",descending=True).unique(subset=["account_id"],maintain_order=True)
print(f"\n📌 Top-K:")
for k in [100,500,1000,5000,10000]:
    ck=min(k,len(df_dist)); hits=df_dist.head(ck)["is_laundering"].sum()
    print(f"  Top-{k:5d}: {hits:5d}명 ({hits/ck*100:.2f}%)")
print(f"\n📌 일별 Top-100:")
for d in df_dist["date"].unique(maintain_order=True).sort(descending=True):
    hits=df_dist.filter(pl.col("date")==d).head(100)["is_laundering"].sum(); print(f"  {d}: {hits}")

tn,fp,fn,tp=confusion_matrix(y_test,y_pred).ravel()
print(f"\n📌 v7c.18→v7c.19: AUPRC {auprc:.4f} | F1 {best_f1:.4f}")
print(f"  TP:{tp:,} FP:{fp:,} FN:{fn:,} TN:{tn:,}")
print(f"  Precision:{prec:.4f} Recall:{rec:.4f}")

print(f"\n📌 GNN 탐색 결과:")
for r in gnn_results:
    tag = " ★" if r["label"]==best_gnn["label"] else ""
    print(f"  {r['label']:<12s}: QVal={r['q_val']:.5f} QTest={r['q_test']:.4f} Loss={r['loss']:.4f} Params={r['params']:,}{tag}")

print(f"\n📌 3-Stage 결과:")
print(f"  S1 전체({len(xgb_feature_cols)}개): Val={s1_val:.5f} Test={s1_test:.4f}")
print(f"  S2 Forward Selection: {len(selected_fs)}개 채택, Val={best_fs_val:.5f}")
print(f"  S3 HP Grid ({best_s2['label']}): Val={best_s2['val']:.5f} Test={auprc:.4f}")

print(f"\n✨ v7c.19 변경 요약 (v7c.18 대비):")
print(f"  GNN 레이어 비교: TransformerConv(4) + GATv2(2) + NNConv_Max(2) + GINE(2) = 10개 config")
print(f"  Best conv_type : {best_gnn.get('conv_type','transformer')} ({best_gnn['label']})")
print(f"  GNN 입력 차원  : {IN_DIM}차원")
print(f"  XGB 후보       : {len(xgb_feature_cols)}개")
print(f"  FS 채택        : {len(selected_fs)}개")

del X_test; gc.collect(); log_memory("최종 완료")
print(f"\n✅ TGAT v7c.19 완료!")

🛡️ [TGAT v7c.19] 레이어 비교 실험
   ▸ TransformerConv (기존, SUM+attention) × 4
   ▸ GATv2Conv (동적 어텐션 SUM) × 2
   ▸ NNConv+MAX (엣지 변환행렬×MAX 집계) × 2  ← AML 핵심 실험
   ▸ GINEConv+SUM (WL 이론적 최강 표현력) × 2
   총 10개 config → QVal 기준 best 선택 후 3-Stage

📂 데이터 로딩 중...
⏱️ Train cutoff: 2022-09-09 20:00:00
⏱️ Val   cutoff: 2022-09-14 05:00:00
📊 전체 노드 수: 2,076,999
  💾 [데이터 로드 완료] RAM: 103.71GB | GPU: 1.19GB

📐 [Step 2] 전체 노드 피처 계산...
  📊 그래프 구조 피처...
  📊 PageRank...
  ✅ 그래프 피처 13개
  📊 조건부행동 피처...
  ✅ 조건부행동 10개
  📊 AML 점수...
  ✅ AML 점수 5개
  📊 규칙성 피처...
  ✅ 규칙성 7개
  📊 금액 분포 피처...
  ✅ 금액 분포 8개
  📊 ★ Cross-border 정밀화 피처 (신규)...
  ✅ Cross-border 정밀화 4개: ['cb_new_partner_ratio', 'cb_oneway_new', 'cb_amount_asymmetry', 'cb_burst_cross']
  📊 ★ 거래량 대비 상대 정규화 피처 (신규)...
  ✅ 거래량 대비 정규화 5개: ['vol_amount_per_peer', 'vol_concentrated_amount', 'vol_low_freq_high_amt', 'vol_degree_normalized_amount', 'vol_outlier_score']
  📊 ★ 시간 윈도우별 행동 변화 피처 (신규)...
  ✅ 시간 윈도우 변화 3개: ['ws_recent_surge', 'ws_recent_only', 'ws_partner_exp

    [tc_h96_L2] Ep 1 Loss=1.3398


    [tc_h96_L2] Ep10 Loss=1.0351


    [tc_h96_L2] Ep20 Loss=1.0112


    [tc_h96_L2] Ep30 Loss=1.0010


...

    [gine_h128_L2] Ep50 Loss=0.7394


    [gine_h128_L2] Ep60 Loss=0.7305


    [gine_h128_L2] Ep70 Loss=0.7229


    [gine_h128_L2] Ep80 Loss=0.7189


  10 gine_h128_L2             gine |    128      2 |   0.7189  157,283 |  0.33925   0.3079
  💾 [gine_h128_L2] RAM: 86.43GB | GPU: 1.19GB

  🏆 Best GNN: nn_max_h96_L3 (QVal=0.41760 QTest=0.4446)

🔗 Interaction 피처 (nn_max_h96_L3)...
  ✅ Interaction 10개
  💾 [GNN 탐색 완료] RAM: 78.47GB | GPU: 1.19GB

🚀 [Step 7] XGBoost 데이터 구성...
📊 XGBoost 총 후보 피처: 205
   V2:38 G:13 Cond:10 AML:5
   ★추가 Reg:7 Amt:8  (v7c.6 누락 수정)
   CB:4 VolNorm:5
   WinShift:3 InOut:6
   Emb:96 Inter:10
   ★ Forward Selection이 전체 후보에서 유효한 것 선별
📊 Train: 17,172,747 → 619,437 (4.76%)
📊 Val: 5,790,644
📊 Test: 5,792,913
  💾 [XGBoost 준비] RAM: 109.22GB | GPU: 1.19GB

🔥 [Step 8] 3-Stage XGBoost (Full → Forward Selection → HP Grid)...

── Stage 1: 전체 피처 학습 (importance 순위 산출) ──
[0]	validation_0-aucpr:0.11812
[100]	validation_0-aucpr:0.27819
[200]	validation_0-aucpr:0.37525
[300]	validation_0-aucpr:0.39754
[400]	validation_0-aucpr:0.40828
[500]	validation_0-aucpr:0.41543
[600]	validation_0-aucpr:0.42085
[700]	validation_0-aucpr:0.42421